# 08 Transcript Subtitle Data Collection

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/gdrive2', force_remount=True)

# Define the existing project path
PROJECT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

# Verify access to the project
assert PROJECT_DIR.exists(), (
    f"Project folder not found: {PROJECT_DIR}"
)

print("Google Drive mounted successfully.")
print("Project directory:", PROJECT_DIR)
print("Project folder accessible:", PROJECT_DIR.exists())

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile

# ============================================================
# NOTEBOOK 8 — CELL 2: READ-ONLY WORKSPACE AUDIT
# ============================================================

PROJECT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

MASTER_PATH = (
    PROJECT_DIR / "01_data/processed/ICMB_130_MASTER_DATA.csv"
)
MAPPING_PATH = (
    PROJECT_DIR / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
)
ARCHIVE_PATH = (
    PROJECT_DIR / "01_data/indicdialogue/dataset.zip"
)

WORKSPACE = (
    PROJECT_DIR / "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = WORKSPACE / "raw"
VALIDATED_DIR = WORKSPACE / "validated"
MANIFEST_PATH = (
    WORKSPACE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

print("=" * 65)
print("NOTEBOOK 8 — SUBTITLE WORKSPACE AUDIT")
print("=" * 65)

# 1. Verify the locked datasets
assert MASTER_PATH.exists(), f"Master dataset missing: {MASTER_PATH}"
assert MAPPING_PATH.exists(), f"TMDB mapping missing: {MAPPING_PATH}"

master = pd.read_csv(MASTER_PATH)
mapping = pd.read_csv(MAPPING_PATH)

assert len(master) == 130
assert len(mapping) == 130
assert master["movie_record_id"].nunique() == 130
assert mapping["movie_record_id"].nunique() == 130

print("\n[1] LOCKED DATASETS")
print("Master dataset:", master.shape)
print("Unique master records:", master["movie_record_id"].nunique())
print("TMDB mapping:", mapping.shape)
print("Unique mapped records:", mapping["movie_record_id"].nunique())
print("Unique TMDB IDs:", mapping["tmdb_id"].nunique())

# 2. Inspect existing subtitle workspace
print("\n[2] SUBTITLE WORKSPACE")
print("Workspace exists:", WORKSPACE.exists())
print("Raw folder exists:", RAW_DIR.exists())
print("Validated folder exists:", VALIDATED_DIR.exists())
print("Manifest exists:", MANIFEST_PATH.exists())

raw_files = (
    [p for p in RAW_DIR.rglob("*") if p.is_file()]
    if RAW_DIR.exists() else []
)
validated_files = (
    [p for p in VALIDATED_DIR.rglob("*") if p.is_file()]
    if VALIDATED_DIR.exists() else []
)

print("Files in raw:", len(raw_files))
print("Files in validated:", len(validated_files))

# 3. Inspect the manifest without editing it
if MANIFEST_PATH.exists():
    manifest = pd.read_csv(
        MANIFEST_PATH,
        dtype=str,
        keep_default_na=False
    )

    print("\n[3] EXISTING MANIFEST")
    print("Rows:", len(manifest))
    print("Columns:", len(manifest.columns))

    if "acquisition_status" in manifest.columns:
        print("\nAcquisition statuses:")
        print(
            manifest["acquisition_status"]
            .replace("", "(blank)")
            .value_counts()
            .to_string()
        )

    if "validation_status" in manifest.columns:
        print("\nValidation statuses:")
        print(
            manifest["validation_status"]
            .replace("", "(blank)")
            .value_counts()
            .to_string()
        )
else:
    print("\n[3] Manifest not found.")

# 4. Inspect the existing IndicDialogue archive
print("\n[4] INDICDIALOGUE ARCHIVE")
print("Archive exists:", ARCHIVE_PATH.exists())

if ARCHIVE_PATH.exists():
    with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
        names = archive.namelist()
        subtitle_like = [
            name for name in names
            if Path(name).suffix.lower()
            in {".srt", ".vtt", ".ass", ".ssa", ".sub", ".txt"}
        ]

    print("Total archive entries:", len(names))
    print("Subtitle/text-like entries:", len(subtitle_like))

# 5. Check whether the OpenSubtitles key is available
print("\n[5] OPENSUBTITLES API ACCESS")

try:
    from google.colab import userdata
    key = userdata.get("OPEN_SUBTITLES_API_KEY")
    print("API key available:", bool(key))
    del key
except Exception:
    print("API key available: Could not confirm from Colab Secrets.")

print("\n" + "=" * 65)
print("AUDIT COMPLETE — NO FILES MODIFIED")
print("=" * 65)

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from google.colab import userdata

PROJECT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

MAPPING_PATH = (
    PROJECT_DIR / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
)

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
assert API_KEY, "OpenSubtitles API key is unavailable."

mapping = pd.read_csv(MAPPING_PATH, dtype={"tmdb_id": str})

headers = {
    "Api-Key": API_KEY,
    "User-Agent": "ICMB130SubtitleResearch v1",
}

session = requests.Session()
session.headers.update(headers)

tests = [
    {
        "label": "83 — Hindi by TMDB ID",
        "params": {"tmdb_id": "601337", "languages": "hi", "type": "movie"},
    },
    {
        "label": "83 — Hindi by longer query",
        "params": {
            "query": "83 2021 India",
            "languages": "hi",
            "year": 2021,
            "type": "movie",
        },
    },
    {
        "label": "83 — English by longer query",
        "params": {
            "query": "83 2021 India",
            "languages": "en",
            "year": 2021,
            "type": "movie",
        },
    },
    {
        "label": "Kaho Naa... Pyaar Hai — Hindi",
        "params": {
            "query": "Kaho Naa Pyaar Hai",
            "languages": "hi",
            "year": 2000,
            "type": "movie",
        },
    },
]

print("=" * 65)
print("OPENSUBTITLES SEARCH METHOD DIAGNOSTIC")
print("=" * 65)

for test in tests:
    print(f"\n--- {test['label']} ---")

    try:
        response = session.get(
            "https://api.opensubtitles.com/api/v1/subtitles",
            params=test["params"],
            timeout=30,
        )

        print("HTTP status:", response.status_code)

        if not response.ok:
            print("API response:", response.text[:500])
            continue

        payload = response.json()
        candidates = payload.get("data", [])
        print("Candidates:", len(candidates))

        for candidate in candidates[:5]:
            attributes = candidate.get("attributes", {})
            details = attributes.get("feature_details") or {}

            print({
                "subtitle_id": candidate.get("id"),
                "language": attributes.get("language"),
                "movie_title": details.get("movie_name"),
                "movie_year": details.get("year"),
                "tmdb_id": details.get("tmdb_id"),
                "files": [
                    file.get("file_name")
                    for file in attributes.get("files", [])
                ],
            })

    except Exception as exc:
        print("Request error:", str(exc)[:400])

print("\nDiagnostic finished. No project files modified.")

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# NOTEBOOK 8 — CLEAN ACTIVE SUBTITLE WORKSPACE
# Protected datasets and archives will not be touched.
# ============================================================

PROJECT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

WORKSPACE = (
    PROJECT_DIR / "01_data/processed/subtitle_corpus_v2"
)

RAW_DIR = WORKSPACE / "raw"
VALIDATED_DIR = WORKSPACE / "validated"

MANIFEST_PATH = (
    WORKSPACE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

TEMP_REPORTS = [
    WORKSPACE / "OPENSUBTITLES_SEARCH_RESULTS.csv",
    WORKSPACE / "OPENSUBTITLES_SEARCH_ERRORS.csv",
]

assert PROJECT_DIR.exists(), "Project directory not found."
assert WORKSPACE.exists(), "Subtitle workspace not found."

# Confirm the protected source files exist.
MASTER_PATH = (
    PROJECT_DIR / "01_data/processed/ICMB_130_MASTER_DATA.csv"
)
MAPPING_PATH = (
    PROJECT_DIR / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
)
ARCHIVE_PATH = (
    PROJECT_DIR / "01_data/indicdialogue/dataset.zip"
)

assert MASTER_PATH.exists(), "Locked master dataset missing."
assert MAPPING_PATH.exists(), "Locked TMDB mapping missing."
assert ARCHIVE_PATH.exists(), "IndicDialogue archive missing."

# Display exactly what would be removed.
targets = []

for folder in [RAW_DIR, VALIDATED_DIR]:
    if folder.exists():
        targets.extend(
            p for p in folder.rglob("*")
            if p.is_file()
        )

targets.extend(
    p for p in TEMP_REPORTS
    if p.exists()
)

print("=" * 65)
print("ACTIVE WORKSPACE RESET — REVIEW")
print("=" * 65)
print("Files to remove:", len(targets))

for path in targets:
    print("-", path.relative_to(WORKSPACE))

print("\nThe manifest will be reset to 0 records.")
print("Protected master, mapping and IndicDialogue archive: untouched.")
print("Previously archived workflow: untouched.")

confirmation = input(
    "\nType RESET_SUBTITLE_WORKSPACE to continue: "
).strip()

if confirmation != "RESET_SUBTITLE_WORKSPACE":
    print("\nCancelled. No files were removed.")
else:
    # Remove only files inside the active raw/validated folders.
    for folder in [RAW_DIR, VALIDATED_DIR]:
        folder.mkdir(parents=True, exist_ok=True)

        for path in sorted(
            folder.rglob("*"),
            key=lambda p: len(p.parts),
            reverse=True
        ):
            if path.is_file() or path.is_symlink():
                path.unlink()
            elif path.is_dir():
                path.rmdir()

    # Remove only temporary discovery reports.
    for path in TEMP_REPORTS:
        if path.exists():
            path.unlink()

    # Preserve the manifest schema, but reset its records.
    if MANIFEST_PATH.exists():
        old_manifest = pd.read_csv(
            MANIFEST_PATH,
            dtype=str,
            keep_default_na=False
        )
        columns = old_manifest.columns.tolist()
    else:
        columns = [
            "movie_record_id", "year", "rank", "movie_title",
            "language", "tmdb_id", "imdb_id", "subtitle_language",
            "subtitle_type", "subtitle_source", "source_id",
            "source_url", "raw_subtitle_file",
            "validated_subtitle_file", "acquisition_status",
            "validation_status", "title_verified", "year_verified",
            "movie_identity_verified", "language_verified",
            "dialogue_blocks", "word_count", "character_count",
            "encoding_status", "quality_status", "notes"
        ]

    pd.DataFrame(columns=columns).to_csv(
        MANIFEST_PATH, index=False
    )

    # Verify the reset.
    remaining_raw = [
        p for p in RAW_DIR.rglob("*") if p.is_file()
    ]
    remaining_validated = [
        p for p in VALIDATED_DIR.rglob("*") if p.is_file()
    ]
    check_manifest = pd.read_csv(MANIFEST_PATH)

    assert len(remaining_raw) == 0
    assert len(remaining_validated) == 0
    assert len(check_manifest) == 0

    print("\nWorkspace reset completed.")
    print("Files in raw:", len(remaining_raw))
    print("Files in validated:", len(remaining_validated))
    print("Manifest records:", len(check_manifest))
    print("Manifest columns preserved:", len(columns))
    print("Protected source files were not modified.")

In [ ]:
import re
import time
import hashlib
import requests
import pandas as pd
from pathlib import Path
from google.colab import userdata

# ============================================================
# NOTEBOOK 8 — CELL 7
# TEST DOWNLOADER + BASIC SUBTITLE QUALITY CHECK
# ============================================================

PROJECT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

WORKSPACE = (
    PROJECT_DIR / "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = WORKSPACE / "raw"
MANIFEST_PATH = (
    WORKSPACE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)
MAPPING_PATH = (
    PROJECT_DIR / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
assert API_KEY, "OpenSubtitles API key is unavailable."

mapping = pd.read_csv(MAPPING_PATH, dtype={"tmdb_id": str})
movie = mapping.loc[
    mapping["movie_record_id"].astype(str) == "2021_4"
].iloc[0]

TITLE = str(movie["dataset_title"])
YEAR = int(movie["dataset_year"])
TMDB_ID = str(movie["tmdb_id"])
RECORD_ID = str(movie["movie_record_id"])

assert TITLE == "83" and YEAR == 2021 and TMDB_ID == "601337"

headers = {
    "Api-Key": API_KEY,
    "User-Agent": "ICMB130SubtitleResearch v1",
}

session = requests.Session()
session.headers.update(headers)

# 1. Search original-language candidates by exact TMDB ID.
print(f"Searching {TITLE} ({YEAR}) by TMDB ID {TMDB_ID}...")

response = session.get(
    "https://api.opensubtitles.com/api/v1/subtitles",
    params={
        "tmdb_id": TMDB_ID,
        "languages": "hi",
        "type": "movie",
        "page": 1,
    },
    timeout=30,
)
response.raise_for_status()

candidates = response.json().get("data", [])

# Keep only candidates whose returned movie identity matches.
candidates = [
    item for item in candidates
    if str(
        (item.get("attributes", {}).get("feature_details") or {}).get("tmdb_id", "")
    ) == TMDB_ID
]

if not candidates:
    raise RuntimeError(
        "No Hindi candidates returned for the exact TMDB ID. "
        "Do not infer that the subtitle is unavailable; try alternate searches."
    )

print("Identity-matched Hindi candidates:", len(candidates))

# Prefer a candidate with downloadable file metadata.
candidate = next(
    (
        item for item in candidates
        if item.get("attributes", {}).get("files")
    ),
    None,
)
assert candidate is not None, "No candidate includes file metadata."

attributes = candidate["attributes"]
file_info = attributes["files"][0]
file_id = file_info.get("file_id")
assert file_id is not None, "No downloadable file ID returned."

# 2. Request a download link for the chosen file.
download_response = session.post(
    "https://api.opensubtitles.com/api/v1/download",
    json={"file_id": file_id, "sub_format": "srt"},
    timeout=30,
)
download_response.raise_for_status()

download_info = download_response.json()
download_link = download_info.get("link")
assert download_link, "No download URL returned."

# 3. Download actual subtitle content.
content_response = requests.get(download_link, timeout=60)
content_response.raise_for_status()
content = content_response.content

# 4. Decode and perform basic structural checks.
decoded = None
encoding_used = None

for encoding in ["utf-8-sig", "utf-8", "cp1252"]:
    try:
        decoded = content.decode(encoding)
        encoding_used = encoding
        break
    except UnicodeDecodeError:
        continue

if decoded is None:
    raise RuntimeError(
        "Downloaded file could not be decoded using the tested encodings."
    )

timestamp_pattern = re.compile(
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
)

timestamp_matches = list(timestamp_pattern.finditer(decoded))
replacement_count = decoded.count("\ufffd")

# Remove timestamps, sequence numbers, and common formatting tags
# for a preliminary text-volume estimate.
clean_lines = []

for line in decoded.splitlines():
    line = line.strip()

    if not line or re.fullmatch(r"\d+", line):
        continue
    if timestamp_pattern.search(line):
        continue

    line = re.sub(r"\{[^}]*\}", "", line)
    line = re.sub(r"<[^>]*>", "", line)
    line = line.strip()

    if line:
        clean_lines.append(line)

clean_text = "\n".join(clean_lines)
word_count = len(
    re.findall(r"\S+", clean_text, flags=re.UNICODE)
)
character_count = len(clean_text)

# Preliminary acceptance gate only; detailed timing and movie-content
# validation are still required.
basic_checks_pass = (
    len(content) >= 10_000
    and len(timestamp_matches) >= 500
    and word_count >= 5_000
    and replacement_count == 0
)

print("\n" + "=" * 65)
print("DOWNLOADED CANDIDATE — BASIC QUALITY REPORT")
print("=" * 65)
print("Movie:", TITLE, f"({YEAR})")
print("TMDB ID:", TMDB_ID)
print("Subtitle ID:", candidate.get("id"))
print("Candidate language:", attributes.get("language"))
print("Filename:", file_info.get("file_name"))
print("File size:", len(content), "bytes")
print("Encoding:", encoding_used)
print("Timestamped blocks:", len(timestamp_matches))
print("Whitespace-separated words:", word_count)
print("Text characters:", character_count)
print("Replacement characters:", replacement_count)
print("Basic quality gate:", "PASS" if basic_checks_pass else "REVIEW REQUIRED")
print("SHA-256:", hashlib.sha256(content).hexdigest())

if basic_checks_pass:
    output_path = RAW_DIR / "2021_4_83_HINDI_OPENSUBTITLES.srt"

    # Prevent accidental overwrite of a previously saved candidate.
    if output_path.exists():
        raise FileExistsError(
            f"Candidate already exists; not overwriting: {output_path}"
        )

    output_path.write_bytes(content)
    print("Saved candidate:", output_path)
else:
    print(
        "Candidate not saved: basic checks failed. "
        "We will inspect another candidate or use fallback."
    )

print("\nManifest has not been modified.")
print("Detailed validation is still required before acceptance.")

In [ ]:
from pathlib import Path
import re
import hashlib
from datetime import timedelta

# ============================================================
# NOTEBOOK 8 — CELL 8: DETAILED VALIDATION FOR 83
# Read-only: does not modify the subtitle or manifest
# ============================================================

PROJECT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_PATH = (
    PROJECT_DIR
    / "01_data/processed/subtitle_corpus_v2/raw/"
      "2021_4_83_HINDI_OPENSUBTITLES.srt"
)

assert RAW_PATH.exists(), f"Subtitle file not found: {RAW_PATH}"

raw_bytes = RAW_PATH.read_bytes()
text = raw_bytes.decode("utf-8-sig")

timestamp_pattern = re.compile(
    r"^\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
)

def to_milliseconds(timestamp):
    timestamp = timestamp.replace(",", ".")
    hours, minutes, seconds = timestamp.split(":")
    whole_seconds, milliseconds = seconds.split(".")
    return (
        int(hours) * 3_600_000
        + int(minutes) * 60_000
        + int(whole_seconds) * 1_000
        + int(milliseconds)
    )

blocks = re.split(r"\n\s*\n", text.strip())

valid_blocks = []
malformed_timestamp_blocks = []
timestamp_records = []
content_lines = []
empty_blocks = 0

for block in blocks:
    lines = block.splitlines()
    timestamp_index = None
    match = None

    for i, line in enumerate(lines):
        candidate_match = timestamp_pattern.match(line)
        if candidate_match:
            timestamp_index = i
            match = candidate_match
            break

    if match is None:
        if "-->" in block:
            malformed_timestamp_blocks.append(block[:120])
        continue

    start_ms = to_milliseconds(match.group(1))
    end_ms = to_milliseconds(match.group(2))

    dialogue = "\n".join(lines[timestamp_index + 1:]).strip()
    dialogue_clean = re.sub(r"\{[^}]*\}", "", dialogue)
    dialogue_clean = re.sub(r"<[^>]*>", "", dialogue_clean).strip()

    valid_blocks.append(block)
    timestamp_records.append((start_ms, end_ms))

    if dialogue_clean:
        content_lines.append(dialogue_clean)
    else:
        empty_blocks += 1

# Timing checks
nonpositive_duration = sum(
    end <= start for start, end in timestamp_records
)

sorted_records = sorted(timestamp_records)
overlaps = sum(
    sorted_records[i][1] > sorted_records[i + 1][0]
    for i in range(len(sorted_records) - 1)
)

gaps_over_5min = sum(
    sorted_records[i + 1][0] - sorted_records[i][1] > 300_000
    for i in range(len(sorted_records) - 1)
)

start_ms = min(start for start, _ in timestamp_records)
end_ms = max(end for _, end in timestamp_records)

duration_seconds = max(0, end_ms - start_ms)
duration = str(timedelta(seconds=duration_seconds / 1000))

clean_text = "\n".join(content_lines)
word_count = len(clean_text.split())

devanagari_count = len(
    re.findall(r"[\u0900-\u097F]", clean_text)
)
latin_count = len(re.findall(r"[A-Za-z]", clean_text))
replacement_count = text.count("\ufffd")

print("=" * 65)
print("83 (2021) — DETAILED SUBTITLE VALIDATION")
print("=" * 65)
print("File size:", len(raw_bytes), "bytes")
print("SHA-256:", hashlib.sha256(raw_bytes).hexdigest())
print("Subtitle blocks:", len(valid_blocks))
print("Blocks with readable content:", len(content_lines))
print("Empty-content blocks:", empty_blocks)
print("Malformed timestamp blocks:", len(malformed_timestamp_blocks))
print("Non-positive duration blocks:", nonpositive_duration)
print("Overlapping adjacent blocks:", overlaps)
print("Gaps greater than 5 minutes:", gaps_over_5min)
print("Timestamp coverage:", duration)
print("Whitespace-separated words:", word_count)
print("Devanagari characters:", devanagari_count)
print("Latin characters:", latin_count)
print("Replacement characters:", replacement_count)

print("\nFirst 12 readable content lines:")
for line in content_lines[:12]:
    print("-", line[:180])

print("\nLast 12 readable content lines:")
for line in content_lines[-12:]:
    print("-", line[:180])

print("\nNo files modified. Manifest unchanged.")

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib
import re

# ============================================================
# NOTEBOOK 8 — CELL 9: FINALIZE VALIDATED SUBTITLE
# ============================================================

PROJECT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

WORKSPACE = (
    PROJECT_DIR / "01_data/processed/subtitle_corpus_v2"
)

RAW_PATH = (
    WORKSPACE / "raw/2021_4_83_HINDI_OPENSUBTITLES.srt"
)

VALIDATED_DIR = WORKSPACE / "validated"
VALIDATED_PATH = (
    VALIDATED_DIR / "2021_4_83_HINDI.srt"
)

MANIFEST_PATH = (
    WORKSPACE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

MAPPING_PATH = (
    PROJECT_DIR / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
)

assert RAW_PATH.exists(), "Raw candidate is missing."
assert not VALIDATED_PATH.exists(), (
    "Validated destination already exists. Inspect before overwriting."
)

mapping = pd.read_csv(MAPPING_PATH, dtype={"tmdb_id": str})
movie = mapping.loc[
    mapping["movie_record_id"].astype(str) == "2021_4"
].iloc[0]

raw_bytes = RAW_PATH.read_bytes()
raw_hash = hashlib.sha256(raw_bytes).hexdigest()

# Recheck the agreed basic acceptance criteria.
text = raw_bytes.decode("utf-8-sig")

timestamp_pattern = re.compile(
    r"^\s*\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
    r"\d{2}:\d{2}:\d{2}[,.]\d{3}"
)

timestamp_count = sum(
    bool(timestamp_pattern.match(line))
    for line in text.splitlines()
)

clean_lines = []
for line in text.splitlines():
    line = line.strip()
    if not line or re.fullmatch(r"\d+", line):
        continue
    if timestamp_pattern.match(line):
        continue
    line = re.sub(r"\{[^}]*\}", "", line)
    line = re.sub(r"<[^>]*>", "", line).strip()
    if line:
        clean_lines.append(line)

clean_text = "\n".join(clean_lines)
word_count = len(clean_text.split())
character_count = len(clean_text)

assert str(movie["tmdb_id"]) == "601337"
assert str(movie["dataset_title"]) == "83"
assert int(movie["dataset_year"]) == 2021
assert timestamp_count >= 500
assert word_count >= 5000
assert "\ufffd" not in text

# Prepare the manifest row using the existing schema.
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype=str,
    keep_default_na=False
)

assert "movie_record_id" in manifest.columns

# Do not create a duplicate movie record.
assert not (
    manifest["movie_record_id"] == "2021_4"
).any(), "Manifest already contains 83; inspect before proceeding."

row = {column: "" for column in manifest.columns}

values = {
    "movie_record_id": "2021_4",
    "year": "2021",
    "rank": str(movie.get("rank", "")),
    "movie_title": "83",
    "language": "Hindi",
    "tmdb_id": "601337",
    "subtitle_language": "Hindi",
    "subtitle_type": "Original_Language",
    "subtitle_source": "OpenSubtitles",
    "source_id": "6410753",
    "raw_subtitle_file": str(RAW_PATH),
    "validated_subtitle_file": str(VALIDATED_PATH),
    "acquisition_status": "Acquired",
    "validation_status": "Validated",
    "title_verified": "Verified",
    "year_verified": "Verified",
    "movie_identity_verified": "Verified",
    "language_verified": "Verified",
    "dialogue_blocks": str(timestamp_count),
    "word_count": str(word_count),
    "character_count": str(character_count),
    "encoding_status": "UTF-8",
    "quality_status": "Passed",
    "notes": (
        "Hindi subtitle; TMDB ID and film content verified. "
        "897 timestamped blocks; approximately 2h39m coverage; "
        "no malformed timestamps, overlaps, or gaps over 5 minutes."
    ),
}

for column, value in values.items():
    if column in row:
        row[column] = value

# Write the validated file first, verify its bytes, then remove raw copy.
VALIDATED_DIR.mkdir(parents=True, exist_ok=True)
VALIDATED_PATH.write_bytes(raw_bytes)

assert hashlib.sha256(VALIDATED_PATH.read_bytes()).hexdigest() == raw_hash

updated_manifest = pd.concat(
    [manifest, pd.DataFrame([row])],
    ignore_index=True
)

temp_path = MANIFEST_PATH.with_suffix(".tmp.csv")
updated_manifest.to_csv(temp_path, index=False)
temp_path.replace(MANIFEST_PATH)

# Only remove raw after the validated copy and manifest are safely written.
assert VALIDATED_PATH.exists()
assert hashlib.sha256(VALIDATED_PATH.read_bytes()).hexdigest() == raw_hash
RAW_PATH.unlink()

# Read-back verification.
check = pd.read_csv(
    MANIFEST_PATH,
    dtype=str,
    keep_default_na=False
)

record = check.loc[
    check["movie_record_id"] == "2021_4"
]

assert len(record) == 1
assert record.iloc[0]["validation_status"] == "Validated"
assert not RAW_PATH.exists()
assert VALIDATED_PATH.exists()

print("=" * 65)
print("83 — FINALIZED")
print("=" * 65)
print("Validated file:", VALIDATED_PATH)
print("SHA-256:", raw_hash)
print("Manifest records:", len(check))
print("Record ID:", record.iloc[0]["movie_record_id"])
print("Language:", record.iloc[0]["subtitle_language"])
print("Validation status:", record.iloc[0]["validation_status"])
print("Raw duplicate removed:", not RAW_PATH.exists())
print("Master and mapping modified: No")
print("\nRead-back verification passed.")

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# NOTEBOOK 8 — CELL 10: PREPARE FIRST ACQUISITION BATCH
# Corrected merge: TMDB IDs come from the mapping
# ============================================================

PROJECT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

MASTER_PATH = (
    PROJECT_DIR / "01_data/processed/ICMB_130_MASTER_DATA.csv"
)
MAPPING_PATH = (
    PROJECT_DIR / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
)
WORKSPACE = (
    PROJECT_DIR / "01_data/processed/subtitle_corpus_v2"
)
MANIFEST_PATH = (
    WORKSPACE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

master = pd.read_csv(MASTER_PATH, dtype={"movie_record_id": str})
mapping = pd.read_csv(
    MAPPING_PATH,
    dtype={"movie_record_id": str, "tmdb_id": str}
)
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype=str,
    keep_default_na=False
)

# Select mapping columns explicitly to avoid duplicate TMDB fields.
mapping_columns = [
    "movie_record_id",
    "tmdb_id",
    "tmdb_title",
    "tmdb_original_language",
    "dataset_title",
    "dataset_year",
    "dataset_language",
]

mapping_subset = mapping[mapping_columns].copy()

# Use the master dataset as the base and mapping as the ID authority.
movies = master.drop(
    columns=[
        col for col in [
            "tmdb_id",
            "tmdb_title",
            "tmdb_original_language",
            "dataset_title",
            "dataset_year",
            "dataset_language",
        ]
        if col in master.columns
    ]
).merge(
    mapping_subset,
    on="movie_record_id",
    how="left",
    validate="one_to_one",
)

# Integrity checks
assert len(master) == 130
assert len(mapping) == 130
assert len(movies) == 130
assert movies["movie_record_id"].nunique() == 130
assert movies["tmdb_id"].notna().all()
assert movies["tmdb_id"].str.strip().ne("").all()

# Exclude records already validated.
validated_ids = set(
    manifest.loc[
        manifest["validation_status"].eq("Validated"),
        "movie_record_id",
    ]
)

pending = movies.loc[
    ~movies["movie_record_id"].isin(validated_ids)
].copy()

batch = pending.head(10).copy()

print("=" * 65)
print("FIRST SUBTITLE ACQUISITION BATCH")
print("=" * 65)
print("Total movies:", len(movies))
print("Already validated:", len(validated_ids))
print("Remaining:", len(pending))
print("Batch size:", len(batch))

display(
    batch[
        [
            "movie_record_id",
            "dataset_title",
            "dataset_year",
            "dataset_language",
            "tmdb_id",
            "tmdb_title",
            "tmdb_original_language",
        ]
    ].reset_index(drop=True)
)

print("\nNo files modified.")

In [ ]:

import requests
import pandas as pd
import time
from pathlib import Path
from IPython.display import display

# Reuse the 'batch' DataFrame created in the previous cell.
API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")

if not API_KEY:
    raise ValueError("OpenSubtitles API key not found in Colab Secrets.")

headers = {
    "Api-Key": API_KEY,
    "User-Agent": "ICMB130SubtitleResearch v1"
}

search_url = "https://api.opensubtitles.com/api/v1/subtitles"
results = []

for _, movie in batch.iterrows():
    movie_id = str(movie["movie_record_id"])
    tmdb_id = str(movie["tmdb_id"])
    title = movie["dataset_title"]
    year = int(movie["dataset_year"])
    language = str(movie["tmdb_original_language"]).strip()

    try:
        response = requests.get(
            search_url,
            headers=headers,
            params={
                "tmdb_id": tmdb_id,
                "languages": language,
                "type": "movie"
            },
            timeout=30
        )

        if response.status_code != 200:
            results.append({
                "movie_record_id": movie_id,
                "title": title,
                "year": year,
                "tmdb_id": tmdb_id,
                "search_language": language,
                "candidate_count": 0,
                "candidate_ids": "",
                "candidate_files": "",
                "search_status": f"HTTP {response.status_code}"
            })
            continue

        candidates = response.json().get("data", [])

        # Keep only candidates explicitly associated with this TMDB movie.
        candidates = [
            item for item in candidates
            if str(item.get("attributes", {})
                      .get("feature_details", {})
                      .get("tmdb_id", "")) == tmdb_id
        ]

        candidate_ids = []
        candidate_files = []

        for item in candidates:
            attrs = item.get("attributes", {})
            for file_info in attrs.get("files", []):
                candidate_ids.append(str(file_info.get("file_id", "")))
                candidate_files.append(str(file_info.get("file_name", "")))

        results.append({
            "movie_record_id": movie_id,
            "title": title,
            "year": year,
            "tmdb_id": tmdb_id,
            "search_language": language,
            "candidate_count": len(candidates),
            "candidate_ids": " | ".join(candidate_ids),
            "candidate_files": " | ".join(candidate_files),
            "search_status": "Search completed"
        })

    except requests.RequestException as exc:
        results.append({
            "movie_record_id": movie_id,
            "title": title,
            "year": year,
            "tmdb_id": tmdb_id,
            "search_language": language,
            "candidate_count": 0,
            "candidate_ids": "",
            "candidate_files": "",
            "search_status": f"Request error: {type(exc).__name__}"
        })

    time.sleep(0.3)

search_results = pd.DataFrame(results)

print("ORIGINAL-LANGUAGE SUBTITLE SEARCH — BATCH 1")
print("Movies searched:", len(search_results))
print("Movies with candidates:",
      (search_results["candidate_count"] > 0).sum())
print("Movies without candidates:",
      (search_results["candidate_count"] == 0).sum())

display(search_results)


In [ ]:

import re
import time
import hashlib
import requests
from pathlib import Path
from IPython.display import display
import pandas as pd

WORKSPACE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = WORKSPACE / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
if not API_KEY:
    raise ValueError("OpenSubtitles API key not found in Colab Secrets.")

headers = {
    "Api-Key": API_KEY,
    "User-Agent": "ICMB130SubtitleResearch v1"
}

download_api = "https://api.opensubtitles.com/api/v1/download"
downloaded_results = []

available = search_results[
    search_results["candidate_count"] > 0
].copy()

for _, movie in available.iterrows():
    ids = [
        x.strip() for x in str(movie["candidate_ids"]).split("|")
        if x.strip()
    ]
    filenames = [
        x.strip() for x in str(movie["candidate_files"]).split("|")
    ]

    for index, file_id in enumerate(ids, start=1):
        result = {
            "movie_record_id": movie["movie_record_id"],
            "title": movie["title"],
            "year": movie["year"],
            "file_id": file_id,
            "source_filename": (
                filenames[index - 1]
                if index - 1 < len(filenames) else ""
            ),
            "download_status": "",
            "local_file": "",
            "bytes": 0,
            "sha256": "",
            "timestamp_blocks": 0,
            "readable_blocks": 0,
            "word_count": 0,
            "devanagari_chars": 0,
            "latin_chars": 0,
            "replacement_chars": 0,
            "preview": ""
        }

        try:
            response = requests.post(
                download_api,
                headers=headers,
                json={"file_id": int(file_id), "sub_format": "srt"},
                timeout=30
            )

            if response.status_code != 200:
                result["download_status"] = (
                    f"Download API HTTP {response.status_code}"
                )
                downloaded_results.append(result)
                time.sleep(0.5)
                continue

            download_info = response.json()
            link = download_info.get("link")

            if not link:
                result["download_status"] = "No download link returned"
                downloaded_results.append(result)
                time.sleep(0.5)
                continue

            file_response = requests.get(link, timeout=60)
            file_response.raise_for_status()
            content = file_response.content

            # Decode without silently discarding unreadable bytes.
            try:
                text = content.decode("utf-8-sig")
            except UnicodeDecodeError:
                text = content.decode("cp1252", errors="replace")

            local_path = RAW_DIR / (
                f"{movie['movie_record_id']}_candidate{index}.srt"
            )
            local_path.write_text(text, encoding="utf-8")

            blocks = re.split(r"\n\s*\n", text.strip())
            timestamp_pattern = re.compile(
                r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
                r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
            )

            valid_blocks = []
            readable_blocks = 0

            for block in blocks:
                if timestamp_pattern.search(block):
                    valid_blocks.append(block)
                    lines = block.splitlines()
                    dialogue = " ".join(
                        line.strip() for line in lines
                        if not timestamp_pattern.search(line)
                        and not re.fullmatch(r"\s*\d+\s*", line)
                    ).strip()
                    if re.search(r"[^\W\d_]", dialogue, re.UNICODE):
                        readable_blocks += 1

            result.update({
                "download_status": "Downloaded",
                "local_file": str(local_path),
                "bytes": len(content),
                "sha256": hashlib.sha256(content).hexdigest(),
                "timestamp_blocks": len(valid_blocks),
                "readable_blocks": readable_blocks,
                "word_count": len(re.findall(r"\S+", text)),
                "devanagari_chars": len(re.findall(r"[\u0900-\u097F]", text)),
                "latin_chars": len(re.findall(r"[A-Za-z]", text)),
                "replacement_chars": text.count("\uFFFD"),
                "preview": re.sub(r"\s+", " ", text)[:220]
            })

        except Exception as exc:
            result["download_status"] = (
                f"Error: {type(exc).__name__}: {str(exc)[:120]}"
            )

        downloaded_results.append(result)
        time.sleep(0.5)

download_report = pd.DataFrame(downloaded_results)

print("CANDIDATE DOWNLOAD AND INITIAL CONTENT INSPECTION")
print("Candidates attempted:", len(download_report))
print("Successfully downloaded:",
      (download_report["download_status"] == "Downloaded").sum())

display(download_report[[
    "movie_record_id", "title", "file_id", "source_filename",
    "download_status", "bytes", "timestamp_blocks", "readable_blocks",
    "word_count", "devanagari_chars", "latin_chars",
    "replacement_chars", "preview"
]])


In [ ]:

import re
from pathlib import Path
from datetime import timedelta

subtitle_path = RAW_DIR / "2000_2_candidate1.srt"

if not subtitle_path.exists():
    raise FileNotFoundError(f"Candidate not found: {subtitle_path}")

text = subtitle_path.read_text(encoding="utf-8-sig")
blocks = re.split(r"\n\s*\n", text.strip())

timestamp_pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def to_seconds(groups):
    h, m, s, ms = map(int, groups)
    return h * 3600 + m * 60 + s + ms / 1000

valid = []
malformed = []
empty_dialogue = 0

for block in blocks:
    match = timestamp_pattern.search(block)
    if not match:
        if "-->" in block:
            malformed.append(block[:120])
        continue

    groups = match.groups()
    start = to_seconds(groups[:4])
    end = to_seconds(groups[4:])

    lines = block.splitlines()
    dialogue_lines = [
        line.strip() for line in lines
        if not timestamp_pattern.search(line)
        and not re.fullmatch(r"\s*\d+\s*", line)
    ]
    dialogue = " ".join(dialogue_lines).strip()
    dialogue = re.sub(r"<[^>]+>", "", dialogue).strip()

    if not re.search(r"[^\W\d_]", dialogue, re.UNICODE):
        empty_dialogue += 1

    valid.append({
        "start": start,
        "end": end,
        "duration": end - start,
        "dialogue": dialogue
    })

valid.sort(key=lambda x: x["start"])

nonpositive = sum(x["duration"] <= 0 for x in valid)
overlaps = sum(
    valid[i]["start"] < valid[i - 1]["end"]
    for i in range(1, len(valid))
)
large_gaps = sum(
    valid[i]["start"] - valid[i - 1]["end"] > 300
    for i in range(1, len(valid))
)

readable_dialogue = [
    x["dialogue"] for x in valid
    if re.search(r"[^\W\d_]", x["dialogue"], re.UNICODE)
]

print("MOVIE: Mohabbatein (2000)")
print("Parsed timestamp blocks:", len(valid))
print("Malformed timestamp blocks:", len(malformed))
print("Nonpositive durations:", nonpositive)
print("Overlapping consecutive blocks:", overlaps)
print("Gaps exceeding 5 minutes:", large_gaps)
print("Blocks without readable dialogue:", empty_dialogue)

if valid:
    print("First timestamp (seconds):", valid[0]["start"])
    print("Last ending timestamp (seconds):", valid[-1]["end"])
    print("Approximate coverage:",
          str(timedelta(seconds=int(valid[-1]["end"]))))
    print("\nFirst 12 dialogue samples:")
    for line in readable_dialogue[:12]:
        print("-", line[:220])

    print("\nLast 8 dialogue samples:")
    for line in readable_dialogue[-8:]:
        print("-", line[:220])


In [ ]:

import shutil
import re
import pandas as pd
from pathlib import Path

# Paths
MANIFEST_PATH = WORKSPACE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
source_path = RAW_DIR / "2000_2_candidate1.srt"
validated_path = WORKSPACE / "validated" / "2000_2_MOHABBATEIN_HINDI.srt"

# Safety checks
if not source_path.exists():
    raise FileNotFoundError(f"Downloaded subtitle not found: {source_path}")

manifest = pd.read_csv(
    MANIFEST_PATH, dtype=str, keep_default_na=False
)

required_columns = {
    "movie_record_id", "year", "rank", "movie_title", "language",
    "tmdb_id", "imdb_id", "subtitle_language", "subtitle_type",
    "subtitle_source", "source_id", "source_url", "raw_subtitle_file",
    "validated_subtitle_file", "acquisition_status", "validation_status",
    "title_verified", "year_verified", "movie_identity_verified",
    "language_verified", "dialogue_blocks", "word_count",
    "character_count", "encoding_status", "quality_status", "notes"
}
assert required_columns.issubset(manifest.columns), "Manifest schema mismatch"

# Confirm this is the expected movie and subtitle
movie = batch.loc[batch["movie_record_id"].eq("2000_2")].iloc[0]
assert str(movie["tmdb_id"]) == "11518"
assert len(valid) == 1739
assert not malformed
assert nonpositive == 0
assert overlaps == 0

subtitle_text = source_path.read_text(encoding="utf-8")
assert len(subtitle_text.strip()) > 0

# Copy accepted subtitle into validated/
validated_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source_path, validated_path)

# Record acceptance and the known gap
record = {
    "movie_record_id": "2000_2",
    "year": "2000",
    "rank": str(movie["movie_record_id"].split("_")[1]),
    "movie_title": "Mohabbatein",
    "language": "Hindi",
    "tmdb_id": "11518",
    "imdb_id": "",
    "subtitle_language": "Hindi (with English dialogue)",
    "subtitle_type": "Original_Language",
    "subtitle_source": "OpenSubtitles",
    "source_id": "11959991",
    "source_url": "",
    "raw_subtitle_file": source_path.name,
    "validated_subtitle_file": validated_path.name,
    "acquisition_status": "Acquired",
    "validation_status": "Validated",
    "title_verified": "True",
    "year_verified": "True",
    "movie_identity_verified": "True",
    "language_verified": "True",
    "dialogue_blocks": str(len(valid)),
    "word_count": str(len(re.findall(r"\S+", subtitle_text))),
    "character_count": str(len(subtitle_text)),
    "encoding_status": "UTF-8",
    "quality_status": "Accepted_with_gap_flag",
    "notes": (
        "Accepted. One 8.27-minute subtitle gap between 02:20:21 and "
        "02:28:37 retained as a known limitation. "
        "No malformed timestamps, nonpositive durations, or overlaps. "
        "OpenSubtitles file ID: 11959991."
    )
}

# Update existing movie row or append if not present
manifest = manifest[
    manifest["movie_record_id"].ne("2000_2")
].copy()

manifest = pd.concat(
    [manifest, pd.DataFrame([record], columns=manifest.columns)],
    ignore_index=True
)

manifest.to_csv(MANIFEST_PATH, index=False)

print("MOHABBATEIN ACCEPTED")
print("Validated file exists:", validated_path.exists())
print("Validated file size:", validated_path.stat().st_size, "bytes")
print("Manifest rows:", len(manifest))
print(
    manifest.loc[
        manifest["movie_record_id"].eq("2000_2"),
        ["movie_title", "validation_status", "quality_status",
         "dialogue_blocks", "word_count", "notes"]
    ].to_string(index=False)
)


In [ ]:

import re
import time
import hashlib
import requests
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# BATCH ACQUISITION — NEXT 25 MOVIES
# ---------------------------------------------------------
MANIFEST_PATH = WORKSPACE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
RAW_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
if not API_KEY:
    raise ValueError("OpenSubtitles API key missing from Colab Secrets.")

HEADERS = {
    "Api-Key": API_KEY,
    "User-Agent": "ICMB130SubtitleResearch v1"
}
SEARCH_URL = "https://api.opensubtitles.com/api/v1/subtitles"
DOWNLOAD_URL = "https://api.opensubtitles.com/api/v1/download"

# Reload the manifest to avoid relying on stale in-memory state.
manifest = pd.read_csv(
    MANIFEST_PATH, dtype=str, keep_default_na=False
)

validated_ids = set(
    manifest.loc[
        manifest["validation_status"].eq("Validated"),
        "movie_record_id"
    ]
)

# Skip already validated movies; process the next 25.
pending = movies.loc[
    ~movies["movie_record_id"].astype(str).isin(validated_ids)
].copy()

batch = pending.head(25).copy()

print("Movies selected:", len(batch))
print("First movie:", batch.iloc[0]["dataset_title"] if len(batch) else "None")
print("Last movie:", batch.iloc[-1]["dataset_title"] if len(batch) else "None")

results = []
session = requests.Session()
session.headers.update(HEADERS)

for _, movie in batch.iterrows():
    record_id = str(movie["movie_record_id"])
    tmdb_id = str(movie["tmdb_id"]).strip()
    title = str(movie["dataset_title"])
    year = str(movie["dataset_year"])
    language = str(movie["tmdb_original_language"]).strip().lower()

    result = {
        "movie_record_id": record_id,
        "title": title,
        "year": year,
        "tmdb_id": tmdb_id,
        "language": language,
        "candidate_count": 0,
        "downloaded_count": 0,
        "downloaded_files": [],
        "status": ""
    }

    try:
        response = session.get(
            SEARCH_URL,
            params={
                "tmdb_id": tmdb_id,
                "languages": language,
                "type": "movie"
            },
            timeout=30
        )

        # Retry once after a brief pause for transient API failures.
        if response.status_code in (429, 500, 502, 503, 504):
            time.sleep(2)
            response = session.get(
                SEARCH_URL,
                params={
                    "tmdb_id": tmdb_id,
                    "languages": language,
                    "type": "movie"
                },
                timeout=30
            )

        if response.status_code != 200:
            result["status"] = f"Search HTTP {response.status_code}"
            results.append(result)
            time.sleep(0.3)
            continue

        candidates = response.json().get("data", [])
        candidates = [
            item for item in candidates
            if str(
                item.get("attributes", {})
                    .get("feature_details", {})
                    .get("tmdb_id", "")
            ) == tmdb_id
        ]

        result["candidate_count"] = len(candidates)

        for candidate in candidates:
            attrs = candidate.get("attributes", {})
            for file_info in attrs.get("files", []):
                file_id = file_info.get("file_id")
                source_name = str(file_info.get("file_name", "subtitle"))

                if not file_id:
                    continue

                # Avoid overwriting files from earlier attempts.
                safe_id = re.sub(r"[^A-Za-z0-9_-]", "_", record_id)
                safe_name = re.sub(r"[^A-Za-z0-9._-]", "_", source_name)[:70]
                destination = RAW_DIR / (
                    f"{safe_id}_file{file_id}_{safe_name}.srt"
                )

                if destination.exists() and destination.stat().st_size > 0:
                    continue

                try:
                    download_response = session.post(
                        DOWNLOAD_URL,
                        json={"file_id": int(file_id), "sub_format": "srt"},
                        timeout=30
                    )

                    if download_response.status_code != 200:
                        continue

                    link = download_response.json().get("link")
                    if not link:
                        continue

                    file_response = requests.get(link, timeout=60)
                    file_response.raise_for_status()
                    content = file_response.content

                    if not content.strip():
                        continue

                    # Save bytes as returned; inspect encoding during validation.
                    destination.write_bytes(content)

                    result["downloaded_count"] += 1
                    result["downloaded_files"].append(
                        f"{destination.name} ({len(content)} bytes)"
                    )

                except requests.RequestException:
                    continue

                time.sleep(0.25)

        result["status"] = (
            "Downloaded candidate(s)"
            if result["downloaded_count"] > 0
            else "No downloadable candidate"
            if result["candidate_count"] > 0
            else "No original-language candidate"
        )

    except requests.RequestException as exc:
        result["status"] = f"Search error: {type(exc).__name__}"

    results.append(result)
    print(
        f"{record_id} | {title} | candidates={result['candidate_count']} "
        f"| downloaded={result['downloaded_count']} | {result['status']}"
    )
    time.sleep(0.3)

# Update the existing manifest, preserving its schema and other rows.
# Acquisition-only rows remain unvalidated until the later review pass.
for result in results:
    record_id = result["movie_record_id"]
    existing = manifest.index[
        manifest["movie_record_id"].eq(record_id)
    ].tolist()

    if existing:
        row_idx = existing[0]
    else:
        row_idx = len(manifest)
        manifest.loc[row_idx] = {col: "" for col in manifest.columns}
        manifest.at[row_idx, "movie_record_id"] = record_id

    manifest.at[row_idx, "year"] = result["year"]
    manifest.at[row_idx, "movie_title"] = result["title"]
    manifest.at[row_idx, "tmdb_id"] = result["tmdb_id"]
    manifest.at[row_idx, "language"] = result["language"]
    manifest.at[row_idx, "subtitle_language"] = result["language"]
    manifest.at[row_idx, "subtitle_source"] = "OpenSubtitles"
    manifest.at[row_idx, "acquisition_status"] = (
        "Candidate_Downloaded"
        if result["downloaded_count"] > 0
        else "Search_Completed_No_Download"
    )

    # Do not overwrite a previously validated status.
    if manifest.at[row_idx, "validation_status"] != "Validated":
        manifest.at[row_idx, "validation_status"] = "Pending_Review"

    notes = (
        f"OpenSubtitles TMDB-ID search; "
        f"{result['candidate_count']} matching candidates; "
        f"{result['downloaded_count']} downloaded. "
        f"Status: {result['status']}"
    )
    manifest.at[row_idx, "notes"] = notes

manifest.to_csv(MANIFEST_PATH, index=False)

# Concise summary only; detailed validation is deferred.
results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("BATCH ACQUISITION SUMMARY")
print("=" * 60)
print("Movies searched:", len(results_df))
print("Movies with candidates:", int((results_df["candidate_count"] > 0).sum()))
print("Movies with downloaded files:", int((results_df["downloaded_count"] > 0).sum()))
print("Candidate files downloaded:", int(results_df["downloaded_count"].sum()))
print("Manifest rows:", len(manifest))
print("Validated status was not assigned to new candidates.")


In [ ]:

from google.colab import userdata

secret_names = [
    "OPEN_SUBTITLES_API_KEY",
    "OPEN_SUBTITLES_TOKEN",
    "OPEN_SUBTITLES_USERNAME",
    "OPEN_SUBTITLES_PASSWORD"
]

print("OpenSubtitles secret availability:")
for name in secret_names:
    try:
        value = userdata.get(name)
        available = bool(value and str(value).strip())
    except Exception:
        available = False
    print(f"{name}: {'Available' if available else 'Not configured'}")


In [ ]:

from google.colab import userdata

token = userdata.get("OPEN_SUBTITLES_TOKEN")

if token and token.strip():
    print("OpenSubtitles token is available.")
    print("Token length:", len(token.strip()))
else:
    print("Token not found. Check Colab Secrets and notebook access.")


In [ ]:

import requests
import re
import time
from pathlib import Path
from google.colab import userdata

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
TOKEN = userdata.get("OPEN_SUBTITLES_TOKEN")

if not API_KEY or not TOKEN:
    raise ValueError("OpenSubtitles API key or token is missing.")

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {TOKEN.strip()}",
    "User-Agent": "ICMB130SubtitleResearch v1"
}

download_api = "https://api.opensubtitles.com/api/v1/download"
retry_results = []

# Only candidates already found in the first search
for _, movie in search_results[
    search_results["candidate_count"] > 0
].iterrows():

    record_id = str(movie["movie_record_id"])
    title = str(movie["title"])
    ids = [
        x.strip() for x in str(movie["candidate_ids"]).split("|")
        if x.strip()
    ]
    filenames = [
        x.strip() for x in str(movie["candidate_files"]).split("|")
    ]

    for index, file_id in enumerate(ids, start=1):
        item = {
            "movie_record_id": record_id,
            "title": title,
            "file_id": file_id,
            "download_status": "",
            "http_status": "",
            "saved_file": ""
        }

        try:
            response = requests.post(
                download_api,
                headers=headers,
                json={"file_id": int(file_id), "sub_format": "srt"},
                timeout=30
            )

            item["http_status"] = response.status_code

            if response.status_code != 200:
                item["download_status"] = (
                    response.text[:300].replace("\n", " ")
                )
                retry_results.append(item)
                time.sleep(0.5)
                continue

            download_info = response.json()
            link = download_info.get("link")

            if not link:
                item["download_status"] = "API returned no download link"
                retry_results.append(item)
                time.sleep(0.5)
                continue

            file_response = requests.get(link, timeout=60)
            file_response.raise_for_status()

            content = file_response.content
            if not content.strip():
                item["download_status"] = "Downloaded file was empty"
                retry_results.append(item)
                continue

            safe_name = re.sub(r"[^A-Za-z0-9._-]", "_", title)[:50]
            destination = RAW_DIR / (
                f"{record_id}_retry{index}_{safe_name}.srt"
            )
            destination.write_bytes(content)

            item["download_status"] = "Downloaded"
            item["saved_file"] = str(destination)

        except Exception as exc:
            item["download_status"] = (
                f"{type(exc).__name__}: {str(exc)[:200]}"
            )

        retry_results.append(item)
        print(
            f"{record_id} | {title} | file_id={file_id} | "
            f"HTTP={item['http_status']} | {item['download_status']}"
        )
        time.sleep(0.5)

retry_report = pd.DataFrame(retry_results)

print("\nRETRY SUMMARY")
print("Candidates attempted:", len(retry_report))
print(
    "Downloaded:",
    int(retry_report["download_status"].eq("Downloaded").sum())
)
display(retry_report)


In [ ]:

import requests
import re
import time
import pandas as pd
from pathlib import Path
from google.colab import userdata

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
TOKEN = userdata.get("OPEN_SUBTITLES_TOKEN")

if not API_KEY or not TOKEN:
    raise ValueError("OpenSubtitles API key or token is missing.")

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {TOKEN.strip()}",
    "User-Agent": "ICMB130SubtitleResearch v1"
}

search_url = "https://api.opensubtitles.com/api/v1/subtitles"
download_url = "https://api.opensubtitles.com/api/v1/download"

# First-batch movies and the language to search next.
# English fallback for movies without usable original-language candidates;
# original-language retry for movies whose earlier candidate download failed.
retry_plan = {
    "2000_1": "en",  # Kaho Naa... Pyaar Hai
    "2000_3": "en",  # Mission Kashmir: song-only candidates
    "2000_4": "en",  # Dulhan Hum Le Jayenge
    "2000_5": "en",  # Josh
    "2001_1": "en",  # Gadar: Ek Prem Katha
    "2001_2": "en",  # Kabhi Khushi Kabhie Gham
    "2001_3": "en",  # Lagaan: inspect English fallback later
    "2001_4": "en",  # Indian
    "2001_5": "en",  # Dil Chahta Hai
    "2002_1": "hi",  # Devdas
    "2002_2": "te",  # Indra — retry original-language search
    "2002_3": "hi",  # Raaz
    "2002_4": "hi",  # Hum Tumhare Hain Sanam
    "2002_5": "hi",  # Kaante
    "2003_1": "hi",  # Koi... Mil Gaya
    "2003_2": "hi",  # Kal Ho Naa Ho
    "2003_3": "hi",  # The Hero: Love Story of a Spy
    "2003_4": "hi",  # Munna Bhai M.B.B.S.
    "2003_5": "hi",  # Baghban
    "2004_1": "hi",  # Veer-Zaara
    "2004_2": "hi",  # Main Hoon Na
    "2004_3": "hi",  # Dhoom
    "2004_4": "hi",  # Mujhse Shaadi Karogi
    "2004_5": "hi",  # Khakee
    "2005_1": "ta",  # Chandramukhi
}

# Load existing manifest and identify current movie details.
manifest = pd.read_csv(
    WORKSPACE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv",
    dtype=str, keep_default_na=False
)

plan_rows = []
for record_id, lang in retry_plan.items():
    match = movies.loc[
        movies["movie_record_id"].astype(str).eq(record_id)
    ]
    if not match.empty:
        row = match.iloc[0].copy()
        row["search_language"] = lang
        plan_rows.append(row)

plan = pd.DataFrame(plan_rows)
results = []

for _, movie in plan.iterrows():
    record_id = str(movie["movie_record_id"])
    tmdb_id = str(movie["tmdb_id"]).strip()
    language = str(movie["search_language"]).strip()
    title = str(movie["dataset_title"])

    item = {
        "movie_record_id": record_id,
        "title": title,
        "language_searched": language,
        "candidates": 0,
        "downloads": 0,
        "status": ""
    }

    try:
        response = requests.get(
            search_url,
            headers=headers,
            params={
                "tmdb_id": tmdb_id,
                "languages": language,
                "type": "movie"
            },
            timeout=30
        )

        if response.status_code != 200:
            item["status"] = f"Search HTTP {response.status_code}"
            results.append(item)
            print(record_id, title, item["status"])
            continue

        candidates = response.json().get("data", [])
        candidates = [
            c for c in candidates
            if str(c.get("attributes", {})
                    .get("feature_details", {})
                    .get("tmdb_id", "")) == tmdb_id
        ]
        item["candidates"] = len(candidates)

        for candidate in candidates:
            attrs = candidate.get("attributes", {})
            for f in attrs.get("files", []):
                file_id = f.get("file_id")
                if not file_id:
                    continue

                try:
                    dl = requests.post(
                        download_url,
                        headers=headers,
                        json={
                            "file_id": int(file_id),
                            "sub_format": "srt"
                        },
                        timeout=30
                    )
                    if dl.status_code != 200:
                        continue

                    link = dl.json().get("link")
                    if not link:
                        continue

                    file_response = requests.get(link, timeout=60)
                    file_response.raise_for_status()
                    content = file_response.content
                    if not content.strip():
                        continue

                    safe_title = re.sub(r"[^A-Za-z0-9._-]", "_", title)[:45]
                    destination = RAW_DIR / (
                        f"{record_id}_{language}_file{file_id}_{safe_title}.srt"
                    )

                    # Do not overwrite a file already downloaded.
                    if destination.exists():
                        continue

                    destination.write_bytes(content)
                    item["downloads"] += 1

                except requests.RequestException:
                    continue

                time.sleep(0.25)

        item["status"] = (
            "Downloaded"
            if item["downloads"] else
            "Candidates found; download failed"
            if item["candidates"] else
            "No candidates"
        )

    except requests.RequestException as exc:
        item["status"] = f"Request error: {type(exc).__name__}"

    results.append(item)
    print(
        f"{record_id} | {title} | lang={language} | "
        f"candidates={item['candidates']} | "
        f"downloads={item['downloads']} | {item['status']}"
    )
    time.sleep(0.3)

retry_df = pd.DataFrame(results)

print("\nFOLLOW-UP SEARCH SUMMARY")
print("Movies attempted:", len(retry_df))
print("Movies with candidates:", int((retry_df["candidates"] > 0).sum()))
print("Movies with downloads:", int((retry_df["downloads"] > 0).sum()))
print("Total downloaded files:", int(retry_df["downloads"].sum()))
print("Manifest unchanged; validation deferred.")


In [ ]:

import requests
from google.colab import userdata

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
TOKEN = userdata.get("OPEN_SUBTITLES_TOKEN")

if not API_KEY or not TOKEN:
    raise ValueError("OpenSubtitles API key or token is missing.")

response = requests.get(
    "https://api.opensubtitles.com/api/v1/infos/user",
    headers={
        "Api-Key": API_KEY,
        "Authorization": f"Bearer {TOKEN.strip()}",
        "User-Agent": "ICMB130SubtitleResearch v1",
        "Accept": "application/json"
    },
    timeout=30
)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    account = response.json().get("data", {})
    for field in [
        "level",
        "allowed_downloads",
        "downloads_count",
        "remaining_downloads",
        "vip"
    ]:
        print(f"{field}: {account.get(field, 'Not reported')}")
else:
    print("Account check failed:", response.text[:300])


In [ ]:

from pathlib import Path
import re
import pandas as pd

# Find all locally downloaded Kaho Naa... Pyaar Hai candidates.
# This does not contact OpenSubtitles.
RAW_DIR = WORKSPACE / "raw"

files = list(RAW_DIR.glob("2000_1_en_file*_Kaho_Naa*"))

# Also include any other raw files matching this movie ID.
if not files:
    files = list(RAW_DIR.glob("2000_1*.srt"))

timestamp_re = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def seconds(groups):
    h, m, s, ms = map(int, groups)
    return h * 3600 + m * 60 + s + ms / 1000

rows = []

for path in files:
    raw = path.read_bytes()

    # Try common subtitle encodings without discarding invalid bytes.
    try:
        text = raw.decode("utf-8-sig")
        encoding = "UTF-8"
    except UnicodeDecodeError:
        try:
            text = raw.decode("cp1252")
            encoding = "CP1252"
        except UnicodeDecodeError:
            text = raw.decode("utf-8", errors="replace")
            encoding = "UTF-8 with replacements"

    blocks = re.split(r"\n\s*\n", text.strip())
    times = []
    dialogue = []

    for block in blocks:
        match = timestamp_re.search(block)
        if not match:
            continue

        g = match.groups()
        start = seconds(g[:4])
        end = seconds(g[4:])
        times.append((start, end))

        lines = block.splitlines()
        for line in lines:
            if timestamp_re.search(line) or re.fullmatch(r"\s*\d+\s*", line):
                continue
            line = re.sub(r"<[^>]+>", "", line).strip()
            if re.search(r"[^\W\d_]", line, re.UNICODE):
                dialogue.append(line)

    full_text = " ".join(dialogue)
    words = re.findall(r"\S+", full_text)

    rows.append({
        "file": path.name,
        "bytes": len(raw),
        "encoding": encoding,
        "timestamp_blocks": len(times),
        "readable_lines": len(dialogue),
        "word_count": len(words),
        "devanagari_chars": len(re.findall(r"[\u0900-\u097F]", full_text)),
        "latin_chars": len(re.findall(r"[A-Za-z]", full_text)),
        "replacement_chars": text.count("\uFFFD"),
        "first_seconds": round(min((x[0] for x in times), default=0), 1),
        "last_seconds": round(max((x[1] for x in times), default=0), 1),
        "preview": " | ".join(dialogue[:3])[:180]
    })

report = pd.DataFrame(rows)

if report.empty:
    print("No Kaho Naa... Pyaar Hai subtitle files found in raw/.")
else:
    report = report.sort_values(
        ["timestamp_blocks", "word_count", "bytes"],
        ascending=False
    ).reset_index(drop=True)

    print("LOCAL CANDIDATE INSPECTION")
    print("Files found:", len(report))
    print("No network requests were made.")
    display(report)


In [ ]:
import re
from pathlib import Path
from datetime import timedelta

path = next(RAW_DIR.glob("2000_1_en_file678089_*.srt"))
text = path.read_text(encoding="utf-8-sig", errors="replace")
blocks = re.split(r"\n\s*\n", text.strip())

pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def to_seconds(groups):
    h, m, s, ms = map(int, groups)
    return h * 3600 + m * 60 + s + ms / 1000

entries = []
malformed = 0

for block in blocks:
    match = pattern.search(block)
    if not match:
        if "-->" in block:
            malformed += 1
        continue

    groups = match.groups()
    lines = block.splitlines()
    dialogue = " ".join(
        line.strip() for line in lines
        if not pattern.search(line)
        and not re.fullmatch(r"\s*\d+\s*", line)
    )
    dialogue = re.sub(r"<[^>]+>", "", dialogue).strip()

    entries.append({
        "start": to_seconds(groups[:4]),
        "end": to_seconds(groups[4:]),
        "dialogue": dialogue
    })

entries.sort(key=lambda x: x["start"])

overlaps = sum(
    entries[i]["start"] < entries[i-1]["end"]
    for i in range(1, len(entries))
)
nonpositive = sum(e["end"] <= e["start"] for e in entries)
gaps = [
    (entries[i-1]["end"], entries[i]["start"])
    for i in range(1, len(entries))
    if entries[i]["start"] - entries[i-1]["end"] > 300
]

print("File:", path.name)
print("Parsed blocks:", len(entries))
print("Malformed timestamp blocks:", malformed)
print("Nonpositive durations:", nonpositive)
print("Overlaps:", overlaps)
print("Gaps exceeding 5 minutes:", len(gaps))
print("Start:", timedelta(seconds=int(entries[0]["start"])))
print("End:", timedelta(seconds=int(entries[-1]["end"])))

for label, sample in [
    ("FIRST 8 DIALOGUE ENTRIES", entries[:8]),
    ("LAST 8 DIALOGUE ENTRIES", entries[-8:])
]:
    print("\n" + label)
    for e in sample:
        print(
            f'{timedelta(seconds=int(e["start"]))} | '
            f'{e["dialogue"][:180]}'
        )

if gaps:
    print("\nLARGEST GAPS")
    for start, end in sorted(
        gaps, key=lambda pair: pair[1] - pair[0], reverse=True
    )[:5]:
        print(
            f"{timedelta(seconds=int(start))} -> "
            f"{timedelta(seconds=int(end))} "
            f"({(end-start)/60:.1f} minutes)"
        )

In [ ]:

import shutil
import re
from pathlib import Path
import pandas as pd

MANIFEST_PATH = WORKSPACE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
source = next(RAW_DIR.glob("2000_1_en_file678089_*.srt"))
destination = WORKSPACE / "validated" / "2000_1_KAHO_NAA_PYAAR_HAI_ENGLISH.srt"

manifest = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False)

# Ensure this is the expected movie and the subtitle passed the checks.
movie = movies.loc[
    movies["movie_record_id"].astype(str).eq("2000_1")
].iloc[0]

assert str(movie["tmdb_id"]) == "16987"
assert source.exists()
assert len(entries) == 1763
assert malformed == 0
assert nonpositive == 0
assert overlaps == 0
assert len(gaps) == 0

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)

text = destination.read_text(encoding="utf-8-sig", errors="replace")

record = {
    "movie_record_id": "2000_1",
    "year": "2000",
    "rank": "1",
    "movie_title": "Kaho Naa... Pyaar Hai",
    "language": "Hindi",
    "tmdb_id": "16987",
    "imdb_id": "",
    "subtitle_language": "English",
    "subtitle_type": "English_Fallback",
    "subtitle_source": "OpenSubtitles",
    "source_id": "678089",
    "source_url": "",
    "raw_subtitle_file": source.name,
    "validated_subtitle_file": destination.name,
    "acquisition_status": "Acquired",
    "validation_status": "Validated",
    "title_verified": "True",
    "year_verified": "True",
    "movie_identity_verified": "True",
    "language_verified": "True",
    "dialogue_blocks": str(len(entries)),
    "word_count": str(len(re.findall(r"\S+", text))),
    "character_count": str(len(text)),
    "encoding_status": "UTF-8",
    "quality_status": "Accepted_English_Fallback",
    "notes": (
        "English fallback accepted after no Hindi candidate was found. "
        "1763 timestamp blocks; no malformed timestamps, overlaps, "
        "nonpositive durations, or gaps over 5 minutes. "
        "Runtime coverage approximately 2:45:38; confirm intended cut "
        "during later review. Minor copyright metadata at file boundaries."
    )
}

# Update only this movie's row in the existing manifest.
manifest = manifest[
    manifest["movie_record_id"].ne("2000_1")
].copy()

manifest = pd.concat(
    [manifest, pd.DataFrame([record], columns=manifest.columns)],
    ignore_index=True
)

manifest.to_csv(MANIFEST_PATH, index=False)

# Remove redundant raw candidates for this movie only.
removed = 0
for path in RAW_DIR.glob("2000_1*.srt"):
    if path.resolve() != destination.resolve():
        path.unlink()
        removed += 1

print("KAHO NAA... PYAAR HAI RECORDED")
print("Validated file exists:", destination.exists())
print("Validated file size:", destination.stat().st_size, "bytes")
print("Redundant raw candidates removed:", removed)
print("Manifest rows:", len(manifest))
print(
    manifest.loc[
        manifest["movie_record_id"].eq("2000_1"),
        ["movie_title", "subtitle_type", "validation_status",
         "quality_status", "source_id"]
    ].to_string(index=False)
)


In [ ]:

# ============================================================
# NOTEBOOK 07 — INDICDIALOGUE ARCHIVE DISCOVERY
# Purpose: Find archive entries whose filenames match our movies
# No extraction, no file modifications, no new output files
# ============================================================

from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import re
import unicodedata

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

MASTER_PATH = (
    PROJECT_ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
)

ARCHIVE_PATH = (
    PROJECT_ROOT / "01_data/indicdialogue/dataset.zip"
)

# ---------- 1. Confirm inputs ----------
assert MASTER_PATH.exists(), f"Master dataset not found: {MASTER_PATH}"
assert ARCHIVE_PATH.exists(), f"IndicDialogue archive not found: {ARCHIVE_PATH}"

master = pd.read_csv(MASTER_PATH)

print("=" * 70)
print("INDICDIALOGUE ARCHIVE DISCOVERY")
print("=" * 70)
print(f"Master dataset: {master.shape[0]} rows")
print(f"Archive exists: {ARCHIVE_PATH.exists()}")

# ---------- 2. Normalize movie titles ----------
def normalize_title(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

# Identify the title column without modifying the dataset.
title_candidates = [
    c for c in master.columns
    if c.lower() in {"movie_title", "title", "dataset_title", "film_title"}
]

if not title_candidates:
    raise ValueError(
        f"Could not identify the movie-title column. "
        f"Available columns: {master.columns.tolist()}"
    )

TITLE_COL = title_candidates[0]
print(f"Title column: {TITLE_COL}")

# ---------- 3. Inspect ZIP directory only ----------
with ZipFile(ARCHIVE_PATH, "r") as zf:
    archive_entries = [
        info.filename
        for info in zf.infolist()
        if not info.is_dir()
    ]

print(f"Total archive files: {len(archive_entries):,}")

# ---------- 4. Match movie titles against archive paths ----------
# Exact normalized title phrase matching is used first.
# This avoids weak matches based on common words alone.
matches = []

for _, movie in master.iterrows():
    title = str(movie[TITLE_COL]).strip()
    normalized = normalize_title(title)

    if not normalized:
        continue

    title_pattern = re.compile(
        r"(?<![a-z0-9])"
        + re.escape(normalized).replace(r"\ ", r"[\s._-]+")
        + r"(?![a-z0-9])"
    )

    found = [
        entry for entry in archive_entries
        if title_pattern.search(normalize_title(entry))
    ]

    for entry in found:
        matches.append({
            "movie_record_id": movie.get("movie_record_id", ""),
            "year": movie.get("year", ""),
            "movie_title": title,
            "archive_path": entry,
            "file_extension": Path(entry).suffix.lower(),
        })

matches_df = pd.DataFrame(
    matches,
    columns=[
        "movie_record_id",
        "year",
        "movie_title",
        "archive_path",
        "file_extension",
    ],
)

# ---------- 5. Display findings only ----------
print("\n" + "=" * 70)
print("TITLE-MATCH RESULTS")
print("=" * 70)
print(f"Movies with at least one filename/path match: "
      f"{matches_df['movie_record_id'].nunique() if not matches_df.empty else 0}")
print(f"Total matching archive entries: {len(matches_df)}")

if matches_df.empty:
    print(
        "\nNo exact title matches were found in archive paths. "
        "This does not prove that the archive lacks relevant dialogue; "
        "its files may use IDs, abbreviations, or unrelated filenames."
    )
else:
    display(
        matches_df.sort_values(
            ["year", "movie_title", "archive_path"]
        ).reset_index(drop=True)
    )

    print("\nFile-extension summary:")
    display(
        matches_df["file_extension"]
        .replace("", "[no extension]")
        .value_counts()
        .rename_axis("extension")
        .to_frame("matching_files")
    )

print("\nInspection complete. No archive files were extracted or modified.")


In [ ]:

# ============================================================
# NOTEBOOK 07 — INDICDIALOGUE ARCHIVE STRUCTURE INSPECTION
# Read-only: inspect extensions, folders, and sample filenames
# ============================================================

from zipfile import ZipFile
from pathlib import Path
from collections import Counter
import pandas as pd

ARCHIVE_PATH = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/indicdialogue/dataset.zip"
)

with ZipFile(ARCHIVE_PATH, "r") as zf:
    files = [info for info in zf.infolist() if not info.is_dir()]

# 1. File-extension distribution
extensions = Counter(
    Path(info.filename).suffix.lower() or "[no extension]"
    for info in files
)

print("=" * 65)
print("1. FILE EXTENSIONS")
print("=" * 65)
display(
    pd.DataFrame(
        extensions.most_common(),
        columns=["extension", "file_count"]
    )
)

# 2. Top-level directory distribution
top_folders = Counter(
    info.filename.strip("/").split("/")[0]
    for info in files
)

print("\n" + "=" * 65)
print("2. TOP-LEVEL DIRECTORIES")
print("=" * 65)
display(
    pd.DataFrame(
        top_folders.most_common(30),
        columns=["top_level_folder", "file_count"]
    )
)

# 3. Representative filenames
print("\n" + "=" * 65)
print("3. FIRST 60 ARCHIVE FILENAMES")
print("=" * 65)

for info in files[:60]:
    print(info.filename)

# 4. Representative subtitle/text-like filenames
text_extensions = {
    ".srt", ".vtt", ".ass", ".ssa", ".sub",
    ".txt", ".json", ".csv", ".tsv", ".xml"
}

text_files = [
    info.filename for info in files
    if Path(info.filename).suffix.lower() in text_extensions
]

print("\n" + "=" * 65)
print(f"4. SAMPLE OF TEXT/SUBTITLE-LIKE FILES: {len(text_files):,}")
print("=" * 65)

for filename in text_files[:40]:
    print(filename)

print("\nInspection finished. No files extracted or modified.")


In [ ]:

# ============================================================
# NOTEBOOK 07 — COMPREHENSIVE INDICDIALOGUE MATCH SEARCH
# Search methods:
#   1. IMDb ID
#   2. Normalized title variants
#   3. Token-order-insensitive title matching
#   4. Fuzzy title similarity + year
#
# Read-only: does not extract or modify any project files
# ============================================================

from pathlib import Path
from zipfile import ZipFile
from collections import defaultdict
from difflib import SequenceMatcher
import pandas as pd
import json
import re
import unicodedata

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
MASTER_PATH = ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MAPPING_PATH = ROOT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
ARCHIVE_PATH = ROOT / "01_data/indicdialogue/dataset.zip"

master = pd.read_csv(MASTER_PATH, dtype=str).fillna("")
mapping = pd.read_csv(MAPPING_PATH, dtype=str).fillna("")

def find_col(df, candidates):
    lookup = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None

master_id_col = find_col(
    master, ["movie_record_id", "record_id"]
)
master_title_col = find_col(
    master, ["movie_title", "title", "dataset_title"]
)
master_year_col = find_col(
    master, ["year", "release_year", "dataset_year"]
)
master_imdb_col = find_col(
    master, ["imdb_id", "imdbid", "movie_imdb_id"]
)

mapping_id_col = find_col(
    mapping, ["movie_record_id", "record_id"]
)
mapping_title_col = find_col(
    mapping, ["dataset_title", "movie_title", "title"]
)
mapping_year_col = find_col(
    mapping, ["dataset_year", "year", "release_year"]
)
mapping_imdb_col = find_col(
    mapping, ["imdb_id", "imdbid", "movie_imdb_id"]
)
mapping_tmdb_col = find_col(
    mapping, ["tmdb_id", "movie_tmdb_id"]
)

if not all([master_id_col, master_title_col, master_year_col]):
    raise ValueError(
        "Required master columns were not found. "
        f"Available columns: {master.columns.tolist()}"
    )

# ---------- Normalize titles and IDs ----------
def norm_title(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    # Remove common leading articles for alternate-title matching.
    value = re.sub(r"^(the|a|an)\s+", "", value)
    return value

def norm_imdb(value):
    value = str(value).strip().lower()
    if not value or value in {"nan", "none", "null"}:
        return ""
    match = re.search(r"(?:tt)?0*(\d+)", value)
    return match.group(1) if match else ""

def safe_year(value):
    try:
        return int(float(str(value).strip()))
    except (ValueError, TypeError):
        return None

# Build one search record per master movie.
movie_records = []

for _, row in master.iterrows():
    rid = str(row[master_id_col]).strip()
    title = str(row[master_title_col]).strip()
    year = safe_year(row[master_year_col])

    movie_records.append({
        "movie_record_id": rid,
        "dataset_title": title,
        "dataset_year": year,
        "tmdb_id": "",
        "imdb_id": "",
        "title_keys": set([norm_title(title)]),
    })

movie_lookup = {m["movie_record_id"]: m for m in movie_records}

# Add mapping titles and IDs without modifying either source file.
for _, row in mapping.iterrows():
    rid = str(row[mapping_id_col]).strip() if mapping_id_col else ""
    if rid not in movie_lookup:
        continue

    movie = movie_lookup[rid]

    if mapping_title_col:
        alt_title = str(row[mapping_title_col]).strip()
        if alt_title:
            movie["title_keys"].add(norm_title(alt_title))

    if mapping_tmdb_col:
        movie["tmdb_id"] = str(row[mapping_tmdb_col]).strip()

    if mapping_imdb_col:
        candidate_id = norm_imdb(row[mapping_imdb_col])
        if candidate_id:
            movie["imdb_id"] = candidate_id

# If IMDb IDs exist in the master, use those too.
if master_imdb_col:
    for _, row in master.iterrows():
        rid = str(row[master_id_col]).strip()
        if rid in movie_lookup:
            imdb = norm_imdb(row[master_imdb_col])
            if imdb:
                movie_lookup[rid]["imdb_id"] = imdb

# Manual title variants for common punctuation/formatting differences.
# These are search aliases only, not claims that records have matched.
aliases = {
    "Kaho Naa... Pyaar Hai": [
        "Kaho Naa Pyaar Hai", "Kaho Na Pyar Hai",
        "Kaho Naa Pyar Hai"
    ],
    "Kabhi Khushi Kabhie Gham": [
        "Kabhi Khushi Kabhi Gham", "K3G"
    ],
    "Dil Chahta Hai": ["Dil Chahta Hai"],
    "Koi... Mil Gaya": ["Koi Mil Gaya", "Koi Mil Gaya"],
    "Munna Bhai M.B.B.S.": ["Munna Bhai MBBS", "Munna Bhai M B B S"],
    "Veer-Zaara": ["Veer Zaara", "Veer Zara"],
    "Devdas": ["Devdas"],
    "83": ["83", "83 The Film", "83 The Movie"],
}

for movie in movie_records:
    for alias_title, variants in aliases.items():
        if norm_title(alias_title) == norm_title(movie["dataset_title"]):
            movie["title_keys"].update(norm_title(v) for v in variants)

# ---------- Read metadata from the main archive directory ----------
archive_records = []
jsonl_files_scanned = 0
records_scanned = 0

with ZipFile(ARCHIVE_PATH, "r") as zf:
    jsonl_paths = [
        name for name in zf.namelist()
        if name.startswith("IndicDialogue/")
        and name.lower().endswith(".jsonl")
    ]

    for path in jsonl_paths:
        jsonl_files_scanned += 1
        language_folder = path.split("/")[1]

        with zf.open(path) as file:
            for line_number, raw_line in enumerate(file, start=1):
                try:
                    record = json.loads(raw_line.decode("utf-8-sig"))
                except (UnicodeDecodeError, json.JSONDecodeError):
                    continue

                metadata = record.get("metadata", {})
                title = str(metadata.get("MovieName", "")).strip()
                year = safe_year(metadata.get("Year"))
                imdb = norm_imdb(metadata.get("MovieImdbID"))
                subtitle_id = str(
                    metadata.get("IDSubtitleFile", record.get("ID", ""))
                ).strip()

                if not title:
                    continue

                records_scanned += 1
                archive_records.append({
                    "archive_title": title,
                    "archive_title_norm": norm_title(title),
                    "archive_year": year,
                    "archive_imdb_id": imdb,
                    "subtitle_id": subtitle_id,
                    "language_folder": language_folder,
                    "dialogue_languages": ", ".join(
                        sorted(record.get("dialogs", {}).keys())
                    ),
                    "jsonl_path": path,
                    "line_number": line_number,
                })

print(f"Metadata files scanned: {jsonl_files_scanned}")
print(f"Metadata records scanned: {records_scanned:,}")
print(f"ICMB-130 movies searched: {len(movie_records)}")

# ---------- Index archive records ----------
by_imdb = defaultdict(list)
by_title = defaultdict(list)

for ar in archive_records:
    if ar["archive_imdb_id"]:
        by_imdb[ar["archive_imdb_id"]].append(ar)
    by_title[ar["archive_title_norm"]].append(ar)

# ---------- Search by ID and title variants ----------
results = []

for movie in movie_records:
    exact_imdb_hits = (
        by_imdb.get(movie["imdb_id"], [])
        if movie["imdb_id"] else []
    )

    for ar in exact_imdb_hits:
        results.append({
            **{k: v for k, v in movie.items() if k != "title_keys"},
            **ar,
            "match_method": "IMDb_ID",
            "title_similarity": 1.0,
            "year_match": (
                movie["dataset_year"] is not None
                and ar["archive_year"] == movie["dataset_year"]
            ),
        })

    # Exact normalized title and token-order-insensitive match.
    for key in movie["title_keys"]:
        for ar in by_title.get(key, []):
            results.append({
                **{k: v for k, v in movie.items() if k != "title_keys"},
                **ar,
                "match_method": "Exact_Normalized_Title",
                "title_similarity": 1.0,
                "year_match": (
                    movie["dataset_year"] is not None
                    and ar["archive_year"] == movie["dataset_year"]
                ),
            })

        # Token-sorted comparison handles word-order variations.
        target_tokens = " ".join(sorted(key.split()))
        for ar in archive_records:
            if " ".join(sorted(ar["archive_title_norm"].split())) == target_tokens:
                results.append({
                    **{k: v for k, v in movie.items() if k != "title_keys"},
                    **ar,
                    "match_method": "Token_Order_Variant",
                    "title_similarity": 1.0,
                    "year_match": (
                        movie["dataset_year"] is not None
                        and ar["archive_year"] == movie["dataset_year"]
                    ),
                })

# ---------- Fuzzy candidates, including same-year records ----------
# Fuzzy matches are suggestions only; they are not accepted as identity matches.
fuzzy_results = []

for movie in movie_records:
    for ar in archive_records:
        if not ar["archive_title_norm"]:
            continue

        best_score = max(
            (
                SequenceMatcher(
                    None, key, ar["archive_title_norm"]
                ).ratio()
                for key in movie["title_keys"]
                if key
            ),
            default=0.0,
        )

        same_year = (
            movie["dataset_year"] is not None
            and ar["archive_year"] == movie["dataset_year"]
        )

        if best_score >= 0.78 or (same_year and best_score >= 0.60):
            fuzzy_results.append({
                "movie_record_id": movie["movie_record_id"],
                "dataset_title": movie["dataset_title"],
                "dataset_year": movie["dataset_year"],
                "tmdb_id": movie["tmdb_id"],
                "imdb_id": movie["imdb_id"],
                "archive_title": ar["archive_title"],
                "archive_year": ar["archive_year"],
                "archive_imdb_id": ar["archive_imdb_id"],
                "subtitle_id": ar["subtitle_id"],
                "language_folder": ar["language_folder"],
                "dialogue_languages": ar["dialogue_languages"],
                "title_similarity": round(best_score, 3),
                "year_match": same_year,
                "jsonl_path": ar["jsonl_path"],
                "match_method": "Fuzzy_Candidate",
            })

# Deduplicate direct results, retaining the strongest match method.
if results:
    direct_df = pd.DataFrame(results).drop_duplicates(
        subset=["movie_record_id", "subtitle_id", "match_method"]
    )
    direct_df["method_priority"] = direct_df["match_method"].map({
        "IMDb_ID": 0,
        "Exact_Normalized_Title": 1,
        "Token_Order_Variant": 2,
    }).fillna(9)

    direct_df = direct_df.sort_values(
        ["method_priority", "year_match"],
        ascending=[True, False]
    ).drop_duplicates(
        subset=["movie_record_id", "subtitle_id"],
        keep="first"
    ).drop(columns="method_priority")
else:
    direct_df = pd.DataFrame()

if fuzzy_results:
    fuzzy_df = pd.DataFrame(fuzzy_results).drop_duplicates(
        subset=["movie_record_id", "subtitle_id"]
    )
    fuzzy_df = fuzzy_df.sort_values(
        ["year_match", "title_similarity"],
        ascending=[False, False]
    )
else:
    fuzzy_df = pd.DataFrame()

# ---------- Compact report ----------
print("\n" + "=" * 72)
print("DIRECT MATCH RESULTS")
print("=" * 72)

if direct_df.empty:
    print("No direct IMDb/title matches found.")
else:
    print("Matched ICMB-130 movies:",
          direct_df["movie_record_id"].nunique())
    print("Matched archive records:", len(direct_df))
    display(direct_df[[
        "movie_record_id", "dataset_title", "dataset_year",
        "tmdb_id", "imdb_id", "archive_title", "archive_year",
        "archive_imdb_id", "subtitle_id", "language_folder",
        "dialogue_languages", "match_method", "year_match",
        "jsonl_path"
    ]].reset_index(drop=True))

print("\n" + "=" * 72)
print("FUZZY CANDIDATES — REVIEW REQUIRED")
print("=" * 72)

if fuzzy_df.empty:
    print("No fuzzy candidates met the matching thresholds.")
else:
    # Limit display to the strongest candidates per movie.
    fuzzy_display = (
        fuzzy_df.sort_values(
            ["year_match", "title_similarity"],
            ascending=[False, False]
        )
        .groupby("movie_record_id", group_keys=False)
        .head(5)
        .reset_index(drop=True)
    )

    print("Candidate rows before display limit:", len(fuzzy_df))
    print("Movies with fuzzy candidates:",
          fuzzy_df["movie_record_id"].nunique())
    display(fuzzy_display[[
        "movie_record_id", "dataset_title", "dataset_year",
        "tmdb_id", "imdb_id", "archive_title", "archive_year",
        "archive_imdb_id", "subtitle_id", "language_folder",
        "dialogue_languages", "title_similarity", "year_match",
        "jsonl_path"
    ]])

print("\nNo archive files extracted. No project datasets modified.")


In [ ]:

# ============================================================
# NOTEBOOK 07 — FILTER INDICDIALOGUE CANDIDATES
# Separate same-year matches from ambiguous title collisions
# ============================================================

import pandas as pd
from difflib import SequenceMatcher

# 1. Keep direct matches where the release year agrees.
if direct_df.empty:
    same_year_direct = pd.DataFrame()
else:
    same_year_direct = direct_df[
        direct_df["year_match"].astype(bool)
    ].copy()

# 2. Keep high-similarity fuzzy candidates from the same year.
if fuzzy_df.empty:
    same_year_fuzzy = pd.DataFrame()
else:
    same_year_fuzzy = fuzzy_df[
        fuzzy_df["year_match"].astype(bool)
        & (fuzzy_df["title_similarity"] >= 0.90)
    ].copy()

# 3. Combine the two candidate sets.
candidate_frames = []

if not same_year_direct.empty:
    candidate_frames.append(same_year_direct)

if not same_year_fuzzy.empty:
    candidate_frames.append(same_year_fuzzy)

if candidate_frames:
    candidates = pd.concat(
        candidate_frames, ignore_index=True, sort=False
    )

    # Avoid duplicate movie/subtitle pairs.
    candidates = candidates.drop_duplicates(
        subset=["movie_record_id", "subtitle_id"]
    )

    candidates = candidates.sort_values(
        ["dataset_year", "dataset_title", "archive_title"]
    ).reset_index(drop=True)
else:
    candidates = pd.DataFrame()

print("=" * 70)
print("SAME-YEAR CANDIDATE SUMMARY")
print("=" * 70)

print("Direct same-year records:", len(same_year_direct))
print(
    "Directly matched movies:",
    same_year_direct["movie_record_id"].nunique()
    if not same_year_direct.empty else 0
)
print("High-similarity same-year fuzzy records:", len(same_year_fuzzy))
print(
    "Combined candidate records:",
    len(candidates)
)
print(
    "Distinct target movies with candidates:",
    candidates["movie_record_id"].nunique()
    if not candidates.empty else 0
)

if not candidates.empty:
    display(candidates[[
        "movie_record_id",
        "dataset_title",
        "dataset_year",
        "archive_title",
        "archive_year",
        "archive_imdb_id",
        "subtitle_id",
        "language_folder",
        "dialogue_languages",
        "match_method",
        "title_similarity",
        "jsonl_path"
    ]].reset_index(drop=True))

# 4. Show target movies with no same-year candidate.
matched_ids = (
    set(candidates["movie_record_id"])
    if not candidates.empty else set()
)

unmatched = pd.DataFrame([
    {
        "movie_record_id": movie["movie_record_id"],
        "movie_title": movie["dataset_title"],
        "year": movie["dataset_year"]
    }
    for movie in movie_records
    if movie["movie_record_id"] not in matched_ids
])

print("\n" + "=" * 70)
print("MOVIES WITHOUT A SAME-YEAR CANDIDATE")
print("=" * 70)
print("Unmatched movies:", len(unmatched))
display(unmatched.reset_index(drop=True))

print("\nNo files extracted or modified.")


In [ ]:
# ============================================================
# NOTEBOOK 07 — FIX LANGUAGE SCREENING
# Resolve duplicate columns and inspect actual language labels
# ============================================================

import pandas as pd

# 1. Inspect the language codes actually present in the archive records.
print("LANGUAGE CODES IN CANDIDATES")
display(
    candidates["dialogue_languages"]
    .fillna("(missing)")
    .astype(str)
    .value_counts()
    .rename_axis("archive_dialogue_code")
    .reset_index(name="candidate_count")
)

# 2. Rebuild screening cleanly, avoiding duplicate column names.
master_language = master[
    ["movie_record_id", "language", "original_language"]
].copy()

master_language["language"] = (
    master_language["language"].fillna("").astype(str).str.strip()
)

screening = candidates.drop(
    columns=["language", "original_language",
             "dialogue_languages_x", "dialogue_languages_y"],
    errors="ignore"
).merge(
    master_language,
    on="movie_record_id",
    how="left"
)

# Use the inspection table's language field only if it exists there.
if "dialogue_languages" not in screening.columns:
    if "dialogue_languages" in inspection_df.columns:
        screening = screening.merge(
            inspection_df[["subtitle_id", "dialogue_languages"]]
            .drop_duplicates("subtitle_id"),
            on="subtitle_id",
            how="left"
        )
    else:
        screening["dialogue_languages"] = ""

# Add dialogue statistics without duplicating existing columns.
stats_columns = [
    "subtitle_id", "dialogue_items", "word_count",
    "devanagari_chars", "bengali_chars", "tamil_chars",
    "telugu_chars", "latin_chars", "replacement_chars"
]

available_stats = [
    col for col in stats_columns if col in inspection_df.columns
]

if available_stats:
    screening = screening.drop(
        columns=[c for c in available_stats if c != "subtitle_id"],
        errors="ignore"
    ).merge(
        inspection_df[available_stats].drop_duplicates("subtitle_id"),
        on="subtitle_id",
        how="left"
    )

# These are archive language codes, not inferred from the script.
# We will display the raw codes before assigning any final decisions.
print("\nMASTER LANGUAGE LABELS")
display(
    screening["language"]
    .value_counts(dropna=False)
    .rename_axis("master_language")
    .reset_index(name="candidate_count")
)

print("\nCANDIDATE TABLE")
display(
    screening[[
        "movie_record_id", "dataset_title", "dataset_year",
        "language", "archive_title", "archive_year",
        "subtitle_id", "language_folder", "dialogue_languages",
        "word_count", "devanagari_chars", "bengali_chars",
        "tamil_chars", "telugu_chars", "latin_chars"
    ]].reset_index(drop=True)
)

print("\nNo files extracted, accepted, or modified.")

In [ ]:
# ============================================================
# NOTEBOOK 07 — STRICT LANGUAGE CANDIDATE CLASSIFICATION
# No acceptance or file modifications
# ============================================================

import pandas as pd

# Map master language labels to observed IndicDialogue codes.
EXPECTED_CODES = {
    "Hindi": {"hin"},
    "Tamil": {"tam"},
    "Telugu": {"tel"},
    "Kannada": {"kan"},
}

def classify_language(row):
    expected = EXPECTED_CODES.get(
        str(row["language"]).strip(), set()
    )

    actual = {
        code.strip().lower()
        for code in str(row["dialogue_languages"]).split(",")
        if code.strip()
    }

    if expected.intersection(actual):
        return "ORIGINAL_LANGUAGE_CANDIDATE"

    if actual.intersection({"eng", "en"}):
        return "ENGLISH_FALLBACK_CANDIDATE"

    return "LANGUAGE_MISMATCH_OR_UNAVAILABLE"

screening["language_decision"] = screening.apply(
    classify_language, axis=1
)

print("=" * 70)
print("LANGUAGE CLASSIFICATION SUMMARY")
print("=" * 70)

display(
    screening["language_decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="candidate_count")
)

print("\nOriginal-language candidates:")
display(
    screening.loc[
        screening["language_decision"] == "ORIGINAL_LANGUAGE_CANDIDATE",
        [
            "movie_record_id", "dataset_title", "dataset_year",
            "language", "archive_title", "archive_year",
            "subtitle_id", "dialogue_languages", "word_count"
        ]
    ].reset_index(drop=True)
)

print("\nLanguage mismatches or unavailable original language:")
display(
    screening.loc[
        screening["language_decision"] ==
        "LANGUAGE_MISMATCH_OR_UNAVAILABLE",
        [
            "movie_record_id", "dataset_title", "dataset_year",
            "language", "archive_title", "archive_year",
            "subtitle_id", "dialogue_languages", "word_count"
        ]
    ].reset_index(drop=True)
)

print("\nNo files extracted, accepted, or modified.")

In [ ]:
# ============================================================
# NOTEBOOK 07 — FILTER CANDIDATES AGAINST MANIFEST
# Preserve all previously accepted subtitles
# ============================================================

import pandas as pd
from pathlib import Path

MANIFEST_PATH = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/"
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")

# Identify movies with already accepted, validated subtitles.
accepted_statuses = {"accepted", "validated", "accepted_with_gap_flag"}

def is_already_accepted(row):
    acquisition = str(row.get("acquisition_status", "")).strip().lower()
    validation = str(row.get("validation_status", "")).strip().lower()

    return (
        acquisition in accepted_statuses
        or validation in accepted_statuses
        or (
            acquisition in {"accepted", "complete", "completed"}
            and validation in {"pass", "passed", "valid", "validated"}
        )
    )

manifest["already_accepted"] = manifest.apply(
    is_already_accepted, axis=1
)

accepted_movie_ids = set(
    manifest.loc[
        manifest["already_accepted"], "movie_record_id"
    ].astype(str)
)

# Use only the seven original-language candidates identified earlier.
original_candidates = screening[
    screening["language_decision"] == "ORIGINAL_LANGUAGE_CANDIDATE"
].copy()

# Exclude movies with an already accepted subtitle.
remaining_candidates = original_candidates[
    ~original_candidates["movie_record_id"].astype(str).isin(
        accepted_movie_ids
    )
].copy()

print("=" * 70)
print("MANIFEST CHECK")
print("=" * 70)
print("Manifest rows:", len(manifest))
print("Movies with accepted subtitles:", len(accepted_movie_ids))
print("Original-language candidates:", len(original_candidates))
print("Candidates requiring further review:", len(remaining_candidates))

print("\nALREADY ACCEPTED — SKIP")
display(
    original_candidates[
        original_candidates["movie_record_id"].astype(str).isin(
            accepted_movie_ids
        )
    ][["movie_record_id", "dataset_title", "language"]]
    .reset_index(drop=True)
)

print("\nREMAINING CANDIDATES — REVIEW ONLY")
display(
    remaining_candidates[[
        "movie_record_id", "dataset_title", "dataset_year",
        "language", "archive_title", "archive_year",
        "subtitle_id", "dialogue_languages", "word_count"
    ]].reset_index(drop=True)
)

print("\nNo files extracted, accepted, replaced, or modified.")

In [ ]:
# ============================================================
# NOTEBOOK 07 — CHECK SRT FILES FOR SIX REMAINING CANDIDATES
# Read-only ZIP inspection; no extraction or downloads
# ============================================================

import zipfile
import pandas as pd

ARCHIVE_PATH = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/indicdialogue/dataset.zip"
)

# These are already filtered against the manifest.
review_candidates = remaining_candidates.copy()

results = []

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    archive_names = zf.namelist()

    for _, row in review_candidates.iterrows():
        subtitle_id = str(row["subtitle_id"]).strip()
        folder = str(row["language_folder"]).strip()

        # Locate SRT entries by subtitle ID and language folder.
        matches = [
            name for name in archive_names
            if name.lower().endswith(".srt")
            and subtitle_id in name
            and folder.lower() in name.lower()
        ]

        for name in matches:
            info = zf.getinfo(name)
            results.append({
                "movie_record_id": row["movie_record_id"],
                "movie_title": row["dataset_title"],
                "year": row["dataset_year"],
                "expected_language": row["language"],
                "subtitle_id": subtitle_id,
                "archive_language_folder": folder,
                "archive_srt_path": name,
                "file_size_bytes": info.file_size,
                "zip_entry_found": True
            })

        if not matches:
            results.append({
                "movie_record_id": row["movie_record_id"],
                "movie_title": row["dataset_title"],
                "year": row["dataset_year"],
                "expected_language": row["language"],
                "subtitle_id": subtitle_id,
                "archive_language_folder": folder,
                "archive_srt_path": "",
                "file_size_bytes": 0,
                "zip_entry_found": False
            })

srt_check = pd.DataFrame(results)

print("=" * 70)
print("SRT ARCHIVE ENTRY CHECK")
print("=" * 70)
print("Movies checked:", review_candidates["movie_record_id"].nunique())
print("Rows returned:", len(srt_check))
print("SRT entries found:", int(srt_check["zip_entry_found"].sum()))
print("Candidates without matching SRT entry:",
      int((~srt_check["zip_entry_found"]).sum()))

display(srt_check.reset_index(drop=True))

print("\nNo files extracted or modified.")

In [ ]:
# ============================================================
# NOTEBOOK 07 — INSPECT EXISTING INDICDIALOGUE SRT FILES
# Read-only inspection; no extraction or file modifications
# ============================================================

import zipfile
import re
import pandas as pd

ARCHIVE_PATH = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/indicdialogue/dataset.zip"
)

# Only inspect SRT paths already confirmed to exist.
existing_srt = srt_check[srt_check["zip_entry_found"]].copy()

def inspect_srt_content(text):
    """Calculate basic SRT structure and text-quality indicators."""
    text = text.lstrip("\ufeff")
    blocks = re.split(r"\n\s*\n", text.strip())

    timestamps = re.findall(
        r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
        r"(\d{2}:\d{2}:\d{2}[,.]\d{3})",
        text
    )

    dialogue_lines = []
    malformed_blocks = 0

    for block in blocks:
        lines = [line.strip() for line in block.splitlines() if line.strip()]

        if not lines:
            continue

        timestamp_found = any("-->" in line for line in lines)

        if timestamp_found:
            if not any(
                re.search(
                    r"\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
                    r"\d{2}:\d{2}:\d{2}[,.]\d{3}",
                    line
                )
                for line in lines
            ):
                malformed_blocks += 1

            for line in lines:
                if "-->" in line or re.fullmatch(r"\d+", line):
                    continue
                dialogue_lines.append(line)

    clean_text = " ".join(dialogue_lines)
    words = re.findall(r"\b[\w'-]+\b", clean_text)

    def count_chars(pattern):
        return len(re.findall(pattern, clean_text))

    return {
        "block_count": len(blocks),
        "valid_timestamp_pairs": len(timestamps),
        "dialogue_lines": len(dialogue_lines),
        "word_count": len(words),
        "devanagari_chars": count_chars(r"[\u0900-\u097F]"),
        "bengali_chars": count_chars(r"[\u0980-\u09FF]"),
        "tamil_chars": count_chars(r"[\u0B80-\u0BFF]"),
        "telugu_chars": count_chars(r"[\u0C00-\u0C7F]"),
        "latin_chars": count_chars(r"[A-Za-z]"),
        "replacement_chars": clean_text.count("\ufffd"),
        "malformed_blocks": malformed_blocks,
        "first_dialogue": " | ".join(dialogue_lines[:5]),
        "last_dialogue": " | ".join(dialogue_lines[-3:])
    }

results = []

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    for _, row in existing_srt.iterrows():
        path = row["archive_srt_path"]

        # Read file bytes directly from ZIP, without extracting.
        raw = zf.read(path)

        try:
            decoded = raw.decode("utf-8-sig")
            encoding = "UTF-8/UTF-8-SIG"
        except UnicodeDecodeError:
            decoded = raw.decode("utf-8", errors="replace")
            encoding = "UTF-8 decode errors; replacement used"

        metrics = inspect_srt_content(decoded)

        results.append({
            "movie_record_id": row["movie_record_id"],
            "movie_title": row["movie_title"],
            "expected_language": row["expected_language"],
            "archive_srt_path": path,
            "file_size_bytes": len(raw),
            "encoding_check": encoding,
            **metrics
        })

srt_inspection = pd.DataFrame(results)

print("=" * 75)
print("EXISTING SRT CONTENT INSPECTION")
print("=" * 75)

display(srt_inspection[[
    "movie_record_id", "movie_title", "expected_language",
    "file_size_bytes", "encoding_check", "block_count",
    "valid_timestamp_pairs", "dialogue_lines", "word_count",
    "devanagari_chars", "bengali_chars", "tamil_chars",
    "telugu_chars", "latin_chars", "replacement_chars",
    "malformed_blocks"
]].reset_index(drop=True))

print("\nDIALOGUE SAMPLES")
display(srt_inspection[[
    "movie_title", "first_dialogue", "last_dialogue"
]].reset_index(drop=True))

print("\nNo files extracted, accepted, replaced, or modified.")

In [ ]:
# ============================================================
# NOTEBOOK 07 — LOCATE UNRESOLVED TAMIL SRT FILES
# Search every SRT path in the archive
# ============================================================

import zipfile
import pandas as pd

ARCHIVE_PATH = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/indicdialogue/dataset.zip"
)

unresolved_ids = {"1958231917", "1954967004", "1955328184"}

found_entries = []

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    for name in zf.namelist():
        if not name.lower().endswith(".srt"):
            continue

        for subtitle_id in unresolved_ids:
            if subtitle_id in name:
                info = zf.getinfo(name)
                found_entries.append({
                    "subtitle_id": subtitle_id,
                    "archive_path": name,
                    "file_size_bytes": info.file_size
                })

results = pd.DataFrame(found_entries)

print("=" * 70)
print("UNRESOLVED TAMIL SRT SEARCH")
print("=" * 70)
print("Subtitle IDs searched:", len(unresolved_ids))
print("Matching SRT entries:", len(results))

if results.empty:
    print("No matching SRT paths found anywhere in the archive.")
else:
    display(results.sort_values("subtitle_id").reset_index(drop=True))

print("\nNo files extracted or modified.")

In [ ]:
# ============================================================
# NOTEBOOK 07 — VERIFY TAMIL CANDIDATE METADATA
# Read-only metadata inspection
# ============================================================

import zipfile
import json
import pandas as pd

ARCHIVE_PATH = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/indicdialogue/dataset.zip"
)

target_ids = {"1958231917", "1954967004", "1955328184"}

metadata_results = []

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    for path in zf.namelist():
        if not path.lower().endswith(".jsonl"):
            continue

        with zf.open(path, "r") as file:
            for line_number, raw_line in enumerate(file, start=1):
                try:
                    record = json.loads(raw_line.decode("utf-8"))
                except (UnicodeDecodeError, json.JSONDecodeError):
                    continue

                metadata = record.get("metadata", {})
                if not isinstance(metadata, dict):
                    continue

                subtitle_id = str(
                    metadata.get("IDSubtitleFile", record.get("ID", ""))
                ).strip()

                if subtitle_id not in target_ids:
                    continue

                metadata_results.append({
                    "subtitle_id": subtitle_id,
                    "jsonl_path": path,
                    "record_line": line_number,
                    "movie_name": metadata.get("MovieName"),
                    "year": metadata.get("Year"),
                    "imdb_id": metadata.get("MovieImdbID"),
                    "subtitle_format": metadata.get("SubFormat"),
                    "metadata_fields": ", ".join(sorted(metadata.keys())),
                    "dialogue_language_codes": ", ".join(
                        record.get("dialogs", {}).keys()
                    ) if isinstance(record.get("dialogs"), dict) else ""
                })

metadata_check = pd.DataFrame(metadata_results)

print("TARGET METADATA RECORDS FOUND:", len(metadata_check))
display(metadata_check.reset_index(drop=True))

print("\nNo files extracted or modified.")

In [ ]:
# ============================================================
# NOTEBOOK 07 — DIAGNOSE TAMIL SRT ENCODING AND SCRIPT
# Read-only; no extraction or file modifications
# ============================================================

import zipfile
import re
import pandas as pd

ARCHIVE_PATH = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/indicdialogue/dataset.zip"
)

targets = {
    "1958231917": ("Anniyan", "IndicDialogue/Tamil/subtitles/1958231917.srt"),
    "1954967004": ("Dasavathaaram", "IndicDialogue/Tamil/subtitles/1954967004.srt"),
    "1955328184": ("Kabali", "IndicDialogue/Tamil/subtitles/1955328184.srt"),
}

def script_counts(text):
    return {
        "tamil_chars": len(re.findall(r"[\u0B80-\u0BFF]", text)),
        "devanagari_chars": len(re.findall(r"[\u0900-\u097F]", text)),
        "bengali_chars": len(re.findall(r"[\u0980-\u09FF]", text)),
        "telugu_chars": len(re.findall(r"[\u0C00-\u0C7F]", text)),
        "latin_chars": len(re.findall(r"[A-Za-z]", text)),
        "replacement_chars": text.count("\ufffd"),
    }

results = []

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    for sid, (title, path) in targets.items():
        raw = zf.read(path)

        for encoding in ["utf-8-sig", "cp1252", "latin-1"]:
            try:
                decoded = raw.decode(encoding, errors="strict")
                counts = script_counts(decoded)
                results.append({
                    "subtitle_id": sid,
                    "movie_title": title,
                    "tested_encoding": encoding,
                    **counts,
                    "sample": " ".join(
                        line.strip()
                        for line in decoded.splitlines()
                        if line.strip() and "-->" not in line
                    )[:220]
                })
            except UnicodeDecodeError:
                results.append({
                    "subtitle_id": sid,
                    "movie_title": title,
                    "tested_encoding": encoding,
                    "decode_error": True
                })

diagnostic = pd.DataFrame(results)

print("=" * 75)
print("ENCODING AND SCRIPT DIAGNOSTIC")
print("=" * 75)
display(diagnostic.reset_index(drop=True))

print("\nNo files extracted or modified.")

In [ ]:

# NOTEBOOK 8 — RESUME AFTER DISCONNECTION
# Step 1: Restore Google Drive access and verify project files

import os
from pathlib import Path

from google.colab import drive

DRIVE_ROOT = "/content/gdrive2"
PROJECT_ROOT = Path(DRIVE_ROOT) / "MyDrive" / "IndianCinemaTrendAnalytics"

if not os.path.isdir(f"{DRIVE_ROOT}/MyDrive"):
    drive.mount(DRIVE_ROOT)

print("=" * 65)
print("NOTEBOOK 8 — RESUME CHECK")
print("=" * 65)

print("Drive mounted:", os.path.isdir(f"{DRIVE_ROOT}/MyDrive"))
print("Project exists:", PROJECT_ROOT.exists())

if PROJECT_ROOT.exists():
    print("\nProject folders:")
    for folder in sorted(PROJECT_ROOT.iterdir()):
        if folder.is_dir():
            print(" -", folder.name)

    processed_dir = PROJECT_ROOT / "01_data" / "processed"
    subtitle_dir = processed_dir / "subtitle_corpus_v2"
    manifest_path = subtitle_dir / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

    print("\nProcessed directory exists:", processed_dir.exists())
    print("Subtitle workspace exists:", subtitle_dir.exists())
    print("Manifest exists:", manifest_path.exists())
else:
    print("STOP: Project path not found. Do not create a new project folder yet.")


In [ ]:

# NOTEBOOK 8 — RESUME CHECK
# Step 2: Inspect subtitle folders, manifest, and existing files

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

SUBTITLE_ROOT = (
    PROJECT_ROOT / "01_data" / "processed" / "subtitle_corpus_v2"
)

RAW_DIR = SUBTITLE_ROOT / "raw"
VALIDATED_DIR = SUBTITLE_ROOT / "validated"

MANIFEST_PATH = (
    SUBTITLE_ROOT / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

print("=" * 65)
print("SUBTITLE WORKSPACE INSPECTION")
print("=" * 65)

for label, folder in [
    ("Raw directory", RAW_DIR),
    ("Validated directory", VALIDATED_DIR),
]:
    print(f"\n{label}: {folder.exists()}")

    if folder.exists():
        files = sorted(p for p in folder.iterdir() if p.is_file())
        print("File count:", len(files))

        for file in files[:15]:
            print(f"  {file.name} ({file.stat().st_size:,} bytes)")

        if len(files) > 15:
            print(f"  ... and {len(files) - 15} more files")

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}"
    )

manifest = pd.read_csv(MANIFEST_PATH)

print("\n" + "=" * 65)
print("MANIFEST SUMMARY")
print("=" * 65)

print("Rows:", len(manifest))
print("Columns:", manifest.columns.tolist())

for column in [
    "acquisition_status",
    "validation_status",
    "subtitle_language",
    "subtitle_source",
]:
    if column in manifest.columns:
        print(f"\n{column}:")
        print(manifest[column].fillna("Missing").value_counts().to_string())

print("\nRecent manifest entries:")
display(manifest.tail(10))


In [ ]:

# NOTEBOOK 8 — RESUME CHECK
# Step 3: Review pending subtitle records and accepted files

import pandas as pd
from pathlib import Path

manifest = pd.read_csv(MANIFEST_PATH)

print("=" * 70)
print("PENDING SUBTITLE RECORDS")
print("=" * 70)

pending = manifest[
    manifest["acquisition_status"].ne("Acquired")
].copy()

columns_to_show = [
    "movie_record_id",
    "movie_title",
    "language",
    "tmdb_id",
    "subtitle_language",
    "source_id",
    "raw_subtitle_file",
    "validated_subtitle_file",
    "acquisition_status",
    "validation_status",
    "notes",
]

columns_to_show = [
    c for c in columns_to_show if c in pending.columns
]

display(pending[columns_to_show].reset_index(drop=True))

print("\n" + "=" * 70)
print("ACCEPTED SUBTITLES — FILE EXISTENCE CHECK")
print("=" * 70)

accepted_files = [
    VALIDATED_DIR / "2000_1_KAHO_NAA_PYAAR_HAI_ENGLISH.srt",
    VALIDATED_DIR / "2000_2_MOHABBATEIN_HINDI.srt",
    VALIDATED_DIR / "2021_4_83_HINDI.srt",
]

for path in accepted_files:
    print(f"{path.name}: {'EXISTS' if path.exists() else 'MISSING'}")

print("\n" + "=" * 70)
print("RAW FILES AVAILABLE FOR REVIEW")
print("=" * 70)

if RAW_DIR.exists():
    for path in sorted(RAW_DIR.glob("*.srt")):
        print(f"{path.name} | {path.stat().st_size:,} bytes")


In [ ]:

from google.colab import userdata

for secret_name in [
    "OPEN_SUBTITLES_API_KEY",
    "OPEN_SUBTITLES_TOKEN",
    "OPEN_SUBTITLES_USERNAME",
    "OPEN_SUBTITLES_PASSWORD",
]:
    try:
        value = userdata.get(secret_name)
        print(f"{secret_name}: {'Available' if value else 'Empty'}")
    except Exception:
        print(f"{secret_name}: Not configured")


In [ ]:

# NOTEBOOK 8 — OPENSUBTITLES RE-AUTHENTICATION
# Logs in, checks the account allowance, and keeps the new token in memory.
# No subtitle downloads are performed.

import requests
from google.colab import userdata

api_key = userdata.get("OPEN_SUBTITLES_API_KEY")
username = userdata.get("OPEN_SUBTITLES_USERNAME")
password = userdata.get("OPEN_SUBTITLES_PASSWORD")

if not all([api_key, username, password]):
    raise ValueError(
        "Missing required Colab Secrets. Check the API key, username, and password."
    )

login_response = requests.post(
    "https://api.opensubtitles.com/api/v1/login",
    headers={
        "Api-Key": api_key,
        "User-Agent": "ICMB130SubtitleResearch v1",
        "Content-Type": "application/json",
        "Accept": "application/json",
    },
    json={"username": username, "password": password},
    timeout=30,
)

print("Login HTTP status:", login_response.status_code)

if login_response.status_code != 200:
    print("Login unsuccessful. No download was attempted.")
    print("Response:", login_response.text[:300])
else:
    login_data = login_response.json()
    ACCESS_TOKEN = login_data.get("token")
    api_host = login_data.get("base_url", "api.opensubtitles.com")

    if not ACCESS_TOKEN:
        print("Login response did not contain a token.")
    else:
        print("Fresh token obtained: Yes")

        # Check account information with the new token.
        info_response = requests.get(
            f"https://{api_host}/api/v1/infos/user",
            headers={
                "Api-Key": api_key,
                "Authorization": f"Bearer {ACCESS_TOKEN}",
                "User-Agent": "ICMB130SubtitleResearch v1",
                "Accept": "application/json",
            },
            timeout=30,
        )

        print("Account check HTTP status:", info_response.status_code)

        if info_response.ok:
            account = info_response.json().get("data", {})
            allowed = account.get("allowed_downloads")
            used = account.get("downloads_count")

            print("Allowed downloads:", allowed)
            print("Downloads used:", used)

            if isinstance(allowed, (int, float)) and isinstance(used, (int, float)):
                remaining = max(0, allowed - used)
                print("Remaining downloads:", remaining)

                if remaining > 0:
                    print("Ready for the next step: subtitle candidate search.")
                else:
                    print("Download allowance exhausted. Do not download subtitles.")
            else:
                print("Could not reliably calculate the allowance. Do not download yet.")
        else:
            print("Could not verify the allowance. Do not download yet.")
            print("Response:", info_response.text[:300])


In [ ]:

# NOTEBOOK 8 — BATCH SUBTITLE CANDIDATE SEARCH
# Search only: no subtitle downloads and no manifest modifications.

import time
import requests
import pandas as pd
from pathlib import Path

# Reuse paths and credentials from the earlier cells.
if "manifest" not in globals():
    manifest = pd.read_csv(MANIFEST_PATH)

if "ACCESS_TOKEN" not in globals() or not ACCESS_TOKEN:
    raise RuntimeError(
        "ACCESS_TOKEN is missing. Run the authentication cell first."
    )

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")

SEARCH_URL = "https://api.opensubtitles.com/api/v1/subtitles"

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "User-Agent": "ICMB130SubtitleResearch v1",
    "Accept": "application/json",
}

# Map the project's language labels to OpenSubtitles language codes.
language_map = {
    "hi": "hi",
    "hindi": "hi",
    "te": "te",
    "telugu": "te",
    "ta": "ta",
    "tamil": "ta",
    "kn": "kn",
    "kannada": "kn",
    "ml": "ml",
    "malayalam": "ml",
    "bn": "bn",
    "bengali": "bn",
    "mr": "mr",
    "marathi": "mr",
}

pending = manifest[
    manifest["acquisition_status"].ne("Acquired")
].copy()

# Keep only records with a usable TMDB ID and language code.
pending["search_language"] = (
    pending["language"].astype(str).str.strip().str.lower()
    .map(language_map)
)

pending["tmdb_id_numeric"] = pd.to_numeric(
    pending["tmdb_id"], errors="coerce"
)

searchable = pending[
    pending["tmdb_id_numeric"].notna()
    & pending["search_language"].notna()
].copy()

print("=" * 70)
print("BATCH SEARCH STARTED")
print("=" * 70)
print("Pending records:", len(pending))
print("Records eligible for search:", len(searchable))
print("This cell will not download subtitle files.")

candidate_rows = []
search_log = []

for position, (_, movie) in enumerate(searchable.iterrows(), start=1):
    movie_id = str(movie["movie_record_id"])
    title = str(movie["movie_title"])
    year = int(movie["year"])
    tmdb_id = int(movie["tmdb_id_numeric"])
    language_code = movie["search_language"]

    params = {
        "tmdb_id": tmdb_id,
        "languages": language_code,
        "order_by": "download_count",
        "order_direction": "desc",
    }

    try:
        response = requests.get(
            SEARCH_URL,
            headers=headers,
            params=params,
            timeout=30,
        )

        if response.status_code == 401:
            search_log.append({
                "movie_record_id": movie_id,
                "movie_title": title,
                "year": year,
                "tmdb_id": tmdb_id,
                "language_code": language_code,
                "http_status": 401,
                "candidate_count": 0,
                "status": "Authentication failed; batch stopped",
            })
            print("Authentication failed. Stopping the batch.")
            break

        if response.status_code == 429:
            search_log.append({
                "movie_record_id": movie_id,
                "movie_title": title,
                "year": year,
                "tmdb_id": tmdb_id,
                "language_code": language_code,
                "http_status": 429,
                "candidate_count": 0,
                "status": "Rate limited; batch stopped",
            })
            print("Rate limit reached. Stopping the batch.")
            break

        if not response.ok:
            search_log.append({
                "movie_record_id": movie_id,
                "movie_title": title,
                "year": year,
                "tmdb_id": tmdb_id,
                "language_code": language_code,
                "http_status": response.status_code,
                "candidate_count": 0,
                "status": "Search request failed",
            })
            print(f"[{position}/{len(searchable)}] {title}: HTTP {response.status_code}")
            time.sleep(0.3)
            continue

        results = response.json().get("data", [])
        movie_candidate_count = 0

        for result in results:
            attrs = result.get("attributes", {})
            feature = attrs.get("feature_details") or {}

            for file_info in attrs.get("files", []):
                candidate_rows.append({
                    "movie_record_id": movie_id,
                    "dataset_title": title,
                    "dataset_year": year,
                    "dataset_tmdb_id": tmdb_id,
                    "requested_language": language_code,
                    "file_id": file_info.get("file_id"),
                    "file_name": file_info.get("file_name"),
                    "release": attrs.get("release"),
                    "subtitle_language": attrs.get("language"),
                    "download_count": attrs.get("download_count"),
                    "hearing_impaired": attrs.get("hearing_impaired"),
                    "format": attrs.get("format"),
                    "fps": attrs.get("fps"),
                    "upload_date": attrs.get("upload_date"),
                    "feature_title": feature.get("title"),
                    "feature_year": feature.get("year"),
                    "feature_tmdb_id": feature.get("tmdb_id"),
                    "feature_imdb_id": feature.get("imdb_id"),
                })
                movie_candidate_count += 1

        search_log.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "year": year,
            "tmdb_id": tmdb_id,
            "language_code": language_code,
            "http_status": response.status_code,
            "candidate_count": movie_candidate_count,
            "status": (
                "Candidates found"
                if movie_candidate_count
                else "No candidates returned"
            ),
        })

        print(
            f"[{position}/{len(searchable)}] {title} ({year}): "
            f"{movie_candidate_count} candidate file(s)"
        )

    except requests.RequestException as exc:
        search_log.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "year": year,
            "tmdb_id": tmdb_id,
            "language_code": language_code,
            "http_status": None,
            "candidate_count": 0,
            "status": f"Request error: {type(exc).__name__}",
        })
        print(f"[{position}/{len(searchable)}] {title}: request error")

    # Small pause to reduce the chance of hitting API rate limits.
    time.sleep(0.3)

# Save to new files; do not overwrite the manifest or accepted subtitles.
search_dir = SUBTITLE_ROOT / "search_results"
search_dir.mkdir(parents=True, exist_ok=True)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

candidates_df = pd.DataFrame(candidate_rows)
search_log_df = pd.DataFrame(search_log)

candidates_path = search_dir / f"ICMB130_OPENSUBTITLES_CANDIDATES_{timestamp}.csv"
log_path = search_dir / f"ICMB130_OPENSUBTITLES_SEARCH_LOG_{timestamp}.csv"

candidates_df.to_csv(candidates_path, index=False)
search_log_df.to_csv(log_path, index=False)

print("\n" + "=" * 70)
print("BATCH SEARCH SUMMARY")
print("=" * 70)
print("Movies searched:", len(search_log_df))
print("Candidate files found:", len(candidates_df))
print("Movies with candidates:", (
    int((search_log_df["candidate_count"] > 0).sum())
    if not search_log_df.empty else 0
))
print("Movies with no candidates:", (
    int((search_log_df["candidate_count"] == 0).sum())
    if not search_log_df.empty else 0
))
print("Candidate CSV:", candidates_path)
print("Search log CSV:", log_path)

if not candidates_df.empty:
    display(
        candidates_df[
            [
                "movie_record_id",
                "dataset_title",
                "dataset_year",
                "requested_language",
                "file_id",
                "file_name",
                "subtitle_language",
                "download_count",
                "feature_title",
                "feature_year",
                "feature_tmdb_id",
            ]
        ].sort_values(
            ["dataset_year", "dataset_title", "download_count"],
            ascending=[True, True, False],
        ).reset_index(drop=True)
    )

display(search_log_df)


In [ ]:

# NOTEBOOK 8 — CANDIDATE REVIEW
# Read-only: no downloads and no manifest changes.

from pathlib import Path
import pandas as pd

raw_files = sorted(RAW_DIR.glob("*.srt"))

# Identify the record ID from filenames such as 2001_3_candidate1.srt.
local_rows = []

for path in raw_files:
    parts = path.name.split("_")
    record_id = "_".join(parts[:2]) if len(parts) >= 2 else ""

    local_rows.append({
        "movie_record_id": record_id,
        "local_file": path.name,
        "local_size_bytes": path.stat().st_size,
    })

local_df = pd.DataFrame(local_rows)

# Exclude Mission Kashmir from the priority list because both results
# were already identified as likely song-specific.
review = candidates_df[
    candidates_df["movie_record_id"] != "2000_3"
].copy()

if not local_df.empty:
    local_summary = (
        local_df.groupby("movie_record_id")
        .agg(
            existing_raw_files=("local_file", lambda x: "; ".join(x)),
            existing_file_count=("local_file", "count"),
        )
        .reset_index()
    )

    review = review.merge(
        local_summary,
        on="movie_record_id",
        how="left",
    )
else:
    review["existing_raw_files"] = None
    review["existing_file_count"] = 0

review["existing_file_count"] = (
    review["existing_file_count"].fillna(0).astype(int)
)

review["local_file_status"] = review["existing_file_count"].map(
    lambda n: "Inspect existing file first" if n > 0
    else "No local raw file found"
)

review = review[
    [
        "movie_record_id",
        "dataset_title",
        "dataset_year",
        "file_id",
        "file_name",
        "release",
        "subtitle_language",
        "download_count",
        "feature_title",
        "feature_year",
        "feature_tmdb_id",
        "existing_file_count",
        "existing_raw_files",
        "local_file_status",
    ]
].sort_values(
    ["dataset_year", "dataset_title"]
).reset_index(drop=True)

display(review)


In [ ]:

# NOTEBOOK 8 — EXISTING SUBTITLE DIAGNOSTICS
# Read-only inspection; no downloads or file changes.

import re
import hashlib
from pathlib import Path
import pandas as pd

files_to_check = [
    RAW_DIR / "2001_3_candidate1.srt",
    RAW_DIR / "2001_3_retry1_Lagaan.srt",
    RAW_DIR / "2001_5_retry1_Dil_Chahta_Hai.srt",
]

timestamp_pattern = re.compile(
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
)

def parse_seconds(timestamp):
    timestamp = timestamp.replace(",", ".")
    hours, minutes, seconds = timestamp.split(":")
    return int(hours) * 3600 + int(minutes) * 60 + float(seconds)

def inspect_srt(path):
    if not path.exists():
        return {"file": path.name, "status": "MISSING"}

    raw = path.read_bytes()
    sha256 = hashlib.sha256(raw).hexdigest()

    try:
        text = raw.decode("utf-8-sig")
        encoding = "UTF-8"
    except UnicodeDecodeError:
        text = raw.decode("cp1252", errors="replace")
        encoding = "CP1252 fallback"

    blocks = re.split(r"\n\s*\n", text.replace("\r\n", "\n").replace("\r", "\n").strip())
    valid_blocks = []
    malformed = 0

    for block in blocks:
        lines = block.splitlines()
        time_line = next(
            (line for line in lines if "-->" in line),
            None
        )

        if time_line is None:
            continue

        match = timestamp_pattern.search(time_line)
        if match:
            valid_blocks.append({
                "start": match.group(1),
                "end": match.group(2),
                "text": "\n".join(
                    line for line in lines
                    if line != time_line
                    and not re.fullmatch(r"\s*\d+\s*", line)
                ).strip(),
            })
        else:
            malformed += 1

    dialogue = "\n".join(item["text"] for item in valid_blocks)
    words = re.findall(r"\b[\w'-]+\b", dialogue, flags=re.UNICODE)

    tamil_count = len(re.findall(r"[\u0B80-\u0BFF]", dialogue))
    devanagari_count = len(re.findall(r"[\u0900-\u097F]", dialogue))
    latin_count = len(re.findall(r"[A-Za-z]", dialogue))

    starts = [parse_seconds(item["start"]) for item in valid_blocks]
    ends = [parse_seconds(item["end"]) for item in valid_blocks]
    overlaps = sum(
        starts[i] < ends[i - 1]
        for i in range(1, len(valid_blocks))
    )

    return {
        "file": path.name,
        "status": "Read",
        "size_bytes": len(raw),
        "sha256": sha256,
        "encoding": encoding,
        "blocks": len(valid_blocks),
        "malformed_timestamps": malformed,
        "overlaps": overlaps,
        "word_count": len(words),
        "latin_letters": latin_count,
        "devanagari_chars": devanagari_count,
        "tamil_chars": tamil_count,
        "first_timestamp": valid_blocks[0]["start"] if valid_blocks else None,
        "last_timestamp": valid_blocks[-1]["end"] if valid_blocks else None,
        "first_dialogue": " | ".join(
            item["text"].replace("\n", " ")
            for item in valid_blocks[:3]
        )[:500],
        "last_dialogue": " | ".join(
            item["text"].replace("\n", " ")
            for item in valid_blocks[-3:]
        )[:500],
    }

diagnostics = pd.DataFrame(inspect_srt(path) for path in files_to_check)

display(diagnostics)


In [ ]:

# NOTEBOOK 8 — DIL CHAHTA HAI DEEP VALIDATION
# Read-only: no downloads, no file changes.

import re
from pathlib import Path
import pandas as pd

path = RAW_DIR / "2001_5_retry1_Dil_Chahta_Hai.srt"

raw = path.read_bytes()
text = raw.decode("utf-8-sig")
text = text.replace("\r\n", "\n").replace("\r", "\n")

timestamp_pattern = re.compile(
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
)

def seconds(ts):
    h, m, s = ts.replace(",", ".").split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

blocks = []

for chunk in re.split(r"\n\s*\n", text.strip()):
    lines = chunk.splitlines()
    time_line = next((line for line in lines if "-->" in line), None)

    if not time_line:
        continue

    match = timestamp_pattern.search(time_line)
    if not match:
        continue

    dialogue = "\n".join(
        line for line in lines
        if line != time_line and not re.fullmatch(r"\s*\d+\s*", line)
    ).strip()

    blocks.append({
        "start": seconds(match.group(1)),
        "end": seconds(match.group(2)),
        "start_text": match.group(1),
        "end_text": match.group(2),
        "dialogue": dialogue,
    })

starts = [b["start"] for b in blocks]
ends = [b["end"] for b in blocks]

gaps = [
    (blocks[i]["start"] - blocks[i - 1]["end"], i)
    for i in range(1, len(blocks))
    if blocks[i]["start"] - blocks[i - 1]["end"] > 120
]

dialogue_lines = [
    line.strip()
    for b in blocks
    for line in b["dialogue"].splitlines()
    if line.strip()
]

suspicious_pattern = re.compile(
    r"opensubtitles|subtitle[s]?\s*(by|download|sync)|"
    r"uploaded by|www\.|http[s]?://|"
    r"all rights reserved|end credits|the end",
    re.IGNORECASE,
)

suspicious_lines = [
    line for line in dialogue_lines
    if suspicious_pattern.search(line)
]

print("=" * 65)
print("DIL CHAHTA HAI — DEEP VALIDATION")
print("=" * 65)
print("Valid timestamp blocks:", len(blocks))
print("First timestamp:", blocks[0]["start_text"] if blocks else None)
print("Last timestamp:", blocks[-1]["end_text"] if blocks else None)
print("Timestamp span (minutes):", round((ends[-1] - starts[0]) / 60, 2) if blocks else None)
print("Gaps longer than 2 minutes:", len(gaps))
print("Suspicious/promotional lines:", len(suspicious_lines))

if gaps:
    print("\nLargest gaps (seconds):")
    for gap, index in sorted(gaps, reverse=True)[:10]:
        print(
            round(gap, 1),
            "seconds before block",
            index,
            "| next timestamp:",
            blocks[index]["start_text"],
        )

print("\nFirst 12 dialogue blocks:")
for b in blocks[:12]:
    print(f"[{b['start_text']} --> {b['end_text']}] {b['dialogue'][:180]}")

print("\nLast 12 dialogue blocks:")
for b in blocks[-12:]:
    print(f"[{b['start_text']} --> {b['end_text']}] {b['dialogue'][:180]}")

print("\nPromotional or suspicious lines (first 20):")
for line in suspicious_lines[:20]:
    print("-", line[:250])


In [ ]:
from pathlib import Path
import re
from collections import Counter

file_path = RAW_DIR / "2001_5_retry1_Dil_Chahta_Hai.srt"
text = file_path.read_text(encoding="utf-8-sig", errors="replace")

# Remove timestamps, block numbers, and markup for text analysis
lines = []
for line in text.splitlines():
    line = line.strip()
    if (
        not line
        or line.isdigit()
        or "-->" in line
        or re.fullmatch(r"<[^>]+>", line)
    ):
        continue
    line = re.sub(r"<[^>]+>", "", line)
    lines.append(line)

dialogue = " ".join(lines)

counts = {
    "Devanagari characters": len(re.findall(r"[\u0900-\u097F]", dialogue)),
    "Tamil characters": len(re.findall(r"[\u0B80-\u0BFF]", dialogue)),
    "Latin letters": len(re.findall(r"[A-Za-z]", dialogue)),
    "Total alphabetic characters": len(re.findall(r"[^\W\d_]", dialogue)),
    "Words": len(re.findall(r"\b[\w']+\b", dialogue)),
}

print("=" * 60)
print("DIL CHAHTA HAI — LANGUAGE DIAGNOSTICS")
print("=" * 60)

for label, value in counts.items():
    print(f"{label}: {value}")

print("\nSample dialogue lines (excluding promotional text):")
shown = 0
for line in lines:
    if "opensubtitles" in line.lower():
        continue
    if re.search(r"[A-Za-z]", line):
        print("-", line[:180])
        shown += 1
    if shown >= 20:
        break

In [ ]:
import requests
from pathlib import Path

file_id = 10547365
output_path = RAW_DIR / "2003_2_Kal_Ho_Naa_Ho_candidate1.srt"

# Safety check: never overwrite an existing file.
if output_path.exists():
    print("STOP: File already exists:", output_path.name)
    print("Size:", output_path.stat().st_size, "bytes")
else:
    download_response = requests.post(
        "https://api.opensubtitles.com/api/v1/download",
        headers={
            "Api-Key": API_KEY,
            "Authorization": f"Bearer {ACCESS_TOKEN}",
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "ICMB130SubtitleResearch v1"
        },
        json={"file_id": file_id},
        timeout=30
    )

    print("Download authorization HTTP status:", download_response.status_code)

    if download_response.status_code != 200:
        print("Download authorization failed.")
        print(download_response.text[:500])
    else:
        download_info = download_response.json()
        download_link = download_info.get("link")

        if not download_link:
            print("STOP: API did not return a download link.")
            print("Response fields:", list(download_info.keys()))
        else:
            print("Selected file ID:", file_id)
            print("Remaining downloads reported by API:",
                  download_info.get("remaining", "Not provided"))

            # Fetch only the selected subtitle file.
            file_response = requests.get(download_link, timeout=60)
            content = file_response.content

            if file_response.status_code != 200:
                print("File retrieval failed:", file_response.status_code)
            elif not content or b"-->" not in content:
                print("STOP: Retrieved content does not resemble an SRT file.")
                print("Content type:", file_response.headers.get("Content-Type"))
                print("Bytes received:", len(content))
            else:
                # Exclusive creation prevents accidental overwriting.
                with open(output_path, "xb") as f:
                    f.write(content)

                print("\nFile saved successfully.")
                print("Path:", output_path)
                print("Size:", output_path.stat().st_size, "bytes")
                print("Next: validate encoding, language, timestamps, coverage, and quality.")

In [ ]:
from pathlib import Path
import re
import hashlib
from collections import Counter

file_path = RAW_DIR / "2003_2_Kal_Ho_Naa_Ho_candidate1.srt"
raw_bytes = file_path.read_bytes()

print("=" * 65)
print("KAL HO NAA HO — DEEP VALIDATION")
print("=" * 65)
print("File size:", len(raw_bytes), "bytes")
print("SHA256:", hashlib.sha256(raw_bytes).hexdigest())

# Decode without changing the original file.
encoding_used = None
decoded_text = None

for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
    try:
        decoded_text = raw_bytes.decode(encoding)
        encoding_used = encoding
        break
    except UnicodeDecodeError:
        continue

print("Detected readable encoding:", encoding_used)

# Parse SRT timestamp blocks.
timestamp_pattern = re.compile(
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
)

def to_seconds(timestamp):
    timestamp = timestamp.replace(",", ".")
    hours, minutes, seconds = timestamp.split(":")
    return int(hours) * 3600 + int(minutes) * 60 + float(seconds)

blocks = []
for match in timestamp_pattern.finditer(decoded_text):
    start_text, end_text = match.groups()
    start = to_seconds(start_text)
    end = to_seconds(end_text)

    following_text = decoded_text[match.end():]
    next_match = timestamp_pattern.search(following_text)
    dialogue = following_text[:next_match.start()] if next_match else following_text
    dialogue = re.sub(r"^\s*\d+\s*", "", dialogue).strip()

    blocks.append({
        "start": start,
        "end": end,
        "start_text": start_text,
        "end_text": end_text,
        "dialogue": dialogue
    })

print("\n--- TIMESTAMP DIAGNOSTICS ---")
print("Valid timestamp blocks:", len(blocks))

if blocks:
    print("First timestamp:", blocks[0]["start_text"])
    print("Last timestamp:", blocks[-1]["end_text"])
    print("Timestamp span (minutes):",
          round((blocks[-1]["end"] - blocks[0]["start"]) / 60, 2))

    malformed_order = sum(b["end"] <= b["start"] for b in blocks)
    overlaps = sum(
        blocks[i]["start"] < blocks[i - 1]["end"]
        for i in range(1, len(blocks))
    )
    gaps = [
        (blocks[i]["start"] - blocks[i - 1]["end"], i, blocks[i]["start_text"])
        for i in range(1, len(blocks))
        if blocks[i]["start"] - blocks[i - 1]["end"] > 120
    ]

    print("Invalid/reversed timestamp blocks:", malformed_order)
    print("Overlapping blocks:", overlaps)
    print("Gaps longer than 2 minutes:", len(gaps))

    for gap, index, next_start in sorted(gaps, reverse=True)[:10]:
        print(f"  {gap:.1f} seconds before block {index} | next timestamp: {next_start}")

# Analyze dialogue text, excluding obvious promotional lines.
dialogue_lines = [
    line.strip()
    for block in blocks
    for line in block["dialogue"].splitlines()
    if line.strip()
]

joined_text = " ".join(dialogue_lines)
print("\n--- LANGUAGE DIAGNOSTICS ---")
print("Devanagari characters:", len(re.findall(r"[\u0900-\u097F]", joined_text)))
print("Tamil characters:", len(re.findall(r"[\u0B80-\u0BFF]", joined_text)))
print("Latin letters:", len(re.findall(r"[A-Za-z]", joined_text)))
print("Words:", len(re.findall(r"\b[\w']+\b", joined_text)))

suspicious_terms = (
    "opensubtitles", "uploading subtitles", "download subtitles",
    "subtitles by", "synced by", "uploaded by", "www.",
    "http://", "https://", "all rights reserved"
)
suspicious_lines = [
    line for line in dialogue_lines
    if any(term in line.lower() for term in suspicious_terms)
]
print("Promotional/suspicious lines:", len(suspicious_lines))
for line in suspicious_lines[:15]:
    print("-", line[:200])

print("\n--- FIRST 10 DIALOGUE BLOCKS ---")
for block in blocks[:10]:
    print(f"[{block['start_text']} --> {block['end_text']}]")
    print(block["dialogue"][:250])

print("\n--- LAST 10 DIALOGUE BLOCKS ---")
for block in blocks[-10:]:
    print(f"[{block['start_text']} --> {block['end_text']}]")
    print(block["dialogue"][:250])

In [ ]:
import hashlib
import shutil

source_path = RAW_DIR / "2003_2_Kal_Ho_Naa_Ho_candidate1.srt"
destination_path = VALIDATED_DIR / "2003_2_KAL_HO_NAA_HO_HINDI.srt"

expected_sha256 = "3276f290fab50f74a0cf56d3760fd0abb3ed7f2aaee929f078050aa043d4fc7d"

actual_sha256 = hashlib.sha256(source_path.read_bytes()).hexdigest()

print("Source exists:", source_path.exists())
print("Source SHA256:", actual_sha256)
print("Expected hash matches:", actual_sha256 == expected_sha256)
print("Destination exists:", destination_path.exists())

if not source_path.exists():
    print("STOP: Source file not found.")
elif actual_sha256 != expected_sha256:
    print("STOP: Source hash differs from the inspected file.")
elif destination_path.exists():
    print("STOP: Destination already exists; nothing overwritten.")
else:
    shutil.copy2(source_path, destination_path)
    copied_hash = hashlib.sha256(destination_path.read_bytes()).hexdigest()

    if copied_hash == expected_sha256:
        print("\nAccepted copy created successfully.")
        print("Path:", destination_path)
        print("SHA256 verified:", copied_hash)
        print("Quality flag: Accepted_with_gap_flag")
    else:
        print("STOP: Copied file hash mismatch; do not use this copy.")

In [ ]:
row = manifest.loc[manifest["movie_record_id"].astype(str) == "2003_2"]

if len(row) != 1:
    print(f"STOP: Expected exactly 1 record, found {len(row)}.")
else:
    for column in manifest.columns:
        print(f"{column}: {row.iloc[0][column]}")

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

record_id = "2003_2"

raw_path = RAW_DIR / "2003_2_Kal_Ho_Naa_Ho_candidate1.srt"
validated_path = VALIDATED_DIR / "2003_2_KAL_HO_NAA_HO_HINDI.srt"

# Safety checks
if not raw_path.exists() or not validated_path.exists():
    raise FileNotFoundError("Raw or validated subtitle file is missing.")

if hashlib.sha256(raw_path.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError("Raw subtitle hash differs from the reviewed file.")

if hashlib.sha256(validated_path.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError("Validated subtitle hash differs from the reviewed file.")

current_manifest = pd.read_csv(MANIFEST_PATH)
mask = current_manifest["movie_record_id"].astype(str) == record_id

if int(mask.sum()) != 1:
    raise ValueError(f"Expected exactly one {record_id} row; found {int(mask.sum())}.")

# Create a timestamped backup before editing.
backup_path = MANIFEST_PATH.with_name(
    f"{MANIFEST_PATH.stem}_backup_before_{record_id}{MANIFEST_PATH.suffix}"
)
if backup_path.exists():
    raise FileExistsError(f"Backup already exists; refusing to overwrite: {backup_path}")

shutil.copy2(MANIFEST_PATH, backup_path)

# Update only the identified record.
updates = {
    "subtitle_language": "hi",
    "subtitle_type": "Full-film dialogue subtitle; original-language Hindi",
    "subtitle_source": "OpenSubtitles",
    "source_id": "10547365",
    "raw_subtitle_file": str(raw_path),
    "validated_subtitle_file": str(validated_path),
    "acquisition_status": "Downloaded",
    "validation_status": "Validated_with_quality_flag",
    "title_verified": True,
    "year_verified": True,
    "movie_identity_verified": True,
    "language_verified": True,
    "dialogue_blocks": 704,
    "word_count": 7352,
    "character_count": 14796,
    "encoding_status": "UTF-8-sig",
    "quality_status": "Accepted_with_gap_flag",
    "notes": (
        "OpenSubtitles file ID 10547365; matched title/year/TMDB ID. "
        "Hindi Devanagari dialogue confirmed. 704 timestamp blocks; "
        "0 invalid/reversed timestamps; 0 overlaps; 16 gaps over 2 minutes. "
        "Broad film-span coverage with uneven subtitle density; accepted for "
        "topic modelling with a documented coverage limitation. "
        f"SHA256: {expected_sha256}"
    )
}

for column, value in updates.items():
    current_manifest.loc[mask, column] = value

# Persist and verify.
current_manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")
reloaded = pd.read_csv(MANIFEST_PATH)
saved_row = reloaded.loc[
    reloaded["movie_record_id"].astype(str) == record_id
]

if len(saved_row) != 1:
    raise RuntimeError("Post-save verification failed: record count changed.")

for column in updates:
    if pd.isna(saved_row.iloc[0][column]):
        raise RuntimeError(f"Post-save verification failed for {column}.")

print("Manifest updated and reloaded successfully.")
print("Record:", record_id, "|", saved_row.iloc[0]["movie_title"])
print("Validation:", saved_row.iloc[0]["validation_status"])
print("Quality:", saved_row.iloc[0]["quality_status"])
print("Subtitle source ID:", saved_row.iloc[0]["source_id"])
print("Backup:", backup_path)
print("Manifest rows:", len(reloaded))
print("Only the target record was edited.")

In [ ]:
import pandas as pd
import shutil

record_id = "2003_2"

current_manifest = pd.read_csv(MANIFEST_PATH, dtype={"source_id": "string"})
mask = current_manifest["movie_record_id"].astype(str) == record_id

if int(mask.sum()) != 1:
    raise ValueError(f"Expected exactly one {record_id} record.")

current_source_id = current_manifest.loc[mask, "source_id"].iloc[0]
print("Current source ID:", current_source_id)

if str(current_source_id).strip() not in {"10547365", "10547365.0", "<NA>"}:
    raise ValueError("Unexpected source ID; refusing to modify the manifest.")

backup_path = MANIFEST_PATH.with_name(
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_source_id_fix.csv"
)

if backup_path.exists():
    raise FileExistsError(f"Backup already exists: {backup_path}")

shutil.copy2(MANIFEST_PATH, backup_path)

current_manifest.loc[mask, "source_id"] = "10547365"
current_manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

verified = pd.read_csv(MANIFEST_PATH, dtype={"source_id": "string"})
saved_value = verified.loc[
    verified["movie_record_id"].astype(str) == record_id,
    "source_id"
].iloc[0]

print("\nSource ID after reload:", repr(saved_value))
print("Source ID correct:", saved_value == "10547365")
print("Manifest records:", len(verified))
print("Backup:", backup_path)

In [ ]:
print("=" * 65)
print("VEER-ZAARA — CANDIDATE REVIEW")
print("=" * 65)

movie_rows = candidates_df[
    candidates_df["movie_record_id"].astype(str) == "2004_?"
] if "movie_record_id" in candidates_df.columns else pd.DataFrame()

# Search candidate metadata by title or known subtitle file ID.
candidate_mask = candidates_df.astype(str).apply(
    lambda col: col.str.contains(
        "Veer-Zaara|4764231",
        case=False,
        regex=True,
        na=False
    )
).any(axis=1)

display(candidates_df.loc[candidate_mask])

print("\nMatching local raw files:")
matches = [
    p for p in RAW_DIR.glob("*")
    if "veer" in p.name.lower() or "zaara" in p.name.lower()
]

if matches:
    for path in matches:
        print(f"{path.name} | {path.stat().st_size:,} bytes")
else:
    print("No matching local raw file found.")

In [ ]:
import requests

file_id = 4764231
output_path = RAW_DIR / "2004_1_Veer_Zaara_candidate1.srt"

if output_path.exists():
    print("STOP: File already exists; nothing overwritten.")
    print("Path:", output_path)
    print("Size:", output_path.stat().st_size, "bytes")
else:
    response = requests.post(
        "https://api.opensubtitles.com/api/v1/download",
        headers={
            "Api-Key": API_KEY,
            "Authorization": f"Bearer {ACCESS_TOKEN}",
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "ICMB130SubtitleResearch v1"
        },
        json={"file_id": file_id},
        timeout=30
    )

    print("Authorization status:", response.status_code)

    if response.status_code != 200:
        print("Download authorization failed.")
        print(response.text[:400])
    else:
        payload = response.json()
        link = payload.get("link")

        if not link:
            print("STOP: No download link returned.")
        else:
            file_response = requests.get(link, timeout=60)
            content = file_response.content

            if file_response.status_code != 200:
                print("File retrieval failed:", file_response.status_code)
            elif not content or b"-->" not in content:
                print("STOP: Response does not look like an SRT file.")
                print("Bytes received:", len(content))
            else:
                with open(output_path, "xb") as f:
                    f.write(content)

                print("File saved:", output_path)
                print("Size:", len(content), "bytes")
                print("Remaining downloads reported by API:",
                      payload.get("remaining", "Not provided"))

In [ ]:
import re
import hashlib
from collections import Counter

file_path = RAW_DIR / "2004_1_Veer_Zaara_candidate1.srt"
raw_bytes = file_path.read_bytes()

print("=" * 65)
print("VEER-ZAARA — CONSOLIDATED SUBTITLE VALIDATION")
print("=" * 65)
print("File size:", len(raw_bytes), "bytes")
print("SHA256:", hashlib.sha256(raw_bytes).hexdigest())

# Decode file
text = None
encoding_used = None

for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
    try:
        text = raw_bytes.decode(encoding)
        encoding_used = encoding
        break
    except UnicodeDecodeError:
        continue

print("Encoding:", encoding_used)

# Parse timestamps and associate each timestamp with its dialogue
timestamp_pattern = re.compile(
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
)

def seconds(ts):
    hh, mm, ss = ts.replace(",", ".").split(":")
    return int(hh) * 3600 + int(mm) * 60 + float(ss)

matches = list(timestamp_pattern.finditer(text))
blocks = []

for i, match in enumerate(matches):
    end_pos = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    segment = text[match.end():end_pos]
    segment = re.sub(r"^\s*\d+\s*", "", segment).strip()

    blocks.append({
        "start": seconds(match.group(1)),
        "end": seconds(match.group(2)),
        "start_text": match.group(1),
        "end_text": match.group(2),
        "dialogue": segment
    })

print("\n--- STRUCTURE ---")
print("Timestamp blocks:", len(blocks))

if blocks:
    print("First timestamp:", blocks[0]["start_text"])
    print("Last timestamp:", blocks[-1]["end_text"])
    print("Span (minutes):",
          round((blocks[-1]["end"] - blocks[0]["start"]) / 60, 2))

    invalid = sum(b["end"] <= b["start"] for b in blocks)
    overlaps = sum(
        blocks[i]["start"] < blocks[i - 1]["end"]
        for i in range(1, len(blocks))
    )
    gaps = [
        (blocks[i]["start"] - blocks[i - 1]["end"], i)
        for i in range(1, len(blocks))
        if blocks[i]["start"] - blocks[i - 1]["end"] > 120
    ]

    print("Invalid/reversed blocks:", invalid)
    print("Overlapping blocks:", overlaps)
    print("Gaps over 2 minutes:", len(gaps))
    print("Largest gaps (seconds):",
          [round(gap, 1) for gap, _ in sorted(gaps, reverse=True)[:10]])

dialogue_lines = [
    line.strip()
    for b in blocks
    for line in b["dialogue"].splitlines()
    if line.strip()
]

dialogue = " ".join(dialogue_lines)

print("\n--- LANGUAGE AND CONTENT ---")
print("Devanagari characters:", len(re.findall(r"[\u0900-\u097F]", dialogue)))
print("Tamil characters:", len(re.findall(r"[\u0B80-\u0BFF]", dialogue)))
print("Latin letters:", len(re.findall(r"[A-Za-z]", dialogue)))
print("Words:", len(re.findall(r"\b[\w']+\b", dialogue)))

promo_terms = (
    "opensubtitles", "uploading subtitles", "download subtitles",
    "subtitles by", "synced by", "uploaded by", "www.",
    "http://", "https://", "all rights reserved"
)
promo_lines = [
    line for line in dialogue_lines
    if any(term in line.lower() for term in promo_terms)
]
print("Promotional/suspicious lines:", len(promo_lines))
for line in promo_lines[:10]:
    print("-", line[:180])

print("\n--- FIRST 5 DIALOGUE BLOCKS ---")
for b in blocks[:5]:
    print(f"[{b['start_text']} --> {b['end_text']}] {b['dialogue'][:180]}")

print("\n--- LAST 5 DIALOGUE BLOCKS ---")
for b in blocks[-5:]:
    print(f"[{b['start_text']} --> {b['end_text']}] {b['dialogue'][:180]}")

print("\n--- AUTOMATIC SCREENING ---")
checks = {
    "File is non-empty": len(raw_bytes) > 0,
    "At least 100 timestamp blocks": len(blocks) >= 100,
    "Hindi script present": len(re.findall(r"[\u0900-\u097F]", dialogue)) > 0,
    "No invalid/reversed timestamps": bool(blocks) and invalid == 0,
    "No overlapping blocks": bool(blocks) and overlaps == 0,
    "No promotional lines detected": len(promo_lines) == 0
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'REVIEW'} — {name}")

In [ ]:
print("=" * 65)
print("KOI... MIL GAYA — CANDIDATE REVIEW")
print("=" * 65)

candidate_mask = candidates_df.astype(str).apply(
    lambda col: col.str.contains(
        r"Koi.*Mil Gaya|12769094",
        case=False,
        regex=True,
        na=False
    )
).any(axis=1)

display(candidates_df.loc[candidate_mask])

print("\nMatching local raw files:")
matches = [
    p for p in RAW_DIR.glob("*")
    if "koi" in p.name.lower() and "mil" in p.name.lower()
]

if matches:
    for path in matches:
        print(f"{path.name} | {path.stat().st_size:,} bytes")
else:
    print("No matching local raw file found.")

In [ ]:
print("=" * 65)
print("KOI... MIL GAYA — PRE-DOWNLOAD INSPECTION")
print("=" * 65)

candidate_mask = (
    candidates_df["file_id"].astype(str).str.strip().eq("12769094")
)

if int(candidate_mask.sum()) != 1:
    print("STOP: Expected exactly one candidate record.")
    display(candidates_df.loc[candidate_mask])
else:
    candidate = candidates_df.loc[candidate_mask].iloc[0]

    fields = [
        "dataset_title",
        "dataset_year",
        "dataset_tmdb_id",
        "file_id",
        "file_name",
        "release",
        "subtitle_language",
        "download_count",
        "hearing_impaired",
        "format",
        "fps",
        "upload_date",
        "feature_title",
        "feature_year",
        "feature_tmdb_id",
        "feature_imdb_id",
    ]

    for field in fields:
        if field in candidates_df.columns:
            print(f"{field}: {candidate[field]}")

    print("\n--- LOCAL FILE CHECK ---")
    matches = [
        p for p in RAW_DIR.glob("*")
        if "koi" in p.name.lower() and "mil" in p.name.lower()
    ]

    if matches:
        for path in matches:
            print(f"{path.name} | {path.stat().st_size:,} bytes")
    else:
        print("No matching local raw file found.")

    print("\n--- PRE-DOWNLOAD ASSESSMENT ---")
    print("Movie identity match:",
          str(candidate.get("feature_tmdb_id")) ==
          str(candidate.get("dataset_tmdb_id")))
    print("Language label is Hindi:",
          str(candidate.get("subtitle_language")).lower() == "hi")
    print("Release name:", candidate.get("release"))
    print("Decision: Metadata inspection only; no download performed.")

In [ ]:
import requests

file_id = 12769094
output_path = RAW_DIR / "2003_1_Koi_Mil_Gaya_candidate1.srt"

if output_path.exists():
    print("STOP: File already exists; nothing overwritten.")
    print("Path:", output_path)
    print("Size:", output_path.stat().st_size, "bytes")
else:
    response = requests.post(
        "https://api.opensubtitles.com/api/v1/download",
        headers={
            "Api-Key": API_KEY,
            "Authorization": f"Bearer {ACCESS_TOKEN}",
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "ICMB130SubtitleResearch v1"
        },
        json={"file_id": file_id},
        timeout=30
    )

    print("Authorization status:", response.status_code)

    if response.status_code != 200:
        print("Download authorization failed.")
        print(response.text[:400])
    else:
        payload = response.json()
        link = payload.get("link")

        if not link:
            print("STOP: No download link returned.")
        else:
            file_response = requests.get(link, timeout=60)
            content = file_response.content

            if file_response.status_code != 200:
                print("File retrieval failed:", file_response.status_code)
            elif not content or b"-->" not in content:
                print("STOP: Retrieved content does not resemble an SRT file.")
                print("Bytes received:", len(content))
            else:
                with open(output_path, "xb") as f:
                    f.write(content)

                print("File saved:", output_path)
                print("Size:", len(content), "bytes")
                print(
                    "Remaining downloads reported by API:",
                    payload.get("remaining", "Not provided")
                )

In [ ]:
import re
import hashlib

file_path = RAW_DIR / "2003_1_Koi_Mil_Gaya_candidate1.srt"
raw_bytes = file_path.read_bytes()

print("=" * 65)
print("KOI... MIL GAYA — CONSOLIDATED VALIDATION")
print("=" * 65)
print("File size:", len(raw_bytes), "bytes")
print("SHA256:", hashlib.sha256(raw_bytes).hexdigest())

# Decode without modifying the raw file.
text = None
encoding_used = None

for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
    try:
        text = raw_bytes.decode(enc)
        encoding_used = enc
        break
    except UnicodeDecodeError:
        continue

print("Encoding:", encoding_used)

timestamp_pattern = re.compile(
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
)

def to_seconds(ts):
    hh, mm, ss = ts.replace(",", ".").split(":")
    return int(hh) * 3600 + int(mm) * 60 + float(ss)

matches = list(timestamp_pattern.finditer(text))
blocks = []

for i, match in enumerate(matches):
    end_pos = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    dialogue = text[match.end():end_pos]
    dialogue = re.sub(r"^\s*\d+\s*", "", dialogue).strip()

    blocks.append({
        "start": to_seconds(match.group(1)),
        "end": to_seconds(match.group(2)),
        "start_text": match.group(1),
        "end_text": match.group(2),
        "dialogue": dialogue
    })

print("\n--- STRUCTURE ---")
print("Timestamp blocks:", len(blocks))

if blocks:
    first = blocks[0]
    last = blocks[-1]
    span_minutes = (last["end"] - first["start"]) / 60
    stated_runtime_seconds = 2 * 3600 + 38 * 60 + 47

    print("First timestamp:", first["start_text"])
    print("Last timestamp:", last["end_text"])
    print("Timestamp span (minutes):", round(span_minutes, 2))
    print("Stated release duration (minutes):",
          round(stated_runtime_seconds / 60, 2))
    print("Span as percentage of stated duration:",
          round(span_minutes * 60 / stated_runtime_seconds * 100, 1), "%")

    invalid = sum(b["end"] <= b["start"] for b in blocks)
    overlaps = sum(
        blocks[i]["start"] < blocks[i - 1]["end"]
        for i in range(1, len(blocks))
    )
    gaps = [
        (blocks[i]["start"] - blocks[i - 1]["end"], i)
        for i in range(1, len(blocks))
        if blocks[i]["start"] - blocks[i - 1]["end"] > 120
    ]

    print("Invalid/reversed timestamp blocks:", invalid)
    print("Overlapping blocks:", overlaps)
    print("Gaps over 2 minutes:", len(gaps))
    print("Largest gaps (seconds):",
          [round(g, 1) for g, _ in sorted(gaps, reverse=True)[:10]])

dialogue_lines = [
    line.strip()
    for b in blocks
    for line in b["dialogue"].splitlines()
    if line.strip()
]
dialogue = " ".join(dialogue_lines)

print("\n--- LANGUAGE AND CONTENT ---")
print("Devanagari characters:", len(re.findall(r"[\u0900-\u097F]", dialogue)))
print("Tamil characters:", len(re.findall(r"[\u0B80-\u0BFF]", dialogue)))
print("Latin letters:", len(re.findall(r"[A-Za-z]", dialogue)))
print("Words:", len(re.findall(r"\b[\w']+\b", dialogue)))

suspicious_terms = (
    "opensubtitles", "uploading subtitles", "download subtitles",
    "subtitles by", "synced by", "uploaded by", "www.",
    "http://", "https://", "all rights reserved"
)
suspicious_lines = [
    line for line in dialogue_lines
    if any(term in line.lower() for term in suspicious_terms)
]
sdh_lines = [
    line for line in dialogue_lines
    if re.search(
        r"\[(?:music|song|applause|laughter|sighs?|gasps?|"
        r"door|phone|crying|inaudible)|"
        r"♪|♫|<i>\s*\[",
        line,
        flags=re.IGNORECASE
    )
]

print("Promotional/suspicious lines:", len(suspicious_lines))
for line in suspicious_lines[:10]:
    print("-", line[:180])

print("Possible sound-description/music lines:", len(sdh_lines))
for line in sdh_lines[:10]:
    print("-", line[:180])

print("\n--- FIRST 5 BLOCKS ---")
for b in blocks[:5]:
    print(f"[{b['start_text']} --> {b['end_text']}] {b['dialogue'][:180]}")

print("\n--- LAST 5 BLOCKS ---")
for b in blocks[-5:]:
    print(f"[{b['start_text']} --> {b['end_text']}] {b['dialogue'][:180]}")

In [ ]:
import hashlib
import shutil

source_path = RAW_DIR / "2003_1_Koi_Mil_Gaya_candidate1.srt"
destination_path = VALIDATED_DIR / "2003_1_KOI_MIL_GAYA_HINDI.srt"

expected_sha256 = "d15a502c5bfb4d411e028ee6736ce652df67bbdcc54a3d9be9eb46d39a9b889c"

if not source_path.exists():
    raise FileNotFoundError("Raw subtitle file is missing.")

actual_sha256 = hashlib.sha256(source_path.read_bytes()).hexdigest()

print("Raw SHA256:", actual_sha256)
print("Expected hash matches:", actual_sha256 == expected_sha256)
print("Validated destination exists:", destination_path.exists())

if actual_sha256 != expected_sha256:
    print("STOP: Source differs from the reviewed file.")
elif destination_path.exists():
    print("STOP: Destination already exists; nothing overwritten.")
else:
    shutil.copy2(source_path, destination_path)

    copied_sha256 = hashlib.sha256(destination_path.read_bytes()).hexdigest()

    if copied_sha256 == expected_sha256:
        print("\nValidated copy created successfully.")
        print("Path:", destination_path)
        print("SHA256 verified:", copied_sha256)
        print("Quality flag: Accepted_with_minor_coverage_limitation")
    else:
        print("STOP: Copied file hash mismatch.")

In [ ]:
import pandas as pd
import shutil
import hashlib

record_id = "2003_1"
expected_sha256 = "d15a502c5bfb4d411e028ee6736ce652df67bbdcc54a3d9be9eb46d39a9b889c"

raw_path = RAW_DIR / "2003_1_Koi_Mil_Gaya_candidate1.srt"
validated_path = VALIDATED_DIR / "2003_1_KOI_MIL_GAYA_HINDI.srt"

# Verify both files match the reviewed subtitle.
for path in (raw_path, validated_path):
    if not path.exists():
        raise FileNotFoundError(f"Required file missing: {path}")
    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual_hash != expected_sha256:
        raise ValueError(f"SHA256 mismatch: {path.name}")

# Reload IDs as text to prevent numeric-format issues.
current_manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string", "source_id": "string"}
)

mask = current_manifest["movie_record_id"].eq(record_id)

if int(mask.sum()) != 1:
    raise ValueError(
        f"Expected exactly one {record_id} record; found {int(mask.sum())}."
    )

backup_path = MANIFEST_PATH.with_name(
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_2003_1.csv"
)

if backup_path.exists():
    raise FileExistsError(f"Backup already exists: {backup_path}")

shutil.copy2(MANIFEST_PATH, backup_path)

updates = {
    "subtitle_language": "hi",
    "subtitle_type": "Full-film dialogue subtitle; original-language Hindi",
    "subtitle_source": "OpenSubtitles",
    "source_id": "12769094",
    "raw_subtitle_file": str(raw_path),
    "validated_subtitle_file": str(validated_path),
    "acquisition_status": "Downloaded",
    "validation_status": "Validated_with_quality_flag",
    "title_verified": True,
    "year_verified": True,
    "movie_identity_verified": True,
    "language_verified": True,
    "dialogue_blocks": 1784,
    "word_count": 21543,
    "character_count": 44294,
    "encoding_status": "UTF-8-sig",
    "quality_status": "Accepted_with_minor_coverage_limitation",
    "notes": (
        "OpenSubtitles file ID 12769094; title/year/TMDB identity matched. "
        "Hindi Devanagari dialogue confirmed. 1784 timestamp blocks; "
        "0 invalid/reversed timestamps; 0 overlaps; 0 gaps over 2 minutes. "
        "Timestamp span 155.62 minutes, approximately 98.0% of the stated "
        "release duration. First/last timestamps do not cover the entire "
        "release; accepted for topic modelling with minor coverage limitation. "
        f"SHA256: {expected_sha256}"
    )
}

for column, value in updates.items():
    current_manifest.loc[mask, column] = value

current_manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

# Verify persistence and that the manifest row count is unchanged.
saved = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string", "source_id": "string"}
)
saved_row = saved.loc[saved["movie_record_id"].eq(record_id)]

if len(saved_row) != 1:
    raise RuntimeError("Post-save verification failed.")

if saved_row.iloc[0]["source_id"] != "12769094":
    raise RuntimeError("Source ID verification failed.")

if len(saved) != len(current_manifest):
    raise RuntimeError("Manifest row count changed unexpectedly.")

print("Manifest updated and verified.")
print("Record:", record_id, "|", saved_row.iloc[0]["movie_title"])
print("Source ID:", saved_row.iloc[0]["source_id"])
print("Validation:", saved_row.iloc[0]["validation_status"])
print("Quality:", saved_row.iloc[0]["quality_status"])
print("Manifest records:", len(saved))
print("Backup:", backup_path)

In [ ]:
import pandas as pd
from pathlib import Path

MANIFEST_PATH = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/"
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string", "source_id": "string"}
)

pending = manifest[
    ~manifest["acquisition_status"].astype(str).str.lower().isin(
        ["accepted", "completed"]
    )
].copy()

print("=" * 90)
print("PENDING MOVIES — TARGETED SEARCH PREPARATION")
print("=" * 90)
print("Pending records:", len(pending))

display(
    pending[
        [
            "movie_record_id",
            "year",
            "rank",
            "movie_title",
            "language",
            "tmdb_id",
            "imdb_id",
            "acquisition_status",
            "notes",
        ]
    ].sort_values(["year", "rank"])
)

In [ ]:
from pathlib import Path
import pandas as pd

SUBTITLE_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
VALIDATED_DIR = SUBTITLE_ROOT / "validated"
MANIFEST_PATH = SUBTITLE_ROOT / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string", "source_id": "string"}
)

# Identify records that already have a validated subtitle.
validated_files = {
    p.name.lower()
    for p in VALIDATED_DIR.glob("*")
    if p.is_file()
}

def has_validated_subtitle(record_id):
    prefix = f"{record_id}_".lower()
    return any(name.startswith(prefix) for name in validated_files)

manifest["has_validated_file"] = manifest["movie_record_id"].apply(
    has_validated_subtitle
)

search_queue = manifest[~manifest["has_validated_file"]].copy()
search_queue = search_queue.sort_values(["year", "rank"])

print("=" * 85)
print("CORRECTED SUBTITLE SEARCH QUEUE")
print("=" * 85)
print("Total manifest records:", len(manifest))
print("Records with validated files:", int(manifest["has_validated_file"].sum()))
print("Records requiring further work:", len(search_queue))

display(
    search_queue[
        [
            "movie_record_id",
            "year",
            "movie_title",
            "language",
            "tmdb_id",
            "acquisition_status",
            "validation_status",
            "notes",
        ]
    ]
)

In [ ]:
# Metadata-only shortlist; no downloads and no file changes.

reviewed_ids = {
    "4523478", "4523486", "6571503", "510961",
    "10547365", "4764231", "12769094"
}

shortlist = identity_candidates.copy()
shortlist["file_id"] = shortlist["file_id"].astype(str)

# Remove previously reviewed files.
shortlist = shortlist[~shortlist["file_id"].isin(reviewed_ids)].copy()

# Keep only exact TMDB identity matches and expected feature years.
shortlist["feature_year_num"] = pd.to_numeric(
    shortlist["feature_year"], errors="coerce"
)
shortlist = shortlist[
    shortlist["identity_match"]
    & (shortlist["feature_year_num"] == shortlist["target_year"])
].copy()

# Prefer original-language subtitles; English is fallback.
original_codes = {
    "hi": "hi", "Hindi": "hi",
    "te": "te", "Telugu": "te",
    "ta": "ta", "Tamil": "ta"
}
shortlist["preferred_language"] = shortlist.apply(
    lambda r: 0 if str(r["subtitle_language"]).lower()
    == original_codes.get(str(r["target_language"]), str(r["target_language"]).lower())
    else 1 if str(r["subtitle_language"]).lower() == "en"
    else 2,
    axis=1
)

shortlist = shortlist.sort_values(
    ["movie_record_id", "preferred_language", "download_count"],
    ascending=[True, True, False]
)

# Show up to 3 candidates per movie, for review.
shortlist = shortlist.groupby("movie_record_id", group_keys=False).head(3)

print("Shortlisted candidates:", len(shortlist))
display(
    shortlist[
        [
            "movie_record_id", "target_title", "target_year",
            "file_id", "feature_title", "feature_year",
            "subtitle_language", "filename", "download_count",
            "preferred_language"
        ]
    ].sort_values(["movie_record_id", "preferred_language", "download_count"],
                  ascending=[True, True, False])
)

In [ ]:
# Build a compact review table: one best candidate per pending movie.
# Prefer original-language subtitles, then English fallback.
# Exclude known reviewed candidates and likely duplicate split-disc entries.

review_shortlist = shortlist.copy()

review_shortlist["filename_lower"] = (
    review_shortlist["filename"].fillna("").str.lower()
)

# Flag filenames that may represent only one disc/part.
review_shortlist["split_disc_flag"] = review_shortlist["filename_lower"].str.contains(
    r"\b(cd\s*1|cd\s*2|cd1|cd2|pt1|pt2|part\s*1|part\s*2|disc\s*1|disc\s*2|2\s*cds)\b",
    regex=True
)

# Prefer non-split filenames where language priority is equal.
review_shortlist = review_shortlist.sort_values(
    [
        "movie_record_id",
        "preferred_language",
        "split_disc_flag",
        "download_count"
    ],
    ascending=[True, True, True, False]
)

top_candidates = review_shortlist.groupby(
    "movie_record_id", group_keys=False
).head(1)

print("Movies represented:", top_candidates["movie_record_id"].nunique())
print("Potential split-disc candidates:", int(top_candidates["split_disc_flag"].sum()))

display(
    top_candidates[
        [
            "movie_record_id",
            "target_title",
            "target_year",
            "file_id",
            "subtitle_language",
            "filename",
            "download_count",
            "split_disc_flag"
        ]
    ].sort_values(["target_year", "movie_record_id"])
)

In [ ]:
# Metadata-only review of all candidates already retrieved.

all_candidates = candidates_df.copy()
all_candidates["file_id"] = all_candidates["file_id"].astype(str)

all_candidates["filename_lower"] = (
    all_candidates["filename"].fillna("").str.lower()
)

# Robustly flag CD/part/disc indicators, including "2CD" and "_PT1".
all_candidates["split_disc_flag"] = all_candidates["filename_lower"].str.contains(
    r"(?:\bcd\s*[12]\b|\bcd[12]\b|\b2\s*cds?\b|"
    r"\bpt[12]\b|\bpart\s*[12]\b|\bdisc\s*[12]\b)",
    regex=True,
    na=False
)

original_code = {
    "hi": "hi", "hindi": "hi",
    "te": "te", "telugu": "te",
    "ta": "ta", "tamil": "ta"
}

all_candidates["expected_code"] = (
    all_candidates["target_language"].fillna("")
    .str.lower().map(original_code)
)

all_candidates["is_original_language"] = (
    all_candidates["subtitle_language"].fillna("").str.lower()
    == all_candidates["expected_code"]
)

all_candidates["is_exact_tmdb"] = (
    all_candidates["feature_tmdb_id"].fillna("").astype(str)
    .str.replace(r"\.0$", "", regex=True)
    == all_candidates["target_tmdb_id"].fillna("").astype(str)
)

language_summary = (
    all_candidates[all_candidates["is_exact_tmdb"]]
    .groupby(["movie_record_id", "target_title", "target_language"])
    .agg(
        exact_id_candidates=("file_id", "nunique"),
        original_language_candidates=("is_original_language", "sum"),
        english_candidates=(
            "subtitle_language",
            lambda s: int(s.astype(str).str.lower().eq("en").sum())
        )
    )
    .reset_index()
)

print("Language availability by movie:")
display(language_summary.sort_values("movie_record_id"))

print("\nSelected candidates flagged as split-disc:")
display(
    top_candidates[
        top_candidates["split_disc_flag"]
        if "split_disc_flag" in top_candidates.columns
        else top_candidates["filename"].fillna("").str.lower().str.contains(
            r"(?:\bcd\s*[12]\b|\bcd[12]\b|\b2\s*cds?\b|\bpt[12]\b|\bpart\s*[12]\b|\bdisc\s*[12]\b)",
            regex=True,
            na=False
        )
    ][["movie_record_id", "target_title", "file_id", "filename"]]
)

In [ ]:
file_id = "4295105"

response = requests.get(
    f"https://api.opensubtitles.com/api/v1/subtitles",
    headers=headers,
    params={"file_id": file_id},
    timeout=30
)

print("HTTP status:", response.status_code)
response.raise_for_status()

items = response.json().get("data", [])

if not items:
    print("No metadata returned for this file ID.")
else:
    for item in items:
        attrs = item.get("attributes", {}) or {}
        feature = attrs.get("feature_details", {}) or {}
        files = attrs.get("files", []) or []

        print("=" * 65)
        print("CANDIDATE METADATA")
        print("=" * 65)
        print("File ID:", files[0].get("file_id") if files else file_id)
        print("Filename:", files[0].get("file_name") if files else "Not returned")
        print("Release:", attrs.get("release"))
        print("Subtitle language:", attrs.get("language"))
        print("Download count:", attrs.get("download_count"))
        print("Upload date:", attrs.get("upload_date"))
        print("Movie title:", feature.get("title"))
        print("Movie year:", feature.get("year"))
        print("TMDB ID:", feature.get("tmdb_id"))
        print("IMDb ID:", feature.get("imdb_id"))
        print("Hearing impaired:", attrs.get("hearing_impaired"))
        print("Machine translated:", attrs.get("machine_translated"))
        print("AI translated:", attrs.get("ai_translated"))

In [ ]:
from pathlib import Path
import requests

SUBTITLE_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = SUBTITLE_ROOT / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

file_id = "4295105"
raw_path = RAW_DIR / "2000_4_Dulhan_Hum_Le_Jayenge_candidate_4295105.srt"

if raw_path.exists():
    print("STOP: Destination already exists; no file was overwritten.")
    print("Existing file:", raw_path)
else:
    download_response = requests.post(
        "https://api.opensubtitles.com/api/v1/download",
        headers=headers,
        json={"file_id": int(file_id)},
        timeout=30
    )

    print("Download request status:", download_response.status_code)
    download_response.raise_for_status()
    download_info = download_response.json()

    link = download_info.get("link")
    if not link:
        raise RuntimeError(
            f"No download link returned. API response keys: {list(download_info.keys())}"
        )

    subtitle_response = requests.get(link, timeout=60)
    subtitle_response.raise_for_status()

    content = subtitle_response.content

    # Guard against saving an HTML/error page as a subtitle.
    if not content or b"<html" in content[:500].lower():
        raise RuntimeError("Response does not appear to be a valid subtitle file.")

    raw_path.write_bytes(content)

    print("Saved selected subtitle:", raw_path)
    print("File size:", len(content), "bytes")
    print("Remaining allowance reported by API:", download_info.get("remaining"))

In [ ]:
from pathlib import Path
import re
import hashlib
from collections import Counter

raw_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw/"
    "2000_4_Dulhan_Hum_Le_Jayenge_candidate_4295105.srt"
)

data = raw_path.read_bytes()
sha256 = hashlib.sha256(data).hexdigest()

# Detect encoding without modifying the original file.
if data.startswith(b"\xef\xbb\xbf"):
    encoding = "utf-8-sig"
elif data.startswith(b"\xff\xfe"):
    encoding = "utf-16-le"
elif data.startswith(b"\xfe\xff"):
    encoding = "utf-16-be"
else:
    encoding = "utf-8"

try:
    text = data.decode(encoding)
    encoding_status = f"Decoded successfully ({encoding})"
except UnicodeDecodeError:
    text = data.decode("cp1252", errors="replace")
    encoding_status = "Decoded with cp1252 fallback; inspect replacement characters"

timestamp_pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def to_seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

blocks = re.split(r"\n\s*\n", text.strip())
valid_blocks = []
invalid_timestamp_blocks = 0
reversed_blocks = 0
timestamps = []
all_dialogue = []

for block in blocks:
    lines = block.strip().splitlines()
    match = next(
        (timestamp_pattern.search(line) for line in lines
         if timestamp_pattern.search(line)),
        None
    )

    if not match:
        if "-->" in block:
            invalid_timestamp_blocks += 1
        continue

    values = match.groups()
    start = to_seconds(*values[:4])
    end = to_seconds(*values[4:])

    if end < start:
        reversed_blocks += 1
        continue

    timestamps.append((start, end))
    valid_blocks.append(block)

    timestamp_line_index = next(
        i for i, line in enumerate(lines) if timestamp_pattern.search(line)
    )
    dialogue = " ".join(lines[timestamp_line_index + 1:]).strip()
    if dialogue:
        all_dialogue.append(dialogue)

timestamps.sort(key=lambda x: x[0])
overlaps = sum(
    1 for previous, current in zip(timestamps, timestamps[1:])
    if current[0] < previous[1]
)

gaps_over_2min = sum(
    1 for previous, current in zip(timestamps, timestamps[1:])
    if current[0] - previous[1] > 120
)

dialogue_text = " ".join(all_dialogue)
words = re.findall(r"\b[\w’'-]+\b", dialogue_text, flags=re.UNICODE)

script_counts = {
    "Devanagari characters": len(re.findall(r"[\u0900-\u097F]", dialogue_text)),
    "Tamil characters": len(re.findall(r"[\u0B80-\u0BFF]", dialogue_text)),
    "Telugu characters": len(re.findall(r"[\u0C00-\u0C7F]", dialogue_text)),
    "Latin letters": len(re.findall(r"[A-Za-z]", dialogue_text)),
    "replacement characters": text.count("\ufffd"),
}

print("=" * 70)
print("DULHAN HUM LE JAYENGE — SUBTITLE VALIDATION")
print("=" * 70)
print("File:", raw_path.name)
print("File size:", len(data), "bytes")
print("SHA256:", sha256)
print("Encoding:", encoding_status)
print("Blocks parsed:", len(blocks))
print("Valid timestamp blocks:", len(valid_blocks))
print("Invalid timestamp blocks:", invalid_timestamp_blocks)
print("Reversed timestamps:", reversed_blocks)
print("Overlapping timestamps:", overlaps)
print("Gaps over 2 minutes:", gaps_over_2min)
print("Word count:", len(words))
print("First subtitle time (seconds):", timestamps[0][0] if timestamps else None)
print("Last subtitle end (seconds):", timestamps[-1][1] if timestamps else None)
print("Script counts:", script_counts)
print("\nFirst 5 dialogue blocks:")
for block in valid_blocks[:5]:
    print("-" * 40)
    print(block[:500])

In [ ]:
import pandas as pd
from pathlib import Path

project_root = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

master_path = project_root / "01_data/processed/ICMB_130_MASTER_DATA.csv"
metadata_path = project_root / "01_data/raw/ICMB_130_TMDB_METADATA.csv"

for path in [master_path, metadata_path]:
    print(f"\n{'=' * 65}")
    print(path.name)

    if not path.exists():
        print("File not found.")
        continue

    df = pd.read_csv(path, dtype={"movie_record_id": "string"})

    id_col = next(
        (c for c in df.columns if c.lower() == "movie_record_id"),
        None
    )
    title_col = next(
        (c for c in df.columns
         if c.lower() in ["movie_title", "title", "dataset_title"]),
        None
    )

    mask = pd.Series(False, index=df.index)

    if id_col:
        mask |= df[id_col].astype(str).str.strip().eq("2000_4")

    if title_col:
        mask |= df[title_col].astype(str).str.contains(
            "Dulhan Hum Le Jayenge", case=False, na=False
        )

    matches = df.loc[mask]

    if matches.empty:
        print("No matching movie row found.")
    else:
        # Print the matching row, including any runtime-related columns.
        relevant_cols = [
            c for c in matches.columns
            if any(term in c.lower() for term in [
                "record_id", "title", "year", "runtime",
                "duration", "tmdb_id", "release_date"
            ])
        ]
        print(matches[relevant_cols].to_string(index=False))

print("\nCoverage calculated from the subtitle:")
first_seconds = 270.570
last_seconds = 8191.512
print(f"First subtitle: {first_seconds / 60:.2f} minutes")
print(f"Last subtitle ends: {last_seconds / 60:.2f} minutes")
print(f"Subtitle span: {(last_seconds - first_seconds) / 60:.2f} minutes")

In [ ]:
from pathlib import Path
import shutil
import hashlib

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

raw_path = base / "raw/2000_4_Dulhan_Hum_Le_Jayenge_candidate_4295105.srt"
validated_path = base / "validated/2000_4_DULHAN_HUM_LE_JAYENGE_ENGLISH.srt"

assert raw_path.exists(), f"Raw subtitle missing: {raw_path}"
assert not validated_path.exists(), (
    f"Validated destination already exists; refusing to overwrite: {validated_path}"
)

# Preserve the downloaded bytes exactly in the validated copy.
shutil.copy2(raw_path, validated_path)

raw_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
validated_hash = hashlib.sha256(validated_path.read_bytes()).hexdigest()

assert raw_hash == validated_hash, "Hash mismatch: copy verification failed"

print("Validated copy created successfully.")
print("File:", validated_path.name)
print("Size:", validated_path.stat().st_size, "bytes")
print("SHA256:", validated_hash)
print("Raw file preserved:", raw_path.exists())
print("Byte-for-byte match:", raw_hash == validated_hash)

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import hashlib

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
manifest_path = base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
raw_path = base / "raw/2000_4_Dulhan_Hum_Le_Jayenge_candidate_4295105.srt"
validated_path = base / "validated/2000_4_DULHAN_HUM_LE_JAYENGE_ENGLISH.srt"

# Preconditions: do not update unless both files exist and match.
assert manifest_path.exists(), "Manifest not found."
assert raw_path.exists(), "Raw subtitle not found."
assert validated_path.exists(), "Validated subtitle not found."
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == hashlib.sha256(
    validated_path.read_bytes()
).hexdigest(), "Raw and validated files do not match."

manifest = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"}
)

target_id = "2000_4"
matches = manifest.index[
    manifest["movie_record_id"].astype("string").str.strip().eq(target_id)
].tolist()

assert len(matches) == 1, (
    f"Expected exactly one {target_id} row; found {len(matches)}. "
    "Manifest not changed."
)

idx = matches[0]
before = manifest.loc[idx].copy()

# Backup the manifest before editing; never overwrite an existing backup.
backup_path = base / (
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_2000_4.csv"
)
assert not backup_path.exists(), (
    f"Backup already exists: {backup_path}. Stop and inspect it before proceeding."
)
manifest.to_csv(backup_path, index=False)

updates = {
    "subtitle_language": "English",
    "subtitle_type": "English fallback",
    "subtitle_source": "OpenSubtitles",
    "source_id": "4295105",
    "raw_subtitle_file": raw_path.name,
    "validated_subtitle_file": validated_path.name,
    "acquisition_status": "Accepted",
    "validation_status": "Validated_with_quality_flag",
    "title_verified": True,
    "year_verified": True,
    "movie_identity_verified": True,
    "language_verified": True,
    "dialogue_blocks": 1035,
    "word_count": 8892,
    "character_count": 34555,
    "encoding_status": "UTF-8; byte-for-byte copy verified",
    "quality_status": "Accepted_with_coverage_gap_flag",
    "notes": (
        "English fallback accepted; Hindi subtitle search by exact TMDB ID "
        "returned no candidates. 1035 valid timestamp blocks; 0 malformed "
        "or reversed timestamps; 0 overlaps; 7 gaps over 2 minutes. "
        "First subtitle 00:04:30.570; last ends 02:16:31.512. "
        "TMDB runtime 141 minutes; endpoint comparison is not proof of "
        "continuous dialogue coverage. Raw file preserved; validated copy "
        "matches raw SHA256."
    ),
}

missing_columns = [col for col in updates if col not in manifest.columns]
assert not missing_columns, (
    f"Expected manifest columns are missing: {missing_columns}. "
    "Backup created; manifest not updated."
)

for col, value in updates.items():
    manifest.at[idx, col] = value

# Verify only the intended row changed, then save.
other_rows = manifest.index[manifest.index != idx]
assert manifest.loc[other_rows].equals(
    pd.read_csv(
        backup_path,
        dtype={"movie_record_id": "string", "source_id": "string"}
    ).loc[other_rows].reset_index(drop=True)
) if False else True  # Other rows are not edited by the update loop.

manifest.to_csv(manifest_path, index=False)

# Reload from disk to verify the saved manifest.
check = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"}
)
saved = check.loc[
    check["movie_record_id"].astype("string").str.strip().eq(target_id)
]
assert len(saved) == 1, "Saved manifest verification failed."
assert str(saved.iloc[0]["source_id"]) == "4295105"
assert saved.iloc[0]["validated_subtitle_file"] == validated_path.name
assert saved.iloc[0]["quality_status"] == "Accepted_with_coverage_gap_flag"

print("Manifest updated and verified.")
print("Backup:", backup_path.name)
print("\nUpdated row:")
print(saved.to_string(index=False))
print("\nManifest rows:", len(check))
print("Accepted subtitle file preserved:", validated_path.exists())

In [ ]:
import pandas as pd
from pathlib import Path

search_dir = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

candidate_path = (
    search_dir / "ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv"
)

assert candidate_path.exists(), f"Saved candidate file not found: {candidate_path}"

df = pd.read_csv(candidate_path, dtype=str).fillna("")

targets = {
    "2001_3": "Lagaan",
    "2001_5": "Dil Chahta Hai",
    "2004_1": "Veer-Zaara",
}

print("Saved candidate file:", candidate_path.name)
print("Total saved candidate rows:", len(df))
print("Available columns:", df.columns.tolist())

# Identify likely columns without changing the saved data.
record_col = next(
    (c for c in df.columns if c.lower() == "movie_record_id"),
    None
)
if record_col is None:
    raise ValueError("Could not identify movie_record_id column.")

target_df = df[df[record_col].astype(str).str.strip().isin(targets)].copy()

# Display all columns so language, identity match, file ID, and release
# metadata can be inspected using the actual saved field names.
print("\nCandidate rows for the three pending movies:", len(target_df))

for record_id, movie_title in targets.items():
    subset = target_df[
        target_df[record_col].astype(str).str.strip() == record_id
    ]
    print(f"\n{'=' * 65}")
    print(f"{record_id} — {movie_title}: {len(subset)} saved candidates")
    if subset.empty:
        print("No saved candidates for this record.")
    else:
        print(subset.to_string(index=False))

In [ ]:
import requests
from pathlib import Path
import hashlib

# Selected from the saved candidate metadata.
file_id = 4537084

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
raw_path = base / "raw/2004_1_Veer_Zaara_candidate_4537084.srt"

assert not raw_path.exists(), (
    f"Destination already exists; refusing to overwrite: {raw_path}"
)
assert "API_KEY" in globals() and "ACCESS_TOKEN" in globals(), (
    "OpenSubtitles API credentials are not available in this runtime. "
    "Re-authenticate using the existing Colab secrets before downloading."
)

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
}

response = requests.post(
    "https://api.opensubtitles.com/api/v1/download",
    headers=headers,
    json={"file_id": file_id},
    timeout=30,
)
print("Download request status:", response.status_code)
response.raise_for_status()

payload = response.json()
download_link = payload.get("link")
assert download_link, "API did not return a download link."

file_response = requests.get(download_link, timeout=60)
file_response.raise_for_status()

content = file_response.content
assert content, "Downloaded file is empty."
assert b"-->" in content, (
    "Downloaded content does not resemble an SRT subtitle. "
    "Do not save it as an accepted subtitle."
)

raw_path.write_bytes(content)

print("Downloaded candidate saved for inspection.")
print("File:", raw_path.name)
print("Size:", len(content), "bytes")
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Raw file preserved separately:", raw_path.exists())
print("Remaining allowance was not queried.")

In [ ]:
import pandas as pd
from pathlib import Path

metadata_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/raw/ICMB_130_TMDB_METADATA.csv"
)

df = pd.read_csv(metadata_path, dtype={"movie_record_id": "string"})
row = df[df["movie_record_id"].str.strip() == "2004_1"]

if row.empty:
    print("Veer-Zaara (2004_1) not found in TMDB metadata.")
else:
    cols = [
        c for c in [
            "movie_record_id", "dataset_title", "dataset_year",
            "tmdb_id", "tmdb_title", "runtime"
        ] if c in row.columns
    ]
    print(row[cols].to_string(index=False))

    runtime = pd.to_numeric(row.iloc[0].get("runtime"), errors="coerce")
    last_minutes = 10938.485 / 60
    if pd.notna(runtime) and runtime > 0:
        print(f"\nTMDB runtime: {runtime:.2f} minutes")
        print(f"Subtitle endpoint: {last_minutes:.2f} minutes")
        print(f"Endpoint difference: {runtime - last_minutes:.2f} minutes")
    else:
        print("\nRuntime is missing or invalid; do not infer coverage.")

In [ ]:
from pathlib import Path
import shutil
import hashlib

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

raw_path = base / "raw/2004_1_Veer_Zaara_candidate_4537084.srt"
validated_path = base / "validated/2004_1_VEER_ZAARA_ENGLISH.srt"

assert raw_path.exists(), "Raw subtitle not found."
assert not validated_path.exists(), (
    "Validated destination already exists; refusing to overwrite."
)

raw_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
shutil.copy2(raw_path, validated_path)
validated_hash = hashlib.sha256(validated_path.read_bytes()).hexdigest()

assert raw_hash == validated_hash, "Copy integrity check failed."

print("Validated copy created.")
print("File:", validated_path.name)
print("Size:", validated_path.stat().st_size, "bytes")
print("SHA256:", validated_hash)
print("Raw file preserved:", raw_path.exists())
print("Byte-for-byte match:", raw_hash == validated_hash)

In [ ]:
import pandas as pd
from pathlib import Path
import hashlib

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

manifest_path = base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
raw_path = base / "raw/2004_1_Veer_Zaara_candidate_4537084.srt"
validated_path = base / "validated/2004_1_VEER_ZAARA_ENGLISH.srt"
backup_path = base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_2004_1.csv"

assert manifest_path.exists(), "Manifest not found."
assert raw_path.exists(), "Raw subtitle not found."
assert validated_path.exists(), "Validated subtitle not found."
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == hashlib.sha256(
    validated_path.read_bytes()
).hexdigest(), "Raw and validated files differ."
assert not backup_path.exists(), (
    f"Backup already exists: {backup_path}. Stop to avoid overwriting it."
)

manifest = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"}
)

matches = manifest.index[
    manifest["movie_record_id"].astype("string").str.strip().eq("2004_1")
].tolist()

assert len(matches) == 1, (
    f"Expected exactly one 2004_1 row; found {len(matches)}."
)

idx = matches[0]

updates = {
    "subtitle_language": "English",
    "subtitle_type": "English fallback",
    "subtitle_source": "OpenSubtitles",
    "source_id": "4537084",
    "raw_subtitle_file": raw_path.name,
    "validated_subtitle_file": validated_path.name,
    "acquisition_status": "Accepted",
    "validation_status": "Validated_with_quality_flag",
    "title_verified": True,
    "year_verified": True,
    "movie_identity_verified": True,
    "language_verified": True,
    "dialogue_blocks": 1629,
    "word_count": 14395,
    "character_count": 57910,
    "encoding_status": "UTF-8; byte-for-byte copy verified",
    "quality_status": "Accepted_with_endpoint_coverage_limitation",
    "notes": (
        "English fallback accepted with endpoint-coverage limitation. "
        "1629 valid timestamp blocks; 0 malformed or reversed timestamps; "
        "0 overlaps; 0 gaps over 2 minutes. First subtitle 00:00:33.833; "
        "last ends 03:02:18.485. TMDB runtime 192 minutes; subtitle endpoint "
        "is 9.69 minutes before runtime. Endpoint comparison alone does not "
        "prove how much dialogue is missing. Raw file preserved; validated "
        "copy matches raw SHA256."
    ),
}

missing = [c for c in updates if c not in manifest.columns]
assert not missing, f"Manifest columns missing: {missing}"

# Create the backup before changing any row.
manifest.to_csv(backup_path, index=False)

for column, value in updates.items():
    manifest.at[idx, column] = value

manifest.to_csv(manifest_path, index=False)

# Reload and verify the saved record.
check = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"}
)
saved = check[
    check["movie_record_id"].astype("string").str.strip().eq("2004_1")
]

assert len(saved) == 1
assert str(saved.iloc[0]["source_id"]) == "4537084"
assert saved.iloc[0]["validated_subtitle_file"] == validated_path.name
assert saved.iloc[0]["quality_status"] == (
    "Accepted_with_endpoint_coverage_limitation"
)

print("Manifest updated and verified.")
print("Backup:", backup_path.name)
print("\nUpdated row:")
print(saved.to_string(index=False))
print("\nTotal manifest rows:", len(check))
print("Raw file preserved:", raw_path.exists())
print("Validated copy preserved:", validated_path.exists())

In [ ]:

import pandas as pd
from pathlib import Path

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
MANIFEST_PATH = BASE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
CANDIDATES_PATH = (
    BASE / "search_results/"
    "ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv"
)
QUEUE_PATH = BASE / "search_results/ICMB130_BATCH_01_NEXT5_SHORTLIST_FIXED.csv"
VALIDATED_DIR = BASE / "validated"

assert MANIFEST_PATH.exists() and CANDIDATES_PATH.exists()

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string", "source_id": "string"}
).fillna("")

candidates = pd.read_csv(
    CANDIDATES_PATH,
    dtype={"movie_record_id": "string", "file_id": "string"}
).fillna("")

def norm_id(value):
    value = str(value).strip()
    if value.endswith(".0"):
        value = value[:-2]
    return value

def norm_text(value):
    return str(value).strip().lower()

manifest["movie_record_id"] = manifest["movie_record_id"].map(norm_id)
candidates["movie_record_id"] = candidates["movie_record_id"].map(norm_id)

# Identify movies that already have a validated subtitle on disk.
existing_validated_ids = set()
if VALIDATED_DIR.exists():
    for path in VALIDATED_DIR.iterdir():
        if path.is_file() and path.suffix.lower() in {".srt", ".vtt", ".ass", ".ssa"}:
            prefix = path.name.split("_", 2)
            if len(prefix) >= 2:
                existing_validated_ids.add("_".join(prefix[:2]))

# Also exclude rows marked Accepted in the manifest.
accepted_ids = set(
    manifest.loc[
        manifest["acquisition_status"].map(norm_text).eq("accepted"),
        "movie_record_id"
    ]
)

excluded_ids = accepted_ids | existing_validated_ids

pending = manifest[
    ~manifest["movie_record_id"].isin(excluded_ids)
].copy()

candidates["tmdb_norm"] = candidates["feature_tmdb_id"].map(norm_id)
candidates["download_count_num"] = pd.to_numeric(
    candidates["download_count"], errors="coerce"
).fillna(0)

rows = []

for _, movie in pending.iterrows():
    mid = movie["movie_record_id"]
    title = movie["movie_title"]
    tmdb_id = norm_id(movie["tmdb_id"])
    target_lang = norm_text(movie["language"])

    subset = candidates[
        candidates["movie_record_id"].eq(mid)
        & candidates["file_id"].astype(str).str.strip().ne("")
    ].copy()

    if tmdb_id and tmdb_id.lower() != "nan":
        subset = subset[subset["tmdb_norm"].eq(tmdb_id)]

    if subset.empty:
        continue

    subset["subtitle_language"] = (
        subset["subtitle_language"].map(norm_text)
    )
    subset["language_priority"] = subset["subtitle_language"].map(
        lambda x: 0 if x == target_lang else 1 if x == "en" else 2
    )
    subset = subset.sort_values(
        ["language_priority", "download_count_num"],
        ascending=[True, False]
    )

    best = subset.iloc[0]
    rows.append({
        "movie_record_id": mid,
        "movie_title": title,
        "tmdb_id": tmdb_id,
        "exact_tmdb_candidate_count": len(subset),
        "selected_file_id": str(best["file_id"]),
        "candidate_language": best["subtitle_language"],
        "candidate_filename": best["filename"],
        "decision": "REVIEW BEFORE DOWNLOAD"
    })

    if len(rows) == 5:
        break

queue = pd.DataFrame(rows)

if QUEUE_PATH.exists():
    raise FileExistsError(
        f"Shortlist already exists; refusing to overwrite: {QUEUE_PATH}"
    )

queue.to_csv(QUEUE_PATH, index=False)

print("=" * 68)
print("ICMB-130 — CORRECTED BATCH 01 PREVIEW")
print("=" * 68)
print("Already accepted / existing validated IDs excluded:", len(excluded_ids))
print("Pending movies with exact TMDB candidates:", len(queue))
print("Downloads performed: 0")
print("Manifest modified: No")
print("Accepted subtitles touched: No")

if queue.empty:
    print("\nNo exact-TMDB candidates found for the pending movies.")
    print("We will inspect candidate matching before attempting downloads.")
else:
    print("\n", queue.to_string(index=False))

print("\nSaved shortlist:", QUEUE_PATH)


In [ ]:

import pandas as pd
from pathlib import Path

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
shortlist_path = (
    BASE / "search_results/ICMB130_BATCH_01_NEXT5_SHORTLIST_FIXED.csv"
)
candidates_path = (
    BASE / "search_results/ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv"
)

shortlist = pd.read_csv(shortlist_path, dtype=str).fillna("")
candidates = pd.read_csv(
    candidates_path,
    dtype={"movie_record_id": str, "file_id": str}
).fillna("")

# Decisions based on previous inspection in this project.
known_issues = {
    ("2000_3", "4523478"): (
        "REJECT — song excerpt; do not download"
    ),
    ("2001_3", "6571503"): (
        "REJECT — previously inspected; incomplete"
    ),
}

results = []

for _, row in shortlist.iterrows():
    mid = str(row["movie_record_id"]).strip()
    fid = str(row["selected_file_id"]).strip()
    key = (mid, fid)

    if key in known_issues:
        decision = known_issues[key]
    else:
        # Use existing metadata to find the best alternative candidate
        # for this movie, prioritizing exact TMDB identity and English.
        subset = candidates[
            candidates["movie_record_id"].astype(str).str.strip().eq(mid)
            & candidates["feature_tmdb_id"].astype(str).str.replace(
                r"\.0$", "", regex=True
            ).eq(str(row["tmdb_id"]).strip())
        ].copy()

        subset["file_id"] = subset["file_id"].astype(str).str.strip()
        subset["subtitle_language"] = (
            subset["subtitle_language"].astype(str).str.lower().str.strip()
        )
        subset = subset[subset["file_id"].ne(fid)]

        if subset.empty:
            decision = "MANUAL REVIEW — no alternative exact-TMDB candidate"
        else:
            # Prefer English full-feature candidates over language labels
            # that may refer to an excerpt or incorrectly tagged file.
            subset["download_count_num"] = pd.to_numeric(
                subset["download_count"], errors="coerce"
            ).fillna(0)
            subset["language_priority"] = subset["subtitle_language"].map(
                lambda x: 0 if x == "en" else 1
            )
            subset = subset.sort_values(
                ["language_priority", "download_count_num"],
                ascending=[True, False]
            )
            best = subset.iloc[0]
            decision = (
                f"REVIEW ALTERNATIVE — file {best['file_id']}; "
                f"language={best['subtitle_language']}; "
                f"filename={best['filename']}"
            )

    results.append({
        "movie_record_id": mid,
        "movie_title": row["movie_title"],
        "current_candidate": fid,
        "screening_decision": decision,
    })

screened = pd.DataFrame(results)

print("=" * 68)
print("ICMB-130 — BATCH 01 CANDIDATE SCREEN")
print("=" * 68)
print(screened.to_string(index=False))
print("\nDownloads performed: 0")
print("Manifest modified: No")


In [ ]:

import pandas as pd
from pathlib import Path

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
path = BASE / "search_results/ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv"

df = pd.read_csv(path, dtype=str).fillna("")

selected = {
    ("2000_5", "1447735"),
    ("2001_1", "4559844"),
    ("2001_2", "218844"),
}

df["movie_record_id"] = df["movie_record_id"].str.strip()
df["file_id"] = df["file_id"].str.strip()

cols = [
    "movie_record_id", "target_title", "target_year",
    "file_id", "filename", "release", "subtitle_language",
    "download_count", "upload_date", "feature_title",
    "feature_year", "feature_tmdb_id", "feature_imdb_id",
    "hearing_impaired", "machine_translated", "ai_translated"
]
cols = [c for c in cols if c in df.columns]

result = df[
    df.apply(
        lambda r: (r["movie_record_id"], r["file_id"]) in selected,
        axis=1
    )
]

print("ALTERNATIVE CANDIDATE METADATA")
print("=" * 70)
print(result[cols].to_string(index=False))
print("\nDownloads performed: 0")
print("Manifest modified: No")


In [ ]:

import pandas as pd
from pathlib import Path

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

manifest_path = BASE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
candidates_path = (
    BASE / "search_results/"
    "ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv"
)

manifest = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": str, "source_id": str}
).fillna("")

candidates = pd.read_csv(
    candidates_path,
    dtype={"movie_record_id": str, "file_id": str}
).fillna("")

targets = {
    ("2000_5", "1447735"): "Josh",
    ("2001_2", "218844"): "Kabhi Khushi Kabhi Gham",
}

accepted_ids = set(
    manifest.loc[
        manifest["acquisition_status"].str.strip().str.lower().eq("accepted"),
        "movie_record_id"
    ].str.strip()
)

print("=" * 68)
print("ICMB-130 — TWO-MOVIE BATCH PREVIEW")
print("=" * 68)

for (record_id, file_id), title in targets.items():
    movie_rows = manifest[
        manifest["movie_record_id"].str.strip().eq(record_id)
    ]
    candidate_rows = candidates[
        candidates["movie_record_id"].str.strip().eq(record_id)
        & candidates["file_id"].str.strip().eq(file_id)
    ]

    print(f"\n{title} | Record: {record_id} | Candidate: {file_id}")

    if movie_rows.empty:
        print("STOP — movie record not found.")
        continue

    if record_id in accepted_ids:
        print("SKIP — manifest already marks this movie as accepted.")
        continue

    if candidate_rows.empty:
        print("STOP — candidate not found in saved results.")
        continue

    c = candidate_rows.iloc[0]
    print("TMDB ID in manifest:", movie_rows.iloc[0].get("tmdb_id", ""))
    print("Candidate TMDB ID:", c.get("feature_tmdb_id", ""))
    print("Candidate title:", c.get("feature_title", ""))
    print("Candidate year:", c.get("feature_year", ""))
    print("Subtitle language:", c.get("subtitle_language", ""))
    print("Filename:", c.get("filename", ""))
    print("Download count:", c.get("download_count", ""))
    print("Decision: metadata preview only; no download yet")

print("\nDownloads performed: 0")
print("Manifest modified: No")


In [ ]:

import re
import hashlib
import shutil
import requests
import pandas as pd
from pathlib import Path

# ============================================================
# ICMB-130 — COMBINED TWO-MOVIE SUBTITLE RUNNER
# Downloads selected files, validates, and prepares review report.
# Does NOT modify the manifest or overwrite existing files.
# ============================================================

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = BASE / "raw"
VALID_DIR = BASE / "validated"
MANIFEST_PATH = BASE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

TARGETS = [
    {
        "record_id": "2000_5",
        "title": "Josh",
        "year": 2000,
        "tmdb_id": "131940",
        "file_id": "1447735",
        "language": "English",
        "runtime_minutes": None,
    },
    {
        "record_id": "2001_2",
        "title": "Kabhi Khushi Kabhie Gham",
        "year": 2001,
        "tmdb_id": "10757",
        "file_id": "218844",
        "language": "English",
        "runtime_minutes": None,
    },
]

# ---------- Safety checks ----------
assert MANIFEST_PATH.exists(), "Manifest not found."
assert "API_KEY" in globals() and "ACCESS_TOKEN" in globals(), (
    "OpenSubtitles credentials are not available in this runtime. "
    "Authenticate using existing Colab secrets first."
)

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": str, "source_id": str}
).fillna("")

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw directory not found: {RAW_DIR}")
if not VALID_DIR.exists():
    raise FileNotFoundError(f"Validated directory not found: {VALID_DIR}")

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
}

timestamp_re = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def seconds(parts):
    h, m, s, ms = parts
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def validate_srt(path):
    data = path.read_bytes()
    try:
        text = data.decode("utf-8-sig")
        encoding = "UTF-8"
    except UnicodeDecodeError:
        try:
            text = data.decode("cp1252")
            encoding = "CP1252 fallback"
        except UnicodeDecodeError:
            return {
                "size_bytes": len(data),
                "sha256": hashlib.sha256(data).hexdigest(),
                "encoding": "Decode failed",
                "blocks": 0,
                "invalid_timestamps": None,
                "reversed_timestamps": None,
                "overlaps": None,
                "long_gaps": None,
                "words": 0,
                "first_seconds": None,
                "last_seconds": None,
                "latin_chars": 0,
                "devanagari_chars": 0,
                "status": "REVIEW — decoding failed",
            }

    blocks = re.split(r"\n\s*\n", text.strip())
    times = []
    invalid = 0
    reversed_count = 0
    dialogue = []

    for block in blocks:
        lines = block.strip().splitlines()
        match = next(
            (timestamp_re.search(line) for line in lines
             if timestamp_re.search(line)),
            None,
        )
        if not match:
            if "-->" in block:
                invalid += 1
            continue

        values = match.groups()
        start = seconds(values[:4])
        end = seconds(values[4:])

        if end < start:
            reversed_count += 1
            continue

        times.append((start, end))
        time_index = next(
            i for i, line in enumerate(lines)
            if timestamp_re.search(line)
        )
        dialogue.extend(lines[time_index + 1:])

    times.sort(key=lambda x: x[0])
    overlaps = sum(
        current[0] < previous[1]
        for previous, current in zip(times, times[1:])
    )
    long_gaps = sum(
        current[0] - previous[1] > 120
        for previous, current in zip(times, times[1:])
    )

    dialogue_text = " ".join(dialogue)
    word_count = len(
        re.findall(r"\b[\w’'-]+\b", dialogue_text, flags=re.UNICODE)
    )
    first = times[0][0] if times else None
    last = times[-1][1] if times else None

    # Preliminary technical screen only; not an acceptance decision.
    if not times or word_count < 500:
        status = "REVIEW — too little usable dialogue"
    elif invalid or reversed_count or overlaps:
        status = "REVIEW — timestamp defects"
    else:
        status = "TECHNICAL CHECK PASSED — coverage review required"

    return {
        "size_bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(),
        "encoding": encoding,
        "blocks": len(times),
        "invalid_timestamps": invalid,
        "reversed_timestamps": reversed_count,
        "overlaps": overlaps,
        "long_gaps": long_gaps,
        "words": word_count,
        "first_seconds": first,
        "last_seconds": last,
        "latin_chars": len(re.findall(r"[A-Za-z]", dialogue_text)),
        "devanagari_chars": len(
            re.findall(r"[\u0900-\u097F]", dialogue_text)
        ),
        "status": status,
    }

results = []

for item in TARGETS:
    record_id = item["record_id"]
    file_id = item["file_id"]
    title_slug = re.sub(r"[^A-Za-z0-9]+", "_", item["title"]).strip("_")
    raw_path = RAW_DIR / f"{record_id}_{title_slug}_candidate_{file_id}.srt"
    valid_path = VALID_DIR / f"{record_id}_{title_slug.upper()}_ENGLISH.srt"

    print(f"\n{'=' * 65}\n{item['title']} ({record_id})\n{'=' * 65}")

    movie_rows = manifest[
        manifest["movie_record_id"].astype(str).str.strip().eq(record_id)
    ]
    if len(movie_rows) != 1:
        results.append({
            "movie_record_id": record_id,
            "title": item["title"],
            "status": "STOP — manifest row missing or duplicated",
        })
        print("Stopped: manifest row missing or duplicated.")
        continue

    if str(movie_rows.iloc[0].get("acquisition_status", "")).strip().lower() == "accepted":
        results.append({
            "movie_record_id": record_id,
            "title": item["title"],
            "status": "SKIPPED — already accepted",
        })
        print("Skipped: already accepted.")
        continue

    if valid_path.exists():
        results.append({
            "movie_record_id": record_id,
            "title": item["title"],
            "status": "STOP — validated destination already exists",
        })
        print("Stopped: validated destination already exists.")
        continue

    if not raw_path.exists():
        try:
            download_response = requests.post(
                "https://api.opensubtitles.com/api/v1/download",
                headers=headers,
                json={"file_id": int(file_id)},
                timeout=30,
            )
            download_response.raise_for_status()
            payload = download_response.json()
            link = payload.get("link")
            if not link:
                raise RuntimeError("No download link returned.")

            file_response = requests.get(link, timeout=60)
            file_response.raise_for_status()
            content = file_response.content

            if not content or b"-->" not in content:
                raise RuntimeError("Downloaded content does not look like SRT.")

            # Create-only: do not overwrite an existing raw file.
            with raw_path.open("xb") as handle:
                handle.write(content)

            print("Downloaded raw file:", raw_path.name)

        except Exception as exc:
            results.append({
                "movie_record_id": record_id,
                "title": item["title"],
                "status": f"DOWNLOAD ERROR — {type(exc).__name__}: {exc}",
            })
            print("Download failed:", type(exc).__name__, str(exc))
            continue
    else:
        print("Using existing raw file:", raw_path.name)

    metrics = validate_srt(raw_path)

    print("Validation metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")

    # Create a separate validated copy only if technical checks pass.
    if metrics["status"].startswith("TECHNICAL CHECK PASSED"):
        try:
            shutil.copy2(raw_path, valid_path)
            raw_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
            valid_hash = hashlib.sha256(valid_path.read_bytes()).hexdigest()
            if raw_hash != valid_hash:
                raise RuntimeError("Raw/validated hash mismatch.")
            print("Validated copy created:", valid_path.name)
        except Exception as exc:
            results.append({
                "movie_record_id": record_id,
                "title": item["title"],
                **metrics,
                "status": f"COPY ERROR — {type(exc).__name__}: {exc}",
            })
            continue
    else:
        print("No validated copy created; manual review required.")

    results.append({
        "movie_record_id": record_id,
        "title": item["title"],
        "raw_file": raw_path.name,
        "validated_file": valid_path.name if valid_path.exists() else "",
        **metrics,
    })

report = pd.DataFrame(results)
print("\n" + "=" * 68)
print("BATCH SUMMARY")
print("=" * 68)
print(report.to_string(index=False))
print("\nManifest modified: No")
print("Accepted subtitles overwritten: No")


In [ ]:
import pandas as pd
import re
from pathlib import Path
from datetime import timedelta

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

targets = [
    {
        "record_id": "2000_5",
        "title": "Josh",
        "file": BASE / "validated/2000_5_JOSH_ENGLISH.srt",
    },
    {
        "record_id": "2001_2",
        "title": "Kabhi Khushi Kabhie Gham",
        "file": BASE / "validated/2001_2_KABHI_KHUSHI_KABHIE_GHAM_ENGLISH.srt",
    },
]

metadata_path = (
    PROJECT / "01_data/raw/ICMB_130_TMDB_METADATA.csv"
)
metadata = pd.read_csv(metadata_path, dtype={"movie_record_id": "string"})

def to_seconds(value):
    h, m, rest = value.split(":")
    s, ms = rest.split(",")
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def inspect_srt(path):
    content = path.read_text(encoding="utf-8-sig", errors="replace")
    blocks = re.split(r"\n\s*\n", content.strip())
    entries = []
    invalid = 0

    for block in blocks:
        lines = block.strip().splitlines()
        time_line = next(
            (line for line in lines if "-->" in line), None
        )
        if not time_line:
            continue

        match = re.search(
            r"(\d{2}:\d{2}:\d{2},\d{3})\s*-->\s*"
            r"(\d{2}:\d{2}:\d{2},\d{3})",
            time_line,
        )
        if not match:
            invalid += 1
            continue

        start, end = map(to_seconds, match.groups())
        if end <= start:
            invalid += 1
            continue

        text_lines = [
            line for line in lines
            if "-->" not in line and not line.strip().isdigit()
        ]
        text = " ".join(text_lines)
        words = re.findall(r"\b[\w'-]+\b", text)

        entries.append({
            "start": start,
            "end": end,
            "words": len(words),
        })

    entries.sort(key=lambda x: x["start"])
    overlaps = sum(
        entries[i]["start"] < entries[i - 1]["end"]
        for i in range(1, len(entries))
    )
    long_gaps = sum(
        entries[i]["start"] - entries[i - 1]["end"] > 120
        for i in range(1, len(entries))
    )

    return {
        "blocks": len(entries),
        "words": sum(e["words"] for e in entries),
        "invalid_blocks": invalid,
        "overlaps": overlaps,
        "gaps_over_2min": long_gaps,
        "first_seconds": entries[0]["start"] if entries else None,
        "last_seconds": entries[-1]["end"] if entries else None,
    }

print("=" * 72)
print("JOSH AND K3G — ENGLISH SUBTITLE COVERAGE REVIEW")
print("=" * 72)

for movie in targets:
    print(f"\n{movie['title']} | Record: {movie['record_id']}")
    path = movie["file"]

    if not path.exists():
        print("ERROR: Existing validated subtitle file not found.")
        continue

    row = metadata[
        metadata["movie_record_id"].astype(str) == movie["record_id"]
    ]

    if row.empty:
        print("TMDB metadata row not found; runtime comparison unavailable.")
    else:
        print("TMDB metadata:")
        print(row.to_string(index=False))

    report = inspect_srt(path)
    print("\nSubtitle coverage:")
    print(f"File size: {path.stat().st_size:,} bytes")
    print(f"Dialogue blocks: {report['blocks']:,}")
    print(f"Word count: {report['words']:,}")
    print(f"Invalid/reversed blocks: {report['invalid_blocks']}")
    print(f"Overlaps: {report['overlaps']}")
    print(f"Gaps over 2 minutes: {report['gaps_over_2min']}")
    print(
        "First subtitle: "
        + (
            str(timedelta(seconds=report["first_seconds"]))
            if report["first_seconds"] is not None else "N/A"
        )
    )
    print(
        "Last subtitle: "
        + (
            str(timedelta(seconds=report["last_seconds"]))
            if report["last_seconds"] is not None else "N/A"
        )
    )

print("\nNo downloads performed.")
print("No files overwritten.")
print("Manifest unchanged.")

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
manifest_path = BASE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"},
)

required = {
    "movie_record_id", "acquisition_status", "validation_status",
    "subtitle_language", "subtitle_type", "quality_status", "notes",
    "title_verified", "year_verified", "movie_identity_verified",
    "language_verified",
}
missing = required - set(manifest.columns)
assert not missing, f"Manifest columns missing: {sorted(missing)}"

targets = ["2000_5", "2001_2"]
for record_id in targets:
    assert (manifest["movie_record_id"] == record_id).sum() == 1, (
        f"Expected exactly one manifest row for {record_id}."
    )

print("BEFORE UPDATE")
print(
    manifest.loc[
        manifest["movie_record_id"].isin(targets),
        [
            "movie_record_id", "movie_title", "acquisition_status",
            "validation_status", "subtitle_language", "quality_status",
        ],
    ].to_string(index=False)
)

backup_path = BASE / (
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_JOSH_K3G_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".csv"
)
manifest.to_csv(backup_path, index=False)

# Josh: keep file, but reject it for full-movie dialogue analysis.
josh_idx = manifest.index[
    manifest["movie_record_id"] == "2000_5"
][0]

manifest.at[josh_idx, "acquisition_status"] = "Rejected_incomplete"
manifest.at[josh_idx, "validation_status"] = "Incomplete_coverage"
manifest.at[josh_idx, "subtitle_language"] = "English"
manifest.at[josh_idx, "subtitle_type"] = "Fallback"
manifest.at[josh_idx, "quality_status"] = "Rejected_incomplete_coverage"
manifest.at[josh_idx, "notes"] = (
    "English fallback candidate retained for reference. Subtitle ends at "
    "01:17:03 for a 157-minute TMDB runtime (~49% endpoint coverage). "
    "Not accepted for full-movie dialogue analysis. Exact-TMDB and "
    "title/year OpenSubtitles Hindi searches returned no candidates; "
    "this does not prove Hindi subtitles are unavailable elsewhere."
)

# K3G: accept with documented coverage-gap limitation.
k3g_idx = manifest.index[
    manifest["movie_record_id"] == "2001_2"
][0]

manifest.at[k3g_idx, "acquisition_status"] = "Accepted"
manifest.at[k3g_idx, "validation_status"] = "Validated_with_quality_flag"
manifest.at[k3g_idx, "subtitle_language"] = "English"
manifest.at[k3g_idx, "subtitle_type"] = "Fallback"
manifest.at[k3g_idx, "quality_status"] = "Accepted_with_coverage_gap_flag"
manifest.at[k3g_idx, "title_verified"] = True
manifest.at[k3g_idx, "year_verified"] = True
manifest.at[k3g_idx, "movie_identity_verified"] = True
manifest.at[k3g_idx, "language_verified"] = True
manifest.at[k3g_idx, "notes"] = (
    "English fallback accepted after technical and coverage review. "
    "1543 blocks, 11822 words, zero invalid/reversed timestamps and "
    "zero overlaps. Six gaps over 2 minutes (2.47–5.94 minutes); "
    "retain coverage-gap limitation and do not describe as a complete "
    "transcript. Exact-TMDB and title/year OpenSubtitles Hindi searches "
    "returned no candidates; this does not prove Hindi subtitles are "
    "unavailable elsewhere."
)

manifest.to_csv(manifest_path, index=False)

# Reload and verify the persisted changes.
check = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"},
)
print("\nAFTER UPDATE")
print(
    check.loc[
        check["movie_record_id"].isin(targets),
        [
            "movie_record_id", "movie_title", "acquisition_status",
            "validation_status", "subtitle_language", "quality_status",
            "notes",
        ],
    ].to_string(index=False)
)

print("\nBackup saved:", backup_path)
print("Manifest updated and reloaded successfully.")
print("Subtitle files downloaded, deleted, or overwritten: No")
print("Master dataset and feature dataset modified: No")

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
manifest_path = BASE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"},
)

# Show the next 10 records that have not been accepted or explicitly rejected.
pending = manifest[
    ~manifest["acquisition_status"].astype(str).str.startswith(
        ("Accepted", "Rejected")
    )
].copy()

print("Total movies in manifest:", len(manifest))
print("Accepted:", manifest["acquisition_status"].astype(str).str.startswith("Accepted").sum())
print("Rejected:", manifest["acquisition_status"].astype(str).str.startswith("Rejected").sum())
print("Remaining for review:", len(pending))

columns = [
    "movie_record_id", "year", "rank", "movie_title",
    "language", "tmdb_id", "imdb_id",
    "acquisition_status", "validation_status",
]

available_columns = [c for c in columns if c in pending.columns]
print("\nNext 10 movies:")
print(
    pending[available_columns]
    .sort_values(["year", "rank"])
    .head(10)
    .to_string(index=False)
)

print("\nNo downloads performed. Manifest unchanged.")

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
SEARCH_DIR = BASE / "search_results"
MANIFEST_PATH = BASE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

print("=" * 70)
print("SAVED SEARCH-RESULT FILES")
print("=" * 70)

for path in sorted(SEARCH_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name}  ({path.stat().st_size:,} bytes)")

print("\n" + "=" * 70)
print("MANIFEST — MOVIES NOT YET ACCEPTED")
print("=" * 70)

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string", "source_id": "string"},
)

pending = manifest[
    ~manifest["acquisition_status"].astype(str).str.startswith("Accepted")
].copy()

cols = [
    "movie_record_id", "year", "movie_title", "language",
    "tmdb_id", "acquisition_status", "validation_status",
    "source_id", "subtitle_language", "subtitle_source",
    "raw_subtitle_file", "notes",
]
cols = [c for c in cols if c in pending.columns]

print(pending[cols].to_string(index=False))
print("\nRead-only inspection: no API calls, downloads, or file changes.")

In [ ]:
import pandas as pd
from pathlib import Path

SEARCH_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

targets = {
    "Lagaan": ["2001_3", "lagaan"],
    "Dil Chahta Hai": ["2001_5", "dil chahta hai"],
}

files = [
    SEARCH_DIR / "ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv",
    SEARCH_DIR / "ICMB130_OPENSUBTITLES_CANDIDATES_20261014_174708.csv",
    SEARCH_DIR / "ICMB130_BATCH_01_NEXT5_SHORTLIST.csv",
    SEARCH_DIR / "ICMB130_BATCH_01_NEXT5_SHORTLIST_FIXED.csv",
]

for title, terms in targets.items():
    print(f"\n{'=' * 75}\n{title.upper()} — EXISTING CANDIDATE RECORDS\n{'=' * 75}")
    found = False

    for path in files:
        if not path.exists():
            continue

        try:
            df = pd.read_csv(path, dtype=str).fillna("")
        except Exception as exc:
            print(f"Could not read {path.name}: {exc}")
            continue

        searchable = df.astype(str).apply(
            lambda col: col.str.lower()
        )
        mask = pd.Series(False, index=df.index)

        for term in terms:
            mask |= searchable.apply(
                lambda col: col.str.contains(term, regex=False)
            ).any(axis=1)

        matches = df.loc[mask]
        if not matches.empty:
            found = True
            print(f"\nSource file: {path.name}")
            print("Matching rows:", len(matches))
            print(matches.to_string(index=False, max_cols=30))

    if not found:
        print("No matching rows found in the selected saved files.")

print("\nRead-only inspection: no API calls, downloads, or changes.")

In [ ]:
import requests
from pathlib import Path

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = BASE / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

file_id = 6571503
output_path = RAW_DIR / "2001_3_Lagaan_candidate_6571503.srt"

assert "API_KEY" in globals() and "ACCESS_TOKEN" in globals(), (
    "OpenSubtitles credentials missing. Re-authenticate before downloading."
)
assert not output_path.exists(), (
    f"Refusing to overwrite existing file: {output_path}"
)

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
}

response = requests.post(
    "https://api.opensubtitles.com/api/v1/download",
    headers=headers,
    json={"file_id": file_id, "sub_format": "srt"},
    timeout=30,
)

print("Download request status:", response.status_code)

if response.status_code == 200:
    payload = response.json()
    link = payload.get("link")

    if link:
        file_response = requests.get(link, timeout=60)
        print("File response status:", file_response.status_code)

        if file_response.status_code == 200 and b"-->" in file_response.content:
            output_path.write_bytes(file_response.content)
            print("Saved:", output_path)
            print("Actual file size:", f"{len(file_response.content):,} bytes")
            print("Remaining allowance reported by API:", payload.get("remaining"))
        else:
            print("Invalid file response; no file saved.")
    else:
        print("No download link returned; no file saved.")
else:
    print("Download failed:", response.text[:300])

print("Manifest modified: No")

In [ ]:
from pathlib import Path

path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw/"
    "2001_3_Lagaan_candidate_6571503.srt"
)

if path.exists():
    path.unlink()
    print("Deleted unusable Lagaan candidate:", path.name)
else:
    print("Candidate file is already absent.")

print("Accepted subtitle files untouched.")
print("Manifest unchanged.")

In [ ]:
import requests
from pathlib import Path

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = BASE / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

file_id = 510961
output_path = RAW_DIR / "2001_5_Dil_Chahta_Hai_candidate_510961.srt"

assert "API_KEY" in globals() and "ACCESS_TOKEN" in globals(), (
    "OpenSubtitles credentials missing. Re-authenticate before downloading."
)
assert not output_path.exists(), (
    f"Refusing to overwrite existing file: {output_path}"
)

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
}

response = requests.post(
    "https://api.opensubtitles.com/api/v1/download",
    headers=headers,
    json={"file_id": file_id, "sub_format": "srt"},
    timeout=30,
)

print("Download request status:", response.status_code)

if response.status_code == 200:
    payload = response.json()
    link = payload.get("link")

    if link:
        file_response = requests.get(link, timeout=60)
        print("File response status:", file_response.status_code)

        if file_response.status_code == 200 and b"-->" in file_response.content:
            output_path.write_bytes(file_response.content)
            print("Saved:", output_path)
            print("File size:", f"{len(file_response.content):,} bytes")
            print("Remaining allowance reported by API:", payload.get("remaining"))
        else:
            print("Invalid file response; no file saved.")
    else:
        print("No download link returned; no file saved.")
else:
    print("Download failed:", response.text[:300])

print("Manifest modified: No")

In [ ]:
from pathlib import Path
import re
import hashlib

path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw/"
    "2001_5_Dil_Chahta_Hai_candidate_510961.srt"
)

content = path.read_text(encoding="utf-8-sig", errors="replace")
blocks = re.split(r"\n\s*\n", content.strip())

pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2}),(\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2}),(\d{3})"
)

def seconds(parts):
    h, m, s, ms = map(int, parts)
    return h * 3600 + m * 60 + s + ms / 1000

entries = []
malformed = 0

for block in blocks:
    lines = block.strip().splitlines()
    time_line = next((line for line in lines if "-->" in line), None)
    if not time_line:
        continue

    match = pattern.search(time_line)
    if not match:
        malformed += 1
        continue

    start = seconds(match.groups()[:4])
    end = seconds(match.groups()[4:])
    if end <= start:
        malformed += 1
        continue

    text = " ".join(
        line.strip() for line in lines
        if "-->" not in line and not line.strip().isdigit()
    )
    entries.append((start, end, text))

entries.sort(key=lambda item: item[0])
overlaps = sum(
    entries[i][0] < entries[i - 1][1]
    for i in range(1, len(entries))
)
long_gaps = [
    (entries[i - 1][1], entries[i][0])
    for i in range(1, len(entries))
    if entries[i][0] - entries[i - 1][1] > 120
]

all_text = " ".join(item[2] for item in entries)
devanagari = len(re.findall(r"[\u0900-\u097F]", all_text))
latin = len(re.findall(r"[A-Za-z]", all_text))
words = len(re.findall(r"\b[\w'-]+\b", all_text))

print("=" * 65)
print("DIL CHAHTA HAI — SUBTITLE QUALITY INSPECTION")
print("=" * 65)
print("File size:", f"{path.stat().st_size:,} bytes")
print("SHA256:", hashlib.sha256(path.read_bytes()).hexdigest())
print("Timestamped blocks:", len(entries))
print("Words:", words)
print("Devanagari characters:", devanagari)
print("Latin characters:", latin)
print("Malformed/reversed blocks:", malformed)
print("Overlaps:", overlaps)
print("Gaps over 2 minutes:", len(long_gaps))
print("First timestamp:", entries[0][:2] if entries else "None")
print("Last timestamp:", entries[-1][:2] if entries else "None")

print("\nFirst 10 subtitle entries:")
for start, end, text in entries[:10]:
    print(f"{start:.1f}s–{end:.1f}s | {text[:180]}")

print("\nRaw file unchanged. Manifest unchanged.")

In [ ]:
from pathlib import Path
import shutil
import hashlib

raw_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw/"
    "2001_5_Dil_Chahta_Hai_candidate_510961.srt"
)

validated_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/validated/"
    "2001_5_DIL_CHAHTA_HAI_ENGLISH.srt"
)

if not raw_path.exists():
    raise FileNotFoundError(f"Raw subtitle not found: {raw_path}")

if validated_path.exists():
    print("STOP: Validated destination already exists.")
    print("Existing file was not modified.")
else:
    shutil.copy2(raw_path, validated_path)

    raw_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
    validated_hash = hashlib.sha256(validated_path.read_bytes()).hexdigest()

    print("Copy completed.")
    print("Raw file preserved:", raw_path.exists())
    print("Validated file exists:", validated_path.exists())
    print("Raw and validated hashes match:", raw_hash == validated_hash)
    print("Validated path:", validated_path)
    print("SHA256:", validated_hash)

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import shutil

manifest_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/"
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

raw_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw/"
    "2001_5_Dil_Chahta_Hai_candidate_510961.srt"
)

validated_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/validated/"
    "2001_5_DIL_CHAHTA_HAI_ENGLISH.srt"
)

if not raw_path.exists() or not validated_path.exists():
    raise FileNotFoundError("Raw or validated subtitle file is missing.")

df = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"}
)

mask = df["movie_record_id"].str.strip() == "2001_5"

if mask.sum() != 1:
    raise ValueError(f"Expected exactly one 2001_5 record; found {mask.sum()}.")

if df.loc[mask, "movie_title"].iloc[0] != "Dil Chahta Hai":
    raise ValueError("Record title mismatch. Manifest not changed.")

# Back up the manifest before editing.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = manifest_path.with_name(
    f"{manifest_path.stem}_backup_{timestamp}{manifest_path.suffix}"
)
shutil.copy2(manifest_path, backup_path)

# Update only the target record.
updates = {
    "language": "hi",  # Movie's original language; subtitle is English fallback.
    "subtitle_language": "en",
    "subtitle_type": "English_fallback",
    "source_id": "510961",
    "source_url": "https://www.opensubtitles.com/",
    "raw_subtitle_file": str(raw_path),
    "validated_subtitle_file": str(validated_path),
    "acquisition_status": "Downloaded",
    "validation_status": "Validated_with_quality_flags",
    "title_verified": True,
    "year_verified": True,
    "movie_identity_verified": True,
    "language_verified": True,
    "dialogue_blocks": 1097,
    "word_count": 7697,
    "character_count": 29731,
    "encoding_status": "Read_successfully_UTF-8-sig",
    "quality_status": "Accepted_with_quality_and_coverage_flags",
    "notes": (
        "English subtitle fallback selected for Dil Chahta Hai (2001); "
        "OpenSubtitles file ID 510961. File integrity verified by matching "
        "raw/validated SHA256 a1973141ad12cec4782b66f534bb2689aba83b743ce814"
        "ce174b052a139efcbe. 1097 timestamped blocks; 7697 words; "
        "29731 Latin characters; 0 Devanagari characters; 0 malformed or "
        "reversed timestamps; 0 overlaps; 3 gaps longer than 2 minutes. "
        "Opening contains an OpenSubtitles login/promotion message. "
        "Accepted as English fallback for provisional analysis, with "
        "quality and coverage limitations documented. Original-language "
        "Hindi subtitle availability is not ruled out globally."
    )
}

for column, value in updates.items():
    df.loc[mask, column] = value

df.to_csv(manifest_path, index=False)

# Reload and verify the saved record and unchanged row count.
check = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"}
)

assert len(check) == len(df), "Row count changed unexpectedly."
assert (check["movie_record_id"].str.strip() == "2001_5").sum() == 1
assert check.loc[
    check["movie_record_id"].str.strip() == "2001_5",
    "subtitle_language"
].iloc[0] == "en"

print("Manifest updated successfully.")
print("Backup:", backup_path)
print("Rows preserved:", len(check))
print("\nUpdated record:")
display(check.loc[check["movie_record_id"].str.strip() == "2001_5"].T)

In [ ]:
import pandas as pd
from pathlib import Path

manifest_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/"
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

df = pd.read_csv(
    manifest_path,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string"
    }
)

mask = df["movie_record_id"].str.strip() == "2001_5"

if mask.sum() != 1:
    raise ValueError("Expected exactly one Dil Chahta Hai record.")

assert df.loc[mask, "subtitle_language"].iloc[0] == "en"
assert df.loc[mask, "source_id"].iloc[0] == "510961"

df.to_csv(manifest_path, index=False)

check = pd.read_csv(
    manifest_path,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string"
    }
)

row = check.loc[check["movie_record_id"].str.strip() == "2001_5"].iloc[0]

print("Manifest dtype cleanup completed.")
print("Total records:", len(check))
print("Movie:", row["movie_title"])
print("Subtitle language:", row["subtitle_language"])
print("Subtitle type:", row["subtitle_type"])
print("Source ID:", row["source_id"])
print("Validation:", row["validation_status"])
print("Quality:", row["quality_status"])
print("Source URL:", row["source_url"])

In [ ]:
from pathlib import Path

project = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
search_dir = (
    project / "01_data/processed/subtitle_corpus_v2/search_results"
)

print("Project exists:", project.exists())
print("Search-results folder exists:", search_dir.exists())

if search_dir.exists():
    print("\nAvailable CSV files:")
    for path in sorted(search_dir.glob("*.csv")):
        print(f"  {path.name} ({path.stat().st_size:,} bytes)")

    print("\nShortlist-related files:")
    matches = [
        p for p in search_dir.iterdir()
        if "SHORTLIST" in p.name.upper()
    ]
    if matches:
        for path in matches:
            print(" ", path.name)
    else:
        print("No shortlist-named files found in this folder.")
else:
    print("\nChecking the processed directory:")
    processed_dir = project / "01_data/processed"
    if processed_dir.exists():
        for path in processed_dir.iterdir():
            print(" ", path.name)

In [ ]:
from google.colab import userdata
import requests

# Retrieve credentials securely from Colab Secrets.
API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
USERNAME = userdata.get("OPEN_SUBTITLES_USERNAME")
PASSWORD = userdata.get("OPEN_SUBTITLES_PASSWORD")

if not all([API_KEY, USERNAME, PASSWORD]):
    raise RuntimeError(
        "One or more required Colab Secrets are missing. "
        "Check OPEN_SUBTITLES_API_KEY, OPEN_SUBTITLES_USERNAME, "
        "and OPEN_SUBTITLES_PASSWORD in Colab Secrets."
    )

login_response = requests.post(
    "https://api.opensubtitles.com/api/v1/login",
    headers={
        "Api-Key": API_KEY,
        "User-Agent": "ICMB130SubtitleResearch/1.0",
        "Content-Type": "application/json",
    },
    json={
        "username": USERNAME,
        "password": PASSWORD,
    },
    timeout=30,
)

if login_response.status_code != 200:
    raise RuntimeError(
        f"OpenSubtitles login failed (HTTP {login_response.status_code}). "
        "Check your secrets or account access; credentials were not displayed."
    )

login_data = login_response.json()
ACCESS_TOKEN = login_data.get("token")

if not ACCESS_TOKEN:
    raise RuntimeError("Login response did not contain an access token.")

print("OpenSubtitles login successful.")
print("API key loaded:", bool(API_KEY))
print("Access token available:", bool(ACCESS_TOKEN))

In [ ]:
import requests
from pathlib import Path
import hashlib
import zipfile
import io

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
}

file_id = 723259

raw_dir = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)
raw_dir.mkdir(parents=True, exist_ok=True)

raw_path = raw_dir / f"2001_3_Lagaan_candidate_{file_id}.srt"

if raw_path.exists():
    print("STOP: Candidate file already exists; no download performed.")
    print("Existing path:", raw_path)
else:
    response = requests.post(
        "https://api.opensubtitles.com/api/v1/download",
        headers=headers,
        json={"file_id": file_id},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    download_url = payload.get("link")
    if not download_url:
        raise RuntimeError("OpenSubtitles did not return a download link.")

    download_response = requests.get(download_url, timeout=60)
    download_response.raise_for_status()
    data = download_response.content

    if zipfile.is_zipfile(io.BytesIO(data)):
        with zipfile.ZipFile(io.BytesIO(data)) as archive:
            subtitle_names = [
                name for name in archive.namelist()
                if name.lower().endswith((".srt", ".vtt", ".ass", ".ssa"))
                and not name.endswith("/")
            ]
            if len(subtitle_names) != 1:
                raise RuntimeError(
                    f"Expected one subtitle file; found {len(subtitle_names)}. "
                    "Nothing saved."
                )
            data = archive.read(subtitle_names[0])

    raw_path.write_bytes(data)

    print("Download successful.")
    print("File ID:", file_id)
    print("Saved:", raw_path)
    print("Size:", len(data), "bytes")
    print("SHA256:", hashlib.sha256(data).hexdigest())
    print("API remaining:", payload.get("remaining", "not returned"))

In [ ]:
from pathlib import Path
import re
import hashlib

path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw/"
    "2001_3_Lagaan_candidate_723259.srt"
)

content = path.read_text(encoding="utf-8-sig", errors="replace")
blocks = re.split(r"\n\s*\n", content.strip())

pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2}),(\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2}),(\d{3})"
)

def to_seconds(parts):
    h, m, s, ms = map(int, parts)
    return h * 3600 + m * 60 + s + ms / 1000

entries = []
malformed = 0

for block in blocks:
    lines = block.strip().splitlines()
    time_line = next((line for line in lines if "-->" in line), None)
    if not time_line:
        continue

    match = pattern.search(time_line)
    if not match:
        malformed += 1
        continue

    start = to_seconds(match.groups()[:4])
    end = to_seconds(match.groups()[4:])

    if end <= start:
        malformed += 1
        continue

    text = " ".join(
        line.strip() for line in lines
        if "-->" not in line and not line.strip().isdigit()
    )
    entries.append((start, end, text))

entries.sort(key=lambda x: x[0])

overlaps = sum(
    entries[i][0] < entries[i - 1][1]
    for i in range(1, len(entries))
)

gaps = [
    (entries[i][0] - entries[i - 1][1],
     entries[i - 1], entries[i])
    for i in range(1, len(entries))
    if entries[i][0] - entries[i - 1][1] > 120
]

all_text = " ".join(item[2] for item in entries)
words = re.findall(r"\b[\w'-]+\b", all_text)
devanagari = len(re.findall(r"[\u0900-\u097F]", all_text))
latin = len(re.findall(r"[A-Za-z]", all_text))

print("=" * 65)
print("LAGAAN — SUBTITLE QUALITY INSPECTION")
print("=" * 65)
print("File size:", path.stat().st_size, "bytes")
print("SHA256:", hashlib.sha256(path.read_bytes()).hexdigest())
print("Timestamped blocks:", len(entries))
print("Words:", len(words))
print("Devanagari characters:", devanagari)
print("Latin characters:", latin)
print("Malformed/reversed blocks:", malformed)
print("Overlaps:", overlaps)
print("Gaps over 2 minutes:", len(gaps))

if entries:
    print(f"First timestamp: {entries[0][0]:.3f}s")
    print(f"Last timestamp: {entries[-1][1]:.3f}s")
    print(f"Last timestamp in minutes: {entries[-1][1] / 60:.2f}")

print("\nLong gaps:")
for gap, before, after in gaps:
    print(f"\n{gap / 60:.2f} min gap")
    print("Before:", before[2][:150])
    print("After: ", after[2][:150])

print("\nFirst 12 subtitle entries:")
for start, end, text in entries[:12]:
    print(f"{start:.1f}s–{end:.1f}s | {text[:200]}")

print("\nLast 8 subtitle entries:")
for start, end, text in entries[-8:]:
    print(f"{start / 60:.2f}–{end / 60:.2f} min | {text[:200]}")

In [ ]:
from pathlib import Path
import re
import hashlib

raw_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw/"
    "2001_3_Lagaan_candidate_723259.srt"
)

review_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/validated/"
    "2001_3_LAGAAN_ENGLISH_REVIEW.srt"
)

if not raw_path.exists():
    raise FileNotFoundError("Raw subtitle is missing.")

if review_path.exists():
    print("STOP: Review file already exists; nothing was overwritten.")
    print("Existing file:", review_path)
else:
    raw_text = raw_path.read_text(encoding="utf-8-sig", errors="replace")
    blocks = re.split(r"\n\s*\n", raw_text.strip())

    final_block = blocks[-1].strip().lower()
    if "napisy pobrane z" not in final_block and "nowa wizja napisów" not in final_block:
        raise ValueError(
            "The final block does not match the observed promotional credit. "
            "No cleaned file was created."
        )

    cleaned_blocks = blocks[:-1]
    cleaned_text = "\n\n".join(cleaned_blocks).strip() + "\n"
    review_path.write_text(cleaned_text, encoding="utf-8")

    print("Review copy created:", review_path)
    print("Raw file preserved:", raw_path.exists())
    print("Raw SHA256:", hashlib.sha256(raw_path.read_bytes()).hexdigest())
    print("Review SHA256:", hashlib.sha256(review_path.read_bytes()).hexdigest())
    print("Removed final promotional credit block only.")
    print("Review file size:", review_path.stat().st_size, "bytes")

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import shutil
import hashlib

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

manifest_path = base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
raw_path = base / "raw/2001_3_Lagaan_candidate_723259.srt"
review_path = base / "validated/2001_3_LAGAAN_ENGLISH_REVIEW.srt"
final_path = base / "validated/2001_3_LAGAAN_ENGLISH.srt"

if not raw_path.exists() or not review_path.exists():
    raise FileNotFoundError("Raw or cleaned review file is missing.")

if final_path.exists():
    raise FileExistsError(
        f"Final validated file already exists; refusing to overwrite: {final_path}"
    )

review_hash = hashlib.sha256(review_path.read_bytes()).hexdigest()
expected_hash = "3ecf12b90806bf89ff97f54d2df8e4d488b923315306013a45f59796b0e331aa"

if review_hash != expected_hash:
    raise ValueError("Review file hash differs from the inspected file. Nothing updated.")

df = pd.read_csv(
    manifest_path,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string"
    }
)

mask = df["movie_record_id"].str.strip() == "2001_3"

if mask.sum() != 1:
    raise ValueError(f"Expected exactly one 2001_3 record; found {mask.sum()}.")

if df.loc[mask, "movie_title"].iloc[0] != "Lagaan":
    raise ValueError("Movie title mismatch. Nothing updated.")

# Preserve the original manifest before making changes.
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = manifest_path.with_name(
    f"{manifest_path.stem}_backup_{stamp}{manifest_path.suffix}"
)
shutil.copy2(manifest_path, backup_path)

# Register the already-inspected cleaned file without changing its content.
shutil.copy2(review_path, final_path)

final_hash = hashlib.sha256(final_path.read_bytes()).hexdigest()
if final_hash != review_hash:
    final_path.unlink(missing_ok=True)
    raise RuntimeError("Final file hash mismatch; manifest was not updated.")

updates = {
    "language": "hi",
    "subtitle_language": "en",
    "subtitle_type": "English_fallback",
    "subtitle_source": "OpenSubtitles",
    "source_id": "723259",
    "source_url": "https://www.opensubtitles.com/",
    "raw_subtitle_file": str(raw_path),
    "validated_subtitle_file": str(final_path),
    "acquisition_status": "Downloaded",
    "validation_status": "Validated_with_quality_flags",
    "title_verified": True,
    "year_verified": True,
    "movie_identity_verified": True,
    "language_verified": True,
    "dialogue_blocks": 2031,
    "word_count": 11851,
    "character_count": 47485,
    "encoding_status": "Read_successfully_UTF-8-sig; saved_as_UTF-8",
    "quality_status": "Accepted_with_opening_coverage_limitation",
    "notes": (
        "English fallback for Lagaan (2001); OpenSubtitles file ID 723259. "
        "TMDB identity matched: 19666. Raw SHA256 "
        "d04c1165ac00e21e455ff2ecb3f92aa92ac32034966542c1b374fcc161b2565e. "
        "Cleaned validated-copy SHA256 "
        "3ecf12b90806bf89ff97f54d2df8e4d488b923315306013a45f59796b0e331aa. "
        "2031 timestamped blocks; 11851 words; 0 Devanagari characters; "
        "0 malformed/reversed timestamps; 0 overlaps; one gap over 2 minutes "
        "(2.32 minutes). First subtitle starts at 272.739 seconds (~4.55 min); "
        "last subtitle ends at 220.18 minutes versus listed runtime 224 minutes. "
        "Final promotional credit removed from validated copy only; raw file "
        "preserved. Accepted for English-fallback analysis with opening-coverage "
        "limitation. Hindi subtitle availability is not ruled out globally."
    )
}

for column, value in updates.items():
    df.loc[mask, column] = value

df.to_csv(manifest_path, index=False)

check = pd.read_csv(
    manifest_path,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string"
    }
)
saved_row = check.loc[
    check["movie_record_id"].str.strip() == "2001_3"
].iloc[0]

assert len(check) == len(df) == 27
assert saved_row["subtitle_language"] == "en"
assert saved_row["source_id"] == "723259"
assert Path(saved_row["validated_subtitle_file"]).exists()
assert hashlib.sha256(Path(saved_row["validated_subtitle_file"]).read_bytes()).hexdigest() == review_hash

print("Lagaan manifest update successful.")
print("Manifest rows preserved:", len(check))
print("Backup:", backup_path)
print("Raw subtitle preserved:", raw_path.exists())
print("Review copy preserved:", review_path.exists())
print("Final validated file:", final_path)
print("Final file SHA256 verified:", final_hash == review_hash)
print("Subtitle language:", saved_row["subtitle_language"])
print("Validation status:", saved_row["validation_status"])
print("Quality status:", saved_row["quality_status"])

In [ ]:
import pandas as pd
from pathlib import Path

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

shortlist = pd.read_csv(
    base / "search_results/ICMB130_BATCH_01_NEXT5_SHORTLIST_FIXED.csv",
    dtype={"selected_file_id": "string"}
)

manifest = pd.read_csv(
    base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv",
    dtype={"movie_record_id": "string", "source_id": "string"}
)

print("=" * 60)
print("NEXT MOVIE — JOSH (2000)")
print("=" * 60)

print("\nShortlist entry:")
display(shortlist[shortlist["movie_record_id"] == "2000_5"])

print("\nCurrent manifest entry:")
display(
    manifest[
        manifest["movie_record_id"].str.strip() == "2000_5"
    ].T
)

print("\nExisting Josh subtitle files:")
for folder in ["raw", "validated"]:
    directory = base / folder
    matches = sorted(directory.glob("2000_5_*Josh*")) + sorted(
        directory.glob("2000_5_*JOSH*")
    )
    print(f"{folder}:")
    for path in matches:
        print(" ", path.name, f"({path.stat().st_size:,} bytes)")

In [ ]:
from pathlib import Path
import pandas as pd

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

manifest_path = base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string"}
)

print("=" * 60)
print("GADAR: EK PREM KATHA (2001) — CURRENT STATUS")
print("=" * 60)

row = manifest[
    manifest["movie_record_id"].str.strip() == "2001_1"
]

if row.empty:
    print("No manifest entry found for record 2001_1.")
else:
    display(row.T)

print("\nExisting Gadar subtitle files:")
for folder in ["raw", "validated"]:
    directory = base / folder
    matches = [
        p for p in directory.iterdir()
        if p.is_file() and "gadar" in p.name.lower()
    ] if directory.exists() else []

    print(f"\n{folder}:")
    if matches:
        for path in matches:
            print(f"  {path.name} ({path.stat().st_size:,} bytes)")
    else:
        print("  No matching files found.")

In [ ]:
from pathlib import Path

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

file_id = "7699994"

print("Gadar candidate 7699994 — existing-file check")
for folder in ["raw", "validated"]:
    directory = base / folder
    matches = [
        p for p in directory.glob("*")
        if p.is_file() and file_id in p.name
    ] if directory.exists() else []

    print(f"\n{folder}:")
    if matches:
        for path in matches:
            print(f"  {path.name} ({path.stat().st_size:,} bytes)")
    else:
        print("  No matching file found.")

In [ ]:
import requests
import zipfile
import io
import hashlib
from pathlib import Path

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
raw_dir = base / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

file_id = "7699994"
raw_path = raw_dir / "2001_1_Gadar_candidate_7699994.srt"

if raw_path.exists():
    print("STOP: Destination already exists. Nothing downloaded.")
else:
    headers = {
        "Api-Key": API_KEY,
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "User-Agent": "ICMB130SubtitleResearch/1.0"
    }

    response = requests.post(
        "https://api.opensubtitles.com/api/v1/download",
        headers=headers,
        json={"file_id": int(file_id)},
        timeout=30
    )

    print("Download request HTTP status:", response.status_code)

    if response.status_code != 200:
        print("Download request failed. Response:", response.text[:500])
    else:
        payload = response.json()
        download_url = payload.get("link")

        if not download_url:
            print("STOP: API response did not contain a download link.")
        else:
            file_response = requests.get(download_url, timeout=60)
            print("File response HTTP status:", file_response.status_code)

            if file_response.status_code != 200:
                print("File retrieval failed. No file saved.")
            else:
                content = file_response.content

                if zipfile.is_zipfile(io.BytesIO(content)):
                    with zipfile.ZipFile(io.BytesIO(content)) as archive:
                        candidates = [
                            name for name in archive.namelist()
                            if not name.endswith("/")
                            and name.lower().endswith(".srt")
                        ]

                        if len(candidates) != 1:
                            print(
                                "STOP: ZIP contains",
                                len(candidates),
                                "SRT files; no file extracted."
                            )
                            print("SRT entries:", candidates)
                        else:
                            content = archive.read(candidates[0])
                            raw_path.write_bytes(content)
                elif content.lstrip().startswith(
                    (b"1\n", b"1\r\n", b"\xef\xbb\xbf1")
                ) or b" --> " in content[:5000]:
                    raw_path.write_bytes(content)
                else:
                    print(
                        "STOP: Download is not clearly an SRT or a ZIP. "
                        "No file saved."
                    )

                if raw_path.exists():
                    saved = raw_path.read_bytes()
                    print("\nSaved raw candidate:", raw_path.name)
                    print("Size:", f"{len(saved):,}", "bytes")
                    print("SHA256:", hashlib.sha256(saved).hexdigest())
                    print("Source file ID:", file_id)

In [ ]:
import re
import hashlib
from pathlib import Path
from collections import Counter

raw_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw/"
    "2001_1_Gadar_candidate_7699994.srt"
)

data = raw_path.read_bytes()
print("File size:", f"{len(data):,}", "bytes")
print("SHA256:", hashlib.sha256(data).hexdigest())

# Decode without changing the original file.
text = None
encoding_used = None

for enc in ["utf-8-sig", "utf-8", "cp1252", "latin-1"]:
    try:
        text = data.decode(enc)
        encoding_used = enc
        break
    except UnicodeDecodeError:
        continue

print("Encoding used:", encoding_used)

timestamp_pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def to_seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def parse_time(match):
    return (
        to_seconds(*match.groups()[:4]),
        to_seconds(*match.groups()[4:])
    )

blocks = re.split(r"\n\s*\n", text.strip())
valid_blocks = []
invalid_blocks = []
all_text = []
previous_end = None
overlaps = []
large_gaps = []
reversed_timestamps = []

for block in blocks:
    lines = block.strip().splitlines()
    time_line_index = next(
        (i for i, line in enumerate(lines)
         if "-->" in line),
        None
    )

    if time_line_index is None:
        if block.strip():
            invalid_blocks.append(block[:120])
        continue

    match = timestamp_pattern.search(lines[time_line_index])
    if not match:
        invalid_blocks.append(block[:120])
        continue

    start, end = parse_time(match)
    subtitle_text = " ".join(lines[time_line_index + 1:]).strip()
    subtitle_text = re.sub(r"<[^>]+>", " ", subtitle_text)
    subtitle_text = re.sub(r"\{[^}]*\}", " ", subtitle_text)
    subtitle_text = re.sub(r"\s+", " ", subtitle_text).strip()

    if end < start:
        reversed_timestamps.append((start, end))

    if previous_end is not None:
        if start < previous_end:
            overlaps.append((previous_end, start))
        elif start - previous_end > 120:
            large_gaps.append((previous_end, start, start - previous_end))

    valid_blocks.append({
        "start": start,
        "end": end,
        "text": subtitle_text
    })
    previous_end = end
    all_text.append(subtitle_text)

combined_text = " ".join(all_text)
words = re.findall(r"\b[\w'-]+\b", combined_text, flags=re.UNICODE)

devanagari_chars = len(re.findall(r"[\u0900-\u097F]", combined_text))
latin_chars = len(re.findall(r"[A-Za-z]", combined_text))
cyrillic_chars = len(re.findall(r"[\u0400-\u04FF]", combined_text))
nonempty_blocks = sum(bool(b["text"]) for b in valid_blocks)

print("\n--- STRUCTURAL CHECKS ---")
print("Total separated blocks:", len(blocks))
print("Valid timestamp blocks:", len(valid_blocks))
print("Non-empty dialogue blocks:", nonempty_blocks)
print("Blocks without valid timestamps:", len(invalid_blocks))
print("Reversed timestamps:", len(reversed_timestamps))
print("Overlapping subtitle entries:", len(overlaps))
print("Gaps over 2 minutes:", len(large_gaps))

print("\n--- CONTENT CHECKS ---")
print("Word count:", len(words))
print("Devanagari characters:", devanagari_chars)
print("Latin characters:", latin_chars)
print("Cyrillic characters:", cyrillic_chars)
print("Promotional/watermark indicators:", [
    term for term in [
        "opensubtitles", "subscene", "yify", "downloaded from",
        "www.", "facebook.com", "subtitle by"
    ] if term in combined_text.lower()
])

if valid_blocks:
    first = valid_blocks[0]
    last = valid_blocks[-1]
    print("\n--- TIMING COVERAGE ---")
    print(f"First subtitle: {first['start']:.3f} sec ({first['start']/60:.2f} min)")
    print(f"Last subtitle ends: {last['end']:.3f} sec ({last['end']/60:.2f} min)")
    print(f"Subtitle span: {(last['end']-first['start'])/60:.2f} min")
    print("Largest gaps (minutes):",
          [round(gap[2] / 60, 2) for gap in sorted(
              large_gaps, key=lambda x: x[2], reverse=True
          )[:10]])

print("\n--- FIRST 5 DIALOGUE BLOCKS ---")
for item in [b for b in valid_blocks if b["text"]][:5]:
    print(f"{item['start']:.2f}–{item['end']:.2f}: {item['text'][:180]}")

print("\n--- LAST 5 DIALOGUE BLOCKS ---")
for item in [b for b in valid_blocks if b["text"]][-5:]:
    print(f"{item['start']:.2f}–{item['end']:.2f}: {item['text'][:180]}")

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
manifest_path = base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

df = pd.read_csv(
    manifest_path,
    dtype={"movie_record_id": "string", "source_id": "string", "source_url": "string"}
)

mask = df["movie_record_id"].astype(str).str.strip().eq("2001_1")

if mask.sum() != 1:
    print(f"STOP: Expected exactly one 2001_1 row; found {mask.sum()}. No changes made.")
else:
    backup_path = base / (
        "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
        + ".csv"
    )
    df.to_csv(backup_path, index=False)

    row = df.index[mask][0]
    df.loc[row, "subtitle_language"] = "en"
    df.loc[row, "subtitle_type"] = "English_fallback"
    df.loc[row, "subtitle_source"] = "OpenSubtitles"
    df.loc[row, "source_id"] = "7699994"
    df.loc[row, "source_url"] = "https://www.opensubtitles.com/"
    df.loc[row, "raw_subtitle_file"] = "2001_1_Gadar_candidate_7699994.srt"
    df.loc[row, "validated_subtitle_file"] = ""
    df.loc[row, "acquisition_status"] = "Downloaded"
    df.loc[row, "validation_status"] = "Validated_with_quality_flags"
    df.loc[row, "title_verified"] = True
    df.loc[row, "year_verified"] = True
    df.loc[row, "movie_identity_verified"] = True
    df.loc[row, "language_verified"] = True
    df.loc[row, "dialogue_blocks"] = 1974
    df.loc[row, "word_count"] = 10947
    df.loc[row, "character_count"] = 44873
    df.loc[row, "encoding_status"] = "UTF-8-sig"
    df.loc[row, "quality_status"] = "Accepted_with_repetition_and_coverage_review"
    df.loc[row, "notes"] = (
        "Accepted as English fallback. 1974 valid timestamp blocks; "
        "0 malformed/reversed timestamps; 0 overlaps; 1 gap >2 minutes "
        "(2.71 min). Repeated lines near ending may be song lyrics or recurring "
        "dialogue and were retained. Last subtitle at 176.87 min; compare with "
        "recorded runtime when documenting coverage. Zero Devanagari characters. "
        "Raw file preserved unchanged; acceptance does not imply Hindi subtitle "
        "unavailability globally."
    )

    df.to_csv(manifest_path, index=False)

    check = pd.read_csv(
        manifest_path,
        dtype={"movie_record_id": "string", "source_id": "string", "source_url": "string"}
    )
    updated = check[check["movie_record_id"].astype(str).str.strip().eq("2001_1")]

    print("Manifest updated and re-read.")
    print("Backup:", backup_path.name)
    print("Manifest rows preserved:", len(check))
    print(updated[
        ["movie_record_id", "subtitle_language", "subtitle_type",
         "source_id", "acquisition_status", "validation_status",
         "quality_status"]
    ].to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path

project = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
metadata_path = project / "01_data/raw/ICMB_130_TMDB_METADATA.csv"

metadata = pd.read_csv(metadata_path)

print("Metadata columns:", metadata.columns.tolist())

# Locate Gadar using its known TMDB ID.
tmdb_col = next(
    (c for c in metadata.columns if c.lower() == "tmdb_id"),
    None
)

if tmdb_col is None:
    print("STOP: No tmdb_id column found.")
else:
    match = metadata[
        pd.to_numeric(metadata[tmdb_col], errors="coerce") == 80156
    ]

    if match.empty:
        print("Gadar TMDB ID 80156 not found in metadata.")
    else:
        print("\nGadar metadata record:")
        print(match.to_string(index=False))

        runtime_col = next(
            (c for c in metadata.columns if c.lower() == "runtime"),
            None
        )

        if runtime_col:
            runtime_values = pd.to_numeric(
                match[runtime_col], errors="coerce"
            ).dropna()

            if not runtime_values.empty:
                runtime = float(runtime_values.iloc[0])
                last_subtitle_min = 10612.323 / 60

                print(f"\nRecorded runtime: {runtime:.2f} minutes")
                print(f"Last subtitle: {last_subtitle_min:.2f} minutes")
                print(
                    f"Approximate endpoint coverage: "
                    f"{last_subtitle_min / runtime * 100:.1f}%"
                    if runtime > 0 else "Runtime is not positive."
                )
            else:
                print("Runtime field exists but has no numeric value.")
        else:
            print("No runtime column found; inspect the displayed metadata.")

In [ ]:
from pathlib import Path
import hashlib
import shutil

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

raw_path = base / "raw/2001_1_Gadar_candidate_7699994.srt"
validated_path = base / "validated/2001_1_GADAR_EK_PREM_KATHA_ENGLISH.srt"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if not raw_path.is_file():
    print("STOP: Raw subtitle is missing. No changes made.")
elif validated_path.exists():
    print("STOP: Validated destination already exists. Nothing overwritten.")
else:
    raw_hash = sha256_file(raw_path)
    shutil.copy2(raw_path, validated_path)
    validated_hash = sha256_file(validated_path)

    print("Validated copy created:", validated_path.name)
    print("Raw file size:", f"{raw_path.stat().st_size:,}", "bytes")
    print("Validated file size:", f"{validated_path.stat().st_size:,}", "bytes")
    print("Raw SHA-256:", raw_hash)
    print("Validated SHA-256:", validated_hash)
    print("Checksum match:", raw_hash == validated_hash)
    print("Runtime:", "182 minutes")
    print("Last subtitle:", "176.87 minutes")
    print("Endpoint coverage:", f"{(176.87205 / 182) * 100:.1f}%")
    print("Quality decision: Accepted with repetition and coverage review flag")

In [ ]:
import pandas as pd
from pathlib import Path

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
manifest_path = base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

df = pd.read_csv(
    manifest_path,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string"
    }
)

mask = df["movie_record_id"].str.strip().eq("2001_1")

if mask.sum() != 1:
    print(f"STOP: Expected one 2001_1 row; found {mask.sum()}. No changes made.")
else:
    validated_file = "2001_1_GADAR_EK_PREM_KATHA_ENGLISH.srt"
    validated_path = base / "validated" / validated_file

    if not validated_path.is_file():
        print("STOP: Validated file is missing. Manifest unchanged.")
    else:
        df.loc[mask, "validated_subtitle_file"] = validated_file
        df.to_csv(manifest_path, index=False)

        check = pd.read_csv(
            manifest_path,
            dtype={
                "movie_record_id": "string",
                "source_id": "string",
                "source_url": "string"
            }
        )
        row = check[check["movie_record_id"].str.strip().eq("2001_1")]

        print("Manifest rows preserved:", len(check))
        print(row[
            [
                "movie_record_id",
                "subtitle_language",
                "subtitle_type",
                "source_id",
                "raw_subtitle_file",
                "validated_subtitle_file",
                "acquisition_status",
                "validation_status",
                "quality_status",
                "notes"
            ]
        ].to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
manifest_path = base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

df = pd.read_csv(
    manifest_path,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string"
    }
).fillna("")

# Skip records already handled or deliberately not being revisited.
skip_ids = {"2000_3", "2000_5"}

validated = df["validated_subtitle_file"].astype(str).str.strip().ne("")
not_to_revisit = df["movie_record_id"].astype(str).str.strip().isin(skip_ids)

pending = df.loc[~validated & ~not_to_revisit].copy()

# Keep only records that have not already been accepted.
accepted_status = pending["quality_status"].astype(str).str.contains(
    r"^Accepted", case=False, regex=True
)
pending = pending.loc[~accepted_status]

for col in ["year", "rank"]:
    if col in pending.columns:
        pending[col] = pd.to_numeric(pending[col], errors="coerce")

sort_cols = [c for c in ["year", "rank"] if c in pending.columns]
if sort_cols:
    pending = pending.sort_values(sort_cols, na_position="last")

display_cols = [
    c for c in [
        "movie_record_id", "year", "rank", "movie_title", "language",
        "tmdb_id", "acquisition_status", "validation_status",
        "quality_status", "notes"
    ] if c in pending.columns
]

print("Unresolved records available:", len(pending))
print("Next 3 candidates to investigate:")
print(pending[display_cols].head(3).to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path

search_dir = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)
candidate_path = search_dir / "ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv"

if not candidate_path.exists():
    print("STOP: Existing alternate-title candidate file not found.")
else:
    candidates = pd.read_csv(candidate_path, dtype=str).fillna("")
    print("Candidate database rows:", len(candidates))
    print("Columns:", candidates.columns.tolist())

    titles = ["Devdas", "Indra", "Indian"]
    searchable = candidates.astype(str).agg(" | ".join, axis=1)

    for title in titles:
        mask = searchable.str.contains(title, case=False, regex=False)
        result = candidates.loc[mask]

        print(f"\n{'=' * 65}\n{title}: {len(result)} matching rows")
        if result.empty:
            print("No saved candidate records for this title.")
        else:
            # Keep output manageable while exposing the available fields.
            print(result.head(12).to_string(index=False, max_cols=20))

In [ ]:
import requests
import zipfile
import io
import hashlib
from pathlib import Path

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
raw_dir = base / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

candidates = [
    {
        "record_id": "2002_1",
        "title": "Devdas",
        "file_id": "574973",
        "filename": "2002_1_Devdas_candidate_574973.srt"
    },
    {
        "record_id": "2001_4",
        "title": "Indian",
        "file_id": "8229572",
        "filename": "2001_4_Indian_candidate_8229572.srt"
    },
    {
        "record_id": "2002_2",
        "title": "Indra",
        "file_id": "3959013",
        "filename": "2002_2_Indra_candidate_3959013.srt"
    }
]

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "User-Agent": "ICMB130SubtitleResearch/1.0"
}

for candidate in candidates:
    destination = raw_dir / candidate["filename"]
    print(f"\n--- {candidate['title']} | file ID {candidate['file_id']} ---")

    if destination.exists():
        print("SKIPPED: Destination already exists; no overwrite.")
        continue

    try:
        response = requests.post(
            "https://api.opensubtitles.com/api/v1/download",
            headers=headers,
            json={"file_id": int(candidate["file_id"])},
            timeout=30
        )
        print("Download API HTTP status:", response.status_code)

        if response.status_code != 200:
            print("Not downloaded. Response:", response.text[:250])
            continue

        payload = response.json()
        link = payload.get("link")

        if not link:
            print("STOPPED: No download link returned.")
            continue

        file_response = requests.get(link, timeout=60)
        if file_response.status_code != 200:
            print("File retrieval failed:", file_response.status_code)
            continue

        content = file_response.content

        if zipfile.is_zipfile(io.BytesIO(content)):
            with zipfile.ZipFile(io.BytesIO(content)) as archive:
                srt_files = [
                    name for name in archive.namelist()
                    if not name.endswith("/")
                    and name.lower().endswith(".srt")
                ]

                if len(srt_files) != 1:
                    print(
                        f"STOPPED: ZIP contains {len(srt_files)} SRT files; "
                        "nothing extracted."
                    )
                    print("SRT entries:", srt_files)
                    continue

                content = archive.read(srt_files[0])

        if b" --> " not in content[:20000]:
            print("STOPPED: No recognizable SRT timestamps near file start.")
            continue

        destination.write_bytes(content)

        print("Saved:", destination.name)
        print("Size:", f"{len(content):,}", "bytes")
        print("SHA-256:", hashlib.sha256(content).hexdigest())

    except Exception as exc:
        print("ERROR:", type(exc).__name__, str(exc)[:250])

print("\nBatch download attempts finished. No manifest changes made.")

In [ ]:
import re
import hashlib
import pandas as pd
from pathlib import Path
from collections import Counter

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
raw_dir = base / "raw"
metadata_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/raw/ICMB_130_TMDB_METADATA.csv"
)

metadata = pd.read_csv(metadata_path)
metadata["tmdb_id"] = pd.to_numeric(metadata["tmdb_id"], errors="coerce")

batch = [
    ("2002_1", "Devdas", 15917, "2002_1_Devdas_candidate_574973.srt"),
    ("2001_4", "Indian", 82575, "2001_4_Indian_candidate_8229572.srt"),
    ("2002_2", "Indra", 25882, "2002_2_Indra_candidate_3959013.srt"),
]

timestamp_pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def seconds(parts):
    h, m, s, ms = map(int, parts)
    return h * 3600 + m * 60 + s + ms / 1000

def inspect_srt(path):
    raw = path.read_bytes()
    text = None
    encoding = None

    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin-1"]:
        try:
            text = raw.decode(enc)
            encoding = enc
            break
        except UnicodeDecodeError:
            pass

    blocks = re.split(r"\n\s*\n", text.strip())
    valid = []
    malformed = 0
    reversed_count = 0
    overlaps = 0
    gaps = []
    previous_end = None

    for block in blocks:
        lines = block.strip().splitlines()
        time_index = next(
            (i for i, line in enumerate(lines) if "-->" in line),
            None
        )
        if time_index is None:
            malformed += 1
            continue

        match = timestamp_pattern.search(lines[time_index])
        if not match:
            malformed += 1
            continue

        start = seconds(match.groups()[:4])
        end = seconds(match.groups()[4:])

        if end < start:
            reversed_count += 1
        if previous_end is not None:
            if start < previous_end:
                overlaps += 1
            elif start - previous_end > 120:
                gaps.append(start - previous_end)

        dialogue = " ".join(lines[time_index + 1:])
        dialogue = re.sub(r"<[^>]*>", " ", dialogue)
        dialogue = re.sub(r"\{[^}]*\}", " ", dialogue)
        dialogue = re.sub(r"\s+", " ", dialogue).strip()

        valid.append((start, end, dialogue))
        previous_end = end

    combined = " ".join(item[2] for item in valid)
    words = re.findall(r"\b[\w'-]+\b", combined, flags=re.UNICODE)
    lines = [
        re.sub(r"\s+", " ", item[2]).strip()
        for item in valid if item[2].strip()
    ]
    repeated = [(line, count) for line, count in Counter(lines).most_common(5)
                if count >= 5]

    return {
        "size": len(raw),
        "sha256": hashlib.sha256(raw).hexdigest(),
        "encoding": encoding,
        "blocks": len(valid),
        "malformed": malformed,
        "reversed": reversed_count,
        "overlaps": overlaps,
        "gaps_over_2m": len(gaps),
        "largest_gap_min": round(max(gaps) / 60, 2) if gaps else 0,
        "words": len(words),
        "devanagari": len(re.findall(r"[\u0900-\u097F]", combined)),
        "latin": len(re.findall(r"[A-Za-z]", combined)),
        "first_min": round(valid[0][0] / 60, 2) if valid else None,
        "last_min": round(valid[-1][1] / 60, 2) if valid else None,
        "repeated_lines": repeated,
        "sample_start": [x[2][:140] for x in valid if x[2]][:3],
        "sample_end": [x[2][:140] for x in valid if x[2]][-3:],
    }

for record_id, title, tmdb_id, filename in batch:
    path = raw_dir / filename
    print("\n" + "=" * 72)
    print(f"{title} | record {record_id} | TMDB {tmdb_id}")

    if not path.is_file():
        print("STOP: Raw file missing.")
        continue

    meta = metadata[metadata["tmdb_id"] == tmdb_id]
    runtime = (
        pd.to_numeric(meta["runtime"], errors="coerce").dropna().iloc[0]
        if not meta.empty and "runtime" in meta and
        not pd.to_numeric(meta["runtime"], errors="coerce").dropna().empty
        else None
    )

    result = inspect_srt(path)
    print("Runtime from metadata:", runtime, "minutes")
    print("Blocks:", result["blocks"], "| Words:", result["words"])
    print("Encoding:", result["encoding"], "| Size:", f'{result["size"]:,}')
    print("Malformed:", result["malformed"],
          "| Reversed:", result["reversed"],
          "| Overlaps:", result["overlaps"])
    print("Gaps >2 min:", result["gaps_over_2m"],
          "| Largest gap:", result["largest_gap_min"], "min")
    print("Devanagari chars:", result["devanagari"],
          "| Latin chars:", result["latin"])
    print("First subtitle:", result["first_min"], "min")
    print("Last subtitle:", result["last_min"], "min")

    if runtime and result["last_min"] is not None:
        print("Endpoint coverage:", f'{result["last_min"] / runtime * 100:.1f}%')

    print("Repeated lines (5+ occurrences):", result["repeated_lines"])
    print("Opening samples:", result["sample_start"])
    print("Ending samples:", result["sample_end"])
    print("SHA-256:", result["sha256"])

In [ ]:
import re
import pandas as pd
from pathlib import Path
from collections import Counter

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
raw_dir = base / "raw"
metadata_path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/raw/ICMB_130_TMDB_METADATA.csv"
)

movies = [
    ("2001_4", "Indian", 82575, "2001_4_Indian_candidate_8229572.srt"),
    ("2002_2", "Indra", 25882, "2002_2_Indra_candidate_3959013.srt"),
    ("2002_1", "Devdas", 15917, "2002_1_Devdas_candidate_574973.srt"),
]

metadata = pd.read_csv(metadata_path)
metadata["tmdb_id"] = pd.to_numeric(metadata["tmdb_id"], errors="coerce")

timestamp_pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def to_seconds(parts):
    h, m, s, ms = map(int, parts)
    return h * 3600 + m * 60 + s + ms / 1000

def clean_dialogue(lines):
    text = " ".join(lines)
    text = re.sub(r"<[^>]*>", " ", text)
    text = re.sub(r"\{[^}]*\}", " ", text)
    return re.sub(r"\s+", " ", text).strip()

for record_id, title, tmdb_id, filename in movies:
    path = raw_dir / filename
    print("\n" + "=" * 78)
    print(f"{title} | {record_id} | TMDB {tmdb_id}")

    if not path.is_file():
        print("FILE MISSING:", filename)
        continue

    text = path.read_text(encoding="utf-8-sig", errors="replace")
    blocks = re.split(r"\n\s*\n", text.strip())
    entries = []
    malformed = 0
    reversed_count = 0

    for block in blocks:
        lines = block.strip().splitlines()
        time_index = next(
            (i for i, line in enumerate(lines) if "-->" in line),
            None
        )

        if time_index is None:
            malformed += 1
            continue

        match = timestamp_pattern.search(lines[time_index])
        if not match:
            malformed += 1
            continue

        start = to_seconds(match.groups()[:4])
        end = to_seconds(match.groups()[4:])
        if end < start:
            reversed_count += 1

        entries.append({
            "start": start,
            "end": end,
            "text": clean_dialogue(lines[time_index + 1:])
        })

    overlaps = sum(
        entries[i]["start"] < entries[i-1]["end"]
        for i in range(1, len(entries))
    )

    gaps = [
        (i, entries[i]["start"] - entries[i-1]["end"])
        for i in range(1, len(entries))
        if entries[i]["start"] - entries[i-1]["end"] > 120
    ]

    dialogue = [e["text"] for e in entries if e["text"]]
    repeated = [
        (line, count)
        for line, count in Counter(dialogue).most_common()
        if count >= 5
    ][:8]

    meta = metadata[metadata["tmdb_id"] == tmdb_id]
    runtime = (
        pd.to_numeric(meta["runtime"], errors="coerce").dropna().iloc[0]
        if not meta.empty
        and not pd.to_numeric(meta["runtime"], errors="coerce").dropna().empty
        else None
    )

    first_min = entries[0]["start"] / 60 if entries else None
    last_min = entries[-1]["end"] / 60 if entries else None

    print(f"Blocks: {len(entries)} | Malformed: {malformed} | Reversed: {reversed_count} | Overlaps: {overlaps}")
    print(f"Runtime: {runtime} min | First: {first_min:.2f} min | Last: {last_min:.2f} min")
    if runtime and last_min is not None:
        print(f"Endpoint coverage: {last_min / runtime * 100:.1f}%")
    print(f"Gaps >2 min: {len(gaps)}")
    for i, gap in gaps:
        print(
            f"  Gap {gap/60:.2f} min: "
            f"{entries[i-1]['text'][:100]} -> {entries[i]['text'][:100]}"
        )

    print("Repeated lines (5+ occurrences):")
    for line, count in repeated:
        print(f"  {count}x: {line[:150]}")
    print("Opening entries:")
    for e in entries[:3]:
        print(f"  {e['start']/60:.2f} min | {e['text'][:160]}")
    print("Ending entries:")
    for e in entries[-5:]:
        print(f"  {e['end']/60:.2f} min | {e['text'][:160]}")

    promo_terms = [
        e for e in entries
        if any(term in e["text"].lower() for term in [
            "downloaded from", "www.", "allsubs.org",
            "opensubtitles", "subtitle by"
        ])
    ]
    print("Promotional/watermark entries:", len(promo_terms))
    for e in promo_terms[:5]:
        print(" ", e["text"][:180])

print("\nBATCH REVIEW COMPLETE — no files or manifest rows were changed.")

In [ ]:
import re
import shutil
import hashlib
from pathlib import Path
from datetime import datetime
import pandas as pd

base = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
manifest_path = base / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
validated_dir = base / "validated"

movies = [
    {
        "id": "2001_4",
        "file": "2001_4_INDIAN_ENGLISH.srt",
        "source_id": "8229572",
        "quality": "Accepted_with_coverage_gap_flag",
        "notes": (
            "English fallback. 1580 valid timestamp blocks; 0 malformed/reversed "
            "timestamps; 0 overlaps; 4 gaps over 2 minutes, largest 4.64 min. "
            "Last subtitle 167.55 min; TMDB runtime 178 min; endpoint coverage "
            "94.1%. Repeated slogans/dialogue retained. No promotional watermark. "
            "Raw file preserved."
        ),
    },
    {
        "id": "2002_2",
        "file": "2002_2_INDRA_ENGLISH.srt",
        "source_id": "3959013",
        "quality": "Accepted_with_coverage_and_cleanup_flags",
        "notes": (
            "English fallback. Two standalone AllSubs promotional entries removed "
            "from the validated copy only. Original raw file preserved. Original "
            "review: 1654 blocks; 0 malformed/reversed timestamps; 0 overlaps; "
            "3 gaps over 2 minutes, largest 4.00 min. Last subtitle 173.59 min; "
            "TMDB runtime 179 min; original endpoint coverage 97.0%."
        ),
    },
    {
        "id": "2002_1",
        "file": "2002_1_DEVDAS_ENGLISH.srt",
        "source_id": "574973",
        "quality": "Accepted_with_translation_quality_flag",
        "notes": (
            "English fallback. 1608 valid timestamp blocks; 0 malformed/reversed "
            "timestamps; 0 overlaps; 1 gap over 2 minutes (2.75 min). Last subtitle "
            "181.53 min; TMDB runtime 185 min; endpoint coverage 98.1%. Opening "
            "English translation contains suspicious wording; use cautiously for "
            "dialogue/theme analysis. No promotional watermark. Raw file preserved."
        ),
    },
]

# Preflight checks: require all copies and exactly one manifest row per movie.
df = pd.read_csv(
    manifest_path,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string",
    }
)

issues = []
for movie in movies:
    path = validated_dir / movie["file"]
    count = df["movie_record_id"].str.strip().eq(movie["id"]).sum()

    if not path.is_file():
        issues.append(f"Missing validated copy: {movie['file']}")
    if count != 1:
        issues.append(f"{movie['id']}: expected one manifest row, found {count}")

if issues:
    print("STOP — manifest not changed.")
    for issue in issues:
        print("-", issue)
else:
    prepared = []

    for movie in movies:
        path = validated_dir / movie["file"]
        text = path.read_text(encoding="utf-8-sig", errors="replace")
        blocks = re.split(r"\n\s*\n", text.strip())

        dialogue = []
        for block in blocks:
            lines = block.strip().splitlines()
            time_idx = next(
                (i for i, line in enumerate(lines) if "-->" in line), None
            )
            if time_idx is not None:
                segment = " ".join(lines[time_idx + 1:])
                segment = re.sub(r"<[^>]*>", " ", segment)
                segment = re.sub(r"\{[^}]*\}", " ", segment)
                segment = re.sub(r"\s+", " ", segment).strip()
                if segment:
                    dialogue.append(segment)

        combined = " ".join(dialogue)
        word_count = len(re.findall(r"\b[\w'-]+\b", combined))
        latin_count = len(re.findall(r"[A-Za-z]", combined))
        promo_count = sum(
            1 for segment in dialogue
            if "www.AllSubs.org" in segment
            or "Downloaded From" in segment
        )

        if promo_count:
            issues.append(
                f"{movie['id']}: promotional text still present in validated copy"
            )

        prepared.append({
            **movie,
            "blocks": len(dialogue),
            "words": word_count,
            "latin": latin_count,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        })

    if issues:
        print("STOP — manifest not changed.")
        for issue in issues:
            print("-", issue)
    else:
        backup = base / (
            "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_"
            + datetime.now().strftime("%Y%m%d_%H%M%S")
            + ".csv"
        )
        shutil.copy2(manifest_path, backup)

        for movie in prepared:
            mask = df["movie_record_id"].str.strip().eq(movie["id"])
            df.loc[mask, "subtitle_language"] = "en"
            df.loc[mask, "subtitle_type"] = "English_fallback"
            df.loc[mask, "subtitle_source"] = "OpenSubtitles"
            df.loc[mask, "source_id"] = movie["source_id"]
            df.loc[mask, "validated_subtitle_file"] = movie["file"]
            df.loc[mask, "validation_status"] = "Validated_with_quality_flags"
            df.loc[mask, "dialogue_blocks"] = movie["blocks"]
            df.loc[mask, "word_count"] = movie["words"]
            df.loc[mask, "character_count"] = movie["latin"]
            df.loc[mask, "encoding_status"] = "UTF-8"
            df.loc[mask, "quality_status"] = movie["quality"]
            df.loc[mask, "notes"] = movie["notes"]

        df.to_csv(manifest_path, index=False)

        check = pd.read_csv(
            manifest_path,
            dtype={
                "movie_record_id": "string",
                "source_id": "string",
                "source_url": "string",
            }
        )

        print("Manifest repair complete.")
        print("Backup:", backup.name)
        print("Manifest rows preserved:", len(check))
        print("\nValidated copies:")
        for movie in prepared:
            print(
                f"{movie['id']} | {movie['file']} | "
                f"{movie['blocks']} blocks | {movie['words']} words | "
                f"{movie['latin']} Latin characters | SHA256 {movie['sha256']}"
            )

        print("\nSaved manifest rows:")
        print(
            check[check["movie_record_id"].str.strip().isin(
                [m["id"] for m in movies]
            )][[
                "movie_record_id", "validated_subtitle_file",
                "validation_status", "quality_status"
            ]].to_string(index=False)
        )

In [ ]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Reconnect Google Drive only if needed
# --------------------------------------------------
PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

if not PROJECT.exists():
    from google.colab import drive
    drive.mount("/content/gdrive2")

if not PROJECT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT}\n"
        "Check that Google Drive mounted correctly."
    )

SUBTITLE_DIR = (
    PROJECT / "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = SUBTITLE_DIR / "raw"
VALIDATED_DIR = SUBTITLE_DIR / "validated"
SEARCH_DIR = SUBTITLE_DIR / "search_results"

MANIFEST_PATH = (
    SUBTITLE_DIR / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)
CANDIDATES_PATH = (
    SEARCH_DIR / "ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv"
)
METADATA_PATH = PROJECT / "01_data/raw/ICMB_130_TMDB_METADATA.csv"

# Locked datasets: verify access only; never write to them here.
LOCKED_FILES = {
    "Master dataset": PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv",
    "TMDB mapping": PROJECT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv",
    "Feature dataset": PROJECT / "01_data/processed/ICMB_130_FEATURES.csv",
}

print("=" * 68)
print("ICMB-130 SUBTITLE WORKSPACE — RESUME CHECK")
print("=" * 68)
print("Project:", PROJECT)
print("Subtitle workspace exists:", SUBTITLE_DIR.exists())
print("Raw folder exists:", RAW_DIR.exists())
print("Validated folder exists:", VALIDATED_DIR.exists())
print("Search-results folder exists:", SEARCH_DIR.exists())

required_files = {
    "Subtitle manifest": MANIFEST_PATH,
    "Existing candidate database": CANDIDATES_PATH,
    "TMDB metadata": METADATA_PATH,
}

print("\nRequired files:")
for name, path in required_files.items():
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'}")

print("\nLocked datasets — access check only:")
for name, path in LOCKED_FILES.items():
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'}")

if not all(path.is_file() for path in required_files.values()):
    raise FileNotFoundError(
        "A required file is missing. Stop here and share the output."
    )

# --------------------------------------------------
# 2. Load current manifest and candidate database
# --------------------------------------------------
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string",
    }
).fillna("")

candidates = pd.read_csv(
    CANDIDATES_PATH,
    dtype=str
).fillna("")

print("\nManifest records:", len(manifest))
print("Candidate database records:", len(candidates))
print("Raw subtitle files:", sum(p.is_file() for p in RAW_DIR.iterdir()))
print("Validated subtitle files:", sum(
    p.is_file() for p in VALIDATED_DIR.iterdir()
))

# --------------------------------------------------
# 3. Identify the next 10 unresolved movies
# --------------------------------------------------
# These two records are deliberately excluded from repeated work.
skip_ids = {"2000_3", "2000_5"}

record_ids = manifest["movie_record_id"].astype(str).str.strip()
validated_names = manifest["validated_subtitle_file"].astype(str).str.strip()
quality = manifest["quality_status"].astype(str).str.strip()

pending_mask = (
    validated_names.eq("")
    & ~record_ids.isin(skip_ids)
    & ~quality.str.startswith("Accepted", na=False)
)

pending = manifest.loc[pending_mask].copy()

if "year" in pending.columns:
    pending["year"] = pd.to_numeric(
        pending["year"], errors="coerce"
    )

pending = pending.sort_values(
    ["year", "movie_record_id"],
    na_position="last"
).head(10)

display_cols = [
    c for c in [
        "movie_record_id", "year", "movie_title",
        "language", "tmdb_id", "acquisition_status",
        "validation_status", "quality_status"
    ]
    if c in pending.columns
]

print("\nNEXT BATCH — UP TO 10 UNRESOLVED MOVIES")
print("Movies in batch:", len(pending))
print(pending[display_cols].to_string(index=False))

# --------------------------------------------------
# 4. Count existing candidate records for this batch
# --------------------------------------------------
if not pending.empty and "movie_record_id" in candidates.columns:
    selected_ids = set(
        pending["movie_record_id"].astype(str).str.strip()
    )
    candidate_subset = candidates[
        candidates["movie_record_id"].astype(str).str.strip().isin(
            selected_ids
        )
    ]

    print("\nEXISTING CANDIDATE RECORDS FOR THIS BATCH")
    if candidate_subset.empty:
        print("No saved candidates for these records.")
    else:
        print(
            candidate_subset.groupby(
                ["movie_record_id", "target_title"],
                dropna=False
            ).size().rename("candidate_rows").to_string()
        )

print("\nRead-only check complete. No files were changed.")

In [ ]:

import pandas as pd
from IPython.display import display

# The 10 movies selected by the resume check
BATCH_IDS = [
    "2002_3", "2002_4", "2002_5",
    "2003_3", "2003_4", "2003_5",
    "2004_2", "2004_3", "2004_4", "2004_5"
]

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string", "source_id": "string"},
).fillna("")

candidates = pd.read_csv(
    CANDIDATES_PATH,
    dtype=str,
).fillna("")

batch_candidates = candidates[
    candidates["movie_record_id"].astype(str).isin(BATCH_IDS)
].copy()

print("=" * 75)
print("ICMB-130 — EXISTING SUBTITLE CANDIDATE REVIEW")
print("=" * 75)
print("Candidate rows for this batch:", len(batch_candidates))
print("\nAvailable candidate columns:")
print(batch_candidates.columns.tolist())

# Display useful identifying, language, quality, and download fields.
keywords = [
    "movie_record_id", "target_title", "title", "release", "year",
    "language", "lang", "file_id", "subtitle_id", "source",
    "download", "url", "format", "type", "score", "rating",
    "hearing", "fps", "comment", "detail", "feature"
]
display_cols = [
    col for col in batch_candidates.columns
    if any(word in col.lower() for word in keywords)
]

# Keep one row per candidate; show at most 8 candidates per movie.
for record_id in BATCH_IDS:
    movie_rows = batch_candidates[
        batch_candidates["movie_record_id"].astype(str) == record_id
    ].copy()

    movie_info = manifest[
        manifest["movie_record_id"].astype(str) == record_id
    ]

    if movie_info.empty:
        print(f"\n{record_id} — manifest record not found")
        continue

    movie = movie_info.iloc[0]
    print(
        f"\n{'-' * 75}\n"
        f"{record_id} | {movie['movie_title']} ({movie['year']})\n"
        f"Expected language: {movie['language']} | "
        f"TMDB ID: {movie['tmdb_id']}\n"
        f"Existing candidates: {len(movie_rows)}"
    )

    if movie_rows.empty:
        print("No existing candidate records.")
        continue

    # Prefer Hindi-labelled candidates for these Hindi-language films.
    language_cols = [
        c for c in movie_rows.columns
        if "language" in c.lower() or c.lower() in ["lang", "iso639"]
    ]
    if language_cols:
        def language_priority(row):
            values = " ".join(
                str(row[c]).lower() for c in language_cols
            )
            if "hindi" in values or "hi" == values.strip():
                return 0
            if "english" in values or "en" == values.strip():
                return 1
            return 2

        movie_rows["_language_priority"] = movie_rows.apply(
            language_priority, axis=1
        )
        movie_rows = movie_rows.sort_values("_language_priority")

    display(movie_rows[display_cols].head(8))


In [ ]:

# Read-only shortlist: prioritize correct movie identity and likely full-film subtitles

import pandas as pd
from IPython.display import display

batch_manifest = manifest[
    manifest["movie_record_id"].astype(str).isin(BATCH_IDS)
].copy()

cand = candidates.copy()
cand["file_id"] = cand["file_id"].astype(str).str.strip()
cand["feature_tmdb_id_num"] = pd.to_numeric(
    cand["feature_tmdb_id"], errors="coerce"
)
cand["feature_year_num"] = pd.to_numeric(
    cand["feature_year"], errors="coerce"
)
cand["download_count_num"] = pd.to_numeric(
    cand["download_count"], errors="coerce"
).fillna(0)

results = []

for _, movie in batch_manifest.iterrows():
    rid = str(movie["movie_record_id"])
    tmdb = pd.to_numeric(movie["tmdb_id"], errors="coerce")
    year = pd.to_numeric(movie["year"], errors="coerce")

    rows = cand[cand["movie_record_id"].astype(str) == rid].copy()

    # Exact movie identity first; don't trust a similar title alone.
    exact = rows[
        (rows["feature_tmdb_id_num"] == tmdb) &
        (rows["feature_year_num"] == year)
    ].copy()

    if exact.empty:
        exact = rows[
            rows["feature_year_num"] == year
        ].copy()
        exact["identity_match_note"] = "Year match only — identity needs review"
    else:
        exact["identity_match_note"] = "TMDB ID + year match"

    exact = exact.drop_duplicates(subset="file_id")
    exact["subtitle_language"] = exact["subtitle_language"].str.lower().str.strip()

    # These are subtitle-language labels, not the video release language.
    exact["language_priority"] = exact["subtitle_language"].map(
        {"hi": 0, "hin": 0, "hindi": 0, "en": 1, "eng": 1, "english": 1}
    ).fillna(2)

    exact = exact.sort_values(
        ["language_priority", "download_count_num"],
        ascending=[True, False]
    )

    for _, row in exact.head(5).iterrows():
        results.append({
            "movie_record_id": rid,
            "movie_title": movie["movie_title"],
            "file_id": row["file_id"],
            "subtitle_language": row["subtitle_language"],
            "download_count": int(row["download_count_num"]),
            "release": row["release"],
            "feature_title": row["feature_title"],
            "feature_year": row["feature_year"],
            "feature_tmdb_id": row["feature_tmdb_id"],
            "identity_check": row["identity_match_note"],
        })

shortlist = pd.DataFrame(results)

print("SHORTLIST — up to 5 candidates per movie")
print("Rows:", len(shortlist))
if shortlist.empty:
    print("No candidates passed the identity/year filter.")
else:
    display(shortlist)
    print("\nCandidate language counts:")
    display(
        shortlist.groupby(
            ["movie_record_id", "movie_title", "subtitle_language"],
            dropna=False
        ).size().rename("candidate_count").reset_index()
    )


In [ ]:

# NOTEBOOK 8 — RESTORE OPENSUBTITLES CREDENTIALS FROM COLAB SECRETS

from google.colab import userdata
import requests

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
USERNAME = userdata.get("OPEN_SUBTITLES_USERNAME")
PASSWORD = userdata.get("OPEN_SUBTITLES_PASSWORD")

if not all([API_KEY, USERNAME, PASSWORD]):
    raise RuntimeError(
        "One or more required OpenSubtitles secrets are missing. "
        "Check OPEN_SUBTITLES_API_KEY, OPEN_SUBTITLES_USERNAME, "
        "and OPEN_SUBTITLES_PASSWORD in Colab Secrets."
    )

login_response = requests.post(
    "https://api.opensubtitles.com/api/v1/login",
    headers={
        "Api-Key": API_KEY,
        "Content-Type": "application/json",
        "User-Agent": "ICMB130SubtitleResearch/1.0",
    },
    json={
        "username": USERNAME,
        "password": PASSWORD,
    },
    timeout=30,
)

if login_response.status_code == 200:
    ACCESS_TOKEN = login_response.json().get("token")

    if not ACCESS_TOKEN:
        raise RuntimeError("Login response did not contain an access token.")

    print("OpenSubtitles login successful. Credentials remain hidden.")

else:
    ACCESS_TOKEN = None
    print("Login failed. HTTP status:", login_response.status_code)
    print("Check your Colab Secrets and account/API access; do not print credentials.")


In [ ]:

# NOTEBOOK 8 — BATCH 1: DOWNLOAD ENGLISH FALLBACK CANDIDATES
# Downloads only; validation and manifest updates come later.

import re
import time
import requests
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
SUBTITLE_ROOT = (
    PROJECT_ROOT / "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = SUBTITLE_ROOT / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_CANDIDATES = {
    "2002_3": ("Raaz", "687862"),
    "2002_4": ("Hum Tumhare Hain Sanam", "685064"),
    "2002_5": ("Kaante", "4206080"),
    "2003_3": ("The Hero Love Story of a Spy", "3235"),
    "2003_4": ("Munna Bhai M.B.B.S.", "582775"),
    "2003_5": ("Baghban", "710859"),
    "2004_2": ("Main Hoon Na", "670527"),
    "2004_3": ("Dhoom", "1458752"),
    "2004_4": ("Mujhse Shaadi Karogi", "4093113"),
    "2004_5": ("Khakee", "4555612"),
}

if "API_KEY" not in globals() or "ACCESS_TOKEN" not in globals():
    raise RuntimeError(
        "OpenSubtitles credentials are missing. "
        "Run the credential restoration cell first."
    )

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
    "Content-Type": "application/json",
}

results = []

for record_id, (title, file_id) in SELECTED_CANDIDATES.items():
    safe_title = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")
    output_path = RAW_DIR / (
        f"{record_id}_{safe_title}_candidate_{file_id}.srt"
    )

    if output_path.exists():
        print(f"SKIPPED — raw file already exists: {output_path.name}")
        results.append({
            "movie_record_id": record_id,
            "title": title,
            "file_id": file_id,
            "status": "Skipped_existing_file",
            "bytes": output_path.stat().st_size,
            "path": str(output_path),
        })
        continue

    try:
        response = requests.post(
            "https://api.opensubtitles.com/api/v1/download",
            headers=headers,
            json={"file_id": int(file_id)},
            timeout=30,
        )

        if response.status_code != 200:
            print(f"FAILED — {title}: HTTP {response.status_code}")
            results.append({
                "movie_record_id": record_id,
                "title": title,
                "file_id": file_id,
                "status": f"Download_API_HTTP_{response.status_code}",
                "bytes": 0,
                "path": "",
            })
            time.sleep(1)
            continue

        download_data = response.json()
        download_link = download_data.get("link")

        if not download_link:
            print(f"FAILED — {title}: no download link returned")
            results.append({
                "movie_record_id": record_id,
                "title": title,
                "file_id": file_id,
                "status": "No_download_link",
                "bytes": 0,
                "path": "",
            })
            time.sleep(1)
            continue

        file_response = requests.get(download_link, timeout=60)
        file_response.raise_for_status()
        content = file_response.content

        # Avoid saving empty or obviously invalid responses.
        if len(content) < 100:
            print(f"FAILED — {title}: response too small to be a subtitle")
            results.append({
                "movie_record_id": record_id,
                "title": title,
                "file_id": file_id,
                "status": "Response_too_small",
                "bytes": len(content),
                "path": "",
            })
            time.sleep(1)
            continue

        # Save only to a new raw path; never overwrite.
        with open(output_path, "xb") as f:
            f.write(content)

        print(f"DOWNLOADED — {title}: {len(content):,} bytes")
        results.append({
            "movie_record_id": record_id,
            "title": title,
            "file_id": file_id,
            "status": "Downloaded_pending_validation",
            "bytes": len(content),
            "path": str(output_path),
        })

    except Exception as exc:
        print(f"ERROR — {title}: {type(exc).__name__}: {exc}")
        results.append({
            "movie_record_id": record_id,
            "title": title,
            "file_id": file_id,
            "status": f"Error_{type(exc).__name__}",
            "bytes": 0,
            "path": "",
        })

    time.sleep(1)

print("\nDOWNLOAD SUMMARY")
for item in results:
    print(
        f"{item['movie_record_id']} | {item['title']} | "
        f"{item['status']} | {item['bytes']:,} bytes"
    )


In [ ]:

# NOTEBOOK 8 — BATCH 1: VALIDATE DOWNLOADED SUBTITLE CANDIDATES
# Read-only validation. No manifest updates or file modifications.

import re
import hashlib
from pathlib import Path
from datetime import datetime
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
RAW_DIR = (
    PROJECT_ROOT / "01_data/processed/subtitle_corpus_v2/raw"
)

BATCH = [
    ("2002_3", "Raaz", 2002, 20731, 150),
    ("2002_4", "Hum Tumhare Hain Sanam", 2002, 90634, 162),
    ("2002_5", "Kaante", 2002, 35907, 154),
    ("2003_3", "The Hero Love Story of a Spy", 2003, 271677, 160),
    ("2003_4", "Munna Bhai M.B.B.S.", 2003, 19625, 156),
    ("2003_5", "Baghban", 2003, 39839, 181),
    ("2004_2", "Main Hoon Na", 2004, 14134, 179),
    ("2004_3", "Dhoom", 2004, 17478, 129),
    ("2004_4", "Mujhse Shaadi Karogi", 2004, 20495, 163),
    ("2004_5", "Khakee", 2004, 21665, 174),
]

def to_seconds(timestamp):
    timestamp = timestamp.replace(",", ".")
    h, m, s = timestamp.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

def parse_srt(text):
    blocks = re.split(r"\n\s*\n", text.strip())
    parsed = []
    malformed = 0

    for block in blocks:
        lines = block.splitlines()
        time_index = next(
            (
                i for i, line in enumerate(lines)
                if "-->" in line
            ),
            None,
        )

        if time_index is None:
            malformed += 1
            continue

        match = re.search(
            r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
            r"(\d{2}:\d{2}:\d{2}[,.]\d{3})",
            lines[time_index],
        )

        if not match:
            malformed += 1
            continue

        try:
            start = to_seconds(match.group(1))
            end = to_seconds(match.group(2))
        except Exception:
            malformed += 1
            continue

        dialogue = " ".join(lines[time_index + 1:]).strip()
        dialogue = re.sub(r"<[^>]+>", " ", dialogue)
        dialogue = re.sub(r"\{[^}]*\}", " ", dialogue)
        dialogue = re.sub(r"\s+", " ", dialogue).strip()

        parsed.append({
            "start": start,
            "end": end,
            "dialogue": dialogue,
        })

    return parsed, malformed

rows = []

for record_id, title, year, tmdb_id, runtime_min in BATCH:
    safe_title = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")
    file_id = {
        "2002_3": "687862",
        "2002_4": "685064",
        "2002_5": "4206080",
        "2003_3": "3235",
        "2003_4": "582775",
        "2003_5": "710859",
        "2004_2": "670527",
        "2004_3": "1458752",
        "2004_4": "4093113",
        "2004_5": "4555612",
    }[record_id]

    path = RAW_DIR / (
        f"{record_id}_{safe_title}_candidate_{file_id}.srt"
    )

    result = {
        "movie_record_id": record_id,
        "movie_title": title,
        "year": year,
        "tmdb_id": tmdb_id,
        "file_id": file_id,
        "raw_file": str(path),
    }

    if not path.exists():
        result.update({
            "validation_status": "File_missing",
            "bytes": None,
        })
        rows.append(result)
        continue

    raw_bytes = path.read_bytes()
    result["bytes"] = len(raw_bytes)
    result["sha256"] = hashlib.sha256(raw_bytes).hexdigest()

    # Decode without altering the raw file.
    try:
        text = raw_bytes.decode("utf-8-sig")
        encoding = "UTF-8"
    except UnicodeDecodeError:
        try:
            text = raw_bytes.decode("cp1252")
            encoding = "CP1252_fallback"
        except UnicodeDecodeError:
            text = raw_bytes.decode("latin-1")
            encoding = "Latin-1_fallback"

    result["encoding"] = encoding
    result["replacement_characters"] = text.count("\ufffd")

    normalized = text.replace("\r\n", "\n").replace("\r", "\n")
    parsed, malformed = parse_srt(normalized)

    dialogue_text = " ".join(
        item["dialogue"] for item in parsed
    )
    words = re.findall(r"\b[\w']+\b", dialogue_text)
    devanagari_chars = len(
        re.findall(r"[\u0900-\u097F]", dialogue_text)
    )
    latin_chars = len(re.findall(r"[A-Za-z]", dialogue_text))

    reversed_timestamps = sum(
        item["end"] < item["start"] for item in parsed
    )

    sorted_blocks = sorted(parsed, key=lambda item: item["start"])
    overlaps = sum(
        sorted_blocks[i]["start"] < sorted_blocks[i - 1]["end"]
        for i in range(1, len(sorted_blocks))
    )

    gaps = [
        sorted_blocks[i]["start"] - sorted_blocks[i - 1]["end"]
        for i in range(1, len(sorted_blocks))
        if sorted_blocks[i]["start"] - sorted_blocks[i - 1]["end"] > 120
    ]

    runtime_seconds = runtime_min * 60
    first_start = min(
        (item["start"] for item in parsed), default=None
    )
    last_end = max(
        (item["end"] for item in parsed), default=None
    )

    result.update({
        "subtitle_blocks": len(parsed),
        "malformed_blocks": malformed,
        "reversed_timestamps": reversed_timestamps,
        "overlapping_blocks": overlaps,
        "gaps_over_2min": len(gaps),
        "largest_gap_min": round(max(gaps) / 60, 2) if gaps else 0,
        "first_subtitle_min": round(first_start / 60, 2)
            if first_start is not None else None,
        "last_subtitle_min": round(last_end / 60, 2)
            if last_end is not None else None,
        "runtime_min_reference": runtime_min,
        "ending_coverage_pct": round(
            min(last_end / runtime_seconds * 100, 100), 1
        ) if last_end is not None else None,
        "word_count": len(words),
        "devanagari_chars": devanagari_chars,
        "latin_chars": latin_chars,
        "empty_dialogue_blocks": sum(
            not item["dialogue"] for item in parsed
        ),
        "sha256": result["sha256"],
    })

    if not parsed:
        status = "Reject_or_manual_review_no_valid_blocks"
    elif malformed > 0 or reversed_timestamps > 0:
        status = "Manual_review_structural_defects"
    elif not words:
        status = "Manual_review_no_dialogue"
    else:
        status = "Pending_coverage_and_language_review"

    result["validation_status"] = status
    rows.append(result)

validation_df = pd.DataFrame(rows)

print("BATCH 1 — READ-ONLY VALIDATION SUMMARY")
display(
    validation_df[
        [
            "movie_record_id",
            "movie_title",
            "bytes",
            "encoding",
            "subtitle_blocks",
            "malformed_blocks",
            "reversed_timestamps",
            "overlapping_blocks",
            "gaps_over_2min",
            "largest_gap_min",
            "first_subtitle_min",
            "last_subtitle_min",
            "ending_coverage_pct",
            "word_count",
            "devanagari_chars",
            "latin_chars",
            "validation_status",
        ]
    ]
)

print("\nNo files or manifests were modified.")


In [ ]:

# NOTEBOOK 8 — INSPECT SAVED CANDIDATE CSV SCHEMA

from pathlib import Path
import pandas as pd

SEARCH_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

CANDIDATE_FILE = (
    SEARCH_DIR / "ICMB130_OPENSUBTITLES_CANDIDATES_20261014_174708.csv"
)

if not CANDIDATE_FILE.exists():
    raise FileNotFoundError(CANDIDATE_FILE)

df = pd.read_csv(
    CANDIDATE_FILE,
    dtype={
        "movie_record_id": "string",
        "file_id": "string",
        "feature_tmdb_id": "string",
        "feature_year": "string",
        "target_year": "string",
    },
)

print("CSV shape:", df.shape)
print("\nActual column names:")
for col in df.columns:
    print(f" - {col}")

print("\nFirst 3 rows:")
display(df.head(3))

print("\nColumns potentially related to filenames/releases:")
display(
    pd.DataFrame({
        "column": df.columns,
        "sample_value": [
            str(df[col].dropna().iloc[0])
            if df[col].notna().any() else "<empty>"
            for col in df.columns
        ],
    })[
        pd.DataFrame({
            "column": df.columns,
        })["column"].str.contains(
            "file|name|release|title|language|download|tmdb|year|record",
            case=False,
            regex=True,
        ).values
    ]
)


In [ ]:

# NOTEBOOK 8 — LOCATE SAVED GENERAL CANDIDATE DATABASE
# Read-only inspection; no API calls or file changes.

from pathlib import Path
import pandas as pd

SEARCH_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

for path in sorted(SEARCH_DIR.glob("*.csv")):
    try:
        sample = pd.read_csv(path, nrows=3)
        print(f"\n{path.name}")
        print("Columns:", list(sample.columns))
        print("Sample rows:", len(sample))
    except Exception as exc:
        print(f"\nCould not inspect {path.name}: {exc}")


In [ ]:

# NOTEBOOK 8 — ALTERNATIVE ENGLISH CANDIDATES
# Read-only search in the existing alternate-title candidate database.

from pathlib import Path
import pandas as pd
from IPython.display import display

SEARCH_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

CANDIDATE_FILE = (
    SEARCH_DIR / "ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv"
)

INCOMPLETE = {
    "2002_4": {
        "title": "Hum Tumhare Hain Sanam",
        "tmdb_id": 90634,
        "year": 2002,
        "current_file_id": "685064",
    },
    "2004_2": {
        "title": "Main Hoon Na",
        "tmdb_id": 14134,
        "year": 2004,
        "current_file_id": "670527",
    },
    "2004_3": {
        "title": "Dhoom",
        "tmdb_id": 17478,
        "year": 2004,
        "current_file_id": "1458752",
    },
    "2004_5": {
        "title": "Khakee",
        "tmdb_id": 21665,
        "year": 2004,
        "current_file_id": "4555612",
    },
}

if not CANDIDATE_FILE.exists():
    raise FileNotFoundError(CANDIDATE_FILE)

df = pd.read_csv(
    CANDIDATE_FILE,
    dtype={
        "movie_record_id": "string",
        "file_id": "string",
        "feature_tmdb_id": "string",
        "feature_year": "string",
        "target_year": "string",
    },
)

required = {
    "movie_record_id", "file_id", "filename",
    "subtitle_language", "download_count",
    "feature_tmdb_id", "feature_year",
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

print("Saved alternate-title candidate rows:", len(df))

for record_id, info in INCOMPLETE.items():
    print(f"\n{'=' * 70}")
    print(f"{info['title']} ({info['year']}) — alternative candidates")

    # Match target movie record and exact TMDB/year where available.
    subset = df[df["movie_record_id"] == record_id].copy()

    exact = subset[
        (subset["feature_tmdb_id"] == str(info["tmdb_id"]))
        & (subset["feature_year"] == str(info["year"]))
    ].copy()

    # Exclude the already downloaded candidate.
    exact = exact[
        exact["file_id"] != info["current_file_id"]
    ].copy()

    exact["download_count"] = pd.to_numeric(
        exact["download_count"], errors="coerce"
    )

    if exact.empty:
        print("No alternative exact-TMDB/year matches in this file.")
        continue

    print(f"Exact-TMDB/year alternatives: {len(exact)}")

    display(
        exact.sort_values(
            "download_count",
            ascending=False,
            na_position="last",
        )[
            [
                "file_id",
                "filename",
                "release",
                "subtitle_language",
                "download_count",
                "upload_date",
                "feature_title",
                "feature_year",
                "feature_tmdb_id",
                "machine_translated",
                "ai_translated",
            ]
        ].head(8)
    )


In [ ]:

# NOTEBOOK 8 — BATCH 1: QUALITY REVIEW OF SIX CANDIDATES
# Read-only review; no manifest updates or file changes.

import re
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
RAW_DIR = (
    PROJECT_ROOT / "01_data/processed/subtitle_corpus_v2/raw"
)

CANDIDATES = [
    ("2002_3", "Raaz", "687862"),
    ("2002_5", "Kaante", "4206080"),
    ("2003_3", "The Hero Love Story of a Spy", "3235"),
    ("2003_4", "Munna Bhai M.B.B.S.", "582775"),
    ("2003_5", "Baghban", "710859"),
    ("2004_4", "Mujhse Shaadi Karogi", "4093113"),
]

def get_blocks(text):
    blocks = re.split(r"\n\s*\n", text.strip())
    output = []

    for block in blocks:
        lines = block.splitlines()
        ti = next(
            (i for i, line in enumerate(lines) if "-->" in line),
            None,
        )
        if ti is None:
            continue

        match = re.search(
            r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
            r"(\d{2}:\d{2}:\d{2}[,.]\d{3})",
            lines[ti],
        )
        if not match:
            continue

        def seconds(ts):
            h, m, s = ts.replace(",", ".").split(":")
            return int(h) * 3600 + int(m) * 60 + float(s)

        dialogue = " ".join(lines[ti + 1:]).strip()
        dialogue = re.sub(r"<[^>]+>", " ", dialogue)
        dialogue = re.sub(r"\s+", " ", dialogue).strip()

        output.append({
            "start": seconds(match.group(1)),
            "end": seconds(match.group(2)),
            "time": lines[ti],
            "dialogue": dialogue,
        })

    return output

summary = []

for record_id, title, file_id in CANDIDATES:
    safe_title = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")
    path = RAW_DIR / (
        f"{record_id}_{safe_title}_candidate_{file_id}.srt"
    )

    print("\n" + "=" * 72)
    print(f"{title} | {record_id}")

    if not path.exists():
        print("RAW FILE MISSING")
        continue

    text = path.read_text(encoding="utf-8-sig", errors="replace")
    blocks = get_blocks(text)
    dialogues = [b["dialogue"] for b in blocks if b["dialogue"]]

    # Exact repeated dialogue counts (repetition can be legitimate).
    normalized = [
        re.sub(r"[^a-z0-9]+", " ", d.lower()).strip()
        for d in dialogues
    ]
    counts = pd.Series(normalized).value_counts() if normalized else pd.Series(dtype=int)
    repeated = counts[counts >= 3]

    # Potential non-dialogue lines, to inspect rather than auto-delete.
    suspicious = [
        b for b in blocks
        if re.search(
            r"subtitles? by|www\.|\.com\b|frame ?rate|^\d+(\.\d+)?$|"
            r"downloaded from|opensubtitles",
            b["dialogue"],
            re.I,
        )
    ]

    # Timestamp overlaps and gaps over 2 minutes.
    ordered = sorted(blocks, key=lambda b: b["start"])
    overlaps = sum(
        ordered[i]["start"] < ordered[i - 1]["end"]
        for i in range(1, len(ordered))
    )
    gaps = [
        (ordered[i]["start"] - ordered[i - 1]["end"]) / 60
        for i in range(1, len(ordered))
        if ordered[i]["start"] - ordered[i - 1]["end"] > 120
    ]

    summary.append({
        "movie_record_id": record_id,
        "movie_title": title,
        "blocks": len(blocks),
        "dialogue_blocks": len(dialogues),
        "repeated_phrases_3plus": len(repeated),
        "suspicious_non_dialogue_entries": len(suspicious),
        "overlapping_blocks": overlaps,
        "gaps_over_2min": len(gaps),
        "largest_gap_min": round(max(gaps), 2) if gaps else 0,
        "first_min": round(min((b["start"] for b in blocks), default=0) / 60, 2),
        "last_min": round(max((b["end"] for b in blocks), default=0) / 60, 2),
    })

    print(f"Blocks: {len(blocks)} | Dialogue blocks: {len(dialogues)}")
    print(f"Repeated normalized phrases occurring 3+ times: {len(repeated)}")
    print(f"Potential non-dialogue entries: {len(suspicious)}")
    print(f"Timestamp overlaps: {overlaps} | Gaps >2 min: {len(gaps)}")

    if not repeated.empty:
        print("\nMost repeated phrases (inspect; do not auto-delete):")
        for phrase, count in repeated.head(5).items():
            print(f"  [{count}x] {phrase[:180]}")

    if suspicious:
        print("\nPotential non-dialogue entries:")
        for item in suspicious[:8]:
            print(f"  {item['time']} | {item['dialogue'][:200]}")

print("\n" + "=" * 72)
print("QUALITY REVIEW SUMMARY")
display(pd.DataFrame(summary))
print("\nNo files or manifests were modified.")


In [ ]:

# NOTEBOOK 8 — BATCH 1: INSPECT CURRENT MANIFEST ROWS
# Read-only: no backup or changes in this cell.

from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
MANIFEST_PATH = (
    PROJECT_ROOT
    / "01_data/processed/subtitle_corpus_v2/"
    / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

BATCH_IDS = [
    "2002_3", "2002_4", "2002_5",
    "2003_3", "2003_4", "2003_5",
    "2004_2", "2004_3", "2004_4", "2004_5",
]

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string",
    },
)

print("Manifest rows:", len(manifest))
print("Manifest columns:", list(manifest.columns))

selected = manifest[
    manifest["movie_record_id"].isin(BATCH_IDS)
].copy()

print("\nMatching rows:", len(selected))
display(selected)


In [ ]:

# NOTEBOOK 8 — BATCH 1: BACK UP AND UPDATE SUBTITLE MANIFEST
# Updates only subtitle-related manifest fields for the 10 batch records.

from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd
import re

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
SUBTITLE_ROOT = (
    PROJECT_ROOT / "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = SUBTITLE_ROOT / "raw"
VALIDATED_DIR = SUBTITLE_ROOT / "validated"
MANIFEST_PATH = (
    SUBTITLE_ROOT / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

# Decisions based on the current validation and quality review.
DECISIONS = {
    "2002_3": {
        "title": "Raaz",
        "file_id": "687862",
        "blocks": 1035,
        "words": 7533,
        "chars": 29755,
        "status": "Accepted_with_quality_flags",
        "quality": "Accepted_with_review_flags",
        "notes": "English fallback; no original Hindi subtitle confirmed. "
                 "8 gaps over 2 minutes (largest 6.54 min); opening "
                 "technical entry '23.976' retained in raw. "
                 "Accepted provisionally for dialogue analysis.",
    },
    "2002_5": {
        "title": "Kaante",
        "file_id": "4206080",
        "blocks": 1515,
        "words": 11562,
        "chars": 45340,
        "status": "Accepted_with_quality_flags",
        "quality": "Accepted_with_review_flags",
        "notes": "English fallback; no original Hindi subtitle confirmed. "
                 "4 gaps over 2 minutes (largest 6.03 min); repeated "
                 "phrases may reflect songs or dialogue and were retained.",
    },
    "2003_3": {
        "title": "The Hero Love Story of a Spy",
        "file_id": "3235",
        "blocks": 1390,
        "words": 11123,
        "chars": 45738,
        "status": "Accepted_with_quality_flags",
        "quality": "Accepted_with_review_flags",
        "notes": "English fallback; no original Hindi subtitle confirmed. "
                 "6 gaps over 2 minutes (largest 4.43 min); final subtitle "
                 "credit retained in raw.",
    },
    "2003_4": {
        "title": "Munna Bhai M.B.B.S.",
        "file_id": "582775",
        "blocks": 1665,
        "words": 11495,
        "chars": 45784,
        "status": "Accepted_English_fallback",
        "quality": "Good_structural_quality",
        "notes": "English fallback; no original Hindi subtitle confirmed. "
                 "No gaps over 2 minutes, overlaps, or malformed blocks "
                 "observed in current checks.",
    },
    "2003_5": {
        "title": "Baghban",
        "file_id": "710859",
        "blocks": 1782,
        "words": 14379,
        "chars": 56489,
        "status": "Accepted_English_fallback",
        "quality": "Good_structural_quality",
        "notes": "English fallback; no original Hindi subtitle confirmed. "
                 "No gaps over 2 minutes, overlaps, or malformed blocks "
                 "observed in current checks.",
    },
    "2004_4": {
        "title": "Mujhse Shaadi Karogi",
        "file_id": "4093113",
        "blocks": 1696,
        "words": 12827,
        "chars": 49490,
        "status": "Accepted_with_quality_flags",
        "quality": "Accepted_with_review_flags",
        "notes": "English fallback; no original Hindi subtitle confirmed. "
                 "One timestamp overlap and one gap over 2 minutes "
                 "(largest 2.83 min); retained for further analysis.",
    },
    "2002_4": {
        "title": "Hum Tumhare Hain Sanam",
        "file_id": "685064",
        "status": "Rejected_incomplete",
        "quality": "Rejected_incomplete_coverage",
        "notes": "English fallback candidate rejected: subtitles end at "
                 "82.89 min against the current screening runtime reference. "
                 "No alternative exact-TMDB/year candidate in saved search data.",
    },
    "2004_2": {
        "title": "Main Hoon Na",
        "file_id": "670527",
        "status": "Rejected_incomplete",
        "quality": "Rejected_incomplete_coverage",
        "notes": "English fallback candidate rejected: subtitles end at "
                 "86.80 min against the current screening runtime reference. "
                 "No alternative exact-TMDB/year candidate in saved search data.",
    },
    "2004_3": {
        "title": "Dhoom",
        "file_id": "1458752",
        "status": "Rejected_incomplete",
        "quality": "Rejected_incomplete_coverage",
        "notes": "English fallback candidate rejected: subtitles end at "
                 "59.98 min against the current screening runtime reference. "
                 "No alternative exact-TMDB/year candidate in saved search data.",
    },
    "2004_5": {
        "title": "Khakee",
        "file_id": "4555612",
        "status": "Rejected_incomplete",
        "quality": "Rejected_incomplete_coverage",
        "notes": "English fallback candidate rejected: subtitles end at "
                 "92.07 min against the current screening runtime reference. "
                 "No alternative exact-TMDB/year candidate in saved search data.",
    },
}

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string",
    },
)

expected_ids = set(DECISIONS)
matching = manifest["movie_record_id"].isin(expected_ids)

if int(matching.sum()) != 10:
    raise RuntimeError(
        f"Expected exactly 10 manifest rows; found {int(matching.sum())}. "
        "No changes written."
    )

if manifest.loc[matching, "movie_record_id"].duplicated().any():
    raise RuntimeError("Duplicate record IDs found. No changes written.")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = MANIFEST_PATH.with_name(
    f"{MANIFEST_PATH.stem}_backup_batch01_{timestamp}.csv"
)

# Preserve current manifest before modifying.
shutil.copy2(MANIFEST_PATH, backup_path)

for record_id, decision in DECISIONS.items():
    idx = manifest.index[
        manifest["movie_record_id"] == record_id
    ][0]

    safe_title = re.sub(
        r"[^A-Za-z0-9]+", "_", decision["title"]
    ).strip("_")

    raw_path = RAW_DIR / (
        f"{record_id}_{safe_title}_candidate_{decision['file_id']}.srt"
    )

    manifest.at[idx, "subtitle_language"] = "en"
    manifest.at[idx, "subtitle_type"] = "English_fallback"
    manifest.at[idx, "subtitle_source"] = "OpenSubtitles"
    manifest.at[idx, "source_id"] = decision["file_id"]
    manifest.at[idx, "source_url"] = (
        f"https://www.opensubtitles.com/en/subtitles/{decision['file_id']}"
    )
    manifest.at[idx, "raw_subtitle_file"] = (
        str(raw_path) if raw_path.exists() else ""
    )
    manifest.at[idx, "validated_subtitle_file"] = ""

    manifest.at[idx, "acquisition_status"] = decision["status"]
    manifest.at[idx, "validation_status"] = (
        "Accepted_pending_validated_copy"
        if decision["status"].startswith("Accepted")
        else "Rejected_incomplete_coverage"
    )

    manifest.at[idx, "title_verified"] = True
    manifest.at[idx, "year_verified"] = True
    manifest.at[idx, "movie_identity_verified"] = True
    manifest.at[idx, "language_verified"] = True

    manifest.at[idx, "encoding_status"] = "UTF-8"
    manifest.at[idx, "quality_status"] = decision["quality"]
    manifest.at[idx, "notes"] = decision["notes"]

    for col in ["dialogue_blocks", "word_count", "character_count"]:
        manifest.at[idx, col] = decision.get(col, pd.NA)

# Write updated manifest only after backup exists.
manifest.to_csv(MANIFEST_PATH, index=False)

print("Manifest updated.")
print("Backup:", backup_path)
print("Rows updated:", int(matching.sum()))
print("\nUpdated batch status:")
display(
    manifest.loc[
        matching,
        [
            "movie_record_id", "movie_title",
            "subtitle_language", "source_id",
            "acquisition_status", "validation_status",
            "quality_status", "raw_subtitle_file",
            "validated_subtitle_file",
        ],
    ].sort_values("movie_record_id")
)

print("\nNo locked master, mapping, or feature dataset was modified.")


In [ ]:

# NOTEBOOK 8 — BATCH 1: CREATE VALIDATED COPIES
# Raw files remain untouched. Manifest is not updated in this cell.

import re
import hashlib
from pathlib import Path
from datetime import datetime
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
SUBTITLE_ROOT = (
    PROJECT_ROOT / "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = SUBTITLE_ROOT / "raw"
VALIDATED_DIR = SUBTITLE_ROOT / "validated"
VALIDATED_DIR.mkdir(parents=True, exist_ok=True)

ACCEPTED = [
    ("2002_3", "Raaz", "687862"),
    ("2002_5", "Kaante", "4206080"),
    ("2003_3", "The Hero Love Story of a Spy", "3235"),
    ("2003_4", "Munna Bhai M.B.B.S.", "582775"),
    ("2003_5", "Baghban", "710859"),
    ("2004_4", "Mujhse Shaadi Karogi", "4093113"),
]

def parse_blocks(text):
    parts = re.split(r"\n\s*\n", text.strip())
    result = []

    for part in parts:
        lines = part.splitlines()
        time_idx = next(
            (i for i, line in enumerate(lines) if "-->" in line),
            None,
        )
        if time_idx is None:
            continue

        match = re.search(
            r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
            r"(\d{2}:\d{2}:\d{2}[,.]\d{3})",
            lines[time_idx],
        )
        if not match:
            continue

        dialogue = " ".join(lines[time_idx + 1:]).strip()
        result.append({
            "start": match.group(1),
            "end": match.group(2),
            "dialogue": dialogue,
            "original": part.strip(),
        })

    return result

results = []

for record_id, title, file_id in ACCEPTED:
    safe_title = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")
    raw_path = RAW_DIR / (
        f"{record_id}_{safe_title}_candidate_{file_id}.srt"
    )
    output_path = VALIDATED_DIR / (
        f"{record_id}_{safe_title}_ENGLISH.srt"
    )

    print(f"\nProcessing: {title}")

    if not raw_path.exists():
        print("  ERROR: raw file missing; skipped")
        results.append({
            "movie_record_id": record_id,
            "movie_title": title,
            "status": "Raw_file_missing",
        })
        continue

    if output_path.exists():
        print(f"  SKIPPED: validated file already exists: {output_path.name}")
        results.append({
            "movie_record_id": record_id,
            "movie_title": title,
            "status": "Destination_already_exists_no_overwrite",
            "validated_file": str(output_path),
        })
        continue

    raw_bytes = raw_path.read_bytes()
    text = raw_bytes.decode("utf-8-sig", errors="strict")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = parse_blocks(text)

    # Remove only the entries explicitly identified during review:
    # Raaz: opening frame-rate entry.
    # The Hero: final subtitle-credit entry.
    remove_indices = set()

    if record_id == "2002_3":
        for i, block in enumerate(blocks):
            if block["dialogue"].strip() == "23.976":
                remove_indices.add(i)

    if record_id == "2003_3":
        for i, block in enumerate(blocks):
            if re.fullmatch(
                r"Subtitles by:\s*Mr\.John",
                block["dialogue"].strip(),
                flags=re.I,
            ):
                remove_indices.add(i)

    kept = [
        block for i, block in enumerate(blocks)
        if i not in remove_indices
    ]

    # Renumber blocks in the validated copy; preserve timestamps and dialogue.
    output_text = "\n\n".join(
        f"{i}\n{block['start']} --> {block['end']}\n"
        f"{block['dialogue']}"
        for i, block in enumerate(kept, start=1)
    ) + "\n"

    # Exclusive create prevents overwriting any existing validated file.
    with open(output_path, "x", encoding="utf-8", newline="\n") as f:
        f.write(output_text)

    kept_dialogue = " ".join(block["dialogue"] for block in kept)
    words = re.findall(r"\b[\w']+\b", kept_dialogue)
    devanagari_chars = len(
        re.findall(r"[\u0900-\u097F]", kept_dialogue)
    )
    latin_chars = len(re.findall(r"[A-Za-z]", kept_dialogue))

    results.append({
        "movie_record_id": record_id,
        "movie_title": title,
        "status": "Validated_copy_created_pending_final_review",
        "raw_blocks": len(blocks),
        "removed_non_dialogue_entries": len(remove_indices),
        "validated_blocks": len(kept),
        "word_count": len(words),
        "devanagari_chars": devanagari_chars,
        "latin_chars": latin_chars,
        "raw_sha256": hashlib.sha256(raw_bytes).hexdigest(),
        "validated_sha256": hashlib.sha256(
            output_path.read_bytes()
        ).hexdigest(),
        "validated_file": str(output_path),
    })

    print(f"  Validated copy created: {output_path.name}")
    print(f"  Entries removed from copy: {len(remove_indices)}")
    print(f"  Validated blocks: {len(kept)}")

print("\nVALIDATED COPY SUMMARY")
display(pd.DataFrame(results))
print("\nRaw files and manifest remain unchanged.")


In [ ]:

from pathlib import Path
import hashlib
import re
import pandas as pd

VALIDATED_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/validated"
)

# Expected values from the validated-copy creation output.
expected = [
    ("2002_3", "Raaz", "2002_3_Raaz_ENGLISH.srt",
     "e513514b1598bdbee951982e7ab7172dff580d398681ad", 1034),
    ("2002_5", "Kaante", "2002_5_Kaante_ENGLISH.srt",
     "bb234ed5afaddf2e73a011460292a6a0f7c1bbbff9b508", 1515),
    ("2003_3", "The Hero Love Story of a Spy",
     "2003_3_The_Hero_Love_Story_of_a_Spy_ENGLISH.srt",
     "d45ba9f73c8a215d98fee9c8532821a62c28e161f6a275", 1389),
    ("2003_4", "Munna Bhai M.B.B.S.",
     "2003_4_Munna_Bhai_M_B_B_S_ENGLISH.srt",
     "d4557b29a99cfd2fd4e6fbaf824d83626e1c65ab9daf6d", 1665),
    ("2003_5", "Baghban", "2003_5_Baghban_ENGLISH.srt",
     "7c9e84ae0edc14f29e387ff792711773829d6a388cd810", 1782),
    ("2004_4", "Mujhse Shaadi Karogi",
     "2004_4_Mujhse_Shaadi_Karogi_ENGLISH.srt",
     "899ff8c52ca16dc783844892bb674ee8e4e7950c50a9fa", 1696),
]

results = []

for record_id, title, filename, hash_prefix, expected_blocks in expected:
    path = VALIDATED_DIR / filename

    if not path.exists():
        results.append({
            "movie_record_id": record_id,
            "movie_title": title,
            "file_exists": False,
            "blocks": None,
            "hash_prefix_matches": False,
            "status": "FAIL: file missing"
        })
        continue

    content = path.read_bytes()
    sha256 = hashlib.sha256(content).hexdigest()
    text = content.decode("utf-8-sig", errors="strict")

    blocks = [
        b for b in re.split(r"\r?\n\s*\r?\n", text.strip())
        if b.strip()
    ]

    hash_match = sha256.startswith(hash_prefix)
    block_match = len(blocks) == expected_blocks

    results.append({
        "movie_record_id": record_id,
        "movie_title": title,
        "file_exists": True,
        "blocks": len(blocks),
        "expected_blocks": expected_blocks,
        "hash_prefix_matches": hash_match,
        "block_count_matches": block_match,
        "sha256": sha256,
        "status": "PASS" if hash_match and block_match else
                  "FAIL: verify hash/count"
    })

verification_df = pd.DataFrame(results)
display(verification_df)

print("\nOverall result:",
      "PASS — all six files verified"
      if all(r["status"] == "PASS" for r in results)
      else "CHECK REQUIRED — do not update the manifest yet")


In [ ]:

from pathlib import Path
from datetime import datetime
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

SUBTITLE_DIR = (
    PROJECT_ROOT / "01_data/processed/subtitle_corpus_v2"
)

MANIFEST_PATH = (
    SUBTITLE_DIR / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

assert MANIFEST_PATH.exists(), "Manifest not found."

df = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string"
    }
)

target_ids = [
    "2002_3", "2002_5", "2003_3",
    "2003_4", "2003_5", "2004_4"
]

matches = df["movie_record_id"].isin(target_ids)

assert matches.sum() == 6, (
    f"Expected exactly 6 target records; found {matches.sum()}."
)

assert df.loc[matches, "movie_record_id"].nunique() == 6, (
    "Duplicate target record IDs detected."
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = SUBTITLE_DIR / (
    f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_prevalidated_update_{timestamp}.csv"
)

# Preserve the current manifest before any changes.
df.to_csv(backup_path, index=False)

print("Manifest record count:", len(df))
print("Target records verified:", matches.sum())
print("Backup created:", backup_path)
print("\nTarget records:")
display(
    df.loc[matches, [
        "movie_record_id",
        "movie_title",
        "acquisition_status",
        "validation_status",
        "validated_subtitle_file"
    ]].sort_values("movie_record_id")
)
print("\nManifest itself remains unchanged.")


In [ ]:

from pathlib import Path
from datetime import datetime
import hashlib
import re
import shutil
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
SUBTITLE_DIR = (
    PROJECT_ROOT / "01_data/processed/subtitle_corpus_v2"
)
RAW_DIR = SUBTITLE_DIR / "raw"
VALIDATED_DIR = SUBTITLE_DIR / "validated"
MANIFEST_PATH = (
    SUBTITLE_DIR / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

# Only these six records may be updated.
targets = {
    "2002_3": {
        "file": "2002_3_Raaz_ENGLISH.srt",
        "expected_blocks": 1034,
        "cleanup_note": "Removed one technical opening entry from validated copy."
    },
    "2002_5": {
        "file": "2002_5_Kaante_ENGLISH.srt",
        "expected_blocks": 1515,
        "cleanup_note": ""
    },
    "2003_3": {
        "file": "2003_3_The_Hero_Love_Story_of_a_Spy_ENGLISH.srt",
        "expected_blocks": 1389,
        "cleanup_note": "Removed one closing subtitle credit from validated copy."
    },
    "2003_4": {
        "file": "2003_4_Munna_Bhai_M_B_B_S_ENGLISH.srt",
        "expected_blocks": 1665,
        "cleanup_note": ""
    },
    "2003_5": {
        "file": "2003_5_Baghban_ENGLISH.srt",
        "expected_blocks": 1782,
        "cleanup_note": ""
    },
    "2004_4": {
        "file": "2004_4_Mujhse_Shaadi_Karogi_ENGLISH.srt",
        "expected_blocks": 1696,
        "cleanup_note": ""
    }
}

assert MANIFEST_PATH.exists(), "Manifest not found."

df = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string"
    }
)

# Ensure all targets exist exactly once.
for record_id in targets:
    assert (df["movie_record_id"] == record_id).sum() == 1, (
        f"Expected exactly one manifest row for {record_id}"
    )

# Build and validate metrics before modifying the manifest.
updates = {}

for record_id, spec in targets.items():
    validated_path = VALIDATED_DIR / spec["file"]
    assert validated_path.exists(), f"Missing validated file: {validated_path}"

    content = validated_path.read_bytes()
    text = content.decode("utf-8-sig", errors="strict")
    blocks = [
        b for b in re.split(r"\r?\n\s*\r?\n", text.strip())
        if b.strip()
    ]

    assert len(blocks) == spec["expected_blocks"], (
        f"Block count mismatch for {record_id}: {len(blocks)}"
    )

    # Count words and character-script metrics from subtitle dialogue only.
    dialogue_lines = []
    for block in blocks:
        lines = block.splitlines()
        if len(lines) >= 3:
            dialogue_lines.extend(lines[2:])

    dialogue = "\n".join(dialogue_lines)
    word_count = len(re.findall(r"\b[\w'-]+\b", dialogue))
    devanagari_chars = len(re.findall(r"[\u0900-\u097F]", dialogue))
    latin_chars = len(re.findall(r"[A-Za-z]", dialogue))
    sha256 = hashlib.sha256(content).hexdigest()

    updates[record_id] = {
        "validated_path": str(validated_path),
        "blocks": len(blocks),
        "word_count": word_count,
        "devanagari_chars": devanagari_chars,
        "latin_chars": latin_chars,
        "sha256": sha256,
        "cleanup_note": spec["cleanup_note"]
    }

# Confirm each validated file is associated with the intended raw-file record.
# This is a metadata check only; raw files are never written or deleted.
for record_id in targets:
    row = df.loc[df["movie_record_id"] == record_id].iloc[0]
    assert str(row.get("raw_subtitle_file", "")).strip() not in ("", "nan"), (
        f"Raw-file reference missing for {record_id}; stop and inspect."
    )

# Preserve an additional immediate backup before writing the manifest.
backup_path = SUBTITLE_DIR / (
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_write_"
    + datetime.now().strftime("%Y%m%d_%H%M%S") + ".csv"
)
shutil.copy2(MANIFEST_PATH, backup_path)

# Update only subtitle-related fields in the six target rows.
for record_id, metrics in updates.items():
    idx = df.index[df["movie_record_id"] == record_id][0]
    existing_notes = df.at[idx, "notes"]
    existing_notes = (
        "" if pd.isna(existing_notes) else str(existing_notes).strip()
    )

    note_parts = [existing_notes] if existing_notes else []
    if metrics["cleanup_note"]:
        note_parts.append(metrics["cleanup_note"])
    note_parts.append(
        f"Validated-copy SHA-256: {metrics['sha256']}; "
        "English fallback; original raw file preserved."
    )

    df.at[idx, "validated_subtitle_file"] = metrics["validated_path"]
    df.at[idx, "validation_status"] = (
        "Accepted_with_quality_flags"
        if record_id in {"2002_3", "2002_5", "2003_3", "2004_4"}
        else "Accepted"
    )
    df.at[idx, "dialogue_blocks"] = metrics["blocks"]
    df.at[idx, "word_count"] = metrics["word_count"]
    df.at[idx, "character_count"] = (
        metrics["devanagari_chars"] + metrics["latin_chars"]
    )
    df.at[idx, "encoding_status"] = "UTF-8"
    df.at[idx, "notes"] = " ".join(note_parts)

# Atomic replacement: write temporary CSV, then replace manifest.
temp_path = MANIFEST_PATH.with_name(
    MANIFEST_PATH.stem + "_temporary.csv"
)
df.to_csv(temp_path, index=False)
pd.read_csv(temp_path, dtype={"movie_record_id": "string"})
temp_path.replace(MANIFEST_PATH)

print("Manifest updated successfully.")
print("Backup:", backup_path)
print("Rows updated:", len(updates))
print("Locked master, mapping, features, and raw subtitles were not modified.")

summary = []
for record_id, metrics in updates.items():
    row = df.loc[df["movie_record_id"] == record_id].iloc[0]
    summary.append({
        "movie_record_id": record_id,
        "movie_title": row["movie_title"],
        "validation_status": row["validation_status"],
        "dialogue_blocks": metrics["blocks"],
        "word_count": metrics["word_count"],
        "validated_subtitle_file": metrics["validated_path"]
    })

display(pd.DataFrame(summary))


In [ ]:

import pandas as pd
from pathlib import Path

MANIFEST_PATH = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/"
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string"
    }
)

# Classify completion using both acquisition and validation status.
validation = manifest["validation_status"].fillna("").str.lower()
acquisition = manifest["acquisition_status"].fillna("").str.lower()

manifest["review_category"] = "Needs status review"

manifest.loc[
    validation.str.startswith("validated")
    | validation.str.startswith("accepted"),
    "review_category"
] = "Validated or accepted"

manifest.loc[
    validation.str.contains("rejected")
    | acquisition.str.contains("rejected"),
    "review_category"
] = "Rejected — do not repeat"

manifest.loc[
    manifest["movie_record_id"].isin(["2000_3", "2005_1"]),
    "review_category"
] = "Search decision required"

display(
    manifest[
        [
            "movie_record_id",
            "year",
            "movie_title",
            "acquisition_status",
            "validation_status",
            "review_category"
        ]
    ].sort_values(["year", "movie_title"])
)

print("\nCategory counts:")
display(manifest["review_category"].value_counts().rename_axis("category").to_frame("records"))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
MASTER_PATH = ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST_PATH = (
    ROOT / "01_data/processed/subtitle_corpus_v2/"
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

master = pd.read_csv(
    MASTER_PATH,
    dtype={"movie_record_id": "string"}
)
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string"}
)

assert "movie_record_id" in master.columns, (
    f"Master columns need review: {master.columns.tolist()}"
)
assert "movie_record_id" in manifest.columns

# Select records absent from the manifest; previously handled movies
# will not be searched again.
known_ids = set(manifest["movie_record_id"].dropna().astype(str))
new_movies = master[
    ~master["movie_record_id"].astype(str).isin(known_ids)
].copy()

# Sort chronologically, then by rank if that field is available.
sort_cols = []
for col in ["year", "rank"]:
    if col in new_movies.columns:
        sort_cols.append(col)

if sort_cols:
    new_movies = new_movies.sort_values(sort_cols, na_position="last")

batch = new_movies.head(5)

print("Master records:", len(master))
print("Already represented in manifest:", len(known_ids))
print("New records available:", len(new_movies))
print("Next batch size:", len(batch))
print("\nMaster columns:", master.columns.tolist())

display(batch)


In [ ]:

from pathlib import Path
import pandas as pd
import re
import unicodedata

SEARCH_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

# Locate the saved Batch 2 search CSV without calling the API.
CSV_PATH = SEARCH_DIR / "batch02_subtitle_search_20261015_165928.csv"

if not CSV_PATH.exists():
    available = sorted(SEARCH_DIR.glob("batch02_subtitle_search_*.csv"))
    print("Expected search file not found.")
    print("Available Batch 2 search CSVs:")
    for p in available:
        print(p)
    raise FileNotFoundError(
        "Use the actual Batch 2 search CSV path printed above."
    )

candidates = pd.read_csv(CSV_PATH)

def normalize_title(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value)).lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())

target_languages = {
    "2005_2": "ta",
    "2005_3": "hi",
    "2005_4": "hi",
    "2005_5": "hi",
    "2006_1": "hi",
}

candidates["requested_norm"] = candidates["requested_title"].map(normalize_title)
candidates["candidate_norm"] = candidates["candidate_title"].map(normalize_title)
candidates["candidate_year_num"] = pd.to_numeric(
    candidates["candidate_year"], errors="coerce"
)
candidates["file_id_num"] = pd.to_numeric(
    candidates["file_id"], errors="coerce"
)

exact = candidates[
    (candidates["candidate_norm"] == candidates["requested_norm"])
    & (candidates["candidate_year_num"] == candidates["requested_year"])
    & candidates["file_id_num"].notna()
].copy()

exact["file_id"] = exact["file_id_num"].astype("int64").astype(str)
exact["language_priority"] = exact.apply(
    lambda r: (
        0 if str(r["candidate_language"]).lower()
        == target_languages.get(str(r["movie_record_id"]), "")
        else 1 if str(r["candidate_language"]).lower() == "en"
        else 2
    ),
    axis=1
)
exact["download_count"] = pd.to_numeric(
    exact["download_count"], errors="coerce"
)

exact = exact.sort_values(
    ["movie_record_id", "candidate_language", "language_priority", "download_count"],
    ascending=[True, True, True, False],
    na_position="last"
).drop_duplicates(
    subset=["movie_record_id", "candidate_language", "file_id"]
)

# Save the filtered results for reuse.
filtered_path = SEARCH_DIR / "batch02_exact_title_year_candidates.csv"
exact.to_csv(filtered_path, index=False)

print("Original search CSV:", CSV_PATH)
print("Exact title/year candidates:", len(exact))
print("Filtered candidates saved:", filtered_path)

print("\nCounts by movie and language:")
display(
    exact.groupby(
        ["movie_record_id", "requested_title", "candidate_language"]
    ).agg(
        unique_files=("file_id", "nunique"),
        best_download_count=("download_count", "max")
    ).reset_index()
)

print("\nTop 5 candidates per movie:")
best = (
    exact.sort_values(
        ["movie_record_id", "language_priority", "download_count"],
        ascending=[True, True, False],
        na_position="last"
    )
    .groupby("movie_record_id", group_keys=False)
    .head(5)
)

display(best[[
    "movie_record_id", "requested_title",
    "candidate_title", "candidate_year",
    "candidate_language", "subtitle_result_id",
    "file_id", "download_count", "release"
]])

print("\nNo new API calls. No downloads or manifest changes.")


In [ ]:

from pathlib import Path
from google.colab import userdata
import requests
import pandas as pd
import time

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
SUBTITLE_DIR = ROOT / "01_data/processed/subtitle_corpus_v2"
RAW_DIR = SUBTITLE_DIR / "raw"
SEARCH_DIR = SUBTITLE_DIR / "search_results"
RAW_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
ACCESS_TOKEN = userdata.get("OPEN_SUBTITLES_TOKEN")

assert API_KEY, "Missing OPEN_SUBTITLES_API_KEY in Colab Secrets."
assert ACCESS_TOKEN, (
    "Missing OPEN_SUBTITLES_TOKEN. If expired, renew it using the "
    "existing login procedure before downloading."
)

HEADERS = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
    "Content-Type": "application/json"
}

selected = [
    ("2005_2", "Anniyan", 2005, "ta", "8250919"),
    ("2005_3", "No_Entry", 2005, "en", "3753295"),
    ("2005_4", "Bunty_Aur_Babli", 2005, "en", "698104"),
    ("2005_5", "Garam_Masala", 2005, "en", "4310686"),
    ("2006_1", "Dhoom_2", 2006, "en", "4579045"),
]

download_url = "https://api.opensubtitles.com/api/v1/download"
results = []

for record_id, safe_title, year, lang, file_id in selected:
    destination = RAW_DIR / (
        f"{record_id}_{safe_title}_candidate_{file_id}.srt"
    )

    if destination.exists():
        results.append({
            "movie_record_id": record_id,
            "file_id": file_id,
            "status": "SKIPPED — destination already exists",
            "path": str(destination)
        })
        print(f"Skipping existing file: {destination.name}")
        continue

    try:
        response = requests.post(
            download_url,
            headers=HEADERS,
            json={"file_id": int(file_id)},
            timeout=30
        )

        if response.status_code != 200:
            results.append({
                "movie_record_id": record_id,
                "file_id": file_id,
                "status": f"DOWNLOAD REQUEST FAILED ({response.status_code})",
                "path": ""
            })
            print(f"Failed: {safe_title} — HTTP {response.status_code}")
            time.sleep(1.1)
            continue

        payload = response.json()
        link = payload.get("link")
        if not link:
            results.append({
                "movie_record_id": record_id,
                "file_id": file_id,
                "status": "FAILED — no download link returned",
                "path": ""
            })
            print(f"No download link: {safe_title}")
            time.sleep(1.1)
            continue

        file_response = requests.get(link, timeout=60)
        file_response.raise_for_status()

        # Basic content guard: avoid saving an HTML error page as an SRT.
        content = file_response.content
        if not content or b"<html" in content[:500].lower():
            results.append({
                "movie_record_id": record_id,
                "file_id": file_id,
                "status": "FAILED — empty or HTML response",
                "path": ""
            })
            print(f"Invalid response: {safe_title}")
            time.sleep(1.1)
            continue

        with open(destination, "xb") as f:
            f.write(content)

        results.append({
            "movie_record_id": record_id,
            "file_id": file_id,
            "status": "DOWNLOADED",
            "bytes": len(content),
            "path": str(destination)
        })
        print(f"Downloaded: {safe_title} ({len(content):,} bytes)")

    except FileExistsError:
        results.append({
            "movie_record_id": record_id,
            "file_id": file_id,
            "status": "SKIPPED — destination appeared during download",
            "path": str(destination)
        })
    except Exception as exc:
        results.append({
            "movie_record_id": record_id,
            "file_id": file_id,
            "status": f"ERROR — {type(exc).__name__}: {str(exc)[:160]}",
            "path": ""
        })
        print(f"Error: {safe_title} — {type(exc).__name__}")

    time.sleep(1.1)

print("\nBATCH 2 DOWNLOAD SUMMARY")
display(pd.DataFrame(results))
print("\nManifest unchanged. No validated files were overwritten.")


In [ ]:

from pathlib import Path
import re
import hashlib
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
RAW_DIR = ROOT / "01_data/processed/subtitle_corpus_v2/raw"

candidates = [
    {
        "record_id": "2005_2", "title": "Anniyan",
        "language": "ta", "file_id": "8250919",
        "filename": "2005_2_Anniyan_candidate_8250919.srt",
        "runtime_min": None
    },
    {
        "record_id": "2005_3", "title": "No Entry",
        "language": "en", "file_id": "3753295",
        "filename": "2005_3_No_Entry_candidate_3753295.srt",
        "runtime_min": None
    },
    {
        "record_id": "2005_4", "title": "Bunty Aur Babli",
        "language": "en", "file_id": "698104",
        "filename": "2005_4_Bunty_Aur_Babli_candidate_698104.srt",
        "runtime_min": None
    },
    {
        "record_id": "2005_5", "title": "Garam Masala",
        "language": "en", "file_id": "4310686",
        "filename": "2005_5_Garam_Masala_candidate_4310686.srt",
        "runtime_min": None
    },
    {
        "record_id": "2006_1", "title": "Dhoom 2",
        "language": "en", "file_id": "4579045",
        "filename": "2006_1_Dhoom_2_candidate_4579045.srt",
        "runtime_min": None
    },
]

TIMESTAMP = re.compile(
    r"^\s*(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def to_seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def parse_stamp(line):
    match = TIMESTAMP.match(line)
    if not match:
        return None
    vals = match.groups()
    return (
        to_seconds(*vals[:4]),
        to_seconds(*vals[4:])
    )

def analyze_srt(path, expected_language):
    raw = path.read_bytes()
    text = raw.decode("utf-8-sig", errors="replace")
    blocks = [
        block for block in re.split(r"\r?\n\s*\r?\n", text.strip())
        if block.strip()
    ]

    malformed = 0
    reversed_times = 0
    overlaps = 0
    starts = []
    ends = []
    dialogue_lines = []
    devanagari_chars = 0
    tamil_chars = 0
    latin_chars = 0

    previous_end = None

    for block in blocks:
        lines = block.splitlines()
        timestamp_line = next(
            (line for line in lines if "-->" in line), None
        )

        if timestamp_line is None:
            malformed += 1
            continue

        times = parse_stamp(timestamp_line)
        if times is None:
            malformed += 1
            continue

        start, end = times
        if end <= start:
            reversed_times += 1
            continue

        if previous_end is not None and start < previous_end:
            overlaps += 1

        starts.append(start)
        ends.append(end)
        previous_end = max(previous_end or 0, end)

        try:
            time_index = lines.index(timestamp_line)
            dialogue_lines.extend(lines[time_index + 1:])
        except ValueError:
            pass

    dialogue = "\n".join(dialogue_lines)
    word_count = len(re.findall(r"\b[\w'-]+\b", dialogue))
    devanagari_chars = len(re.findall(r"[\u0900-\u097F]", dialogue))
    tamil_chars = len(re.findall(r"[\u0B80-\u0BFF]", dialogue))
    latin_chars = len(re.findall(r"[A-Za-z]", dialogue))

    gaps = [
        starts[i] - ends[i - 1]
        for i in range(1, len(starts))
        if starts[i] > ends[i - 1]
    ]
    long_gaps = [gap for gap in gaps if gap > 120]

    runtime = None
    coverage = None
    if starts and ends:
        last_end = max(ends)
        # Runtime not assumed; coverage calculated only if supplied.
        if runtime is not None:
            coverage = round(last_end / runtime * 100, 1)

    return {
        "bytes": len(raw),
        "encoding_replacement_chars": text.count("\ufffd"),
        "blocks": len(blocks),
        "valid_timestamp_blocks": len(starts),
        "malformed_blocks": malformed,
        "reversed_or_zero_duration": reversed_times,
        "overlaps": overlaps,
        "first_subtitle_min": round(min(starts) / 60, 2) if starts else None,
        "last_subtitle_min": round(max(ends) / 60, 2) if ends else None,
        "gaps_over_2min": len(long_gaps),
        "largest_gap_min": round(max(long_gaps) / 60, 2) if long_gaps else 0,
        "word_count": word_count,
        "devanagari_chars": devanagari_chars,
        "tamil_chars": tamil_chars,
        "latin_chars": latin_chars,
        "sha256": hashlib.sha256(raw).hexdigest(),
        "language_script_check": (
            "Tamil script present" if expected_language == "ta" and tamil_chars > 0
            else "No Tamil script detected" if expected_language == "ta"
            else "English fallback; Latin-script check only"
        )
    }

results = []

for item in candidates:
    path = RAW_DIR / item["filename"]
    if not path.exists():
        results.append({
            "movie_record_id": item["record_id"],
            "movie_title": item["title"],
            "status": "MISSING RAW FILE",
            "path": str(path)
        })
        continue

    metrics = analyze_srt(path, item["language"])
    results.append({
        "movie_record_id": item["record_id"],
        "movie_title": item["title"],
        "candidate_language": item["language"],
        "file_id": item["file_id"],
        "status": "STRUCTURE CHECK COMPLETE — REVIEW REQUIRED",
        **metrics
    })

validation_df = pd.DataFrame(results)
display(validation_df)

print("\nNo raw files changed.")
print("No validated copies created.")
print("Manifest unchanged.")


In [ ]:

from pathlib import Path
import re
import pandas as pd

RAW_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

# Screening runtimes in minutes; verify against the master metadata
# before treating coverage percentages as final.
movies = [
    ("2005_2", "Anniyan", "2005_2_Anniyan_candidate_8250919.srt", 181, "ta"),
    ("2005_3", "No Entry", "2005_3_No_Entry_candidate_3753295.srt", 160, "hi"),
    ("2005_4", "Bunty Aur Babli", "2005_4_Bunty_Aur_Babli_candidate_698104.srt", 170, "hi"),
    ("2005_5", "Garam Masala", "2005_5_Garam_Masala_candidate_4310686.srt", 150, "hi"),
    ("2006_1", "Dhoom 2", "2006_1_Dhoom_2_candidate_4579045.srt", 152, "hi"),
]

timestamp_re = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

results = []

for record_id, title, filename, runtime_min, target_lang in movies:
    path = RAW_DIR / filename
    if not path.exists():
        results.append({
            "movie_record_id": record_id,
            "movie_title": title,
            "status": "FILE MISSING"
        })
        continue

    text = path.read_text(encoding="utf-8-sig", errors="replace")
    blocks = [
        b for b in re.split(r"\r?\n\s*\r?\n", text.strip())
        if b.strip()
    ]

    ends = []
    dialogue_lines = []

    for block in blocks:
        lines = block.splitlines()
        time_line = next((line for line in lines if "-->" in line), None)
        if not time_line:
            continue
        match = timestamp_re.search(time_line)
        if not match:
            continue
        values = match.groups()
        start = seconds(*values[:4])
        end = seconds(*values[4:])
        if end > start:
            ends.append(end)
            idx = lines.index(time_line)
            dialogue_lines.extend(lines[idx + 1:])

    dialogue = "\n".join(dialogue_lines)
    devanagari = len(re.findall(r"[\u0900-\u097F]", dialogue))
    tamil = len(re.findall(r"[\u0B80-\u0BFF]", dialogue))
    latin = len(re.findall(r"[A-Za-z]", dialogue))
    total_script_chars = devanagari + tamil + latin

    last_min = max(ends) / 60 if ends else 0
    coverage = min(100, last_min / runtime_min * 100) if runtime_min else None

    results.append({
        "movie_record_id": record_id,
        "movie_title": title,
        "target_language": target_lang,
        "runtime_reference_min": runtime_min,
        "last_subtitle_min": round(last_min, 2),
        "endpoint_coverage_pct": round(coverage, 1),
        "devanagari_chars": devanagari,
        "tamil_chars": tamil,
        "latin_chars": latin,
        "script_chars_total": total_script_chars,
        "script_check": (
            "Tamil present" if target_lang == "ta" and tamil > 0
            else "Tamil absent — inspect/reject as Tamil"
            if target_lang == "ta"
            else "English fallback; inspect coverage"
        )
    })

display(pd.DataFrame(results))
print("\nRead-only check: raw files and manifest unchanged.")


In [ ]:

from pathlib import Path
import re
import hashlib
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
SUBTITLE_DIR = ROOT / "01_data/processed/subtitle_corpus_v2"
RAW_DIR = SUBTITLE_DIR / "raw"
VALIDATED_DIR = SUBTITLE_DIR / "validated"
VALIDATED_DIR.mkdir(parents=True, exist_ok=True)

accepted = [
    {
        "record_id": "2005_3",
        "title": "No Entry",
        "raw": "2005_3_No_Entry_candidate_3753295.srt",
        "validated": "2005_3_NO_ENTRY_ENGLISH.srt",
        "expected_blocks": 2053
    },
    {
        "record_id": "2005_4",
        "title": "Bunty Aur Babli",
        "raw": "2005_4_Bunty_Aur_Babli_candidate_698104.srt",
        "validated": "2005_4_BUNTY_AUR_BABLI_ENGLISH.srt",
        "expected_blocks": 1546
    },
    {
        "record_id": "2005_5",
        "title": "Garam Masala",
        "raw": "2005_5_Garam_Masala_candidate_4310686.srt",
        "validated": "2005_5_GARAM_MASALA_ENGLISH.srt",
        "expected_blocks": 1922
    },
    {
        "record_id": "2006_1",
        "title": "Dhoom 2",
        "raw": "2006_1_Dhoom_2_candidate_4579045.srt",
        "validated": "2006_1_DHOOM_2_ENGLISH.srt",
        "expected_blocks": 1410
    }
]

def count_blocks(text):
    return len([
        block for block in re.split(r"\r?\n\s*\r?\n", text.strip())
        if block.strip()
    ])

results = []

for item in accepted:
    raw_path = RAW_DIR / item["raw"]
    out_path = VALIDATED_DIR / item["validated"]

    if not raw_path.exists():
        results.append({
            "movie_record_id": item["record_id"],
            "status": "FAIL — raw file missing"
        })
        continue

    if out_path.exists():
        results.append({
            "movie_record_id": item["record_id"],
            "status": "SKIPPED — validated file already exists",
            "validated_file": str(out_path)
        })
        continue

    raw_bytes = raw_path.read_bytes()
    text = raw_bytes.decode("utf-8-sig", errors="strict")
    blocks = [
        block for block in re.split(r"\r?\n\s*\r?\n", text.strip())
        if block.strip()
    ]

    # No content cleanup: retain dialogue and timestamps as downloaded.
    assert len(blocks) == item["expected_blocks"], (
        f"Block count mismatch for {item['title']}"
    )

    with open(out_path, "xb") as f:
        f.write(text.encode("utf-8"))

    validated_bytes = out_path.read_bytes()
    results.append({
        "movie_record_id": item["record_id"],
        "movie_title": item["title"],
        "status": "Validated copy created — pending manifest update",
        "blocks": count_blocks(
            validated_bytes.decode("utf-8-sig", errors="strict")
        ),
        "raw_sha256": hashlib.sha256(raw_bytes).hexdigest(),
        "validated_sha256": hashlib.sha256(validated_bytes).hexdigest(),
        "validated_file": str(out_path)
    })

display(pd.DataFrame(results))
print("\nRaw files preserved. Manifest unchanged.")
print("Anniyan was intentionally excluded from the accepted copies.")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
MASTER_PATH = ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST_PATH = ROOT / (
    "01_data/processed/subtitle_corpus_v2/"
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

master = pd.read_csv(
    MASTER_PATH,
    dtype={"movie_record_id": "string"}
)
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string"}
)

target_ids = ["2005_2", "2005_3", "2005_4", "2005_5", "2006_1"]

print("Master columns:", master.columns.tolist())
print("Manifest rows before update:", len(manifest))
print("Target records in master:")

master_ids = master["movie_record_id"].astype("string").str.strip()
manifest_ids = manifest["movie_record_id"].astype("string").str.strip()

for record_id in target_ids:
    matches = master.loc[master_ids == record_id]
    print(f"\n{record_id}: {len(matches)} master match(es)")
    if len(matches):
        display(matches)

    print(
        "Already in manifest:",
        int((manifest_ids == record_id).sum())
    )

print("\nNo files were modified.")


In [ ]:

from pathlib import Path
from datetime import datetime
import pandas as pd
import hashlib
import os

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
BASE = ROOT / "01_data/processed/subtitle_corpus_v2"
RAW_DIR = BASE / "raw"
VALIDATED_DIR = BASE / "validated"
MANIFEST_PATH = BASE / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
MASTER_PATH = ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"

TARGETS = {
    "2005_3": {
        "raw": "2005_3_No_Entry_candidate_3753295.srt",
        "validated": "2005_3_NO_ENTRY_ENGLISH.srt",
        "file_id": "3753295",
        "blocks": 2053,
        "runtime": 160,
        "status": "Accepted_English_fallback",
    },
    "2005_4": {
        "raw": "2005_4_Bunty_Aur_Babli_candidate_698104.srt",
        "validated": "2005_4_BUNTY_AUR_BABLI_ENGLISH.srt",
        "file_id": "698104",
        "blocks": 1546,
        "runtime": 170,
        "status": "Accepted_English_fallback",
    },
    "2005_5": {
        "raw": "2005_5_Garam_Masala_candidate_4310686.srt",
        "validated": "2005_5_GARAM_MASALA_ENGLISH.srt",
        "file_id": "4310686",
        "blocks": 1922,
        "runtime": 150,
        "status": "Accepted_English_fallback",
    },
    "2006_1": {
        "raw": "2006_1_Dhoom_2_candidate_4579045.srt",
        "validated": "2006_1_DHOOM_2_ENGLISH.srt",
        "file_id": "4579045",
        "blocks": 1410,
        "runtime": 152,
        "status": "Accepted_English_fallback",
    },
}

TARGET_IDS = ["2005_2", "2005_3", "2005_4", "2005_5", "2006_1"]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def inspect_srt(path):
    text = path.read_text(encoding="utf-8-sig", errors="replace")
    blocks = [
        b for b in __import__("re").split(r"\n\s*\n", text.strip())
        if __import__("re").search(
            r"\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
            r"\d{2}:\d{2}:\d{2}[,.]\d{3}", b
        )
    ]
    dialogue = []
    for block in blocks:
        lines = block.splitlines()
        for line in lines:
            line = line.strip()
            if not line or "-->" in line or line.isdigit():
                continue
            if line.startswith(("{", "[")) and line.endswith(("}", "]")):
                continue
            dialogue.append(line)

    joined = " ".join(dialogue)
    words = __import__("re").findall(r"\b[\w'-]+\b", joined)
    devanagari = len(__import__("re").findall(r"[\u0900-\u097F]", joined))
    latin = len(__import__("re").findall(r"[A-Za-z]", joined))
    return len(blocks), len(words), devanagari, latin, len(text)

# Read existing files; no writes until all checks pass.
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string",
    },
)
master = pd.read_csv(
    MASTER_PATH,
    dtype={"movie_record_id": "string"},
)

manifest_ids = manifest["movie_record_id"].astype("string").str.strip()
master_ids = master["movie_record_id"].astype("string").str.strip()

assert manifest_ids.is_unique, "Manifest already has duplicate record IDs."
assert all((master_ids == rid).sum() == 1 for rid in TARGET_IDS), (
    "Each target must occur exactly once in the master."
)
assert not manifest_ids.isin(TARGET_IDS).any(), (
    "At least one target already exists in the manifest; stop to avoid duplicates."
)

# Verify every accepted raw/validated pair and its expected block count.
metrics = {}
for rid, cfg in TARGETS.items():
    raw_path = RAW_DIR / cfg["raw"]
    val_path = VALIDATED_DIR / cfg["validated"]

    assert raw_path.exists(), f"Missing raw file: {raw_path}"
    assert val_path.exists(), f"Missing validated file: {val_path}"
    assert sha256_file(raw_path) == sha256_file(val_path), (
        f"Raw/validated hash mismatch for {rid}"
    )

    result = inspect_srt(val_path)
    assert result[0] == cfg["blocks"], (
        f"Unexpected block count for {rid}: {result[0]}"
    )
    metrics[rid] = result

# Build five new rows using metadata from the locked master.
new_rows = []
for rid in TARGET_IDS:
    m = master.loc[master_ids == rid].iloc[0]
    row = {col: "" for col in manifest.columns}

    for col in ["movie_record_id", "year", "rank", "movie_title",
                "language", "tmdb_id", "imdb_id"]:
        if col in row and col in master.columns:
            row[col] = m[col]

    if rid in TARGETS:
        cfg = TARGETS[rid]
        blocks, words, devanagari, latin, _ = metrics[rid]
        raw_path = RAW_DIR / cfg["raw"]
        val_path = VALIDATED_DIR / cfg["validated"]

        row.update({
            "subtitle_language": "en",
            "subtitle_type": "English_fallback",
            "subtitle_source": "OpenSubtitles",
            "source_id": cfg["file_id"],
            "source_url": (
                f"https://www.opensubtitles.com/en/subtitles/{cfg['file_id']}"
            ),
            "raw_subtitle_file": str(raw_path),
            "validated_subtitle_file": str(val_path),
            "acquisition_status": cfg["status"],
            "validation_status": "Accepted_with_quality_flags",
            "dialogue_blocks": blocks,
            "word_count": words,
            "character_count": devanagari + latin,
            "encoding_status": "UTF-8",
            "quality_status": "Accepted_with_quality_flags",
            "notes": (
                f"Batch 2. English fallback; source subtitle file "
                f"{cfg['file_id']}. Raw and validated SHA-256 match. "
                f"Screening endpoint coverage approximately "
                f"{(metrics[rid][0] and cfg['runtime'])} min reference; "
                f"runtime-based coverage is approximate. "
                f"Devanagari characters={devanagari}; Latin characters={latin}. "
                f"SHA256={sha256_file(val_path)}."
            ),
        })
    else:
        # Anniyan: keep raw candidate for audit, but do not accept as Tamil.
        raw_path = RAW_DIR / "2005_2_Anniyan_candidate_8250919.srt"
        assert raw_path.exists(), f"Missing Anniyan raw candidate: {raw_path}"

        row.update({
            "subtitle_language": "ta",
            "subtitle_type": "Original_language_candidate_rejected",
            "subtitle_source": "OpenSubtitles",
            "source_id": "8250919",
            "source_url": "https://www.opensubtitles.com/en/subtitles/8250919",
            "raw_subtitle_file": str(raw_path),
            "validated_subtitle_file": "",
            "acquisition_status": "Rejected_language_mismatch",
            "validation_status": "Rejected_language_mismatch",
            "quality_status": "Rejected_language_mismatch",
            "notes": (
                "Batch 2 candidate rejected for Tamil-language analysis: "
                "candidate contained zero Tamil-script characters. "
                "Raw candidate retained for audit; no validated copy accepted."
            ),
        })

    new_rows.append(row)

new_df = pd.DataFrame(new_rows, columns=manifest.columns)
combined = pd.concat([manifest, new_df], ignore_index=True)

# Final safety checks before writing.
combined_ids = combined["movie_record_id"].astype("string").str.strip()
assert len(combined) == len(manifest) + 5
assert combined_ids.is_unique
assert all((combined_ids == rid).sum() == 1 for rid in TARGET_IDS)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = BASE / (
    f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_batch02_{timestamp}.csv"
)
temp_path = MANIFEST_PATH.with_suffix(".batch02.tmp.csv")

# Preserve the exact pre-update manifest, then atomically replace it.
manifest.to_csv(backup_path, index=False)
combined.to_csv(temp_path, index=False)
os.replace(temp_path, MANIFEST_PATH)

# Re-read and verify persisted result.
saved = pd.read_csv(
    MANIFEST_PATH,
    dtype={"movie_record_id": "string", "source_id": "string"}
)
saved_ids = saved["movie_record_id"].astype("string").str.strip()

assert len(saved) == len(manifest) + 5
assert saved_ids.is_unique
assert all((saved_ids == rid).sum() == 1 for rid in TARGET_IDS)

print("BATCH 2 MANIFEST UPDATE SUCCESSFUL")
print("Rows before:", len(manifest))
print("Rows after:", len(saved))
print("Backup:", backup_path)
print("\nAdded records:")
display(
    saved.loc[saved_ids.isin(TARGET_IDS), [
        "movie_record_id", "movie_title", "subtitle_language",
        "acquisition_status", "validation_status",
        "dialogue_blocks", "word_count", "quality_status"
    ]]
)


In [ ]:

from pathlib import Path
from zipfile import ZipFile, BadZipFile
import hashlib
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
SUB_ROOT = ROOT / "01_data/processed/subtitle_corpus_v2"

ZIP_DIR = SUB_ROOT / "raw/batch03_zip_originals"
STAGING_DIR = SUB_ROOT / "raw/batch03_extracted"
STAGING_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_DIR.exists():
    raise FileNotFoundError(
        f"Upload your 10 ZIP files to this folder first:\n{ZIP_DIR}"
    )

zip_files = sorted(ZIP_DIR.glob("*.zip"))

if not zip_files:
    raise FileNotFoundError(
        f"No ZIP files found in {ZIP_DIR}"
    )

expected_movies = [
    "Lage Raho Munnabhai",
    "Krrish",
    "Rang De Basanti",
    "Fanaa",
    "Om Shanti Om",
    "Sivaji",
    "Welcome",
    "Chak De",
    "Partner",
    "Ghajini",
]

inventory = []
errors = []

for zip_path in zip_files:
    try:
        with ZipFile(zip_path, "r") as zf:
            members = [
                item for item in zf.infolist()
                if not item.is_dir()
            ]
            srt_members = [
                item for item in members
                if Path(item.filename).suffix.lower() == ".srt"
            ]

            print(f"\nZIP: {zip_path.name}")
            print(f"  Files inside: {len(members)}")
            print(f"  SRT files: {len(srt_members)}")

            for item in srt_members:
                # Use only the basename, preventing ZIP path traversal.
                basename = Path(item.filename).name
                target = STAGING_DIR / (
                    f"{zip_path.stem}__{basename}"
                )

                content = zf.read(item)
                target.write_bytes(content)

                inventory.append({
                    "zip_filename": zip_path.name,
                    "srt_inside_zip": item.filename,
                    "staged_filename": target.name,
                    "size_bytes": len(content),
                    "sha256": hashlib.sha256(content).hexdigest(),
                })
                print(f"    Extracted: {target.name}")

            if not srt_members:
                inventory.append({
                    "zip_filename": zip_path.name,
                    "srt_inside_zip": None,
                    "staged_filename": None,
                    "size_bytes": None,
                    "sha256": None,
                })

    except BadZipFile:
        errors.append({
            "zip_filename": zip_path.name,
            "error": "Invalid or corrupted ZIP archive",
        })
    except Exception as exc:
        errors.append({
            "zip_filename": zip_path.name,
            "error": str(exc)[:300],
        })

inventory_df = pd.DataFrame(inventory)

print("\n" + "=" * 60)
print("BATCH 03 ZIP INVENTORY")
print("=" * 60)
print("ZIP files found:", len(zip_files))
print("Expected movie count:", len(expected_movies))
print("SRT files extracted:", sum(
    1 for row in inventory if row["staged_filename"]
))
print("Extraction errors:", len(errors))
print("Staging folder:", STAGING_DIR)

if not inventory_df.empty:
    display(inventory_df)

if errors:
    display(pd.DataFrame(errors))

print("\nNext: review the ZIP-to-movie mapping before validation.")


In [ ]:

# ============================================================
# BATCH 03 — FINAL VALIDATION + VALIDATED COPIES + MANIFEST
# Original ZIPs and staged source subtitles remain untouched.
# Locked master/mapping/metadata/features files are not modified.
# ============================================================

from pathlib import Path
from datetime import datetime
from zipfile import ZipFile
import re
import hashlib
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
SUB_ROOT = ROOT / "01_data/processed/subtitle_corpus_v2"
RAW_DIR = SUB_ROOT / "raw"
STAGING_DIR = RAW_DIR / "batch03_extracted"
VALIDATED_DIR = SUB_ROOT / "validated"
MANIFEST_PATH = SUB_ROOT / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

VALIDATED_DIR.mkdir(parents=True, exist_ok=True)

# Verified project identities and source ZIP filename prefixes
movies = [
    ("2006_2", "Lage Raho Munnabhai", 2006, 2, "Hindi", "21461",
     "lage-raho-munna-bhai_english-2179411", "LAGE_RAHO_MUNNABHAI", 144),
    ("2006_3", "Krrish", 2006, 3, "Hindi", "32740",
     "SUBDL.com__krrish444651", "KRRISH", 175),
    ("2006_4", "Rang De Basanti", 2006, 4, "Hindi", "7913",
     "SUBDL.com__rang.de.basanti1143457", "RANG_DE_BASANTI", 157),
    ("2006_5", "Fanaa", 2006, 5, "Hindi", "8453",
     "SUBDL.com__fanaa67793", "FANAA", 168),
    ("2007_1", "Om Shanti Om", 2007, 1, "Hindi", "8079",
     "SUBDL.com__om.shanti.om952462", "OM_SHANTI_OM", 162),
    ("2007_2", "Sivaji: The Boss", 2007, 2, "Tamil", "24049",
     "sivaji-sivaji-the-boss_english-2229486", "SIVAJI_THE_BOSS", 185),
    ("2007_3", "Welcome", 2007, 3, "Hindi", "20294",
     "welcome_english-2561737", "WELCOME", 157),
    ("2007_4", "Chak De! India", 2007, 4, "Hindi", "14163",
     "SUBDL.com__chak.de.india388776", "CHAK_DE_INDIA", 153),
    ("2007_5", "Partner", 2007, 5, "Hindi", "27298",
     "SUBDL.com__partner386977", "PARTNER", 143),
    ("2008_1", "Ghajini", 2008, 1, "Hindi", "14070",
     "SUBDL.com__ghajini.2008212177", "GHAJINI", 183),
]

def parse_timestamp(value):
    match = re.search(
        r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})",
        value
    )
    if not match:
        return None
    h, m, s, ms = match.groups()
    return (
        int(h) * 3600 + int(m) * 60 + int(s)
        + int(ms.ljust(3, "0")[:3]) / 1000
    )

def format_timestamp(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    milliseconds = round((secs - int(secs)) * 1000)
    if milliseconds == 1000:
        secs = int(secs) + 1
        milliseconds = 0
    else:
        secs = int(secs)
    return (
        f"{hours:02}:{minutes:02}:{secs:02},"
        f"{milliseconds:03}"
    )

def decode_subtitle(path):
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return raw.decode(encoding), encoding, raw
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Cannot decode {path.name}")

def clean_srt(source_text, strip_markup=False):
    text = source_text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = re.split(r"\n\s*\n", text.strip()) if text.strip() else []

    output_blocks = []
    malformed_dropped = 0
    reversed_dropped = 0
    valid_entries = []

    for original_number, block in enumerate(blocks, start=1):
        lines = [line.strip("\ufeff") for line in block.splitlines()]
        timing_idx = next(
            (i for i, line in enumerate(lines) if "-->" in line),
            None
        )

        if timing_idx is None:
            malformed_dropped += 1
            continue

        timing_parts = lines[timing_idx].split("-->")
        if len(timing_parts) != 2:
            malformed_dropped += 1
            continue

        start = parse_timestamp(timing_parts[0])
        end = parse_timestamp(timing_parts[1])

        if start is None or end is None:
            malformed_dropped += 1
            continue

        if end < start:
            reversed_dropped += 1
            continue

        dialogue_lines = [
            line for line in lines[timing_idx + 1:]
            if line.strip()
        ]

        if strip_markup:
            cleaned_lines = []
            for line in dialogue_lines:
                line = re.sub(r"\{\\[^}]*\}", "", line)
                line = re.sub(r"</?(?:font|i|b|u|s|ruby|rt)\b[^>]*>", "", line, flags=re.I)
                line = re.sub(r"<[^>]{1,100}>", "", line)
                cleaned_lines.append(line.strip())
            dialogue_lines = cleaned_lines

        # Keep the original timing, dialogue, and ordering for valid entries.
        block_lines = [
            str(len(valid_entries) + 1),
            f"{format_timestamp(start)} --> {format_timestamp(end)}",
            *dialogue_lines
        ]
        output_blocks.append("\n".join(block_lines))

        valid_entries.append({
            "original_block": original_number,
            "start": start,
            "end": end,
            "text": " ".join(dialogue_lines).strip()
        })

    if not valid_entries:
        raise ValueError("No valid subtitle entries remain.")

    cleaned_text = "\n\n".join(output_blocks) + "\n"
    all_dialogue = " ".join(x["text"] for x in valid_entries)
    words = re.findall(r"\b[\w’'-]+\b", all_dialogue, flags=re.UNICODE)

    ordered = sorted(valid_entries, key=lambda x: x["start"])
    gaps = []
    overlaps = 0
    for previous, current in zip(ordered, ordered[1:]):
        delta = current["start"] - previous["end"]
        if delta < -0.5:
            overlaps += 1
        elif delta > 120:
            gaps.append(delta)

    # Check ordering in the original file, not only chronological sorting.
    out_of_order = sum(
        current["start"] < previous["start"]
        for previous, current in zip(valid_entries, valid_entries[1:])
    )

    first_start = min(x["start"] for x in valid_entries)
    last_end = max(x["end"] for x in valid_entries)

    return {
        "text": cleaned_text,
        "blocks": len(valid_entries),
        "word_count": len(words),
        "character_count": len(all_dialogue),
        "first_start": first_start,
        "last_end": last_end,
        "gaps_over_2min": len(gaps),
        "largest_gap_sec": round(max(gaps), 1) if gaps else 0,
        "overlaps": overlaps,
        "out_of_order": out_of_order,
        "malformed_dropped": malformed_dropped,
        "reversed_dropped": reversed_dropped,
        "devanagari_chars": len(re.findall(r"[\u0900-\u097F]", all_dialogue)),
        "tamil_chars": len(re.findall(r"[\u0B80-\u0BFF]", all_dialogue)),
        "latin_chars": len(re.findall(r"[A-Za-z]", all_dialogue)),
        "replacement_chars": all_dialogue.count("\ufffd"),
    }

# ---------- Preflight: map and validate every file first ----------
prepared = []
problems = []

for record_id, title, year, rank, source_language, tmdb_id, prefix, slug, runtime in movies:
    matches = sorted(
        p for p in STAGING_DIR.glob("*.srt")
        if p.name.startswith(prefix)
    )

    if len(matches) != 1:
        problems.append(
            f"{record_id} {title}: expected exactly one staged SRT; found {len(matches)}"
        )
        continue

    source_path = matches[0]
    try:
        source_text, encoding, raw_bytes = decode_subtitle(source_path)

        # Sivaji has extensive HTML/ASS formatting plus two reversed entries.
        # Clean formatting only in its separate validated copy.
        result = clean_srt(
            source_text,
            strip_markup=(record_id == "2007_2")
        )

        runtime_seconds = runtime * 60
        coverage = result["last_end"] / runtime_seconds * 100

        if result["blocks"] < 500 or result["word_count"] < 3000:
            problems.append(
                f"{record_id} {title}: insufficient dialogue after cleanup "
                f"({result['blocks']} blocks, {result['word_count']} words)"
            )
            continue

        if result["replacement_chars"] > 0:
            problems.append(
                f"{record_id} {title}: replacement characters found"
            )
            continue

        # Quality flags are retained rather than hiding known limitations.
        flags = ["English_fallback"]
        if result["gaps_over_2min"]:
            flags.append("Large_timing_gaps")
        if result["overlaps"]:
            flags.append("Overlapping_timestamps")
        if result["out_of_order"]:
            flags.append("Original_order_anomalies")
        if result["malformed_dropped"] or result["reversed_dropped"]:
            flags.append("Invalid_blocks_removed")
        if record_id == "2007_2":
            flags.append("Formatting_stripped")
            flags.append("Timing_review_flag")
        if record_id == "2007_4":
            flags.append("Opening_source_credit_present")
        if coverage < 85:
            flags.append("Endpoint_coverage_review")

        validated_path = VALIDATED_DIR / f"{record_id}_{slug}_ENGLISH.srt"

        prepared.append({
            "record_id": record_id,
            "title": title,
            "year": year,
            "rank": rank,
            "source_language": source_language,
            "tmdb_id": tmdb_id,
            "source_path": source_path,
            "source_encoding": encoding,
            "source_sha256": hashlib.sha256(raw_bytes).hexdigest(),
            "validated_path": validated_path,
            "validated_text": result["text"],
            "validated_sha256": hashlib.sha256(
                result["text"].encode("utf-8")
            ).hexdigest(),
            "runtime_reference_min": runtime,
            "endpoint_coverage_pct_approx": round(coverage, 1),
            "flags": flags,
            **{k: v for k, v in result.items() if k != "text"}
        })

    except Exception as exc:
        problems.append(f"{record_id} {title}: {type(exc).__name__}: {exc}")

if problems:
    print("PREFLIGHT FAILED — no manifest updates made.")
    for problem in problems:
        print("-", problem)
    raise RuntimeError("Resolve preflight problems before updating the manifest.")

assert len(prepared) == 10, f"Expected 10 validated candidates, got {len(prepared)}"

# ---------- Check manifest before any update ----------
if MANIFEST_PATH.exists():
    manifest = pd.read_csv(
        MANIFEST_PATH,
        dtype={
            "movie_record_id": "string",
            "source_id": "string",
            "source_url": "string",
        }
    )
else:
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

required_columns = [
    "movie_record_id", "year", "rank", "movie_title", "language",
    "tmdb_id", "imdb_id", "subtitle_language", "subtitle_type",
    "subtitle_source", "source_id", "source_url", "raw_subtitle_file",
    "validated_subtitle_file", "acquisition_status", "validation_status",
    "title_verified", "year_verified", "movie_identity_verified",
    "language_verified", "dialogue_blocks", "word_count",
    "character_count", "encoding_status", "quality_status", "notes"
]

for column in required_columns:
    if column not in manifest.columns:
        manifest[column] = pd.NA

# Ensure one manifest row per movie record ID.
if manifest["movie_record_id"].duplicated().any():
    duplicates = manifest.loc[
        manifest["movie_record_id"].duplicated(keep=False),
        "movie_record_id"
    ].tolist()
    raise ValueError(f"Duplicate manifest record IDs found: {duplicates}")

# ---------- Write separate validated copies ----------
for item in prepared:
    item["validated_path"].write_text(
        item["validated_text"],
        encoding="utf-8",
        newline="\n"
    )

# Verify every written copy by hash before updating the manifest.
for item in prepared:
    actual_hash = hashlib.sha256(
        item["validated_path"].read_bytes()
    ).hexdigest()
    expected_hash = hashlib.sha256(
        item["validated_text"].encode("utf-8")
    ).hexdigest()
    if actual_hash != expected_hash:
        raise IOError(f"Hash verification failed: {item['validated_path']}")

# ---------- Back up manifest before updating ----------
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = SUB_ROOT / (
    f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_batch03_{stamp}.csv"
)
manifest.to_csv(backup_path, index=False)

# ---------- Update/append only the 10 target records ----------
for item in prepared:
    record_id = item["record_id"]
    existing = manifest.index[
        manifest["movie_record_id"].astype("string") == record_id
    ]

    if len(existing) == 1:
        idx = existing[0]
    else:
        idx = len(manifest)
        manifest.loc[idx, :] = pd.NA
        manifest.loc[idx, "movie_record_id"] = record_id

    quality_status = (
        "Accepted_English_fallback_with_flags"
        if item["flags"] else "Accepted_English_fallback"
    )

    notes = (
        "SUBDL English fallback; source ZIP/SRT preserved. "
        f"Source file: {item['source_path'].name}. "
        f"Source encoding: {item['source_encoding']}. "
        f"Approximate endpoint coverage: "
        f"{item['endpoint_coverage_pct_approx']}%. "
        f"Cleanup: removed {item['malformed_dropped']} malformed and "
        f"{item['reversed_dropped']} reversed-timing blocks; "
        f"formatting stripped only for Sivaji. "
        f"Flags: {', '.join(item['flags']) if item['flags'] else 'none'}."
    )

    manifest.loc[idx, "year"] = item["year"]
    manifest.loc[idx, "rank"] = item["rank"]
    manifest.loc[idx, "movie_title"] = item["title"]
    manifest.loc[idx, "language"] = item["source_language"]
    manifest.loc[idx, "tmdb_id"] = item["tmdb_id"]
    manifest.loc[idx, "subtitle_language"] = "English"
    manifest.loc[idx, "subtitle_type"] = "English_fallback"
    manifest.loc[idx, "subtitle_source"] = "SUBDL"
    manifest.loc[idx, "source_id"] = item["source_path"].name
    manifest.loc[idx, "source_url"] = ""
    manifest.loc[idx, "raw_subtitle_file"] = str(item["source_path"])
    manifest.loc[idx, "validated_subtitle_file"] = str(item["validated_path"])
    manifest.loc[idx, "acquisition_status"] = "Accepted_English_fallback"
    manifest.loc[idx, "validation_status"] = "Validated_with_quality_flags"
    manifest.loc[idx, "title_verified"] = True
    manifest.loc[idx, "year_verified"] = True
    manifest.loc[idx, "movie_identity_verified"] = True
    manifest.loc[idx, "language_verified"] = True
    manifest.loc[idx, "dialogue_blocks"] = item["blocks"]
    manifest.loc[idx, "word_count"] = item["word_count"]
    manifest.loc[idx, "character_count"] = item["character_count"]
    manifest.loc[idx, "encoding_status"] = (
        f"Decoded_{item['source_encoding']}_and_saved_UTF8"
    )
    manifest.loc[idx, "quality_status"] = quality_status
    manifest.loc[idx, "notes"] = notes

manifest.to_csv(MANIFEST_PATH, index=False)

# ---------- Reload and verify persisted updates ----------
check = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string",
    }
)

summary = []
for item in prepared:
    row = check.loc[
        check["movie_record_id"].astype("string") == item["record_id"]
    ]
    assert len(row) == 1, f"Manifest row verification failed: {item['record_id']}"
    row = row.iloc[0]

    assert Path(row["validated_subtitle_file"]).is_file()
    assert row["acquisition_status"] == "Accepted_English_fallback"

    summary.append({
        "movie_record_id": item["record_id"],
        "movie_title": item["title"],
        "validated_blocks": item["blocks"],
        "word_count": item["word_count"],
        "malformed_removed": item["malformed_dropped"],
        "reversed_timing_removed": item["reversed_dropped"],
        "overlaps_remaining": item["overlaps"],
        "gaps_over_2min": item["gaps_over_2min"],
        "coverage_pct_approx": item["endpoint_coverage_pct_approx"],
        "quality_flags": ", ".join(item["flags"]),
        "manifest_status": row["acquisition_status"],
        "validated_file": Path(row["validated_subtitle_file"]).name,
    })

print("=" * 72)
print("BATCH 03 — VALIDATED COPIES AND MANIFEST UPDATE COMPLETE")
print("=" * 72)
print("Movies processed:", len(prepared))
print("Manifest backup:", backup_path)
print("Updated manifest:", MANIFEST_PATH)
print("Validated directory:", VALIDATED_DIR)
print("\nOriginal ZIPs and staged source SRTs were preserved.")
print("Locked master, mapping, metadata, and feature datasets were not modified.")

display(pd.DataFrame(summary))


In [ ]:
from pathlib import Path
from datetime import datetime
import re
import hashlib
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
VALIDATED = ROOT / "validated"
MANIFEST = ROOT / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

record_id = "2006_2"
old_file = VALIDATED / "2006_2_LAGE_RAHO_MUNNABHAI_ENGLISH.srt"
new_file = VALIDATED / "2006_2_LAGE_RAHO_MUNNABHAI_ENGLISH_v2.srt"

assert old_file.exists(), f"Original validated file missing: {old_file}"
assert MANIFEST.exists(), f"Manifest missing: {MANIFEST}"
assert not new_file.exists(), f"Correction file already exists: {new_file}"

def timestamp_seconds(value):
    value = value.strip().replace(",", ".")
    parts = value.split(":")
    if len(parts) != 3:
        raise ValueError(f"Unexpected timestamp: {value}")
    h, m, s = parts
    return int(h) * 3600 + int(m) * 60 + float(s)

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Parse the existing validated file without modifying it.
text = old_file.read_text(encoding="utf-8-sig", errors="replace")
blocks = re.split(r"\n\s*\n", text.strip())
timestamp_re = re.compile(
    r"(\d{2,}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2,}:\d{2}:\d{2}[,.]\d{3})"
)

kept = []
removed = []
for block in blocks:
    match = timestamp_re.search(block)
    if not match:
        continue

    start_s = timestamp_seconds(match.group(1))
    end_s = timestamp_seconds(match.group(2))

    # A 31-hour timestamp cannot belong to this ~144-minute film.
    # Also reject reversed timings and timestamps beyond six hours.
    if start_s > 6 * 3600 or end_s > 6 * 3600 or end_s < start_s:
        removed.append({
            "start": match.group(1),
            "end": match.group(2),
            "reason": "Implausible timestamp or reversed timing"
        })
        continue

    # Preserve subtitle text and timing; normalize numbering only.
    body = timestamp_re.sub(
        f"{match.group(1)} --> {match.group(2)}", block, count=1
    )
    kept.append(body)

assert removed, (
    "No implausible timestamps were found. Stop and inspect the file manually; "
    "no output or manifest changes have been made."
)
assert len(kept) > 500, f"Unexpectedly few blocks remain: {len(kept)}"

# Reindex the retained subtitle blocks.
renumbered = []
for i, block in enumerate(kept, start=1):
    lines = block.splitlines()
    if lines and lines[0].strip().isdigit():
        lines = lines[1:]
    renumbered.append(f"{i}\n" + "\n".join(lines))

new_text = "\n\n".join(renumbered) + "\n"
new_file.write_text(new_text, encoding="utf-8")

# Re-read and calculate corrected metrics.
check_text = new_file.read_text(encoding="utf-8")
check_blocks = re.split(r"\n\s*\n", check_text.strip())
times = []
words = 0

for block in check_blocks:
    match = timestamp_re.search(block)
    if not match:
        continue
    start_s = timestamp_seconds(match.group(1))
    end_s = timestamp_seconds(match.group(2))
    times.append((start_s, end_s))

    dialogue = timestamp_re.sub("", block)
    dialogue = re.sub(r"<[^>]*>", " ", dialogue)
    dialogue = re.sub(r"\{[^}]*\}", " ", dialogue)
    dialogue = re.sub(r"^\s*\d+\s*", "", dialogue)
    words += len(re.findall(r"\b[\w'-]+\b", dialogue))

assert len(times) == len(check_blocks), "Some output blocks failed timing validation."
assert all(s <= e for s, e in times), "Reversed timing remains."
assert max(e for _, e in times) <= 6 * 3600, "Implausible timestamp remains."

# Back up manifest BEFORE changing it.
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = ROOT / f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_lage_raho_fix_{stamp}.csv"

manifest = pd.read_csv(
    MANIFEST,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string",
        "tmdb_id": "string"
    }
)
assert (manifest["movie_record_id"] == record_id).sum() == 1, (
    "Expected exactly one Lage Raho manifest row."
)
manifest.to_csv(backup, index=False)

idx = manifest.index[manifest["movie_record_id"] == record_id][0]
last_end = max(e for _, e in times)
first_start = min(s for s, _ in times)
runtime_seconds = 144 * 60
coverage_pct = last_end / runtime_seconds * 100

manifest.at[idx, "validated_subtitle_file"] = str(new_file)
manifest.at[idx, "dialogue_blocks"] = len(times)
manifest.at[idx, "word_count"] = words
manifest.at[idx, "character_count"] = len(
    re.sub(r"\s+", "", re.sub(r"\d+\s*\n", "", check_text))
)
manifest.at[idx, "quality_status"] = "Accepted_English_fallback_with_timestamp_correction"
old_notes = str(manifest.at[idx, "notes"])
manifest.at[idx, "notes"] = (
    old_notes
    + " | Correction: removed implausible 31-hour timestamp block; "
      "previous validated file preserved; corrected v2 file used."
)

manifest.to_csv(MANIFEST, index=False)

# Reload and verify the changed row and file.
verified = pd.read_csv(
    MANIFEST,
    dtype={"movie_record_id": "string", "tmdb_id": "string"}
)
row = verified.loc[verified["movie_record_id"] == record_id].iloc[0]

assert Path(row["validated_subtitle_file"]).exists()
assert row["validated_subtitle_file"] == str(new_file)

print("=" * 72)
print("LAGE RAHO MUNNABHAI — TIMESTAMP CORRECTION")
print("=" * 72)
print("Implausible blocks removed:", len(removed))
print("Removed timing(s):", removed)
print("Corrected subtitle blocks:", len(times))
print("Recalculated word count:", words)
print(f"First subtitle: {first_start / 60:.2f} min")
print(f"Last subtitle:  {last_end / 60:.2f} min")
print(f"Approximate endpoint coverage: {coverage_pct:.1f}%")
print("Original validated file preserved:", old_file.exists())
print("Corrected file:", new_file)
print("Manifest backup:", backup)
print("Manifest updated and verified:", True)

In [ ]:
# ============================================================
# OPEN SUBTITLES — REFRESH TOKEN AND CHECK ACCOUNT STATUS
# No subtitle downloads. No project files or manifest changes.
# ============================================================

import requests
from google.colab import userdata

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
USERNAME = userdata.get("OPEN_SUBTITLES_USERNAME")
PASSWORD = userdata.get("OPEN_SUBTITLES_PASSWORD")

if not all([API_KEY, USERNAME, PASSWORD]):
    raise ValueError(
        "Missing OPEN_SUBTITLES_API_KEY, "
        "OPEN_SUBTITLES_USERNAME, or OPEN_SUBTITLES_PASSWORD in Colab Secrets."
    )

BASE_URL = "https://api.opensubtitles.com/api/v1"

headers = {
    "Api-Key": API_KEY,
    "User-Agent": "ICMB130SubtitleResearch/1.0",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

# Step 1: Log in to obtain a fresh token.
login_response = requests.post(
    f"{BASE_URL}/login",
    headers=headers,
    json={"username": USERNAME, "password": PASSWORD},
    timeout=30,
)

print("Login HTTP status:", login_response.status_code)

try:
    login_data = login_response.json()
except ValueError:
    raise RuntimeError("Login returned a non-JSON response.")

if login_response.status_code != 200:
    print("Login error:", login_data.get("message", login_data.get("errors", "Unknown error")))
    raise RuntimeError("Login failed. No downloads were attempted.")

fresh_token = login_data.get("token")

if not fresh_token:
    raise RuntimeError("Login response did not contain a token.")

print("Fresh token obtained: YES (token not displayed)")

# Step 2: Check account information using the fresh token.
auth_headers = {
    **headers,
    "Authorization": f"Bearer {fresh_token}",
}

info_response = requests.get(
    f"{BASE_URL}/infos/user",
    headers=auth_headers,
    timeout=30,
)

print("\nAccount info HTTP status:", info_response.status_code)

try:
    info_data = info_response.json()
except ValueError:
    print("Account info response:", info_response.text[:500])
    raise RuntimeError("Account info returned a non-JSON response.")

if info_response.status_code == 200:
    account = info_data.get("data", {})
    print("Account information retrieved successfully.")

    # Display only relevant non-secret fields, if supplied.
    for key in (
        "allowed_downloads",
        "downloads_count",
        "remaining_downloads",
        "level",
    ):
        if key in account:
            print(f"{key}: {account[key]}")

else:
    print("Account info response:", info_data)

print("\nNo subtitles downloaded.")
print("Manifest unchanged.")
print("Locked datasets unchanged.")
print(
    "\nKeep this fresh token in memory for the current Colab session. "
    "Do not print or share it."
)

In [ ]:

from google.colab import drive
from pathlib import Path

# Mount Drive using the existing mount location
drive.mount("/content/drive")

# Check the project in the standard Colab mount
PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")

print("\nProject exists:", PROJECT.exists())

if PROJECT.exists():
    print("Project folders:", [p.name for p in PROJECT.iterdir()])
else:
    print("\nProject not found at this path.")
    print("MyDrive exists:", Path("/content/drive/MyDrive").exists())
    if Path("/content/drive/MyDrive").exists():
        print("Possible project folders:")
        for p in Path("/content/drive/MyDrive").iterdir():
            if "IndianCinema" in p.name or "Movie" in p.name:
                print(p)


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

print("=" * 60)
print("NOTEBOOK 8 — WORKSPACE VERIFICATION")
print("=" * 60)

for name, path in [
    ("Project", PROJECT),
    ("Subtitle workspace", WORK),
    ("Raw folder", RAW),
    ("Validated folder", VALIDATED),
    ("Manifest", MANIFEST),
]:
    print(f"{name}: {path.exists()}")

if MANIFEST.exists():
    df = pd.read_csv(
        MANIFEST,
        dtype={
            "movie_record_id": "string",
            "source_id": "string",
            "source_url": "string",
            "tmdb_id": "string",
        },
    )

    print("\nManifest rows:", len(df))

    batch04_ids = [
        "2008_2", "2008_3", "2008_4", "2008_5",
        "2009_1", "2009_2", "2009_3", "2009_4",
        "2009_5", "2010_1",
    ]

    cols = [
        c for c in [
            "movie_record_id", "movie_title", "language",
            "subtitle_language", "source_id",
            "raw_subtitle_file", "validated_subtitle_file",
            "acquisition_status", "validation_status",
        ] if c in df.columns
    ]

    print("\nBatch 04 manifest records:")
    print(
        df[df["movie_record_id"].isin(batch04_ids)][cols]
        .to_string(index=False)
    )

    if RAW.exists():
        print("\nRaw files:", len(list(RAW.iterdir())))
    if VALIDATED.exists():
        print("Validated files:", len(list(VALIDATED.iterdir())))
else:
    print("\nManifest not found at the expected path.")
    print("Checking for the manifest within the project...")
    for p in PROJECT.rglob("ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"):
        print("Found:", p)

print("\nRead-only verification complete.")


In [ ]:

print("MANIFEST ID FORMAT")
print("Unique record IDs:", df["movie_record_id"].nunique())
print("\nAll manifest records:")
print(
    df[[
        "movie_record_id",
        "year",
        "rank",
        "movie_title",
        "language",
        "acquisition_status",
        "validation_status",
    ]].to_string(index=False)
)

print("\nExisting subtitle files that may belong to Batch 04:")
for folder in [RAW, VALIDATED]:
    print(f"\n{folder.name}/")
    for p in sorted(folder.iterdir()):
        print(p.name)


In [ ]:

from pathlib import Path

candidate_ids = [
    4292288, 4197321, 4246518, 4093059, 8326640,
    11596863, 42380, 4936757, 4210730, 11186717
]

print("BATCH 04 — LOCAL FILE CHECK")
print("=" * 55)

for candidate_id in candidate_ids:
    matches = []
    for folder in [RAW, VALIDATED]:
        if folder.exists():
            matches.extend(
                str(p.relative_to(WORK))
                for p in folder.rglob("*")
                if p.is_file() and str(candidate_id) in p.name
            )
    print(f"{candidate_id}: {matches if matches else 'No matching local file'}")

print("\nManifest last records:")
print(
    df[["movie_record_id", "movie_title", "acquisition_status"]]
    .tail(5)
    .to_string(index=False)
)

print("\nNo downloads or modifications performed.")


In [ ]:

import requests
from google.colab import userdata

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
USERNAME = userdata.get("OPEN_SUBTITLES_USERNAME")
PASSWORD = userdata.get("OPEN_SUBTITLES_PASSWORD")

BASE_URL = "https://api.opensubtitles.com/api/v1"
HEADERS = {
    "Api-Key": API_KEY,
    "User-Agent": "ICMB130SubtitleResearch/1.0",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

# Log in without printing credentials or the token
login = requests.post(
    f"{BASE_URL}/login",
    headers=HEADERS,
    json={"username": USERNAME, "password": PASSWORD},
    timeout=30,
)

print("Login HTTP status:", login.status_code)

if login.status_code == 200:
    fresh_token = login.json().get("token")
    auth_headers = {
        **HEADERS,
        "Authorization": f"Bearer {fresh_token}",
    }

    info = requests.get(
        f"{BASE_URL}/infos/user",
        headers=auth_headers,
        timeout=30,
    )

    print("Account info HTTP status:", info.status_code)

    if info.status_code == 200:
        account = info.json().get("data", {})
        print("Allowed downloads:", account.get("allowed_downloads"))
        print("Downloads used:", account.get("downloads_count"))
        print("Downloads remaining:", account.get("remaining_downloads"))
        print("Account level:", account.get("level"))
    else:
        print("Could not retrieve account allowance.")
        print("Response:", info.text[:500])
else:
    print("Login failed. Response:", login.text[:500])

print("\nNo subtitle download requests were made.")


In [ ]:

import io
import re
import zipfile
import requests
from pathlib import Path

# Existing Notebook 8 variables: PROJECT, WORK, RAW, fresh_token
API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
BASE_URL = "https://api.opensubtitles.com/api/v1"

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {fresh_token}",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

candidate_id = 4292288
movie_prefix = "2008_2_Rab_Ne_Bana_Di_Jodi"

# Request one download link
response = requests.post(
    f"{BASE_URL}/download",
    headers=headers,
    json={"file_id": candidate_id},
    timeout=30,
)

print("Download-link HTTP status:", response.status_code)

if response.status_code != 200:
    try:
        info = response.json()
        print("API message:", info.get("message", "No message"))
        print("Remaining quota:", info.get("remaining", "Not provided"))
    except Exception:
        print("Response:", response.text[:300])
else:
    info = response.json()
    print("Download-link response received.")
    print("Remaining quota reported:", info.get("remaining", "Not provided"))

    link = info.get("link")
    supplied_name = info.get("file_name") or f"{movie_prefix}_candidate_{candidate_id}.srt"

    if not link:
        print("No download URL was supplied; no file saved.")
    else:
        # Fetch the actual subtitle content without printing the temporary URL
        file_response = requests.get(link, timeout=60)
        content = file_response.content

        print("File-response HTTP status:", file_response.status_code)
        print("Content-Type:", file_response.headers.get("Content-Type", "Unknown"))
        print("Bytes received:", len(content))

        if file_response.status_code != 200:
            print("File fetch failed; no file saved.")
        else:
            subtitle_bytes = None
            output_ext = Path(supplied_name).suffix.lower()

            # Handle ZIP responses
            if zipfile.is_zipfile(io.BytesIO(content)):
                with zipfile.ZipFile(io.BytesIO(content)) as zf:
                    choices = [
                        n for n in zf.namelist()
                        if Path(n).suffix.lower() in {".srt", ".ass", ".ssa", ".vtt"}
                        and not n.endswith("/")
                    ]
                    if choices:
                        # Prefer SRT when present
                        choices.sort(key=lambda n: (
                            Path(n).suffix.lower() != ".srt",
                            len(n)
                        ))
                        selected = choices[0]
                        subtitle_bytes = zf.read(selected)
                        output_ext = Path(selected).suffix.lower()
                        print("ZIP subtitle selected:", Path(selected).name)
                    else:
                        print("ZIP contains no recognizable subtitle file.")
            else:
                # Handle direct subtitle text
                text_sample = content[:5000].decode("utf-8-sig", errors="replace")
                looks_like_subtitle = (
                    bool(re.search(
                        r"\d{1,2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*\d{1,2}:\d{2}:\d{2}[,.]\d{3}",
                        text_sample
                    ))
                    or "[Script Info]" in text_sample
                    or text_sample.lstrip().startswith("WEBVTT")
                )

                if looks_like_subtitle:
                    subtitle_bytes = content
                    if output_ext not in {".srt", ".ass", ".ssa", ".vtt"}:
                        output_ext = ".srt"
                else:
                    print("Response does not look like subtitle text.")
                    print("Response preview:", text_sample[:250].replace("\n", " "))

            if subtitle_bytes:
                # Validate that the extracted/direct file contains subtitle timing cues
                decoded = subtitle_bytes.decode("utf-8-sig", errors="replace")
                cue_count = len(re.findall(
                    r"\d{1,2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*\d{1,2}:\d{2}:\d{2}[,.]\d{3}",
                    decoded
                ))

                if cue_count == 0:
                    print("No standard timing cues detected; file not saved.")
                else:
                    RAW.mkdir(parents=True, exist_ok=True)
                    output_path = RAW / f"{movie_prefix}_candidate_{candidate_id}{output_ext}"

                    if output_path.exists():
                        print("File already exists; refusing to overwrite:", output_path.name)
                    else:
                        output_path.write_bytes(subtitle_bytes)
                        print("Saved raw subtitle:", output_path.name)
                        print("Timing cues detected:", cue_count)
                        print("Saved size (bytes):", output_path.stat().st_size)

print("\nManifest and locked datasets were not modified.")


In [ ]:

import requests
import pandas as pd

BASE_URL = "https://api.opensubtitles.com/api/v1"

headers = {
    "Api-Key": userdata.get("OPEN_SUBTITLES_API_KEY"),
    "Authorization": f"Bearer {fresh_token}",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
    "Accept": "application/json",
}

movies = [
    {"title": "Dasavathaaram", "year": 2008, "lang": "ta", "tmdb": 17888},
    {"title": "Singh Is Kinng", "year": 2008, "lang": "hi", "tmdb": 12273},
    {"title": "Jodhaa Akbar", "year": 2008, "lang": "hi", "tmdb": 14073},
    {"title": "3 Idiots", "year": 2009, "lang": "hi", "tmdb": 20453},
    {"title": "Magadheera", "year": 2009, "lang": "te", "tmdb": 23790},
    {"title": "Love Aaj Kal", "year": 2009, "lang": "hi", "tmdb": 19663},
    {"title": "Ajab Prem Ki Ghazab Kahani", "year": 2009, "lang": "hi", "tmdb": 24827},
    {"title": "Wanted", "year": 2009, "lang": "hi", "tmdb": 22717},
    {"title": "Enthiran", "year": 2010, "lang": "ta", "tmdb": 148284},
]

def search_subtitles(movie, language):
    response = requests.get(
        f"{BASE_URL}/subtitles",
        headers=headers,
        params={
            "query": movie["title"],
            "year": movie["year"],
            "languages": language,
            "type": "movie",
            "per_page": 50,
        },
        timeout=30,
    )
    if response.status_code != 200:
        return [], f"HTTP {response.status_code}"
    return response.json().get("data", []), None

def extract_candidates(results, movie, language, fallback=False):
    found = []
    for item in results:
        attr = item.get("attributes", {})
        feature = attr.get("feature_details") or attr.get("feature") or {}
        files = attr.get("files", [])

        title = feature.get("title") or ""
        year = feature.get("year")
        tmdb = feature.get("tmdb_id")
        imdb = feature.get("imdb_id")

        # Prefer matching TMDB identity; require matching title/year
        # if the API omits TMDB metadata.
        title_match = title.strip().casefold() == movie["title"].casefold()
        year_match = str(year) == str(movie["year"])
        tmdb_match = str(tmdb) == str(movie["tmdb"])

        if not tmdb_match and not (title_match and year_match):
            continue

        for f in files:
            found.append({
                "movie": movie["title"],
                "year": movie["year"],
                "requested_language": language,
                "fallback_english": fallback,
                "feature_title": title,
                "feature_year": year,
                "tmdb_id": tmdb,
                "imdb_id": imdb,
                "subtitle_language": attr.get("language"),
                "file_id": f.get("file_id"),
                "release": f.get("file_name") or attr.get("release"),
                "downloads": attr.get("download_count"),
                "hearing_impaired": attr.get("hearing_impaired"),
                "identity_match": "TMDB" if tmdb_match else "Title+year",
            })
    return found

rows = []

for movie in movies:
    print(f"\nSearching original language: {movie['title']} ({movie['year']})")
    results, error = search_subtitles(movie, movie["lang"])

    if error:
        print("Original-language search error:", error)
        continue

    original = extract_candidates(results, movie, movie["lang"])
    # Only use English fallback when no credible original-language candidate exists.
    if original:
        rows.extend(original)
        print("Credible original-language candidates:", len(original))
    else:
        print("No credible original-language candidate; searching English fallback.")
        english_results, english_error = search_subtitles(movie, "en")

        if english_error:
            print("English fallback search error:", english_error)
            continue

        english = extract_candidates(
            english_results, movie, "en", fallback=True
        )
        rows.extend(english)
        print("Credible English fallback candidates:", len(english))

review_df = pd.DataFrame(rows)

if review_df.empty:
    print("\nNo candidates passed the metadata identity filter.")
else:
    review_df = review_df.sort_values(
        ["movie", "fallback_english", "downloads"],
        ascending=[True, True, False],
        na_position="last",
    )
    display(review_df.groupby("movie", group_keys=False).head(5))

print("\nSearch-only review complete. No download requests were made.")


In [ ]:

import io
import re
import zipfile
import requests
from pathlib import Path

BASE_URL = "https://api.opensubtitles.com/api/v1"

# Seven candidates selected after search-only metadata checks.
candidates = [
    (8900291, "2008_3_DASAVATHAARAM", "en"),
    (4246518, "2008_4_SINGH_IS_KINNG", "en"),
    (4093059, "2008_5_JODHAA_AKBAR", "en"),
    (8326640, "2009_1_3_IDIOTS", "hi"),
    (42380, "2009_3_LOVE_AAJ_KAL", "en"),
    (4210730, "2009_5_WANTED", "en"),
    (11186717, "2010_1_ENTHIRAN", "ta"),
]

headers = {
    "Api-Key": userdata.get("OPEN_SUBTITLES_API_KEY"),
    "Authorization": f"Bearer {fresh_token}",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

timing_pattern = re.compile(
    r"\d{1,2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
    r"\d{1,2}:\d{2}:\d{2}[,.]\d{3}"
)

results = []

for file_id, prefix, expected_language in candidates:
    print(f"\n{prefix} | ID {file_id} | Expected language: {expected_language}")

    try:
        # Check existing file first; never overwrite.
        existing = [
            p for p in RAW.glob(f"{prefix}_candidate_{file_id}.*")
            if p.is_file()
        ]
        if existing:
            print("SKIPPED — file already exists:", existing[0].name)
            results.append((prefix, "Already exists", 0))
            continue

        # Request download link.
        r = requests.post(
            f"{BASE_URL}/download",
            headers=headers,
            json={"file_id": file_id},
            timeout=30,
        )

        if r.status_code != 200:
            try:
                data = r.json()
                message = data.get("message", "No API message")
                remaining = data.get("remaining", "unknown")
            except Exception:
                message, remaining = r.text[:200], "unknown"

            print("STOP — link request failed:", r.status_code)
            print("Message:", message, "| Remaining quota:", remaining)
            results.append((prefix, "Link request failed", 0))
            break

        data = r.json()
        link = data.get("link")

        if not link:
            print("STOP — no download link returned.")
            results.append((prefix, "No link", 0))
            break

        # Fetch the file from the supplied temporary link.
        file_response = requests.get(link, timeout=60)
        if file_response.status_code != 200:
            print("STOP — file fetch failed:", file_response.status_code)
            results.append((prefix, "File fetch failed", 0))
            break

        content = file_response.content
        extension = Path(data.get("file_name") or "").suffix.lower()
        subtitle_bytes = None

        # Handle ZIP or direct subtitle text.
        if zipfile.is_zipfile(io.BytesIO(content)):
            with zipfile.ZipFile(io.BytesIO(content)) as zf:
                names = [
                    n for n in zf.namelist()
                    if not n.endswith("/")
                    and Path(n).suffix.lower() in {".srt", ".ass", ".ssa", ".vtt"}
                ]
                if names:
                    names.sort(key=lambda n: Path(n).suffix.lower() != ".srt")
                    chosen = names[0]
                    subtitle_bytes = zf.read(chosen)
                    extension = Path(chosen).suffix.lower()
        else:
            sample = content[:10000].decode("utf-8-sig", errors="replace")
            if (
                timing_pattern.search(sample)
                or "[Script Info]" in sample
                or sample.lstrip().startswith("WEBVTT")
            ):
                subtitle_bytes = content

        if not subtitle_bytes:
            print("STOP — unrecognized subtitle response; no file saved.")
            print("Content-Type:", file_response.headers.get("Content-Type"))
            results.append((prefix, "Unrecognized response", 0))
            break

        decoded = subtitle_bytes.decode("utf-8-sig", errors="replace")
        cue_count = len(timing_pattern.findall(decoded))

        if cue_count == 0:
            print("STOP — no standard timing cues; no file saved.")
            results.append((prefix, "No timing cues", 0))
            break

        if extension not in {".srt", ".ass", ".ssa", ".vtt"}:
            extension = ".srt"

        RAW.mkdir(parents=True, exist_ok=True)
        output_path = RAW / f"{prefix}_candidate_{file_id}{extension}"

        if output_path.exists():
            print("STOP — refusing to overwrite:", output_path.name)
            results.append((prefix, "Overwrite prevented", cue_count))
            break

        output_path.write_bytes(subtitle_bytes)

        print("Saved:", output_path.name)
        print("Timing cues:", cue_count, "| Bytes:", len(subtitle_bytes))
        results.append((prefix, "Downloaded — validation pending", cue_count))

    except Exception as exc:
        print("STOP —", type(exc).__name__, str(exc)[:180])
        results.append((prefix, "Error", 0))
        break

print("\n" + "=" * 60)
print("DOWNLOAD SUMMARY")
for name, status, cues in results:
    print(f"{name}: {status} | cues: {cues}")

print("\nManifest unchanged. Locked datasets untouched.")


In [ ]:
from pathlib import Path
import re

review_files = {
    "3 Idiots": RAW / "2009_1_3_IDIOTS_candidate_8326640.srt",
    "Love Aaj Kal": RAW / "2009_3_LOVE_AAJ_KAL_candidate_42380.srt",
    "Wanted": RAW / "2009_5_WANTED_candidate_4210730.srt",
    "Enthiran": RAW / "2010_1_ENTHIRAN_candidate_11186717.srt",
}

TIME_RE = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def to_seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

for movie, path in review_files.items():
    print("\n" + "=" * 72)
    print(f"{movie} | Exists: {path.exists()}")

    if not path.exists():
        continue

    text = path.read_text(encoding="utf-8-sig", errors="replace")
    blocks = re.split(r"\n\s*\n", text.strip())
    cues = []

    for block in blocks:
        match = TIME_RE.search(block)
        if not match:
            continue

        values = match.groups()
        start = to_seconds(*values[:4])
        end = to_seconds(*values[4:])

        subtitle_text = "\n".join(
            line for line in block.splitlines()
            if "-->" not in line and not line.strip().isdigit()
        ).strip()

        cues.append((start, end, subtitle_text))

    cues.sort(key=lambda x: x[0])

    print("Parsed cues:", len(cues))

    if not cues:
        print("No valid cues found.")
        continue

    print(f"First cue: {cues[0][0] / 60:.2f} min")
    print(f"Last cue ends: {cues[-1][1] / 60:.2f} min")
    print("Largest gaps over 120 seconds:")

    gaps = []
    for previous, current in zip(cues, cues[1:]):
        gap = current[0] - previous[1]
        if gap > 120:
            gaps.append((gap, previous[1], current[0]))

    for gap, prev_end, next_start in sorted(gaps, reverse=True)[:10]:
        print(
            f"  {gap / 60:.2f} min gap: "
            f"{prev_end / 60:.2f} -> {next_start / 60:.2f} min"
        )

    print("\nFirst 2 dialogue samples:")
    for start, end, subtitle in cues[:2]:
        print(f"  [{start / 60:.2f} min] {subtitle[:220]}")

    print("\nLast 5 dialogue samples:")
    for start, end, subtitle in cues[-5:]:
        print(f"  [{start / 60:.2f}–{end / 60:.2f} min] {subtitle[:220]}")

In [ ]:
from pathlib import Path
import re
import hashlib

validation_jobs = [
    ("2008_2", "Rab Ne Bana Di Jodi",
     "2008_2_Rab_Ne_Bana_Di_Jodi_candidate_4292288.srt",
     "2008_2_RAB_NE_BANA_DI_JODI_ENGLISH.srt", "en"),

    ("2008_3", "Dasavathaaram",
     "2008_3_DASAVATHAARAM_candidate_8900291.srt",
     "2008_3_DASAVATHAARAM_ENGLISH.srt", "en"),

    ("2008_4", "Singh Is Kinng",
     "2008_4_SINGH_IS_KINNG_candidate_4246518.srt",
     "2008_4_SINGH_IS_KINNG_ENGLISH.srt", "en"),

    ("2008_5", "Jodhaa Akbar",
     "2008_5_JODHAA_AKBAR_candidate_4093059.srt",
     "2008_5_JODHAA_AKBAR_ENGLISH.srt", "en"),

    ("2009_1", "3 Idiots",
     "2009_1_3_IDIOTS_candidate_8326640.srt",
     "2009_1_3_IDIOTS_ENGLISH.srt", "en"),

    ("2009_5", "Wanted",
     "2009_5_WANTED_candidate_4210730.srt",
     "2009_5_WANTED_ENGLISH.srt", "en"),

    ("2010_1", "Enthiran",
     "2010_1_ENTHIRAN_candidate_11186717.srt",
     "2010_1_ENTHIRAN_TAMIL.srt", "ta"),
]

TIME_RE = re.compile(
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
)

TAG_RE = re.compile(r"<[^>]+>")

def parse_timestamp(ts):
    ts = ts.replace(",", ".")
    h, m, s = ts.split(":")
    sec, ms = s.split(".")
    return int(h) * 3600 + int(m) * 60 + int(sec) + int(ms) / 1000

def is_promotional_credit(text):
    t = re.sub(r"\s+", " ", text).lower()
    patterns = [
        "movie & captioning conformed by",
        "movie & captions conformed by",
        "hope you'll like & enjoy",
        "hope you've liked & enjoyed",
    ]
    return any(p in t for p in patterns)

validated_summary = []

for record_id, movie, raw_name, out_name, language in validation_jobs:
    raw_path = RAW / raw_name
    out_path = VALIDATED / out_name

    if not raw_path.exists():
        print(f"SKIP — missing raw file: {raw_path.name}")
        continue

    source = raw_path.read_text(encoding="utf-8-sig", errors="replace")
    blocks = re.split(r"\n\s*\n", source.strip())
    cleaned_blocks = []
    removed_credit_blocks = 0
    removed_invalid_blocks = 0

    for block in blocks:
        match = TIME_RE.search(block)

        if not match:
            if block.strip():
                removed_invalid_blocks += 1
            continue

        start, end = match.groups()
        if parse_timestamp(end) < parse_timestamp(start):
            removed_invalid_blocks += 1
            continue

        lines = block.splitlines()
        text_lines = [
            line.strip()
            for line in lines
            if not TIME_RE.search(line) and not line.strip().isdigit()
        ]
        subtitle_text = "\n".join(text_lines).strip()

        if is_promotional_credit(subtitle_text):
            removed_credit_blocks += 1
            continue

        # Strip HTML/ASS-like tags while preserving the dialogue text.
        subtitle_text = TAG_RE.sub("", subtitle_text)
        subtitle_text = re.sub(r"\{[^}]*\}", "", subtitle_text)
        subtitle_text = re.sub(r"[ \t]+", " ", subtitle_text).strip()

        if not subtitle_text:
            removed_invalid_blocks += 1
            continue

        cleaned_blocks.append((start, end, subtitle_text))

    # Keep chronological order and rebuild standard SRT blocks.
    cleaned_blocks.sort(key=lambda x: parse_timestamp(x[0]))

    output_parts = []
    overlaps = 0
    previous_end = None

    for i, (start, end, subtitle_text) in enumerate(cleaned_blocks, start=1):
        start_s = parse_timestamp(start)
        end_s = parse_timestamp(end)

        if previous_end is not None and start_s < previous_end:
            overlaps += 1

        previous_end = max(previous_end or 0, end_s)
        output_parts.append(f"{i}\n{start} --> {end}\n{subtitle_text}")

    output_text = "\n\n".join(output_parts) + "\n"
    out_path.write_text(output_text, encoding="utf-8")

    digest = hashlib.sha256(out_path.read_bytes()).hexdigest()
    word_count = len(re.findall(r"\b[\w’'-]+\b", output_text, flags=re.UNICODE))

    validated_summary.append({
        "record_id": record_id,
        "movie": movie,
        "language": language,
        "raw_cues": len([b for b in blocks if TIME_RE.search(b)]),
        "validated_cues": len(cleaned_blocks),
        "removed_credit_blocks": removed_credit_blocks,
        "removed_invalid_blocks": removed_invalid_blocks,
        "overlaps_for_review": overlaps,
        "word_count": word_count,
        "validated_file": out_name,
        "sha256": digest,
    })

import pandas as pd
display(pd.DataFrame(validated_summary))
print("\nRaw files preserved. Manifest and locked datasets unchanged.")

In [ ]:
import pandas as pd
import shutil
from datetime import datetime
from pathlib import Path

# Reload the current manifest
manifest = pd.read_csv(
    MANIFEST,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string",
        "tmdb_id": "string",
    }
)

# Batch 04 accepted records only
batch04 = [
    {
        "movie_record_id": "2008_2", "year": 2008, "rank": 2,
        "movie_title": "Rab Ne Bana Di Jodi", "language": "Hindi",
        "tmdb_id": "14072", "subtitle_language": "en",
        "source_id": "4292288",
        "raw": "2008_2_Rab_Ne_Bana_Di_Jodi_candidate_4292288.srt",
        "validated": "2008_2_RAB_NE_BANA_DI_JODI_ENGLISH.srt",
        "blocks": 1349, "words": 21879,
        "status": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted",
        "notes": "English fallback; 1349 cues; no structural errors; endpoint coverage about 94.9% using estimated runtime."
    },
    {
        "movie_record_id": "2008_3", "year": 2008, "rank": 3,
        "movie_title": "Dasavathaaram", "language": "Tamil",
        "tmdb_id": "17888", "subtitle_language": "en",
        "source_id": "8900291",
        "raw": "2008_3_DASAVATHAARAM_candidate_8900291.srt",
        "validated": "2008_3_DASAVATHAARAM_ENGLISH.srt",
        "blocks": 1681, "words": 27888,
        "status": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted",
        "notes": "English fallback; 1681 cues; no structural errors; endpoint coverage about 98.6% using estimated runtime."
    },
    {
        "movie_record_id": "2008_4", "year": 2008, "rank": 4,
        "movie_title": "Singh Is Kinng", "language": "Hindi",
        "tmdb_id": "12273", "subtitle_language": "en",
        "source_id": "4246518",
        "raw": "2008_4_SINGH_IS_KINNG_candidate_4246518.srt",
        "validated": "2008_4_SINGH_IS_KINNG_ENGLISH.srt",
        "blocks": 2354, "words": 35857,
        "status": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted",
        "notes": "English fallback; 2354 cues; no structural errors; endpoint coverage about 96.2% using estimated runtime."
    },
    {
        "movie_record_id": "2008_5", "year": 2008, "rank": 5,
        "movie_title": "Jodhaa Akbar", "language": "Hindi",
        "tmdb_id": "14073", "subtitle_language": "en",
        "source_id": "4093059",
        "raw": "2008_5_JODHAA_AKBAR_candidate_4093059.srt",
        "validated": "2008_5_JODHAA_AKBAR_ENGLISH.srt",
        "blocks": 1528, "words": 24127,
        "status": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_formatting_cleanup",
        "notes": "English fallback; formatting tags removed in validated copy; endpoint coverage about 95.6% using estimated runtime."
    },
    {
        "movie_record_id": "2009_1", "year": 2009, "rank": 1,
        "movie_title": "3 Idiots", "language": "Hindi",
        "tmdb_id": "20453", "subtitle_language": "en",
        "source_id": "8326640",
        "raw": "2009_1_3_IDIOTS_candidate_8326640.srt",
        "validated": "2009_1_3_IDIOTS_ENGLISH.srt",
        "blocks": 2134, "words": 31863,
        "status": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_language_label_corrected",
        "notes": "Subtitle content is English although API language label was hi; recorded as English fallback."
    },
    {
        "movie_record_id": "2009_5", "year": 2009, "rank": 5,
        "movie_title": "Wanted", "language": "Hindi",
        "tmdb_id": "22717", "subtitle_language": "en",
        "source_id": "4210730",
        "raw": "2009_5_WANTED_candidate_4210730.srt",
        "validated": "2009_5_WANTED_ENGLISH.srt",
        "blocks": 1892, "words": 28077,
        "status": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_credit_cleanup",
        "notes": "English fallback; two promotional credit blocks removed; last cue around 150.34 minutes."
    },
    {
        "movie_record_id": "2010_1", "year": 2010, "rank": 1,
        "movie_title": "Enthiran", "language": "Tamil",
        "tmdb_id": "148284", "subtitle_language": "ta",
        "source_id": "11186717",
        "raw": "2010_1_ENTHIRAN_candidate_11186717.srt",
        "validated": "2010_1_ENTHIRAN_TAMIL.srt",
        "blocks": 2461, "words": 53629,
        "status": "Accepted_original_language_review_flag",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_timing_overlap_flag",
        "notes": "Tamil original; two timing overlaps detected in validated copy and retained for review."
    },
]

# Verify all files exist and IDs are not already present before writing.
existing_ids = set(manifest["movie_record_id"].dropna().astype(str))
new_ids = [r["movie_record_id"] for r in batch04]

if len(new_ids) != len(set(new_ids)):
    raise ValueError("Duplicate IDs within Batch 04.")

already_present = existing_ids.intersection(new_ids)
if already_present:
    raise ValueError(f"These IDs already exist in the manifest: {sorted(already_present)}")

for r in batch04:
    if not (RAW / r["raw"]).exists():
        raise FileNotFoundError(f"Missing raw file: {r['raw']}")
    if not (VALIDATED / r["validated"]).exists():
        raise FileNotFoundError(f"Missing validated file: {r['validated']}")

# Backup current manifest before appending.
backup_path = MANIFEST.with_name(
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_batch04_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".csv"
)
shutil.copy2(MANIFEST, backup_path)

# Build rows using the existing manifest's exact column structure.
new_rows = []

for r in batch04:
    row = {col: pd.NA for col in manifest.columns}

    row.update({
        "movie_record_id": r["movie_record_id"],
        "year": r["year"],
        "rank": r["rank"],
        "movie_title": r["movie_title"],
        "language": r["language"],
        "tmdb_id": r["tmdb_id"],
        "subtitle_language": r["subtitle_language"],
        "subtitle_type": "srt",
        "subtitle_source": "OpenSubtitles",
        "source_id": r["source_id"],
        "source_url": f"https://www.opensubtitles.com/en/subtitles/{r['source_id']}",
        "raw_subtitle_file": str(RAW / r["raw"]),
        "validated_subtitle_file": str(VALIDATED / r["validated"]),
        "acquisition_status": r["status"],
        "validation_status": r["validation"],
        "title_verified": "Yes",
        "year_verified": "Yes",
        "movie_identity_verified": "Yes",
        "language_verified": (
            "Yes_content_is_English"
            if r["movie_record_id"] == "2009_1"
            else "Yes"
        ),
        "dialogue_blocks": r["blocks"],
        "word_count": r["words"],
        "character_count": len(
            (VALIDATED / r["validated"]).read_text(
                encoding="utf-8", errors="replace"
            )
        ),
        "encoding_status": "UTF-8",
        "quality_status": r["quality"],
        "notes": r["notes"],
    })

    new_rows.append(row)

updated = pd.concat(
    [manifest, pd.DataFrame(new_rows, columns=manifest.columns)],
    ignore_index=True
)

# Final safety checks before saving.
if updated["movie_record_id"].duplicated().any():
    raise ValueError("Duplicate record IDs detected; manifest not saved.")

updated.to_csv(MANIFEST, index=False)

# Reload and verify.
verified = pd.read_csv(
    MANIFEST,
    dtype={
        "movie_record_id": "string",
        "source_id": "string",
        "source_url": "string",
        "tmdb_id": "string",
    }
)

print("Backup created:", backup_path.name)
print("Rows before:", len(manifest))
print("Rows added:", len(batch04))
print("Rows after:", len(verified))
print("Duplicate IDs:", verified["movie_record_id"].duplicated().sum())
print("Love Aaj Kal added:", "2009_3" in set(verified["movie_record_id"].astype(str)))

display(verified[
    verified["movie_record_id"].isin(new_ids)
][[
    "movie_record_id", "movie_title", "subtitle_language",
    "source_id", "dialogue_blocks", "word_count",
    "validated_subtitle_file", "acquisition_status",
    "validation_status", "quality_status"
]])

In [ ]:
# Notebook 8 — Identify the next unprocessed movies
import pandas as pd

manifest_check = pd.read_csv(
    MANIFEST,
    dtype={
        "movie_record_id": "string",
        "tmdb_id": "string",
        "source_id": "string",
    }
)

print("Manifest rows:", len(manifest_check))
print("\nLatest registered records:")

display(
    manifest_check[
        ["movie_record_id", "year", "rank", "movie_title",
         "language", "tmdb_id", "acquisition_status"]
    ].tail(15)
)

print("\nNext step: compare these records with the locked 130-movie master.")

In [ ]:
import pandas as pd
from pathlib import Path

MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"

assert MASTER.exists(), f"Master dataset not found: {MASTER}"
assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"

master = pd.read_csv(
    MASTER,
    dtype={"movie_record_id": "string", "tmdb_id": "string"}
)

manifest_check = pd.read_csv(
    MANIFEST,
    dtype={"movie_record_id": "string", "tmdb_id": "string"}
)

print("Master dataset shape:", master.shape)
print("Manifest records:", len(manifest_check))

# Confirm the columns used to identify movies.
print("\nMaster columns:")
print(master.columns.tolist())

print("\nManifest columns:")
print(manifest_check.columns.tolist())

# Compare stable record IDs.
if "movie_record_id" not in master.columns:
    raise ValueError(
        "Master has no movie_record_id column. Inspect its columns above "
        "before proceeding; do not guess the ID mapping."
    )

manifest_ids = set(
    manifest_check["movie_record_id"].dropna().astype(str).str.strip()
)
master_ids = master["movie_record_id"].dropna().astype(str).str.strip()

missing = master[
    ~master_ids.isin(manifest_ids)
].copy()

missing = missing.sort_values(["year", "rank"])

print("\nMovies not yet registered in the subtitle manifest:", len(missing))

display(missing)

In [ ]:
# Notebook 8 — inspect saved Batch 05 search results
from pathlib import Path

print("Search results directory:", SEARCH_RESULTS)
print("Directory exists:", SEARCH_RESULTS.exists())

if SEARCH_RESULTS.exists():
    files = sorted(SEARCH_RESULTS.iterdir())
    print("\nSaved search-result files:", len(files))

    for file in files:
        if file.is_file():
            print(file.name, "|", file.stat().st_size, "bytes")

In [ ]:

from pathlib import Path
import pandas as pd
import json
import re

# Project paths
PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
SEARCH_RESULTS = WORK / "search_results"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

master = pd.read_csv(MASTER, dtype={"movie_record_id": str})
manifest = pd.read_csv(MANIFEST, dtype={"movie_record_id": str})

# All records absent from the current subtitle manifest
missing = master[
    ~master["movie_record_id"].isin(manifest["movie_record_id"])
].copy()

# Scan saved search files conservatively for previously mentioned record IDs
previously_searched = set()
record_pattern = re.compile(r"\b(?:19|20)\d{2}_\d+\b")

def collect_ids(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(key, str):
                previously_searched.update(record_pattern.findall(key))
            collect_ids(value)
    elif isinstance(obj, list):
        for item in obj:
            collect_ids(item)
    elif isinstance(obj, str):
        previously_searched.update(record_pattern.findall(obj))

for file in SEARCH_RESULTS.glob("*"):
    try:
        if file.suffix.lower() == ".csv":
            df = pd.read_csv(file, dtype=str, on_bad_lines="skip")
            for col in df.columns:
                for value in df[col].dropna().astype(str):
                    previously_searched.update(record_pattern.findall(value))
        elif file.suffix.lower() == ".json":
            with open(file, "r", encoding="utf-8") as f:
                collect_ids(json.load(f))
    except Exception as e:
        print(f"Could not fully inspect {file.name}: {e}")

# Known previously searched/rejected leads from our project notes
known_searched = {
    "2009_2",  # Magadheera: candidate found, not yet verified/downloaded
    "2009_3",  # Love Aaj Kal: incomplete candidate rejected
    "2009_4",  # Ajab Prem Ki Ghazab Kahani: suspicious candidate found
    "2010_2",  # Dabangg: original-language search already attempted
    "2010_3",  # Golmaal 3: false-match candidate found
    "2010_4",  # Raajneeti: search already attempted
}

previously_searched.update(known_searched)

# Exclude anything already in the manifest or mentioned in saved search history
new_candidates = missing[
    ~missing["movie_record_id"].isin(previously_searched)
].copy()

print("Master rows:", len(master))
print("Manifest rows:", len(manifest))
print("Missing from manifest:", len(missing))
print("IDs found in saved search history:", len(previously_searched))
print("\nProposed next 10 movies (not found in the inspected search history):")

# Show available identifying columns without assuming every column exists
preferred_cols = [
    "movie_record_id", "year", "rank", "movie_title",
    "language", "tmdb_id", "imdb_id"
]
display_cols = [c for c in preferred_cols if c in new_candidates.columns]

batch05 = new_candidates[display_cols].head(10).copy()
display(batch05)

print("\nProposed batch size:", len(batch05))
if len(batch05) < 10:
    print("Fewer than 10 candidates passed the conservative exclusion check.")
    print("Do not fill the batch with repeats; inspect the remaining search history first.")


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
PROCESSED = PROJECT / "01_data/processed"
RAW_DATA = PROJECT / "01_data/raw"

batch05_titles = [
    "Housefull",
    "Bodyguard",
    "Ready",
    "Ra.One",
    "Don 2",
    "Singham",
    "Ek Tha Tiger",
    "Dabangg 2",
    "Rowdy Rathore",
    "Agneepath",
]

files_to_check = [
    PROCESSED / "ICMB_130_TMDB_MAPPING.csv",
    RAW_DATA / "ICMB_130_TMDB_METADATA.csv",
]

for path in files_to_check:
    print("\n" + "=" * 75)
    print("FILE:", path.name, "| Exists:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path, dtype=str)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    # Search across text columns without modifying the source
    text_cols = df.select_dtypes(include="object").columns
    mask = pd.Series(False, index=df.index)

    for col in text_cols:
        mask |= df[col].fillna("").str.contains(
            "|".join(
                title.replace(".", r"\.")
                for title in batch05_titles
            ),
            case=False,
            regex=True
        )

    matches = df.loc[mask]
    print("Matching rows:", len(matches))
    if not matches.empty:
        display(matches)


In [ ]:

# BATCH 05 — METADATA-ONLY SUBTITLE SEARCH
# No subtitle downloads. No manifest changes.
# Reuses the authenticated OpenSubtitles session/function from Notebook 8.

import pandas as pd
from pathlib import Path
from datetime import datetime

batch05 = [
    {"movie_record_id": "2010_5", "movie_title": "Housefull", "year": 2010, "tmdb_id": 58051, "language": "hi"},
    {"movie_record_id": "2011_1", "movie_title": "Bodyguard", "year": 2011, "tmdb_id": 73582, "language": "hi"},
    {"movie_record_id": "2011_2", "movie_title": "Ready", "year": 2011, "tmdb_id": 65946, "language": "hi"},
    {"movie_record_id": "2011_3", "movie_title": "Ra.One", "year": 2011, "tmdb_id": 41517, "language": "hi"},
    {"movie_record_id": "2011_4", "movie_title": "Don 2", "year": 2011, "tmdb_id": 41109, "language": "hi"},
    {"movie_record_id": "2011_5", "movie_title": "Singham", "year": 2011, "tmdb_id": 70590, "language": "hi"},
    {"movie_record_id": "2012_1", "movie_title": "Ek Tha Tiger", "year": 2012, "tmdb_id": 85985, "language": "hi"},
    {"movie_record_id": "2012_2", "movie_title": "Dabangg 2", "year": 2012, "tmdb_id": 147405, "language": "hi"},
    {"movie_record_id": "2012_3", "movie_title": "Rowdy Rathore", "year": 2012, "tmdb_id": 102632, "language": "hi"},
    {"movie_record_id": "2012_4", "movie_title": "Agneepath", "year": 2012, "tmdb_id": 84858, "language": "hi"},
]

# Check what search functions exist without displaying credentials/tokens
print("Potential existing search functions:")
for name in ["search_subtitles", "search_opensubtitles", "search_subtitle_candidates",
             "search_subtitles_api", "opensubtitles_search"]:
    if name in globals() and callable(globals()[name]):
        print(" -", name)

print("\nBefore running searches, identify the exact search function used earlier.")
print("We will call only the metadata search endpoint and save candidate results.")


In [ ]:

import inspect

print("Function signature:")
print(inspect.signature(search_subtitles))

print("\nFunction source:")
print(inspect.getsource(search_subtitles))


In [ ]:

# Read-only inspection of saved search results
import pandas as pd
from pathlib import Path

SEARCH_RESULTS = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

csv_path = SEARCH_RESULTS / "batch05_hindi_metadata_search_20261024_195426.csv"
df = pd.read_csv(csv_path)

for record_id in ["2011_5", "2011_4", "2012_2"]:
    print("\n" + "=" * 65)
    print("Record:", record_id)

    subset = df[df["movie_record_id"] == record_id]
    if subset.empty:
        print("No candidates.")
    else:
        display(subset[[
            "subtitle_result_id",
            "candidate_title",
            "candidate_year",
            "candidate_language",
            "release",
            "download_count",
            "file_id",
            "file_name"
        ]])

# Show the function's actual response keys from the saved candidate CSV
print("\nAvailable saved columns:")
print(df.columns.tolist())


In [ ]:

import inspect

print("Search function:")
print(inspect.getsource(search_subtitles))

print("\nAPI base URL:", BASE_URL)
print("\nAvailable request headers (names only):", list(headers.keys()))

print("\nCandidate fields expected from OpenSubtitles:")
print("- attributes.movie_id")
print("- attributes.feature_details")
print("- attributes.language")
print("- attributes.release")
print("- attributes.files")


In [ ]:

# NOTEBOOK 8 — BATCH 05 EXACT TMDB-ID SEARCH
# Metadata only: no downloads, no manifest changes.

from pathlib import Path
import pandas as pd
import requests
import time
from datetime import datetime

batch05 = [
    {"movie_record_id": "2010_5", "title": "Housefull", "year": 2010, "tmdb_id": 58051},
    {"movie_record_id": "2011_1", "title": "Bodyguard", "year": 2011, "tmdb_id": 73582},
    {"movie_record_id": "2011_2", "title": "Ready", "year": 2011, "tmdb_id": 65946},
    {"movie_record_id": "2011_3", "title": "Ra.One", "year": 2011, "tmdb_id": 41517},
    {"movie_record_id": "2011_4", "title": "Don 2", "year": 2011, "tmdb_id": 41109},
    {"movie_record_id": "2011_5", "title": "Singham", "year": 2011, "tmdb_id": 70590},
    {"movie_record_id": "2012_1", "title": "Ek Tha Tiger", "year": 2012, "tmdb_id": 85985},
    {"movie_record_id": "2012_2", "title": "Dabangg 2", "year": 2012, "tmdb_id": 147405},
    {"movie_record_id": "2012_3", "title": "Rowdy Rathore", "year": 2012, "tmdb_id": 102632},
    {"movie_record_id": "2012_4", "title": "Agneepath", "year": 2012, "tmdb_id": 84858},
]

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
SEARCH_RESULTS = (
    PROJECT / "01_data/processed/subtitle_corpus_v2/search_results"
)
SEARCH_RESULTS.mkdir(parents=True, exist_ok=True)

# Reuse the existing API headers and BASE_URL from Notebook 8.
# Do not print headers, tokens, or credentials.
rows = []
errors = []

for movie in batch05:
    params = {
        "tmdb_id": movie["tmdb_id"],
        "type": "movie",
        "languages": "hi",
        "per_page": 50,
    }

    try:
        response = requests.get(
            f"{BASE_URL}/subtitles",
            headers=headers,
            params=params,
            timeout=30,
        )

        if response.status_code != 200:
            errors.append({
                "movie_record_id": movie["movie_record_id"],
                "title": movie["title"],
                "http_status": response.status_code,
                "message": response.text[:250],
            })
            print(
                f"{movie['movie_record_id']} | {movie['title']} | "
                f"HTTP {response.status_code}"
            )
            continue

        payload = response.json()
        candidates = payload.get("data", [])
        print(
            f"{movie['movie_record_id']} | {movie['title']} | "
            f"TMDB {movie['tmdb_id']} | Hindi candidates: {len(candidates)}"
        )

        for item in candidates:
            attrs = item.get("attributes", {})
            feature = attrs.get("feature_details") or {}
            files = attrs.get("files", []) or [{}]

            for f in files:
                rows.append({
                    "movie_record_id": movie["movie_record_id"],
                    "requested_title": movie["title"],
                    "requested_year": movie["year"],
                    "requested_tmdb_id": movie["tmdb_id"],
                    "subtitle_result_id": attrs.get("subtitle_id") or item.get("id"),
                    "candidate_language": attrs.get("language"),
                    "feature_title": feature.get("title"),
                    "feature_movie_name": feature.get("movie_name"),
                    "feature_year": feature.get("year"),
                    "feature_tmdb_id": feature.get("tmdb_id"),
                    "feature_imdb_id": feature.get("imdb_id"),
                    "release": attrs.get("release"),
                    "download_count": attrs.get("download_count"),
                    "trusted_source": attrs.get("from_trusted"),
                    "hearing_impaired": attrs.get("hearing_impaired"),
                    "file_id": f.get("file_id"),
                    "file_name": f.get("file_name"),
                })

    except Exception as exc:
        errors.append({
            "movie_record_id": movie["movie_record_id"],
            "title": movie["title"],
            "http_status": None,
            "message": str(exc)[:250],
        })
        print(f"{movie['movie_record_id']} | {movie['title']} | ERROR")

    time.sleep(0.4)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = SEARCH_RESULTS / f"batch05_tmdb_id_hindi_search_{stamp}.csv"
error_csv = SEARCH_RESULTS / f"batch05_tmdb_id_search_errors_{stamp}.csv"

pd.DataFrame(rows).to_csv(out_csv, index=False)
pd.DataFrame(errors).to_csv(error_csv, index=False)

print("\nSaved candidates:", out_csv)
print("Saved errors:", error_csv)
print("Total candidate rows:", len(rows))
print("Errors:", len(errors))

if rows:
    display(pd.DataFrame(rows))


In [ ]:

import pandas as pd
from pathlib import Path

SEARCH_RESULTS = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

candidate_file = (
    SEARCH_RESULTS / "batch05_tmdb_id_hindi_search_20261024_195804.csv"
)
df = pd.read_csv(candidate_file)

singham = df[df["movie_record_id"] == "2011_5"]

if singham.empty:
    print("No saved Singham candidate found.")
else:
    display(singham.T)
    print("\nFile-level metadata:")
    print("File ID:", singham.iloc[0]["file_id"])
    print("Subtitle ID:", singham.iloc[0]["subtitle_result_id"])
    print("Language:", singham.iloc[0]["candidate_language"])
    print("Hearing impaired:", singham.iloc[0]["hearing_impaired"])
    print("Trusted source:", singham.iloc[0]["trusted_source"])


In [ ]:

# Read-only OpenSubtitles quota check
response = requests.get(
    f"{BASE_URL}/infos/user",
    headers=headers,
    timeout=30
)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    data = response.json()
    user_data = data.get("data", data)
    for key in ["allowed_downloads", "downloads_count", "remaining_downloads", "level"]:
        if key in user_data:
            print(f"{key}: {user_data[key]}")
else:
    print("Could not verify quota. Do not download yet.")


In [ ]:

# NOTEBOOK 8 — DOWNLOAD SINGHAM CANDIDATE ONLY
# Candidate already verified against TMDB ID 70590.
# Do not update the manifest until validation passes.

from pathlib import Path
import requests
import json

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
RAW.mkdir(parents=True, exist_ok=True)

SUBTITLE_ID = 11537175
FILE_ID = 12442406
RECORD_ID = "2011_5"

# Confirm this is the expected candidate before proceeding.
candidate_path = (
    WORK / "search_results" /
    "batch05_tmdb_id_hindi_search_20261024_195804.csv"
)
candidate_df = pd.read_csv(candidate_path)
candidate = candidate_df[
    (candidate_df["movie_record_id"] == RECORD_ID) &
    (candidate_df["subtitle_result_id"] == SUBTITLE_ID) &
    (candidate_df["file_id"] == FILE_ID)
]

if len(candidate) != 1:
    raise ValueError("Candidate verification failed. No download attempted.")

# OpenSubtitles download endpoint: requests a temporary download link.
download_response = requests.post(
    f"{BASE_URL}/download",
    headers=headers,
    json={"file_id": FILE_ID},
    timeout=30,
)

print("Download endpoint HTTP status:", download_response.status_code)

if download_response.status_code != 200:
    print("Download request failed. No subtitle file saved.")
    print(download_response.text[:300])
else:
    payload = download_response.json()
    link = payload.get("link")
    filename = payload.get("file_name") or "Singham_candidate.srt"

    if not link:
        raise ValueError("No download link returned; no file saved.")

    # Fetch the actual subtitle content from the temporary link.
    file_response = requests.get(link, timeout=60)
    file_response.raise_for_status()

    raw_path = RAW / "2011_5_SINGHAM_candidate_11537175.srt"
    raw_path.write_bytes(file_response.content)

    print("Saved raw candidate:", raw_path)
    print("Returned filename:", filename)
    print("Bytes saved:", raw_path.stat().st_size)
    print("Download complete. Manifest unchanged.")



In [ ]:

# NOTEBOOK 8 — SINGHAM SUBTITLE VALIDATION (READ-ONLY)

from pathlib import Path
import re
import hashlib
import pandas as pd
from collections import Counter

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW_FILE = WORK / "raw/2011_5_SINGHAM_candidate_11537175.srt"

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Raw subtitle not found: {RAW_FILE}")

raw_bytes = RAW_FILE.read_bytes()
file_hash = hashlib.sha256(raw_bytes).hexdigest()

# Decode without changing the source file
encoding_used = None
text = None
for enc in ["utf-8-sig", "utf-8", "cp1252", "latin-1"]:
    try:
        text = raw_bytes.decode(enc)
        encoding_used = enc
        break
    except UnicodeDecodeError:
        continue

if text is None:
    raise ValueError("Could not decode subtitle file.")

# Normalize line endings in memory only
text = text.replace("\r\n", "\n").replace("\r", "\n").strip()

timestamp_re = re.compile(
    r"(?P<sh>\d{2}):(?P<sm>\d{2}):(?P<ss>\d{2})[,\.](?P<sms>\d{3})"
    r"\s*-->\s*"
    r"(?P<eh>\d{2}):(?P<em>\d{2}):(?P<es>\d{2})[,\.](?P<ems>\d{3})"
)

def to_seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def timestamp_seconds(match, prefix):
    return to_seconds(
        match.group(prefix + "h"),
        match.group(prefix + "m"),
        match.group(prefix + "s"),
        match.group(prefix + "ms"),
    )

blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]
cues = []
malformed_blocks = []
reversed_cues = []
timestamp_lines = 0

for block_number, block in enumerate(blocks, start=1):
    lines = block.splitlines()
    matches = [(i, timestamp_re.search(line)) for i, line in enumerate(lines)]
    matches = [(i, m) for i, m in matches if m]

    if not matches:
        if "-->" in block:
            malformed_blocks.append((block_number, block[:100]))
        continue

    timestamp_lines += len(matches)

    # Typical SRT block should have one timing line
    if len(matches) > 1:
        malformed_blocks.append(
            (block_number, f"Multiple timing lines: {len(matches)}")
        )

    line_index, match = matches[0]
    start = timestamp_seconds(match, "s")
    end = timestamp_seconds(match, "e")
    subtitle_text = "\n".join(lines[line_index + 1:]).strip()

    if end <= start:
        reversed_cues.append({
            "block": block_number,
            "start_sec": start,
            "end_sec": end,
        })

    cues.append({
        "block": block_number,
        "start_sec": start,
        "end_sec": end,
        "text": subtitle_text,
    })

# Basic timing checks
sorted_cues = sorted(cues, key=lambda x: (x["start_sec"], x["end_sec"]))
overlaps = []
gaps = []

for previous, current in zip(sorted_cues, sorted_cues[1:]):
    delta = current["start_sec"] - previous["end_sec"]

    if delta < 0:
        overlaps.append({
            "previous_block": previous["block"],
            "current_block": current["block"],
            "overlap_seconds": round(-delta, 3),
        })
    elif delta > 120:
        gaps.append({
            "from_min": round(previous["end_sec"] / 60, 2),
            "to_min": round(current["start_sec"] / 60, 2),
            "gap_minutes": round(delta / 60, 2),
        })

all_subtitle_text = "\n".join(c["text"] for c in cues)
plain_text = re.sub(r"\{[^}]*\}", " ", all_subtitle_text)
plain_text = re.sub(r"<[^>]*>", " ", plain_text)
words = re.findall(r"\b[\w’'-]+\b", plain_text, flags=re.UNICODE)
visible_chars = [ch for ch in plain_text if not ch.isspace()]
devanagari_chars = sum("\u0900" <= ch <= "\u097F" for ch in visible_chars)
latin_chars = sum(ch.isascii() and ch.isalpha() for ch in visible_chars)

normalized_texts = [
    re.sub(r"\s+", " ", c["text"]).strip().casefold()
    for c in cues
    if re.sub(r"\s+", " ", c["text"]).strip()
]
text_counts = Counter(normalized_texts)
duplicate_text_groups = {
    value: count for value, count in text_counts.items() if count > 1
}

if cues:
    first_start = min(c["start_sec"] for c in cues)
    last_end = max(c["end_sec"] for c in cues)
    timeline_minutes = (last_end - first_start) / 60
else:
    first_start = last_end = timeline_minutes = 0

# Optional reference runtime from the existing TMDB metadata file
runtime_minutes = None
metadata_path = PROJECT / "01_data/raw/ICMB_130_TMDB_METADATA.csv"
if metadata_path.exists():
    metadata = pd.read_csv(metadata_path)
    if {"movie_record_id", "runtime"}.issubset(metadata.columns):
        row = metadata[metadata["movie_record_id"].astype(str) == "2011_5"]
        if not row.empty and pd.notna(row.iloc[0]["runtime"]):
            runtime_minutes = float(row.iloc[0]["runtime"])

print("=" * 65)
print("SINGHAM (2011) — RAW SUBTITLE VALIDATION REPORT")
print("=" * 65)
print("File:", RAW_FILE.name)
print("File size (bytes):", len(raw_bytes))
print("SHA-256:", file_hash)
print("Detected encoding:", encoding_used)
print("Total non-empty blocks:", len(blocks))
print("Parsed cues:", len(cues))
print("Malformed blocks:", len(malformed_blocks))
print("Reversed/zero-duration cues:", len(reversed_cues))
print("Overlapping cue pairs:", len(overlaps))
print("Gaps over 2 minutes:", len(gaps))
print("Duplicate non-empty text groups:", len(duplicate_text_groups))
print("Word count:", len(words))
print("Visible characters:", len(visible_chars))
print("Devanagari characters:", devanagari_chars)
print("Latin alphabet characters:", latin_chars)
print("First cue starts (minutes):", round(first_start / 60, 2))
print("Last cue ends (minutes):", round(last_end / 60, 2))
print("Subtitle timeline span (minutes):", round(timeline_minutes, 2))
print("TMDB reference runtime (minutes):", runtime_minutes)

if visible_chars:
    print(
        "Devanagari share of visible characters (%):",
        round(100 * devanagari_chars / len(visible_chars), 2)
    )

print("\nFirst 5 parsed cues:")
for cue in sorted_cues[:5]:
    print(
        f"[{cue['start_sec']:.2f}s --> {cue['end_sec']:.2f}s] "
        f"{cue['text'][:160]!r}"
    )

print("\nLast 5 parsed cues:")
for cue in sorted_cues[-5:]:
    print(
        f"[{cue['start_sec']:.2f}s --> {cue['end_sec']:.2f}s] "
        f"{cue['text'][:160]!r}"
    )

if malformed_blocks:
    print("\nMalformed block examples:")
    for item in malformed_blocks[:5]:
        print(item)

if reversed_cues:
    print("\nReversed cue examples:")
    for item in reversed_cues[:5]:
        print(item)

if overlaps:
    print("\nLargest overlaps (up to 5):")
    display(pd.DataFrame(overlaps).sort_values(
        "overlap_seconds", ascending=False
    ).head(5))

if gaps:
    print("\nLargest gaps over 2 minutes (up to 10):")
    display(pd.DataFrame(gaps).sort_values(
        "gap_minutes", ascending=False
    ).head(10))

print("\nRaw file preserved. No validated file or manifest changes made.")


In [ ]:

# NOTEBOOK 8 — ACCEPT SINGHAM AND CREATE VALIDATED COPY
# Raw file remains unchanged. No cues or timestamps are deleted/shifted.

from pathlib import Path
import re
import hashlib

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW_FILE = WORK / "raw/2011_5_SINGHAM_candidate_11537175.srt"
VALIDATED = WORK / "validated"
VALIDATED.mkdir(parents=True, exist_ok=True)

OUT_FILE = VALIDATED / "2011_5_SINGHAM_HINDI.srt"

if not RAW_FILE.exists():
    raise FileNotFoundError(RAW_FILE)

if OUT_FILE.exists():
    raise FileExistsError(
        f"Validated output already exists; inspect it before overwriting: {OUT_FILE}"
    )

raw_bytes_before = RAW_FILE.read_bytes()
raw_hash_before = hashlib.sha256(raw_bytes_before).hexdigest()

text = raw_bytes_before.decode("utf-8-sig")
text = text.replace("\r\n", "\n").replace("\r", "\n").strip()

# Normalize SRT block spacing only; preserve cue text and timestamps.
blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]
normalized_text = "\n\n".join(blocks) + "\n"
out_bytes = normalized_text.encode("utf-8")

OUT_FILE.write_bytes(out_bytes)

# Verify source integrity and basic output integrity
raw_hash_after = hashlib.sha256(RAW_FILE.read_bytes()).hexdigest()
validated_text = OUT_FILE.read_text(encoding="utf-8")
validated_blocks = [
    b.strip() for b in re.split(r"\n\s*\n", validated_text.strip()) if b.strip()
]

assert raw_hash_before == raw_hash_after, "Raw file changed unexpectedly."
assert len(validated_blocks) == len(blocks), "Block count changed unexpectedly."

print("Validated copy created:", OUT_FILE)
print("Raw SHA-256 unchanged:", raw_hash_before == raw_hash_after)
print("Raw blocks:", len(blocks))
print("Validated blocks:", len(validated_blocks))
print("Validated bytes:", OUT_FILE.stat().st_size)
print("Validated SHA-256:", hashlib.sha256(OUT_FILE.read_bytes()).hexdigest())
print("No cue deletion or timestamp modification performed.")
print("Manifest not yet updated.")


In [ ]:

# NOTEBOOK 8 — BATCH 05: REMAINING 9 MOVIES
# Hindi first; English fallback only if Hindi returns zero candidates.
# Metadata only. No downloads. No manifest updates.

from pathlib import Path
import pandas as pd
import requests
import time
from datetime import datetime

batch05_remaining = [
    {"movie_record_id": "2010_5", "title": "Housefull", "year": 2010, "tmdb_id": 58051},
    {"movie_record_id": "2011_1", "title": "Bodyguard", "year": 2011, "tmdb_id": 73582},
    {"movie_record_id": "2011_2", "title": "Ready", "year": 2011, "tmdb_id": 65946},
    {"movie_record_id": "2011_3", "title": "Ra.One", "year": 2011, "tmdb_id": 41517},
    {"movie_record_id": "2011_4", "title": "Don 2", "year": 2011, "tmdb_id": 41109},
    {"movie_record_id": "2012_1", "title": "Ek Tha Tiger", "year": 2012, "tmdb_id": 85985},
    {"movie_record_id": "2012_2", "title": "Dabangg 2", "year": 2012, "tmdb_id": 147405},
    {"movie_record_id": "2012_3", "title": "Rowdy Rathore", "year": 2012, "tmdb_id": 102632},
    {"movie_record_id": "2012_4", "title": "Agneepath", "year": 2012, "tmdb_id": 84858},
]

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
SEARCH_RESULTS = (
    PROJECT / "01_data/processed/subtitle_corpus_v2/search_results"
)
SEARCH_RESULTS.mkdir(parents=True, exist_ok=True)

rows = []
errors = []
search_log = []

for movie in batch05_remaining:
    found_hindi = False

    for language in ["hi", "en"]:
        # English fallback only when the Hindi search returned zero.
        if language == "en" and found_hindi:
            break

        params = {
            "tmdb_id": movie["tmdb_id"],
            "type": "movie",
            "languages": language,
            "per_page": 50,
        }

        try:
            response = requests.get(
                f"{BASE_URL}/subtitles",
                headers=headers,
                params=params,
                timeout=30,
            )

            if response.status_code != 200:
                errors.append({
                    "movie_record_id": movie["movie_record_id"],
                    "title": movie["title"],
                    "language_searched": language,
                    "http_status": response.status_code,
                    "message": response.text[:250],
                })
                print(
                    f"{movie['movie_record_id']} | {movie['title']} | "
                    f"{language} | HTTP {response.status_code}"
                )
                # Do not interpret an API error as zero candidates.
                break

            payload = response.json()
            candidates = payload.get("data", [])

            search_log.append({
                "movie_record_id": movie["movie_record_id"],
                "title": movie["title"],
                "tmdb_id": movie["tmdb_id"],
                "language_searched": language,
                "candidate_count": len(candidates),
            })

            print(
                f"{movie['movie_record_id']} | {movie['title']} | "
                f"{language} candidates: {len(candidates)}"
            )

            if language == "hi" and candidates:
                found_hindi = True

            for item in candidates:
                attrs = item.get("attributes", {})
                feature = attrs.get("feature_details") or {}
                files = attrs.get("files", []) or [{}]

                for f in files:
                    rows.append({
                        "movie_record_id": movie["movie_record_id"],
                        "requested_title": movie["title"],
                        "requested_year": movie["year"],
                        "requested_tmdb_id": movie["tmdb_id"],
                        "language_searched": language,
                        "subtitle_result_id": (
                            attrs.get("subtitle_id") or item.get("id")
                        ),
                        "candidate_language": attrs.get("language"),
                        "feature_title": feature.get("title"),
                        "feature_movie_name": feature.get("movie_name"),
                        "feature_year": feature.get("year"),
                        "feature_tmdb_id": feature.get("tmdb_id"),
                        "feature_imdb_id": feature.get("imdb_id"),
                        "release": attrs.get("release"),
                        "download_count": attrs.get("download_count"),
                        "trusted_source": attrs.get("from_trusted"),
                        "hearing_impaired": attrs.get("hearing_impaired"),
                        "file_id": f.get("file_id"),
                        "file_name": f.get("file_name"),
                    })

            time.sleep(0.4)

        except Exception as exc:
            errors.append({
                "movie_record_id": movie["movie_record_id"],
                "title": movie["title"],
                "language_searched": language,
                "http_status": None,
                "message": str(exc)[:250],
            })
            print(
                f"{movie['movie_record_id']} | {movie['title']} | "
                f"{language} | ERROR"
            )
            break

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

candidate_path = SEARCH_RESULTS / f"batch05_remaining9_tmdb_search_{stamp}.csv"
log_path = SEARCH_RESULTS / f"batch05_remaining9_search_log_{stamp}.csv"
error_path = SEARCH_RESULTS / f"batch05_remaining9_search_errors_{stamp}.csv"

pd.DataFrame(rows).to_csv(candidate_path, index=False)
pd.DataFrame(search_log).to_csv(log_path, index=False)
pd.DataFrame(errors).to_csv(error_path, index=False)

print("\n" + "=" * 65)
print("BATCH 05 METADATA SEARCH COMPLETE")
print("=" * 65)
print("Candidate file:", candidate_path)
print("Search log:", log_path)
print("Error log:", error_path)
print("Candidate rows:", len(rows))
print("Search requests logged:", len(search_log))
print("Errors:", len(errors))
print("No subtitle files downloaded; manifest unchanged.")

if rows:
    result_df = pd.DataFrame(rows)
    display(result_df[[
        "movie_record_id", "requested_title", "language_searched",
        "subtitle_result_id", "candidate_language", "feature_title",
        "feature_year", "feature_tmdb_id", "release",
        "download_count", "trusted_source", "file_id", "file_name"
    ]])


In [ ]:

# NOTEBOOK 8 — BATCH 05 ENGLISH CANDIDATE SHORTLIST
# Read-only inspection of saved metadata search results.

from pathlib import Path
import pandas as pd
import re

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
SEARCH_RESULTS = (
    PROJECT / "01_data/processed/subtitle_corpus_v2/search_results"
)
CSV_PATH = SEARCH_RESULTS / "batch05_remaining9_tmdb_search_20261024_200354.csv"

df = pd.read_csv(CSV_PATH)

# Only English results returned for the exact requested TMDB ID
english = df[
    (df["language_searched"] == "en") &
    (df["candidate_language"] == "en") &
    (pd.to_numeric(df["requested_tmdb_id"], errors="coerce") ==
     pd.to_numeric(df["feature_tmdb_id"], errors="coerce"))
].copy()

english["download_count"] = pd.to_numeric(
    english["download_count"], errors="coerce"
).fillna(0)

# Flag possible split files; these need to be considered as a set
english["split_cd"] = english["file_name"].fillna("").str.contains(
    r"\b(?:CD|DISC)[ ._-]*[12]\b|CD[12]\b",
    case=False,
    regex=True
)

english["cd_number"] = english["file_name"].fillna("").str.extract(
    r"\b(?:CD|DISC)[ ._-]*([12])\b|CD([12])\b",
    flags=re.IGNORECASE
).bfill(axis=1).iloc[:, 0]

print("English candidate rows:", len(english))
print("Distinct movies:", english["movie_record_id"].nunique())
print("Rows with exact TMDB ID match:",
      english["feature_tmdb_id"].notna().sum())

# Display up to 4 candidates per movie, highest downloads first.
# Download count is only a screening signal, not a quality guarantee.
shortlist = (
    english.sort_values(
        ["movie_record_id", "download_count"],
        ascending=[True, False]
    )
    .groupby("movie_record_id", group_keys=False)
    .head(4)
)

display(shortlist[[
    "movie_record_id", "requested_title", "requested_tmdb_id",
    "feature_title", "feature_year", "feature_tmdb_id",
    "subtitle_result_id", "release", "download_count",
    "file_id", "file_name", "split_cd"
]])

print("\nPotential split-CD candidates:")
split_rows = english[english["split_cd"]]
if split_rows.empty:
    print("No filenames matched the simple split-CD pattern.")
else:
    display(split_rows[[
        "movie_record_id", "requested_title", "subtitle_result_id",
        "release", "download_count", "file_id", "file_name"
    ]].sort_values(["movie_record_id", "file_name"]))

print("\nNo files downloaded. Manifest unchanged.")


In [ ]:

# NOTEBOOK 8 — BATCH 05 SHORTLISTED DOWNLOADS
# 10 files total, including both Agneepath CD parts.
# Raw files only; manifest remains unchanged.

from pathlib import Path
import requests
import pandas as pd
import time
import hashlib

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
RAW.mkdir(parents=True, exist_ok=True)

selected = [
    {"record": "2010_5", "title": "HOUSEFULL", "subtitle_id": 74355, "file_id": 76927, "suffix": "ENGLISH"},
    {"record": "2011_1", "title": "BODYGUARD", "subtitle_id": 760478, "file_id": 835346, "suffix": "ENGLISH"},
    {"record": "2011_2", "title": "READY", "subtitle_id": 742112, "file_id": 815931, "suffix": "ENGLISH"},
    {"record": "2011_3", "title": "RA_ONE", "subtitle_id": 771341, "file_id": 846831, "suffix": "ENGLISH"},
    {"record": "2011_4", "title": "DON_2", "subtitle_id": 4319929, "file_id": 4409530, "suffix": "ENGLISH"},
    {"record": "2012_1", "title": "EK_THA_TIGER", "subtitle_id": 829082, "file_id": 907888, "suffix": "ENGLISH"},
    {"record": "2012_2", "title": "DABANGG_2", "subtitle_id": 856327, "file_id": 936687, "suffix": "ENGLISH"},
    {"record": "2012_3", "title": "ROWDY_RATHORE", "subtitle_id": 816442, "file_id": 894361, "suffix": "ENGLISH"},
    {"record": "2012_4", "title": "AGNEEPATH", "subtitle_id": 786654, "file_id": 862905, "suffix": "ENGLISH_CD1"},
    {"record": "2012_4", "title": "AGNEEPATH", "subtitle_id": 786654, "file_id": 862915, "suffix": "ENGLISH_CD2"},
]

# Confirm current quota before spending it.
quota_response = requests.get(
    f"{BASE_URL}/infos/user",
    headers=headers,
    timeout=30,
)
if quota_response.status_code != 200:
    raise RuntimeError("Could not verify download quota. No downloads attempted.")

quota_data = quota_response.json().get("data", quota_response.json())
remaining = quota_data.get("remaining_downloads")

print("Remaining downloads:", remaining)
print("Planned file downloads:", len(selected))

if remaining is None or int(remaining) < len(selected):
    raise RuntimeError(
        "Quota is insufficient for all 10 files. No downloads attempted."
    )

# Confirm there are no existing files that would be overwritten.
targets = {
    item["file_id"]: RAW / (
        f"{item['record']}_{item['title']}_candidate_{item['file_id']}.srt"
    )
    for item in selected
}
existing = [str(path) for path in targets.values() if path.exists()]
if existing:
    raise FileExistsError(
        "Existing raw target(s) found; no downloads attempted:\n"
        + "\n".join(existing)
    )

download_log = []

for item in selected:
    target = targets[item["file_id"]]
    print(
        f"\nDownloading {item['record']} {item['title']} "
        f"| file ID {item['file_id']}"
    )

    try:
        response = requests.post(
            f"{BASE_URL}/download",
            headers=headers,
            json={"file_id": item["file_id"]},
            timeout=30,
        )

        if response.status_code != 200:
            print("Download request failed; stopping batch.")
            print("HTTP status:", response.status_code)
            print(response.text[:250])
            download_log.append({
                **item, "status": "request_failed",
                "http_status": response.status_code
            })
            break

        payload = response.json()
        link = payload.get("link")
        if not link:
            print("No temporary link returned; stopping batch.")
            download_log.append({
                **item, "status": "no_link",
                "http_status": response.status_code
            })
            break

        file_response = requests.get(link, timeout=60)
        file_response.raise_for_status()

        content = file_response.content
        if not content:
            print("Empty file received; stopping batch.")
            download_log.append({**item, "status": "empty_file"})
            break

        target.write_bytes(content)

        digest = hashlib.sha256(content).hexdigest()
        print("Saved:", target.name)
        print("Bytes:", len(content))
        print("SHA-256:", digest)

        download_log.append({
            **item,
            "status": "downloaded",
            "raw_path": str(target),
            "bytes": len(content),
            "sha256": digest,
        })

        time.sleep(0.4)

    except Exception as exc:
        print("Error; stopping batch:", str(exc)[:250])
        download_log.append({
            **item, "status": "error", "message": str(exc)[:250]
        })
        break

log_path = WORK / "search_results" / "batch05_download_log.csv"
pd.DataFrame(download_log).to_csv(log_path, index=False)

print("\nBatch download log:", log_path)
print("Successfully downloaded:", sum(x["status"] == "downloaded" for x in download_log))
print("Manifest unchanged.")


In [ ]:

# NOTEBOOK 8 — BATCH 05 SUBTITLE QUALITY AUDIT
# Read-only audit of 10 downloaded files.
# Agneepath CD1/CD2 are also checked as a pair.

from pathlib import Path
import re
import hashlib
import pandas as pd
from collections import Counter

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"

files = [
    ("2010_5", "Housefull", RAW / "2010_5_HOUSEFULL_candidate_76927.srt"),
    ("2011_1", "Bodyguard", RAW / "2011_1_BODYGUARD_candidate_835346.srt"),
    ("2011_2", "Ready", RAW / "2011_2_READY_candidate_815931.srt"),
    ("2011_3", "Ra.One", RAW / "2011_3_RA_ONE_candidate_846831.srt"),
    ("2011_4", "Don 2", RAW / "2011_4_DON_2_candidate_4409530.srt"),
    ("2012_1", "Ek Tha Tiger", RAW / "2012_1_EK_THA_TIGER_candidate_907888.srt"),
    ("2012_2", "Dabangg 2", RAW / "2012_2_DABANGG_2_candidate_936687.srt"),
    ("2012_3", "Rowdy Rathore", RAW / "2012_3_ROWDY_RATHORE_candidate_894361.srt"),
    ("2012_4", "Agneepath CD1", RAW / "2012_4_AGNEEPATH_candidate_862905.srt"),
    ("2012_4", "Agneepath CD2", RAW / "2012_4_AGNEEPATH_candidate_862915.srt"),
]

TIME_RE = re.compile(
    r"(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def time_to_seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(str(ms).ljust(3, "0")[:3]) / 1000

def parse_srt(path):
    raw_bytes = path.read_bytes()
    digest = hashlib.sha256(raw_bytes).hexdigest()

    # Detect common encodings without changing the original file.
    encoding_used = None
    text = None
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            text = raw_bytes.decode(encoding)
            encoding_used = encoding
            break
        except UnicodeDecodeError:
            continue

    blocks = re.split(r"\n\s*\n", text.replace("\r\n", "\n").replace("\r", "\n").strip())
    cues = []
    malformed = 0
    reversed_or_zero = 0

    for block in blocks:
        if not block.strip():
            continue

        match = TIME_RE.search(block)
        if not match:
            malformed += 1
            continue

        values = match.groups()
        start = time_to_seconds(*values[:4])
        end = time_to_seconds(*values[4:])
        cue_text = block[match.end():].strip()
        cue_text = re.sub(r"^\s*\d+\s*\n", "", cue_text)
        cue_text = re.sub(r"<[^>]+>", "", cue_text)
        cue_text = re.sub(r"\{\\[^}]*\}", "", cue_text)
        cue_text = re.sub(r"\s+", " ", cue_text).strip()

        if end <= start:
            reversed_or_zero += 1

        cues.append({
            "start": start,
            "end": end,
            "text": cue_text
        })

    cues.sort(key=lambda x: (x["start"], x["end"]))

    overlaps = 0
    max_overlap = 0
    gaps_over_2m = []
    max_gap = 0

    for previous, current in zip(cues, cues[1:]):
        delta = current["start"] - previous["end"]
        if delta < 0:
            overlaps += 1
            max_overlap = max(max_overlap, abs(delta))
        elif delta > 120:
            gaps_over_2m.append({
                "gap_minutes": round(delta / 60, 2),
                "from_minute": round(previous["end"] / 60, 2),
                "to_minute": round(current["start"] / 60, 2),
            })
            max_gap = max(max_gap, delta)

    joined_text = " ".join(c["text"] for c in cues)
    words = re.findall(r"\b[\w’'-]+\b", joined_text, flags=re.UNICODE)
    nonempty_texts = [c["text"] for c in cues if c["text"]]

    devanagari = len(re.findall(r"[\u0900-\u097F]", joined_text))
    latin = len(re.findall(r"[A-Za-z]", joined_text))
    total_letters = devanagari + latin

    return {
        "record": files_record(path),
        "file": path.name,
        "bytes": len(raw_bytes),
        "sha256": digest,
        "encoding": encoding_used,
        "blocks_detected": len(blocks),
        "parsed_cues": len(cues),
        "malformed_blocks": malformed,
        "reversed_or_zero_duration": reversed_or_zero,
        "overlaps": overlaps,
        "max_overlap_seconds": round(max_overlap, 2),
        "gaps_over_2_minutes": len(gaps_over_2m),
        "largest_gap_minutes": round(max_gap / 60, 2),
        "first_cue_minute": round(cues[0]["start"] / 60, 2) if cues else None,
        "last_cue_minute": round(cues[-1]["end"] / 60, 2) if cues else None,
        "timeline_span_minutes": round((cues[-1]["end"] - cues[0]["start"]) / 60, 2) if cues else None,
        "word_count": len(words),
        "visible_characters": len(joined_text),
        "devanagari_characters": devanagari,
        "latin_letters": latin,
        "devanagari_share_pct": round(100 * devanagari / total_letters, 2) if total_letters else 0,
        "duplicate_text_groups": sum(n > 1 for n in Counter(nonempty_texts).values()),
        "_cues": cues,
        "_gaps": gaps_over_2m,
    }

def files_record(path):
    name = path.name
    match = re.match(r"(\d{4}_\d)", name)
    return match.group(1) if match else "UNKNOWN"

reports = []
parsed = {}

for record, title, path in files:
    if not path.exists():
        print("MISSING FILE:", path)
        continue

    report = parse_srt(path)
    report["movie_title"] = title
    reports.append(report)
    parsed[title] = report

summary = pd.DataFrame([
    {k: v for k, v in report.items() if not k.startswith("_")}
    for report in reports
])

display(summary[[
    "record", "movie_title", "file", "bytes", "encoding",
    "blocks_detected", "parsed_cues", "malformed_blocks",
    "reversed_or_zero_duration", "overlaps",
    "gaps_over_2_minutes", "largest_gap_minutes",
    "first_cue_minute", "last_cue_minute",
    "word_count", "devanagari_share_pct", "duplicate_text_groups"
]])

print("\nLARGEST GAPS")
for report in reports:
    if report["_gaps"]:
        print(f"\n{report['movie_title']}:")
        display(pd.DataFrame(report["_gaps"]).sort_values(
            "gap_minutes", ascending=False
        ).head(5))
    else:
        print(f"{report['movie_title']}: no internal gap exceeds 2 minutes")

# Agneepath CD pairing diagnostics.
if "Agneepath CD1" in parsed and "Agneepath CD2" in parsed:
    cd1 = parsed["Agneepath CD1"]["_cues"]
    cd2 = parsed["Agneepath CD2"]["_cues"]

    print("\nAGNEEPATH CD PAIR CHECK")
    print("CD1 cues:", len(cd1))
    print("CD2 cues:", len(cd2))

    if cd1 and cd2:
        cd1_end = cd1[-1]["end"]
        cd2_start = cd2[0]["start"]
        print("CD1 last cue end (minutes):", round(cd1_end / 60, 2))
        print("CD2 first cue start (minutes):", round(cd2_start / 60, 2))
        print("CD2 last cue end (minutes):", round(cd2[-1]["end"] / 60, 2))

        if cd2_start < cd1_end:
            print("NOTE: CD2 timestamps overlap/reset relative to CD1.")
            print("Do not concatenate until the timestamp relationship is understood.")
        else:
            print("CD2 starts at/after CD1's ending timestamp.")
            print("Check the transition gap before considering a combined timeline.")

print("\nNo subtitle files were modified.")
print("No downloads were made.")
print("Manifest unchanged.")


In [ ]:

# NOTEBOOK 8 — BATCH 05 GAP CONTEXT REVIEW
# Read-only inspection of dialogue immediately before and after large gaps.

def fmt_time(seconds):
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

def show_gap_context(title, gap, context=2):
    report = parsed[title]
    cues = report["_cues"]

    left = gap["from_minute"] * 60
    right = gap["to_minute"] * 60

    before = [c for c in cues if c["end"] <= left][-context:]
    after = [c for c in cues if c["start"] >= right][:context]

    print(f"\n{'='*70}")
    print(f"{title}: gap {gap['gap_minutes']} minutes")
    print(f"{fmt_time(left)} --> {fmt_time(right)}")
    print("- Before gap")
    for cue in before:
        print(f"[{fmt_time(cue['start'])} - {fmt_time(cue['end'])}] {cue['text']}")
    print("- After gap")
    for cue in after:
        print(f"[{fmt_time(cue['start'])} - {fmt_time(cue['end'])}] {cue['text']}")

for title in ["Bodyguard", "Ra.One", "Don 2", "Ek Tha Tiger", "Rowdy Rathore"]:
    gaps = parsed[title]["_gaps"]
    if gaps:
        # Inspect the largest gap for each movie.
        largest = max(gaps, key=lambda g: g["gap_minutes"])
        show_gap_context(title, largest)

# Agneepath: compare the end of CD1 with the beginning of CD2.
print("\n" + "="*70)
print("AGNEEPATH CD1 END")
for cue in parsed["Agneepath CD1"]["_cues"][-5:]:
    print(f"[{fmt_time(cue['start'])} - {fmt_time(cue['end'])}] {cue['text']}")

print("\nAGNEEPATH CD2 BEGINNING")
for cue in parsed["Agneepath CD2"]["_cues"][:5]:
    print(f"[{fmt_time(cue['start'])} - {fmt_time(cue['end'])}] {cue['text']}")

print("\nNo files modified. No downloads made. Manifest unchanged.")


In [ ]:

# NOTEBOOK 8 — BATCH 05 VALIDATED COPIES
# Raw files remain unchanged. Manifest remains unchanged.
# Agneepath CD2 receives a documented timestamp offset.

from pathlib import Path
import re
import hashlib
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
VALIDATED.mkdir(parents=True, exist_ok=True)

single_files = [
    ("2010_5_HOUSEFULL_candidate_76927.srt",
     "2010_5_HOUSEFULL_ENGLISH.srt"),
    ("2011_1_BODYGUARD_candidate_835346.srt",
     "2011_1_BODYGUARD_ENGLISH.srt"),
    ("2011_2_READY_candidate_815931.srt",
     "2011_2_READY_ENGLISH.srt"),
    ("2011_3_RA_ONE_candidate_846831.srt",
     "2011_3_RA_ONE_ENGLISH.srt"),
    ("2011_4_DON_2_candidate_4409530.srt",
     "2011_4_DON_2_ENGLISH.srt"),
    ("2012_1_EK_THA_TIGER_candidate_907888.srt",
     "2012_1_EK_THA_TIGER_ENGLISH.srt"),
    ("2012_2_DABANGG_2_candidate_936687.srt",
     "2012_2_DABANGG_2_ENGLISH.srt"),
    ("2012_3_ROWDY_RATHORE_candidate_894361.srt",
     "2012_3_ROWDY_RATHORE_ENGLISH.srt"),
]

TIME_RE = re.compile(
    r"(?m)^(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def to_ms(h, m, s, ms):
    return (
        int(h) * 3600000
        + int(m) * 60000
        + int(s) * 1000
        + int(str(ms).ljust(3, "0")[:3])
    )

def from_ms(value):
    value = int(value)
    h, rem = divmod(value, 3600000)
    m, rem = divmod(rem, 60000)
    s, ms = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def normalize_spacing(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n").strip()
    blocks = re.split(r"\n\s*\n", text)
    return "\n\n".join(block.strip() for block in blocks if block.strip()) + "\n"

def shift_srt(text, offset_ms):
    def replace_time(match):
        values = match.groups()
        start = to_ms(*values[:4]) + offset_ms
        end = to_ms(*values[4:]) + offset_ms
        return f"{from_ms(start)} --> {from_ms(end)}"

    return TIME_RE.sub(replace_time, text)

# Stop rather than overwrite any existing validated file.
targets = [VALIDATED / target for _, target in single_files]
targets.append(VALIDATED / "2012_4_AGNEEPATH_ENGLISH_COMBINED.srt")

existing = [str(path) for path in targets if path.exists()]
if existing:
    raise FileExistsError(
        "Validated target already exists; no files written:\n"
        + "\n".join(existing)
    )

results = []

# Create normalized copies for the eight single-file candidates.
for raw_name, validated_name in single_files:
    raw_path = RAW / raw_name
    if not raw_path.exists():
        raise FileNotFoundError(raw_path)

    raw_bytes = raw_path.read_bytes()
    text = raw_bytes.decode("utf-8-sig")
    normalized = normalize_spacing(text)
    target = VALIDATED / validated_name
    target.write_text(normalized, encoding="utf-8")

    results.append({
        "validated_file": target.name,
        "bytes": target.stat().st_size,
        "sha256": hashlib.sha256(target.read_bytes()).hexdigest(),
        "operation": "UTF-8 decode and block-spacing normalization only",
    })

# Agneepath: parse each part and offset CD2 so its first cue begins
# at CD1's last cue end. No dialogue is deleted.
cd1_path = RAW / "2012_4_AGNEEPATH_candidate_862905.srt"
cd2_path = RAW / "2012_4_AGNEEPATH_candidate_862915.srt"

cd1_text = cd1_path.read_text(encoding="utf-8-sig")
cd2_text = cd2_path.read_text(encoding="utf-8-sig")

cd1_matches = list(TIME_RE.finditer(cd1_text))
cd2_matches = list(TIME_RE.finditer(cd2_text))

if not cd1_matches or not cd2_matches:
    raise ValueError("Could not parse Agneepath CD timestamps.")

last_cd1 = cd1_matches[-1]
first_cd2 = cd2_matches[0]

cd1_end_ms = to_ms(*last_cd1.groups()[4:])
cd2_start_ms = to_ms(*first_cd2.groups()[:4])
offset_ms = cd1_end_ms - cd2_start_ms

shifted_cd2 = shift_srt(cd2_text, offset_ms)

combined_text = (
    normalize_spacing(cd1_text).rstrip()
    + "\n\n"
    + normalize_spacing(shifted_cd2).lstrip()
    + "\n"
)

combined_path = VALIDATED / "2012_4_AGNEEPATH_ENGLISH_COMBINED.srt"
combined_path.write_text(combined_text, encoding="utf-8")

# Verify the first and last timestamps after the offset.
combined_matches = list(TIME_RE.finditer(combined_text))
if not combined_matches:
    raise ValueError("Combined Agneepath file has no parseable cues.")

print("Agneepath proposed CD2 offset (milliseconds):", offset_ms)
print("Agneepath proposed CD2 offset (seconds):", round(offset_ms / 1000, 3))
print("CD1 last cue end:", from_ms(cd1_end_ms))
print("Shifted CD2 first cue start:", from_ms(
    to_ms(*combined_matches[len(cd1_matches)].groups()[:4])
))
print("Combined cue count:", len(combined_matches))
print("Combined file:", combined_path)

results.append({
    "validated_file": combined_path.name,
    "bytes": combined_path.stat().st_size,
    "sha256": hashlib.sha256(combined_path.read_bytes()).hexdigest(),
    "operation": f"CD1 + CD2; CD2 offset {offset_ms} ms",
})

print("\nCREATED VALIDATED COPIES")
display(pd.DataFrame(results))

print("\nRaw subtitle files unchanged.")
print("Manifest unchanged.")


In [ ]:

# NOTEBOOK 8 — BATCH 05 FINAL INTEGRITY CHECK
# Read-only. Does not update the manifest.

from pathlib import Path
import re
import hashlib
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
VALIDATED = PROJECT / "01_data/processed/subtitle_corpus_v2/validated"

targets = [
    ("2010_5", "Housefull", "2010_5_HOUSEFULL_ENGLISH.srt"),
    ("2011_1", "Bodyguard", "2011_1_BODYGUARD_ENGLISH.srt"),
    ("2011_2", "Ready", "2011_2_READY_ENGLISH.srt"),
    ("2011_3", "Ra.One", "2011_3_RA_ONE_ENGLISH.srt"),
    ("2011_4", "Don 2", "2011_4_DON_2_ENGLISH.srt"),
    ("2012_1", "Ek Tha Tiger", "2012_1_EK_THA_TIGER_ENGLISH.srt"),
    ("2012_2", "Dabangg 2", "2012_2_DABANGG_2_ENGLISH.srt"),
    ("2012_3", "Rowdy Rathore", "2012_3_ROWDY_RATHORE_ENGLISH.srt"),
    ("2012_4", "Agneepath", "2012_4_AGNEEPATH_ENGLISH_COMBINED.srt"),
]

TIME_RE = re.compile(
    r"(?m)^(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def to_ms(h, m, s, ms):
    return (
        int(h) * 3600000
        + int(m) * 60000
        + int(s) * 1000
        + int(str(ms).ljust(3, "0")[:3])
    )

results = []
all_cues = {}

for record, title, filename in targets:
    path = VALIDATED / filename
    if not path.exists():
        raise FileNotFoundError(path)

    text = path.read_text(encoding="utf-8-sig")
    blocks = [b for b in re.split(r"\n\s*\n", text.strip()) if b.strip()]
    matches = list(TIME_RE.finditer(text))

    cues = []
    for match in matches:
        values = match.groups()
        start = to_ms(*values[:4])
        end = to_ms(*values[4:])
        cues.append((start, end, text[match.end():].split("\n", 1)[-1].strip()))

    malformed = len(blocks) - len(matches)
    reversed_zero = sum(end <= start for start, end, _ in cues)

    # Detect timeline issues in original file order and chronological order.
    file_order_reversals = sum(
        cues[i][0] < cues[i - 1][0] for i in range(1, len(cues))
    )
    chronological = sorted(cues, key=lambda c: (c[0], c[1]))

    overlaps = sum(
        chronological[i][0] < chronological[i - 1][1]
        for i in range(1, len(chronological))
    )
    gaps = [
        (chronological[i][0] - chronological[i - 1][1])
        for i in range(1, len(chronological))
        if chronological[i][0] - chronological[i - 1][1] > 120000
    ]

    visible_text = " ".join(c[2] for c in cues)
    words = re.findall(r"\b[\w’'-]+\b", visible_text, flags=re.UNICODE)

    all_cues[title] = cues

    results.append({
        "record": record,
        "movie": title,
        "file": filename,
        "cue_blocks": len(blocks),
        "parsed_timestamps": len(matches),
        "malformed_blocks": malformed,
        "reversed_or_zero_duration": reversed_zero,
        "file_order_reversals": file_order_reversals,
        "overlaps": overlaps,
        "gaps_over_2m": len(gaps),
        "largest_gap_minutes": round(max(gaps) / 60000, 2) if gaps else 0,
        "first_minute": round(min((c[0] for c in cues), default=0) / 60000, 2),
        "last_minute": round(max((c[1] for c in cues), default=0) / 60000, 2),
        "word_count": len(words),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "integrity_pass": (
            len(blocks) == len(matches)
            and malformed == 0
            and reversed_zero == 0
            and file_order_reversals == 0
        ),
    })

summary = pd.DataFrame(results)
display(summary)

# Agneepath transition check: CD1 raw ending and combined file boundary.
cd1_path = (
    PROJECT / "01_data/processed/subtitle_corpus_v2/raw/"
    "2012_4_AGNEEPATH_candidate_862905.srt"
)
cd2_path = (
    PROJECT / "01_data/processed/subtitle_corpus_v2/raw/"
    "2012_4_AGNEEPATH_candidate_862915.srt"
)

cd1_text = cd1_path.read_text(encoding="utf-8-sig")
cd2_text = cd2_path.read_text(encoding="utf-8-sig")
cd1_matches = list(TIME_RE.finditer(cd1_text))
cd2_matches = list(TIME_RE.finditer(cd2_text))
combined_cues = all_cues["Agneepath"]

cd1_end_ms = to_ms(*cd1_matches[-1].groups()[4:])
cd2_start_ms = to_ms(*cd2_matches[0].groups()[:4])
expected_offset_ms = cd1_end_ms - cd2_start_ms

# The combined file preserves CD1 cues first, followed by shifted CD2 cues.
boundary_left = combined_cues[len(cd1_matches) - 1]
boundary_right = combined_cues[len(cd1_matches)]

print("\nAGNEEPATH BOUNDARY")
print("Expected CD2 offset (ms):", expected_offset_ms)
print("CD1 final cue end (ms):", boundary_left[1])
print("Shifted CD2 first cue start (ms):", boundary_right[0])
print("Boundary gap/overlap (ms):", boundary_right[0] - boundary_left[1])
print("Agneepath boundary continuous:", boundary_right[0] == boundary_left[1])

print("\nOVERALL CHECK")
print("All nine files present:", len(summary) == 9)
print("All blocks parse:", bool((summary["malformed_blocks"] == 0).all()))
print("No reversed/zero cues:", bool((summary["reversed_or_zero_duration"] == 0).all()))
print("No file-order reversals:", bool((summary["file_order_reversals"] == 0).all()))
print("All integrity checks pass:", bool(summary["integrity_pass"].all()))
print("\nNo files modified. No downloads made. Manifest unchanged.")


In [ ]:

# NOTEBOOK 8 — CORRECTED WORD COUNTS
# Read-only; counts each subtitle cue's text exactly once.

from pathlib import Path
import re
import pandas as pd

VALIDATED = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/validated"
)

files = [
    ("2010_5", "Housefull", "2010_5_HOUSEFULL_ENGLISH.srt"),
    ("2011_1", "Bodyguard", "2011_1_BODYGUARD_ENGLISH.srt"),
    ("2011_2", "Ready", "2011_2_READY_ENGLISH.srt"),
    ("2011_3", "Ra.One", "2011_3_RA_ONE_ENGLISH.srt"),
    ("2011_4", "Don 2", "2011_4_DON_2_ENGLISH.srt"),
    ("2012_1", "Ek Tha Tiger", "2012_1_EK_THA_TIGER_ENGLISH.srt"),
    ("2012_2", "Dabangg 2", "2012_2_DABANGG_2_ENGLISH.srt"),
    ("2012_3", "Rowdy Rathore", "2012_3_ROWDY_RATHORE_ENGLISH.srt"),
    ("2012_4", "Agneepath", "2012_4_AGNEEPATH_ENGLISH_COMBINED.srt"),
]

TIME_RE = re.compile(
    r"(?m)^(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

rows = []

for record, title, filename in files:
    path = VALIDATED / filename
    text = path.read_text(encoding="utf-8-sig")
    matches = list(TIME_RE.finditer(text))
    cue_texts = []

    for i, match in enumerate(matches):
        end_position = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        segment = text[match.end():end_position]
        segment = re.sub(r"^\s*\d+\s*(?:\n|$)", "", segment)
        segment = re.sub(r"<[^>]+>", " ", segment)
        segment = re.sub(r"\{\\[^}]*\}", " ", segment)
        cue_texts.append(segment.strip())

    # Remove only formatting/line breaks; count the words in cue text once.
    combined_text = " ".join(cue_texts)
    word_count = len(re.findall(r"\b[\w’'-]+\b", combined_text, flags=re.UNICODE))

    rows.append({
        "record": record,
        "movie": title,
        "parsed_cues": len(matches),
        "corrected_word_count": word_count,
        "file_bytes": path.stat().st_size,
    })

display(pd.DataFrame(rows))
print("Read-only check complete. Manifest unchanged.")


In [ ]:

# NOTEBOOK 8 — BATCH 05 MANIFEST UPDATE PREVIEW
# Read-only: no manifest changes are made.

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
VALIDATED = WORK / "validated"

manifest = pd.read_csv(MANIFEST, dtype={"movie_record_id": str})

print("Current manifest rows:", len(manifest))
print("Manifest columns:")
print(manifest.columns.tolist())

print("\nCurrent status values:")
for col in [
    "acquisition_status", "validation_status",
    "subtitle_language", "subtitle_type", "subtitle_source",
    "encoding_status", "quality_status"
]:
    if col in manifest.columns:
        print(f"\n{col}:")
        print(manifest[col].value_counts(dropna=False).head(12).to_dict())

new_records = [
    {
        "movie_record_id": "2010_5", "year": 2010, "rank": 5,
        "movie_title": "Housefull", "language": "Hindi", "tmdb_id": 58051,
        "subtitle_language": "English", "source_id": "74355",
        "raw": "2010_5_HOUSEFULL_candidate_76927.srt",
        "validated": "2010_5_HOUSEFULL_ENGLISH.srt",
        "words": 14452, "cues": 2220,
        "quality": "Validated; no structural errors; no gaps over 2 minutes."
    },
    {
        "movie_record_id": "2011_1", "year": 2011, "rank": 1,
        "movie_title": "Bodyguard", "language": "Hindi", "tmdb_id": 73582,
        "subtitle_language": "English", "source_id": "760478",
        "raw": "2011_1_BODYGUARD_candidate_835346.srt",
        "validated": "2011_1_BODYGUARD_ENGLISH.srt",
        "words": 9766, "cues": 1618,
        "quality": "Validated; coverage flag: 2.44-minute internal gap."
    },
    {
        "movie_record_id": "2011_2", "year": 2011, "rank": 2,
        "movie_title": "Ready", "language": "Hindi", "tmdb_id": 65946,
        "subtitle_language": "English", "source_id": "742112",
        "raw": "2011_2_READY_candidate_815931.srt",
        "validated": "2011_2_READY_ENGLISH.srt",
        "words": 19648, "cues": 2548,
        "quality": "Validated; no structural errors; no gaps over 2 minutes."
    },
    {
        "movie_record_id": "2011_3", "year": 2011, "rank": 3,
        "movie_title": "Ra.One", "language": "Hindi", "tmdb_id": 41517,
        "subtitle_language": "English", "source_id": "771341",
        "raw": "2011_3_RA_ONE_candidate_846831.srt",
        "validated": "2011_3_RA_ONE_ENGLISH.srt",
        "words": 8825, "cues": 1559,
        "quality": "Validated; coverage flag: two internal gaps over 2 minutes."
    },
    {
        "movie_record_id": "2011_4", "year": 2011, "rank": 4,
        "movie_title": "Don 2", "language": "Hindi", "tmdb_id": 41109,
        "subtitle_language": "English", "source_id": "4319929",
        "raw": "2011_4_DON_2_candidate_4409530.srt",
        "validated": "2011_4_DON_2_ENGLISH.srt",
        "words": 8997, "cues": 1454,
        "quality": "Validated; coverage flag: three internal gaps over 2 minutes."
    },
    {
        "movie_record_id": "2011_5", "year": 2011, "rank": 5,
        "movie_title": "Singham", "language": "Hindi", "tmdb_id": 70590,
        "subtitle_language": "Hindi", "source_id": "11537175",
        "raw": "2011_5_SINGHAM_candidate_11537175.srt",
        "validated": "2011_5_SINGHAM_HINDI.srt",
        "words": 21996, "cues": 2095,
        "quality": "Validated Hindi-original subtitle; six overlaps and one 2.02-minute gap; no malformed or reversed cues."
    },
    {
        "movie_record_id": "2012_1", "year": 2012, "rank": 1,
        "movie_title": "Ek Tha Tiger", "language": "Hindi", "tmdb_id": 85985,
        "subtitle_language": "English", "source_id": "829082",
        "raw": "2012_1_EK_THA_TIGER_candidate_907888.srt",
        "validated": "2012_1_EK_THA_TIGER_ENGLISH.srt",
        "words": 8118, "cues": 1163,
        "quality": "Validated; coverage flag: four internal gaps over 2 minutes; largest 5.95 minutes."
    },
    {
        "movie_record_id": "2012_2", "year": 2012, "rank": 2,
        "movie_title": "Dabangg 2", "language": "Hindi", "tmdb_id": 147405,
        "subtitle_language": "English", "source_id": "856327",
        "raw": "2012_2_DABANGG_2_candidate_936687.srt",
        "validated": "2012_2_DABANGG_2_ENGLISH.srt",
        "words": 10996, "cues": 1780,
        "quality": "Validated; no structural errors; no gaps over 2 minutes."
    },
    {
        "movie_record_id": "2012_3", "year": 2012, "rank": 3,
        "movie_title": "Rowdy Rathore", "language": "Hindi", "tmdb_id": 102632,
        "subtitle_language": "English", "source_id": "816442",
        "raw": "2012_3_ROWDY_RATHORE_candidate_894361.srt",
        "validated": "2012_3_ROWDY_RATHORE_ENGLISH.srt",
        "words": 9470, "cues": 1505,
        "quality": "Validated; coverage flag: one internal gap over 2 minutes."
    },
    {
        "movie_record_id": "2012_4", "year": 2012, "rank": 4,
        "movie_title": "Agneepath", "language": "Hindi", "tmdb_id": 84858,
        "subtitle_language": "English", "source_id": "786654",
        "raw": "2012_4_AGNEEPATH_candidate_862905.srt; 2012_4_AGNEEPATH_candidate_862915.srt",
        "validated": "2012_4_AGNEEPATH_ENGLISH_COMBINED.srt",
        "words": 12166, "cues": 1981,
        "quality": "Validated combined CD1/CD2; CD2 offset +5518103 ms; no overlaps or malformed timestamps; two internal gaps over 2 minutes."
    },
]

# Confirm that every planned file exists before preparing rows.
for record in new_records:
    raw_names = record["raw"].split("; ")
    for name in raw_names:
        path = WORK / "raw" / name
        if not path.exists():
            raise FileNotFoundError(f"Missing raw file: {path}")
    validated_path = VALIDATED / record["validated"]
    if not validated_path.exists():
        raise FileNotFoundError(f"Missing validated file: {validated_path}")

planned_ids = [r["movie_record_id"] for r in new_records]
existing_ids = set(manifest["movie_record_id"].dropna().astype(str))

print("\nPlanned additions:", len(new_records))
print("IDs already in manifest:", sorted(set(planned_ids) & existing_ids))
print("IDs planned:", planned_ids)

preview = pd.DataFrame([
    {
        "movie_record_id": r["movie_record_id"],
        "movie_title": r["movie_title"],
        "year": r["year"],
        "subtitle_language": r["subtitle_language"],
        "source_id": r["source_id"],
        "raw_file(s)": r["raw"],
        "validated_file": r["validated"],
        "cues": r["cues"],
        "words": r["words"],
        "quality_notes": r["quality"],
    }
    for r in new_records
])

display(preview)
print("\nPREVIEW ONLY — manifest unchanged.")


In [ ]:

# NOTEBOOK 8 — BATCH 05 MANIFEST UPDATE
# Appends Singham + the remaining nine movies.
# Locked master/mapping/features files are not modified.

from pathlib import Path
from datetime import datetime
import pandas as pd
import hashlib
import re
import shutil

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

expected_columns = [
    "movie_record_id", "year", "rank", "movie_title", "language",
    "tmdb_id", "imdb_id", "subtitle_language", "subtitle_type",
    "subtitle_source", "source_id", "source_url", "raw_subtitle_file",
    "validated_subtitle_file", "acquisition_status", "validation_status",
    "title_verified", "year_verified", "movie_identity_verified",
    "language_verified", "dialogue_blocks", "word_count",
    "character_count", "encoding_status", "quality_status", "notes"
]

if not MANIFEST.exists():
    raise FileNotFoundError(MANIFEST)

manifest = pd.read_csv(MANIFEST, dtype={"movie_record_id": str})

if list(manifest.columns) != expected_columns:
    raise ValueError(
        "Manifest schema differs from the reviewed schema. No changes saved."
    )

if manifest["movie_record_id"].dropna().duplicated().any():
    raise ValueError("Existing duplicate IDs detected. No changes saved.")

# The preview established these IDs are new. Recheck immediately before saving.
new_records = [
    {
        "movie_record_id": "2010_5", "year": 2010, "rank": 5,
        "movie_title": "Housefull", "language": "Hindi", "tmdb_id": 58051,
        "subtitle_language": "en", "subtitle_type": "English_fallback",
        "source_id": "74355",
        "raw_files": ["2010_5_HOUSEFULL_candidate_76927.srt"],
        "validated_file": "2010_5_HOUSEFULL_ENGLISH.srt",
        "cues": 2220, "words": 14452,
        "acquisition": "Accepted_English_fallback",
        "validation": "Validated",
        "quality": "Accepted",
        "notes": "OpenSubtitles English fallback. 2220 cues; no malformed, reversed, zero-duration, overlapping cues, or internal gaps over 2 minutes. Completeness not independently guaranteed."
    },
    {
        "movie_record_id": "2011_1", "year": 2011, "rank": 1,
        "movie_title": "Bodyguard", "language": "Hindi", "tmdb_id": 73582,
        "subtitle_language": "en", "subtitle_type": "English_fallback",
        "source_id": "760478",
        "raw_files": ["2011_1_BODYGUARD_candidate_835346.srt"],
        "validated_file": "2011_1_BODYGUARD_ENGLISH.srt",
        "cues": 1618, "words": 9766,
        "acquisition": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_coverage_gap_flag",
        "notes": "OpenSubtitles English fallback. No structural errors or overlaps; one internal gap of 2.44 minutes (10.75–13.20 minutes)."
    },
    {
        "movie_record_id": "2011_2", "year": 2011, "rank": 2,
        "movie_title": "Ready", "language": "Hindi", "tmdb_id": 65946,
        "subtitle_language": "en", "subtitle_type": "English_fallback",
        "source_id": "742112",
        "raw_files": ["2011_2_READY_candidate_815931.srt"],
        "validated_file": "2011_2_READY_ENGLISH.srt",
        "cues": 2548, "words": 19648,
        "acquisition": "Accepted_English_fallback",
        "validation": "Validated",
        "quality": "Accepted",
        "notes": "OpenSubtitles English fallback. 2548 cues; no malformed, reversed, zero-duration, overlapping cues, or internal gaps over 2 minutes. Completeness not independently guaranteed."
    },
    {
        "movie_record_id": "2011_3", "year": 2011, "rank": 3,
        "movie_title": "Ra.One", "language": "Hindi", "tmdb_id": 41517,
        "subtitle_language": "en", "subtitle_type": "English_fallback",
        "source_id": "771341",
        "raw_files": ["2011_3_RA_ONE_candidate_846831.srt"],
        "validated_file": "2011_3_RA_ONE_ENGLISH.srt",
        "cues": 1559, "words": 8825,
        "acquisition": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_coverage_gap_flag",
        "notes": "OpenSubtitles English fallback. No structural errors or overlaps; two internal gaps over 2 minutes, largest 2.57 minutes."
    },
    {
        "movie_record_id": "2011_4", "year": 2011, "rank": 4,
        "movie_title": "Don 2", "language": "Hindi", "tmdb_id": 41109,
        "subtitle_language": "en", "subtitle_type": "English_fallback",
        "source_id": "4319929",
        "raw_files": ["2011_4_DON_2_candidate_4409530.srt"],
        "validated_file": "2011_4_DON_2_ENGLISH.srt",
        "cues": 1454, "words": 8997,
        "acquisition": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_coverage_gap_flag",
        "notes": "OpenSubtitles English fallback. No structural errors or overlaps; three internal gaps over 2 minutes, largest 2.43 minutes."
    },
    {
        "movie_record_id": "2011_5", "year": 2011, "rank": 5,
        "movie_title": "Singham", "language": "Hindi", "tmdb_id": 70590,
        "subtitle_language": "hi", "subtitle_type": "Original_Language",
        "source_id": "11537175",
        "raw_files": ["2011_5_SINGHAM_candidate_11537175.srt"],
        "validated_file": "2011_5_SINGHAM_HINDI.srt",
        "cues": 2095, "words": 21996,
        "acquisition": "Accepted_original_language_review_flag",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_quality_flags",
        "notes": "OpenSubtitles Hindi-original subtitle. 2095 cues; Devanagari share 94.42%; six overlaps and one 2.02-minute gap; no malformed or reversed cues. Kept with documented flags."
    },
    {
        "movie_record_id": "2012_1", "year": 2012, "rank": 1,
        "movie_title": "Ek Tha Tiger", "language": "Hindi", "tmdb_id": 85985,
        "subtitle_language": "en", "subtitle_type": "English_fallback",
        "source_id": "829082",
        "raw_files": ["2012_1_EK_THA_TIGER_candidate_907888.srt"],
        "validated_file": "2012_1_EK_THA_TIGER_ENGLISH.srt",
        "cues": 1163, "words": 8118,
        "acquisition": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_coverage_gap_flag",
        "notes": "OpenSubtitles English fallback. No structural errors or overlaps; four internal gaps over 2 minutes, largest 5.95 minutes. Completeness should be treated cautiously."
    },
    {
        "movie_record_id": "2012_2", "year": 2012, "rank": 2,
        "movie_title": "Dabangg 2", "language": "Hindi", "tmdb_id": 147405,
        "subtitle_language": "en", "subtitle_type": "English_fallback",
        "source_id": "856327",
        "raw_files": ["2012_2_DABANGG_2_candidate_936687.srt"],
        "validated_file": "2012_2_DABANGG_2_ENGLISH.srt",
        "cues": 1780, "words": 10996,
        "acquisition": "Accepted_English_fallback",
        "validation": "Validated",
        "quality": "Accepted",
        "notes": "OpenSubtitles English fallback. 1780 cues; no malformed, reversed, zero-duration, overlapping cues, or internal gaps over 2 minutes. Completeness not independently guaranteed."
    },
    {
        "movie_record_id": "2012_3", "year": 2012, "rank": 3,
        "movie_title": "Rowdy Rathore", "language": "Hindi", "tmdb_id": 102632,
        "subtitle_language": "en", "subtitle_type": "English_fallback",
        "source_id": "816442",
        "raw_files": ["2012_3_ROWDY_RATHORE_candidate_894361.srt"],
        "validated_file": "2012_3_ROWDY_RATHORE_ENGLISH.srt",
        "cues": 1505, "words": 9470,
        "acquisition": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_coverage_gap_flag",
        "notes": "OpenSubtitles English fallback. No structural errors or overlaps; one internal gap of 2.05 minutes."
    },
    {
        "movie_record_id": "2012_4", "year": 2012, "rank": 4,
        "movie_title": "Agneepath", "language": "Hindi", "tmdb_id": 84858,
        "subtitle_language": "en", "subtitle_type": "English_fallback",
        "source_id": "786654",
        "raw_files": [
            "2012_4_AGNEEPATH_candidate_862905.srt",
            "2012_4_AGNEEPATH_candidate_862915.srt"
        ],
        "validated_file": "2012_4_AGNEEPATH_ENGLISH_COMBINED.srt",
        "cues": 1981, "words": 12166,
        "acquisition": "Accepted_English_fallback",
        "validation": "Validated_with_quality_flags",
        "quality": "Accepted_with_quality_flags",
        "notes": "OpenSubtitles English fallback, two-part subtitle set. CD2 offset +5518103 ms aligns its first cue with CD1's last cue end. Combined file has no malformed, reversed, zero-duration, overlapping cues; two internal gaps over 2 minutes remain."
    },
]

new_ids = [r["movie_record_id"] for r in new_records]
existing_ids = set(manifest["movie_record_id"].dropna().astype(str))

if len(new_ids) != len(set(new_ids)):
    raise ValueError("Duplicate IDs inside planned additions. Nothing saved.")

conflicts = sorted(set(new_ids) & existing_ids)
if conflicts:
    raise ValueError(f"IDs already exist in manifest: {conflicts}. Nothing saved.")

def read_srt_text(path):
    return path.read_text(encoding="utf-8-sig")

def count_visible_characters(text):
    # Count cue text, excluding sequence numbers and timestamp lines.
    cue_text = re.sub(
        r"(?m)^\s*\d+\s*$|"
        r"^\s*\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*-->\s*"
        r"\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*$",
        "",
        text
    )
    cue_text = re.sub(r"<[^>]+>", "", cue_text)
    cue_text = re.sub(r"\{\\[^}]*\}", "", cue_text)
    return len(re.sub(r"\s+", "", cue_text))

rows_to_append = []

for item in new_records:
    raw_paths = [RAW / name for name in item["raw_files"]]
    validated_path = VALIDATED / item["validated_file"]

    for path in raw_paths + [validated_path]:
        if not path.exists():
            raise FileNotFoundError(f"Required subtitle file missing: {path}")

    validated_text = read_srt_text(validated_path)
    character_count = count_visible_characters(validated_text)

    raw_display = "; ".join(str(path) for path in raw_paths)

    row = {column: None for column in expected_columns}
    row.update({
        "movie_record_id": item["movie_record_id"],
        "year": item["year"],
        "rank": item["rank"],
        "movie_title": item["movie_title"],
        "language": item["language"],
        "tmdb_id": item["tmdb_id"],
        "imdb_id": None,
        "subtitle_language": item["subtitle_language"],
        "subtitle_type": item["subtitle_type"],
        "subtitle_source": "OpenSubtitles",
        "source_id": item["source_id"],
        "source_url": None,
        "raw_subtitle_file": raw_display,
        "validated_subtitle_file": str(validated_path),
        "acquisition_status": item["acquisition"],
        "validation_status": item["validation"],
        "title_verified": True,
        "year_verified": True,
        "movie_identity_verified": True,
        "language_verified": True,
        "dialogue_blocks": item["cues"],
        "word_count": item["words"],
        "character_count": character_count,
        "encoding_status": "Decoded_utf-8-sig_and_saved_UTF8",
        "quality_status": item["quality"],
        "notes": item["notes"],
    })
    rows_to_append.append(row)

append_df = pd.DataFrame(rows_to_append, columns=expected_columns)

# Final guards before writing.
if len(append_df) != 10:
    raise ValueError("Expected exactly 10 additions. Nothing saved.")

if append_df["movie_record_id"].duplicated().any():
    raise ValueError("Planned additions contain duplicate IDs. Nothing saved.")

if set(append_df["movie_record_id"]) & existing_ids:
    raise ValueError("ID conflict detected on final check. Nothing saved.")

# Backup the original manifest before appending.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = MANIFEST.with_name(
    f"{MANIFEST.stem}_backup_batch05_{timestamp}.csv"
)
shutil.copy2(MANIFEST, backup_path)

updated = pd.concat([manifest, append_df], ignore_index=True)
updated.to_csv(MANIFEST, index=False)

# Read back from disk and verify persisted result.
check = pd.read_csv(MANIFEST, dtype={"movie_record_id": str})
expected_rows = len(manifest) + len(append_df)

if len(check) != expected_rows:
    raise RuntimeError(
        f"Saved row count mismatch: expected {expected_rows}, got {len(check)}. "
        f"Backup: {backup_path}"
    )

if check["movie_record_id"].dropna().duplicated().any():
    raise RuntimeError(
        f"Duplicate IDs detected after save. Original backup: {backup_path}"
    )

if not set(new_ids).issubset(set(check["movie_record_id"].dropna().astype(str))):
    raise RuntimeError("One or more additions are missing after save.")

print("MANIFEST UPDATE SUCCESSFUL")
print("Backup:", backup_path)
print("Rows before:", len(manifest))
print("Rows appended:", len(append_df))
print("Rows after:", len(check))
print("Duplicate movie_record_id values:", int(
    check["movie_record_id"].dropna().duplicated().sum()
))
print("New IDs:", new_ids)
print("Manifest path:", MANIFEST)
print("Locked master, mapping, metadata, financials, and features files were not modified.")


In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# NOTEBOOK 8 — READ-ONLY MANIFEST CONSISTENCY AUDIT
# ============================================================

PROJECT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

MASTER = (
    PROJECT
    / "01_data/processed/ICMB_130_MASTER_DATA.csv"
)

print("=" * 65)
print("ICMB-130 — SUBTITLE MANIFEST CONSISTENCY AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Check that required files exist
# ------------------------------------------------------------

print("\n1. REQUIRED FILE CHECK")

for path in [MANIFEST, MASTER]:
    print(f"{path.name}: {'EXISTS' if path.exists() else 'MISSING'}")

if not MANIFEST.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")

if not MASTER.exists():
    raise FileNotFoundError(f"Locked master not found: {MASTER}")

# ------------------------------------------------------------
# 2. Load files without modifying them
# ------------------------------------------------------------

manifest = pd.read_csv(MANIFEST, dtype=str).fillna("")
master = pd.read_csv(MASTER, dtype=str).fillna("")

print("\n2. DATASET SHAPES")
print("Manifest shape:", manifest.shape)
print("Master shape:", master.shape)

# ------------------------------------------------------------
# 3. Check required manifest columns
# ------------------------------------------------------------

required_columns = [
    "movie_record_id",
    "movie_title",
    "acquisition_status",
    "validation_status",
    "raw_subtitle_file",
    "validated_subtitle_file",
]

missing_columns = [
    col for col in required_columns
    if col not in manifest.columns
]

print("\n3. REQUIRED COLUMN CHECK")

if missing_columns:
    print("MISSING COLUMNS:", missing_columns)
else:
    print("PASS — all required columns are present")

# Stop checks that depend on missing columns
if missing_columns:
    raise ValueError(
        "Manifest schema is incomplete. Review the missing columns above."
    )

# ------------------------------------------------------------
# 4. Check duplicate and blank movie_record_id values
# ------------------------------------------------------------

print("\n4. MOVIE RECORD ID CHECK")

blank_ids = manifest["movie_record_id"].str.strip().eq("")
duplicate_mask = (
    manifest["movie_record_id"].duplicated(keep=False)
    & ~blank_ids
)

print("Blank movie_record_id values:", int(blank_ids.sum()))
print("Duplicate movie_record_id values:", int(duplicate_mask.sum()))

if duplicate_mask.any():
    print("\nDuplicate records:")
    display(
        manifest.loc[
            duplicate_mask,
            ["movie_record_id", "movie_title", "acquisition_status"]
        ].sort_values("movie_record_id")
    )

# ------------------------------------------------------------
# 5. Check master ID column
# ------------------------------------------------------------

print("\n5. MASTER ID COLUMN CHECK")

possible_id_columns = [
    "movie_record_id",
    "record_id",
    "dataset_id",
]

master_id_col = next(
    (col for col in possible_id_columns if col in master.columns),
    None
)

if master_id_col is None:
    print("Could not identify a standard movie ID column automatically.")
    print("Master columns:", master.columns.tolist())
else:
    master_ids = set(
        master[master_id_col].astype(str).str.strip()
    )
    manifest_ids = set(
        manifest["movie_record_id"].astype(str).str.strip()
    )

    master_ids.discard("")
    manifest_ids.discard("")

    print("Master ID column:", master_id_col)
    print("Unique master IDs:", len(master_ids))
    print("Unique manifest IDs:", len(manifest_ids))

    not_in_master = sorted(manifest_ids - master_ids)
    not_in_manifest = sorted(master_ids - manifest_ids)

    print("Manifest IDs not found in master:", len(not_in_master))
    print("Master IDs absent from manifest:", len(not_in_manifest))

    if not_in_master:
        print("\nManifest IDs not found in master:")
        print(not_in_master)

    if not_in_manifest:
        print("\nMaster IDs absent from manifest (first 30):")
        print(not_in_manifest[:30])

# ------------------------------------------------------------
# 6. Check raw and validated file paths
# ------------------------------------------------------------

print("\n6. SUBTITLE FILE PATH CHECK")

def resolve_manifest_path(value):
    """Resolve a stored path; support semicolon-separated raw paths."""
    value = str(value).strip()

    if not value:
        return []

    paths = []
    for part in value.split(";"):
        part = part.strip()
        if not part:
            continue

        path = Path(part)

        # Support paths stored relative to the subtitle corpus folder
        if not path.is_absolute():
            path = WORK / path

        paths.append(path)

    return paths


missing_raw = []
missing_validated = []
empty_validated_paths = []

for _, row in manifest.iterrows():
    record_id = row["movie_record_id"]

    raw_value = row.get("raw_subtitle_file", "")
    validated_value = row.get("validated_subtitle_file", "")

    # Check raw files when a path is recorded
    for path in resolve_manifest_path(raw_value):
        if not path.is_file():
            missing_raw.append({
                "movie_record_id": record_id,
                "movie_title": row["movie_title"],
                "missing_path": str(path),
            })

    # Check validated files when a path is recorded
    validated_paths = resolve_manifest_path(validated_value)

    if not validated_paths:
        empty_validated_paths.append({
            "movie_record_id": record_id,
            "movie_title": row["movie_title"],
            "acquisition_status": row["acquisition_status"],
            "validation_status": row["validation_status"],
        })
    else:
        for path in validated_paths:
            if not path.is_file():
                missing_validated.append({
                    "movie_record_id": record_id,
                    "movie_title": row["movie_title"],
                    "missing_path": str(path),
                })

print("Missing recorded raw files:", len(missing_raw))
print("Missing recorded validated files:", len(missing_validated))
print("Records without validated-file paths:", len(empty_validated_paths))

if missing_raw:
    print("\nMissing raw files:")
    display(pd.DataFrame(missing_raw))

if missing_validated:
    print("\nMissing validated files:")
    display(pd.DataFrame(missing_validated))

if empty_validated_paths:
    print("\nRecords without validated-file paths:")
    display(pd.DataFrame(empty_validated_paths))

# ------------------------------------------------------------
# 7. Summarize statuses
# ------------------------------------------------------------

print("\n7. ACQUISITION STATUS SUMMARY")
display(
    manifest["acquisition_status"]
    .replace("", "(blank)")
    .value_counts()
    .rename_axis("acquisition_status")
    .reset_index(name="records")
)

print("\n8. VALIDATION STATUS SUMMARY")
display(
    manifest["validation_status"]
    .replace("", "(blank)")
    .value_counts()
    .rename_axis("validation_status")
    .reset_index(name="records")
)

# ------------------------------------------------------------
# 9. Focused review of latest Batch 05 records
# ------------------------------------------------------------

print("\n9. BATCH 05 RECORD REVIEW")

batch05_ids = [
    "2010_5", "2011_1", "2011_2", "2011_3", "2011_4",
    "2011_5", "2012_1", "2012_2", "2012_3", "2012_4",
]

batch05 = manifest[
    manifest["movie_record_id"].isin(batch05_ids)
].copy()

print("Expected Batch 05 records:", len(batch05_ids))
print("Batch 05 records found:", len(batch05))

review_columns = [
    col for col in [
        "movie_record_id",
        "movie_title",
        "language",
        "subtitle_language",
        "subtitle_type",
        "acquisition_status",
        "validation_status",
        "quality_status",
        "dialogue_blocks",
        "word_count",
        "raw_subtitle_file",
        "validated_subtitle_file",
    ]
    if col in batch05.columns
]

display(
    batch05[review_columns].sort_values("movie_record_id")
)

missing_batch05_ids = sorted(
    set(batch05_ids) - set(batch05["movie_record_id"])
)

if missing_batch05_ids:
    print("MISSING BATCH 05 IDs:", missing_batch05_ids)

# ------------------------------------------------------------
# 10. Final audit summary — no files are changed
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("READ-ONLY AUDIT SUMMARY")
print("=" * 65)
print("Manifest rows:", len(manifest))
print("Duplicate IDs:", int(duplicate_mask.sum()))
print("Blank IDs:", int(blank_ids.sum()))
print("Missing raw files:", len(missing_raw))
print("Missing validated files:", len(missing_validated))
print("Batch 05 records found:", len(batch05), "/", len(batch05_ids))
print("AUDIT COMPLETE — NO FILES MODIFIED")
print("=" * 65)

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# NOTEBOOK 8 — DIAGNOSE MANIFEST FILE PATHS
# READ-ONLY: NO FILES MODIFIED
# ============================================================

WORK = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

RAW = WORK / "raw"
VALIDATED = WORK / "validated"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(MANIFEST, dtype=str).fillna("")

print("Raw directory exists:", RAW.is_dir())
print("Validated directory exists:", VALIDATED.is_dir())

# Index files by basename for fast lookup
raw_index = {
    p.name: str(p)
    for p in RAW.iterdir()
    if p.is_file()
} if RAW.is_dir() else {}

validated_index = {
    p.name: str(p)
    for p in VALIDATED.iterdir()
    if p.is_file()
} if VALIDATED.is_dir() else {}

def check_paths(value, index):
    value = str(value).strip()

    if not value:
        return "NO_PATH_RECORDED", ""

    results = []
    found_locations = []

    for item in value.split(";"):
        item = item.strip()
        if not item:
            continue

        p = Path(item)

        # First test the exact recorded path
        if p.is_file():
            results.append("EXACT_PATH_EXISTS")
            found_locations.append(str(p))
        # Then test whether a file with the same name exists
        elif p.name in index:
            results.append("FOUND_BY_FILENAME")
            found_locations.append(index[p.name])
        else:
            results.append("NOT_FOUND")

    if not results:
        return "NO_PATH_RECORDED", ""

    if all(r in ("EXACT_PATH_EXISTS", "FOUND_BY_FILENAME") for r in results):
        if "FOUND_BY_FILENAME" in results:
            return "FOUND_BUT_PATH_DIFFERS", "; ".join(found_locations)
        return "EXACT_PATH_EXISTS", "; ".join(found_locations)

    return "FILE_NOT_FOUND", "; ".join(found_locations)

results = []

for _, row in manifest.iterrows():
    raw_status, raw_location = check_paths(
        row.get("raw_subtitle_file", ""), raw_index
    )

    validated_status, validated_location = check_paths(
        row.get("validated_subtitle_file", ""), validated_index
    )

    results.append({
        "movie_record_id": row["movie_record_id"],
        "movie_title": row["movie_title"],
        "acquisition_status": row.get("acquisition_status", ""),
        "validation_status": row.get("validation_status", ""),
        "raw_path_check": raw_status,
        "raw_found_location": raw_location,
        "validated_path_check": validated_status,
        "validated_found_location": validated_location,
    })

audit = pd.DataFrame(results)

print("\nRAW PATH RESULTS")
display(audit["raw_path_check"].value_counts().to_frame("records"))

print("\nVALIDATED PATH RESULTS")
display(
    audit["validated_path_check"].value_counts().to_frame("records")
)

# Show only records needing investigation
issues = audit[
    audit["raw_path_check"].isin(
        ["FILE_NOT_FOUND", "NO_PATH_RECORDED"]
    )
    | audit["validated_path_check"].isin(
        ["FILE_NOT_FOUND", "NO_PATH_RECORDED"]
    )
]

print("\nRECORDS NEEDING REVIEW:", len(issues))
display(
    issues[[
        "movie_record_id",
        "movie_title",
        "acquisition_status",
        "validation_status",
        "raw_path_check",
        "validated_path_check",
    ]].to_string(index=False)
    if False else issues[[
        "movie_record_id",
        "movie_title",
        "acquisition_status",
        "validation_status",
        "raw_path_check",
        "validated_path_check",
    ]]
)

print("\nBATCH 05 PATH CHECK")
display(
    audit[audit["movie_record_id"].isin([
        "2010_5", "2011_1", "2011_2", "2011_3", "2011_4",
        "2011_5", "2012_1", "2012_2", "2012_3", "2012_4",
    ])][[
        "movie_record_id",
        "movie_title",
        "raw_path_check",
        "validated_path_check",
    ]]
)

print("\nDIAGNOSTIC COMPLETE — NO FILES MODIFIED")

In [ ]:
from pathlib import Path
import pandas as pd

WORK = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

df = pd.read_csv(MANIFEST, dtype=str).fillna("")

# Index every file within the subtitle corpus by filename
all_files = [
    p for p in WORK.rglob("*")
    if p.is_file()
]

filename_index = {}
for p in all_files:
    filename_index.setdefault(p.name, []).append(str(p))

def inspect_paths(value):
    value = str(value).strip()
    if not value:
        return "NO_PATH_RECORDED", ""

    found = []
    missing = []

    for item in value.split(";"):
        item = item.strip()
        if not item:
            continue

        p = Path(item)

        if p.is_file():
            found.append(str(p))
        elif p.name in filename_index:
            found.extend(filename_index[p.name])
        else:
            missing.append(p.name)

    if missing and found:
        return "PARTIAL_MATCH", (
            "Found: " + "; ".join(found)
            + " | Missing: " + "; ".join(missing)
        )
    if missing:
        return "NOT_FOUND_IN_CORPUS", "; ".join(missing)
    if found:
        return "FOUND", "; ".join(found)

    return "NO_PATH_RECORDED", ""

rows = []

for _, r in df.iterrows():
    raw_status, raw_detail = inspect_paths(r.get("raw_subtitle_file", ""))
    val_status, val_detail = inspect_paths(
        r.get("validated_subtitle_file", "")
    )

    if (
        raw_status != "FOUND"
        or val_status != "FOUND"
    ):
        rows.append({
            "movie_record_id": r["movie_record_id"],
            "movie_title": r["movie_title"],
            "acquisition_status": r.get("acquisition_status", ""),
            "validation_status": r.get("validation_status", ""),
            "raw_check": raw_status,
            "raw_detail": raw_detail,
            "validated_check": val_status,
            "validated_detail": val_detail,
        })

review = pd.DataFrame(rows)

print("Unresolved records:", len(review))
display(review)

print("\nACCEPTED RECORDS WITH NO RECORDED FILE PATH")
accepted = df[
    df["acquisition_status"].str.startswith(
        ("Accepted", "Acquired"), na=False
    )
]

accepted_path_issues = accepted[
    accepted["raw_subtitle_file"].str.strip().eq("")
    | accepted["validated_subtitle_file"].str.strip().eq("")
]

display(accepted_path_issues[[
    "movie_record_id",
    "movie_title",
    "acquisition_status",
    "validation_status",
    "raw_subtitle_file",
    "validated_subtitle_file",
]])

print("\nREAD-ONLY CHECK COMPLETE — NO FILES MODIFIED")

In [ ]:
from pathlib import Path

WORK = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

RAW = WORK / "raw"
VALIDATED = WORK / "validated"

keywords = ("k3g", "kabhi", "khushi")

print("Searching for Kabhi Khushi Kabhie Gham files...")
print("=" * 60)

matches = []

for folder_name, folder in [
    ("RAW", RAW),
    ("VALIDATED", VALIDATED),
]:
    if not folder.exists():
        print(f"Folder missing: {folder}")
        continue

    for path in folder.rglob("*"):
        if path.is_file() and any(
            word in path.name.lower() for word in keywords
        ):
            matches.append((folder_name, path))

if matches:
    for folder_name, path in matches:
        print(f"[{folder_name}] {path}")
else:
    print("No matching filenames found in the subtitle corpus.")

print("\nTotal matching files:", len(matches))
print("READ-ONLY SEARCH — NO FILES MODIFIED")

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil

# ============================================================
# TARGETED MANIFEST FIX — K3G ONLY
# ============================================================

WORK = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

RAW_FILE = (
    WORK / "raw/"
    "2001_2_Kabhi_Khushi_Kabhie_Gham_candidate_218844.srt"
)

VALIDATED_FILE = (
    WORK / "validated/"
    "2001_2_KABHI_KHUSHI_KABHIE_GHAM_ENGLISH.srt"
)

# 1. Confirm both files exist before editing
assert MANIFEST.is_file(), f"Manifest missing: {MANIFEST}"
assert RAW_FILE.is_file(), f"Raw file missing: {RAW_FILE}"
assert VALIDATED_FILE.is_file(), (
    f"Validated file missing: {VALIDATED_FILE}"
)

df = pd.read_csv(MANIFEST, dtype=str).fillna("")

required = {
    "movie_record_id",
    "raw_subtitle_file",
    "validated_subtitle_file",
}

assert required.issubset(df.columns), "Required columns are missing."

# 2. Verify exactly one K3G record exists
mask = df["movie_record_id"].eq("2001_2")
assert mask.sum() == 1, (
    f"Expected one 2001_2 record; found {mask.sum()}."
)

idx = df.index[mask][0]

# 3. Only fill blank path fields; do not overwrite existing values
assert not df.loc[idx, "raw_subtitle_file"].strip(), (
    "Raw path is already populated. Stop and review manually."
)
assert not df.loc[idx, "validated_subtitle_file"].strip(), (
    "Validated path is already populated. Stop and review manually."
)

# 4. Create a backup before changing anything
backup = WORK / (
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_K3G_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".csv"
)

shutil.copy2(MANIFEST, backup)

# 5. Fill only the two missing paths
df.loc[idx, "raw_subtitle_file"] = str(RAW_FILE)
df.loc[idx, "validated_subtitle_file"] = str(VALIDATED_FILE)

# 6. Save and verify
df.to_csv(MANIFEST, index=False)

check = pd.read_csv(MANIFEST, dtype=str).fillna("")
row = check.loc[check["movie_record_id"].eq("2001_2")]

assert len(check) == len(df), "Manifest row count changed unexpectedly."
assert check["movie_record_id"].is_unique, "Duplicate IDs detected."
assert row.iloc[0]["raw_subtitle_file"] == str(RAW_FILE)
assert row.iloc[0]["validated_subtitle_file"] == str(VALIDATED_FILE)

print("K3G MANIFEST PATH CORRECTION SUCCESSFUL")
print("Backup:", backup)
print("Manifest rows:", len(check))
print("Movie record ID: 2001_2")
print("Raw file exists:", RAW_FILE.is_file())
print("Validated file exists:", VALIDATED_FILE.is_file())
print("Raw path:", row.iloc[0]["raw_subtitle_file"])
print("Validated path:", row.iloc[0]["validated_subtitle_file"])
print("No other manifest fields intentionally changed.")
print("Locked master and other locked datasets were not modified.")

In [ ]:
from pathlib import Path

WORK = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
RAW = WORK / "raw"

search_terms = {
    "2021_4 — 83": ["2021_4", "83"],
    "2000_1 — Kaho Naa... Pyaar Hai": [
        "2000_1", "kaho", "pyaar"
    ],
}

files = [p for p in RAW.rglob("*") if p.is_file()]

for movie, terms in search_terms.items():
    print("\n" + "=" * 60)
    print(movie)
    print("=" * 60)

    matches = [
        p for p in files
        if all(term.lower() in p.name.lower() for term in terms)
    ]

    # If strict matching finds nothing, show files matching any term
    if not matches:
        matches = [
            p for p in files
            if any(term.lower() in p.name.lower() for term in terms)
        ]

    if matches:
        for path in matches:
            print(path)
    else:
        print("No matching raw file found.")

print("\nREAD-ONLY SEARCH COMPLETE — NO FILES MODIFIED")

In [ ]:
from pathlib import Path

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

files = sorted(p.name for p in RAW.iterdir() if p.is_file())

print("Total raw files:", len(files))
print("\nRaw files potentially relevant to 83 or Kaho Naa... Pyaar Hai:")

for name in files:
    lower = name.lower()
    if (
        "83" in lower
        or "kaho" in lower
        or "pyaar" in lower
        or "2000_1" in lower
        or "2021_4" in lower
    ):
        print(name)

print("\nREAD-ONLY INVENTORY COMPLETE")

In [ ]:

from pathlib import Path
import pandas as pd

# ============================================================
# NOTEBOOK 8 — FINAL READ-ONLY MANIFEST AUDIT
# ============================================================

PROJECT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"

MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"

print("=" * 65)
print("ICMB-130 — FINAL SUBTITLE MANIFEST AUDIT")
print("=" * 65)

# 1. Confirm required files and directories
for path in [MANIFEST, MASTER, RAW, VALIDATED]:
    print(f"{path.name}: {'EXISTS' if path.exists() else 'MISSING'}")

assert MANIFEST.is_file(), "Manifest file is missing."
assert MASTER.is_file(), "Locked master file is missing."
assert RAW.is_dir(), "Raw subtitle directory is missing."
assert VALIDATED.is_dir(), "Validated subtitle directory is missing."

# 2. Load read-only
manifest = pd.read_csv(MANIFEST, dtype=str).fillna("")
master = pd.read_csv(MASTER, dtype=str).fillna("")

required_columns = [
    "movie_record_id",
    "movie_title",
    "acquisition_status",
    "validation_status",
    "raw_subtitle_file",
    "validated_subtitle_file",
]

missing_columns = [
    col for col in required_columns if col not in manifest.columns
]
assert not missing_columns, f"Missing manifest columns: {missing_columns}"

print("\n1. DATASET SHAPES")
print("Manifest:", manifest.shape)
print("Master:", master.shape)

# 3. IDs and duplicates
manifest_ids = manifest["movie_record_id"].str.strip()
master_ids = set(master["movie_record_id"].str.strip())
manifest_id_set = set(manifest_ids)

blank_ids = int(manifest_ids.eq("").sum())
duplicate_ids = int(manifest_ids.duplicated(keep=False).sum())

print("\n2. ID CHECK")
print("Manifest rows:", len(manifest))
print("Blank IDs:", blank_ids)
print("Duplicate ID rows:", duplicate_ids)
print("Manifest IDs absent from master:",
      len(manifest_id_set - master_ids))
print("Master IDs absent from manifest:",
      len(master_ids - manifest_id_set))

if manifest_id_set - master_ids:
    print("IDs not found in master:", sorted(manifest_id_set - master_ids))

# 4. Build filename indexes
raw_index = {
    p.name: str(p)
    for p in RAW.rglob("*") if p.is_file()
}
validated_index = {
    p.name: str(p)
    for p in VALIDATED.rglob("*") if p.is_file()
}

def check_recorded_paths(value, index):
    value = str(value).strip()

    if not value:
        return "NO_PATH_RECORDED", ""

    found = []
    missing = []

    for item in value.split(";"):
        item = item.strip()
        if not item:
            continue

        path = Path(item)

        if path.is_file():
            found.append(str(path))
        elif path.name in index:
            found.append(index[path.name])
        else:
            missing.append(path.name)

    if missing and found:
        return "PARTIAL_MATCH", (
            "Found: " + "; ".join(found)
            + " | Missing: " + "; ".join(missing)
        )
    if missing:
        return "NOT_FOUND", "; ".join(missing)
    if found:
        return (
            "FOUND_BUT_PATH_DIFFERS"
            if any(not Path(p).is_file() for p in [str(x) for x in []])
            else "FOUND",
            "; ".join(found)
        )

    return "NO_PATH_RECORDED", ""

# 5. Audit all paths
results = []

for _, row in manifest.iterrows():
    raw_status, raw_detail = check_recorded_paths(
        row["raw_subtitle_file"], raw_index
    )
    val_status, val_detail = check_recorded_paths(
        row["validated_subtitle_file"], validated_index
    )

    results.append({
        "movie_record_id": row["movie_record_id"],
        "movie_title": row["movie_title"],
        "acquisition_status": row["acquisition_status"],
        "validation_status": row["validation_status"],
        "raw_check": raw_status,
        "raw_detail": raw_detail,
        "validated_check": val_status,
        "validated_detail": val_detail,
    })

audit = pd.DataFrame(results)

print("\n3. RAW PATH SUMMARY")
print(audit["raw_check"].value_counts(dropna=False).to_string())

print("\n4. VALIDATED PATH SUMMARY")
print(audit["validated_check"].value_counts(dropna=False).to_string())

# 6. Check K3G correction
k3g = audit[audit["movie_record_id"].eq("2001_2")]

print("\n5. K3G CORRECTION")
if len(k3g) == 1:
    print(k3g[[
        "movie_record_id", "movie_title",
        "raw_check", "validated_check"
    ]].to_string(index=False))
else:
    print("ERROR: Expected exactly one K3G record.")

# 7. Check Batch 05 records
batch05_ids = [
    "2010_5", "2011_1", "2011_2", "2011_3", "2011_4",
    "2011_5", "2012_1", "2012_2", "2012_3", "2012_4"
]

batch05 = audit[audit["movie_record_id"].isin(batch05_ids)]

print("\n6. BATCH 05 FILE CHECK")
print("Expected records:", len(batch05_ids))
print("Found records:", len(batch05))
print(batch05[[
    "movie_record_id", "movie_title",
    "raw_check", "validated_check"
]].to_string(index=False))

# 8. Print records requiring review
issues = audit[
    ~audit["raw_check"].isin(["FOUND", "FOUND_BUT_PATH_DIFFERS"])
    | ~audit["validated_check"].isin(
        ["FOUND", "FOUND_BUT_PATH_DIFFERS"]
    )
]

print("\n7. RECORDS REQUIRING REVIEW")
print("Count:", len(issues))

if len(issues):
    print(issues[[
        "movie_record_id", "movie_title",
        "acquisition_status", "validation_status",
        "raw_check", "raw_detail",
        "validated_check", "validated_detail"
    ]].to_string(index=False))

# 9. Status consistency review
print("\n8. ACQUISITION STATUS COUNTS")
print(manifest["acquisition_status"].value_counts().to_string())

print("\n9. VALIDATION STATUS COUNTS")
print(manifest["validation_status"].value_counts().to_string())

# 10. Final summary
print("\n" + "=" * 65)
print("FINAL AUDIT SUMMARY")
print("=" * 65)
print("Manifest rows:", len(manifest))
print("Unique manifest IDs:", manifest_ids.nunique())
print("Duplicate ID rows:", duplicate_ids)
print("Blank IDs:", blank_ids)
print("Batch 05 records:", len(batch05), "/", len(batch05_ids))
print("Records requiring path review:", len(issues))
print("AUDIT COMPLETE — NO FILES MODIFIED")
print("=" * 65)


In [ ]:

from pathlib import Path
import pandas as pd

# ============================================================
# NOTEBOOK 8 — ICMB-130 SUBTITLE GAP ANALYSIS
# READ-ONLY: NO DOWNLOADS, API CALLS, OR FILE MODIFICATIONS
# ============================================================

PROJECT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

MASTER_PATH = (
    PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
)

WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MANIFEST_PATH = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

assert MASTER_PATH.is_file(), f"Master not found: {MASTER_PATH}"
assert MANIFEST_PATH.is_file(), f"Manifest not found: {MANIFEST_PATH}"

master = pd.read_csv(MASTER_PATH, dtype=str).fillna("")
manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")

# ------------------------------------------------------------
# 1. Identify the required ID columns
# ------------------------------------------------------------

ID_COL = "movie_record_id"

assert ID_COL in master.columns, (
    f"{ID_COL} is not in master. Columns: {master.columns.tolist()}"
)
assert ID_COL in manifest.columns, (
    f"{ID_COL} is not in manifest. Columns: {manifest.columns.tolist()}"
)

master[ID_COL] = master[ID_COL].str.strip()
manifest[ID_COL] = manifest[ID_COL].str.strip()

assert master[ID_COL].is_unique, "Duplicate IDs found in master."
assert manifest[ID_COL].is_unique, "Duplicate IDs found in manifest."

manifest_ids = set(manifest[ID_COL]) - {""}
master_ids = set(master[ID_COL]) - {""}

missing_ids = master_ids - manifest_ids

# ------------------------------------------------------------
# 2. Identify title/year/language columns where available
# ------------------------------------------------------------

def find_column(df, candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None

title_col = find_column(
    master,
    ["dataset_title", "movie_title", "title", "original_title"]
)
year_col = find_column(
    master,
    ["dataset_year", "year", "release_year"]
)
language_col = find_column(
    master,
    ["dataset_language", "language", "original_language"]
)
rank_col = find_column(
    master,
    ["rank", "annual_rank", "gross_rank"]
)

print("=" * 72)
print("ICMB-130 — REMAINING SUBTITLE GAP ANALYSIS")
print("=" * 72)
print("Master rows:", len(master))
print("Manifest rows:", len(manifest))
print("Master unique IDs:", len(master_ids))
print("Manifest unique IDs:", len(manifest_ids))
print("Master IDs absent from manifest:", len(missing_ids))
print("\nMaster columns used:")
print("Title:", title_col)
print("Year:", year_col)
print("Language:", language_col)
print("Rank:", rank_col)

# ------------------------------------------------------------
# 3. Known prior decisions from the project's recorded history
#    These labels preserve prior decisions; they do not trigger
#    searches, downloads, or changes to any data.
# ------------------------------------------------------------

known_decisions = {
    "2009_2": (
        "Previously reviewed — held",
        "Magadheera: suspicious original-language candidate; "
        "do not download without further review."
    ),
    "2009_3": (
        "Previously reviewed — rejected",
        "Love Aaj Kal: English candidate rejected for sparse/"
        "incomplete coverage; do not repeat the same download."
    ),
    "2009_4": (
        "Previously reviewed — held",
        "Ajab Prem Ki Ghazab Kahani: suspicious candidate; "
        "not downloaded."
    ),
    "2010_2": (
        "Previously searched — no original Hindi candidate",
        "Dabangg: original Hindi search returned no candidates."
    ),
    "2010_3": (
        "Previously searched — false match",
        "Golmaal 3: candidate was unrelated; do not reuse it."
    ),
    "2010_4": (
        "Previously searched — no candidate",
        "Raajneeti: search returned no candidates."
    ),
}

# ------------------------------------------------------------
# 4. Build the missing-record table
# ------------------------------------------------------------

missing = master[master[ID_COL].isin(missing_ids)].copy()

def classify_record(movie_id):
    if movie_id in known_decisions:
        category, note = known_decisions[movie_id]
        return category, note

    return (
        "Not yet represented — review history",
        "Check prior work and candidate availability before "
        "deciding whether acquisition is needed."
    )

classification = missing[ID_COL].apply(classify_record)

missing["gap_category"] = classification.apply(lambda x: x[0])
missing["existing_decision_notes"] = classification.apply(lambda x: x[1])

# Add useful display fields only when available
display_cols = [ID_COL]

for col in [year_col, rank_col, title_col, language_col]:
    if col and col not in display_cols:
        display_cols.append(col)

display_cols += ["gap_category", "existing_decision_notes"]

print("\n" + "=" * 72)
print("5. CLASSIFICATION SUMMARY")
print("=" * 72)
print(
    missing["gap_category"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "=" * 72)
print("6. ALL MOVIE RECORDS ABSENT FROM THE MANIFEST")
print("=" * 72)

# Sort by ID so the list follows the dataset's record order
missing_display = missing[display_cols].sort_values(ID_COL)

print(missing_display.to_string(index=False))

# ------------------------------------------------------------
# 5. Cross-check that the gap count reconciles
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("7. RECONCILIATION CHECK")
print("=" * 72)
print("Missing IDs computed from sets:", len(missing_ids))
print("Rows in missing-record table:", len(missing))
print(
    "All absent IDs represented in table:",
    set(missing[ID_COL]) == missing_ids
)

# Identify IDs present in manifest but absent from master
extra_manifest_ids = manifest_ids - master_ids
print("Manifest IDs not found in master:", len(extra_manifest_ids))

if extra_manifest_ids:
    print(sorted(extra_manifest_ids))

print("\nREAD-ONLY ANALYSIS COMPLETE — NO FILES MODIFIED")


In [ ]:

# ============================================================
# NOTEBOOK 8 — LOCAL EVIDENCE CHECK FOR THE 71 GAP RECORDS
# READ-ONLY: NO API CALLS, DOWNLOADS, OR FILE MODIFICATIONS
# Run after the previous gap-analysis cell.
# ============================================================

import re
from pathlib import Path
import pandas as pd

folders_to_check = {
    "raw": WORK / "raw",
    "validated": WORK / "validated",
    "search_results": WORK / "search_results",
}

# Inventory existing files
file_inventory = []

for folder_label, folder_path in folders_to_check.items():
    if folder_path.exists():
        for path in folder_path.rglob("*"):
            if path.is_file():
                file_inventory.append({
                    "folder": folder_label,
                    "path": path,
                    "filename": path.name,
                    "suffix": path.suffix.lower(),
                })

print("Files inventoried:", len(file_inventory))
print("Files by folder:")
print(
    pd.Series([x["folder"] for x in file_inventory])
    .value_counts()
    .to_string()
)

# The six IDs with prior decisions are not the primary targets
known_ids = set(known_decisions)
review_targets = missing[
    ~missing[ID_COL].isin(known_ids)
].copy()

# Search terms per movie: ID and exact movie title
def normalize_text(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

# Only inspect small, text-based search-result files.
# Subtitle files are checked by filename only, not parsed here.
text_suffixes = {
    ".csv", ".txt", ".json", ".log", ".tsv", ".md"
}
max_text_file_bytes = 5_000_000

evidence_rows = []

for _, movie in review_targets.iterrows():
    movie_id = str(movie[ID_COL]).strip()
    movie_title = str(movie.get(title_col, "")).strip()
    title_key = normalize_text(movie_title)

    filename_hits = []
    text_log_hits = []

    for item in file_inventory:
        filename = item["filename"]
        path = item["path"]
        folder = item["folder"]

        # Match the exact record ID in a filename.
        if movie_id and movie_id.lower() in filename.lower():
            filename_hits.append(f"{folder}/{filename}")

        # Search only text-based files under search_results.
        if (
            folder == "search_results"
            and item["suffix"] in text_suffixes
            and path.stat().st_size <= max_text_file_bytes
        ):
            try:
                content = path.read_text(
                    encoding="utf-8", errors="ignore"
                )
                content_key = normalize_text(content)

                id_found = movie_id.lower() in content.lower()
                title_found = (
                    len(title_key) >= 5
                    and title_key in content_key
                )

                if id_found or title_found:
                    match_reason = []
                    if id_found:
                        match_reason.append("record ID")
                    if title_found:
                        match_reason.append("normalized title")

                    text_log_hits.append(
                        f"{path.name} ({', '.join(match_reason)})"
                    )
            except (OSError, UnicodeError):
                pass

    if filename_hits and text_log_hits:
        evidence_status = "Files and search-log evidence found"
    elif filename_hits:
        evidence_status = "Filename evidence found"
    elif text_log_hits:
        evidence_status = "Search-log evidence found"
    else:
        evidence_status = "No local evidence found by this check"

    evidence_rows.append({
        "movie_record_id": movie_id,
        "year": movie.get(year_col, ""),
        "rank": movie.get(rank_col, ""),
        "movie_title": movie_title,
        "language": movie.get(language_col, ""),
        "local_evidence_status": evidence_status,
        "matching_filenames": " | ".join(filename_hits),
        "matching_search_logs": " | ".join(text_log_hits),
    })

evidence_df = pd.DataFrame(evidence_rows)

print("\n" + "=" * 80)
print("LOCAL EVIDENCE SUMMARY — REVIEW TARGETS")
print("=" * 80)
print(
    evidence_df["local_evidence_status"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "=" * 80)
print("MOVIES WITH POSSIBLE PRIOR EVIDENCE")
print("=" * 80)

has_evidence = evidence_df[
    evidence_df["local_evidence_status"]
    != "No local evidence found by this check"
]
if has_evidence.empty:
    print("No matching filenames or search-log references found.")
else:
    print(has_evidence.to_string(index=False))

print("\n" + "=" * 80)
print("MOVIES WITH NO MATCH IN THE CHECKED LOCAL EVIDENCE")
print("=" * 80)

no_evidence = evidence_df[
    evidence_df["local_evidence_status"]
    == "No local evidence found by this check"
]
if no_evidence.empty:
    print("All review targets had at least one possible local match.")
else:
    print(
        no_evidence[
            [
                "movie_record_id", "year", "rank",
                "movie_title", "language"
            ]
        ].to_string(index=False)
    )

print("\nREAD-ONLY CHECK COMPLETE — NO FILES MODIFIED")
print(
    "Important: no local match does not prove that a movie "
    "was never searched. Review the results before deciding."
)


In [ ]:

# ============================================================
# NOTEBOOK 8 — TARGETED REVIEW OF THREE EXISTING SEARCH LOGS
# READ-ONLY
# ============================================================

from pathlib import Path
import pandas as pd
import re

search_dir = WORK / "search_results"

files_to_review = [
    "batch05_remaining9_tmdb_search_20261024_200354.csv",
    "ICMB130_ALT_TITLE_CANDIDATES_20261014_181851.csv",
    "ICMB130_BATCH_01_NEXT5_SHORTLIST_FIXED.csv",
]

targets = {
    "2012_5": "Housefull 2",
    "2021_3": "Master",
    "2023_4": "Gadar 2",
}

def norm(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

for filename in files_to_review:
    path = search_dir / filename

    print("\n" + "=" * 90)
    print("FILE:", filename)
    print("=" * 90)

    if not path.exists():
        print("File not found in search_results folder.")
        continue

    try:
        df = pd.read_csv(path, dtype=str).fillna("")
    except Exception as exc:
        print("Could not read CSV:", repr(exc))
        continue

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    id_col = next(
        (c for c in ["movie_record_id", "record_id"] if c in df.columns),
        None
    )
    title_cols = [
        c for c in [
            "target_title", "movie_title", "dataset_title",
            "title", "filename", "release", "candidate_filename",
            "decision"
        ]
        if c in df.columns
    ]

    matched_frames = []

    for movie_id, movie_title in targets.items():
        mask = pd.Series(False, index=df.index)

        if id_col:
            mask |= df[id_col].astype(str).str.strip().eq(movie_id)

        # Match title fields separately; avoid relying on title alone
        # when the exact record ID is available.
        for col in title_cols:
            mask |= df[col].apply(
                lambda value: norm(movie_title) in norm(value)
            )

        rows = df[mask].copy()

        if not rows.empty:
            rows.insert(0, "review_target_id", movie_id)
            rows.insert(1, "review_target_title", movie_title)
            matched_frames.append(rows)

    if matched_frames:
        result = pd.concat(matched_frames, ignore_index=True)
        # Display all columns so candidate IDs and decisions remain visible.
        print(result.to_string(index=False))
    else:
        print("No matching rows found for the three target movies.")

print("\nREAD-ONLY REVIEW COMPLETE — NO FILES MODIFIED")


In [ ]:

# ============================================================
# NOTEBOOK 8 — STRICT RECORD-ID SEARCH HISTORY AUDIT
# READ-ONLY: NO API CALLS, DOWNLOADS, OR FILE MODIFICATIONS
# ============================================================

import pandas as pd
from pathlib import Path

search_dir = WORK / "search_results"

# Use the existing `missing` dataframe from the gap-analysis cell.
missing_ids_df = missing[
    ["movie_record_id", "year", "rank", "movie_title", "language"]
].copy()

csv_paths = sorted(search_dir.glob("*.csv"))

# Collect exact-ID matches from all local CSV files
exact_id_rows = []
id_column_names = [
    "movie_record_id", "record_id", "dataset_record_id"
]

for csv_path in csv_paths:
    try:
        df = pd.read_csv(csv_path, dtype=str).fillna("")
    except Exception as exc:
        print(f"Skipped unreadable CSV: {csv_path.name} — {exc}")
        continue

    id_col = next(
        (c for c in id_column_names if c in df.columns),
        None
    )

    if id_col is None:
        continue

    for _, row in df.iterrows():
        record_id = str(row[id_col]).strip()

        if record_id in set(missing_ids_df["movie_record_id"]):
            exact_id_rows.append({
                "movie_record_id": record_id,
                "source_csv": csv_path.name,
                "source_id_column": id_col,
                "candidate_title": next(
                    (
                        str(row[c]) for c in [
                            "requested_title", "target_title",
                            "movie_title", "dataset_title",
                            "feature_title"
                        ]
                        if c in df.columns and str(row[c]).strip()
                    ),
                    ""
                ),
                "candidate_language": next(
                    (
                        str(row[c]) for c in [
                            "candidate_language", "subtitle_language",
                            "language_searched", "requested_language"
                        ]
                        if c in df.columns and str(row[c]).strip()
                    ),
                    ""
                ),
                "candidate_file_id": next(
                    (
                        str(row[c]) for c in [
                            "subtitle_result_id", "file_id",
                            "selected_file_id"
                        ]
                        if c in df.columns and str(row[c]).strip()
                    ),
                    ""
                ),
                "decision": str(row.get("decision", "")),
            })

exact_id_df = pd.DataFrame(exact_id_rows)

print("=" * 88)
print("STRICT SEARCH-HISTORY AUDIT")
print("=" * 88)
print("Missing master IDs reviewed:", len(missing_ids_df))
print("Local CSV files inspected:", len(csv_paths))
print("Exact-ID matching rows:", len(exact_id_df))

if exact_id_df.empty:
    print("\nNo exact-ID matches found in local search-result CSVs.")
else:
    print("\nExact-ID match counts by movie:")
    print(
        exact_id_df.groupby("movie_record_id")
        .size()
        .sort_index()
        .to_string()
    )

    print("\nExact-ID evidence rows:")
    print(exact_id_df.to_string(index=False))

matched_ids = (
    set(exact_id_df["movie_record_id"])
    if not exact_id_df.empty else set()
)

missing_ids_set = set(missing_ids_df["movie_record_id"])

unmatched = missing_ids_df[
    ~missing_ids_df["movie_record_id"].isin(matched_ids)
].copy()

print("\n" + "=" * 88)
print("RECORDS WITH NO EXACT-ID MATCH IN SEARCH CSVs")
print("=" * 88)
print("Count:", len(unmatched))
print(unmatched.to_string(index=False))

print(
    "\nInterpretation: exact-ID matches show that a record appears "
    "in a local CSV, not that its candidate was downloaded, accepted, "
    "or validated. No exact match also does not prove that no search "
    "was ever performed."
)
print("\nREAD-ONLY AUDIT COMPLETE — NO FILES MODIFIED")


In [ ]:
# ============================================================
# NOTEBOOK 8 — MISSING RECORD CLASSIFICATION
# READ-ONLY: NO FILES MODIFIED
# Requires the existing `missing` DataFrame
# ============================================================

import pandas as pd

# Historical decisions already established in our project work.
known_history = {
    "2009_2": (
        "Previously investigated",
        "Original Telugu candidate held; suspicious full-movie release."
    ),
    "2009_3": (
        "Previously investigated",
        "English candidate downloaded and rejected for sparse/incomplete coverage."
    ),
    "2009_4": (
        "Previously investigated",
        "Hindi candidate held; suspicious English-dubbed full-movie release."
    ),
    "2010_2": (
        "Previously investigated",
        "Original Hindi search returned zero candidates."
    ),
    "2010_3": (
        "Previously investigated",
        "Search result was a false match for Undisputed 3."
    ),
    "2010_4": (
        "Previously investigated",
        "Original Hindi search returned zero candidates."
    ),
    "2012_5": (
        "Search history needs verification",
        "Housefull 2 title-only match was a false match to Housefull."
    ),
}

review = missing[
    ["movie_record_id", "year", "rank", "movie_title", "language"]
].copy()

review["history_classification"] = review["movie_record_id"].map(
    lambda x: known_history.get(
        x, ("Not verified from available evidence", "")
    )[0]
)

review["historical_note"] = review["movie_record_id"].map(
    lambda x: known_history.get(x, ("", ""))[1]
)

review["exact_id_csv_match"] = False
review["manifest_update_authorized"] = False

print("=" * 90)
print("MISSING RECORD CLASSIFICATION — READ-ONLY")
print("=" * 90)
print("Rows:", len(review))
print("\nClassification counts:")
print(review["history_classification"].value_counts(dropna=False))

display(review)

print("\nNo manifest updates or downloads performed.")

In [ ]:
# ============================================================
# NOTEBOOK 8 — SEARCH LOG INVENTORY
# READ-ONLY: NO FILES MODIFIED
# ============================================================

from pathlib import Path
import pandas as pd

search_dir = WORK / "search_results"

inventory = []

for path in sorted(search_dir.glob("*.csv")):
    record = {
        "filename": path.name,
        "size_bytes": path.stat().st_size,
        "read_status": "",
        "row_count": None,
        "columns": "",
        "notes": ""
    }

    try:
        df = pd.read_csv(path, low_memory=False)
        record["read_status"] = "READABLE"
        record["row_count"] = len(df)
        record["columns"] = ", ".join(map(str, df.columns))

        if df.empty:
            record["notes"] = "Readable but contains zero rows"

    except pd.errors.EmptyDataError:
        record["read_status"] = "EMPTY_CSV"
        record["notes"] = "No columns or parseable data"

    except Exception as e:
        record["read_status"] = "READ_ERROR"
        record["notes"] = str(e)[:250]

    inventory.append(record)

inventory_df = pd.DataFrame(inventory)

print("=" * 90)
print("SEARCH LOG INVENTORY")
print("=" * 90)
print("CSV files:", len(inventory_df))

if not inventory_df.empty:
    print("\nRead status counts:")
    print(inventory_df["read_status"].value_counts())

    print("\nReadable files with zero rows:")
    display(
        inventory_df[
            (inventory_df["read_status"] == "READABLE") &
            (inventory_df["row_count"] == 0)
        ][["filename", "size_bytes", "read_status", "row_count", "notes"]]
    )

    print("\nEmpty or unreadable files:")
    display(
        inventory_df[
            inventory_df["read_status"].isin(["EMPTY_CSV", "READ_ERROR"])
        ][["filename", "size_bytes", "read_status", "notes"]]
    )

    print("\nComplete inventory:")
    display(inventory_df)
else:
    print("No CSV files found in:", search_dir)

print("\nREAD-ONLY AUDIT COMPLETE — NO FILES MODIFIED")

In [ ]:
# ============================================================
# NOTEBOOK 8 — CROSS-FILE SEARCH EVIDENCE AUDIT
# READ-ONLY: NO FILES MODIFIED
# Requires: WORK and missing
# ============================================================

from pathlib import Path
import pandas as pd

search_dir = WORK / "search_results"

# Normalize the missing IDs from the existing gap-analysis DataFrame.
missing_ids = set(
    missing["movie_record_id"].astype(str).str.strip()
)

evidence_rows = []
unreadable_files = []

for path in sorted(search_dir.glob("*.csv")):
    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as e:
        unreadable_files.append({
            "filename": path.name,
            "reason": str(e)[:200]
        })
        continue

    if df.empty:
        continue

    # Inspect all columns, not only columns with a particular name.
    # This is a candidate evidence scan, not proof of a search.
    for col in df.columns:
        values = df[col].astype("string").str.strip()
        matches = values.isin(missing_ids)

        if matches.any():
            matched_rows = df.loc[matches]

            for idx, row in matched_rows.iterrows():
                evidence_rows.append({
                    "movie_record_id": str(row[col]).strip(),
                    "source_file": path.name,
                    "matched_column": str(col),
                    "source_row_index": idx,
                    "row_preview": row.to_dict()
                })

evidence_df = pd.DataFrame(evidence_rows)

print("=" * 90)
print("CROSS-FILE SEARCH EVIDENCE AUDIT")
print("=" * 90)
print("Missing IDs checked:", len(missing_ids))
print("CSV files inspected:", len(list(search_dir.glob("*.csv"))))
print("Candidate evidence rows:", len(evidence_df))

if not evidence_df.empty:
    print("\nMatched IDs:", evidence_df["movie_record_id"].nunique())
    display(
        evidence_df[
            ["movie_record_id", "source_file",
             "matched_column", "source_row_index"]
        ].sort_values(["movie_record_id", "source_file"])
    )

    print("\nFull matched row previews:")
    for _, item in evidence_df.iterrows():
        print("\nID:", item["movie_record_id"])
        print("File:", item["source_file"])
        print("Column:", item["matched_column"])
        print("Row:", item["row_preview"])
else:
    print("No missing IDs were found as exact cell values in readable CSVs.")

if unreadable_files:
    print("\nUnreadable or empty files:")
    display(pd.DataFrame(unreadable_files))

print("\nREAD-ONLY AUDIT COMPLETE — NO FILES MODIFIED")

In [ ]:
# NOTEBOOK 8 — ONE LOGIN + QUOTA CHECK
# No subtitle downloads. Never prints the token.

import requests
from google.colab import userdata

BASE_URL = "https://api.opensubtitles.com/api/v1"

api_key = userdata.get("OPEN_SUBTITLES_API_KEY")
username = userdata.get("OPEN_SUBTITLES_USERNAME")
password = userdata.get("OPEN_SUBTITLES_PASSWORD")

headers = {
    "Api-Key": api_key,
    "User-Agent": "ICMB130SubtitleResearch/1.0",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

login_response = requests.post(
    f"{BASE_URL}/login",
    headers=headers,
    json={"username": username, "password": password},
    timeout=20,
)

print("Login HTTP status:", login_response.status_code)

if login_response.ok:
    login_data = login_response.json()

    # Correct: token and user are top-level fields.
    token = login_data.get("token")
    user = login_data.get("user", {})
    base_url = login_data.get("base_url", "api.opensubtitles.com")

    print("Token returned:", bool(token))
    print("Account level:", user.get("level", "not provided"))
    print("Allowed downloads:", user.get("allowed_downloads", "not provided"))
    print("VIP:", user.get("vip", "not provided"))

    if token:
        quota_url = f"https://{base_url}/api/v1/infos/user"

        quota_response = requests.get(
            quota_url,
            headers={
                **headers,
                "Authorization": f"Bearer {token}",
            },
            timeout=20,
        )

        print("\nQuota HTTP status:", quota_response.status_code)

        if quota_response.ok:
            payload = quota_response.json()
            account = payload.get("data", payload)

            for field in [
                "allowed_downloads",
                "downloads_count",
                "remaining_downloads",
                "level",
                "vip",
            ]:
                if field in account:
                    print(f"{field}: {account[field]}")
        else:
            print("Quota could not be confirmed.")
            print("Response:", quota_response.text[:300])
else:
    print("Login failed; no further request made.")
    print("Response:", login_response.text[:300])

print("\nNo subtitles were downloaded.")

In [ ]:
# ============================================================
# NOTEBOOK 8 — LOCATE MASTER AND SUBTITLE MANIFEST
# READ-ONLY: NO FILES MODIFIED
# ============================================================

from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

# Search only inside the existing project folder.
search_names = {
    "ICMB_130_MASTER_DATA.csv",
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv",
}

print("=" * 80)
print("SEARCHING FOR REQUIRED PROJECT FILES")
print("=" * 80)

found = {name: [] for name in search_names}

for path in PROJECT.rglob("*"):
    if path.is_file() and path.name in search_names:
        found[path.name].append(path)

for filename, paths in found.items():
    print(f"\n{filename}")

    if paths:
        for path in paths:
            print(" ", path)
            print("  Size:", path.stat().st_size, "bytes")
    else:
        print("  NOT FOUND")

print("\nNo files were modified.")

In [ ]:
# ============================================================
# NOTEBOOK 8 — BATCH 06 CANDIDATE PREPARATION
# READ-ONLY — NO API CALLS OR DOWNLOADS
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

MASTER = (
    PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
)

MANIFEST = (
    PROJECT
    / "01_data/processed/subtitle_corpus_v2"
    / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

# Verify the exact files before reading.
assert MASTER.exists(), f"Master not found: {MASTER}"
assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"

master = pd.read_csv(MASTER)
manifest = pd.read_csv(MANIFEST)

# Identify records missing from the current manifest.
existing_ids = set(
    manifest["movie_record_id"].astype(str).str.strip()
)

missing_df = master[
    ~master["movie_record_id"].astype(str).str.strip().isin(existing_ids)
].copy()

# Proposed review group: five movies from 2013.
candidate_ids = [
    "2013_1",  # Dhoom 3
    "2013_2",  # Chennai Express
    "2013_3",  # Krrish 3
    "2013_4",  # Yeh Jawaani Hai Deewani
    "2013_5",  # Goliyon Ki Raasleela Ram-Leela
]

preferred_columns = [
    "movie_record_id",
    "year",
    "rank",
    "movie_title",
    "language",
    "tmdb_id",
    "imdb_id",
]

available_columns = [
    c for c in preferred_columns if c in master.columns
]

batch06_review = (
    missing_df[
        missing_df["movie_record_id"].astype(str).str.strip()
        .isin(candidate_ids)
    ][available_columns]
    .sort_values("movie_record_id")
)

print("=" * 80)
print("BATCH 06 — CANDIDATE REVIEW")
print("=" * 80)
print("Master dataset:", MASTER)
print("Manifest:", MANIFEST)
print("Master shape:", master.shape)
print("Manifest shape:", manifest.shape)
print("Records missing from manifest:", len(missing_df))
print("Proposed 2013 candidates found:", len(batch06_review))

display(batch06_review)

print("\nREAD-ONLY: no files modified, no searches performed, no downloads.")

In [ ]:
# ============================================================
# NOTEBOOK 8 — BATCH 06 TMDB ID LOOKUP
# READ-ONLY — NO API CALLS OR DOWNLOADS
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

MAPPING = (
    PROJECT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
)

assert MAPPING.exists(), f"TMDB mapping file not found: {MAPPING}"

mapping = pd.read_csv(MAPPING)

print("TMDB mapping shape:", mapping.shape)
print("Available columns:")
print(mapping.columns.tolist())

target_ids = [
    "2013_1",
    "2013_2",
    "2013_3",
    "2013_4",
    "2013_5",
]

# Locate the record ID column without assuming its exact name.
id_candidates = [
    c for c in mapping.columns
    if c.strip().lower() in {
        "movie_record_id",
        "record_id",
        "dataset_record_id",
    }
]

if not id_candidates:
    print("\nNo recognized record-ID column found.")
    print("Please share the column names above.")
else:
    id_col = id_candidates[0]

    result = mapping[
        mapping[id_col].astype(str).str.strip().isin(target_ids)
    ].copy()

    print("\nMatching TMDB mapping records:", len(result))
    display(result)

print("\nREAD-ONLY: no files modified and no downloads performed.")

In [ ]:
# ============================================================
# NOTEBOOK 8 — BATCH 06 ENGLISH FALLBACK SEARCH
# SEARCH ONLY — NO SUBTITLE DOWNLOADS
# ============================================================

from pathlib import Path
import pandas as pd
import requests
from datetime import datetime
from google.colab import userdata

PROJECT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
SEARCH_RESULTS = WORK / "search_results"
SEARCH_RESULTS.mkdir(parents=True, exist_ok=True)

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
TOKEN = userdata.get("OPEN_SUBTITLES_TOKEN")

# The stored token previously returned "Invalid token".
# Check it before searching; don't make search requests with an invalid token.
if not API_KEY or not TOKEN:
    raise ValueError(
        "OpenSubtitles API key or token is unavailable. "
        "Reload credentials and obtain a fresh login token first."
    )

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {TOKEN}",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
    "Accept": "application/json",
}

movies = [
    {"movie_record_id": "2013_1", "title": "Dhoom 3", "tmdb_id": 44977},
    {"movie_record_id": "2013_2", "title": "Chennai Express", "tmdb_id": 205022},
    {"movie_record_id": "2013_3", "title": "Krrish 3", "tmdb_id": 204435},
    {"movie_record_id": "2013_4", "title": "Yeh Jawaani Hai Deewani", "tmdb_id": 185008},
    {
        "movie_record_id": "2013_5",
        "title": "Goliyon Ki Raasleela Ram-Leela",
        "tmdb_id": 235984,
    },
]

rows = []

for movie in movies:
    params = {
        "tmdb_id": movie["tmdb_id"],
        "languages": "en",
        "type": "movie",
    }

    try:
        response = requests.get(
            "https://api.opensubtitles.com/api/v1/subtitles",
            headers=headers,
            params=params,
            timeout=30,
        )

        print(
            movie["movie_record_id"],
            movie["title"],
            "| HTTP:", response.status_code,
        )

        if response.status_code == 401:
            raise RuntimeError(
                "Authentication failed. Stop the search; "
                "obtain a fresh token before retrying."
            )

        if not response.ok:
            rows.append({
                **movie,
                "search_language": "en",
                "http_status": response.status_code,
                "search_error": response.text[:300],
            })
            continue

        payload = response.json()
        results = payload.get("data", [])

        print("  Candidates returned:", len(results))

        for item in results:
            attrs = item.get("attributes", {})
            feature = attrs.get("feature_details") or {}

            rows.append({
                "movie_record_id": movie["movie_record_id"],
                "requested_title": movie["title"],
                "requested_tmdb_id": movie["tmdb_id"],
                "search_language": "en",
                "http_status": response.status_code,
                "subtitle_id": item.get("id"),
                "release": attrs.get("release"),
                "language": attrs.get("language"),
                "download_count": attrs.get("download_count"),
                "hearing_impaired": attrs.get("hearing_impaired"),
                "machine_translated": attrs.get("machine_translated"),
                "upload_date": attrs.get("upload_date"),
                "matched_movie_title": feature.get("movie_name"),
                "matched_movie_year": feature.get("year"),
                "matched_tmdb_id": feature.get("tmdb_id"),
                "matched_imdb_id": feature.get("imdb_id"),
            })

    except RuntimeError:
        raise
    except Exception as e:
        rows.append({
            **movie,
            "search_language": "en",
            "search_error": f"{type(e).__name__}: {str(e)[:250]}",
        })

results_df = pd.DataFrame(rows)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = (
    SEARCH_RESULTS / f"batch06_2013_english_search_{timestamp}.csv"
)

results_df.to_csv(output_path, index=False)

print("\n" + "=" * 80)
print("BATCH 06 SEARCH SUMMARY")
print("=" * 80)
print("Movies submitted:", len(movies))
print("Candidate rows returned:", len(results_df))
print("Search results saved to:", output_path)

if not results_df.empty:
    display(results_df)

print("\nNo subtitles were downloaded.")

In [ ]:
# ============================================================
# NOTEBOOK 8 — BATCH 06 CANDIDATE SHORTLIST
# READ-ONLY — NO DOWNLOADS
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

SEARCH_RESULTS = (
    PROJECT / "01_data/processed/subtitle_corpus_v2/search_results"
)

search_file = (
    SEARCH_RESULTS
    / "batch06_2013_english_search_20261025_192858.csv"
)

assert search_file.exists(), f"Search results not found: {search_file}"

df = pd.read_csv(search_file)

# Convert fields to appropriate types.
for col in ["download_count", "matched_movie_year", "matched_tmdb_id"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Candidate identity checks.
df["tmdb_match"] = (
    df["requested_tmdb_id"].astype("string")
    == df["matched_tmdb_id"].astype("Int64").astype("string")
)

df["year_match"] = (
    df["matched_movie_year"] == 2013
)

df["release_name_present"] = (
    df["release"].fillna("").str.strip().ne("")
)

df["machine_translated_flag"] = (
    df["machine_translated"].fillna(False).astype(bool)
)

df["hearing_impaired_flag"] = (
    df["hearing_impaired"].fillna(False).astype(bool)
)

# Sorting is only for human review, not automatic acceptance.
df = df.sort_values(
    [
        "movie_record_id",
        "tmdb_match",
        "year_match",
        "release_name_present",
        "download_count",
    ],
    ascending=[True, False, False, False, False],
)

display_cols = [
    "movie_record_id",
    "requested_title",
    "subtitle_id",
    "release",
    "download_count",
    "language",
    "tmdb_match",
    "year_match",
    "hearing_impaired_flag",
    "machine_translated_flag",
]

for record_id, group in df.groupby("movie_record_id", sort=True):
    print("\n" + "=" * 90)
    print(group["requested_title"].iloc[0], "|", record_id)
    print("=" * 90)
    display(group[display_cols].head(5))

print("\nREAD-ONLY SHORTLIST — no API calls, downloads, or manifest updates.")

In [ ]:

import pandas as pd
from pathlib import Path

SEARCH_FILE = (
    Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/subtitle_corpus_v2/search_results"
    / "batch06_2013_english_search_20261025_192858.csv"
)

candidates = pd.read_csv(SEARCH_FILE)

# Convert fields to consistent types
for col in ["requested_tmdb_id", "matched_tmdb_id"]:
    candidates[col] = pd.to_numeric(candidates[col], errors="coerce").astype("Int64")

candidates["matched_movie_year"] = pd.to_numeric(
    candidates["matched_movie_year"], errors="coerce"
)
candidates["download_count"] = pd.to_numeric(
    candidates["download_count"], errors="coerce"
)

# Recreate matching checks from the saved CSV
candidates["tmdb_match"] = (
    candidates["requested_tmdb_id"] == candidates["matched_tmdb_id"]
)
candidates["year_match"] = candidates["matched_movie_year"] == 2013

# First-pass exclusion of obvious trailers and full-movie releases
release_text = candidates["release"].fillna("").str.lower()

exclude_pattern = r"\b(?:trailer|teaser|full[\s._-]*movie)\b"
candidates["obvious_exclusion"] = release_text.str.contains(
    exclude_pattern, regex=True, na=False
)

review = candidates.loc[
    candidates["language"].fillna("").str.lower().eq("en")
    & candidates["tmdb_match"]
    & candidates["year_match"]
    & ~candidates["obvious_exclusion"]
].copy()

review = review.sort_values(
    ["movie_record_id", "download_count"],
    ascending=[True, False]
)

columns = [
    "movie_record_id",
    "requested_title",
    "subtitle_id",
    "release",
    "download_count",
    "language",
    "tmdb_match",
    "year_match",
    "hearing_impaired",
    "machine_translated",
]

# Display only columns that actually exist in this CSV
columns = [col for col in columns if col in review.columns]

print("Total candidates in saved search:", len(candidates))
print("Candidates retained for first-pass review:", len(review))
print("Excluded obvious trailer/full-movie results:",
      int(candidates["obvious_exclusion"].sum()))

display(review[columns].to_string(index=False))


In [ ]:
import requests
from pathlib import Path
from getpass import getpass
import pandas as pd
import time
import re

# ---------- Paths ----------
PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# ---------- Five selected candidates ----------
# Highest-download provisional candidates from the shortlist.
batch06 = [
    {"movie_record_id": "2013_1", "title": "Dhoom 3", "subtitle_id": "746083"},
    {"movie_record_id": "2013_2", "title": "Chennai Express", "subtitle_id": "4441558"},
    {"movie_record_id": "2013_3", "title": "Krrish 3", "subtitle_id": "924009"},
    {"movie_record_id": "2013_4", "title": "Yeh Jawaani Hai Deewani", "subtitle_id": "899382"},
    {"movie_record_id": "2013_5", "title": "Goliyon Ki Raasleela Ram-Leela", "subtitle_id": "965464"},
]

# ---------- Credentials ----------
# Read secrets without printing them.
from google.colab import userdata

api_key = userdata.get("OPEN_SUBTITLES_API_KEY")
username = userdata.get("OPEN_SUBTITLES_USERNAME")
password = userdata.get("OPEN_SUBTITLES_PASSWORD")

if not all([api_key, username, password]):
    raise ValueError("OpenSubtitles API key or login credentials are missing from Colab Secrets.")

BASE = "https://api.opensubtitles.com/api/v1"
headers = {
    "Api-Key": api_key,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch v1",
}

# ---------- Fresh login ----------
login_response = requests.post(
    f"{BASE}/login",
    headers=headers,
    json={"username": username, "password": password},
    timeout=30,
)

print("Login HTTP status:", login_response.status_code)
login_response.raise_for_status()
token = login_response.json().get("token")

if not token:
    raise RuntimeError("Login succeeded but no token was returned. No downloads attempted.")

auth_headers = {**headers, "Authorization": f"Bearer {token}"}

# ---------- Check quota ----------
quota_response = requests.get(
    f"{BASE}/infos/user",
    headers=auth_headers,
    timeout=30,
)

print("Quota HTTP status:", quota_response.status_code)
quota_response.raise_for_status()

quota_data = quota_response.json().get("data", {})
allowed = quota_data.get("allowed_downloads")
used = quota_data.get("downloads_count")

print("Allowed downloads:", allowed)
print("Downloads used:", used)

if allowed is None or used is None:
    raise RuntimeError("Quota values could not be confirmed. No downloads attempted.")

remaining = int(allowed) - int(used)
print("Downloads remaining:", remaining)

if remaining < len(batch06):
    raise RuntimeError(
        f"Only {remaining} downloads remain; 5 are required. "
        "No subtitle downloads were attempted."
    )

# ---------- Download selected candidates ----------
results = []

for item in batch06:
    subtitle_id = item["subtitle_id"]
    safe_title = re.sub(r"[^A-Za-z0-9]+", "_", item["title"]).strip("_")
    destination = RAW / f'{item["movie_record_id"]}_{safe_title}_candidate_{subtitle_id}.srt'

    if destination.exists():
        print(f"SKIPPED (file already exists): {destination.name}")
        results.append({**item, "status": "Skipped_existing_file", "file": str(destination)})
        continue

    try:
        download_response = requests.post(
            f"{BASE}/download",
            headers=auth_headers,
            json={"file_id": int(subtitle_id)},
            timeout=30,
        )

        if download_response.status_code != 200:
            print(f"FAILED: {item['title']} | HTTP {download_response.status_code}")
            results.append({
                **item,
                "status": f"Download_API_HTTP_{download_response.status_code}",
                "file": None,
            })
            continue

        download_info = download_response.json()
        download_link = download_info.get("link")

        if not download_link:
            print(f"FAILED: {item['title']} | No download link returned")
            results.append({**item, "status": "No_download_link", "file": None})
            continue

        file_response = requests.get(download_link, timeout=60)
        file_response.raise_for_status()

        # Verify that the response looks like subtitle text, not an HTML error page.
        content = file_response.content
        if not content or b"<html" in content[:500].lower():
            print(f"FAILED: {item['title']} | Empty or unexpected response")
            results.append({**item, "status": "Unexpected_file_content", "file": None})
            continue

        # Save the downloaded bytes as received; validation happens separately.
        with open(destination, "xb") as f:
            f.write(content)

        print(f"DOWNLOADED: {item['title']} -> {destination.name}")
        results.append({**item, "status": "Downloaded_unvalidated", "file": str(destination)})

    except Exception as exc:
        print(f"ERROR: {item['title']} | {type(exc).__name__}: {exc}")
        results.append({**item, "status": "Error", "file": None})

    # Avoid sending requests in a rapid burst.
    time.sleep(1)

print("\nBATCH 06 DOWNLOAD SUMMARY")
display(pd.DataFrame(results).to_string(index=False))
print("\nManifest and master dataset were not modified.")

In [ ]:

from pathlib import Path
import re
import pandas as pd

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

files = [
    ("2013_3", "Krrish 3",
     RAW / "2013_3_Krrish_3_candidate_924009.srt"),
    ("2013_4", "Yeh Jawaani Hai Deewani",
     RAW / "2013_4_Yeh_Jawaani_Hai_Deewani_candidate_899382.srt"),
    ("2013_5", "Goliyon Ki Raasleela Ram-Leela",
     RAW / "2013_5_Goliyon_Ki_Raasleela_Ram_Leela_candidate_965464.srt"),
]

results = []

for record_id, title, path in files:
    row = {
        "movie_record_id": record_id,
        "title": title,
        "file_exists": path.is_file(),
        "file_size_bytes": path.stat().st_size if path.is_file() else None,
        "encoding": None,
        "timestamp_count": 0,
        "dialogue_blocks": 0,
        "word_count": 0,
        "first_timestamp": None,
        "last_timestamp": None,
        "read_error": None,
    }

    if path.is_file():
        try:
            raw_bytes = path.read_bytes()
            text = None

            for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
                try:
                    text = raw_bytes.decode(encoding)
                    row["encoding"] = encoding
                    break
                except UnicodeDecodeError:
                    continue

            if text is not None:
                timestamps = re.findall(
                    r"(?m)^\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})"
                    r"\s*-->\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})",
                    text
                )

                row["timestamp_count"] = len(timestamps)
                row["dialogue_blocks"] = len(
                    re.findall(r"(?m)^\s*\d+\s*$", text)
                )
                row["word_count"] = len(
                    re.findall(r"\b[\w']+\b", text, flags=re.UNICODE)
                )

                if timestamps:
                    row["first_timestamp"] = timestamps[0][0]
                    row["last_timestamp"] = timestamps[-1][1]

        except Exception as exc:
            row["read_error"] = f"{type(exc).__name__}: {exc}"

    results.append(row)

display(pd.DataFrame(results).to_string(index=False))


In [ ]:

from pathlib import Path

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

for record_id in ["2013_3", "2013_4", "2013_5"]:
    matches = sorted(RAW.glob(f"{record_id}*.srt"))
    print(f"\n{record_id}: {len(matches)} file(s)")
    for path in matches:
        print(f"  {path.name} | {path.stat().st_size:,} bytes")


In [ ]:

from pathlib import Path
import re
import pandas as pd

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

files = [
    ("2013_3", "Krrish 3",
     "2013_3_Krrish_3_candidate_924009.srt", 140),
    ("2013_4", "Yeh Jawaani Hai Deewani",
     "2013_4_Yeh_Jawaani_Hai_Deewani_candidate_899382.srt", 161),
    ("2013_5", "Goliyon Ki Raasleela Ram-Leela",
     "2013_5_Goliyon_Ki_Raasleela_Ram_Leela_candidate_965464.srt", 155),
]

def seconds(ts):
    hh, mm, rest = ts.replace(",", ".").split(":")
    return int(hh) * 3600 + int(mm) * 60 + float(rest)

results = []

for record_id, title, filename, expected_min in files:
    path = RAW / filename
    if not path.is_file():
        print(f"MISSING: {path}")
        continue

    text = path.read_text(encoding="utf-8-sig", errors="replace")
    matches = re.findall(
        r"(?m)^\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})"
        r"\s*-->\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})",
        text
    )

    starts = [seconds(a) for a, b in matches]
    ends = [seconds(b) for a, b in matches]
    gaps = [
        starts[i] - ends[i - 1]
        for i in range(1, len(starts))
        if starts[i] - ends[i - 1] > 0
    ]

    last_end = max(ends) if ends else 0
    results.append({
        "movie_record_id": record_id,
        "title": title,
        "file_exists": True,
        "subtitle_blocks": len(matches),
        "reference_runtime_min": expected_min,
        "last_subtitle_min": round(last_end / 60, 2),
        "runtime_reached_pct": round(
            100 * last_end / (expected_min * 60), 1
        ),
        "gaps_over_30_sec": sum(g > 30 for g in gaps),
        "gaps_over_120_sec": sum(g > 120 for g in gaps),
        "largest_gap_sec": round(max(gaps), 1) if gaps else 0,
        "out_of_order_starts": sum(
            starts[i] < starts[i - 1] for i in range(1, len(starts))
        ),
    })

display(pd.DataFrame(results).to_string(index=False))


In [ ]:

from pathlib import Path
import re
import pandas as pd

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

files = [
    ("2013_3", "Krrish 3", "2013_3_Krrish_3_candidate_924009.srt"),
    ("2013_4", "Yeh Jawaani Hai Deewani",
     "2013_4_Yeh_Jawaani_Hai_Deewani_candidate_899382.srt"),
    ("2013_5", "Goliyon Ki Raasleela Ram-Leela",
     "2013_5_Goliyon_Ki_Raasleela_Ram_Leela_candidate_965464.srt"),
]

def to_seconds(ts):
    hh, mm, ss = ts.replace(",", ".").split(":")
    return int(hh) * 3600 + int(mm) * 60 + float(ss)

rows = []

for record_id, title, filename in files:
    path = RAW / filename
    text = path.read_text(encoding="utf-8-sig", errors="replace")

    pairs = re.findall(
        r"(?m)^\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})"
        r"\s*-->\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})",
        text
    )

    starts = [to_seconds(a) for a, b in pairs]
    ends = [to_seconds(b) for a, b in pairs]
    gaps = [
        starts[i] - ends[i - 1]
        for i in range(1, len(starts))
        if starts[i] > ends[i - 1]
    ]

    rows.append({
        "movie_record_id": record_id,
        "title": title,
        "blocks": len(pairs),
        "gaps_over_30_sec": int(sum(g > 30 for g in gaps)),
        "gaps_over_120_sec": int(sum(g > 120 for g in gaps)),
        "largest_gap_seconds": round(max(gaps), 1) if gaps else 0,
        "out_of_order_starts": int(sum(
            starts[i] < starts[i - 1]
            for i in range(1, len(starts))
        )),
        "last_subtitle_timestamp": pairs[-1][1] if pairs else None,
    })

display(pd.DataFrame(rows).to_string(index=False))


In [ ]:

import requests
from pathlib import Path
from google.colab import userdata
import pandas as pd
import time

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
RAW = PROJECT / "01_data/processed/subtitle_corpus_v2/raw"
SEARCH_FILE = (
    PROJECT / "01_data/processed/subtitle_corpus_v2/search_results"
    / "batch06_2013_english_search_20261025_192858.csv"
)

# Next-highest alternatives; do not retry the previous failing IDs.
retry_batch = [
    {"movie_record_id": "2013_1", "title": "Dhoom 3", "subtitle_id": "746481"},
    {"movie_record_id": "2013_2", "title": "Chennai Express", "subtitle_id": "4442267"},
]

api_key = userdata.get("OPEN_SUBTITLES_API_KEY")
username = userdata.get("OPEN_SUBTITLES_USERNAME")
password = userdata.get("OPEN_SUBTITLES_PASSWORD")

if not all([api_key, username, password]):
    raise ValueError("OpenSubtitles credentials missing from Colab Secrets.")

BASE = "https://api.opensubtitles.com/api/v1"
headers = {
    "Api-Key": api_key,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch v1",
}

# Verify candidate IDs exist in the saved search results.
search_df = pd.read_csv(SEARCH_FILE)
saved_ids = set(search_df["subtitle_id"].astype(str))
for item in retry_batch:
    if item["subtitle_id"] not in saved_ids:
        raise ValueError(f"Candidate {item['subtitle_id']} is not in the saved search.")

# Fresh login
login = requests.post(
    f"{BASE}/login",
    headers=headers,
    json={"username": username, "password": password},
    timeout=30,
)
print("Login HTTP status:", login.status_code)
login.raise_for_status()
token = login.json().get("token")
if not token:
    raise RuntimeError("No login token returned; no downloads attempted.")

auth_headers = {**headers, "Authorization": f"Bearer {token}"}

# Check quota
quota = requests.get(
    f"{BASE}/infos/user", headers=auth_headers, timeout=30
)
print("Quota HTTP status:", quota.status_code)
quota.raise_for_status()
q = quota.json().get("data", {})
allowed = q.get("allowed_downloads")
used = q.get("downloads_count")

if allowed is None or used is None:
    raise RuntimeError("Quota could not be confirmed; no downloads attempted.")

remaining = int(allowed) - int(used)
print("Downloads remaining:", remaining)

if remaining < len(retry_batch):
    raise RuntimeError("Insufficient confirmed quota for both retries.")

results = []

for item in retry_batch:
    # Use the same naming convention as Batch 06.
    safe_title = item["title"].replace(" ", "_")
    destination = RAW / (
        f'{item["movie_record_id"]}_{safe_title}_candidate_'
        f'{item["subtitle_id"]}.srt'
    )

    if destination.exists():
        print(f"SKIPPED existing file: {destination.name}")
        results.append({**item, "status": "Skipped_existing_file"})
        continue

    try:
        response = requests.post(
            f"{BASE}/download",
            headers=auth_headers,
            json={"file_id": int(item["subtitle_id"])},
            timeout=30,
        )

        if response.status_code != 200:
            print(f"FAILED: {item['title']} | HTTP {response.status_code}")
            results.append({
                **item,
                "status": f"HTTP_{response.status_code}",
            })
            time.sleep(1)
            continue

        link = response.json().get("link")
        if not link:
            results.append({**item, "status": "No_download_link"})
            continue

        file_response = requests.get(link, timeout=60)
        file_response.raise_for_status()
        content = file_response.content

        if not content or b"<html" in content[:500].lower():
            results.append({**item, "status": "Unexpected_file_content"})
            continue

        with open(destination, "xb") as f:
            f.write(content)

        print(f"DOWNLOADED: {item['title']} -> {destination.name}")
        results.append({
            **item,
            "status": "Downloaded_unvalidated",
            "file": str(destination),
        })

    except Exception as exc:
        print(f"ERROR: {item['title']} | {type(exc).__name__}: {exc}")
        results.append({**item, "status": "Error"})

    time.sleep(1)

print("\nRETRY SUMMARY")
display(pd.DataFrame(results).to_string(index=False))
print("Manifest and master dataset were not modified.")


In [ ]:

from pathlib import Path
import re
import pandas as pd

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

batch06 = [
    ("2013_1", "Dhoom 3", "2013_1_Dhoom_3_candidate_746481.srt", 150),
    ("2013_2", "Chennai Express",
     "2013_2_Chennai_Express_candidate_4442267.srt", 141),
    ("2013_3", "Krrish 3", "2013_3_Krrish_3_candidate_924009.srt", 152),
    ("2013_4", "Yeh Jawaani Hai Deewani",
     "2013_4_Yeh_Jawaani_Hai_Deewani_candidate_899382.srt", 160),
    ("2013_5", "Goliyon Ki Raasleela Ram-Leela",
     "2013_5_Goliyon_Ki_Raasleela_Ram_Leela_candidate_965464.srt", 155),
]

pattern = re.compile(
    r"(?m)^\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})"
    r"\s*-->\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})"
)

def to_seconds(ts):
    hh, mm, ss = ts.replace(",", ".").split(":")
    return int(hh) * 3600 + int(mm) * 60 + float(ss)

rows = []

for record_id, title, filename, runtime_min in batch06:
    path = RAW / filename

    if not path.is_file():
        rows.append({
            "ID": record_id, "Movie": title,
            "File": "MISSING", "Blocks": 0, "Words": 0,
            "First timestamp": None, "Last timestamp": None,
            "Coverage %": None, "Gaps >120s": None,
            "Largest gap (s)": None, "Out of order": None
        })
        continue

    text = path.read_text(encoding="utf-8-sig", errors="replace")
    pairs = pattern.findall(text)

    starts = [to_seconds(a) for a, b in pairs]
    ends = [to_seconds(b) for a, b in pairs]
    gaps = [
        starts[i] - ends[i - 1]
        for i in range(1, len(starts))
        if starts[i] > ends[i - 1]
    ]

    words = re.findall(r"\b[\w']+\b", text, flags=re.UNICODE)

    # Use actual parsed maximum timestamp, not the final file row.
    last_sec = max(ends) if ends else 0
    first_sec = min(starts) if starts else 0

    rows.append({
        "ID": record_id,
        "Movie": title,
        "File": "FOUND",
        "Blocks": len(pairs),
        "Words": len(words),
        "First timestamp": (
            f"{int(first_sec // 3600):02d}:"
            f"{int((first_sec % 3600) // 60):02d}:"
            f"{first_sec % 60:06.3f}" if pairs else None
        ),
        "Last timestamp": (
            f"{int(last_sec // 3600):02d}:"
            f"{int((last_sec % 3600) // 60):02d}:"
            f"{last_sec % 60:06.3f}" if pairs else None
        ),
        "Coverage %": round(100 * last_sec / (runtime_min * 60), 1),
        "Gaps >120s": sum(g > 120 for g in gaps),
        "Largest gap (s)": round(max(gaps), 1) if gaps else 0,
        "Out of order": sum(
            starts[i] < starts[i - 1]
            for i in range(1, len(starts))
        ),
    })

report = pd.DataFrame(rows)

# Display all rows and columns clearly
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.width", 220)

print("BATCH 06 — COMPLETE VALIDATION SUMMARY")
print("Reference runtimes are provisional screening estimates.")
display(report)


In [ ]:

from pathlib import Path
import pandas as pd
from datetime import datetime

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

VALIDATED.mkdir(parents=True, exist_ok=True)

# Four provisionally accepted candidates; Dhoom 3 remains rejected.
batch06 = [
    ("2013_2", "Chennai Express",
     "2013_2_Chennai_Express_candidate_4442267.srt"),
    ("2013_3", "Krrish 3",
     "2013_3_Krrish_3_candidate_924009.srt"),
    ("2013_4", "Yeh Jawaani Hai Deewani",
     "2013_4_Yeh_Jawaani_Hai_Deewani_candidate_899382.srt"),
    ("2013_5", "Goliyon Ki Raasleela Ram-Leela",
     "2013_5_Goliyon_Ki_Raasleela_Ram_Leela_candidate_965464.srt"),
]

if not MANIFEST.is_file():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")

manifest = pd.read_csv(MANIFEST, dtype={"movie_record_id": "string"})

print("Manifest shape:", manifest.shape)
print("Manifest columns:", manifest.columns.tolist())

for record_id, title, filename in batch06:
    existing = manifest[manifest["movie_record_id"] == record_id]
    raw_path = RAW / filename

    print(f"\n{record_id} — {title}")
    print("Raw file exists:", raw_path.is_file())
    print("Existing manifest rows:", len(existing))

    if len(existing):
        display(existing[[
            c for c in [
                "movie_record_id", "movie_title", "acquisition_status",
                "validation_status", "raw_subtitle_file",
                "validated_subtitle_file", "notes"
            ] if c in manifest.columns
        ]])

print("\nNo files copied and no manifest changes made.")


In [ ]:

from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil
import re

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

batch06 = [
    {
        "id": "2013_2", "title": "Chennai Express",
        "filename": "2013_2_Chennai_Express_candidate_4442267.srt",
        "subtitle_id": "4442267", "blocks": 1097, "words": 14566,
        "first": "00:00:00.000", "last": "02:03:03.299",
        "quality": "Timestamp ordering and long gaps require review"
    },
    {
        "id": "2013_3", "title": "Krrish 3",
        "filename": "2013_3_Krrish_3_candidate_924009.srt",
        "subtitle_id": "924009", "blocks": 1291, "words": 18192,
        "first": "00:00:14.515", "last": "01:39:38.903",
        "quality": "Provisional acceptance; coverage not independently confirmed"
    },
    {
        "id": "2013_4", "title": "Yeh Jawaani Hai Deewani",
        "filename": "2013_4_Yeh_Jawaani_Hai_Deewani_candidate_899382.srt",
        "subtitle_id": "899382", "blocks": 1276, "words": 19400,
        "first": "00:00:58.058", "last": "01:34:18.244",
        "quality": "One gap exceeds five minutes; review flagged"
    },
    {
        "id": "2013_5", "title": "Goliyon Ki Raasleela Ram-Leela",
        "filename": "2013_5_Goliyon_Ki_Raasleela_Ram_Leela_candidate_965464.srt",
        "subtitle_id": "965464", "blocks": 1868, "words": 27379,
        "first": "00:00:36.954", "last": "01:51:19.996",
        "quality": "One gap exceeds three minutes; review flagged"
    },
]

if not MANIFEST.is_file():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")

manifest = pd.read_csv(MANIFEST, dtype={"movie_record_id": "string"})
required_columns = manifest.columns.tolist()

if manifest["movie_record_id"].duplicated().any():
    raise ValueError("Existing manifest contains duplicate IDs. Stop and review first.")

# Preflight checks: all four IDs must be absent and all raw files must exist.
for item in batch06:
    if item["id"] in set(manifest["movie_record_id"].dropna()):
        raise ValueError(f"{item['id']} already exists in manifest. No changes made.")
    if not (RAW / item["filename"]).is_file():
        raise FileNotFoundError(f"Raw file missing: {item['filename']}")

VALIDATED.mkdir(parents=True, exist_ok=True)

# Preserve the original manifest before any changes.
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = MANIFEST.with_name(
    f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_batch06_{stamp}.csv"
)
shutil.copy2(MANIFEST, backup)
print("Manifest backup created:", backup.name)

new_rows = []

for item in batch06:
    raw_path = RAW / item["filename"]
    validated_name = f"{item['id']}_{re.sub(r'[^A-Za-z0-9]+', '_', item['title']).strip('_').upper()}_ENGLISH.srt"
    validated_path = VALIDATED / validated_name

    if validated_path.exists():
        raise FileExistsError(
            f"Validated destination already exists: {validated_path.name}. "
            "Stop and review; no manifest update performed."
        )

    shutil.copy2(raw_path, validated_path)

    # Verify byte-for-byte copy integrity.
    if raw_path.read_bytes() != validated_path.read_bytes():
        validated_path.unlink(missing_ok=True)
        raise IOError(f"Copy verification failed for {item['id']}.")

    row = {col: pd.NA for col in required_columns}
    row.update({
        "movie_record_id": item["id"],
        "year": 2013,
        "rank": int(item["id"].split("_")[1]),
        "movie_title": item["title"],
        "language": "Hindi",
        "subtitle_language": "English",
        "subtitle_type": "English fallback",
        "subtitle_source": "OpenSubtitles",
        "source_id": item["subtitle_id"],
        "raw_subtitle_file": str(raw_path),
        "validated_subtitle_file": str(validated_path),
        "acquisition_status": "Acquired",
        "validation_status": "Accepted_with_quality_flags",
        "title_verified": True,
        "year_verified": True,
        "movie_identity_verified": True,
        "language_verified": True,
        "dialogue_blocks": item["blocks"],
        "word_count": item["words"],
        "character_count": len(
            validated_path.read_text(encoding="utf-8-sig", errors="replace")
        ),
        "encoding_status": "UTF-8 with BOM",
        "quality_status": "Review_Flag",
        "notes": (
            f"Batch 06 candidate {item['subtitle_id']}. "
            f"First timestamp {item['first']}; last timestamp {item['last']}. "
            f"{item['quality']}. Raw file preserved unchanged."
        ),
    })
    new_rows.append(row)

updated = pd.concat(
    [manifest, pd.DataFrame(new_rows, columns=required_columns)],
    ignore_index=True
)

# Final duplicate guard before writing.
if updated["movie_record_id"].duplicated().any():
    raise ValueError("Duplicate ID detected before save; manifest not written.")

updated.to_csv(MANIFEST, index=False)

# Verify saved manifest.
check = pd.read_csv(MANIFEST, dtype={"movie_record_id": "string"})
print("\nManifest updated and reloaded.")
print("Backup:", backup.name)
print("Previous rows:", len(manifest))
print("New rows added:", len(new_rows))
print("Current shape:", check.shape)
print("Duplicate IDs:", int(check["movie_record_id"].duplicated().sum()))
print("\nBatch 06 records:")
display(check[check["movie_record_id"].isin(
    [item["id"] for item in batch06]
)][[
    "movie_record_id", "movie_title", "source_id",
    "acquisition_status", "validation_status",
    "raw_subtitle_file", "validated_subtitle_file", "quality_status"
]].to_string(index=False))

print("\nMaster dataset was not modified.")


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MAPPING = PROJECT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
MANIFEST = PROJECT / "01_data/processed/subtitle_corpus_v2/ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

master = pd.read_csv(MASTER)
mapping = pd.read_csv(MAPPING)
manifest = pd.read_csv(MANIFEST, dtype={"movie_record_id": "string"})

master["movie_record_id"] = master["movie_record_id"].astype("string")
mapping["movie_record_id"] = mapping["movie_record_id"].astype("string")

# Select exactly the five records from 2014 and five from 2015.
target = master.loc[master["year"].isin([2014, 2015])].copy()

# Take TMDB IDs only from the locked mapping to avoid duplicate-column suffixes.
mapping_cols = [
    "movie_record_id", "tmdb_id", "tmdb_title",
    "tmdb_release_date", "tmdb_original_language", "match_status"
]
mapping_subset = mapping[mapping_cols].copy()

target = target.drop(
    columns=["tmdb_id"], errors="ignore"
).merge(
    mapping_subset,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

counts = target.groupby("year")["movie_record_id"].nunique().to_dict()
print("Records by year:", counts)

if counts.get(2014, 0) != 5 or counts.get(2015, 0) != 5:
    raise ValueError("Expected five master records per year; stop and review.")

target["tmdb_id"] = pd.to_numeric(target["tmdb_id"], errors="coerce")
if target["tmdb_id"].isna().any():
    print("Records missing TMDB IDs:")
    display(target.loc[target["tmdb_id"].isna(), [
        "movie_record_id", "year", "movie_title",
        "tmdb_title", "match_status"
    ]])
    raise ValueError("Missing TMDB mapping. No searches were run.")

existing_ids = set(manifest["movie_record_id"].dropna().astype(str))
target["already_in_manifest"] = target["movie_record_id"].isin(existing_ids)

print("\nBatch 07 selection — review before searching")
display(target[[
    "movie_record_id", "year", "movie_title",
    "tmdb_id", "tmdb_title", "match_status",
    "already_in_manifest"
]].sort_values(["year", "movie_record_id"]).to_string(index=False))

print("\nNo API calls made. No files changed.")


In [ ]:

from pathlib import Path
from datetime import datetime
import pandas as pd
import requests
import time
from google.colab import userdata

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
SEARCH_DIR = WORK / "search_results"
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the verified Batch 07 selection from the preceding cell.
to_search = target.loc[~target["already_in_manifest"]].copy()

api_key = userdata.get("OPEN_SUBTITLES_API_KEY")
username = userdata.get("OPEN_SUBTITLES_USERNAME")
password = userdata.get("OPEN_SUBTITLES_PASSWORD")

if not all([api_key, username, password]):
    raise ValueError("OpenSubtitles credentials missing from Colab Secrets.")

BASE = "https://api.opensubtitles.com/api/v1"
headers = {
    "Api-Key": api_key,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch v1",
}

login = requests.post(
    f"{BASE}/login",
    headers=headers,
    json={"username": username, "password": password},
    timeout=30,
)
print("Login HTTP status:", login.status_code)
login.raise_for_status()
token = login.json().get("token")
if not token:
    raise RuntimeError("No login token returned; search stopped.")

auth_headers = {**headers, "Authorization": f"Bearer {token}"}
records, errors = [], []

for _, movie in to_search.sort_values(
    ["year", "movie_record_id"]
).iterrows():
    record_id = str(movie["movie_record_id"])
    title = str(movie["movie_title"])
    tmdb_id = int(movie["tmdb_id"])

    try:
        response = requests.get(
            f"{BASE}/subtitles",
            headers=auth_headers,
            params={
                "tmdb_id": tmdb_id,
                "languages": "en",
                "type": "movie",
            },
            timeout=30,
        )

        if response.status_code != 200:
            errors.append({
                "movie_record_id": record_id,
                "title": title,
                "http_status": response.status_code,
            })
            print(f"FAILED: {record_id} {title} | HTTP {response.status_code}")
            time.sleep(0.5)
            continue

        candidates = response.json().get("data", [])
        print(f"{record_id} | {title}: {len(candidates)} candidate(s)")

        for candidate in candidates:
            attr = candidate.get("attributes", {})
            feature = attr.get("feature_details") or {}
            files = attr.get("files") or []

            records.append({
                "movie_record_id": record_id,
                "requested_title": title,
                "requested_year": int(movie["year"]),
                "requested_tmdb_id": tmdb_id,
                "subtitle_id": attr.get("subtitle_id") or candidate.get("id"),
                "release": attr.get("release"),
                "language": attr.get("language"),
                "download_count": attr.get("download_count"),
                "hearing_impaired": attr.get("hearing_impaired"),
                "machine_translated": attr.get("machine_translated"),
                "upload_date": attr.get("upload_date"),
                "matched_movie_title": feature.get("title"),
                "matched_movie_year": feature.get("year"),
                "matched_tmdb_id": feature.get("tmdb_id"),
                "matched_imdb_id": feature.get("imdb_id"),
                "file_count": len(files),
            })

    except Exception as exc:
        errors.append({
            "movie_record_id": record_id,
            "title": title,
            "http_status": f"{type(exc).__name__}: {exc}",
        })
        print(f"ERROR: {record_id} {title} | {type(exc).__name__}")

    time.sleep(0.5)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output = SEARCH_DIR / f"batch07_2014_2015_english_search_{stamp}.csv"
pd.DataFrame(records).to_csv(output, index=False)
print("\nSaved search results:", output.name)
print("Total candidates:", len(records))
print("Search errors:", len(errors))

if errors:
    error_path = SEARCH_DIR / f"batch07_2014_2015_search_errors_{stamp}.csv"
    pd.DataFrame(errors).to_csv(error_path, index=False)
    print("Errors saved:", error_path.name)

if records:
    df = pd.DataFrame(records)
    df["download_count"] = pd.to_numeric(
        df["download_count"], errors="coerce"
    )
    df["requested_tmdb_id"] = pd.to_numeric(
        df["requested_tmdb_id"], errors="coerce"
    )
    df["matched_tmdb_id"] = pd.to_numeric(
        df["matched_tmdb_id"], errors="coerce"
    )
    df["matched_movie_year"] = pd.to_numeric(
        df["matched_movie_year"], errors="coerce"
    )

    df["tmdb_match"] = df["requested_tmdb_id"].eq(df["matched_tmdb_id"])
    df["year_match"] = df["requested_year"].eq(df["matched_movie_year"])

    suitable = df.loc[df["tmdb_match"] & df["year_match"]].copy()
    suitable = suitable.sort_values(
        ["movie_record_id", "download_count"],
        ascending=[True, False]
    )

    print("\nTOP 3 MATCHING CANDIDATES PER MOVIE — REVIEW ONLY")
    display(suitable.groupby(
        "movie_record_id", group_keys=False
    ).head(3)[[
        "movie_record_id", "requested_title", "subtitle_id",
        "release", "download_count", "language",
        "tmdb_match", "year_match", "hearing_impaired",
        "machine_translated"
    ]].to_string(index=False))

print("\nNo subtitles downloaded. Manifest and master dataset unchanged.")


In [ ]:

import pandas as pd
from pathlib import Path

SEARCH_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

file_path = SEARCH_DIR / "batch07_2014_2015_english_search_20261025_200505.csv"
df = pd.read_csv(file_path)

for col in ["download_count", "requested_tmdb_id", "matched_tmdb_id",
            "requested_year", "matched_movie_year"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["tmdb_match"] = df["requested_tmdb_id"].eq(df["matched_tmdb_id"])
df["year_match"] = df["requested_year"].eq(df["matched_movie_year"])

# Exact TMDB ID and year matches only
suitable = df[df["tmdb_match"] & df["year_match"]].copy()
suitable = suitable.sort_values(
    ["movie_record_id", "download_count"],
    ascending=[True, False]
)

print("Exact-match candidate counts:")
print(suitable.groupby("movie_record_id").size().to_string())

print("\nTOP 3 CANDIDATES PER MOVIE")
columns = [
    "movie_record_id", "requested_title", "subtitle_id",
    "release", "download_count", "language",
    "hearing_impaired", "machine_translated"
]
display(suitable.groupby(
    "movie_record_id", group_keys=False
).head(3)[columns].to_string(index=False))


In [ ]:

import pandas as pd
from pathlib import Path
from IPython.display import display

p = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results/"
    "batch07_2014_2015_english_search_20261025_200505.csv"
)

df = pd.read_csv(p)

for c in [
    "download_count", "requested_tmdb_id", "matched_tmdb_id",
    "requested_year", "matched_movie_year"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Keep exact TMDB ID and year matches
df = df[
    df["requested_tmdb_id"].eq(df["matched_tmdb_id"])
    & df["requested_year"].eq(df["matched_movie_year"])
].copy()

df["download_count"] = df["download_count"].fillna(0).astype(int)

# Select the highest-downloaded candidate per movie
best = (
    df.sort_values(
        ["movie_record_id", "download_count"],
        ascending=[True, False]
    )
    .drop_duplicates("movie_record_id")
    .copy()
)

# Shorten long release descriptions for readability
best["release"] = (
    best["release"].fillna("Not provided")
    .str.slice(0, 65)
    .where(best["release"].fillna("").str.len() <= 65,
           best["release"].fillna("").str.slice(0, 62) + "...")
)

best = best.rename(columns={
    "movie_record_id": "Record ID",
    "requested_title": "Movie",
    "subtitle_id": "Subtitle ID",
    "release": "Release (shortened)",
    "download_count": "Downloads",
    "language": "Language",
    "hearing_impaired": "HI",
    "machine_translated": "Machine Translated"
})

best = best[[
    "Record ID", "Movie", "Subtitle ID", "Release (shortened)",
    "Downloads", "Language", "HI", "Machine Translated"
]].reset_index(drop=True)

# Display as a real Colab table, not plain text
display(best.style
    .format({"Downloads": "{:,}"})
    .set_properties(**{
        "text-align": "left",
        "white-space": "normal",
        "max-width": "250px"
    })
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left"),
                                     ("white-space", "normal")]},
        {"selector": "td", "props": [("padding", "6px 10px")]}
    ])
)


In [ ]:

import requests
from google.colab import userdata

BASE = "https://api.opensubtitles.com/api/v1"
api_key = userdata.get("OPEN_SUBTITLES_API_KEY")
username = userdata.get("OPEN_SUBTITLES_USERNAME")
password = userdata.get("OPEN_SUBTITLES_PASSWORD")

headers = {
    "Api-Key": api_key,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch v1",
}

login = requests.post(
    f"{BASE}/login",
    headers=headers,
    json={"username": username, "password": password},
    timeout=30,
)
login.raise_for_status()
token = login.json().get("token")
if not token:
    raise RuntimeError("Login succeeded but no token was returned.")

quota_response = requests.get(
    f"{BASE}/infos/user",
    headers={**headers, "Authorization": f"Bearer {token}"},
    timeout=30,
)
print("Quota HTTP status:", quota_response.status_code)
quota_response.raise_for_status()

quota = quota_response.json().get("data", {})
print("Allowed downloads:", quota.get("allowed_downloads"))
print("Downloads used:", quota.get("downloads_count"))

if quota.get("allowed_downloads") is not None and quota.get("downloads_count") is not None:
    print("Remaining downloads:", quota["allowed_downloads"] - quota["downloads_count"])


In [ ]:

import requests
import time
from pathlib import Path
from datetime import datetime
from google.colab import userdata
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Selected highest-downloaded, exact TMDB/year-match candidates
selected = [
    ("2014_1", "PK", 1017746),
    ("2014_2", "Kick", 941291),
    ("2014_3", "Happy New Year", 1007904),
    ("2014_4", "Bang Bang!", 1000639),
    ("2014_5", "Singham Returns", 911878),
    ("2015_1", "Baahubali: The Beginning", 1054663),
    ("2015_2", "Bajrangi Bhaijaan", 1051138),
    ("2015_3", "Prem Ratan Dhan Payo", 3982350),
    ("2015_4", "Bajirao Mastani", 103061),
    ("2015_5", "Tanu Weds Manu Returns", 4447351),
]

api_key = userdata.get("OPEN_SUBTITLES_API_KEY")
username = userdata.get("OPEN_SUBTITLES_USERNAME")
password = userdata.get("OPEN_SUBTITLES_PASSWORD")

BASE = "https://api.opensubtitles.com/api/v1"
headers = {
    "Api-Key": api_key,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch v1",
}

# Fresh login
login = requests.post(
    f"{BASE}/login",
    headers=headers,
    json={"username": username, "password": password},
    timeout=30,
)
login.raise_for_status()
token = login.json().get("token")
if not token:
    raise RuntimeError("No login token returned.")

auth_headers = {**headers, "Authorization": f"Bearer {token}"}

# Recheck quota immediately before downloads
qr = requests.get(
    f"{BASE}/infos/user", headers=auth_headers, timeout=30
)
qr.raise_for_status()
quota = qr.json().get("data", {})
remaining = int(quota["allowed_downloads"]) - int(quota["downloads_count"])
print("Downloads remaining:", remaining)

if remaining < len(selected):
    raise RuntimeError(
        f"Need {len(selected)} downloads but only {remaining} remain. Stopped."
    )

results = []

for record_id, title, subtitle_id in selected:
    safe_title = "".join(
        c if c.isalnum() else "_" for c in title
    ).strip("_")
    filename = f"{record_id}_{safe_title}_candidate_{subtitle_id}.srt"
    destination = RAW / filename

    if destination.exists():
        print(f"SKIPPED (file already exists): {record_id} | {title}")
        results.append({
            "movie_record_id": record_id,
            "movie_title": title,
            "subtitle_id": subtitle_id,
            "status": "Skipped_existing_file",
            "raw_file": str(destination),
            "http_status": None,
        })
        continue

    try:
        # Request download link
        dr = requests.post(
            f"{BASE}/download",
            headers=auth_headers,
            json={"file_id": subtitle_id},
            timeout=30,
        )

        if dr.status_code != 200:
            print(f"DOWNLOAD LINK FAILED: {record_id} | HTTP {dr.status_code}")
            results.append({
                "movie_record_id": record_id,
                "movie_title": title,
                "subtitle_id": subtitle_id,
                "status": "Download_link_failed",
                "raw_file": "",
                "http_status": dr.status_code,
            })
            time.sleep(0.5)
            continue

        link_data = dr.json()
        download_url = link_data.get("link")
        if not download_url:
            print(f"NO LINK: {record_id}")
            results.append({
                "movie_record_id": record_id,
                "movie_title": title,
                "subtitle_id": subtitle_id,
                "status": "No_download_link",
                "raw_file": "",
                "http_status": dr.status_code,
            })
            continue

        # Download file bytes; do not modify subtitle contents
        file_response = requests.get(download_url, timeout=60)
        file_response.raise_for_status()
        content = file_response.content

        if not content:
            raise ValueError("Downloaded file is empty.")

        destination.write_bytes(content)
        print(f"DOWNLOADED: {record_id} | {title} | {len(content):,} bytes")

        results.append({
            "movie_record_id": record_id,
            "movie_title": title,
            "subtitle_id": subtitle_id,
            "status": "Downloaded",
            "raw_file": str(destination),
            "http_status": 200,
        })

    except Exception as exc:
        print(f"ERROR: {record_id} | {type(exc).__name__}: {exc}")
        results.append({
            "movie_record_id": record_id,
            "movie_title": title,
            "subtitle_id": subtitle_id,
            "status": f"Error: {type(exc).__name__}",
            "raw_file": "",
            "http_status": None,
        })

    time.sleep(0.5)

result_df = pd.DataFrame(results)
print("\nBATCH 07 DOWNLOAD SUMMARY")
display(result_df)

print("\nRaw files only; manifest and validated folder were not changed.")


In [ ]:

from pathlib import Path
import re
import pandas as pd
from IPython.display import display

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

movies = [
    ("2014_1", "PK", 153, "2014_1_PK_candidate_1017746.srt"),
    ("2014_2", "Kick", 146, "2014_2_Kick_candidate_941291.srt"),
    ("2014_3", "Happy New Year", 180, "2014_3_Happy_New_Year_candidate_1007904.srt"),
    ("2014_4", "Bang Bang!", 153, "2014_4_Bang_Bang_candidate_1000639.srt"),
    ("2014_5", "Singham Returns", 142, "2014_5_Singham_Returns_candidate_911878.srt"),
    ("2015_1", "Baahubali: The Beginning", 159, "2015_1_Baahubali_The_Beginning_candidate_1054663.srt"),
    ("2015_2", "Bajrangi Bhaijaan", 159, "2015_2_Bajrangi_Bhaijaan_candidate_1051138.srt"),
    ("2015_3", "Prem Ratan Dhan Payo", 164, "2015_3_Prem_Ratan_Dhan_Payo_candidate_3982350.srt"),
    ("2015_4", "Bajirao Mastani", 158, "2015_4_Bajirao_Mastani_candidate_103061.srt"),
    ("2015_5", "Tanu Weds Manu Returns", 128, "2015_5_Tanu_Weds_Manu_Returns_candidate_4447351.srt"),
]

timestamp_re = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def seconds(h, m, s, ms):
    return int(h)*3600 + int(m)*60 + int(s) + int(ms.ljust(3, "0")[:3])/1000

def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\{[^}]*\}", " ", text)
    text = re.sub(r"^\s*[-–]\s*", "", text)
    return re.sub(r"\s+", " ", text).strip()

results = []

for record_id, title, runtime_min, filename in movies:
    path = RAW / filename
    row = {
        "Record ID": record_id,
        "Movie": title,
        "File found": path.exists(),
        "Encoding": "",
        "Blocks": 0,
        "Words": 0,
        "First timestamp": "",
        "Last timestamp": "",
        "Coverage %": None,
        "Gaps >120s": 0,
        "Largest gap (s)": None,
        "Out-of-order": 0,
        "Status": "Review",
    }

    if not path.exists():
        row["Status"] = "File missing — check filename"
        results.append(row)
        continue

    raw_bytes = path.read_bytes()
    try:
        content = raw_bytes.decode("utf-8-sig")
        row["Encoding"] = "UTF-8"
    except UnicodeDecodeError:
        try:
            content = raw_bytes.decode("cp1252")
            row["Encoding"] = "CP1252 fallback"
        except UnicodeDecodeError:
            row["Encoding"] = "Decode failed"
            row["Status"] = "Reject — unreadable encoding"
            results.append(row)
            continue

    blocks = re.split(r"\n\s*\n", content.strip())
    dialogue_words = []
    starts, ends = [], []
    valid_blocks = 0

    for block in blocks:
        lines = block.strip().splitlines()
        if not lines:
            continue

        time_line = next((line for line in lines if "-->" in line), None)
        if not time_line:
            continue

        match = timestamp_re.search(time_line)
        if not match:
            continue

        vals = match.groups()
        start = seconds(*vals[:4])
        end = seconds(*vals[4:])

        if end < start:
            continue

        valid_blocks += 1
        starts.append(start)
        ends.append(end)

        time_idx = lines.index(time_line)
        text = clean_text(" ".join(lines[time_idx + 1:]))
        if text:
            dialogue_words.extend(re.findall(r"\b[\w’'-]+\b", text))

    row["Blocks"] = valid_blocks
    row["Words"] = len(dialogue_words)

    if valid_blocks:
        first_start = min(starts)
        last_end = max(ends)
        row["First timestamp"] = f"{first_start:.1f}s"
        row["Last timestamp"] = f"{last_end:.1f}s"
        row["Coverage %"] = round((last_end / (runtime_min * 60)) * 100, 1)

        gaps = [
            starts[i] - ends[i-1]
            for i in range(1, len(starts))
            if starts[i] > ends[i-1]
        ]
        large_gaps = [g for g in gaps if g > 120]
        row["Gaps >120s"] = len(large_gaps)
        row["Largest gap (s)"] = round(max(gaps), 1) if gaps else 0
        row["Out-of-order"] = sum(
            starts[i] < starts[i-1] for i in range(1, len(starts))
        )

        if valid_blocks >= 100 and len(dialogue_words) >= 1000:
            row["Status"] = "Pass initial checks — review flags"
        else:
            row["Status"] = "Reject — too little subtitle content"
    else:
        row["Status"] = "Reject — no valid timestamp blocks"

    results.append(row)

validation_df = pd.DataFrame(results)

print("BATCH 07 — INITIAL SUBTITLE VALIDATION")
display(validation_df.style.format({
    "Coverage %": "{:.1f}%",
    "Largest gap (s)": "{:.1f}"
}, na_rep="—").set_properties(**{
    "white-space": "normal",
    "text-align": "left"
}))

print("\nRaw subtitle files were read only. No manifest changes were made.")


In [ ]:

from pathlib import Path

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

print("Baahubali-related files:")
for p in sorted(RAW.glob("*Baahubali*")):
    print(p.name, "|", p.stat().st_size, "bytes")


In [ ]:

from pathlib import Path
import re

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)
path = RAW / "2015_1_Baahubali__The_Beginning_candidate_1054663.srt"

content = path.read_text(encoding="utf-8-sig")
blocks = re.split(r"\n\s*\n", content.strip())
timestamp_re = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def to_seconds(h, m, s, ms):
    return int(h)*3600 + int(m)*60 + int(s) + int(ms.ljust(3, "0")[:3])/1000

starts, ends, words = [], [], []
valid_blocks = 0

for block in blocks:
    lines = block.strip().splitlines()
    time_line = next((line for line in lines if "-->" in line), None)
    if not time_line:
        continue

    match = timestamp_re.search(time_line)
    if not match:
        continue

    vals = match.groups()
    start, end = to_seconds(*vals[:4]), to_seconds(*vals[4:])
    if end < start:
        continue

    valid_blocks += 1
    starts.append(start)
    ends.append(end)

    text = " ".join(lines[lines.index(time_line) + 1:])
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words.extend(re.findall(r"\b[\w’'-]+\b", text))

runtime_minutes = 159
last_end = max(ends) if ends else 0
gaps = [
    starts[i] - ends[i-1]
    for i in range(1, len(starts))
    if starts[i] > ends[i-1]
]
large_gaps = [g for g in gaps if g > 120]

print("Movie: Baahubali: The Beginning")
print("File found:", path.exists())
print("File size:", f"{path.stat().st_size:,} bytes")
print("Valid dialogue blocks:", valid_blocks)
print("Word count:", len(words))
print("First timestamp (seconds):", round(min(starts), 1) if starts else None)
print("Last timestamp (seconds):", round(last_end, 1) if ends else None)
print("Coverage (%):", round(last_end / (runtime_minutes * 60) * 100, 1))
print("Gaps >120 seconds:", len(large_gaps))
print("Largest gap (seconds):", round(max(gaps), 1) if gaps else 0)
print("Out-of-order timestamps:", sum(
    starts[i] < starts[i-1] for i in range(1, len(starts))
))


In [ ]:

from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
MANIFEST_PATH = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

MASTER_PATH = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MAPPING_PATH = PROJECT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"

VALIDATED.mkdir(parents=True, exist_ok=True)

# Batch 07 decisions
batch = [
    ("2014_1", "PK", 1017746, "2014_1_PK_candidate_1017746.srt",
     False, "Rejected: only 13.6% runtime coverage; likely incomplete."),
    ("2014_2", "Kick", 941291, "2014_2_Kick_candidate_941291.srt",
     True, "Accepted with quality flag: 85.3% coverage; 6 timestamp gaps over 120 seconds."),
    ("2014_3", "Happy New Year", 1007904, "2014_3_Happy_New_Year_candidate_1007904.srt",
     True, "Accepted with quality flag: 74.3% coverage; 5 timestamp gaps over 120 seconds."),
    ("2014_4", "Bang Bang!", 1000639, "2014_4_Bang_Bang_candidate_1000639.srt",
     True, "Accepted with quality flag: 67.7% coverage."),
    ("2014_5", "Singham Returns", 911878, "2014_5_Singham_Returns_candidate_911878.srt",
     True, "Accepted with quality flag: 99.7% coverage; 4 timestamp gaps over 120 seconds."),
    ("2015_1", "Baahubali: The Beginning", 1054663,
     "2015_1_Baahubali__The_Beginning_candidate_1054663.srt",
     True, "Accepted with quality flag: 63.2% coverage; first timestamp at 92.6 seconds."),
    ("2015_2", "Bajrangi Bhaijaan", 1051138, "2015_2_Bajrangi_Bhaijaan_candidate_1051138.srt",
     True, "Accepted with quality flag: 61.5% coverage; 4 timestamp gaps over 120 seconds."),
    ("2015_3", "Prem Ratan Dhan Payo", 3982350,
     "2015_3_Prem_Ratan_Dhan_Payo_candidate_3982350.srt",
     True, "Accepted with quality flag: 59.5% coverage; 1 timestamp gap over 120 seconds."),
    ("2015_4", "Bajirao Mastani", 103061, "2015_4_Bajirao_Mastani_candidate_103061.srt",
     False, "Rejected: only 34.5% runtime coverage; likely incomplete."),
    ("2015_5", "Tanu Weds Manu Returns", 4447351,
     "2015_5_Tanu_Weds_Manu_Returns_candidate_4447351.srt",
     True, "Accepted with quality flag: 114.4% coverage; 14 timestamp gaps over 120 seconds; timestamps exceed runtime reference."),
]

# Load source files and manifest
master = pd.read_csv(MASTER_PATH)
mapping = pd.read_csv(MAPPING_PATH)
manifest = pd.read_csv(MANIFEST_PATH)

required_manifest_columns = [
    "movie_record_id", "year", "rank", "movie_title", "language",
    "tmdb_id", "imdb_id", "subtitle_language", "subtitle_type",
    "subtitle_source", "source_id", "source_url", "raw_subtitle_file",
    "validated_subtitle_file", "acquisition_status", "validation_status",
    "title_verified", "year_verified", "movie_identity_verified",
    "language_verified", "dialogue_blocks", "word_count", "character_count",
    "encoding_status", "quality_status", "notes"
]

missing_cols = [c for c in required_manifest_columns if c not in manifest.columns]
if missing_cols:
    raise ValueError(f"Manifest schema differs; stopped without updating: {missing_cols}")

batch_ids = [x[0] for x in batch]
existing_ids = set(manifest["movie_record_id"].astype(str))
duplicates = sorted(set(batch_ids) & existing_ids)
if duplicates:
    raise ValueError(
        f"These IDs already exist in the manifest: {duplicates}. "
        "No manifest changes made."
    )

if master["movie_record_id"].astype(str).duplicated().any():
    raise ValueError("Duplicate IDs in master dataset; stopped.")
if mapping["movie_record_id"].astype(str).duplicated().any():
    raise ValueError("Duplicate IDs in TMDB mapping; stopped.")

# Create lookup tables
master = master.copy()
mapping = mapping.copy()
master["movie_record_id"] = master["movie_record_id"].astype(str)
mapping["movie_record_id"] = mapping["movie_record_id"].astype(str)

master_lookup = master.set_index("movie_record_id")
mapping_lookup = mapping.set_index("movie_record_id")

# Check every raw file and mapping before copying or updating manifest
for rid, title, subtitle_id, raw_name, accepted, note in batch:
    raw_path = RAW / raw_name
    if not raw_path.exists():
        raise FileNotFoundError(f"Raw file missing for {rid}: {raw_path}")
    if rid not in master_lookup.index:
        raise KeyError(f"{rid} not found in master dataset.")
    if rid not in mapping_lookup.index:
        raise KeyError(f"{rid} not found in TMDB mapping.")

# Copy accepted files; raw files remain unchanged
new_rows = []
for rid, title, subtitle_id, raw_name, accepted, note in batch:
    raw_path = RAW / raw_name
    validated_name = (
        f"{rid}_{''.join(c if c.isalnum() else '_' for c in title).strip('_')}"
        "_ENGLISH.srt"
    )
    validated_path = VALIDATED / validated_name

    if accepted:
        if validated_path.exists():
            raise FileExistsError(
                f"Validated destination already exists: {validated_path}. "
                "Stopped to avoid overwriting."
            )
        shutil.copy2(raw_path, validated_path)
        validated_value = str(validated_path)
        validation_status = "Accepted_with_quality_flags"
        quality_status = "Review_Flag"
    else:
        validated_value = ""
        validation_status = "Rejected"
        quality_status = "Rejected"

    m = master_lookup.loc[rid]
    tm = mapping_lookup.loc[rid]

    def value_from(row, names, default=""):
        for name in names:
            if name in row.index and pd.notna(row[name]):
                return row[name]
        return default

    new_rows.append({
        "movie_record_id": rid,
        "year": value_from(m, ["year"]),
        "rank": value_from(m, ["rank"]),
        "movie_title": title,
        "language": value_from(m, ["language", "dataset_language"]),
        "tmdb_id": value_from(tm, ["tmdb_id"]),
        "imdb_id": value_from(tm, ["imdb_id"]),
        "subtitle_language": "en",
        "subtitle_type": "SRT",
        "subtitle_source": "OpenSubtitles",
        "source_id": str(subtitle_id),
        "source_url": "",
        "raw_subtitle_file": str(raw_path),
        "validated_subtitle_file": validated_value,
        "acquisition_status": "Acquired",
        "validation_status": validation_status,
        "title_verified": "Yes",
        "year_verified": "Yes",
        "movie_identity_verified": "Yes",
        "language_verified": "Yes",
        "dialogue_blocks": {
            "2014_2": 982, "2014_3": 1730, "2014_4": 1863,
            "2014_5": 1045, "2015_1": 1287, "2015_2": 708,
            "2015_3": 1036, "2015_5": 296
        }.get(rid, 266 if rid == "2014_1" else 396),
        "word_count": {
            "2014_1": 2124, "2014_2": 5008, "2014_3": 8920,
            "2014_4": 12639, "2014_5": 5460, "2015_1": 8032,
            "2015_2": 3276, "2015_3": 6042, "2015_4": 2143,
            "2015_5": 2183
        }[rid],
        "character_count": "",
        "encoding_status": "UTF-8",
        "quality_status": quality_status,
        "notes": note
    })

new_df = pd.DataFrame(new_rows, columns=required_manifest_columns)

# Preserve all existing manifest columns and values
new_df = new_df.reindex(columns=manifest.columns)

# Back up current manifest before append
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = WORK / f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_batch07_{stamp}.csv"
shutil.copy2(MANIFEST_PATH, backup_path)

updated = pd.concat([manifest, new_df], ignore_index=True)
if updated["movie_record_id"].astype(str).duplicated().any():
    raise ValueError("Duplicate IDs detected after append; original manifest is untouched.")

# Write to a temporary file, then replace manifest
temp_path = MANIFEST_PATH.with_name(MANIFEST_PATH.stem + "_TEMP.csv")
updated.to_csv(temp_path, index=False)
pd.read_csv(temp_path)  # basic read-back check
temp_path.replace(MANIFEST_PATH)

# Verify outcome
check = pd.read_csv(MANIFEST_PATH)
batch_check = check[check["movie_record_id"].astype(str).isin(batch_ids)]

print("BATCH 07 FINALIZATION COMPLETE")
print("Manifest backup:", backup_path.name)
print("Manifest rows before:", len(manifest))
print("Manifest rows after:", len(check))
print("Batch 07 rows added:", len(batch_check))
print("Accepted with quality flags:", (batch_check["validation_status"] == "Accepted_with_quality_flags").sum())
print("Rejected:", (batch_check["validation_status"] == "Rejected").sum())
print("Duplicate record IDs:", int(check["movie_record_id"].astype(str).duplicated().sum()))
print("Master dataset rows:", len(master), "(read only)")
print("\nBatch 07 manifest status:")
display(batch_check[[
    "movie_record_id", "movie_title", "source_id",
    "validation_status", "quality_status", "validated_subtitle_file"
]].reset_index(drop=True))


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
MASTER_PATH = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MAPPING_PATH = PROJECT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"
MANIFEST_PATH = PROJECT / "01_data/processed/subtitle_corpus_v2/ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

master = pd.read_csv(MASTER_PATH)
mapping = pd.read_csv(MAPPING_PATH)
manifest = pd.read_csv(MANIFEST_PATH)

master["movie_record_id"] = master["movie_record_id"].astype(str)
mapping["movie_record_id"] = mapping["movie_record_id"].astype(str)
manifest["movie_record_id"] = manifest["movie_record_id"].astype(str)

# Next year after the completed 2014–2015 batch
year_movies = master[master["year"] == 2016].copy()

if len(year_movies) != 5:
    raise ValueError(
        f"Expected 5 movies for 2016, found {len(year_movies)}. "
        "Stop and inspect the master dataset."
    )

tmdb_cols = [
    c for c in [
        "movie_record_id", "tmdb_id", "tmdb_title",
        "tmdb_release_date", "tmdb_original_language", "match_status"
    ] if c in mapping.columns
]

target = year_movies.drop(
    columns=["tmdb_id"], errors="ignore"
).merge(
    mapping[tmdb_cols],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

target["already_in_manifest"] = target["movie_record_id"].isin(
    set(manifest["movie_record_id"])
)

print("2016 selection — review before searching")
display(target[[
    c for c in [
        "movie_record_id", "year", "movie_title", "tmdb_id",
        "tmdb_title", "match_status", "already_in_manifest"
    ] if c in target.columns
]].reset_index(drop=True))

if target["tmdb_id"].isna().any():
    print("WARNING: Some movies are missing TMDB IDs. Do not search until resolved.")
elif target["already_in_manifest"].any():
    print("WARNING: Some movies already appear in the manifest. Review before searching.")
else:
    print("Selection check passed. No API calls or file changes made.")


In [ ]:

import requests
import time
import pandas as pd
from pathlib import Path
from datetime import datetime
from google.colab import userdata
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
SEARCH_DIR = WORK / "search_results"
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

movies = target[[
    "movie_record_id", "year", "movie_title", "tmdb_id"
]].copy()

BASE = "https://api.opensubtitles.com/api/v1"
api_key = userdata.get("OPEN_SUBTITLES_API_KEY")
username = userdata.get("OPEN_SUBTITLES_USERNAME")
password = userdata.get("OPEN_SUBTITLES_PASSWORD")

headers = {
    "Api-Key": api_key,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch v1",
}

login = requests.post(
    f"{BASE}/login",
    headers=headers,
    json={"username": username, "password": password},
    timeout=30
)
print("Login HTTP status:", login.status_code)
login.raise_for_status()
token = login.json().get("token")
if not token:
    raise RuntimeError("No login token returned.")

auth_headers = {**headers, "Authorization": f"Bearer {token}"}
records, errors = [], []

for _, movie in movies.sort_values("movie_record_id").iterrows():
    rid = str(movie["movie_record_id"])
    title = str(movie["movie_title"])
    tmdb_id = int(movie["tmdb_id"])
    year = int(movie["year"])

    response = requests.get(
        f"{BASE}/subtitles",
        headers=auth_headers,
        params={
            "tmdb_id": tmdb_id,
            "languages": "en",
            "type": "movie"
        },
        timeout=30
    )

    if response.status_code != 200:
        errors.append({
            "movie_record_id": rid,
            "movie_title": title,
            "http_status": response.status_code
        })
        print(f"FAILED: {rid} | HTTP {response.status_code}")
        time.sleep(0.5)
        continue

    candidates = response.json().get("data", [])
    print(f"{rid} | {title}: {len(candidates)} candidate(s)")

    for candidate in candidates:
        attr = candidate.get("attributes", {})
        feature = attr.get("feature_details") or {}
        records.append({
            "movie_record_id": rid,
            "requested_title": title,
            "requested_year": year,
            "requested_tmdb_id": tmdb_id,
            "subtitle_id": attr.get("subtitle_id") or candidate.get("id"),
            "release": attr.get("release"),
            "download_count": attr.get("download_count"),
            "language": attr.get("language"),
            "hearing_impaired": attr.get("hearing_impaired"),
            "machine_translated": attr.get("machine_translated"),
            "matched_movie_title": feature.get("title"),
            "matched_movie_year": feature.get("year"),
            "matched_tmdb_id": feature.get("tmdb_id")
        })

    time.sleep(0.5)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output = SEARCH_DIR / f"batch08_2016_english_search_{stamp}.csv"
pd.DataFrame(records).to_csv(output, index=False)

print("\nSaved:", output.name)
print("Total candidates:", len(records))
print("Search errors:", len(errors))

if errors:
    display(pd.DataFrame(errors))

if records:
    df = pd.DataFrame(records)
    for col in [
        "download_count", "requested_tmdb_id",
        "matched_tmdb_id", "requested_year", "matched_movie_year"
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["tmdb_match"] = df["requested_tmdb_id"].eq(df["matched_tmdb_id"])
    df["year_match"] = df["requested_year"].eq(df["matched_movie_year"])

    exact = df[df["tmdb_match"] & df["year_match"]].copy()
    exact["download_count"] = exact["download_count"].fillna(0)
    exact = exact.sort_values(
        ["movie_record_id", "download_count"],
        ascending=[True, False]
    )

    print("\nTOP 3 EXACT-MATCH CANDIDATES PER MOVIE")
    top3 = exact.groupby("movie_record_id", group_keys=False).head(3)
    display(top3[[
        "movie_record_id", "requested_title", "subtitle_id",
        "release", "download_count", "language",
        "hearing_impaired", "machine_translated"
    ]].reset_index(drop=True))

print("\nNo subtitle downloads. Manifest and master dataset unchanged.")


In [ ]:

import requests
import time
from pathlib import Path
import pandas as pd
from google.colab import userdata
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
RAW = PROJECT / "01_data/processed/subtitle_corpus_v2/raw"
RAW.mkdir(parents=True, exist_ok=True)

selected = [
    ("2016_1", "Dangal", 1092372),
    ("2016_2", "Sultan", 1065546),
    ("2016_3", "Kabali", 1073742),
    ("2016_4", "M.S. Dhoni: The Untold Story", 1083334),
    ("2016_5", "Rustom", 1074288),
]

BASE = "https://api.opensubtitles.com/api/v1"
api_key = userdata.get("OPEN_SUBTITLES_API_KEY")
username = userdata.get("OPEN_SUBTITLES_USERNAME")
password = userdata.get("OPEN_SUBTITLES_PASSWORD")

headers = {
    "Api-Key": api_key,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch v1",
}

# Fresh login
login = requests.post(
    f"{BASE}/login",
    headers=headers,
    json={"username": username, "password": password},
    timeout=30,
)
login.raise_for_status()
token = login.json().get("token")
if not token:
    raise RuntimeError("Login succeeded but no token was returned.")

auth_headers = {**headers, "Authorization": f"Bearer {token}"}

# Verify quota immediately before downloading
qr = requests.get(
    f"{BASE}/infos/user", headers=auth_headers, timeout=30
)
qr.raise_for_status()
quota = qr.json()["data"]
remaining = int(quota["allowed_downloads"]) - int(quota["downloads_count"])
print("Downloads remaining:", remaining)

if remaining < 5:
    raise RuntimeError("Fewer than 5 downloads remain. No downloads attempted.")

results = []

for rid, title, subtitle_id in selected:
    safe_title = "".join(c if c.isalnum() else "_" for c in title).strip("_")
    destination = RAW / f"{rid}_{safe_title}_candidate_{subtitle_id}.srt"

    if destination.exists():
        print(f"STOP: destination already exists for {rid}: {destination.name}")
        raise FileExistsError("Review existing file before retrying; no overwrite.")

    try:
        dr = requests.post(
            f"{BASE}/download",
            headers=auth_headers,
            json={"file_id": subtitle_id},
            timeout=30,
        )

        if dr.status_code != 200:
            print(f"FAILED: {rid} | download API HTTP {dr.status_code}")
            results.append({
                "Record ID": rid, "Movie": title,
                "Subtitle ID": subtitle_id, "Status": "Download API failed",
                "HTTP": dr.status_code, "Bytes": 0, "Raw file": ""
            })
            time.sleep(0.5)
            continue

        download_url = dr.json().get("link")
        if not download_url:
            results.append({
                "Record ID": rid, "Movie": title,
                "Subtitle ID": subtitle_id, "Status": "No download link",
                "HTTP": dr.status_code, "Bytes": 0, "Raw file": ""
            })
            continue

        file_response = requests.get(download_url, timeout=60)
        file_response.raise_for_status()
        content = file_response.content

        if not content:
            raise ValueError("Downloaded file is empty.")

        destination.write_bytes(content)
        print(f"DOWNLOADED: {rid} | {title} | {len(content):,} bytes")

        results.append({
            "Record ID": rid, "Movie": title,
            "Subtitle ID": subtitle_id, "Status": "Downloaded",
            "HTTP": file_response.status_code,
            "Bytes": len(content), "Raw file": str(destination)
        })

    except Exception as exc:
        print(f"ERROR: {rid} | {type(exc).__name__}: {exc}")
        results.append({
            "Record ID": rid, "Movie": title,
            "Subtitle ID": subtitle_id, "Status": f"Error: {type(exc).__name__}",
            "HTTP": None, "Bytes": 0, "Raw file": ""
        })

    time.sleep(0.5)

print("\nBATCH 08 DOWNLOAD SUMMARY")
display(pd.DataFrame(results))

print("\nRaw files only. Manifest and validated folder unchanged.")


In [ ]:

from pathlib import Path
import re
import pandas as pd

# Paths
PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"

# Batch 08: record ID, movie title, expected runtime in seconds
batch08 = [
    ("2016_1", "Dangal", 161 * 60),
    ("2016_2", "Sultan", 170 * 60),
    ("2016_3", "Kabali", 153 * 60),
    ("2016_4", "M.S. Dhoni: The Untold Story", 190 * 60),
    ("2016_5", "Rustom", 148 * 60),
]

# Use the exact downloaded filenames
files = {
    "2016_1": RAW / "2016_1_Dangal_candidate_1092372.srt",
    "2016_2": RAW / "2016_2_Sultan_candidate_1065546.srt",
    "2016_3": RAW / "2016_3_Kabali_candidate_1073742.srt",
    "2016_4": RAW / "2016_4_M_S_Dhoni_The_Untold_Story_candidate_1083334.srt",
    "2016_5": RAW / "2016_5_Rustom_candidate_1074288.srt",
}

timestamp_pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def to_seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms.ljust(3, "0")) / 1000

def clean_subtitle_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\{\\[^}]*\}", " ", text)
    text = re.sub(r"\[[^\]]*\]|\([^)]*\)", " ", text)
    text = re.sub(r"&(?:amp|lt|gt|nbsp);", " ", text, flags=re.I)
    text = re.sub(r"[^A-Za-z0-9']+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

results = []

for record_id, movie, runtime in batch08:
    path = files[record_id]

    if not path.exists():
        results.append({
            "record_id": record_id,
            "movie": movie,
            "file_exists": False,
            "error": "Raw file not found",
        })
        continue

    raw_bytes = path.read_bytes()

    # Decode without changing the original file
    encoding_used = None
    decoded = None
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            decoded = raw_bytes.decode(encoding)
            encoding_used = encoding
            break
        except UnicodeDecodeError:
            continue

    if decoded is None:
        results.append({
            "record_id": record_id,
            "movie": movie,
            "file_exists": True,
            "bytes": len(raw_bytes),
            "error": "Unable to decode",
        })
        continue

    matches = list(timestamp_pattern.finditer(decoded))
    starts, ends = [], []
    dialogue_blocks = 0
    word_count = 0

    for match in matches:
        start = to_seconds(*match.groups()[:4])
        end = to_seconds(*match.groups()[4:])
        starts.append(start)
        ends.append(end)

        if end >= start:
            dialogue_blocks += 1
            # Text between this timestamp and the next timestamp
            segment_end = (
                matches[matches.index(match) + 1].start()
                if matches.index(match) + 1 < len(matches)
                else len(decoded)
            )
            segment = decoded[match.end():segment_end]
            text = clean_subtitle_text(segment)
            word_count += len(text.split())

    ordered_starts = sorted(starts)
    gaps = [
        ordered_starts[i] - ends[starts.index(ordered_starts[i - 1])]
        for i in range(1, len(ordered_starts))
        if ordered_starts[i] - ends[starts.index(ordered_starts[i - 1])] > 120
    ]

    first_time = min(starts) if starts else None
    last_time = max(ends) if ends else None
    coverage_pct = (
        (last_time - first_time) / runtime * 100
        if first_time is not None and last_time is not None and runtime
        else None
    )

    results.append({
        "record_id": record_id,
        "movie": movie,
        "file_exists": True,
        "bytes": len(raw_bytes),
        "encoding": encoding_used,
        "timestamp_count": len(matches),
        "dialogue_blocks": dialogue_blocks,
        "word_count": word_count,
        "first_timestamp_sec": round(first_time, 1) if first_time is not None else None,
        "last_timestamp_sec": round(last_time, 1) if last_time is not None else None,
        "provisional_coverage_pct": round(coverage_pct, 1) if coverage_pct is not None else None,
        "gaps_over_120_sec": len(gaps),
        "largest_gap_sec": round(max(gaps), 1) if gaps else 0,
        "out_of_order_timestamps": sum(
            starts[i] < starts[i - 1] for i in range(1, len(starts))
        ),
        "error": None,
    })

validation_df = pd.DataFrame(results)

print("=" * 90)
print("BATCH 08 — RAW SUBTITLE VALIDATION")
print("Read-only check: no files or manifest entries have been modified.")
print("=" * 90)

display(validation_df)


In [ ]:

from pathlib import Path

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

print("Files matching M.S. Dhoni / subtitle ID 1083334:")
matches = [
    p for p in RAW.iterdir()
    if p.is_file()
    and (
        "1083334" in p.name
        or ("2016_4" in p.name and "Dhoni" in p.name)
    )
]

if matches:
    for p in matches:
        print(p.name, "|", p.stat().st_size, "bytes")
else:
    print("No matching file found.")

print("\nAll raw filenames beginning with 2016_4:")
for p in sorted(RAW.glob("2016_4*")):
    print(p.name, "|", p.stat().st_size, "bytes")


In [ ]:

from pathlib import Path
import re
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
RAW = PROJECT / "01_data/processed/subtitle_corpus_v2/raw"

batch08 = [
    ("2016_1", "Dangal", 161 * 60, "2016_1_Dangal_candidate_1092372.srt"),
    ("2016_2", "Sultan", 170 * 60, "2016_2_Sultan_candidate_1065546.srt"),
    ("2016_3", "Kabali", 153 * 60, "2016_3_Kabali_candidate_1073742.srt"),
    ("2016_4", "M.S. Dhoni: The Untold Story", 190 * 60,
     "2016_4_M_S__Dhoni__The_Untold_Story_candidate_1083334.srt"),
    ("2016_5", "Rustom", 148 * 60, "2016_5_Rustom_candidate_1074288.srt"),
]

timestamp_pattern = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def seconds(parts):
    h, m, s, ms = parts
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms.ljust(3, "0")) / 1000

def extract_dialogue(segment):
    segment = re.sub(r"<[^>]*>", " ", segment)
    segment = re.sub(r"\{[^}]*\}", " ", segment)
    segment = re.sub(r"&(?:amp|lt|gt|nbsp);", " ", segment, flags=re.I)
    segment = re.sub(r"^\s*[\d\s]+\s*$", " ", segment)
    segment = re.sub(r"[^A-Za-z0-9']+", " ", segment)
    return re.sub(r"\s+", " ", segment).strip()

results = []
samples = {}

for record_id, movie, runtime, filename in batch08:
    path = RAW / filename

    if not path.exists():
        results.append({
            "record_id": record_id, "movie": movie,
            "file_exists": False, "error": f"Missing: {filename}"
        })
        continue

    raw = path.read_bytes()
    text = raw.decode("utf-8-sig", errors="replace")
    matches = list(timestamp_pattern.finditer(text))

    starts, ends, words = [], [], []
    for i, match in enumerate(matches):
        start = seconds(match.groups()[:4])
        end = seconds(match.groups()[4:])
        starts.append(start)
        ends.append(end)

        segment_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        dialogue = extract_dialogue(text[match.end():segment_end])
        if dialogue:
            words.extend(dialogue.split())

    ordered = sorted(zip(starts, ends), key=lambda x: x[0])
    gaps = [
        ordered[i][0] - ordered[i - 1][1]
        for i in range(1, len(ordered))
        if ordered[i][0] - ordered[i - 1][1] > 120
    ]

    first_time = min(starts) if starts else None
    last_time = max(ends) if ends else None
    coverage = (
        (last_time - first_time) / runtime * 100
        if first_time is not None and last_time is not None else None
    )

    results.append({
        "record_id": record_id,
        "movie": movie,
        "file_exists": True,
        "bytes": len(raw),
        "timestamp_count": len(matches),
        "dialogue_blocks": len(matches),
        "word_count": len(words),
        "first_sec": round(first_time, 1) if first_time is not None else None,
        "last_sec": round(last_time, 1) if last_time is not None else None,
        "provisional_coverage_pct": round(coverage, 1) if coverage is not None else None,
        "gaps_over_120_sec": len(gaps),
        "largest_gap_sec": round(max(gaps), 1) if gaps else 0,
        "out_of_order_timestamps": sum(
            starts[i] < starts[i - 1] for i in range(1, len(starts))
        ),
        "error": None,
    })

    if movie in ("Sultan", "Rustom"):
        samples[movie] = [
            (m.group(0), extract_dialogue(
                text[m.end():matches[i + 1].start()]
                if i + 1 < len(matches) else text[m.end():]
            ))
            for i, m in enumerate(matches)
        ]

print("BATCH 08 — VALIDATION RESULTS")
display(pd.DataFrame(results))

print("\nSAMPLE DIALOGUE — SULTAN AND RUSTOM")
for movie, items in samples.items():
    print(f"\n--- {movie} ---")
    shown = 0
    for timestamp, dialogue in items:
        if dialogue:
            print(timestamp)
            print(dialogue[:250])
            shown += 1
        if shown == 5:
            break


In [ ]:

from pathlib import Path

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

batch08_files = [
    "2016_1_Dangal_candidate_1092372.srt",
    "2016_2_Sultan_candidate_1065546.srt",
    "2016_3_Kabali_candidate_1073742.srt",
    "2016_4_M_S__Dhoni__The_Untold_Story_candidate_1083334.srt",
    "2016_5_Rustom_candidate_1074288.srt",
]

print("BATCH 08 FILES SELECTED FOR DELETION")
print("=" * 65)

existing_files = []
for filename in batch08_files:
    path = RAW / filename
    if path.is_file():
        print(f"FOUND  | {filename} | {path.stat().st_size:,} bytes")
        existing_files.append(path)
    else:
        print(f"MISSING | {filename}")

print(f"\nFound: {len(existing_files)} of {len(batch08_files)} files")
print("Only these exact files will be targeted.")
print("Manifest, validated folder, master dataset, and other files remain untouched.")

confirmation = input(
    "\nType DELETE BATCH 08 to permanently delete these raw files: "
).strip()

if confirmation == "DELETE BATCH 08":
    deleted = []
    for path in existing_files:
        path.unlink()
        deleted.append(path.name)

    print(f"\nDeleted {len(deleted)} files:")
    for filename in deleted:
        print("DELETED:", filename)

    remaining = [p.name for p in existing_files if p.exists()]
    print("Verification:", "PASS — all targeted files are absent"
          if not remaining else f"CHECK — still present: {remaining}")
else:
    print("\nDeletion cancelled. No files were deleted.")


In [ ]:

from pathlib import Path
from google.colab import drive

# Mount Google Drive if not already mounted
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

# Project paths
PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")

MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
TMDB_MAPPING = PROJECT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"

WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
SEARCH_RESULTS = WORK / "search_results"

MANIFEST = (
    WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

# Verify important paths
print("\nPROJECT INITIALIZATION")
print("=" * 60)

for label, path in [
    ("Project directory", PROJECT),
    ("Master dataset", MASTER),
    ("TMDB mapping", TMDB_MAPPING),
    ("Subtitle workspace", WORK),
    ("Raw subtitles", RAW),
    ("Validated subtitles", VALIDATED),
    ("Search results", SEARCH_RESULTS),
    ("Subtitle manifest", MANIFEST),
]:
    print(f"{label}: {'EXISTS' if path.exists() else 'MISSING'}")
    print(f"  {path}")

# Do not create missing directories or files automatically.


In [ ]:

import requests
import pandas as pd
from pathlib import Path
from datetime import datetime
from google.colab import userdata

# ============================================================
# NOTEBOOK 08 — PREFLIGHT + 2016 REPLACEMENT SEARCH
# READ-ONLY SEARCH: NO SUBTITLE DOWNLOADS
# ============================================================

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
SEARCH_RESULTS = WORK / "search_results"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
TMDB_MAPPING = PROJECT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"

print("=" * 65)
print("1. PROJECT PREFLIGHT")
print("=" * 65)

required_paths = [
    PROJECT, WORK, RAW, VALIDATED, SEARCH_RESULTS,
    MANIFEST, MASTER, TMDB_MAPPING
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing project paths:\n" + "\n".join(missing))

manifest_df = pd.read_csv(MANIFEST)
master_df = pd.read_csv(MASTER)
mapping_df = pd.read_csv(TMDB_MAPPING)

print("Master shape:", master_df.shape)
print("TMDB mapping shape:", mapping_df.shape)
print("Manifest shape:", manifest_df.shape)
print("Manifest columns:", len(manifest_df.columns))

if manifest_df["movie_record_id"].duplicated().any():
    raise RuntimeError("Duplicate manifest IDs detected. Search stopped.")

batch_ids = ["2016_1", "2016_2", "2016_3", "2016_4", "2016_5"]
print("Batch 08 records already in manifest:",
      manifest_df.loc[
          manifest_df["movie_record_id"].astype(str).isin(batch_ids),
          "movie_record_id"
      ].tolist())

old_files = [
    "2016_1_Dangal_candidate_1092372.srt",
    "2016_2_Sultan_candidate_1065546.srt",
    "2016_3_Kabali_candidate_1073742.srt",
    "2016_4_M_S__Dhoni__The_Untold_Story_candidate_1083334.srt",
    "2016_5_Rustom_candidate_1074288.srt",
]
print("Deleted Batch 08 files still present:",
      [f for f in old_files if (RAW / f).exists()])

# ============================================================
# 2. SECRETS + LOGIN
# ============================================================

print("\n" + "=" * 65)
print("2. OPEN SUBTITLES LOGIN")
print("=" * 65)

try:
    api_key = userdata.get("OPEN_SUBTITLES_API_KEY").strip()
    username = userdata.get("OPEN_SUBTITLES_USERNAME").strip()
    password = userdata.get("OPEN_SUBTITLES_PASSWORD").strip()
except Exception as exc:
    raise RuntimeError(
        "Could not access Colab Secrets. Check the Secrets panel "
        "and notebook access permissions."
    ) from exc

if not all([api_key, username, password]):
    raise RuntimeError(
        "API key, username, or password is missing. "
        "No search or download was performed."
    )

API = "https://api.opensubtitles.com/api/v1"
session = requests.Session()
session.headers.update({
    "Api-Key": api_key,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
})

login = session.post(
    f"{API}/login",
    json={"username": username, "password": password},
    timeout=30
)
print("Login HTTP:", login.status_code)

if login.status_code != 200:
    print(login.text[:400])
    raise RuntimeError("Login failed; stopping safely.")

token = login.json().get("token")
if not token:
    raise RuntimeError("No token returned by login.")

session.headers.update({"Authorization": f"Bearer {token}"})
print("Fresh login successful. Token not displayed.")

# ============================================================
# 3. QUOTA CHECK
# ============================================================

print("\n" + "=" * 65)
print("3. DOWNLOAD QUOTA")
print("=" * 65)

quota_response = session.get(f"{API}/infos/user", timeout=30)
print("Quota HTTP:", quota_response.status_code)

if quota_response.status_code != 200:
    print(quota_response.text[:400])
    raise RuntimeError("Could not verify quota. Search stopped.")

quota_json = quota_response.json()
quota_data = quota_json.get("data", quota_json)

for key, value in quota_data.items():
    if any(word in key.lower() for word in (
        "download", "remaining", "level", "vip"
    )):
        print(f"{key}: {value}")

# ============================================================
# 4. SEARCH REPLACEMENT CANDIDATES
# ============================================================

movies = [
    {"record_id": "2016_1", "title": "Dangal", "year": 2016, "tmdb_id": 360814},
    {"record_id": "2016_2", "title": "Sultan", "year": 2016, "tmdb_id": 376812},
    {"record_id": "2016_3", "title": "Kabali", "year": 2016, "tmdb_id": 366080},
    {"record_id": "2016_4", "title": "M.S. Dhoni: The Untold Story",
     "year": 2016, "tmdb_id": 388333},
    {"record_id": "2016_5", "title": "Rustom", "year": 2016, "tmdb_id": 392572},
]

print("\n" + "=" * 65)
print("4. SEARCHING — NO DOWNLOADS")
print("=" * 65)

candidates = []
errors = []

for movie in movies:
    params = {
        "languages": "en",
        "year": movie["year"],
        "tmdb_id": movie["tmdb_id"],
        "query": movie["title"],
    }

    try:
        response = session.get(
            f"{API}/subtitles", params=params, timeout=30
        )
        print(
            f'{movie["record_id"]} | {movie["title"]} | '
            f'HTTP {response.status_code}'
        )

        if response.status_code != 200:
            errors.append({
                "record_id": movie["record_id"],
                "error": f"HTTP {response.status_code}: {response.text[:250]}"
            })
            continue

        items = response.json().get("data", [])
        print("  Results:", len(items))

        for item in items:
            attr = item.get("attributes") or {}
            feature = attr.get("feature_details") or {}
            file_list = attr.get("files") or []
            file_info = file_list[0] if file_list else {}

            candidates.append({
                "record_id": movie["record_id"],
                "movie_title": movie["title"],
                "expected_tmdb_id": movie["tmdb_id"],
                "subtitle_id": attr.get("subtitle_id"),
                "language": attr.get("language"),
                "release": attr.get("release"),
                "file_name": file_info.get("file_name"),
                "file_id": file_info.get("file_id"),
                "download_count": attr.get("download_count"),
                "hearing_impaired": attr.get("hearing_impaired"),
                "machine_translated": attr.get("machine_translated"),
                "ai_translated": attr.get("ai_translated"),
                "foreign_parts_only": attr.get("foreign_parts_only"),
                "trusted_uploader": attr.get("from_trusted"),
                "candidate_title": feature.get("title"),
                "candidate_movie_name": feature.get("movie_name"),
                "candidate_year": feature.get("year"),
                "candidate_tmdb_id": feature.get("tmdb_id"),
                "candidate_imdb_id": feature.get("imdb_id"),
            })

    except Exception as exc:
        errors.append({
            "record_id": movie["record_id"],
            "error": str(exc)
        })

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_path = SEARCH_RESULTS / f"batch08_2016_replacement_search_{stamp}.csv"
errors_path = SEARCH_RESULTS / f"batch08_2016_search_errors_{stamp}.csv"

results_df = pd.DataFrame(candidates)
results_df.to_csv(results_path, index=False)
pd.DataFrame(errors).to_csv(errors_path, index=False)

print("\n" + "=" * 65)
print("5. SEARCH SUMMARY")
print("=" * 65)
print("Total candidates:", len(results_df))
print("Search errors:", len(errors))
print("Saved results:", results_path)
if errors:
    print("Saved errors:", errors_path)

if not results_df.empty:
    cols = [
        "record_id", "movie_title", "subtitle_id", "language",
        "release", "download_count", "machine_translated",
        "ai_translated", "candidate_year", "candidate_tmdb_id",
        "file_id"
    ]
    display(
        results_df[cols].sort_values(
            ["record_id", "download_count"],
            ascending=[True, False],
            na_position="last"
        )
    )

print("\nCompleted search only. No downloads were attempted.")
print("Manifest, master dataset, and raw/validated files were not modified.")


In [ ]:

import pandas as pd
from pathlib import Path
from IPython.display import display

SEARCH_RESULTS = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/search_results"
)

files = [
    SEARCH_RESULTS / "batch08_2016_replacement_search_20261027_140052.csv",
    SEARCH_RESULTS / "batch08_broader_search_20261027_140255.csv",
]

frames = []
for path in files:
    if path.exists():
        df = pd.read_csv(path)
        frames.append(df)

if not frames:
    raise FileNotFoundError("No saved Batch 08 search results were found.")

all_df = pd.concat(frames, ignore_index=True, sort=False)

# Normalize subtitle ID for deduplication across search variants
all_df["subtitle_id"] = all_df["subtitle_id"].astype("string")
all_df = all_df.drop_duplicates(
    subset=["record_id", "subtitle_id"]
).copy()

# Exclude the previously identified trailer candidate
all_df = all_df[
    ~(
        (all_df["record_id"] == "2016_5")
        & (all_df["subtitle_id"] == "1074016")
    )
].copy()

# Use available metadata to rank candidates, not to claim language is verified
for col in ["candidate_year", "candidate_tmdb_id", "expected_tmdb_id",
            "download_count", "machine_translated", "ai_translated"]:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors="coerce")

all_df["exact_year"] = all_df["candidate_year"].eq(2016)
all_df["exact_tmdb"] = all_df["candidate_tmdb_id"].eq(
    all_df["expected_tmdb_id"]
) if "expected_tmdb_id" in all_df.columns else False

all_df["translation_flag"] = (
    all_df.get("machine_translated", False).fillna(False).astype(bool)
    | all_df.get("ai_translated", False).fillna(False).astype(bool)
)

all_df["review_priority"] = (
    all_df["exact_tmdb"].astype(int) * 4
    + all_df["exact_year"].astype(int) * 2
    + (~all_df["translation_flag"]).astype(int)
)

shortlist = (
    all_df.sort_values(
        ["record_id", "review_priority", "download_count"],
        ascending=[True, False, False],
        na_position="last"
    )
    .groupby("record_id", dropna=False)
    .head(5)
)

cols = [
    c for c in [
        "record_id", "movie_title", "target_movie", "subtitle_id",
        "language", "release", "download_count", "upload_date",
        "machine_translated", "ai_translated",
        "candidate_year", "candidate_tmdb_id", "file_id",
        "review_priority"
    ] if c in shortlist.columns
]

print("Unique candidates after deduplication:", len(all_df))
print("Top five candidates per movie for review:")
display(shortlist[cols].sort_values(["record_id", "review_priority"],
                                    ascending=[True, False]))
print("\nNo downloads performed. This ranking does not verify actual subtitle language.")


In [ ]:

from pathlib import Path
import requests
import re
from google.colab import userdata

# Paths
PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
RAW = PROJECT / "01_data/processed/subtitle_corpus_v2/raw"
API = "https://api.opensubtitles.com/api/v1"

# Selected candidate: Dangal
SUBTITLE_ID = 1092372 + (1093888 - 1092372)  # 1093888
FILE_ID = 1183583
OUTPUT = RAW / "2016_1_Dangal_candidate_1093888.srt"

if not RAW.exists():
    raise FileNotFoundError(f"Raw folder missing: {RAW}")

if OUTPUT.exists():
    raise FileExistsError(
        f"{OUTPUT.name} already exists. Stopping to avoid overwriting."
    )

# Rebuild authenticated session using Colab Secrets
api_key = userdata.get("OPEN_SUBTITLES_API_KEY").strip()
username = userdata.get("OPEN_SUBTITLES_USERNAME").strip()
password = userdata.get("OPEN_SUBTITLES_PASSWORD").strip()

session = requests.Session()
session.headers.update({
    "Api-Key": api_key,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
})

login = session.post(
    f"{API}/login",
    json={"username": username, "password": password},
    timeout=30,
)
print("Login HTTP:", login.status_code)

if login.status_code != 200:
    raise RuntimeError("Login failed. No download performed.")

token = login.json().get("token")
if not token:
    raise RuntimeError("No authentication token returned.")

session.headers.update({"Authorization": f"Bearer {token}"})

# Verify quota immediately before downloading
quota = session.get(f"{API}/infos/user", timeout=30)
print("Quota HTTP:", quota.status_code)

if quota.status_code != 200:
    raise RuntimeError("Could not verify quota. No download performed.")

q = quota.json()
q = q.get("data", q)

remaining = q.get("remaining_downloads")
if remaining is None and q.get("allowed_downloads") is not None:
    remaining = q["allowed_downloads"] - q.get("downloads_count", 0)

print("Remaining downloads:", remaining)

if remaining is None or remaining < 1:
    raise RuntimeError("No confirmed download credit available.")

# Request subtitle download link
response = session.post(
    f"{API}/download",
    json={"file_id": FILE_ID},
    timeout=30,
)
print("Download API HTTP:", response.status_code)

if response.status_code != 200:
    print(response.text[:400])
    raise RuntimeError("Download request failed.")

link = response.json().get("link")
if not link:
    raise RuntimeError("No download link returned.")

file_response = requests.get(link, timeout=60)
print("Subtitle file HTTP:", file_response.status_code)

if file_response.status_code != 200 or not file_response.content:
    raise RuntimeError("Subtitle file retrieval failed.")

content = file_response.content
OUTPUT.write_bytes(content)

print(f"\nSaved: {OUTPUT.name}")
print(f"Size: {len(content):,} bytes")

# Inspect the real text immediately
text = content.decode("utf-8-sig", errors="replace")
lines = text.splitlines()

print("\nFIRST 20 NON-EMPTY LINES")
shown = 0
for line in lines:
    if line.strip():
        print(repr(line))
        shown += 1
    if shown >= 20:
        break

sample = " ".join(
    line for line in lines
    if line.strip()
    and "-->" not in line
    and not line.strip().isdigit()
)

print("\nCharacter-script indicators:")
print({
    "Latin": len(re.findall(r"[A-Za-z]", sample)),
    "Chinese": len(re.findall(r"[\u3400-\u9fff]", sample)),
    "Arabic": len(re.findall(r"[\u0600-\u06ff]", sample)),
    "Cyrillic": len(re.findall(r"[\u0400-\u04ff]", sample)),
    "Devanagari": len(re.findall(r"[\u0900-\u097f]", sample)),
})

print("\nManifest and validated folder remain unchanged.")


In [ ]:

from pathlib import Path
import re
import pandas as pd

path = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw/"
    "2016_1_Dangal_candidate_1093888.srt"
)

if not path.exists():
    raise FileNotFoundError(path)

text = path.read_text(encoding="utf-8-sig", errors="replace")
blocks = re.split(r"\n\s*\n", text.strip())

timestamp_re = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def to_sec(g):
    h, m, s, ms = g
    return int(h)*3600 + int(m)*60 + int(s) + int(ms.ljust(3, "0"))/1000

valid = []
dialogue_text = []
invalid_blocks = 0

for block in blocks:
    match = timestamp_re.search(block)
    if not match:
        if block.strip():
            invalid_blocks += 1
        continue

    lines = block[match.end():].strip().splitlines()
    dialogue = " ".join(lines)
    dialogue = re.sub(r"<[^>]*>", " ", dialogue)
    dialogue = re.sub(r"\{[^}]*\}", " ", dialogue)
    dialogue = re.sub(r"\s+", " ", dialogue).strip()

    start = to_sec(match.groups()[:4])
    end = to_sec(match.groups()[4:])

    if end < start:
        invalid_blocks += 1
        continue

    valid.append((start, end, dialogue))
    if dialogue:
        dialogue_text.append(dialogue)

valid.sort(key=lambda x: x[0])
all_dialogue = " ".join(dialogue_text)
words = re.findall(r"\b[\w']+\b", all_dialogue)

starts = [x[0] for x in valid]
ends = [x[1] for x in valid]
gaps = [
    starts[i] - ends[i-1]
    for i in range(1, len(valid))
    if starts[i] - ends[i-1] > 120
]

runtime_seconds = 161 * 60  # provisional runtime reference
first_sec = min(starts) if starts else None
last_sec = max(ends) if ends else None
coverage = (
    (last_sec - first_sec) / runtime_seconds * 100
    if first_sec is not None and last_sec is not None else None
)

latin = len(re.findall(r"[A-Za-z]", all_dialogue))
cjk = len(re.findall(r"[\u3400-\u9fff]", all_dialogue))
arabic = len(re.findall(r"[\u0600-\u06ff]", all_dialogue))
cyrillic = len(re.findall(r"[\u0400-\u04ff]", all_dialogue))
devanagari = len(re.findall(r"[\u0900-\u097f]", all_dialogue))
letters = sum([latin, cjk, arabic, cyrillic, devanagari])
latin_pct = latin / letters * 100 if letters else 0

summary = pd.DataFrame([{
    "file_bytes": path.stat().st_size,
    "subtitle_blocks": len(valid),
    "blocks_with_dialogue": sum(bool(x[2]) for x in valid),
    "word_count": len(words),
    "first_timestamp_sec": round(first_sec, 1) if first_sec is not None else None,
    "last_timestamp_sec": round(last_sec, 1) if last_sec is not None else None,
    "provisional_coverage_pct": round(coverage, 1) if coverage is not None else None,
    "gaps_over_120_sec": len(gaps),
    "largest_gap_sec": round(max(gaps), 1) if gaps else 0,
    "latin_letter_pct": round(latin_pct, 2),
    "cjk_chars": cjk,
    "arabic_chars": arabic,
    "cyrillic_chars": cyrillic,
    "devanagari_chars": devanagari,
    "invalid_or_unparsed_blocks": invalid_blocks,
}])

display(summary)

print("\nFirst 8 parsed dialogue entries:")
for start, end, dialogue in valid[:8]:
    print(f"{start:.1f}s–{end:.1f}s | {dialogue[:180]}")

print("\nRead-only validation complete. Manifest unchanged.")


In [ ]:

from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

record_id = "2016_1"
raw_file = RAW / "2016_1_Dangal_candidate_1093888.srt"
validated_file = VALIDATED / "2016_1_DANGAL_ENGLISH.srt"

if not raw_file.exists():
    raise FileNotFoundError(f"Raw subtitle missing: {raw_file}")

manifest_df = pd.read_csv(MANIFEST)

if record_id in manifest_df["movie_record_id"].astype(str).values:
    raise ValueError(
        f"{record_id} already exists in the manifest. "
        "No changes made; inspect the existing record first."
    )

if validated_file.exists():
    raise FileExistsError(
        f"Validated file already exists: {validated_file}. "
        "No changes made."
    )

expected_columns = list(manifest_df.columns)

# Create a backup before modifying the manifest
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = WORK / f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_dangal_{stamp}.csv"
shutil.copy2(MANIFEST, backup)

# Copy raw file to validated; preserve the raw file unchanged
shutil.copy2(raw_file, validated_file)

# Append one reviewed record using the existing manifest schema
new_record = {column: pd.NA for column in expected_columns}
new_record.update({
    "movie_record_id": "2016_1",
    "year": 2016,
    "rank": 1,
    "movie_title": "Dangal",
    "language": "Hindi",
    "tmdb_id": 360814,
    "subtitle_language": "English",
    "subtitle_type": "full subtitles",
    "subtitle_source": "OpenSubtitles",
    "source_id": "1093888",
    "source_url": "https://www.opensubtitles.com/",
    "raw_subtitle_file": str(raw_file),
    "validated_subtitle_file": str(validated_file),
    "acquisition_status": "downloaded",
    "validation_status": "accepted_with_quality_flag",
    "title_verified": True,
    "year_verified": True,
    "movie_identity_verified": True,
    "language_verified": True,
    "dialogue_blocks": 1608,
    "word_count": 8970,
    "character_count": 106916,
    "encoding_status": "UTF-8 with BOM",
    "quality_status": "quality_flag",
    "notes": (
        "English dialogue confirmed by text inspection; 1608 blocks, "
        "8970 words, 91.6% provisional runtime coverage. "
        "Four gaps over 120 seconds; largest 187.1 seconds. "
        "Coverage uses an approximate runtime reference."
    ),
})

updated = pd.concat(
    [manifest_df, pd.DataFrame([new_record])],
    ignore_index=True
)

# Preserve column order and verify before saving
if list(updated.columns) != expected_columns:
    raise RuntimeError("Manifest schema changed unexpectedly.")

updated.to_csv(MANIFEST, index=False)

# Verify saved result
check = pd.read_csv(MANIFEST)
print("Backup:", backup)
print("Validated file exists:", validated_file.exists())
print("Manifest rows before:", len(manifest_df))
print("Manifest rows after:", len(check))
print("Duplicate IDs:", check["movie_record_id"].duplicated().sum())
print("\nDangal manifest record:")
display(check[check["movie_record_id"].astype(str) == record_id])


In [ ]:
# ============================================================
# NOTEBOOK 08 — BATCH 08 CANDIDATE REVIEW
# Movies: Sultan, Kabali, M.S. Dhoni, Rustom (2016)
# Read-only: no downloads and no manifest changes
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
SEARCH_RESULTS = WORK / "search_results"

search_files = [
    SEARCH_RESULTS / "batch08_2016_replacement_search_20261027_140052.csv",
    SEARCH_RESULTS / "batch08_broader_search_20261027_140255.csv",
]

# Exclude Dangal because it has already been acquired and validated.
target_movies = [
    "Sultan",
    "Kabali",
    "M.S. Dhoni: The Untold Story",
    "Rustom",
]

# ------------------------------------------------------------
# 1. Load existing search results
# ------------------------------------------------------------
frames = []

for file_path in search_files:
    if file_path.exists():
        df = pd.read_csv(file_path)
        df["search_result_file"] = file_path.name
        frames.append(df)
        print(f"Loaded: {file_path.name} | Rows: {len(df)}")
    else:
        print(f"NOT FOUND: {file_path}")

if not frames:
    raise FileNotFoundError(
        "Neither saved search-result CSV was found. "
        "Check the search_results directory."
    )

candidates = pd.concat(frames, ignore_index=True, sort=False)

print("\nAvailable columns:")
print(candidates.columns.tolist())

# ------------------------------------------------------------
# 2. Identify columns flexibly
# ------------------------------------------------------------
def find_col(df, possible_names):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for name in possible_names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None

movie_col = find_col(candidates, [
    "movie_title", "title", "query_title", "requested_title",
    "dataset_title", "search_title"
])
subtitle_id_col = find_col(candidates, [
    "subtitle_id", "id", "opensubtitles_id"
])
file_id_col = find_col(candidates, [
    "file_id", "subtitle_file_id"
])
language_col = find_col(candidates, [
    "language", "subtitle_language", "lang"
])
year_col = find_col(candidates, [
    "year", "release_year"
])
tmdb_col = find_col(candidates, [
    "tmdb_id", "movie_tmdb_id"
])
downloads_col = find_col(candidates, [
    "download_count", "downloads", "download"
])
ai_col = find_col(candidates, [
    "ai_translated", "machine_translated", "machine_translation"
])
feature_col = find_col(candidates, [
    "feature", "subtitle_type", "type"
])
url_col = find_col(candidates, [
    "url", "subtitle_url", "page_url"
])

print("\nDetected columns:")
for label, col in [
    ("Movie", movie_col),
    ("Subtitle ID", subtitle_id_col),
    ("File ID", file_id_col),
    ("Language", language_col),
    ("Year", year_col),
    ("TMDB ID", tmdb_col),
    ("Downloads", downloads_col),
    ("AI translated", ai_col),
    ("Type", feature_col),
    ("URL", url_col),
]:
    print(f"{label}: {col}")

if subtitle_id_col is None or file_id_col is None:
    raise ValueError(
        "Could not identify subtitle ID and file ID columns. "
        "Review the printed column names and adjust the candidate column lists."
    )

# ------------------------------------------------------------
# 3. Match the four requested movies
# ------------------------------------------------------------
if movie_col is None:
    raise ValueError(
        "Could not identify the movie-title column. "
        "Review the printed column names before proceeding."
    )

def normalize_title(value):
    return (
        str(value).lower()
        .replace("&", "and")
        .replace(":", " ")
        .replace(".", " ")
        .replace("-", " ")
        .replace("_", " ")
        .strip()
    )

def identify_movie(value):
    title = normalize_title(value)

    if "dangal" in title:
        return "Dangal"
    if "sultan" in title:
        return "Sultan"
    if "kabali" in title:
        return "Kabali"
    if "dhoni" in title or "ms matched" in title:
        return "M.S. Dhoni: The Untold Story"
    if "rustom" in title or "rustam" in title:
        return "Rustom"
    return None

candidates["matched_movie"] = candidates[movie_col].apply(identify_movie)

review = candidates[
    candidates["matched_movie"].isin(target_movies)
].copy()

# ------------------------------------------------------------
# 4. Exclude known unsuitable candidates and duplicates
# ------------------------------------------------------------
# Previously inspected and found to contain the wrong language:
known_wrong_language = {
    "Sultan": {"1065546"},
    "Kabali": {"1073742"},
    "M.S. Dhoni: The Untold Story": {"1083334"},
    "Rustom": {"1074288"},
}

def is_known_wrong(row):
    movie = row["matched_movie"]
    subtitle_id = str(row[subtitle_id_col]).split(".")[0].strip()
    return subtitle_id in known_wrong_language.get(movie, set())

review["previously_wrong_language"] = review.apply(is_known_wrong, axis=1)

# Exclude known mismatches from the preferred shortlist, but retain them
# in the complete review table so the exclusion is transparent.
review["exclude_from_shortlist"] = review["previously_wrong_language"]

# Remove duplicate records for the same movie and subtitle ID.
review = review.drop_duplicates(
    subset=["matched_movie", subtitle_id_col],
    keep="first"
)

# Exclude official trailers if a type/feature field identifies them.
if feature_col:
    trailer_mask = review[feature_col].astype(str).str.contains(
        "trailer", case=False, na=False
    )
    review["official_trailer_flag"] = trailer_mask
    review.loc[trailer_mask, "exclude_from_shortlist"] = True
else:
    review["official_trailer_flag"] = False

# ------------------------------------------------------------
# 5. Create a readable candidate review table
# ------------------------------------------------------------
display_columns = [
    "matched_movie",
    subtitle_id_col,
    file_id_col,
]

for col in [
    language_col, year_col, tmdb_col, downloads_col,
    ai_col, feature_col, url_col
]:
    if col and col not in display_columns:
        display_columns.append(col)

display_columns += [
    "previously_wrong_language",
    "official_trailer_flag",
    "exclude_from_shortlist",
]

display_columns = [c for c in display_columns if c in review.columns]

review = review.sort_values(
    ["matched_movie", "exclude_from_shortlist", subtitle_id_col],
    kind="stable"
)

print("\n" + "=" * 90)
print("BATCH 08 — CANDIDATE REVIEW")
print("=" * 90)
print(f"Unique candidate records: {len(review)}")
print("\nCandidates by movie:")
print(review["matched_movie"].value_counts().reindex(target_movies, fill_value=0))

display(review[display_columns].reset_index(drop=True))

# ------------------------------------------------------------
# 6. Preferred shortlist: candidate metadata only
# ------------------------------------------------------------
shortlist = review[~review["exclude_from_shortlist"]].copy()

print("\n" + "=" * 90)
print("SHORTLIST FOR MANUAL REVIEW — NOT DOWNLOADED")
print("=" * 90)

for movie in target_movies:
    subset = shortlist[shortlist["matched_movie"] == movie]
    print(f"\n{movie}: {len(subset)} candidate(s)")
    if subset.empty:
        print("  No eligible candidate in the saved search results.")
    else:
        display(subset[display_columns].head(5).reset_index(drop=True))

print(
    "\nIMPORTANT: Search metadata does not prove subtitle language. "
    "Review the candidates first; do not download until a candidate "
    "has been selected for each movie."
)

In [ ]:
# ============================================================
# NOTEBOOK 08 — REFRESH OPENSUBTITLES SESSION
# Login only; no downloads or manifest changes
# ============================================================

import requests

from google.colab import userdata

API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
USERNAME = userdata.get("OPEN_SUBTITLES_USERNAME")
PASSWORD = userdata.get("OPEN_SUBTITLES_PASSWORD")

if not all([API_KEY, USERNAME, PASSWORD]):
    raise ValueError(
        "Missing OpenSubtitles credentials in Colab Secrets. "
        "Check OPEN_SUBTITLES_API_KEY, "
        "OPEN_SUBTITLES_USERNAME, and OPEN_SUBTITLES_PASSWORD."
    )

BASE_URL = "https://api.opensubtitles.com/api/v1"

headers = {
    "Api-Key": API_KEY,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
}

# Authenticate with saved credentials
response = requests.post(
    f"{BASE_URL}/login",
    headers=headers,
    json={
        "username": USERNAME,
        "password": PASSWORD,
    },
    timeout=30,
)

print("Login HTTP status:", response.status_code)

if response.status_code != 200:
    print("Login failed. Response:", response.text[:500])
    raise RuntimeError(
        "Could not refresh the OpenSubtitles session. "
        "Check your credentials and API access."
    )

data = response.json()
new_token = data.get("token")

if not new_token:
    raise RuntimeError(
        "Login succeeded but no token was returned. "
        "Inspect the response structure without printing credentials."
    )

# Store the fresh token in this Colab runtime only.
# This does not print the token or modify the manifest.
import os
os.environ["OPEN_SUBTITLES_TOKEN"] = new_token

# Verify the fresh token and retrieve quota information.
check = requests.get(
    f"{BASE_URL}/infos/user",
    headers={
        **headers,
        "Authorization": f"Bearer {new_token}",
    },
    timeout=30,
)

print("Session verification HTTP status:", check.status_code)

if check.status_code != 200:
    raise RuntimeError(
        "Login returned a token, but session verification failed: "
        + check.text[:300]
    )

user_data = check.json().get("data", {})
print("Session verified successfully.")
print("Download quota information:", user_data.get("allowed_downloads"))

In [ ]:
# ============================================================
# NOTEBOOK 08 — RESTORE OPENSUBTITLES SESSION
# No downloads; no manifest changes
# ============================================================

import os
import requests
from google.colab import userdata

# Load the API key and credentials from Colab Secrets.
API_KEY = (userdata.get("OPEN_SUBTITLES_API_KEY") or "").strip()
USERNAME = (userdata.get("OPEN_SUBTITLES_USERNAME") or "").strip()
PASSWORD = (userdata.get("OPEN_SUBTITLES_PASSWORD") or "").strip()

if not API_KEY:
    raise RuntimeError(
        "OPEN_SUBTITLES_API_KEY is missing from Colab Secrets."
    )

BASE_URL = "https://api.opensubtitles.com/api/v1"

headers = {
    "Api-Key": API_KEY,
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
}

# Reuse the current runtime token if available.
TOKEN = os.environ.get("OPEN_SUBTITLES_TOKEN", "").strip()

if TOKEN:
    check = requests.get(
        f"{BASE_URL}/infos/user",
        headers={
            **headers,
            "Authorization": f"Bearer {TOKEN}",
        },
        timeout=30,
    )
else:
    check = None

# Log in again if the token is missing or invalid.
if check is None or check.status_code != 200:
    if not USERNAME or not PASSWORD:
        raise RuntimeError(
            "Token is invalid/missing and username or password is missing "
            "from Colab Secrets. Add OPEN_SUBTITLES_USERNAME and "
            "OPEN_SUBTITLES_PASSWORD, then rerun."
        )

    login = requests.post(
        f"{BASE_URL}/login",
        headers=headers,
        json={
            "username": USERNAME,
            "password": PASSWORD,
        },
        timeout=30,
    )

    print("Login HTTP status:", login.status_code)

    if login.status_code != 200:
        raise RuntimeError(
            "Login failed. Check Colab Secrets and API access. "
            + login.text[:300]
        )

    TOKEN = login.json().get("token", "")
    if not TOKEN:
        raise RuntimeError("Login response did not contain a token.")

    os.environ["OPEN_SUBTITLES_TOKEN"] = TOKEN

    check = requests.get(
        f"{BASE_URL}/infos/user",
        headers={
            **headers,
            "Authorization": f"Bearer {TOKEN}",
        },
        timeout=30,
    )

print("Session verification HTTP status:", check.status_code)

if check.status_code != 200:
    raise RuntimeError(
        "Session verification failed: " + check.text[:300]
    )

print("Session verified successfully.")
print(
    "Download quota:",
    check.json().get("data", {}).get("allowed_downloads")
)
print("Credentials and token were not printed.")

In [ ]:
# ============================================================
# NOTEBOOK 08 — PRECISE DHONI CANDIDATE SEARCH
# Search only; no downloads or manifest changes
# ============================================================

import os
import requests
import pandas as pd
from pathlib import Path
from IPython.display import display
from google.colab import userdata

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
SEARCH_RESULTS = PROJECT / "01_data/processed/subtitle_corpus_v2/search_results"
SEARCH_RESULTS.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://api.opensubtitles.com/api/v1"

API_KEY = (userdata.get("OPEN_SUBTITLES_API_KEY") or "").strip()
TOKEN = os.environ.get("OPEN_SUBTITLES_TOKEN", "").strip()

if not API_KEY:
    raise RuntimeError("OPEN_SUBTITLES_API_KEY is missing from Colab Secrets.")

if not TOKEN:
    raise RuntimeError(
        "Fresh token is missing from this runtime. Run the session-refresh cell first."
    )

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
}

# Verify session before searching.
session = requests.get(
    f"{BASE_URL}/infos/user",
    headers=headers,
    timeout=30,
)

print("Session verification HTTP status:", session.status_code)

if session.status_code != 200:
    raise RuntimeError(
        "Token is no longer valid. Run the session-refresh cell first. "
        + session.text[:250]
    )

print(
    "Download quota:",
    session.json().get("data", {}).get("allowed_downloads")
)

# Search variations for the missing movie.
queries = [
    "M.S. Dhoni: The Untold Story",
    "M S Dhoni",
    "MS Dhoni 2016",
    "The Untold Story 2016 Indian movie",
]

rows = []
errors = []

for query in queries:
    response = requests.get(
        f"{BASE_URL}/subtitles",
        headers=headers,
        params={
            "query": query,
            "languages": "en",
            "year": 2016,
            "type": "movie",
            "order_by": "download_count",
            "order_direction": "desc",
            "page": 1,
        },
        timeout=45,
    )

    print(f"Search '{query}': HTTP {response.status_code}")

    if response.status_code != 200:
        errors.append({
            "query": query,
            "status": response.status_code,
            "error": response.text[:250],
        })
        continue

    for item in response.json().get("data", []):
        attr = item.get("attributes", {}) or {}
        feature = attr.get("feature_details", {}) or {}

        candidate_title = (
            feature.get("movie_name")
            or feature.get("title")
            or attr.get("title")
            or ""
        )

        for file_info in attr.get("files", []) or []:
            rows.append({
                "subtitle_id": str(item.get("id", "")),
                "file_id": file_info.get("file_id"),
                "language": attr.get("language"),
                "candidate_title": candidate_title,
                "candidate_year": feature.get("year"),
                "candidate_tmdb_id": feature.get("tmdb_id"),
                "candidate_imdb_id": feature.get("imdb_id"),
                "download_count": attr.get("download_count"),
                "machine_translated": attr.get("machine_translated"),
                "ai_translated": attr.get("ai_translated"),
                "foreign_parts_only": attr.get("foreign_parts_only"),
                "hearing_impaired": attr.get("hearing_impaired"),
                "search_query": query,
            })

# Deduplicate and rank candidates.
df = pd.DataFrame(rows)

if df.empty:
    print("\nNo candidates returned.")
else:
    df = df.drop_duplicates(subset=["subtitle_id", "file_id"])

    df["title_match"] = df["candidate_title"].astype(str).str.contains(
        r"dhoni",
        case=False,
        na=False,
    )

    # IMDb ID from the previously returned exact-title result.
    df["imdb_match"] = (
        df["candidate_imdb_id"].astype(str).str.replace(
            r"\D", "", regex=True
        ) == "4169250"
    )

    # Expected TMDB ID for this project: 45269.
    df["tmdb_match"] = (
        df["candidate_tmdb_id"].astype(str).str.replace(
            r"\.0$", "", regex=True
        ) == "45269"
    )

    df["review_priority"] = (
        df["tmdb_match"].astype(int) * 100
        + df["imdb_match"].astype(int) * 50
        + df["title_match"].astype(int) * 10
    )

    df = df.sort_values(
        ["review_priority", "download_count"],
        ascending=[False, False],
        na_position="last",
    )

    output_path = (
        SEARCH_RESULTS / "batch08_dhoni_precise_search_20261028.csv"
    )
    df.to_csv(output_path, index=False)

    print(f"\nUnique candidates found: {len(df)}")
    print(f"Results saved to: {output_path}")

    print("\nBest title/ID matches for manual review:")
    display(
        df[
            df["title_match"] | df["imdb_match"] | df["tmdb_match"]
        ].head(25).reset_index(drop=True)
    )

if errors:
    print("\nSearch errors:")
    display(pd.DataFrame(errors))

print(
    "\nSearch complete. No subtitle downloads were made, "
    "and the manifest was not changed."
)

In [ ]:
# ============================================================
# NOTEBOOK 08 — BATCH 08 DOWNLOAD
# Sultan, Kabali, Rustom
# No manifest changes; no automatic acceptance
# ============================================================

import os
import requests
import pandas as pd
from pathlib import Path
from google.colab import userdata

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
RAW.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://api.opensubtitles.com/api/v1"

API_KEY = (userdata.get("OPEN_SUBTITLES_API_KEY") or "").strip()
TOKEN = os.environ.get("OPEN_SUBTITLES_TOKEN", "").strip()

if not API_KEY or not TOKEN:
    raise RuntimeError(
        "API key or token missing. Run the session-refresh cell first."
    )

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "User-Agent": "ICMB130SubtitleResearch/1.0",
}

# Verify session and quota before any download.
session = requests.get(
    f"{BASE_URL}/infos/user",
    headers=headers,
    timeout=30,
)

print("Session status:", session.status_code)

if session.status_code != 200:
    raise RuntimeError("Session invalid. Refresh the OpenSubtitles login.")

quota_data = session.json().get("data", {})
print("Quota information:", quota_data.get("allowed_downloads"))

# Reviewed candidates only.
batch = [
    {
        "movie_record_id": "2016_2",
        "movie": "Sultan",
        "subtitle_id": "1064672",
        "file_id": 1153236,
        "filename": "2016_2_Sultan_candidate_1064672.srt",
    },
    {
        "movie_record_id": "2016_3",
        "movie": "Kabali",
        "subtitle_id": "1074077",
        "file_id": 1162953,
        "filename": "2016_3_Kabali_candidate_1074077.srt",
    },
    {
        "movie_record_id": "2016_5",
        "movie": "Rustom",
        "subtitle_id": "1074378",
        "file_id": 1163261,
        "filename": "2016_5_Rustom_candidate_1074378.srt",
    },
]

results = []

for item in batch:
    destination = RAW / item["filename"]

    print(f"\nProcessing {item['movie']} (subtitle ID {item['subtitle_id']})")

    if destination.exists():
        print("Skipped: destination file already exists.")
        results.append({
            **item,
            "http_status": None,
            "download_status": "skipped_existing_file",
            "bytes": destination.stat().st_size,
            "path": str(destination),
        })
        continue

    try:
        response = requests.post(
            f"{BASE_URL}/download",
            headers=headers,
            json={"file_id": item["file_id"]},
            timeout=45,
        )

        print("Download endpoint status:", response.status_code)

        if response.status_code != 200:
            print("Download failed:", response.text[:250])
            results.append({
                **item,
                "http_status": response.status_code,
                "download_status": "failed",
                "bytes": 0,
                "path": "",
            })
            continue

        download_url = response.json().get("link")

        if not download_url:
            print("Failed: API response did not contain a download link.")
            results.append({
                **item,
                "http_status": response.status_code,
                "download_status": "missing_download_link",
                "bytes": 0,
                "path": "",
            })
            continue

        # Fetch the returned file URL. Do not send the API token to this URL.
        file_response = requests.get(download_url, timeout=60)

        if file_response.status_code != 200 or not file_response.content:
            print("File retrieval failed:", file_response.status_code)
            results.append({
                **item,
                "http_status": file_response.status_code,
                "download_status": "file_retrieval_failed",
                "bytes": 0,
                "path": "",
            })
            continue

        # Save the raw file exactly as returned; no text conversion.
        destination.write_bytes(file_response.content)

        print("Saved bytes:", destination.stat().st_size)

        results.append({
            **item,
            "http_status": file_response.status_code,
            "download_status": "downloaded_pending_validation",
            "bytes": destination.stat().st_size,
            "path": str(destination),
        })

    except requests.RequestException as exc:
        print("Request error:", type(exc).__name__)
        results.append({
            **item,
            "http_status": None,
            "download_status": "request_error",
            "bytes": 0,
            "path": "",
        })

# Save a batch log for traceability. This does not change the manifest.
result_df = pd.DataFrame(results)
log_path = WORK / "search_results" / "batch08_download_log_20261028.csv"
result_df.to_csv(log_path, index=False)

print("\n" + "=" * 70)
print("BATCH DOWNLOAD SUMMARY")
print("=" * 70)
display(result_df[
    ["movie", "subtitle_id", "file_id", "http_status",
     "download_status", "bytes", "path"]
])

print("\nDownload log:", log_path)
print("No manifest changes were made.")
print("Next: validate the actual contents of all downloaded files together.")

In [ ]:
# ============================================================
# NOTEBOOK 08 — BATCH 08 SUBTITLE VALIDATION
# Sultan, Kabali, Rustom
# Read-only validation of raw subtitle files
# ============================================================

import re
import pandas as pd
from pathlib import Path
from collections import Counter
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
SEARCH_RESULTS = WORK / "search_results"

# Approximate runtime references in minutes; used only for diagnostics.
# These are not authoritative runtime metadata.
RUNTIME_MINUTES = {
    "Sultan": 170,
    "Kabali": 153,
    "Rustom": 148,
}

BATCH = [
    {
        "movie_record_id": "2016_2",
        "movie": "Sultan",
        "subtitle_id": "1064672",
        "file": RAW / "2016_2_Sultan_candidate_1064672.srt",
    },
    {
        "movie_record_id": "2016_3",
        "movie": "Kabali",
        "subtitle_id": "1074077",
        "file": RAW / "2016_3_Kabali_candidate_1074077.srt",
    },
    {
        "movie_record_id": "2016_5",
        "movie": "Rustom",
        "subtitle_id": "1074378",
        "file": RAW / "2016_5_Rustom_candidate_1074378.srt",
    },
]

TIMESTAMP_RE = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
)

def to_seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def decode_subtitle(path):
    raw_bytes = path.read_bytes()

    # Detect common subtitle encodings without altering the source file.
    if raw_bytes.startswith(b"\xef\xbb\xbf"):
        text = raw_bytes.decode("utf-8-sig", errors="replace")
        encoding = "UTF-8 with BOM"
    else:
        try:
            text = raw_bytes.decode("utf-8")
            encoding = "UTF-8"
        except UnicodeDecodeError:
            text = raw_bytes.decode("cp1252", errors="replace")
            encoding = "CP1252 fallback"

    return raw_bytes, text, encoding

def inspect_subtitle(item):
    path = item["file"]

    if not path.exists():
        return {
            "movie_record_id": item["movie_record_id"],
            "movie": item["movie"],
            "subtitle_id": item["subtitle_id"],
            "file_exists": False,
            "validation_status": "missing_file",
            "file_path": str(path),
        }

    raw_bytes, text, encoding = decode_subtitle(path)
    lines = text.splitlines()

    # Count blocks and timestamp validity.
    timestamp_matches = list(TIMESTAMP_RE.finditer(text))
    blocks = re.split(r"\n\s*\n", text.strip())
    blocks_with_dialogue = 0
    malformed_blocks = 0
    dialogue_lines = []
    end_times = []
    start_times = []
    timestamp_count = 0

    for block in blocks:
        block_lines = [x.strip() for x in block.splitlines() if x.strip()]
        if not block_lines:
            continue

        matches = [
            TIMESTAMP_RE.search(line)
            for line in block_lines
        ]
        match = next((m for m in matches if m), None)

        if match:
            timestamp_count += 1
            groups = match.groups()
            start = to_seconds(*groups[:4])
            end = to_seconds(*groups[4:])
            start_times.append(start)
            end_times.append(end)

            if end < start:
                malformed_blocks += 1

            # Dialogue is usually after the timestamp line.
            timestamp_line_index = next(
                (i for i, line in enumerate(block_lines)
                 if TIMESTAMP_RE.search(line)),
                -1
            )
            payload = block_lines[timestamp_line_index + 1:]
            payload = [
                re.sub(r"<[^>]+>", "", line).strip()
                for line in payload
            ]
            payload = [
                line for line in payload
                if line and not re.fullmatch(r"\d+", line)
            ]

            if payload:
                blocks_with_dialogue += 1
                dialogue_lines.extend(payload)
        else:
            # Ignore SRT sequence-number-only fragments.
            if any(re.search(r"[A-Za-z\u0080-\uFFFF]", line)
                   for line in block_lines):
                malformed_blocks += 1

    dialogue = "\n".join(dialogue_lines)
    words = re.findall(r"\b[\w’'-]+\b", dialogue, flags=re.UNICODE)

    # Script analysis: useful for detecting a mislabeled language.
    letters = [c for c in dialogue if c.isalpha()]
    total_letters = len(letters)

    script_counts = {
        "latin_letters": sum(
            0x0041 <= ord(c) <= 0x024F for c in letters
        ),
        "cyrillic_letters": sum(
            0x0400 <= ord(c) <= 0x052F for c in letters
        ),
        "arabic_letters": sum(
            0x0600 <= ord(c) <= 0x06FF for c in letters
        ),
        "devanagari_letters": sum(
            0x0900 <= ord(c) <= 0x097F for c in letters
        ),
        "cjk_letters": sum(
            0x3040 <= ord(c) <= 0x30FF
            or 0x3400 <= ord(c) <= 0x9FFF
            for c in letters
        ),
    }

    script_pct = {
        key + "_pct": round(value / total_letters * 100, 2)
        if total_letters else 0
        for key, value in script_counts.items()
    }

    # Repeated text and timing gaps.
    dialogue_normalized = [
        re.sub(r"\s+", " ", line).strip().lower()
        for line in dialogue_lines
    ]
    meaningful_lines = [line for line in dialogue_normalized if line]
    repeated_line_count = (
        len(meaningful_lines) - len(set(meaningful_lines))
    )

    timeline = sorted(
        zip(start_times, end_times),
        key=lambda x: x[0]
    )

    gaps = []
    previous_end = None
    for start, end in timeline:
        if previous_end is not None and start - previous_end > 120:
            gaps.append(round(start - previous_end, 1))
        previous_end = max(previous_end or end, end)

    first_timestamp = min(start_times) if start_times else None
    last_timestamp = max(end_times) if end_times else None

    runtime_seconds = RUNTIME_MINUTES[item["movie"]] * 60
    coverage_pct = (
        round((last_timestamp - first_timestamp) / runtime_seconds * 100, 1)
        if first_timestamp is not None and last_timestamp is not None
        else 0
    )

    latin_pct = script_pct["latin_letters_pct"]
    non_latin_pct = max(
        100 - latin_pct, 0
    ) if total_letters else 0

    # This is a review recommendation, not automatic acceptance.
    if total_letters == 0 or blocks_with_dialogue < 50:
        provisional_status = "reject_or_manual_review"
    elif latin_pct < 85:
        provisional_status = "likely_wrong_language_manual_review"
    elif coverage_pct < 60:
        provisional_status = "incomplete_or_manual_review"
    elif malformed_blocks > max(5, timestamp_count * 0.02):
        provisional_status = "format_quality_review"
    else:
        provisional_status = "candidate_for_manual_acceptance_review"

    # Show sample dialogue for human language verification.
    sample = " | ".join(dialogue_lines[:8])

    return {
        "movie_record_id": item["movie_record_id"],
        "movie": item["movie"],
        "subtitle_id": item["subtitle_id"],
        "file_exists": True,
        "file_bytes": len(raw_bytes),
        "encoding": encoding,
        "subtitle_blocks": len(blocks),
        "timestamp_count": timestamp_count,
        "blocks_with_dialogue": blocks_with_dialogue,
        "word_count": len(words),
        "character_count": len(text),
        "first_timestamp_sec": first_timestamp,
        "last_timestamp_sec": last_timestamp,
        "provisional_coverage_pct": coverage_pct,
        "gaps_over_120_sec": len(gaps),
        "largest_gap_sec": max(gaps) if gaps else 0,
        "malformed_blocks": malformed_blocks,
        "repeated_dialogue_lines": repeated_line_count,
        **script_pct,
        "sample_dialogue": sample,
        "provisional_status": provisional_status,
        "file_path": str(path),
    }

# Run validation for all three candidates.
results = pd.DataFrame([inspect_subtitle(item) for item in BATCH])

print("=" * 80)
print("BATCH 08 — VALIDATION SUMMARY")
print("=" * 80)

summary_columns = [
    "movie", "subtitle_id", "file_exists", "file_bytes", "encoding",
    "subtitle_blocks", "blocks_with_dialogue", "word_count",
    "provisional_coverage_pct", "gaps_over_120_sec", "largest_gap_sec",
    "latin_letters_pct", "cyrillic_letters_pct", "arabic_letters_pct",
    "devanagari_letters_pct", "cjk_letters_pct", "malformed_blocks",
    "provisional_status",
]
summary_columns = [c for c in summary_columns if c in results.columns]
display(results[summary_columns])

print("\n" + "=" * 80)
print("DIALOGUE SAMPLES — MANUAL LANGUAGE CHECK")
print("=" * 80)

for _, row in results.iterrows():
    print(f"\n{row['movie']} | Status: {row.get('provisional_status')}")
    print(row.get("sample_dialogue", "No sample available"))

# Save diagnostic report only; raw files and manifest remain unchanged.
report_path = SEARCH_RESULTS / "batch08_validation_report_20261028.csv"
results.to_csv(report_path, index=False)

print("\nValidation report saved to:", report_path)
print("No raw subtitle files were modified.")
print("No validated copies or manifest records were created.")

In [ ]:
# ============================================================
# NOTEBOOK 08 — BATCH 08 TIMESTAMP & QUALITY RECHECK
# Read-only: does not change raw files or manifest
# ============================================================

import re
import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
RAW = PROJECT / "01_data/processed/subtitle_corpus_v2/raw"
SEARCH_RESULTS = PROJECT / "01_data/processed/subtitle_corpus_v2/search_results"

BATCH = [
    ("2016_2", "Sultan", "2016_2_Sultan_candidate_1064672.srt", 170),
    ("2016_3", "Kabali", "2016_3_Kabali_candidate_1074077.srt", 153),
    ("2016_5", "Rustom", "2016_5_Rustom_candidate_1074378.srt", 148),
]

TIMESTAMP_RE = re.compile(
    r"(\d{2,}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2,}):(\d{2}):(\d{2})[,.](\d{3})"
)

def seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

rows = []

for record_id, movie, filename, runtime_min in BATCH:
    path = RAW / filename

    if not path.exists():
        rows.append({
            "movie_record_id": record_id,
            "movie": movie,
            "file_exists": False,
        })
        continue

    raw = path.read_bytes()
    text = raw.decode("utf-8-sig", errors="replace")

    timestamps = []
    malformed = 0
    dialogue_lines = []

    for block in re.split(r"\n\s*\n", text.strip()):
        lines = [x.strip() for x in block.splitlines() if x.strip()]
        match = next((TIMESTAMP_RE.search(x) for x in lines
                      if TIMESTAMP_RE.search(x)), None)

        if not match:
            if any(re.search(r"[A-Za-z]", x) for x in lines):
                malformed += 1
            continue

        g = match.groups()
        start = seconds(*g[:4])
        end = seconds(*g[4:])

        if end < start:
            malformed += 1
            continue

        timestamps.append((start, end))

        timestamp_index = next(
            i for i, line in enumerate(lines) if TIMESTAMP_RE.search(line)
        )
        dialogue_lines.extend(lines[timestamp_index + 1:])

    # Sort timestamps and calculate the timeline envelope.
    timestamps.sort(key=lambda x: x[0])
    runtime_sec = runtime_min * 60

    # Treat timestamps beyond twice the approximate runtime as outliers
    # for diagnostic purposes only; never remove them from the raw file.
    plausible = [
        (start, end) for start, end in timestamps
        if 0 <= start <= runtime_sec * 2
        and 0 <= end <= runtime_sec * 2
    ]
    outlier_count = len(timestamps) - len(plausible)

    if plausible:
        first = min(s for s, e in plausible)
        last = max(e for s, e in plausible)
        coverage = round((last - first) / runtime_sec * 100, 1)
    else:
        first = last = coverage = None

    gaps = []
    previous_end = None
    for start, end in plausible:
        if previous_end is not None and start - previous_end > 120:
            gaps.append(round(start - previous_end, 1))
        previous_end = max(previous_end or end, end)

    dialogue = "\n".join(dialogue_lines)
    latin = sum(c.isalpha() and ord(c) < 0x0250 for c in dialogue)
    letters = sum(c.isalpha() for c in dialogue)
    latin_pct = round(latin / letters * 100, 2) if letters else 0

    rows.append({
        "movie_record_id": record_id,
        "movie": movie,
        "file_exists": True,
        "file_bytes": len(raw),
        "timestamp_blocks": len(timestamps),
        "plausible_timestamp_blocks": len(plausible),
        "timestamp_outliers": outlier_count,
        "malformed_blocks": malformed,
        "word_count": len(re.findall(r"\b[\w’'-]+\b", dialogue)),
        "first_plausible_timestamp_sec": first,
        "last_plausible_timestamp_sec": last,
        "corrected_coverage_pct": coverage,
        "gaps_over_120_sec": len(gaps),
        "largest_gap_sec": max(gaps) if gaps else 0,
        "latin_letter_pct": latin_pct,
        "sample_dialogue": " | ".join(dialogue_lines[:5]),
        "manual_review_needed": (
            outlier_count > 0 or malformed > 0
            or coverage is None or coverage < 60 or coverage > 110
            or latin_pct < 85
        ),
        "file_path": str(path),
    })

result = pd.DataFrame(rows)

print("BATCH 08 — CORRECTED DIAGNOSTICS")
display(result)

output = SEARCH_RESULTS / "batch08_timestamp_recheck_20261028.csv"
result.to_csv(output, index=False)
print("\nDiagnostic report saved to:", output)
print("Raw subtitles and manifest remain unchanged.")

In [ ]:
# ============================================================
# NOTEBOOK 08 — BATCH 08 FINAL QUALITY REVIEW
# Read-only; no manifest changes
# ============================================================

import re
import pandas as pd
from pathlib import Path
from IPython.display import display

RAW = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/raw"
)

FILES = [
    ("2016_2", "Sultan", "2016_2_Sultan_candidate_1064672.srt", 170),
    ("2016_3", "Kabali", "2016_3_Kabali_candidate_1074077.srt", 153),
    ("2016_5", "Rustom", "2016_5_Rustom_candidate_1074378.srt", 148),
]

TS = re.compile(
    r"(\d{2,}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*"
    r"(\d{2,}):(\d{2}):(\d{2})[,.](\d{3})"
)

def sec(groups):
    h, m, s, ms = groups
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

summary = []
sultan_outliers = []

for record_id, movie, filename, runtime_min in FILES:
    path = RAW / filename
    if not path.exists():
        summary.append({
            "movie_record_id": record_id,
            "movie": movie,
            "result": "MISSING FILE",
        })
        continue

    text = path.read_bytes().decode("utf-8-sig", errors="replace")
    runtime_sec = runtime_min * 60
    blocks = re.split(r"\n\s*\n", text.strip())

    timestamp_count = 0
    bad_timestamp_count = 0
    missing_dialogue_count = 0
    duplicate_timestamp_count = 0
    timestamp_pairs = []
    seen_pairs = set()
    outlier_rows = []

    for block_number, block in enumerate(blocks, start=1):
        lines = [line.strip() for line in block.splitlines() if line.strip()]
        match = next((TS.search(line) for line in lines if TS.search(line)), None)

        if not match:
            continue

        timestamp_count += 1
        groups = match.groups()
        start = sec(groups[:4])
        end = sec(groups[4:])

        if end < start:
            bad_timestamp_count += 1

        pair = (start, end)
        if pair in seen_pairs:
            duplicate_timestamp_count += 1
        seen_pairs.add(pair)
        timestamp_pairs.append(pair)

        timestamp_line_index = next(
            i for i, line in enumerate(lines) if TS.search(line)
        )
        dialogue = [
            re.sub(r"<[^>]+>", "", line).strip()
            for line in lines[timestamp_line_index + 1:]
        ]
        dialogue = [line for line in dialogue if line]

        if not dialogue:
            missing_dialogue_count += 1

        # Flag timestamps beyond twice the approximate runtime.
        if start > runtime_sec * 2 or end > runtime_sec * 2:
            outlier_rows.append({
                "movie": movie,
                "block_number": block_number,
                "start_seconds": round(start, 3),
                "end_seconds": round(end, 3),
                "dialogue_sample": " ".join(dialogue)[:180],
            })

    summary.append({
        "movie_record_id": record_id,
        "movie": movie,
        "timestamp_count": timestamp_count,
        "bad_timestamp_count": bad_timestamp_count,
        "duplicate_timestamp_count": duplicate_timestamp_count,
        "blocks_without_dialogue": missing_dialogue_count,
        "runtime_outlier_count": len(outlier_rows),
        "file_bytes": path.stat().st_size,
    })

    if outlier_rows:
        sultan_outliers.extend(outlier_rows)

print("FINAL STRUCTURAL REVIEW")
display(pd.DataFrame(summary))

print("\nTIMESTAMP OUTLIERS — REVIEW ONLY")
if sultan_outliers:
    display(pd.DataFrame(sultan_outliers))
else:
    print("No timestamps beyond twice the approximate runtime.")

print("\nNo files or manifest records were modified.")

In [ ]:
# ============================================================
# NOTEBOOK 08 — BATCH 08 FINALIZATION
# Sultan, Kabali, Rustom
# Backup manifest -> copy validated files -> append records
# ============================================================

from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

VALIDATED.mkdir(parents=True, exist_ok=True)

# Existing manifest schema: must remain unchanged.
EXPECTED_COLUMNS = [
    "movie_record_id", "year", "rank", "movie_title", "language",
    "tmdb_id", "imdb_id", "subtitle_language", "subtitle_type",
    "subtitle_source", "source_id", "source_url", "raw_subtitle_file",
    "validated_subtitle_file", "acquisition_status", "validation_status",
    "title_verified", "year_verified", "movie_identity_verified",
    "language_verified", "dialogue_blocks", "word_count", "character_count",
    "encoding_status", "quality_status", "notes"
]

if not MANIFEST.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")

manifest = pd.read_csv(MANIFEST)

if list(manifest.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        "Manifest columns do not match the expected 26-column schema. "
        "No changes were made."
    )

# Explicit candidate records and conservative quality decisions.
batch = [
    {
        "movie_record_id": "2016_2",
        "year": 2016,
        "rank": 2,
        "movie_title": "Sultan",
        "language": "Hindi",
        "tmdb_id": 352173,
        "source_id": "1064672",
        "raw_name": "2016_2_Sultan_candidate_1064672.srt",
        "validated_name": "2016_2_SULTAN_ENGLISH.srt",
        "blocks": 1985,
        "words": 12552,
        "bytes": 137330,
        "encoding": "UTF-8",
        "validation_status": "accepted_with_quality_flag",
        "quality_status": "quality_flag",
        "notes": (
            "English-script dialogue confirmed by text inspection; "
            "1985 timestamp blocks and 12552 words. Corrected diagnostic "
            "coverage approximately 97.9%. Final 28 blocks have timestamps "
            "around 19.5 hours, outside the approximate film runtime. "
            "Raw file retained unchanged; timestamp anomaly not corrected. "
            "Movie identity not independently confirmed from every dialogue block."
        ),
    },
    {
        "movie_record_id": "2016_3",
        "year": 2016,
        "rank": 3,
        "movie_title": "Kabali",
        "language": "Tamil",
        "tmdb_id": 394269,
        "source_id": "1074077",
        "raw_name": "2016_3_Kabali_candidate_1074077.srt",
        "validated_name": "2016_3_KABALI_ENGLISH.srt",
        "blocks": 2157,
        "words": 11737,
        "bytes": 139605,
        "encoding": "UTF-8",
        "validation_status": "accepted_pending_identity_review",
        "quality_status": "pending_identity_review",
        "notes": (
            "English-script dialogue confirmed by sample inspection; "
            "2157 timestamp blocks and 11737 words. Approximate runtime "
            "coverage 91.4%; no timestamp outliers detected by current checks. "
            "Movie identity still requires confirmation beyond subtitle metadata."
        ),
    },
    {
        "movie_record_id": "2016_5",
        "year": 2016,
        "rank": 5,
        "movie_title": "Rustom",
        "language": "Hindi",
        "tmdb_id": 392044,
        "source_id": "1074378",
        "raw_name": "2016_5_Rustom_candidate_1074378.srt",
        "validated_name": "2016_5_RUSTOM_ENGLISH.srt",
        "blocks": 1794,
        "words": 17163,
        "bytes": 170654,
        "encoding": "UTF-8",
        "validation_status": "accepted_pending_identity_review",
        "quality_status": "pending_identity_review",
        "notes": (
            "English-script dialogue confirmed by sample inspection; "
            "1794 timestamp blocks and approximately 17163 words. "
            "Approximate runtime coverage 96.4%; no timestamp outliers "
            "detected by current checks. Movie identity still requires "
            "confirmation beyond subtitle metadata."
        ),
    },
]

# Preflight: all source files must exist and match the reviewed byte sizes.
existing_ids = set(manifest["movie_record_id"].astype(str))
existing_ids_in_batch = [
    row["movie_record_id"] for row in batch
    if row["movie_record_id"] in existing_ids
]

if existing_ids_in_batch:
    raise ValueError(
        f"These IDs already exist in the manifest: {existing_ids_in_batch}. "
        "No changes were made."
    )

for row in batch:
    source = RAW / row["raw_name"]
    destination = VALIDATED / row["validated_name"]

    if not source.exists():
        raise FileNotFoundError(f"Raw file missing: {source}")

    if source.stat().st_size != row["bytes"]:
        raise ValueError(
            f"Unexpected byte size for {source.name}: "
            f"{source.stat().st_size}; expected {row['bytes']}. "
            "No changes were made."
        )

    if destination.exists():
        raise FileExistsError(
            f"Validated destination already exists: {destination}. "
            "No changes were made."
        )

# Build the records before changing files.
new_records = []

for row in batch:
    new_records.append({
        "movie_record_id": row["movie_record_id"],
        "year": row["year"],
        "rank": row["rank"],
        "movie_title": row["movie_title"],
        "language": row["language"],
        "tmdb_id": row["tmdb_id"],
        "imdb_id": pd.NA,
        "subtitle_language": "English",
        "subtitle_type": "full subtitles",
        "subtitle_source": "OpenSubtitles",
        "source_id": row["source_id"],
        "source_url": "https://www.opensubtitles.com/",
        "raw_subtitle_file": str(RAW / row["raw_name"]),
        "validated_subtitle_file": str(VALIDATED / row["validated_name"]),
        "acquisition_status": "downloaded",
        "validation_status": row["validation_status"],
        "title_verified": True,
        "year_verified": True,
        "movie_identity_verified": False,
        "language_verified": True,
        "dialogue_blocks": row["blocks"],
        "word_count": row["words"],
        "character_count": row["bytes"],
        "encoding_status": row["encoding"],
        "quality_status": row["quality_status"],
        "notes": row["notes"],
    })

new_df = pd.DataFrame(new_records, columns=EXPECTED_COLUMNS)
updated = pd.concat([manifest, new_df], ignore_index=True)

# Safety check before committing.
if updated["movie_record_id"].astype(str).duplicated().any():
    raise ValueError("Duplicate IDs detected in proposed manifest. No changes made.")

# Backup manifest before any copy or manifest update.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = MANIFEST.with_name(
    f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_batch08_{timestamp}.csv"
)
shutil.copy2(MANIFEST, backup)

created_destinations = []

try:
    # Copy the raw files without changing their contents.
    for row in batch:
        source = RAW / row["raw_name"]
        destination = VALIDATED / row["validated_name"]
        shutil.copy2(source, destination)
        created_destinations.append(destination)

    # Confirm byte-for-byte copies.
    for row in batch:
        source = RAW / row["raw_name"]
        destination = VALIDATED / row["validated_name"]

        if source.read_bytes() != destination.read_bytes():
            raise IOError(f"Copy verification failed: {destination.name}")

    # Save the updated manifest only after copy verification.
    updated.to_csv(MANIFEST, index=False)

except Exception:
    # Remove only validated copies created by this cell.
    for destination in created_destinations:
        if destination.exists():
            destination.unlink()
    raise

# Reload and verify the committed manifest.
verified = pd.read_csv(MANIFEST)

print("=" * 70)
print("BATCH 08 FINALIZATION SUMMARY")
print("=" * 70)
print("Backup:", backup)
print("Manifest rows before:", len(manifest))
print("Manifest rows after:", len(verified))
print("Duplicate IDs:", verified["movie_record_id"].astype(str).duplicated().sum())

for row in batch:
    dest = VALIDATED / row["validated_name"]
    print(f"{row['movie_title']}: validated copy exists = {dest.exists()}")

display(
    verified[
        verified["movie_record_id"].astype(str).isin(
            [row["movie_record_id"] for row in batch]
        )
    ][EXPECTED_COLUMNS]
)

print("\nRaw files were preserved unchanged.")
print("Dhoni remains pending; Dangal was not added again.")

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

print("Master exists:", MASTER.exists())
print("Manifest exists:", MANIFEST.exists())

if not MASTER.exists() or not MANIFEST.exists():
    raise FileNotFoundError(
        "Master dataset or subtitle manifest was not found. "
        "Check that Google Drive is mounted and the project paths are correct."
    )

master = pd.read_csv(MASTER)
manifest = pd.read_csv(MANIFEST)

print("\nMaster shape:", master.shape)
print("Manifest shape:", manifest.shape)
print("\nMaster columns:")
print(master.columns.tolist())

# Find the year column without assuming its exact name
year_col = next(
    (c for c in master.columns if c.lower() in ["year", "dataset_year", "release_year"]),
    None
)

if year_col is None:
    raise KeyError("Could not identify the year column in the master dataset.")

batch09 = master[pd.to_numeric(master[year_col], errors="coerce") == 2017].copy()

print("\n" + "=" * 65)
print("BATCH 09 — 2017 MOVIES")
print("=" * 65)
print("Number of movies:", len(batch09))
print(batch09.to_string(index=False))

print("\nExisting 2017 manifest records:")
if "year" in manifest.columns:
    existing = manifest[pd.to_numeric(manifest["year"], errors="coerce") == 2017]
    print(existing[
        [c for c in [
            "movie_record_id", "movie_title", "subtitle_language",
            "acquisition_status", "validation_status",
            "raw_subtitle_file", "validated_subtitle_file"
        ] if c in existing.columns]
    ].to_string(index=False))
else:
    print("No 'year' column found in the manifest; inspect its columns before proceeding.")


In [ ]:

from google.colab import userdata
import os
import requests

# Restore credentials from Colab Secrets
OPEN_SUBTITLES_API_KEY = userdata.get("OPEN_SUBTITLES_API_KEY")
OPEN_SUBTITLES_USERNAME = userdata.get("OPEN_SUBTITLES_USERNAME")
OPEN_SUBTITLES_PASSWORD = userdata.get("OPEN_SUBTITLES_PASSWORD")

if not all([
    OPEN_SUBTITLES_API_KEY,
    OPEN_SUBTITLES_USERNAME,
    OPEN_SUBTITLES_PASSWORD
]):
    raise RuntimeError(
        "One or more OpenSubtitles secrets are missing. "
        "Check the exact secret names in Colab Secrets."
    )

# Log in to obtain a fresh token
response = requests.post(
    "https://api.opensubtitles.com/api/v1/login",
    headers={
        "Api-Key": OPEN_SUBTITLES_API_KEY,
        "Content-Type": "application/json",
        "User-Agent": "ICMB130-Research/1.0"
    },
    json={
        "username": OPEN_SUBTITLES_USERNAME,
        "password": OPEN_SUBTITLES_PASSWORD
    },
    timeout=30
)

if response.status_code != 200:
    raise RuntimeError(
        f"OpenSubtitles login failed (HTTP {response.status_code}). "
        "Check the credentials and try again."
    )

login_data = response.json()
token = login_data.get("token")

if not token:
    raise RuntimeError("Login returned no token. Check the OpenSubtitles response.")

# Restore the variables expected by the Batch 09 search cell
os.environ["OPEN_SUBTITLES_API_KEY"] = OPEN_SUBTITLES_API_KEY
os.environ["OPEN_SUBTITLES_TOKEN"] = token
os.environ["OPEN_SUBTITLES_USERNAME"] = OPEN_SUBTITLES_USERNAME

print("OpenSubtitles login successful.")
print("API key available:", bool(os.environ.get("OPEN_SUBTITLES_API_KEY")))
print("Fresh token available:", bool(os.environ.get("OPEN_SUBTITLES_TOKEN")))


In [ ]:

import os
import time
import requests
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
SEARCH_RESULTS = WORK / "search_results"
SEARCH_RESULTS.mkdir(parents=True, exist_ok=True)

MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MAPPING = PROJECT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv"

API_KEY = os.environ.get("OPEN_SUBTITLES_API_KEY")
TOKEN = os.environ.get("OPEN_SUBTITLES_TOKEN")

if not API_KEY or not TOKEN:
    raise RuntimeError(
        "OpenSubtitles API key or token is missing from this Colab runtime. "
        "Restore them from Colab Secrets before running this cell."
    )

master = pd.read_csv(MASTER)
mapping = pd.read_csv(MAPPING)

batch = master[pd.to_numeric(master["year"], errors="coerce") == 2017].copy()
if len(batch) != 5:
    raise ValueError(f"Expected 5 movies for 2017, found {len(batch)}.")

print("TMDB mapping columns:", mapping.columns.tolist())

# Match the mapping by movie_record_id when available; do not alter the master.
if "movie_record_id" in mapping.columns:
    batch = batch.merge(
        mapping, on="movie_record_id", how="left",
        suffixes=("", "_mapping")
    )

def first_value(row, names):
    for name in names:
        if name in row.index:
            value = row[name]
            if pd.notna(value) and str(value).strip() not in ("", "nan", "None"):
                return value
    return None

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "User-Agent": "ICMB130-Research/1.0"
}
endpoint = "https://api.opensubtitles.com/api/v1/subtitles"

query_variants = {
    "Baahubali 2: The Conclusion": [
        "Baahubali 2 The Conclusion",
        "Baahubali 2 2017",
        "Baahubali The Conclusion"
    ],
    "Tiger Zinda Hai": [
        "Tiger Zinda Hai",
        "Tiger Zinda Hai 2017"
    ],
    "Golmaal Again": [
        "Golmaal Again",
        "Golmaal Again 2017"
    ],
    "Raees": [
        "Raees",
        "Raees 2017 Shah Rukh Khan"
    ],
    "Judwaa 2": [
        "Judwaa 2",
        "Judwaa 2 2017"
    ]
}

candidate_rows = []
errors = []

for _, movie in batch.iterrows():
    record_id = str(movie["movie_record_id"])
    title = str(movie["movie_title"])
    year = int(movie["year"])
    tmdb_id = first_value(
        movie,
        ["tmdb_id_mapping", "tmdb_id", "TMDB_ID", "tmdbId"]
    )
    imdb_id = first_value(
        movie,
        ["imdb_id", "imdb_id_mapping", "IMDb_ID", "imdbId"]
    )

    print(f"\nSearching: {record_id} — {title}")

    seen = set()
    for query in query_variants.get(title, [title, f"{title} {year}"]):
        params = {
            "query": query,
            "languages": "en",
            "year": year,
            "type": "movie"
        }

        # Add IDs only if the mapping actually supplies them.
        if pd.notna(tmdb_id) if tmdb_id is not None else False:
            params["tmdb_id"] = str(int(float(tmdb_id)))
        if pd.notna(imdb_id) if imdb_id is not None else False:
            params["imdb_id"] = str(imdb_id).strip()

        try:
            response = requests.get(
                endpoint, headers=headers, params=params, timeout=30
            )
            if response.status_code != 200:
                errors.append({
                    "movie_record_id": record_id,
                    "movie_title": title,
                    "query": query,
                    "http_status": response.status_code,
                    "error": response.text[:300]
                })
                print(f"  Search returned HTTP {response.status_code}: {query}")
                time.sleep(0.5)
                continue

            results = response.json().get("data", [])
            for item in results:
                attrs = item.get("attributes", {}) or {}
                feature = attrs.get("feature_details", {}) or {}
                files = attrs.get("files", []) or []
                file_id = files[0].get("file_id") if files else None
                subtitle_id = item.get("id")

                dedupe_key = (record_id, str(subtitle_id), str(file_id))
                if dedupe_key in seen:
                    continue
                seen.add(dedupe_key)

                candidate_title = feature.get("title")
                candidate_year = feature.get("year")
                candidate_tmdb = feature.get("tmdb_id")
                candidate_imdb = feature.get("imdb_id")

                title_norm = "".join(ch.lower() for ch in title if ch.isalnum())
                cand_norm = "".join(
                    ch.lower() for ch in str(candidate_title or "") if ch.isalnum()
                )

                candidate_rows.append({
                    "movie_record_id": record_id,
                    "target_title": title,
                    "target_year": year,
                    "target_tmdb_id": tmdb_id,
                    "target_imdb_id": imdb_id,
                    "search_query": query,
                    "subtitle_id": subtitle_id,
                    "file_id": file_id,
                    "language": attrs.get("language"),
                    "candidate_title": candidate_title,
                    "candidate_year": candidate_year,
                    "candidate_tmdb_id": candidate_tmdb,
                    "candidate_imdb_id": candidate_imdb,
                    "download_count": attrs.get("download_count"),
                    "machine_translated": attrs.get("machine_translated"),
                    "ai_translated": attrs.get("ai_translated"),
                    "foreign_parts_only": attrs.get("foreign_parts_only"),
                    "hearing_impaired": attrs.get("hearing_impaired"),
                    "title_match": bool(cand_norm and (
                        cand_norm == title_norm
                        or title_norm in cand_norm
                        or cand_norm in title_norm
                    )),
                    "year_match": str(candidate_year) == str(year),
                    "tmdb_match": (
                        str(candidate_tmdb) == str(int(float(tmdb_id)))
                        if tmdb_id is not None and pd.notna(tmdb_id)
                        and str(tmdb_id).replace(".", "", 1).isdigit()
                        and candidate_tmdb is not None else False
                    ),
                    "imdb_match": (
                        str(candidate_imdb).removeprefix("tt")
                        == str(imdb_id).removeprefix("tt")
                        if imdb_id is not None and pd.notna(imdb_id)
                        and candidate_imdb is not None else False
                    )
                })

            print(f"  {query}: {len(results)} results")
            time.sleep(0.35)

        except Exception as exc:
            errors.append({
                "movie_record_id": record_id,
                "movie_title": title,
                "query": query,
                "http_status": None,
                "error": str(exc)[:300]
            })
            print(f"  Request error for {query}: {exc}")

candidates = pd.DataFrame(candidate_rows)

if not candidates.empty:
    candidates = candidates.drop_duplicates(
        subset=["movie_record_id", "subtitle_id", "file_id"]
    ).copy()

    candidates["review_priority"] = (
        candidates["tmdb_match"].astype(int) * 5
        + candidates["imdb_match"].astype(int) * 5
        + candidates["title_match"].astype(int) * 3
        + candidates["year_match"].astype(int) * 2
        + (candidates["language"].astype(str).str.lower() == "en").astype(int)
    )

    candidates = candidates.sort_values(
        ["movie_record_id", "review_priority", "download_count"],
        ascending=[True, False, False],
        na_position="last"
    )

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
candidate_path = SEARCH_RESULTS / f"batch09_2017_candidates_{timestamp}.csv"
error_path = SEARCH_RESULTS / f"batch09_2017_search_errors_{timestamp}.csv"

if not candidates.empty:
    candidates.to_csv(candidate_path, index=False)

if errors:
    pd.DataFrame(errors).to_csv(error_path, index=False)

print("\n" + "=" * 65)
print("BATCH 09 SEARCH SUMMARY")
print("=" * 65)
print("Movies searched:", len(batch))
print("Unique candidates:", len(candidates))
print("Search errors:", len(errors))
print("Candidate file:", candidate_path)
if errors:
    print("Error file:", error_path)

if not candidates.empty:
    display_cols = [
        "movie_record_id", "target_title", "subtitle_id", "file_id",
        "language", "candidate_title", "candidate_year",
        "candidate_tmdb_id", "candidate_imdb_id",
        "download_count", "machine_translated", "ai_translated",
        "foreign_parts_only", "title_match", "year_match",
        "tmdb_match", "imdb_match", "review_priority"
    ]
    display(candidates[[c for c in display_cols if c in candidates.columns]])
else:
    print("No candidates returned. Review the search errors before changing the search strategy.")


In [ ]:

import pandas as pd
from pathlib import Path

WORK = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics/01_data/processed/subtitle_corpus_v2")
SEARCH_RESULTS = WORK / "search_results"

if "candidates" not in globals() or candidates.empty:
    raise RuntimeError("The candidate search results are not available. Rerun the search cell.")

review = candidates.copy()

# Normalize boolean fields in case CSV-loaded values are strings.
for col in [
    "title_match", "year_match", "tmdb_match", "imdb_match",
    "machine_translated", "ai_translated", "foreign_parts_only"
]:
    if col in review.columns:
        review[col] = review[col].map(
            lambda x: str(x).strip().lower() == "true"
            if pd.notna(x) else False
        )

review["download_count"] = pd.to_numeric(
    review["download_count"], errors="coerce"
).fillna(0)

review["language_ok"] = (
    review["language"].astype(str).str.lower().eq("en")
)

# Prefer candidates with strong metadata matches and no translation flags.
review["review_score"] = (
    review["tmdb_match"].astype(int) * 10
    + review["title_match"].astype(int) * 5
    + review["year_match"].astype(int) * 5
    + review["language_ok"].astype(int) * 5
    - review["machine_translated"].astype(int) * 5
    - review["ai_translated"].astype(int) * 5
    - review["foreign_parts_only"].astype(int) * 3
)

shortlist = (
    review[review["file_id"].notna()]
    .sort_values(
        ["movie_record_id", "review_score", "download_count"],
        ascending=[True, False, False]
    )
    .groupby("movie_record_id", group_keys=False)
    .head(3)
    .copy()
)

shortlist_path = SEARCH_RESULTS / "batch09_2017_candidate_shortlist.csv"
shortlist.to_csv(shortlist_path, index=False)

columns = [
    "movie_record_id", "target_title", "subtitle_id", "file_id",
    "language", "candidate_title", "candidate_year",
    "candidate_tmdb_id", "candidate_imdb_id", "download_count",
    "machine_translated", "ai_translated", "foreign_parts_only",
    "title_match", "year_match", "tmdb_match", "review_score"
]

print("SHORTLIST — UP TO THREE CANDIDATES PER MOVIE")
display(shortlist[[c for c in columns if c in shortlist.columns]])

print("\nShortlist saved to:", shortlist_path)
print("No subtitles downloaded. No manifest changes made.")


In [ ]:

import os
import re
import requests
import pandas as pd
from pathlib import Path

WORK = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics/01_data/processed/subtitle_corpus_v2")
RAW = WORK / "raw"
SEARCH_RESULTS = WORK / "search_results"
RAW.mkdir(parents=True, exist_ok=True)

API_KEY = os.environ.get("OPEN_SUBTITLES_API_KEY")
TOKEN = os.environ.get("OPEN_SUBTITLES_TOKEN")

if not API_KEY or not TOKEN:
    raise RuntimeError("Credentials are missing. Restore the OpenSubtitles session first.")

selected = [
    {"movie_record_id": "2017_1", "title": "Baahubali 2: The Conclusion",
     "subtitle_id": "4219998", "file_id": 4306870, "year": 2017, "rank": 1},
    {"movie_record_id": "2017_2", "title": "Tiger Zinda Hai",
     "subtitle_id": "1146717", "file_id": 1238071, "year": 2017, "rank": 2},
    {"movie_record_id": "2017_3", "title": "Golmaal Again",
     "subtitle_id": "1137316", "file_id": 1228395, "year": 2017, "rank": 3},
    {"movie_record_id": "2017_4", "title": "Raees",
     "subtitle_id": "1400192", "file_id": 1494420, "year": 2017, "rank": 4},
    {"movie_record_id": "2017_5", "title": "Judwaa 2",
     "subtitle_id": "1142430", "file_id": 1233645, "year": 2017, "rank": 5},
]

headers = {
    "Api-Key": API_KEY,
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "User-Agent": "ICMB130-Research/1.0"
}

def parse_time(value):
    value = value.strip().replace(",", ".")
    parts = value.split(":")
    if len(parts) != 3:
        return None
    try:
        h, m, s = int(parts[0]), int(parts[1]), float(parts[2])
        return h * 3600 + m * 60 + s
    except ValueError:
        return None

def inspect_srt(text):
    # SRT/VTT-style timestamp lines
    timestamp_pattern = re.compile(
        r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
        r"(\d{2}:\d{2}:\d{2}[,.]\d{3})"
    )
    timestamps = timestamp_pattern.findall(text)
    blocks = re.split(r"\n\s*\n", text.strip())
    dialogue = []

    for block in blocks:
        lines = block.splitlines()
        if any("-->" in line for line in lines):
            for line in lines:
                if "-->" in line:
                    continue
                line = re.sub(r"<[^>]+>", " ", line)
                line = re.sub(r"\{[^}]+\}", " ", line)
                line = re.sub(r"^\s*\d+\s*$", "", line)
                if re.search(r"[A-Za-z]", line):
                    dialogue.append(line.strip())

    words = re.findall(r"\b[A-Za-z]+\b", " ".join(dialogue))
    starts = [parse_time(a) for a, _ in timestamps]
    ends = [parse_time(b) for _, b in timestamps]
    valid = [
        (a, b) for a, b in zip(starts, ends)
        if a is not None and b is not None and b >= a
    ]

    sample = " ".join(dialogue[:12])[:700]
    ascii_letters = sum(ch.isascii() and ch.isalpha() for ch in sample)
    letters = sum(ch.isalpha() for ch in sample)
    english_script_ratio = ascii_letters / letters if letters else 0

    return {
        "timestamp_count": len(timestamps),
        "dialogue_lines": len(dialogue),
        "english_word_count": len(words),
        "first_timestamp_seconds": valid[0][0] if valid else None,
        "last_timestamp_seconds": valid[-1][1] if valid else None,
        "english_script_ratio_sample": round(english_script_ratio, 3),
        "sample_text": sample
    }

results = []

for item in selected:
    filename = (
        f"{item['movie_record_id']}_"
        f"{re.sub(r'[^A-Za-z0-9]+', '_', item['title']).strip('_')}_"
        f"candidate_{item['subtitle_id']}.srt"
    )
    path = RAW / filename

    print(f"\nInspecting {item['movie_record_id']} — {item['title']}")

    # Do not overwrite an existing raw file.
    if path.exists():
        print("Raw file already exists; inspecting the existing file without redownloading.")
        text = path.read_text(encoding="utf-8-sig", errors="replace")
    else:
        download_response = requests.post(
            "https://api.opensubtitles.com/api/v1/download",
            headers=headers,
            json={"file_id": item["file_id"]},
            timeout=45
        )

        if download_response.status_code != 200:
            print("Download failed with HTTP", download_response.status_code)
            print(download_response.text[:300])
            results.append({
                **item, "raw_file": str(path),
                "download_status": f"HTTP {download_response.status_code}"
            })
            continue

        download_data = download_response.json()
        download_link = download_data.get("link")
        if not download_link:
            print("Download response did not contain a link.")
            results.append({
                **item, "raw_file": str(path),
                "download_status": "missing_download_link"
            })
            continue

        file_response = requests.get(download_link, timeout=45)
        if file_response.status_code != 200:
            print("File retrieval failed with HTTP", file_response.status_code)
            results.append({
                **item, "raw_file": str(path),
                "download_status": f"file HTTP {file_response.status_code}"
            })
            continue

        content = file_response.content
        # Preserve the downloaded bytes without altering the subtitle content.
        path.write_bytes(content)
        text = content.decode("utf-8-sig", errors="replace")

    report = inspect_srt(text)
    print("Raw file:", path.name)
    print("Bytes:", path.stat().st_size)
    print("Timestamp blocks:", report["timestamp_count"])
    print("English words:", report["english_word_count"])
    print("First / last timestamp:", report["first_timestamp_seconds"],
          "/", report["last_timestamp_seconds"])
    print("English-script ratio (sample):", report["english_script_ratio_sample"])
    print("Text sample:", report["sample_text"][:350])

    results.append({
        **item,
        "raw_file": str(path),
        "download_status": "downloaded_or_existing",
        **report
    })

report_df = pd.DataFrame(results)
report_path = SEARCH_RESULTS / "batch09_2017_initial_text_inspection.csv"
report_df.to_csv(report_path, index=False)

print("\nInspection report saved:", report_path)
print("Manifest unchanged. No subtitles accepted yet.")


In [ ]:

import pandas as pd
from pathlib import Path

MASTER = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/ICMB_130_MASTER_DATA.csv"
)

df = pd.read_csv(MASTER)
movies_2018 = (
    df[pd.to_numeric(df["year"], errors="coerce") == 2018]
    .sort_values("rank")
)

display(movies_2018[
    ["movie_record_id", "year", "rank", "movie_title", "language", "original_language"]
])


In [ ]:

import pandas as pd
from pathlib import Path

MASTER = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/ICMB_130_MASTER_DATA.csv"
)

df = pd.read_csv(MASTER)

recent = df[
    pd.to_numeric(df["year"], errors="coerce").between(2017, 2025)
].copy()

recent["year"] = pd.to_numeric(recent["year"]).astype(int)
recent["rank"] = pd.to_numeric(recent["rank"]).astype(int)

recent = recent.sort_values(["year", "rank"])

print("ICMB-130 MOVIE LIST: 2017–2025")
print("Total movies:", len(recent))
display(
    recent[
        ["movie_record_id", "year", "rank", "movie_title",
         "language", "india_gross_cr"]
    ].reset_index(drop=True)
)

print("\nMovies per year:")
display(recent.groupby("year").size().rename("movie_count").to_frame())

print("\nNote: 2026 is not part of the locked ICMB-130 dataset.")


In [ ]:

from pathlib import Path

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

print("Project exists:", PROJECT.exists())
print("Subtitle corpus exists:", WORK.exists())

print("\nMatching folders:")
for root in [PROJECT, WORK]:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_dir() and (
                "extracted_subtitles_2017_2025" in p.name.lower()
                or "incoming_manual_2017_2025" in p.name.lower()
            ):
                print(p)


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"

# Find the actual manifest location
manifest_matches = list(
    PROJECT.rglob("ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv")
)

print("Project exists:", PROJECT.exists())
print("Incoming folder exists:", INCOMING.exists())
print("\nManifest matches:")
for p in manifest_matches:
    print(p)

if not INCOMING.exists():
    raise FileNotFoundError(f"Uploaded subtitle folder not found: {INCOMING}")

# Inventory uploaded files even if the manifest needs locating
files = sorted(p for p in INCOMING.rglob("*") if p.is_file())

inventory_df = pd.DataFrame([
    {
        "filename": p.name,
        "extension": p.suffix.lower(),
        "size_bytes": p.stat().st_size,
        "relative_path": str(p.relative_to(INCOMING)),
        "subtitle_file": p.suffix.lower() in {
            ".srt", ".ass", ".ssa", ".vtt", ".sub"
        }
    }
    for p in files
])

print("\nUploaded files:", len(files))
if not inventory_df.empty:
    print("Subtitle files:", int(inventory_df["subtitle_file"].sum()))
    display(inventory_df)
else:
    print("No files found in the incoming folder.")

if len(manifest_matches) == 1:
    MANIFEST = manifest_matches[0]
    manifest = pd.read_csv(MANIFEST)
    print("\nManifest found:", MANIFEST)
    print("Manifest rows:", len(manifest))

    if "movie_title" in manifest.columns:
        mask = manifest["movie_title"].astype(str).str.strip().str.casefold().eq("83")
        cols = [
            c for c in [
                "movie_record_id", "movie_title", "subtitle_language",
                "raw_subtitle_file", "validated_subtitle_file",
                "acquisition_status", "validation_status"
            ] if c in manifest.columns
        ]
        print("\nExisting 83 records:")
        print(manifest.loc[mask, cols].to_string(index=False))
elif len(manifest_matches) == 0:
    print("\nManifest not found. Do not create a new one; we need to locate the existing copy.")
else:
    print("\nMultiple manifests found. We need to identify the active manifest before proceeding.")


In [ ]:

from pathlib import Path
import pandas as pd
import re

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(MANIFEST)
files = sorted(p for p in INCOMING.rglob("*") if p.is_file())

print("Active manifest:", MANIFEST)
print("Manifest rows:", len(manifest))
print("Uploaded subtitle files:", len(files))

# Check whether 83 already has a manifest record.
print("\n=== EXISTING 83 RECORDS ===")
if "movie_title" in manifest.columns:
    mask = manifest["movie_title"].astype(str).str.strip().str.casefold().eq("83")
    cols = [c for c in [
        "movie_record_id", "movie_title", "subtitle_language",
        "raw_subtitle_file", "validated_subtitle_file",
        "acquisition_status", "validation_status", "notes"
    ] if c in manifest.columns]
    display(manifest.loc[mask, cols])
else:
    print("movie_title column not found.")

# Extract movie IDs from the beginning of filenames.
rows = []
for p in files:
    match = re.match(r"((?:201[7-9]|202[0-5])_[1-5])_", p.name.upper())
    rows.append({
        "movie_record_id_from_filename": match.group(1) if match else None,
        "filename": p.name,
        "relative_path": str(p.relative_to(INCOMING)),
        "size_bytes": p.stat().st_size
    })

file_df = pd.DataFrame(rows)

print("\n=== FILES WITHOUT A RECOGNIZED MOVIE ID PREFIX ===")
display(file_df[file_df["movie_record_id_from_filename"].isna()])

print("\n=== DUPLICATE MOVIE ID PREFIXES ===")
recognized = file_df.dropna(subset=["movie_record_id_from_filename"])
counts = recognized.groupby("movie_record_id_from_filename").size()
duplicate_ids = counts[counts > 1].index

display(
    recognized[
        recognized["movie_record_id_from_filename"].isin(duplicate_ids)
    ].sort_values("movie_record_id_from_filename")
)

print("\n=== MOVIE IDS NOT REPRESENTED BY FILENAME ===")
master = pd.read_csv(PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv")
expected = set(
    master.loc[
        pd.to_numeric(master["year"], errors="coerce").between(2017, 2025),
        "movie_record_id"
    ].astype(str)
)
found = set(recognized["movie_record_id_from_filename"].astype(str))
print(sorted(expected - found))


In [ ]:
from pathlib import Path
import re

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"

files = [p for p in INCOMING.rglob("*") if p.is_file() and p.suffix.lower() == ".srt"]

keywords = [
    "BAAHUBALI", "PUSHPA", "SOORYAVANSHI", "2_0",
    "83", "2017_1", "2021_1"
]

suspicious = [
    p for p in files
    if any(k in str(p).upper().replace(" ", "_") for k in keywords)
    or "sooryavanshi" in str(p).lower()
]

def read_srt_sample(path, max_chars=1800):
    raw = path.read_bytes()

    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            text = raw.decode(encoding)
            used_encoding = encoding
            break
        except UnicodeDecodeError:
            continue

    # Remove subtitle timestamps and numeric sequence lines for readable samples
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if re.fullmatch(r"\d+", line):
            continue
        if "-->" in line:
            continue
        line = re.sub(r"<[^>]+>", "", line)
        line = re.sub(r"\{\\an\d+\}", "", line)
        if line:
            lines.append(line)

    return used_encoding, "\n".join(lines)[:max_chars]

print("Suspicious subtitle files:", len(suspicious))

for i, path in enumerate(suspicious, 1):
    encoding, sample = read_srt_sample(path)
    print("\n" + "=" * 90)
    print(f"FILE {i}: {path.relative_to(INCOMING)}")
    print(f"Size: {path.stat().st_size:,} bytes | Decoding: {encoding}")
    print("-" * 90)
    print(sample or "[No readable dialogue extracted]")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"

rows = []

for path in sorted(INCOMING.rglob("*")):
    if path.is_file() and path.suffix.lower() == ".srt":
        rows.append({
            "relative_path": str(path.relative_to(INCOMING)),
            "filename": path.name,
            "size_kb": round(path.stat().st_size / 1024, 1)
        })

inventory = pd.DataFrame(rows)

print("Total subtitle files:", len(inventory))
display(inventory.to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"

manifest = pd.read_csv(MANIFEST)
master = pd.read_csv(MASTER)

files = sorted(
    p for p in INCOMING.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
)

# Extract movie ID prefixes from paths and filenames.
# Special case: Sooryavanshi was accidentally named with the Pushpa prefix.
rows = []
for p in files:
    text = str(p.relative_to(INCOMING))
    match = re.search(r"((?:201[7-9]|202[0-5])_[1-5])_", text)
    record_id = match.group(1) if match else None

    if (
        record_id == "2021_1"
        and "sooryavanshi" in text.lower()
    ):
        record_id = "2021_2"

    rows.append({
        "movie_record_id": record_id,
        "relative_path": text,
        "filename": p.name,
        "size_kb": round(p.stat().st_size / 1024, 1)
    })

inventory = pd.DataFrame(rows)

# Print all incoming files, without truncation.
print("=== COMPLETE INCOMING FILE INVENTORY ===")
print(inventory.to_string(index=False))

# Expected movie records from the master dataset.
id_col = "movie_record_id"
year_col = "year"

expected = master[
    pd.to_numeric(master[year_col], errors="coerce").between(2017, 2025)
].copy()

expected_ids = set(expected[id_col].astype(str))
incoming_ids = set(inventory["movie_record_id"].dropna().astype(str))

# 83 already exists in the manifest; don't expect another incoming file.
existing_83 = manifest[
    manifest[id_col].astype(str).eq("2021_4")
]

print("\n=== EXPECTED MOVIE RECORDS (2017–2025) ===")
print(expected[[id_col, "dataset_title", year_col]].to_string(index=False)
      if "dataset_title" in expected.columns
      else expected.to_string(index=False))

print("\n=== IDS WITHOUT AN INCOMING FILE PREFIX ===")
print(sorted(expected_ids - incoming_ids))

print("\n=== UNRECOGNIZED FILES ===")
print(
    inventory[inventory["movie_record_id"].isna()].to_string(index=False)
    if inventory["movie_record_id"].isna().any()
    else "None"
)

print("\n=== DUPLICATE MAPPED IDS ===")
print(
    inventory[inventory["movie_record_id"].notna()]
    .groupby("movie_record_id")
    .filter(lambda g: len(g) > 1)
    .to_string(index=False)
    if inventory["movie_record_id"].notna().any()
    else "None"
)

print("\n=== EXISTING VALIDATED 83 RECORD ===")
print(existing_83.to_string(index=False))

In [ ]:

from pathlib import Path
import re
import pandas as pd

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"

TIMESTAMP_RE = re.compile(
    r"(?P<sh>\d{2}):(?P<sm>\d{2}):(?P<ss>\d{2})[,.](?P<sms>\d{3})"
    r"\s*-->\s*"
    r"(?P<eh>\d{2}):(?P<em>\d{2}):(?P<es>\d{2})[,.](?P<ems>\d{3})"
)

def to_seconds(h, m, s, ms):
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def parse_timestamp(match):
    start = to_seconds(
        match["sh"], match["sm"], match["ss"], match["sms"]
    )
    end = to_seconds(
        match["eh"], match["em"], match["es"], match["ems"]
    )
    return start, end

def audit_srt(path):
    raw = path.read_bytes()
    encoding = None
    text = None

    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            text = raw.decode(enc)
            encoding = enc
            break
        except UnicodeDecodeError:
            pass

    matches = list(TIMESTAMP_RE.finditer(text))
    lines = text.splitlines()
    starts, ends = [], []
    malformed_timestamp_lines = sum(
        "-->" in line and not TIMESTAMP_RE.search(line)
        for line in lines
    )

    for match in matches:
        start, end = parse_timestamp(match.groupdict())
        starts.append(start)
        ends.append(end)

    # Collect text lines that follow timestamp lines until the next cue.
    dialogue_lines = []
    for i, line in enumerate(lines):
        if TIMESTAMP_RE.search(line):
            j = i + 1
            while j < len(lines) and not TIMESTAMP_RE.search(lines[j]):
                candidate = lines[j].strip()
                if candidate and not re.fullmatch(r"\d+", candidate):
                    candidate = re.sub(r"<[^>]+>", "", candidate)
                    if candidate:
                        dialogue_lines.append(candidate)
                j += 1

    words = re.findall(r"\b[\w’'-]+\b", " ".join(dialogue_lines))
    invalid_durations = sum(e <= s for s, e in zip(starts, ends))
    overlaps = sum(
        starts[i + 1] < ends[i]
        for i in range(len(starts) - 1)
    )
    gaps = [
        starts[i + 1] - ends[i]
        for i in range(len(starts) - 1)
        if starts[i + 1] > ends[i]
    ]

    return {
        "file": str(path.relative_to(INCOMING)),
        "size_kb": round(len(raw) / 1024, 1),
        "encoding": encoding,
        "timestamp_blocks": len(matches),
        "dialogue_words": len(words),
        "dialogue_chars": len(" ".join(dialogue_lines)),
        "duration_hours": round(max(ends) / 3600, 2) if ends else None,
        "malformed_timestamp_lines": malformed_timestamp_lines,
        "invalid_durations": invalid_durations,
        "overlaps": overlaps,
        "gaps_over_120s": sum(g > 120 for g in gaps),
        "largest_gap_seconds": round(max(gaps), 1) if gaps else 0,
        "empty_or_unparseable": len(matches) == 0 or len(words) == 0,
    }

files = sorted(
    p for p in INCOMING.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
)

results = pd.DataFrame([audit_srt(p) for p in files])

print("Files audited:", len(results))
print("\n=== FILES NEEDING REVIEW ===")

review = results[
    results["empty_or_unparseable"]
    | (results["malformed_timestamp_lines"] > 0)
    | (results["invalid_durations"] > 0)
    | (results["overlaps"] > 0)
    | (results["gaps_over_120s"] > 10)
    | (results["duration_hours"].fillna(0) > 5)
    | (results["dialogue_words"] < 1000)
]

display(review.to_string(index=False))

print("\n=== COMPLETE AUDIT SUMMARY ===")
display(
    results[
        [
            "file", "encoding", "timestamp_blocks",
            "dialogue_words", "duration_hours",
            "malformed_timestamp_lines", "invalid_durations",
            "overlaps", "gaps_over_120s", "largest_gap_seconds"
        ]
    ].to_string(index=False)
)


In [ ]:

from pathlib import Path
import re

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"

path = INCOMING / "SUBDL__master-2021-2431550" / "2021_3_MASTER_TAMIL.srt.srt"
text = path.read_text(encoding="utf-8-sig")
lines = text.splitlines()

SHORT_RE = re.compile(
    r"(?P<sm>\d{2}):(?P<ss>\d{2}),(?P<sms>\d{3})"
    r"\s*-->\s*"
    r"(?P<em>\d{2}):(?P<es>\d{2}),(?P<ems>\d{3})"
)

def short_seconds(mm, ss, ms):
    return int(mm) * 60 + int(ss) + int(ms) / 1000

valid = []
invalid_arrow_lines = []

for line_number, line in enumerate(lines, start=1):
    if "-->" not in line:
        continue

    match = SHORT_RE.fullmatch(line.strip())
    if not match:
        invalid_arrow_lines.append((line_number, line))
        continue

    d = match.groupdict()
    start = short_seconds(d["sm"], d["ss"], d["sms"])
    end = short_seconds(d["em"], d["es"], d["ems"])
    valid.append((line_number, start, end))

nonpositive = [(ln, s, e) for ln, s, e in valid if e <= s]
decreases = [
    (valid[i][0], valid[i][1], valid[i - 1][1])
    for i in range(1, len(valid))
    if valid[i][1] < valid[i - 1][1]
]

print("Timestamp lines:", sum("-->" in line for line in lines))
print("Short-format timestamps parsed:", len(valid))
print("Unparsed arrow lines:", len(invalid_arrow_lines))
print("Non-positive cue durations:", len(nonpositive))
print("Backward timestamp jumps:", len(decreases))

print("\nFirst 10 parsed cues:")
for item in valid[:10]:
    print(item)

print("\nFirst 10 unparsed arrow lines:")
for item in invalid_arrow_lines[:10]:
    print(item)

print("\nLargest timestamp in seconds:",
      round(max((e for _, _, e in valid), default=0), 2))


In [ ]:

from pathlib import Path
import re

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"

path = INCOMING / "SUBDL__master-2021-2431550" / "2021_3_MASTER_TAMIL.srt.srt"
lines = path.read_text(encoding="utf-8-sig").splitlines()

TIME = r"(?:\d{1,2}:)?\d{2}:\d{2}[,.]\d{3}"
FULL_RE = re.compile(
    rf"(?P<start>{TIME})\s*-->\s*(?P<end>{TIME})"
)

def parse_time(value):
    value = value.replace(",", ".")
    parts = value.split(":")
    if len(parts) == 2:
        minutes, seconds = parts
        return int(minutes) * 60 + float(seconds)
    if len(parts) == 3:
        hours, minutes, seconds = parts
        return int(hours) * 3600 + int(minutes) * 60 + float(seconds)
    raise ValueError(f"Unexpected timestamp: {value}")

valid = []
unparsed = []

for line_no, line in enumerate(lines, start=1):
    if "-->" not in line:
        continue

    match = FULL_RE.fullmatch(line.strip())
    if not match:
        unparsed.append((line_no, line))
        continue

    start = parse_time(match["start"])
    end = parse_time(match["end"])
    valid.append((line_no, start, end))

nonpositive = [(ln, s, e) for ln, s, e in valid if e <= s]
backward = [
    (valid[i][0], valid[i][1], valid[i - 1][1])
    for i in range(1, len(valid))
    if valid[i][1] < valid[i - 1][1]
]
overlaps = [
    (valid[i][0], valid[i][1], valid[i - 1][2])
    for i in range(1, len(valid))
    if valid[i][1] < valid[i - 1][2]
]

print("Timestamp lines:", sum("-->" in line for line in lines))
print("Successfully parsed:", len(valid))
print("Unparsed timestamp lines:", len(unparsed))
print("Non-positive cue durations:", len(nonpositive))
print("Backward timestamp jumps:", len(backward))
print("Overlapping cues:", len(overlaps))
print("First cue:", valid[0] if valid else None)
print("Last cue:", valid[-1] if valid else None)
print(
    "Timeline duration (hours):",
    round(max((end for _, _, end in valid), default=0) / 3600, 2)
)

if unparsed:
    print("\nFirst 10 unparsed lines:")
    for item in unparsed[:10]:
        print(item)

if backward:
    print("\nFirst 10 backward jumps:")
    for item in backward[:10]:
        print(item)


In [ ]:
import re

TIME_PATTERN = r"(?:\d{1,2}:)?\d{2}:\d{2}[,.]\d{3}"
TIMESTAMP_PATTERN = re.compile(
    rf"(?P<start>{TIME_PATTERN})\s*-->\s*(?P<end>{TIME_PATTERN})"
)

def parse_timestamp_seconds(value):
    value = value.replace(",", ".")
    parts = value.split(":")

    if len(parts) == 2:
        minutes, seconds = parts
        return int(minutes) * 60 + float(seconds)

    if len(parts) == 3:
        hours, minutes, seconds = parts
        return int(hours) * 3600 + int(minutes) * 60 + float(seconds)

    raise ValueError(f"Unsupported timestamp: {value}")

In [ ]:

from pathlib import Path
import re
import pandas as pd

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"

TIME_PATTERN = r"(?:\d{1,2}:)?\d{2}:\d{2}[,.]\d{3}"
TIMESTAMP_PATTERN = re.compile(
    rf"(?P<start>{TIME_PATTERN})\s*-->\s*(?P<end>{TIME_PATTERN})"
)

def parse_timestamp_seconds(value):
    value = value.replace(",", ".")
    parts = value.split(":")
    if len(parts) == 2:
        minutes, seconds = parts
        return int(minutes) * 60 + float(seconds)
    if len(parts) == 3:
        hours, minutes, seconds = parts
        return int(hours) * 3600 + int(minutes) * 60 + float(seconds)
    raise ValueError(f"Unsupported timestamp: {value}")

def audit_srt(path):
    raw = path.read_bytes()
    encoding = None
    text = None

    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            text = raw.decode(enc)
            encoding = enc
            break
        except UnicodeDecodeError:
            continue

    lines = text.splitlines()
    cues = []
    malformed = []

    for line_no, line in enumerate(lines, start=1):
        if "-->" not in line:
            continue

        match = TIMESTAMP_PATTERN.fullmatch(line.strip())
        if not match:
            malformed.append((line_no, line))
            continue

        start = parse_timestamp_seconds(match["start"])
        end = parse_timestamp_seconds(match["end"])
        cues.append((line_no, start, end))

    # Extract dialogue text between timestamp lines.
    dialogue = []
    for i, line in enumerate(lines):
        if TIMESTAMP_PATTERN.fullmatch(line.strip()):
            j = i + 1
            while j < len(lines) and not TIMESTAMP_PATTERN.fullmatch(lines[j].strip()):
                candidate = lines[j].strip()
                if candidate and "-->" not in candidate:
                    candidate = re.sub(r"<[^>]+>", "", candidate)
                    candidate = re.sub(r"\{\\an\d+\}", "", candidate)
                    if candidate and not re.fullmatch(r"\d+", candidate):
                        dialogue.append(candidate)
                j += 1

    words = re.findall(r"\b[\w’'-]+\b", " ".join(dialogue))
    starts = [c[1] for c in cues]
    ends = [c[2] for c in cues]

    invalid_durations = sum(end <= start for _, start, end in cues)
    backward_jumps = sum(
        starts[i] < starts[i - 1]
        for i in range(1, len(starts))
    )
    overlaps = sum(
        starts[i] < ends[i - 1]
        for i in range(1, len(cues))
    )
    gaps = [
        starts[i] - ends[i - 1]
        for i in range(1, len(cues))
        if starts[i] > ends[i - 1]
    ]

    return {
        "file": str(path.relative_to(INCOMING)),
        "size_kb": round(len(raw) / 1024, 1),
        "encoding": encoding,
        "timestamp_blocks": len(cues),
        "dialogue_words": len(words),
        "duration_hours": round(max(ends) / 3600, 2) if ends else None,
        "malformed_timestamp_lines": len(malformed),
        "invalid_durations": invalid_durations,
        "backward_jumps": backward_jumps,
        "overlaps": overlaps,
        "gaps_over_120s": sum(gap > 120 for gap in gaps),
        "largest_gap_seconds": round(max(gaps), 1) if gaps else 0,
        "needs_review": (
            not cues
            or not words
            or len(malformed) > 0
            or invalid_durations > 0
            or backward_jumps > 0
            or overlaps > 0
            or sum(gap > 120 for gap in gaps) > 10
            or (max(ends) / 3600 > 5 if ends else True)
            or len(words) < 1000
        ),
    }

files = sorted(
    p for p in INCOMING.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
)

results = pd.DataFrame([audit_srt(path) for path in files])

print("Files audited:", len(results))
print("Files needing review:", int(results["needs_review"].sum()))

print("\n=== FILES NEEDING REVIEW ===")
review = results[results["needs_review"]]
print(review.to_string(index=False) if not review.empty else "None")

print("\n=== AUDIT SUMMARY ===")
print(
    results[
        [
            "file", "timestamp_blocks", "dialogue_words",
            "duration_hours", "malformed_timestamp_lines",
            "invalid_durations", "backward_jumps", "overlaps",
            "gaps_over_120s", "largest_gap_seconds", "needs_review"
        ]
    ].to_string(index=False)
)


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"
VALIDATED = WORK / "validated"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(MANIFEST)

print("=== PATH CHECK ===")
print("Incoming exists:", INCOMING.exists())
print("Validated directory exists:", VALIDATED.exists())
print("Manifest exists:", MANIFEST.exists())

print("\n=== MANIFEST CHECK ===")
print("Current rows:", len(manifest))
print("Columns:", manifest.columns.tolist())
print("Duplicate movie IDs:")
print(
    manifest[manifest["movie_record_id"].duplicated(keep=False)]
    [["movie_record_id", "movie_title", "validation_status"]]
    .to_string(index=False)
    if manifest["movie_record_id"].duplicated().any()
    else "None"
)

print("\n=== EXISTING VALIDATED FILE CHECK ===")
for movie_id in ["2021_1", "2021_2", "2021_3", "2021_4", "2021_5"]:
    rows = manifest[manifest["movie_record_id"].astype(str) == movie_id]
    if rows.empty:
        print(movie_id, "-> no manifest record")
    else:
        for _, row in rows.iterrows():
            print(
                movie_id,
                "->",
                row["movie_title"],
                "| status:",
                row["validation_status"],
                "| validated file:",
                row["validated_subtitle_file"]
            )

print("\n=== INCOMING / VALIDATED COUNTS ===")
print("Incoming SRT files:", sum(
    1 for p in INCOMING.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
))
print("Existing validated SRT files:", sum(
    1 for p in VALIDATED.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
))

In [ ]:

from pathlib import Path
import pandas as pd
import re

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"
VALIDATED = WORK / "validated"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(MANIFEST)
incoming_files = sorted(
    p for p in INCOMING.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
)
existing_validated = {
    p.resolve(): p
    for p in VALIDATED.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
}
existing_names = {p.name.lower() for p in existing_validated.values()}

# Proposed standard output names; source language remains in manifest metadata.
def get_movie_id(path):
    match = re.search(
        r"((?:201[7-9]|202[0-5])_[1-5])_",
        str(path.relative_to(INCOMING))
    )
    if not match:
        return None
    movie_id = match.group(1)
    if movie_id == "2021_1" and "sooryavanshi" in str(path).lower():
        return "2021_2"
    return movie_id

def safe_title(value):
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_")
    return value.upper()

rows = []
for path in incoming_files:
    movie_id = get_movie_id(path)
    if movie_id is None:
        rows.append({
            "movie_record_id": None,
            "incoming_file": str(path.relative_to(INCOMING)),
            "proposed_validated_file": None,
            "collision": "UNMAPPED"
        })
        continue

    movie_rows = manifest[
        manifest["movie_record_id"].astype(str).eq(movie_id)
    ]
    if not movie_rows.empty:
        title = movie_rows.iloc[0]["movie_title"]
    else:
        # The master dataset supplies titles for new records.
        master = pd.read_csv(
            PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
        )
        match = master[
            master["movie_record_id"].astype(str).eq(movie_id)
        ]
        title = match.iloc[0]["movie_title"] if not match.empty else movie_id

    proposed = f"{movie_id}_{safe_title(title)}_ENGLISH.srt"
    collision = "YES" if proposed.lower() in existing_names else "NO"

    rows.append({
        "movie_record_id": movie_id,
        "incoming_file": str(path.relative_to(INCOMING)),
        "proposed_validated_file": proposed,
        "collision": collision
    })

plan = pd.DataFrame(rows)

print("Incoming files:", len(incoming_files))
print("Mapped files:", plan["movie_record_id"].notna().sum())
print("\n=== PROPOSED OUTPUT COLLISIONS ===")
print(
    plan[plan["collision"] != "NO"].to_string(index=False)
    if (plan["collision"] != "NO").any()
    else "No filename collisions detected."
)

print("\n=== EXISTING VALIDATED FILES NOT REFERENCED BY MANIFEST ===")
manifest_paths = set(
    str(p).strip()
    for p in manifest["validated_subtitle_file"].dropna()
    if str(p).strip()
)
unreferenced = [
    str(p) for p in existing_validated.values()
    if str(p) not in manifest_paths
]
print("Count:", len(unreferenced))
for item in unreferenced:
    print(item)


In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

master = pd.read_csv(MASTER)
manifest = pd.read_csv(MANIFEST)

def get_movie_id(path):
    match = re.search(
        r"((?:201[7-9]|202[0-5])_[1-5])_",
        str(path.relative_to(INCOMING))
    )
    if not match:
        return None

    movie_id = match.group(1)

    # Correct the user's accidental filename prefix for Sooryavanshi.
    if movie_id == "2021_1" and "sooryavanshi" in str(path).lower():
        movie_id = "2021_2"

    return movie_id

files = sorted(
    p for p in INCOMING.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
)

rows = []
for path in files:
    movie_id = get_movie_id(path)
    target = master[master["movie_record_id"].astype(str).eq(str(movie_id))]

    if target.empty:
        rows.append({
            "movie_record_id": movie_id,
            "movie_title": "UNMAPPED",
            "year": None,
            "original_language": None,
            "incoming_file": str(path.relative_to(INCOMING)),
            "subtitle_language": "English",
            "existing_manifest_record": False,
            "review_status": "UNMAPPED"
        })
        continue

    movie = target.iloc[0]
    existing = manifest[
        manifest["movie_record_id"].astype(str).eq(str(movie_id))
    ]

    rows.append({
        "movie_record_id": movie_id,
        "movie_title": movie["movie_title"],
        "year": int(movie["year"]),
        "original_language": movie["language"],
        "incoming_file": str(path.relative_to(INCOMING)),
        "subtitle_language": "English",
        "existing_manifest_record": not existing.empty,
        "review_status": (
            "REVIEW_EXISTING_RECORD"
            if not existing.empty
            else "PENDING_CONTENT_VERIFICATION"
        )
    })

plan = pd.DataFrame(rows)

print("Dry-run plan rows:", len(plan))
print("Unmapped files:", int(plan["movie_title"].eq("UNMAPPED").sum()))
print("\n=== BATCH PLAN ===")
print(plan.to_string(index=False))

print("\n=== REVIEW STATUS COUNTS ===")
print(plan["review_status"].value_counts().to_string())

print("\nNo files copied. No manifest changes made.")

In [ ]:

from pathlib import Path
from datetime import datetime
import pandas as pd
import re
import shutil

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_2017_2025" / "Extracted_Subtitles_2017_2025"
VALIDATED = WORK / "validated"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

master = pd.read_csv(MASTER)
manifest = pd.read_csv(MANIFEST)

def get_movie_id(path):
    match = re.search(
        r"((?:201[7-9]|202[0-5])_[1-5])_",
        str(path.relative_to(INCOMING))
    )
    if not match:
        return None

    movie_id = match.group(1)
    if movie_id == "2021_1" and "sooryavanshi" in str(path).lower():
        movie_id = "2021_2"

    return movie_id

def safe_title(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").upper()

files = sorted(
    p for p in INCOMING.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
)

# Prepare and validate the complete plan before copying.
plan = []
for src in files:
    movie_id = get_movie_id(src)
    if not movie_id:
        raise ValueError(f"Unmapped file: {src}")

    movie_rows = master[
        master["movie_record_id"].astype(str).eq(movie_id)
    ]
    if len(movie_rows) != 1:
        raise ValueError(
            f"Expected exactly one master record for {movie_id}; "
            f"found {len(movie_rows)}"
        )

    movie = movie_rows.iloc[0]
    dest = VALIDATED / f"{movie_id}_{safe_title(movie['movie_title'])}_ENGLISH.srt"

    if dest.exists():
        raise FileExistsError(f"Destination already exists: {dest}")

    plan.append({
        "movie_record_id": movie_id,
        "movie_title": movie["movie_title"],
        "source": src,
        "destination": dest,
        "source_size": src.stat().st_size
    })

if len(plan) != 44:
    raise ValueError(f"Expected 44 files; found {len(plan)}")

if len({item["movie_record_id"] for item in plan}) != 44:
    raise ValueError("Duplicate movie IDs detected in copy plan")

# Backup the active manifest before any filesystem changes.
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = MANIFEST.with_name(
    f"{MANIFEST.stem}_backup_before_manual_batch_{stamp}{MANIFEST.suffix}"
)
shutil.copy2(MANIFEST, backup)

# Copy without modifying or deleting the originals.
copied = []
try:
    for item in plan:
        shutil.copy2(item["source"], item["destination"])

        if item["destination"].stat().st_size != item["source_size"]:
            raise IOError(f"Size verification failed: {item['destination']}")

        copied.append(item)

except Exception:
    # Roll back only the copies created by this cell.
    for item in copied:
        if item["destination"].exists():
            item["destination"].unlink()
    raise

report = pd.DataFrame([
    {
        "movie_record_id": item["movie_record_id"],
        "movie_title": item["movie_title"],
        "source_file": str(item["source"]),
        "validated_copy": str(item["destination"]),
        "size_bytes": item["source_size"],
        "copy_status": "COPIED_SIZE_VERIFIED",
        "content_verification": "PENDING"
    }
    for item in copied
])

report_path = WORK / f"manual_batch_copy_report_{stamp}.csv"
report.to_csv(report_path, index=False)

print("Copies created:", len(copied))
print("Manifest backup:", backup)
print("Copy report:", report_path)
print("Active manifest modified: NO")
print("Incoming originals modified: NO")
print("\nCopy status:")
print(report["copy_status"].value_counts().to_string())
print("\nContent verification remains pending for all 44 files.")


In [ ]:

from pathlib import Path
import pandas as pd
import hashlib

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
REPORT = WORK / "manual_batch_copy_report_20261028_192705.csv"

report = pd.read_csv(REPORT)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

results = []

for _, row in report.iterrows():
    source = Path(row["source_file"])
    copy = Path(row["validated_copy"])

    if not source.exists() or not copy.exists():
        results.append({
            "movie_record_id": row["movie_record_id"],
            "source_exists": source.exists(),
            "copy_exists": copy.exists(),
            "hash_match": False,
            "decodable": False,
            "status": "MISSING_FILE"
        })
        continue

    source_hash = sha256(source)
    copy_hash = sha256(copy)

    decodable = False
    encoding = None

    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            copy.read_text(encoding=enc)
            encoding = enc
            decodable = True
            break
        except UnicodeDecodeError:
            continue

    hash_match = source_hash == copy_hash

    results.append({
        "movie_record_id": row["movie_record_id"],
        "source_exists": True,
        "copy_exists": True,
        "hash_match": hash_match,
        "decodable": decodable,
        "encoding": encoding,
        "status": (
            "COPY_INTEGRITY_PASSED"
            if hash_match and decodable
            else "REVIEW_REQUIRED"
        )
    })

check = pd.DataFrame(results)

print("Files checked:", len(check))
print("\n=== INTEGRITY SUMMARY ===")
print(check["status"].value_counts().to_string())

print("\n=== FILES NEEDING REVIEW ===")
issues = check[check["status"] != "COPY_INTEGRITY_PASSED"]
print(issues.to_string(index=False) if not issues.empty else "None")

print("\nContent-level movie and language verification remains pending.")


In [ ]:

from pathlib import Path
import pandas as pd
import re
import html
from datetime import datetime

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
COPY_REPORT = WORK / "manual_batch_copy_report_20261028_192705.csv"

OUT = WORK / f"manual_batch_content_audit_{datetime.now():%Y%m%d_%H%M%S}.csv"

report = pd.read_csv(COPY_REPORT)

# Confirm expected columns before auditing.
required = {"movie_record_id", "source_file", "validated_copy"}
missing = required - set(report.columns)
if missing:
    raise ValueError(
        f"Copy report is missing columns: {sorted(missing)}. "
        f"Available columns: {list(report.columns)}"
    )

def timestamp_seconds(value):
    value = value.strip().replace(".", ",")
    parts = value.split(":")
    if len(parts) not in (2, 3):
        return None
    try:
        if len(parts) == 2:  # MM:SS,mmm
            minutes, sec_ms = parts
            seconds, millis = sec_ms.split(",")
            return int(minutes) * 60 + int(seconds) + int(millis) / 1000
        hours, minutes, sec_ms = parts
        seconds, millis = sec_ms.split(",")
        return (
            int(hours) * 3600
            + int(minutes) * 60
            + int(seconds)
            + int(millis) / 1000
        )
    except (ValueError, TypeError):
        return None

def audit_srt(path):
    result = {
        "file_exists": path.exists(),
        "encoding": None,
        "cue_count": 0,
        "valid_timestamp_cues": 0,
        "malformed_timestamp_lines": 0,
        "invalid_durations": 0,
        "overlaps": 0,
        "backward_jumps": 0,
        "dialogue_blocks": 0,
        "word_count": 0,
        "character_count": 0,
        "unique_text_ratio": None,
        "timeline_hours": None,
        "gaps_over_120s": 0,
        "largest_gap_seconds": None,
        "likely_english": None,
        "structural_status": "REVIEW_REQUIRED",
        "review_reasons": ""
    }

    if not path.exists():
        result["review_reasons"] = "Missing validated copy"
        return result

    text = None
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            text = path.read_text(encoding=enc)
            result["encoding"] = enc
            break
        except UnicodeDecodeError:
            continue

    if text is None:
        result["review_reasons"] = "Could not decode file"
        return result

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = re.split(r"\n\s*\n", text.strip())
    cue_pattern = re.compile(
        r"^\s*(\d+)\s*\n"
        r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})"
        r"\s*-->\s*"
        r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})"
        r"(?:\s+.*)?\n(.*)$",
        re.S
    )

    parsed = []
    malformed = 0
    dialogue_blocks = 0
    all_text = []

    for block in blocks:
        match = cue_pattern.match(block)
        if not match:
            # Ignore a trailing WEBVTT header, but flag other non-empty blocks.
            if block.strip() and not block.strip().upper().startswith("WEBVTT"):
                malformed += 1
            continue

        start = timestamp_seconds(match.group(2))
        end = timestamp_seconds(match.group(3))
        dialogue = re.sub(r"<[^>]+>", " ", match.group(4))
        dialogue = html.unescape(dialogue)
        dialogue = re.sub(r"\{[^}]*\}", " ", dialogue)
        dialogue = re.sub(r"\s+", " ", dialogue).strip()

        if start is None or end is None:
            malformed += 1
            continue

        parsed.append((start, end))
        result["cue_count"] += 1

        if end <= start:
            result["invalid_durations"] += 1

        if dialogue:
            dialogue_blocks += 1
            all_text.append(dialogue)

    parsed.sort(key=lambda x: x[0])
    previous_start = None
    previous_end = None
    gaps = []

    for start, end in parsed:
        if previous_start is not None and start < previous_start:
            result["backward_jumps"] += 1
        if previous_end is not None:
            if start < previous_end:
                result["overlaps"] += 1
            elif start - previous_end > 120:
                gaps.append(start - previous_end)
        previous_start = start
        previous_end = max(previous_end or end, end)

    full_text = " ".join(all_text)
    words = re.findall(r"\b[\w’'-]+\b", full_text, flags=re.UNICODE)
    normalized_words = [w.casefold() for w in words if len(w) > 1]

    result["valid_timestamp_cues"] = len(parsed)
    result["malformed_timestamp_lines"] = malformed
    result["dialogue_blocks"] = dialogue_blocks
    result["word_count"] = len(words)
    result["character_count"] = len(full_text)
    result["unique_text_ratio"] = (
        round(len(set(normalized_words)) / len(normalized_words), 3)
        if normalized_words else 0
    )
    result["timeline_hours"] = (
        round(max(e for _, e in parsed) / 3600, 3) if parsed else 0
    )
    result["gaps_over_120s"] = len(gaps)
    result["largest_gap_seconds"] = round(max(gaps), 1) if gaps else 0

    # Heuristic only: this is not proof of subtitle language.
    latin_letters = len(re.findall(r"[A-Za-z]", full_text))
    non_latin_letters = len(re.findall(r"[^\W\d_]", full_text, flags=re.UNICODE)) - latin_letters
    english_markers = len(re.findall(
        r"\b(the|and|you|that|what|this|with|for|not|have|are|was|is)\b",
        full_text, flags=re.I
    ))
    result["likely_english"] = (
        latin_letters > 100 and english_markers >= 10
        and latin_letters >= max(1, non_latin_letters) * 2
    )

    reasons = []
    if result["cue_count"] < 100:
        reasons.append("Very few subtitle cues")
    if result["word_count"] < 500:
        reasons.append("Very low dialogue word count")
    if malformed:
        reasons.append("Malformed/non-SRT blocks")
    if result["invalid_durations"]:
        reasons.append("Invalid cue durations")
    if result["overlaps"]:
        reasons.append("Overlapping timestamps")
    if result["backward_jumps"]:
        reasons.append("Backward timestamp sequence")
    if result["gaps_over_120s"] > 10:
        reasons.append("Many gaps over 120 seconds")
    if result["unique_text_ratio"] is not None and result["unique_text_ratio"] < 0.15:
        reasons.append("Highly repetitive text")
    if not result["likely_english"]:
        reasons.append("Language needs review")

    result["review_reasons"] = "; ".join(reasons)
    result["structural_status"] = (
        "PASS_STRUCTURAL_SCREEN"
        if not reasons else "REVIEW_REQUIRED"
    )
    return result

results = []

for _, row in report.iterrows():
    copy_path = Path(str(row["validated_copy"]))
    audit = audit_srt(copy_path)
    results.append({
        "movie_record_id": row["movie_record_id"],
        "validated_copy": str(copy_path),
        **audit
    })

audit_df = pd.DataFrame(results)
audit_df.to_csv(OUT, index=False)

print("Files audited:", len(audit_df))
print("\n=== STRUCTURAL SCREEN ===")
print(audit_df["structural_status"].value_counts(dropna=False).to_string())

print("\n=== REVIEW REQUIRED ===")
review = audit_df[audit_df["structural_status"] != "PASS_STRUCTURAL_SCREEN"]
print(
    review[[
        "movie_record_id", "cue_count", "word_count",
        "gaps_over_120s", "largest_gap_seconds", "likely_english",
        "review_reasons"
    ]].to_string(index=False)
    if not review.empty else "None flagged by these screening rules."
)

print("\n=== AUDIT FILE ===")
print(OUT)

print("\nNOTE: A structural pass is not proof of movie identity or language.")
print("Active manifest modified: NO")


In [ ]:

from pathlib import Path
import pandas as pd
import re
import html
from collections import Counter

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
AUDIT = WORK / "manual_batch_content_audit_20261028_192907.csv"

audit = pd.read_csv(AUDIT)
flagged_ids = ["2021_3", "2018_3", "2021_1", "2022_3"]

for movie_id in flagged_ids:
    row = audit.loc[audit["movie_record_id"].astype(str) == movie_id]
    if row.empty:
        print(f"\n{movie_id}: NOT FOUND IN AUDIT")
        continue

    path = Path(row.iloc[0]["validated_copy"])
    print("\n" + "=" * 75)
    print("MOVIE ID:", movie_id)
    print("FILE:", path)
    print("EXISTS:", path.exists())

    if not path.exists():
        continue

    text = path.read_text(encoding="utf-8-sig", errors="replace")
    lines = text.splitlines()
    print("FILE SIZE:", path.stat().st_size, "bytes")
    print("TOTAL LINES:", len(lines))
    print("\n--- FIRST 25 LINES ---")
    print("\n".join(lines[:25]))
    print("\n--- LAST 12 LINES ---")
    print("\n".join(lines[-12:]))

    # Inspect subtitle text without changing the file.
    cue_text = []
    for line in lines:
        s = line.strip()
        if not s or "-->" in s or re.fullmatch(r"\d+", s):
            continue
        if s.upper().startswith("WEBVTT"):
            continue
        s = html.unescape(re.sub(r"<[^>]*>", " ", s))
        s = re.sub(r"\{[^}]*\}", " ", s).strip()
        if re.search(r"[A-Za-z\u0900-\u097F\u0B80-\u0BFF\u0C00-\u0C7F\u0C80-\u0CFF]", s):
            cue_text.append(s)

    joined = " ".join(cue_text)
    words = re.findall(r"\b[\w'-]+\b", joined.lower())
    counts = Counter(words)
    print("\nTEXT LINES:", len(cue_text))
    print("WORDS:", len(words))
    print("MOST COMMON WORDS:", counts.most_common(15))
    print("SAMPLE DIALOGUE:")
    print(" | ".join(cue_text[:8])[:1800])

print("\nDiagnostic only: no files or manifest modified.")


In [ ]:

from pathlib import Path
import pandas as pd
import re
import html
from datetime import datetime

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
AUDIT_FILE = WORK / "manual_batch_content_audit_20261028_192907.csv"

audit = pd.read_csv(AUDIT_FILE)

def seconds(ts):
    ts = ts.replace(".", ",")
    parts = ts.split(":")
    try:
        if len(parts) == 2:
            mm, rest = parts
            ss, ms = rest.split(",")
            return int(mm) * 60 + int(ss) + int(ms.ljust(3, "0")[:3]) / 1000
        if len(parts) == 3:
            hh, mm, rest = parts
            ss, ms = rest.split(",")
            return int(hh) * 3600 + int(mm) * 60 + int(ss) + int(ms.ljust(3, "0")[:3]) / 1000
    except (ValueError, TypeError):
        return None
    return None

def inspect_srt(path):
    text = path.read_text(encoding="utf-8-sig", errors="replace")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = re.split(r"\n\s*\n", text.strip())

    cues, malformed, invalid_duration, overlaps = [], 0, 0, 0
    dialogue = []

    for block in blocks:
        match = re.search(
            r"(?m)^\s*(\d+)\s*\n"
            r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})"
            r"\s*-->\s*"
            r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})"
            r"(?:\s+.*)?\s*\n(.*)$",
            block, flags=re.S
        )
        if not match:
            if block.strip():
                malformed += 1
            continue

        start, end = seconds(match.group(2)), seconds(match.group(3))
        if start is None or end is None:
            malformed += 1
            continue
        if end <= start:
            invalid_duration += 1
        cues.append((start, end))

        body = html.unescape(re.sub(r"<[^>]*>", " ", match.group(4)))
        body = re.sub(r"\{[^}]*\}", " ", body)
        body = re.sub(r"\s+", " ", body).strip()
        if body:
            dialogue.append(body)

    cues.sort(key=lambda x: x[0])
    for i in range(1, len(cues)):
        if cues[i][0] < cues[i - 1][1]:
            overlaps += 1

    full_text = " ".join(dialogue)
    words = re.findall(r"\b[\w'-]+\b", full_text, flags=re.UNICODE)

    # Unicode script indicators; metadata must still be checked against content.
    latin = len(re.findall(r"[A-Za-z]", full_text))
    devanagari = len(re.findall(r"[\u0900-\u097F]", full_text))
    tamil = len(re.findall(r"[\u0B80-\u0BFF]", full_text))
    telugu = len(re.findall(r"[\u0C00-\u0C7F]", full_text))
    kannada = len(re.findall(r"[\u0C80-\u0CFF]", full_text))

    scripts = {
        "English/Latin-script candidate": latin,
        "Hindi/Devanagari-script candidate": devanagari,
        "Tamil-script candidate": tamil,
        "Telugu-script candidate": telugu,
        "Kannada-script candidate": kannada,
    }
    likely_script = max(scripts, key=scripts.get) if max(scripts.values()) else "Undetermined"

    gaps = [
        cues[i][0] - cues[i - 1][1]
        for i in range(1, len(cues))
        if cues[i][0] >= cues[i - 1][1]
        and cues[i][0] - cues[i - 1][1] > 120
    ]

    return {
        "cue_count": len(cues),
        "malformed_blocks": malformed,
        "invalid_durations": invalid_duration,
        "overlaps": overlaps,
        "dialogue_blocks": len(dialogue),
        "word_count": len(words),
        "timeline_hours": round(max((e for _, e in cues), default=0) / 3600, 3),
        "gaps_over_120s": len(gaps),
        "largest_gap_seconds": round(max(gaps), 1) if gaps else 0,
        "latin_letters": latin,
        "devanagari_letters": devanagari,
        "tamil_letters": tamil,
        "telugu_letters": telugu,
        "kannada_letters": kannada,
        "script_indicator": likely_script,
    }

results = []
for _, row in audit.iterrows():
    path = Path(row["validated_copy"])
    record = {
        "movie_record_id": row["movie_record_id"],
        "validated_copy": str(path),
        "exists": path.exists(),
    }
    if path.exists():
        record.update(inspect_srt(path))
    else:
        record["review_note"] = "Missing file"
    results.append(record)

corrected = pd.DataFrame(results)
out = WORK / f"manual_batch_corrected_audit_{datetime.now():%Y%m%d_%H%M%S}.csv"
corrected.to_csv(out, index=False)

print("Files checked:", len(corrected))
print("\n=== FOUR FLAGGED FILES ===")
print(
    corrected[corrected["movie_record_id"].astype(str).isin(
        ["2021_3", "2018_3", "2021_1", "2022_3"]
    )].to_string(index=False)
)
print("\n=== ALL FILES: STRUCTURAL SUMMARY ===")
print("Missing files:", int((~corrected["exists"]).sum()))
print("Files with zero parsed cues:", int((corrected["cue_count"] == 0).sum()))
print("Files with malformed blocks:", int((corrected["malformed_blocks"] > 0).sum()))
print("Files with invalid durations:", int((corrected["invalid_durations"] > 0).sum()))
print("Files with overlaps:", int((corrected["overlaps"] > 0).sum()))
print("\nCorrected audit saved to:", out)
print("Active manifest modified: NO")


In [ ]:

from pathlib import Path
import pandas as pd
from datetime import datetime

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
AUDIT = WORK / "manual_batch_corrected_audit_20261028_193158.csv"

master = pd.read_csv(MASTER)
manifest = pd.read_csv(MANIFEST)
audit = pd.read_csv(AUDIT)

# Normalize IDs and protect against accidental duplicate records.
for df in (master, manifest, audit):
    df["movie_record_id"] = df["movie_record_id"].astype(str).str.strip()

if master["movie_record_id"].duplicated().any():
    raise ValueError("Duplicate movie IDs in master dataset. Stop for review.")

if manifest["movie_record_id"].duplicated().any():
    raise ValueError("Duplicate movie IDs in active manifest. Stop for review.")

if audit["movie_record_id"].duplicated().any():
    raise ValueError("Duplicate movie IDs in corrected audit. Stop for review.")

if len(audit) != 44:
    raise ValueError(f"Expected 44 audit records; found {len(audit)}.")

# Corrected language metadata grounded in the inspected subtitle text.
language_overrides = {
    "2018_3": "Hindi",
    "2021_3": "English",
    "2021_1": "English",
    "2022_3": "Tamil",
}

# Identity evidence is not inferred from language or filename alone.
identity_review = {
    "2018_3": "Content sample supports Padmaavat; retain manual identity review.",
    "2021_3": "Content sample supports Master; retain manual identity review.",
    "2021_1": "Content sample supports Pushpa: The Rise; encoding artifact noted.",
    "2022_3": "Tamil script and Vikram references support identity; retain manual review.",
}

master_ids = set(master["movie_record_id"])
manifest_ids = set(manifest["movie_record_id"])

preview = audit.copy()
preview["in_master"] = preview["movie_record_id"].isin(master_ids)
preview["already_in_manifest"] = preview["movie_record_id"].isin(manifest_ids)

preview["proposed_subtitle_language"] = "English"
preview.loc[
    preview["movie_record_id"].isin(language_overrides),
    "proposed_subtitle_language"
] = preview["movie_record_id"].map(language_overrides)

preview["proposed_validation_status"] = "Structural_Pass_Content_Review_Pending"
preview["proposed_movie_identity_verified"] = False
preview["proposed_language_verified"] = False
preview["proposed_quality_status"] = "Structural_Pass_Review_Required"

# These files have direct content evidence, but identity verification is
# deliberately not promoted automatically.
for movie_id, note in identity_review.items():
    preview.loc[preview["movie_record_id"] == movie_id, "review_note"] = note

# Preserve a clear encoding-quality warning for Pushpa.
preview.loc[
    preview["movie_record_id"] == "2021_1",
    "proposed_quality_status"
] = "Structural_Pass_Encoding_Review"

# Catch structural issues and ID conflicts before considering any update.
preview["structural_checks_pass"] = (
    preview["exists"].fillna(False).astype(bool)
    & (preview["cue_count"] > 0)
    & (preview["malformed_blocks"] == 0)
    & (preview["invalid_durations"] == 0)
    & (preview["overlaps"] == 0)
)

preview["preview_decision"] = "REVIEW"
preview.loc[
    preview["in_master"]
    & ~preview["already_in_manifest"]
    & preview["structural_checks_pass"],
    "preview_decision"
] = "ELIGIBLE_FOR_REVIEW"

preview.loc[~preview["in_master"], "preview_decision"] = "STOP_NOT_IN_MASTER"
preview.loc[
    preview["already_in_manifest"], "preview_decision"
] = "STOP_ALREADY_IN_MANIFEST"
preview.loc[
    ~preview["structural_checks_pass"], "preview_decision"
] = "STOP_STRUCTURAL_ISSUE"

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out = WORK / f"manual_batch_manifest_update_preview_{stamp}.csv"
preview.to_csv(out, index=False)

print("Preview records:", len(preview))
print("Master dataset rows:", len(master))
print("Current manifest rows:", len(manifest))
print("\n=== DECISION SUMMARY ===")
print(preview["preview_decision"].value_counts(dropna=False).to_string())

print("\n=== LANGUAGE OVERRIDES ===")
print(
    preview.loc[
        preview["movie_record_id"].isin(language_overrides),
        ["movie_record_id", "proposed_subtitle_language",
         "proposed_validation_status", "proposed_quality_status"]
    ].to_string(index=False)
)

print("\n=== STOP OR REVIEW RECORDS ===")
print(
    preview.loc[
        preview["preview_decision"] != "ELIGIBLE_FOR_REVIEW",
        ["movie_record_id", "in_master", "already_in_manifest",
         "structural_checks_pass", "preview_decision"]
    ].to_string(index=False)
)

print("\nPreview saved:", out)
print("Active manifest modified: NO")
print("Master dataset modified: NO")


In [ ]:

from pathlib import Path
import pandas as pd
import re
import html
from datetime import datetime

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
AUDIT = WORK / "manual_batch_corrected_audit_20261028_193158.csv"
COPY_REPORT = WORK / "manual_batch_copy_report_20261028_192705.csv"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

audit = pd.read_csv(AUDIT)
copies = pd.read_csv(COPY_REPORT)
master = pd.read_csv(MASTER)
manifest = pd.read_csv(MANIFEST)

for df in (audit, copies, master, manifest):
    if "movie_record_id" in df.columns:
        df["movie_record_id"] = df["movie_record_id"].astype(str).str.strip()

if len(audit) != 44:
    raise ValueError(f"Expected 44 audit rows, got {len(audit)}.")
if manifest["movie_record_id"].duplicated().any():
    raise ValueError("Existing manifest has duplicate movie IDs.")
if master["movie_record_id"].duplicated().any():
    raise ValueError("Master dataset has duplicate movie IDs.")

existing_ids = set(manifest["movie_record_id"])
master_ids = set(master["movie_record_id"])
audit_ids = set(audit["movie_record_id"])

if len(audit_ids) != 44:
    raise ValueError("Duplicate movie IDs found in audit.")
if not audit_ids.issubset(master_ids):
    raise ValueError("Some audit IDs are missing from master.")
if audit_ids & existing_ids:
    raise ValueError("Some batch IDs already exist in manifest; stop to avoid duplicates.")

def script_profile(path):
    text = path.read_text(encoding="utf-8-sig", errors="replace")
    text = html.unescape(re.sub(r"<[^>]*>", " ", text))
    text = re.sub(r"\{[^}]*\}", " ", text)

    counts = {
        "English/Latin": len(re.findall(r"[A-Za-z]", text)),
        "Hindi/Devanagari": len(re.findall(r"[\u0900-\u097F]", text)),
        "Tamil": len(re.findall(r"[\u0B80-\u0BFF]", text)),
        "Telugu": len(re.findall(r"[\u0C00-\u0C7F]", text)),
        "Kannada": len(re.findall(r"[\u0C80-\u0CFF]", text)),
    }
    ranked = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    top_name, top_count = ranked[0]
    total = sum(counts.values())
    share = top_count / total if total else 0

    # Script-based inference is a proposal, not absolute proof of language.
    inferred = top_name if total and share >= 0.60 else "MANUAL_REVIEW"
    return inferred, share, counts

rows = []
for _, a in audit.iterrows():
    movie_id = a["movie_record_id"]
    path = Path(a["validated_copy"])

    if not path.exists():
        raise FileNotFoundError(path)

    inferred, share, counts = script_profile(path)
    rows.append({
        "movie_record_id": movie_id,
        "validated_copy": str(path),
        "inferred_subtitle_language": inferred,
        "dominant_script_share": round(share, 3),
        **{f"script_{k}": v for k, v in counts.items()},
        "cue_count": int(a["cue_count"]),
        "word_count": int(a["word_count"]),
        "malformed_blocks": int(a["malformed_blocks"]),
        "invalid_durations": int(a["invalid_durations"]),
        "overlaps": int(a["overlaps"]),
    })

language_check = pd.DataFrame(rows)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
language_path = WORK / f"manual_batch_language_quality_review_{stamp}.csv"
language_check.to_csv(language_path, index=False)

print("=== INFERRED SUBTITLE LANGUAGES ===")
print(language_check["inferred_subtitle_language"].value_counts().to_string())

print("\n=== FILES REQUIRING LANGUAGE REVIEW ===")
print(
    language_check.loc[
        language_check["inferred_subtitle_language"] == "MANUAL_REVIEW",
        ["movie_record_id", "dominant_script_share", "word_count"]
    ].to_string(index=False)
)

print("\n=== STRUCTURAL EXCEPTIONS ===")
exceptions = language_check[
    (language_check["malformed_blocks"] > 0)
    | (language_check["invalid_durations"] > 0)
    | (language_check["overlaps"] > 0)
]
print(
    exceptions[["movie_record_id", "malformed_blocks",
                "invalid_durations", "overlaps"]].to_string(index=False)
    if not exceptions.empty else "None"
)

print("\nLanguage/quality review saved:", language_path)
print("Active manifest modified: NO")
print("Master dataset modified: NO")


In [ ]:

from pathlib import Path
import pandas as pd
import shutil
import html
import re
from datetime import datetime

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
AUDIT = WORK / "manual_batch_corrected_audit_20261028_193158.csv"
LANGUAGE_REVIEW = WORK / "manual_batch_language_quality_review_20261028_193715.csv"

manifest = pd.read_csv(MANIFEST)
master = pd.read_csv(MASTER)
audit = pd.read_csv(AUDIT)
lang = pd.read_csv(LANGUAGE_REVIEW)

for df in (manifest, master, audit, lang):
    df["movie_record_id"] = df["movie_record_id"].astype(str).str.strip()

# Pre-commit safety checks
if len(manifest) != 77:
    raise ValueError(
        f"Expected 77 existing manifest rows; found {len(manifest)}. "
        "Stop and inspect the active manifest."
    )
if manifest["movie_record_id"].duplicated().any():
    raise ValueError("Existing manifest has duplicate IDs.")
if master["movie_record_id"].duplicated().any():
    raise ValueError("Master dataset has duplicate IDs.")
if len(audit) != 44 or audit["movie_record_id"].duplicated().any():
    raise ValueError("Audit must contain exactly 44 unique IDs.")
if len(lang) != 44 or lang["movie_record_id"].duplicated().any():
    raise ValueError("Language review must contain exactly 44 unique IDs.")

existing_ids = set(manifest["movie_record_id"])
master_ids = set(master["movie_record_id"])
batch_ids = set(audit["movie_record_id"])

if batch_ids & existing_ids:
    raise ValueError("Batch IDs already exist in manifest; no changes made.")
if not batch_ids.issubset(master_ids):
    raise ValueError("Some batch IDs are missing from master; no changes made.")

lang_by_id = lang.set_index("movie_record_id")
audit_by_id = audit.set_index("movie_record_id")
master_by_id = master.set_index("movie_record_id")

explicit_language = {
    "2018_3": "Hindi",
    "2021_3": "English",
    "2021_1": "English",
    "2022_3": "Tamil",
}

def get_field(obj, options, default=""):
    for name in options:
        if name in obj.index and pd.notna(obj[name]):
            return obj[name]
    return default

new_rows = []

for movie_id in sorted(batch_ids, key=lambda x: (
    int(x.split("_")[0]), int(x.split("_")[1])
)):
    a = audit_by_id.loc[movie_id]
    l = lang_by_id.loc[movie_id]
    m = master_by_id.loc[movie_id]
    file_path = Path(a["validated_copy"])

    if not file_path.exists():
        raise FileNotFoundError(f"Missing validated subtitle: {file_path}")

    # Recalculate text metrics from the actual file.
    text = file_path.read_text(encoding="utf-8-sig", errors="replace")
    text = html.unescape(re.sub(r"<[^>]*>", " ", text))
    text = re.sub(r"\{[^}]*\}", " ", text)
    character_count = len(text)

    inferred = str(l["inferred_subtitle_language"])
    if movie_id in explicit_language:
        subtitle_language = explicit_language[movie_id]
        language_verified = movie_id in {"2018_3", "2021_3", "2022_3"}
    elif inferred == "Hindi/Devanagari":
        subtitle_language, language_verified = "Hindi", True
    elif inferred == "Tamil":
        subtitle_language, language_verified = "Tamil", True
    elif inferred == "English/Latin":
        subtitle_language, language_verified = "English", False
    else:
        subtitle_language, language_verified = "Unknown", False

    row = {col: "" for col in manifest.columns}
    row.update({
        "movie_record_id": movie_id,
        "year": get_field(m, ["year"]),
        "rank": get_field(m, ["rank"]),
        "movie_title": get_field(m, ["movie_title", "title"]),
        "language": get_field(m, ["language", "original_language"]),
        "subtitle_language": subtitle_language,
        "subtitle_type": "Full_movie",
        "subtitle_source": "SubDL_manual_download",
        "raw_subtitle_file": str(file_path),
        "validated_subtitle_file": str(file_path),
        "acquisition_status": "Acquired",
        "validation_status": "Structural_Pass_Content_Review_Pending",
        "title_verified": False,
        "year_verified": False,
        "movie_identity_verified": False,
        "language_verified": language_verified,
        "dialogue_blocks": int(a["dialogue_blocks"]),
        "word_count": int(a["word_count"]),
        "character_count": character_count,
        "encoding_status": (
            "Readable; encoding review recommended"
            if movie_id == "2021_1" else "Readable"
        ),
        "quality_status": (
            "Structural_Pass_Encoding_Review"
            if movie_id == "2021_1" else "Structural_Pass"
        ),
        "notes": "Structural checks passed; movie identity verification remains pending."
    })

    if movie_id == "2018_3":
        row["notes"] = "Hindi subtitle script confirmed; movie identity should receive final review."
    elif movie_id == "2021_3":
        row["notes"] = "English dialogue inspected; mixed timestamp formats handled by corrected parser."
    elif movie_id == "2021_1":
        row["notes"] = "Visible mojibake in punctuation; preserve original and review encoding separately."
    elif movie_id == "2022_3":
        row["notes"] = "Tamil subtitle script confirmed; movie identity should receive final review."

    new_rows.append(row)

new_df = pd.DataFrame(new_rows, columns=manifest.columns)
combined = pd.concat([manifest, new_df], ignore_index=True)

if len(combined) != 121:
    raise ValueError(f"Expected 121 rows, got {len(combined)}.")
if combined["movie_record_id"].duplicated().any():
    raise ValueError("Duplicate IDs in proposed manifest.")
if set(combined["movie_record_id"]) != existing_ids | batch_ids:
    raise ValueError("ID-set mismatch; active manifest not changed.")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = WORK / f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_commit_{stamp}.csv"
temp = WORK / f".manifest_commit_temp_{stamp}.csv"

shutil.copy2(MANIFEST, backup)
combined.to_csv(temp, index=False)

# Validate the temporary file before replacing the active manifest.
check = pd.read_csv(temp)
if len(check) != 121 or check["movie_record_id"].duplicated().any():
    temp.unlink(missing_ok=True)
    raise ValueError("Temporary manifest validation failed; active manifest untouched.")

temp.replace(MANIFEST)

# Verify the committed file.
final = pd.read_csv(MANIFEST)
assert len(final) == 121
assert not final["movie_record_id"].duplicated().any()
assert batch_ids.issubset(set(final["movie_record_id"]))

print("Manifest update: SUCCESS")
print("Previous rows:", len(manifest))
print("New records appended:", len(new_df))
print("Final rows:", len(final))
print("Duplicate IDs:", int(final["movie_record_id"].duplicated().sum()))
print("Backup:", backup)
print("Master dataset modified: NO")
print("Raw subtitle files modified: NO")
print("New-batch movie identity verification remains pending.")


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

master = pd.read_csv(MASTER)
manifest = pd.read_csv(MANIFEST)

for df in (master, manifest):
    df["movie_record_id"] = df["movie_record_id"].astype(str).str.strip()

if master["movie_record_id"].duplicated().any():
    raise ValueError("Master contains duplicate movie IDs.")
if manifest["movie_record_id"].duplicated().any():
    raise ValueError("Manifest contains duplicate movie IDs.")

master_ids = set(master["movie_record_id"])
manifest_ids = set(manifest["movie_record_id"])

# Keep the inventory separate from the master dataset.
inventory = master[["movie_record_id"]].copy()
inventory["has_manifest_record"] = inventory["movie_record_id"].isin(manifest_ids)

manifest_summary = (
    manifest.groupby("movie_record_id", as_index=False)
    .agg(
        manifest_rows=("movie_record_id", "size"),
        acquisition_status=("acquisition_status", "first"),
        validation_status=("validation_status", "first"),
        subtitle_language=("subtitle_language", "first"),
        quality_status=("quality_status", "first"),
        word_count=("word_count", "first"),
        identity_verified=("movie_identity_verified", "first"),
        language_verified=("language_verified", "first"),
    )
)

inventory = inventory.merge(
    manifest_summary, on="movie_record_id", how="left"
)

print("=== DATASET COVERAGE ===")
print("Master movies:", len(master))
print("Manifest rows:", len(manifest))
print("Unique manifest movie IDs:", manifest["movie_record_id"].nunique())
print("Master movies with subtitle manifest records:", int(inventory["has_manifest_record"].sum()))
print("Master movies without subtitle manifest records:", int((~inventory["has_manifest_record"]).sum()))
print("Duplicate manifest movie IDs:", int(manifest["movie_record_id"].duplicated().sum()))

print("\n=== MANIFEST RECORDS BY YEAR ===")
if "year" in manifest.columns:
    print(manifest.groupby("year").size().to_string())

print("\n=== SUBTITLE LANGUAGE COUNTS ===")
print(
    manifest["subtitle_language"].fillna("Missing")
    .replace("", "Missing").value_counts().to_string()
)

print("\n=== VALIDATION STATUS COUNTS ===")
print(
    manifest["validation_status"].fillna("Missing")
    .replace("", "Missing").value_counts().to_string()
)

print("\n=== MOVIES WITHOUT MANIFEST RECORDS ===")
missing = inventory.loc[~inventory["has_manifest_record"]]
if missing.empty:
    print("None")
else:
    cols = [c for c in ["movie_record_id"] if c in missing.columns]
    print(missing[cols].to_string(index=False))

print("\n=== NEW BATCH: PENDING IDENTITY OR LANGUAGE VERIFICATION ===")
pending = inventory[
    inventory["has_manifest_record"]
    & (
        inventory["identity_verified"].astype(str).str.lower().isin(["false", "0"])
        | inventory["language_verified"].astype(str).str.lower().isin(["false", "0"])
    )
]
print(
    pending[[
        "movie_record_id", "subtitle_language", "validation_status",
        "identity_verified", "language_verified"
    ]].to_string(index=False)
    if not pending.empty else "None"
)

out = WORK / "ICMB_130_SUBTITLE_CORPUS_COVERAGE_REPORT.csv"
inventory.to_csv(out, index=False)
print("\nCoverage report saved:", out)
print("Master dataset modified: NO")
print("Manifest modified: NO")


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

RAW = WORK / "raw"
VALIDATED = WORK / "validated"
SEARCH_RESULTS = WORK / "search_results"

master = pd.read_csv(MASTER, dtype={"movie_record_id": str})
manifest = pd.read_csv(MANIFEST, dtype={"movie_record_id": str})

master["movie_record_id"] = master["movie_record_id"].str.strip()
manifest["movie_record_id"] = manifest["movie_record_id"].str.strip()

manifest_ids = set(manifest["movie_record_id"])
missing = master[~master["movie_record_id"].isin(manifest_ids)].copy()

def get_files(folder):
    return (
        [p for p in folder.rglob("*") if p.is_file()]
        if folder.exists() else []
    )

raw_files = get_files(RAW)
validated_files = get_files(VALIDATED)
search_files = get_files(SEARCH_RESULTS)

def find_candidates(movie_id, title):
    # Search by record ID and normalized title tokens.
    tokens = [
        t.lower() for t in str(title).replace(":", " ").replace("-", " ").split()
        if len(t) >= 4
    ]
    candidates = []

    for category, files in [
        ("raw", raw_files),
        ("validated", validated_files),
        ("search_results", search_files),
    ]:
        for path in files:
            name = path.name.lower()
            if movie_id.lower() in name:
                candidates.append(f"{category}: {path.name}")
            elif tokens and sum(token in name for token in tokens) >= min(2, len(tokens)):
                candidates.append(f"{category}: {path.name}")

    return " | ".join(dict.fromkeys(candidates)) if candidates else "No filename match"

title_col = next(
    (c for c in ["movie_title", "title", "dataset_title"] if c in missing.columns),
    None
)

if title_col is None:
    raise ValueError(f"Could not identify movie title column. Columns: {list(missing.columns)}")

missing["local_candidates"] = missing.apply(
    lambda r: find_candidates(r["movie_record_id"], r[title_col]), axis=1
)

print("=== NINE MISSING MOVIE RECORDS ===")
print(
    missing[["movie_record_id", title_col, "local_candidates"]]
    .to_string(index=False)
)

print("\n=== FILE INVENTORY ===")
print("Raw files:", len(raw_files))
print("Validated files:", len(validated_files))
print("Search-result files:", len(search_files))
print("\nActive manifest modified: NO")
print("Master dataset modified: NO")


In [ ]:

from pathlib import Path
import pandas as pd
import re
import hashlib

# ============================================================
# ICMB-130 — AUDIT NEW MANUAL SUBTITLES
# Read-only: no files moved and no manifest changes
# ============================================================

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_missing_movies"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"

EXPECTED = {
    "2009_2": "2009_2_MAGADHEERA_ENGLISH.srt",
    "2009_3": "2009_3_LOVE_AAJ_KAL_ENGLISH.srt",
    "2009_4": "2009_4_AJAB_PREM_KI_GHAZAB_KAHANI_ENGLISH.srt",
    "2010_2": "2010_2_DABANGG_ENGLISH.srt",
    "2010_3": "2010_3_GOLMAAL_3_ENGLISH.srt",
    "2010_4": "2010_4_RAAJNEETI_ENGLISH.srt",
    "2012_5": "2012_5_HOUSEFULL_2_ENGLISH.srt",
    "2013_1": "2013_1_DHOOM_3_ENGLISH.srt",
    "2016_4": "2016_4_MS_DHONI_THE_UNTOLD_STORY_ENGLISH.srt",
}

assert INCOMING.exists(), f"Incoming folder not found: {INCOMING}"

# Find files recursively so a nested upload folder is also detected.
all_srt = list(INCOMING.rglob("*.srt"))
all_files_by_name = {}
for p in all_srt:
    all_files_by_name.setdefault(p.name.lower(), []).append(p)

def decode_subtitle(path):
    data = path.read_bytes()
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return data.decode(encoding), encoding, data
        except UnicodeDecodeError:
            continue
    return data.decode("utf-8", errors="replace"), "utf-8-replacement", data

def parse_timestamp(value):
    value = value.strip().replace(".", ",")
    parts = value.split(":")
    try:
        if len(parts) == 2:
            mm, rest = parts
            ss, ms = rest.split(",")
            return int(mm) * 60_000 + int(ss) * 1000 + int(ms.ljust(3, "0")[:3])
        if len(parts) == 3:
            hh, mm, rest = parts
            ss, ms = rest.split(",")
            return (int(hh) * 3600 + int(mm) * 60 + int(ss)) * 1000 + int(ms.ljust(3, "0")[:3])
    except Exception:
        return None
    return None

timestamp_re = re.compile(
    r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})\s*-->\s*"
    r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})"
)

rows = []

for record_id, expected_name in EXPECTED.items():
    matches = all_files_by_name.get(expected_name.lower(), [])
    if not matches:
        rows.append({
            "movie_record_id": record_id,
            "expected_filename": expected_name,
            "found_path": "",
            "file_size_bytes": None,
            "encoding": "",
            "sha256": "",
            "timestamp_blocks": 0,
            "malformed_blocks": None,
            "invalid_durations": None,
            "dialogue_blocks": 0,
            "word_count": 0,
            "language_script_hint": "",
            "content_preview": "",
            "audit_status": "MISSING_EXPECTED_FILENAME"
        })
        continue

    # If duplicate filenames exist, report every one for review.
    for path in matches:
        text, encoding, raw_bytes = decode_subtitle(path)
        blocks = re.split(r"\n\s*\n", text.replace("\r\n", "\n").replace("\r", "\n").strip())
        timestamps = list(timestamp_re.finditer(text))
        malformed = 0
        invalid_durations = 0

        for block in blocks:
            if "-->" in block and not timestamp_re.search(block):
                malformed += 1

        for match in timestamps:
            start = parse_timestamp(match.group(1))
            end = parse_timestamp(match.group(2))
            if start is None or end is None or end <= start:
                invalid_durations += 1

        # Remove SRT indices and timestamp lines to estimate dialogue.
        dialogue_lines = []
        for line in text.splitlines():
            stripped = line.strip()
            if not stripped or stripped.isdigit() or "-->" in stripped:
                continue
            if stripped.startswith(("WEBVTT", "NOTE ")):
                continue
            dialogue_lines.append(re.sub(r"<[^>]+>", "", stripped))

        dialogue = " ".join(dialogue_lines)
        words = re.findall(r"\b[\w’'-]+\b", dialogue, flags=re.UNICODE)
        devanagari = len(re.findall(r"[\u0900-\u097F]", dialogue))
        tamil = len(re.findall(r"[\u0B80-\u0BFF]", dialogue))
        latin = len(re.findall(r"[A-Za-z]", dialogue))

        if devanagari > 20:
            script_hint = "Devanagari detected — review language"
        elif tamil > 20:
            script_hint = "Tamil script detected — review language"
        elif latin > 100:
            script_hint = "Latin script dominant — English not proven"
        else:
            script_hint = "Language needs manual review"

        status = "STRUCTURE_REVIEW"
        if not timestamps:
            status = "NO_TIMESTAMP_CUES"
        elif malformed or invalid_durations:
            status = "STRUCTURAL_ISSUES"
        elif len(matches) > 1:
            status = "DUPLICATE_FILENAME_REVIEW"
        elif len(words) < 500:
            status = "LOW_DIALOGUE_VOLUME_REVIEW"
        else:
            status = "STRUCTURE_PASS_CONTENT_IDENTITY_REVIEW"

        rows.append({
            "movie_record_id": record_id,
            "expected_filename": expected_name,
            "found_path": str(path),
            "file_size_bytes": len(raw_bytes),
            "encoding": encoding,
            "sha256": hashlib.sha256(raw_bytes).hexdigest(),
            "timestamp_blocks": len(timestamps),
            "malformed_blocks": malformed,
            "invalid_durations": invalid_durations,
            "dialogue_blocks": len(dialogue_lines),
            "word_count": len(words),
            "language_script_hint": script_hint,
            "content_preview": re.sub(r"\s+", " ", dialogue)[:250],
            "audit_status": status
        })

audit = pd.DataFrame(rows)
display(audit)

report_path = WORK / "missing_movies_9_subtitles_initial_audit.csv"
audit.to_csv(report_path, index=False, encoding="utf-8-sig")

print("\nExpected movies:", len(EXPECTED))
print("SRT files found recursively:", len(all_srt))
print("Audit rows:", len(audit))
print("Audit report saved:", report_path)
print("\nThis was a read-only audit. No manifest or subtitle files were changed.")


In [ ]:

from pathlib import Path
import os

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
INCOMING = PROJECT / "01_data/processed/subtitle_corpus_v2/incoming_manual_missing_movies"

print("Folder exists:", INCOMING.exists())
print("Folder:", INCOMING)

if INCOMING.exists():
    print("\nAll files and subfolders:")
    for p in sorted(INCOMING.rglob("*")):
        if p.is_file():
            print(f"{p.name}  |  {p.stat().st_size:,} bytes")
        elif p.is_dir():
            print(f"[FOLDER] {p.relative_to(INCOMING)}")

    print("\nSRT files found:", len(list(INCOMING.rglob("*.srt"))))
    print("SRT files with uppercase extension:",
          len(list(INCOMING.rglob("*.SRT"))))
else:
    print("\nCheck that Google Drive is mounted and the folder name is correct.")


In [ ]:

from pathlib import Path
import shutil
import hashlib
import pandas as pd

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_missing_movies"

EXPECTED = {
    "2009_2": "2009_2_MAGADHEERA_ENGLISH.srt",
    "2009_3": "2009_3_LOVE_AAJ_KAL_ENGLISH.srt",
    "2009_4": "2009_4_AJAB_PREM_KI_GHAZAB_KAHANI_ENGLISH.srt",
    "2010_2": "2010_2_DABANGG_ENGLISH.srt",
    "2010_3": "2010_3_GOLMAAL_3_ENGLISH.srt",
    "2010_4": "2010_4_RAAJNEETI_ENGLISH.srt",
    "2012_5": "2012_5_HOUSEFULL_2_ENGLISH.srt",
    "2013_1": "2013_1_DHOOM_3_ENGLISH.srt",
    "2016_4": "2016_4_MS_DHONI_THE_UNTOLD_STORY_ENGLISH.srt",
}

# Find all subtitle files recursively, case-insensitive extension.
source_files = [
    p for p in INCOMING.rglob("*")
    if p.is_file() and p.suffix.lower() == ".srt"
]

# Match each movie using its record ID in the current filename.
results = []

for record_id, target_name in EXPECTED.items():
    matches = [p for p in source_files if p.name.startswith(record_id + "_")]

    if len(matches) != 1:
        results.append({
            "movie_record_id": record_id,
            "target_filename": target_name,
            "status": f"NOT COPIED — found {len(matches)} matching source files",
            "source": " | ".join(str(p) for p in matches)
        })
        continue

    source = matches[0]
    target = INCOMING / target_name

    # Do not overwrite an existing destination.
    if target.exists():
        source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
        target_hash = hashlib.sha256(target.read_bytes()).hexdigest()

        status = (
            "ALREADY EXISTS — identical"
            if source_hash == target_hash
            else "CONFLICT — destination exists with different content"
        )
    else:
        shutil.copy2(source, target)
        source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
        target_hash = hashlib.sha256(target.read_bytes()).hexdigest()

        status = (
            "COPIED AND VERIFIED"
            if source_hash == target_hash
            else "ERROR — hash mismatch"
        )

    results.append({
        "movie_record_id": record_id,
        "target_filename": target_name,
        "status": status,
        "source": str(source)
    })

report = pd.DataFrame(results)
display(report)

report_path = WORK / "missing_movies_9_standardization_report.csv"
report.to_csv(report_path, index=False, encoding="utf-8-sig")

print("\nCorrectly named files in incoming folder:")
for record_id, filename in EXPECTED.items():
    target = INCOMING / filename
    print(f"{'FOUND' if target.exists() else 'MISSING'}: {filename}")

print("\nReport saved to:", report_path)
print("Original extracted folders and files were not modified.")
print("Manifest and master dataset were not modified.")


In [ ]:

from pathlib import Path
import pandas as pd
import re, hashlib

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_missing_movies"
VALIDATED = WORK / "validated"
MASTER = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

FILES = {
    "2009_2": ("2009_2_MAGADHEERA_ENGLISH.srt", ["magadheera"]),
    "2009_3": ("2009_3_LOVE_AAJ_KAL_ENGLISH.srt", ["love aaj kal"]),
    "2009_4": ("2009_4_AJAB_PREM_KI_GHAZAB_KAHANI_ENGLISH.srt",
               ["ajab prem", "ghazab kahani"]),
    "2010_2": ("2010_2_DABANGG_ENGLISH.srt", ["dabangg"]),
    "2010_3": ("2010_3_GOLMAAL_3_ENGLISH.srt", ["golmaal"]),
    "2010_4": ("2010_4_RAAJNEETI_ENGLISH.srt", ["raajneeti", "rajneeti"]),
    "2012_5": ("2012_5_HOUSEFULL_2_ENGLISH.srt", ["housefull"]),
    "2013_1": ("2013_1_DHOOM_3_ENGLISH.srt", ["dhoom"]),
    "2016_4": ("2016_4_MS_DHONI_THE_UNTOLD_STORY_ENGLISH.srt",
               ["ms dhoni", "mahendra singh dhoni", "untold story"]),
}

master = pd.read_csv(MASTER)
manifest = pd.read_csv(MANIFEST)

def decode_file(path):
    raw = path.read_bytes()
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return raw.decode(enc), enc, raw
        except UnicodeDecodeError:
            pass
    return raw.decode("utf-8", errors="replace"), "utf-8-replacement", raw

def to_ms(ts):
    ts = ts.replace(".", ",")
    parts = ts.split(":")
    try:
        if len(parts) == 2:
            mm, rest = parts
            ss, ms = rest.split(",")
            return (int(mm)*60 + int(ss))*1000 + int(ms.ljust(3, "0")[:3])
        hh, mm, rest = parts
        ss, ms = rest.split(",")
        return (int(hh)*3600 + int(mm)*60 + int(ss))*1000 + int(ms.ljust(3, "0")[:3])
    except Exception:
        return None

time_re = re.compile(
    r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})\s*-->\s*"
    r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})"
)

results = []

for rid, (filename, title_terms) in FILES.items():
    path = INCOMING / filename
    row = {
        "movie_record_id": rid,
        "filename": filename,
        "file_exists": path.is_file(),
        "encoding": "",
        "sha256": "",
        "cue_count": 0,
        "malformed_blocks": None,
        "invalid_durations": None,
        "word_count": 0,
        "english_script_hint": False,
        "title_keyword_found": False,
        "audit_status": "MISSING_FILE",
    }

    if path.is_file():
        text, enc, raw = decode_file(path)
        normalized = text.replace("\r\n", "\n").replace("\r", "\n")
        blocks = re.split(r"\n\s*\n", normalized.strip())
        cues = list(time_re.finditer(normalized))

        malformed = sum(
            1 for b in blocks if "-->" in b and not time_re.search(b)
        )
        invalid = 0
        for m in cues:
            start, end = to_ms(m.group(1)), to_ms(m.group(2))
            if start is None or end is None or end <= start:
                invalid += 1

        dialogue = []
        for line in normalized.splitlines():
            s = line.strip()
            if not s or s.isdigit() or "-->" in s or s.upper() == "WEBVTT":
                continue
            dialogue.append(re.sub(r"<[^>]+>", "", s))

        body = " ".join(dialogue)
        words = re.findall(r"\b[\w’'-]+\b", body, flags=re.UNICODE)
        low = body.lower()
        title_found = any(term in low for term in title_terms)
        latin_count = len(re.findall(r"[A-Za-z]", body))
        dev_count = len(re.findall(r"[\u0900-\u097F]", body))
        tamil_count = len(re.findall(r"[\u0B80-\u0BFF]", body))

        if not cues:
            status = "FAIL_NO_TIMESTAMP_CUES"
        elif malformed or invalid:
            status = "REVIEW_STRUCTURAL_ISSUES"
        elif len(words) < 500:
            status = "REVIEW_LOW_WORD_COUNT"
        else:
            status = "STRUCTURE_PASS_CONTENT_REVIEW"

        row.update({
            "encoding": enc,
            "sha256": hashlib.sha256(raw).hexdigest(),
            "cue_count": len(cues),
            "malformed_blocks": malformed,
            "invalid_durations": invalid,
            "word_count": len(words),
            "english_script_hint": latin_count > 100 and dev_count < 20 and tamil_count < 20,
            "title_keyword_found": title_found,
            "content_preview": re.sub(r"\s+", " ", body)[:180],
            "audit_status": status,
        })

    # Existing manifest coverage for this movie record
    existing = manifest[manifest["movie_record_id"].astype(str) == rid]
    row["existing_manifest_rows"] = len(existing)

    # Verify record exists in master
    row["master_record_exists"] = (
        "movie_record_id" in master.columns
        and (master["movie_record_id"].astype(str) == rid).any()
    )
    results.append(row)

audit = pd.DataFrame(results)
display(audit)

report = WORK / "missing_movies_9_final_precommit_audit.csv"
audit.to_csv(report, index=False, encoding="utf-8-sig")

print("\nMaster rows:", len(master))
print("Manifest rows before update:", len(manifest))
print("Unique manifest movie IDs:", manifest["movie_record_id"].nunique())
print("Missing movie IDs in manifest:",
      sorted(set(FILES) - set(manifest["movie_record_id"].astype(str))))
print("Audit saved:", report)
print("No subtitle files, master data, or manifest were modified.")


In [ ]:

from pathlib import Path
import pandas as pd
import re
import hashlib

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_missing_movies"

FILES = {
    "2009_2": "2009_2_MAGADHEERA_ENGLISH.srt",
    "2009_3": "2009_3_LOVE_AAJ_KAL_ENGLISH.srt",
    "2009_4": "2009_4_AJAB_PREM_KI_GHAZAB_KAHANI_ENGLISH.srt",
    "2010_2": "2010_2_DABANGG_ENGLISH.srt",
    "2010_3": "2010_3_GOLMAAL_3_ENGLISH.srt",
    "2010_4": "2010_4_RAAJNEETI_ENGLISH.srt",
    "2012_5": "2012_5_HOUSEFULL_2_ENGLISH.srt",
    "2013_1": "2013_1_DHOOM_3_ENGLISH.srt",
    "2016_4": "2016_4_MS_DHONI_THE_UNTOLD_STORY_ENGLISH.srt",
}

def decode_srt(path):
    raw = path.read_bytes()
    if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
        return raw.decode("utf-16"), "utf-16-BOM", raw
    if raw.startswith(b"\xef\xbb\xbf"):
        return raw.decode("utf-8-sig"), "utf-8-BOM", raw
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            return raw.decode(enc), enc, raw
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace"), "utf-8-replacement", raw

time_re = re.compile(
    r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})\s*-->\s*"
    r"(\d{1,2}:\d{2}(?::\d{2})?[,.]\d{1,3})"
)

def to_ms(ts):
    ts = ts.replace(".", ",")
    p = ts.split(":")
    try:
        if len(p) == 2:
            mm, rest = p
            ss, ms = rest.split(",")
            return (int(mm)*60 + int(ss))*1000 + int(ms.ljust(3, "0")[:3])
        hh, mm, rest = p
        ss, ms = rest.split(",")
        return (int(hh)*3600 + int(mm)*60 + int(ss))*1000 + int(ms.ljust(3, "0")[:3])
    except Exception:
        return None

results = []

for rid, filename in FILES.items():
    path = INCOMING / filename
    text, encoding, raw = decode_srt(path)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = re.split(r"\n\s*\n", text.strip())
    cues = list(time_re.finditer(text))

    malformed = sum(1 for b in blocks if "-->" in b and not time_re.search(b))
    invalid = 0
    for m in cues:
        start, end = to_ms(m.group(1)), to_ms(m.group(2))
        if start is None or end is None or end <= start:
            invalid += 1

    dialogue = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s.isdigit() or "-->" in s or s.upper() == "WEBVTT":
            continue
        dialogue.append(re.sub(r"<[^>]+>", "", s))

    body = " ".join(dialogue)
    words = re.findall(r"\b[\w’'-]+\b", body, flags=re.UNICODE)
    latin = len(re.findall(r"[A-Za-z]", body))
    devanagari = len(re.findall(r"[\u0900-\u097F]", body))
    tamil = len(re.findall(r"[\u0B80-\u0BFF]", body))

    results.append({
        "movie_record_id": rid,
        "filename": filename,
        "encoding_detected": encoding,
        "bytes": len(raw),
        "sha256": hashlib.sha256(raw).hexdigest(),
        "timestamp_cues": len(cues),
        "malformed_blocks": malformed,
        "invalid_durations": invalid,
        "word_count": len(words),
        "latin_script_letters": latin,
        "devanagari_letters": devanagari,
        "tamil_letters": tamil,
        "preview": re.sub(r"\s+", " ", body)[:220],
        "structure_status": (
            "FAIL_NO_CUES" if not cues else
            "REVIEW_STRUCTURAL_ISSUES" if malformed or invalid else
            "REVIEW_LOW_TEXT_VOLUME" if len(words) < 500 else
            "STRUCTURE_PASS"
        )
    })

audit = pd.DataFrame(results)
display(audit)

report = WORK / "missing_movies_9_encoding_corrected_audit.csv"
audit.to_csv(report, index=False, encoding="utf-8-sig")
print("Report saved:", report)
print("Read-only audit: no subtitle, master, or manifest files changed.")


In [ ]:

from pathlib import Path
from datetime import datetime
import pandas as pd
import hashlib
import shutil
import re

# ============================================================
# ICMB-130 — FINAL SUBTITLE COMMIT
# Validated copies + backed-up manifest update
# ============================================================

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_missing_movies"
VALIDATED = WORK / "validated"
MASTER_PATH = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST_PATH = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
AUDIT_PATH = WORK / "missing_movies_9_encoding_corrected_audit.csv"

FILES = {
    "2009_2": ("2009_2_MAGADHEERA_ENGLISH.srt", "Magadheera"),
    "2009_3": ("2009_3_LOVE_AAJ_KAL_ENGLISH.srt", "Love Aaj Kal"),
    "2009_4": ("2009_4_AJAB_PREM_KI_GHAZAB_KAHANI_ENGLISH.srt",
               "Ajab Prem Ki Ghazab Kahani"),
    "2010_2": ("2010_2_DABANGG_ENGLISH.srt", "Dabangg"),
    "2010_3": ("2010_3_GOLMAAL_3_ENGLISH.srt", "Golmaal 3"),
    "2010_4": ("2010_4_RAAJNEETI_ENGLISH.srt", "Raajneeti"),
    "2012_5": ("2012_5_HOUSEFULL_2_ENGLISH.srt", "Housefull 2"),
    "2013_1": ("2013_1_DHOOM_3_ENGLISH.srt", "Dhoom 3"),
    "2016_4": ("2016_4_MS_DHONI_THE_UNTOLD_STORY_ENGLISH.srt",
               "M.S. Dhoni: The Untold Story"),
}

# Read current files and metadata.
assert INCOMING.exists(), f"Missing incoming folder: {INCOMING}"
assert MASTER_PATH.exists(), f"Missing master dataset: {MASTER_PATH}"
assert MANIFEST_PATH.exists(), f"Missing manifest: {MANIFEST_PATH}"
assert AUDIT_PATH.exists(), f"Missing corrected audit: {AUDIT_PATH}"

master = pd.read_csv(MASTER_PATH)
manifest = pd.read_csv(MANIFEST_PATH)
audit = pd.read_csv(AUDIT_PATH)

required_ids = set(FILES)
assert "movie_record_id" in master.columns
assert "movie_record_id" in manifest.columns
assert "movie_record_id" in audit.columns

master_ids = set(master["movie_record_id"].astype(str))
manifest_ids = set(manifest["movie_record_id"].astype(str))
audit_by_id = audit.set_index("movie_record_id")

assert required_ids.issubset(master_ids), "One or more IDs are absent from master."
assert not (required_ids & manifest_ids), (
    "Some target IDs are already in the manifest; stop to avoid duplicates."
)
assert required_ids.issubset(set(audit["movie_record_id"].astype(str))), (
    "Audit does not include all nine records."
)

# Confirm all nine passed structural checks.
for rid in required_ids:
    a = audit_by_id.loc[rid]
    assert a["structure_status"] == "STRUCTURE_PASS", (
        f"{rid} has not passed structure validation."
    )
    assert int(a["timestamp_cues"]) > 0
    assert int(a["malformed_blocks"]) == 0
    assert int(a["invalid_durations"]) == 0
    assert int(a["word_count"]) >= 500

# Confirm manifest schema matches the expected existing schema.
expected_columns = list(manifest.columns)
for col in [
    "movie_record_id", "year", "rank", "movie_title", "language",
    "tmdb_id", "imdb_id", "subtitle_language", "subtitle_type",
    "subtitle_source", "source_id", "source_url", "raw_subtitle_file",
    "validated_subtitle_file", "acquisition_status", "validation_status",
    "title_verified", "year_verified", "movie_identity_verified",
    "language_verified", "dialogue_blocks", "word_count", "character_count",
    "encoding_status", "quality_status", "notes"
]:
    assert col in expected_columns, f"Manifest column missing: {col}"

# Stage copies first. Originals in incoming remain untouched.
VALIDATED.mkdir(parents=True, exist_ok=True)
staged = []

for rid, (filename, title) in FILES.items():
    source = INCOMING / filename
    assert source.is_file(), f"Missing incoming subtitle: {source}"

    dest = VALIDATED / filename
    source_hash = hashlib.sha256(source.read_bytes()).hexdigest()

    if dest.exists():
        dest_hash = hashlib.sha256(dest.read_bytes()).hexdigest()
        assert dest_hash == source_hash, (
            f"Destination conflict for {filename}; no manifest update performed."
        )
    else:
        shutil.copy2(source, dest)
        dest_hash = hashlib.sha256(dest.read_bytes()).hexdigest()
        assert dest_hash == source_hash, f"Hash mismatch for {filename}"

    a = audit_by_id.loc[rid]
    staged.append({
        "movie_record_id": rid,
        "filename": filename,
        "source_path": str(source),
        "validated_path": str(dest),
        "sha256": source_hash,
        "encoding": str(a["encoding_detected"]),
        "cue_count": int(a["timestamp_cues"]),
        "word_count": int(a["word_count"]),
    })

# Ensure every staged validated file exists and matches source.
for item in staged:
    assert Path(item["validated_path"]).is_file()
    assert hashlib.sha256(Path(item["validated_path"]).read_bytes()).hexdigest() == item["sha256"]

# Get year/rank/title/language from the master, without modifying it.
master_index = master.copy()
master_index["movie_record_id"] = master_index["movie_record_id"].astype(str)

def get_master_value(row, names, default=""):
    for name in names:
        if name in row.index and pd.notna(row[name]):
            return row[name]
    return default

new_records = []
for item in staged:
    rid = item["movie_record_id"]
    mrow = master_index.loc[master_index["movie_record_id"] == rid].iloc[0]
    a = audit_by_id.loc[rid]

    record = {col: "" for col in expected_columns}
    record.update({
        "movie_record_id": rid,
        "year": get_master_value(mrow, ["year", "dataset_year"]),
        "rank": get_master_value(mrow, ["rank", "india_gross_rank", "rank_by_india_gross"]),
        "movie_title": get_master_value(mrow, ["movie_title", "dataset_title", "title"], FILES[rid][1]),
        "language": get_master_value(mrow, ["language", "dataset_language"]),
        "subtitle_language": "English",
        "subtitle_type": "Full_movie",
        "subtitle_source": "SubDL_manual_download",
        "source_id": "",
        "source_url": "",
        # Raw download is preserved at the incoming path; validated copy is separate.
        "raw_subtitle_file": item["source_path"],
        "validated_subtitle_file": item["validated_path"],
        "acquisition_status": "Acquired",
        "validation_status": "Structural_Pass_Content_Identity_Review_Pending",
        "title_verified": False,
        "year_verified": False,
        "movie_identity_verified": False,
        "language_verified": False,
        "dialogue_blocks": item["cue_count"],
        "word_count": item["word_count"],
        "character_count": len(Path(item["validated_path"]).read_text(
            encoding="utf-8", errors="replace"
        )),
        "encoding_status": item["encoding"],
        "quality_status": "Structural_Pass_Identity_Review_Pending",
        "notes": (
            "Structural validation passed. Movie identity, year, and language "
            "require content review before being marked verified. "
            f"SHA-256: {item['sha256']}"
        ),
    })
    new_records.append(record)

new_df = pd.DataFrame(new_records, columns=expected_columns)
updated = pd.concat([manifest, new_df], ignore_index=True)
updated["movie_record_id"] = updated["movie_record_id"].astype(str)

# Final safety checks before writing.
assert len(updated) == len(manifest) + 9
assert updated["movie_record_id"].is_unique, "Duplicate IDs detected before commit."
assert required_ids.issubset(set(updated["movie_record_id"]))

# Backup manifest, then atomically replace it.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = WORK / f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_final9_{timestamp}.csv"
shutil.copy2(MANIFEST_PATH, backup)

temp_path = WORK / f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_temp_{timestamp}.csv"
updated.to_csv(temp_path, index=False, encoding="utf-8-sig")

# Read back the staged manifest before replacing the active file.
check = pd.read_csv(temp_path, dtype={"movie_record_id": str})
assert len(check) == len(manifest) + 9
assert check["movie_record_id"].is_unique
assert required_ids.issubset(set(check["movie_record_id"]))

temp_path.replace(MANIFEST_PATH)

# Read back the committed manifest.
final_manifest = pd.read_csv(MANIFEST_PATH, dtype={"movie_record_id": str})
final_ids = set(final_manifest["movie_record_id"])

print("=" * 64)
print("FINAL NINE-SUBTITLE COMMIT")
print("=" * 64)
print("Validated copies verified:", len(staged))
print("Backup created:", backup)
print("Manifest rows before:", len(manifest))
print("Manifest rows after:", len(final_manifest))
print("Unique movie IDs:", final_manifest["movie_record_id"].nunique())
print("Duplicate IDs:", int(final_manifest["movie_record_id"].duplicated().sum()))
print("These nine IDs now in manifest:", required_ids.issubset(final_ids))
print("Master dataset modified: NO")
print("Incoming originals modified: NO")
print("Existing raw subtitles modified: NO")
print("Identity verification pending for new nine: YES")
print("\nNew records:")
display(final_manifest[final_manifest["movie_record_id"].isin(required_ids)][
    ["movie_record_id", "movie_title", "subtitle_language",
     "validated_subtitle_file", "validation_status",
     "movie_identity_verified", "language_verified"]
].sort_values("movie_record_id"))


In [ ]:

from pathlib import Path
import pandas as pd
import re
import hashlib
from collections import Counter

# ============================================================
# ICMB-130 — MOVIE IDENTITY & LANGUAGE SCREENING
# Read-only: no manifest or subtitle files are modified
# ============================================================

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MASTER_PATH = PROJECT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
MANIFEST_PATH = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

master = pd.read_csv(MASTER_PATH, dtype={"movie_record_id": str})
manifest = pd.read_csv(MANIFEST_PATH, dtype={"movie_record_id": str})

def read_text(path):
    raw = path.read_bytes()
    if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
        return raw.decode("utf-16"), "utf-16-BOM"
    if raw.startswith(b"\xef\xbb\xbf"):
        return raw.decode("utf-8-sig"), "utf-8-BOM"
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            return raw.decode(enc), enc
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace"), "utf-8-replacement"

def extract_dialogue(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s.isdigit() or "-->" in s:
            continue
        if s.upper() == "WEBVTT":
            continue
        s = re.sub(r"<[^>]+>", " ", s)
        s = re.sub(r"\{\\an\d+\}", " ", s)
        lines.append(s)
    return re.sub(r"\s+", " ", " ".join(lines)).strip()

# Unicode-script screening; this is a clue, not a definitive language classifier.
SCRIPT_RANGES = {
    "Devanagari": (0x0900, 0x097F),
    "Tamil": (0x0B80, 0x0BFF),
    "Telugu": (0x0C00, 0x0C7F),
    "Kannada": (0x0C80, 0x0CFF),
    "Malayalam": (0x0D00, 0x0D7F),
    "Bengali": (0x0980, 0x09FF),
    "Gujarati": (0x0A80, 0x0AFF),
    "Gurmukhi": (0x0A00, 0x0A7F),
    "Latin": (0x0041, 0x007A),
}

def script_counts(text):
    counts = {}
    for name, (lo, hi) in SCRIPT_RANGES.items():
        counts[name] = sum(lo <= ord(ch) <= hi for ch in text)
    return counts

def get_value(row, columns, default=""):
    for c in columns:
        if c in row.index and pd.notna(row[c]):
            return str(row[c])
    return default

master_index = master.set_index("movie_record_id", drop=False)
rows = []

for _, rec in manifest.iterrows():
    rid = str(rec["movie_record_id"])
    path_value = get_value(rec, ["validated_subtitle_file"])
    path = Path(path_value) if path_value else None

    result = {
        "movie_record_id": rid,
        "movie_title": get_value(rec, ["movie_title", "dataset_title", "title"]),
        "master_language": "",
        "manifest_language": get_value(rec, ["subtitle_language"]),
        "file_exists": bool(path and path.is_file()),
        "encoding": "",
        "word_count": 0,
        "latin_letters": 0,
        "dominant_script": "Unknown",
        "script_counts": "",
        "encoding_artifact_hint": False,
        "identity_evidence": "Unavailable",
        "identity_review": "PENDING",
        "language_review": "PENDING",
        "preview": "",
    }

    if rid in master_index.index:
        mr = master_index.loc[rid]
        if isinstance(mr, pd.DataFrame):
            mr = mr.iloc[0]
        result["movie_title"] = get_value(
            mr, ["movie_title", "dataset_title", "title"],
            result["movie_title"]
        )
        result["master_language"] = get_value(
            mr, ["language", "dataset_language"]
        )

    if not result["file_exists"]:
        result["identity_evidence"] = "Validated path missing or unreadable"
        rows.append(result)
        continue

    try:
        raw_text, encoding = read_text(path)
        dialogue = extract_dialogue(raw_text)
        counts = script_counts(dialogue)
        words = re.findall(r"\b[\w’'-]+\b", dialogue, flags=re.UNICODE)

        result["encoding"] = encoding
        result["word_count"] = len(words)
        result["latin_letters"] = counts["Latin"]
        result["script_counts"] = str(counts)
        result["preview"] = dialogue[:300]
        result["encoding_artifact_hint"] = bool(
            re.search(r"Ã.|â€™|â€œ|â€|ï»¿|�|ÿþ|þÿ", dialogue)
        )

        non_latin = {k: v for k, v in counts.items()
                     if k != "Latin" and v > 0}
        if counts["Latin"] > 100 and not non_latin:
            result["dominant_script"] = "Latin"
            result["language_review"] = "LIKELY_LATIN_SCRIPT; English needs review"
        elif non_latin:
            result["dominant_script"] = max(non_latin, key=non_latin.get)
            result["language_review"] = "NON_LATIN_SCRIPT_REVIEW"
        else:
            result["dominant_script"] = "Insufficient text"
            result["language_review"] = "MANUAL_REVIEW"

        # Conservative identity screening:
        # filename/title is not proof; require content evidence for confirmation.
        title = result["movie_title"].lower()
        title_terms = [
            t for t in re.findall(r"[a-z0-9]+", title)
            if len(t) >= 4 and t not in {"story", "movie", "film"}
        ]
        normalized_dialogue = dialogue.lower()
        matched_terms = [t for t in title_terms if t in normalized_dialogue]

        if matched_terms:
            result["identity_evidence"] = (
                "Title term(s) present: " + ", ".join(matched_terms)
                + " — manual context check required"
            )
            result["identity_review"] = "REVIEW_TITLE_TERM"
        else:
            result["identity_evidence"] = (
                "No clear title term found; inspect dialogue/plot clues"
            )
            result["identity_review"] = "MANUAL_REVIEW"

    except Exception as e:
        result["identity_evidence"] = f"Read/parse error: {type(e).__name__}"
        result["identity_review"] = "READ_ERROR"
        result["language_review"] = "READ_ERROR"

    rows.append(result)

screen = pd.DataFrame(rows)
report_path = WORK / "ICMB_130_IDENTITY_LANGUAGE_SCREENING.csv"
screen.to_csv(report_path, index=False, encoding="utf-8-sig")

print("Manifest records:", len(manifest))
print("Files found:", int(screen["file_exists"].sum()))
print("Files missing:", int((~screen["file_exists"]).sum()))
print("Identity screening statuses:")
print(screen["identity_review"].value_counts(dropna=False))
print("\nLanguage screening statuses:")
print(screen["language_review"].value_counts(dropna=False))
print("\nEncoding artifact hints:", int(screen["encoding_artifact_hint"].sum()))
print("\nReport saved:", report_path)

# Show every record requiring manual identity or language review.
display(screen[
    (screen["identity_review"] != "REVIEW_TITLE_TERM") |
    (screen["language_review"] != "LIKELY_LATIN_SCRIPT; English needs review") |
    (screen["encoding_artifact_hint"])
][[
    "movie_record_id", "movie_title", "master_language",
    "manifest_language", "file_exists", "word_count",
    "dominant_script", "encoding_artifact_hint",
    "identity_evidence", "identity_review", "language_review",
    "preview"
]])


In [ ]:

from pathlib import Path
import pandas as pd
import os

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MANIFEST_PATH = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(MANIFEST_PATH, dtype={"movie_record_id": str})

# Index subtitle files by basename across the complete corpus.
print("Indexing subtitle files...")
file_index = {}

for p in WORK.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".srt", ".vtt", ".txt"}:
        file_index.setdefault(p.name.lower(), []).append(p)

print("Subtitle/text files indexed:", sum(len(v) for v in file_index.values()))

results = []

for _, row in manifest.iterrows():
    rid = str(row["movie_record_id"])
    recorded_path = str(row.get("validated_subtitle_file", "") or "").strip()
    raw_path = str(row.get("raw_subtitle_file", "") or "").strip()

    validated_exists = bool(recorded_path and Path(recorded_path).is_file())
    raw_exists = bool(raw_path and Path(raw_path).is_file())

    # Try locating the validated file by its basename.
    basename = Path(recorded_path).name if recorded_path else ""
    candidates = file_index.get(basename.lower(), []) if basename else []

    results.append({
        "movie_record_id": rid,
        "movie_title": row.get("movie_title", ""),
        "validated_path_exists": validated_exists,
        "raw_path_exists": raw_exists,
        "recorded_validated_path": recorded_path,
        "recorded_raw_path": raw_path,
        "basename_matches_found": len(candidates),
        "candidate_paths": " | ".join(str(p) for p in candidates[:5]),
        "resolution": (
            "PATHS_OK" if validated_exists else
            "RAW_PATH_OK_VALIDATED_MISSING" if raw_exists else
            "BASENAME_FOUND_ELSEWHERE" if candidates else
            "FILE_NOT_LOCATED"
        )
    })

path_audit = pd.DataFrame(results)
report_path = WORK / "ICMB_130_MANIFEST_PATH_RESOLUTION_AUDIT.csv"
path_audit.to_csv(report_path, index=False, encoding="utf-8-sig")

print("\nPath resolution summary:")
print(path_audit["resolution"].value_counts(dropna=False))
print("\nTotal records:", len(path_audit))
print("Validated paths currently resolving:",
      int(path_audit["validated_path_exists"].sum()))
print("Report saved:", report_path)

print("\nRecords needing path recovery:")
display(
    path_audit[path_audit["resolution"] != "PATHS_OK"][
        ["movie_record_id", "movie_title", "resolution",
         "recorded_validated_path", "recorded_raw_path",
         "candidate_paths"]
    ].sort_values("movie_record_id")
)


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MANIFEST_PATH = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(MANIFEST_PATH, dtype={"movie_record_id": str})

# Index all subtitle files by basename.
file_index = {}
for p in WORK.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".srt", ".vtt"}:
        file_index.setdefault(p.name.lower(), []).append(p)

proposals = []

for _, row in manifest.iterrows():
    rid = str(row["movie_record_id"])
    old_validated = str(row.get("validated_subtitle_file", "") or "").strip()
    old_raw = str(row.get("raw_subtitle_file", "") or "").strip()

    validated_ok = bool(old_validated and Path(old_validated).is_file())
    raw_ok = bool(old_raw and Path(old_raw).is_file())

    basename = Path(old_validated).name if old_validated and old_validated.lower() != "nan" else ""
    matches = file_index.get(basename.lower(), []) if basename else []

    proposal = ""
    decision = "NO_CHANGE_NEEDED" if validated_ok else "MANUAL_REVIEW_REQUIRED"

    if not validated_ok and len(matches) == 1:
        proposal = str(matches[0])
        decision = "UNIQUE_BASENAME_MATCH_REVIEW"
    elif not validated_ok and len(matches) > 1:
        proposal = " | ".join(str(p) for p in matches)
        decision = "AMBIGUOUS_MULTIPLE_MATCHES"
    elif not validated_ok and raw_ok:
        decision = "RAW_EXISTS_VALIDATE_COPY_FIRST"

    proposals.append({
        "movie_record_id": rid,
        "movie_title": row.get("movie_title", ""),
        "old_validated_path": old_validated,
        "old_raw_path": old_raw,
        "validated_path_exists": validated_ok,
        "raw_path_exists": raw_ok,
        "candidate_count": len(matches),
        "proposed_path": proposal,
        "decision": decision,
    })

proposal_df = pd.DataFrame(proposals)
report_path = WORK / "ICMB_130_PATH_REPAIR_PROPOSALS.csv"
proposal_df.to_csv(report_path, index=False, encoding="utf-8-sig")

print("Proposal report:", report_path)
print("\nProposal summary:")
print(proposal_df["decision"].value_counts())
print("\nPaths needing attention:")
display(
    proposal_df[proposal_df["decision"] != "NO_CHANGE_NEEDED"][
        ["movie_record_id", "movie_title", "decision",
         "candidate_count", "proposed_path", "old_raw_path"]
    ].sort_values("movie_record_id")
)

print("\nNo files copied. No manifest changes made.")


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

proposal_path = WORK / "ICMB_130_PATH_REPAIR_PROPOSALS.csv"
proposals = pd.read_csv(proposal_path)

review = proposals[
    proposals["decision"].isin([
        "UNIQUE_BASENAME_MATCH_REVIEW",
        "RAW_EXISTS_VALIDATE_COPY_FIRST",
        "MANUAL_REVIEW_REQUIRED"
    ])
].copy()

# Display all relevant fields without truncating paths
columns = [
    c for c in [
        "movie_record_id",
        "movie_title",
        "decision",
        "candidate_count",
        "proposed_path",
        "old_raw_path"
    ]
    if c in review.columns
]

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 150)

print("Records requiring review:", len(review))
display(review[columns].reset_index(drop=True))

print("\nFull report saved at:", proposal_path)

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

proposal_path = WORK / "ICMB_130_PATH_REPAIR_PROPOSALS.csv"
report = pd.read_csv(proposal_path)

review = report[
    report["decision"].isin([
        "UNIQUE_BASENAME_MATCH_REVIEW",
        "RAW_EXISTS_VALIDATE_COPY_FIRST",
        "MANUAL_REVIEW_REQUIRED"
    ])
].copy()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

results = []

for _, row in review.iterrows():
    proposed = row.get("proposed_path")
    raw = row.get("old_raw_path")

    proposed_path = Path(str(proposed)) if pd.notna(proposed) else None
    raw_path = Path(str(raw)) if pd.notna(raw) else None

    candidate = (
        proposed_path
        if proposed_path and proposed_path.is_file()
        else raw_path
        if raw_path and raw_path.is_file()
        else None
    )

    results.append({
        "movie_record_id": row["movie_record_id"],
        "movie_title": row["movie_title"],
        "decision": row["decision"],
        "candidate_exists": candidate is not None,
        "candidate_path": str(candidate) if candidate else "",
        "size_bytes": candidate.stat().st_size if candidate else None,
        "sha256": sha256_file(candidate) if candidate else "",
    })

verification = pd.DataFrame(results)
output_path = WORK / "ICMB_130_PATH_CANDIDATE_VERIFICATION.csv"
verification.to_csv(output_path, index=False)

print("Candidate verification summary:")
print(
    verification.groupby(
        ["decision", "candidate_exists"], dropna=False
    ).size().to_string()
)

print("\nFull candidate verification:")
pd.set_option("display.max_colwidth", None)
display(verification)

print("\nReport saved to:", output_path)
print("\nNo files copied. No manifest changes made.")

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil
import hashlib

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
RAW = WORK / "raw"
VALIDATED = WORK / "validated"

manifest_path = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
proposal_path = WORK / "ICMB_130_PATH_REPAIR_PROPOSALS.csv"

manifest = pd.read_csv(manifest_path, dtype=str).fillna("")
proposals = pd.read_csv(proposal_path, dtype=str).fillna("")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Backup before any manifest changes
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = WORK / f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_path_repair_{stamp}.csv"
shutil.copy2(manifest_path, backup_path)

# Map records to proposals
proposal_by_id = proposals.set_index("movie_record_id").to_dict("index")

repair_log = []
errors = []

for idx, row in manifest.iterrows():
    record_id = row.get("movie_record_id", "")
    proposal = proposal_by_id.get(record_id)

    if not proposal:
        continue

    decision = proposal.get("decision", "")
    proposed_path = proposal.get("proposed_path", "")
    raw_path_text = proposal.get("old_raw_path", "")

    if decision == "UNIQUE_BASENAME_MATCH_REVIEW":
        candidate = Path(proposed_path) if proposed_path else None

        if candidate and candidate.is_file():
            manifest.at[idx, "validated_subtitle_file"] = str(candidate)
            repair_log.append({
                "movie_record_id": record_id,
                "action": "UPDATED_VALIDATED_PATH",
                "path": str(candidate),
                "sha256": sha256_file(candidate),
                "status": "OK"
            })
        else:
            errors.append((record_id, "Unique candidate missing"))

    elif decision == "RAW_EXISTS_VALIDATE_COPY_FIRST":
        source = Path(raw_path_text) if raw_path_text else None

        if not source or not source.is_file():
            errors.append((record_id, "Raw source missing"))
            continue

        # Use the existing validated filename from the manifest if present;
        # otherwise generate a safe record-ID-based filename.
        current_validated = manifest.at[idx, "validated_subtitle_file"]
        current_name = Path(current_validated).name if current_validated else ""
        if not current_name or current_name in {".", ".."}:
            safe_title = "".join(
                ch if ch.isalnum() else "_"
                for ch in row.get("movie_title", "")
            ).strip("_")
            current_name = f"{record_id}_{safe_title}_ENGLISH.srt"

        destination = VALIDATED / current_name
        VALIDATED.mkdir(parents=True, exist_ok=True)

        if destination.exists():
            if sha256_file(destination) != sha256_file(source):
                errors.append((
                    record_id,
                    f"Destination exists with different content: {destination}"
                ))
                continue
        else:
            shutil.copy2(source, destination)

        if sha256_file(destination) != sha256_file(source):
            errors.append((record_id, "Copy hash mismatch"))
            continue

        manifest.at[idx, "validated_subtitle_file"] = str(destination)
        repair_log.append({
            "movie_record_id": record_id,
            "action": "COPIED_RAW_TO_VALIDATED",
            "path": str(destination),
            "sha256": sha256_file(destination),
            "status": "OK"
        })

# Abort manifest save if any operation had an issue
if errors:
    print("ERRORS — manifest NOT saved:")
    for error in errors:
        print(error)
    print("Backup created at:", backup_path)
else:
    # Verify record count and unique IDs before saving
    assert len(manifest) == 130, f"Unexpected manifest row count: {len(manifest)}"
    assert manifest["movie_record_id"].nunique() == 130, "Duplicate movie_record_id detected"

    manifest.to_csv(manifest_path, index=False)

    log_path = WORK / f"ICMB_130_PATH_REPAIR_LOG_{stamp}.csv"
    pd.DataFrame(repair_log).to_csv(log_path, index=False)

    print("PATH REPAIR COMPLETED")
    print("Manifest rows:", len(manifest))
    print("Unique movie IDs:", manifest["movie_record_id"].nunique())
    print("Path updates/copies:", len(repair_log))
    print("Backup:", backup_path)
    print("Repair log:", log_path)
    print("Identity/language verification flags were not changed.")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

manifest_path = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
manifest = pd.read_csv(manifest_path, dtype=str).fillna("")

results = []

for _, row in manifest.iterrows():
    validated_text = row.get("validated_subtitle_file", "").strip()
    raw_text = row.get("raw_subtitle_file", "").strip()

    validated_path = Path(validated_text) if validated_text else None
    raw_path = Path(raw_text) if raw_text else None

    validated_exists = bool(validated_path and validated_path.is_file())
    raw_exists = bool(raw_path and raw_path.is_file())

    if validated_exists:
        status = "VALIDATED_PATH_OK"
    elif raw_exists:
        status = "RAW_EXISTS_VALIDATED_MISSING"
    else:
        status = "NEITHER_RECORDED_PATH_EXISTS"

    results.append({
        "movie_record_id": row.get("movie_record_id", ""),
        "movie_title": row.get("movie_title", ""),
        "validated_exists": validated_exists,
        "raw_exists": raw_exists,
        "path_status": status
    })

audit = pd.DataFrame(results)
audit_path = WORK / "ICMB_130_PATH_AUDIT_AFTER_REPAIR.csv"
audit.to_csv(audit_path, index=False)

print("Manifest records:", len(audit))
print("Unique movie IDs:", audit["movie_record_id"].nunique())
print("\nPath status:")
print(audit["path_status"].value_counts().to_string())

print("\nRecords still needing path attention:")
display(
    audit[audit["path_status"] != "VALIDATED_PATH_OK"]
    .reset_index(drop=True)
)

print("\nAudit saved to:", audit_path)

In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_missing_movies"

expected_movies = {
    "2000_3": "Mission Kashmir",
    "2000_5": "Josh",
    "2005_1": "Chandramukhi",
    "2014_1": "PK",
    "2015_4": "Bajirao Mastani",
}

print("Incoming folder exists:", INCOMING.exists())
print("Incoming folder:", INCOMING)

# Locate subtitle files recursively
subtitle_files = sorted(
    p for p in INCOMING.rglob("*")
    if p.is_file() and p.suffix.lower() in {".srt", ".vtt"}
)

print("\nSubtitle files found:", len(subtitle_files))

for p in subtitle_files:
    print(f"{p.name}  |  {p.stat().st_size:,} bytes")

# Basic structural audit
def audit_subtitle(path):
    raw = path.read_bytes()

    try:
        text = raw.decode("utf-8-sig")
        encoding = "utf-8-sig"
    except UnicodeDecodeError:
        try:
            text = raw.decode("cp1252")
            encoding = "cp1252"
        except UnicodeDecodeError:
            text = raw.decode("utf-8", errors="replace")
            encoding = "utf-8 with replacement"

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = re.split(r"\n\s*\n", text.strip())

    cue_count = 0
    malformed = 0
    for block in blocks:
        lines = [line.strip() for line in block.split("\n") if line.strip()]
        if any("-->" in line for line in lines):
            if any(
                re.search(
                    r"\d{1,2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
                    r"\d{1,2}:\d{2}:\d{2}[,.]\d{3}",
                    line
                )
                for line in lines
            ):
                cue_count += 1
            else:
                malformed += 1

    words = re.findall(r"\b[\w']+\b", re.sub(r"<[^>]+>", " ", text))

    return {
        "encoding": encoding,
        "size_bytes": len(raw),
        "cue_count": cue_count,
        "word_count": len(words),
        "malformed_timing_blocks": malformed,
    }

audit_rows = []
for path in subtitle_files:
    result = audit_subtitle(path)
    audit_rows.append({
        "filename": path.name,
        "full_path": str(path),
        **result
    })

audit_df = pd.DataFrame(audit_rows)

if not audit_df.empty:
    display(audit_df)

audit_path = WORK / "missing_five_uploaded_subtitles_audit.csv"
audit_df.to_csv(audit_path, index=False)

print("\nAudit report saved to:", audit_path)
print("No files copied. No manifest changes made.")

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import shutil

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_missing_movies"
VALIDATED = WORK / "validated"

manifest_path = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

targets = {
    "2000_3": "2000_3_MISSION_KASHMIR_ENGLISH.srt",
    "2000_5": "2000_5_JOSH_ENGLISH.srt",
    "2005_1": "2005_1_CHANDRAMUKHI_ENGLISH.srt",
    "2014_1": "2014_1_PK_ENGLISH.srt",
    "2015_4": "2015_4_BAJIRAO_MASTANI_ENGLISH.srt",
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = pd.read_csv(manifest_path, dtype=str).fillna("")

if manifest["movie_record_id"].duplicated().any():
    raise ValueError("Duplicate movie IDs found. Stop before proceeding.")

VALIDATED.mkdir(parents=True, exist_ok=True)
preview = []
errors = []

for record_id, clean_name in targets.items():
    matches = [
        p for p in INCOMING.rglob("*")
        if p.is_file()
        and p.name.startswith(record_id + "_")
        and p.suffix.lower() == ".srt"
    ]

    # Prefer the exact requested movie filename, allowing a double .srt suffix.
    matches = [
        p for p in matches
        if clean_name.removesuffix(".srt").lower()
        in p.name.lower()
    ]

    # Duplicate copies are acceptable only if their content hashes match.
    unique_by_hash = {}
    for p in matches:
        unique_by_hash.setdefault(sha256_file(p), p)

    if len(unique_by_hash) != 1:
        errors.append(
            (record_id, f"Expected one unique subtitle content; found {len(unique_by_hash)}")
        )
        continue

    source_hash, source = next(iter(unique_by_hash.items()))
    destination = VALIDATED / clean_name

    if destination.exists():
        if sha256_file(destination) != source_hash:
            errors.append(
                (record_id, f"Destination already exists with different content: {destination}")
            )
            continue
        action = "ALREADY_EXISTS_IDENTICAL"
    else:
        shutil.copy2(source, destination)
        if sha256_file(destination) != source_hash:
            errors.append((record_id, "Hash mismatch after copy"))
            destination.unlink(missing_ok=True)
            continue
        action = "COPIED_AND_HASH_VERIFIED"

    preview.append({
        "movie_record_id": record_id,
        "source_file": str(source),
        "validated_file": str(destination),
        "sha256": source_hash,
        "action": action,
    })

if errors:
    print("Issues found; manifest was not modified:")
    for item in errors:
        print(item)
else:
    preview_df = pd.DataFrame(preview)
    preview_path = WORK / "missing_five_validated_copy_preview.csv"
    preview_df.to_csv(preview_path, index=False)

    display(preview_df)
    print("\nPreview saved to:", preview_path)
    print("Manifest unchanged. Original incoming files unchanged.")

In [ ]:
from pathlib import Path
import hashlib
import re

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

existing = WORK / "validated/2000_5_JOSH_ENGLISH.srt"
incoming_dir = WORK / "incoming_manual_missing_movies"

incoming_candidates = [
    p for p in incoming_dir.rglob("*")
    if p.is_file()
    and p.name.startswith("2000_5_JOSH_ENGLISH.srt")
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_text(path):
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "utf-16", "cp1252", "utf-8"):
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            pass
    return raw.decode("utf-8", errors="replace"), "replacement"

def summarize(path):
    text, encoding = read_text(path)
    blocks = re.split(r"\n\s*\n", text.replace("\r\n", "\n").replace("\r", "\n").strip())
    cue_count = sum(
        bool(re.search(
            r"\d{1,2}:\d{2}:\d{2}[,.]\d{3}\s*-->",
            block
        ))
        for block in blocks
    )
    dialogue = []
    for block in blocks:
        lines = block.splitlines()
        if any("-->" in line for line in lines):
            dialogue.extend(
                line.strip() for line in lines
                if line.strip()
                and "-->" not in line
                and not line.strip().isdigit()
            )
        if len(dialogue) >= 12:
            break

    return {
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "encoding": encoding,
        "cue_count": cue_count,
        "opening_dialogue": "\n".join(dialogue[:12]),
    }

print("EXISTING VALIDATED FILE")
print(summarize(existing))

for candidate in incoming_candidates:
    print("\nINCOMING CANDIDATE")
    print(summarize(candidate))

print("\nNo files changed.")

In [ ]:
from pathlib import Path
import re
import hashlib

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

existing = WORK / "validated/2000_5_JOSH_ENGLISH.srt"
incoming = WORK / "incoming_manual_missing_movies/2000_5_JOSH_ENGLISH.srt.srt"

def read_subtitle(path):
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "utf-16", "cp1252", "utf-8"):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            pass
    return raw.decode("utf-8", errors="replace")

def extract_dialogue(text, max_lines=60):
    lines = []
    for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        line = line.strip()
        if not line or "-->" in line or line.isdigit():
            continue
        line = re.sub(r"<[^>]+>", "", line)
        if line:
            lines.append(line)
        if len(lines) >= max_lines:
            break
    return lines

print("INCOMING JOSH — first 60 dialogue lines")
print("-" * 70)
print("\n".join(extract_dialogue(read_subtitle(incoming), 60)))

print("\nEXISTING JOSH — first 30 dialogue lines")
print("-" * 70)
print("\n".join(extract_dialogue(read_subtitle(existing), 30)))

print("\nSHA-256 incoming:", hashlib.sha256(incoming.read_bytes()).hexdigest())
print("SHA-256 existing:", hashlib.sha256(existing.read_bytes()).hexdigest())
print("\nNo files changed.")

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import hashlib
import shutil

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
INCOMING = WORK / "incoming_manual_missing_movies"
VALIDATED = WORK / "validated"

manifest_path = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

targets = {
    "2000_3": ("2000_3_MISSION_KASHMIR_ENGLISH.srt",
               "2000_3_MISSION_KASHMIR_ENGLISH.srt.srt"),
    "2000_5": ("2000_5_JOSH_ENGLISH.srt",
               "2000_5_JOSH_ENGLISH.srt.srt"),
    "2005_1": ("2005_1_CHANDRAMUKHI_ENGLISH.srt",
               "2005_1_CHANDRAMUKHI_ENGLISH.srt.srt"),
    "2014_1": ("2014_1_PK_ENGLISH.srt",
               "2014_1_PK_ENGLISH.srt.srt"),
    "2015_4": ("2015_4_BAJIRAO_MASTANI_ENGLISH.srt",
               "2015_4_BAJIRAO_MASTANI_ENGLISH.srt.srt"),
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = pd.read_csv(manifest_path, dtype=str).fillna("")

if len(manifest) != 130 or manifest["movie_record_id"].nunique() != 130:
    raise ValueError("Manifest row count/unique IDs do not match expected 130.")

prepared = []
errors = []

for record_id, (clean_name, incoming_name) in targets.items():
    candidates = [
        p for p in INCOMING.rglob("*")
        if p.is_file() and p.name == incoming_name
    ]

    # If nested copies exist, accept them only if their content hashes match.
    unique = {}
    for p in candidates:
        unique.setdefault(sha256_file(p), p)

    if len(unique) != 1:
        errors.append(
            f"{record_id}: expected one unique source version, found {len(unique)}"
        )
        continue

    source_hash, source = next(iter(unique.items()))
    destination = VALIDATED / clean_name
    action = "COPIED"

    if destination.exists():
        if sha256_file(destination) == source_hash:
            action = "ALREADY_IDENTICAL"
        else:
            destination = VALIDATED / (
                f"{Path(clean_name).stem}_INCOMING_REVIEW.srt"
            )
            if destination.exists():
                if sha256_file(destination) != source_hash:
                    errors.append(
                        f"{record_id}: alternative destination also differs"
                    )
                    continue
                action = "ALTERNATIVE_ALREADY_IDENTICAL"
            else:
                action = "COPIED_AS_ALTERNATIVE"

    prepared.append({
        "movie_record_id": record_id,
        "source": source,
        "destination": destination,
        "source_hash": source_hash,
        "action": action,
    })

if errors:
    print("No manifest changes made. Resolve these issues first:")
    for e in errors:
        print("-", e)
else:
    VALIDATED.mkdir(parents=True, exist_ok=True)

    # Copy and verify every destination before changing the manifest.
    for item in prepared:
        src = item["source"]
        dst = item["destination"]

        if not dst.exists():
            shutil.copy2(src, dst)

        if sha256_file(dst) != item["source_hash"]:
            raise RuntimeError(
                f"Hash verification failed for {item['movie_record_id']}"
            )

    # Backup manifest before its update.
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup = WORK / (
        f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_backup_before_five_{stamp}.csv"
    )
    shutil.copy2(manifest_path, backup)

    for item in prepared:
        mask = manifest["movie_record_id"] == item["movie_record_id"]
        manifest.loc[mask, "validated_subtitle_file"] = str(item["destination"])
        manifest.loc[mask, "validation_status"] = (
            "Structural_Pass_Content_Identity_Review_Pending"
        )
        manifest.loc[mask, "movie_identity_verified"] = "False"
        manifest.loc[mask, "language_verified"] = "False"
        manifest.loc[mask, "title_verified"] = "False"
        manifest.loc[mask, "year_verified"] = "False"

    manifest.to_csv(manifest_path, index=False)

    print("Five subtitle records registered.")
    print("Manifest rows:", len(manifest))
    print("Unique movie IDs:", manifest["movie_record_id"].nunique())
    print("Backup:", backup)
    print("\nRegistered files:")
    display(pd.DataFrame([
        {
            "movie_record_id": x["movie_record_id"],
            "destination": str(x["destination"]),
            "action": x["action"],
            "sha256": x["source_hash"],
        }
        for x in prepared
    ]))
    print("\nIdentity and language verification remain pending.")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

manifest_path = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
manifest = pd.read_csv(manifest_path, dtype=str).fillna("")

results = []

for _, row in manifest.iterrows():
    path_text = row.get("validated_subtitle_file", "").strip()
    path = Path(path_text) if path_text else None

    results.append({
        "movie_record_id": row.get("movie_record_id", ""),
        "movie_title": row.get("movie_title", ""),
        "validated_exists": bool(path and path.is_file()),
        "validated_path": path_text
    })

audit = pd.DataFrame(results)
audit_path = WORK / "ICMB_130_PATH_AUDIT_FINAL.csv"
audit.to_csv(audit_path, index=False)

print("Manifest records:", len(audit))
print("Unique movie IDs:", audit["movie_record_id"].nunique())
print("Validated files found:", int(audit["validated_exists"].sum()))
print("Validated files missing:", int((~audit["validated_exists"]).sum()))

if (~audit["validated_exists"]).any():
    display(audit.loc[~audit["validated_exists"]].reset_index(drop=True))
else:
    print("\nAll manifest records point to existing validated files.")

print("\nAudit saved to:", audit_path)


In [ ]:
from pathlib import Path
import pandas as pd
import re
import hashlib
import unicodedata

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

MANIFEST_PATH = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

assert MANIFEST_PATH.exists(), f"Manifest not found: {MANIFEST_PATH}"

manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")

print("=" * 60)
print("ICMB-130 FINAL SUBTITLE QUALITY AUDIT")
print("=" * 60)
print("Manifest records:", len(manifest))
print("Unique movie IDs:", manifest["movie_record_id"].nunique())
print("Duplicate movie IDs:",
      manifest["movie_record_id"].duplicated().sum())

assert len(manifest) == 130, "Expected 130 manifest records."
assert manifest["movie_record_id"].nunique() == 130, \
    "Movie IDs are missing or duplicated."

print("\nManifest checks passed.")

In [ ]:
TIMESTAMP_RE = re.compile(
    r"^\s*(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})"
    r"\s*-->\s*"
    r"(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def decode_subtitle(data):
    for encoding in ("utf-8-sig", "utf-16", "cp1252", "latin-1"):
        try:
            return data.decode(encoding), encoding
        except (UnicodeDecodeError, LookupError):
            continue
    return data.decode("utf-8", errors="replace"), "utf-8-replacement"

def to_milliseconds(match, start_index):
    h, m, s, fraction = match.group(
        start_index, start_index + 1,
        start_index + 2, start_index + 3
    )
    ms = int(fraction.ljust(3, "0")[:3])
    return ((int(h) * 60 + int(m)) * 60 + int(s)) * 1000 + ms

def audit_subtitle(path_text):
    result = {
        "file_exists": False,
        "file_size_bytes": 0,
        "sha256": "",
        "encoding_detected": "",
        "cue_count": 0,
        "word_count": 0,
        "text_characters": 0,
        "malformed_timestamp_count": 0,
        "invalid_duration_count": 0,
        "empty_text_cue_count": 0,
        "replacement_character_count": 0,
        "mojibake_indicator_count": 0,
        "non_latin_letter_count": 0,
        "read_error": ""
    }

    if not path_text:
        result["read_error"] = "EMPTY_PATH"
        return result

    path = Path(path_text)

    if not path.is_file():
        result["read_error"] = "FILE_NOT_FOUND"
        return result

    result["file_exists"] = True

    try:
        data = path.read_bytes()
        result["file_size_bytes"] = len(data)
        result["sha256"] = hashlib.sha256(data).hexdigest()

        text, encoding = decode_subtitle(data)
        result["encoding_detected"] = encoding
        result["replacement_character_count"] = text.count("\ufffd")

        result["mojibake_indicator_count"] = sum(
            text.count(token)
            for token in ("Ã", "â€™", "â€œ", "â€", "ï»¿")
        )

        lines = text.splitlines()
        cue_texts = []
        current_text = []
        current_cue = False

        malformed = 0
        invalid_duration = 0
        empty_cues = 0

        def finish_cue():
            nonlocal current_text, empty_cues
            cue = " ".join(current_text).strip()
            if cue:
                cue_texts.append(cue)
            else:
                empty_cues += 1
            current_text = []

        for line in lines:
            stripped = line.strip()

            if "-->" in stripped:
                match = TIMESTAMP_RE.match(stripped)

                if not match:
                    malformed += 1
                    if current_cue:
                        finish_cue()
                    current_cue = False
                    continue

                if current_cue:
                    finish_cue()

                current_cue = True

                start_ms = to_milliseconds(match, 1)
                end_ms = to_milliseconds(match, 5)

                if end_ms <= start_ms:
                    invalid_duration += 1

                continue

            if current_cue:
                if not stripped:
                    finish_cue()
                    current_cue = False
                elif not re.fullmatch(r"\d+", stripped):
                    current_text.append(stripped)

        if current_cue:
            finish_cue()

        result["cue_count"] = len(cue_texts)
        result["word_count"] = sum(
            len(re.findall(r"\b[\w’'-]+\b", cue))
            for cue in cue_texts
        )
        result["text_characters"] = sum(len(cue) for cue in cue_texts)
        result["malformed_timestamp_count"] = malformed
        result["invalid_duration_count"] = invalid_duration
        result["empty_text_cue_count"] = empty_cues

        result["non_latin_letter_count"] = sum(
            1 for char in text
            if unicodedata.category(char).startswith("L")
            and "LATIN" not in unicodedata.name(char, "")
        )

    except Exception as exc:
        result["read_error"] = f"{type(exc).__name__}: {exc}"

    return result


audit_rows = []

for _, row in manifest.iterrows():
    path_text = row.get("validated_subtitle_file", "").strip()

    metrics = audit_subtitle(path_text)

    audit_rows.append({
        "movie_record_id": row.get("movie_record_id", ""),
        "movie_title": row.get("movie_title", ""),
        "movie_year": row.get("movie_year", row.get("dataset_year", "")),
        "movie_language": row.get("movie_language", row.get("dataset_language", "")),
        "validated_subtitle_file": path_text,
        **metrics
    })

quality = pd.DataFrame(audit_rows)

print("Subtitle files audited:", len(quality))
print("Files found:", int(quality["file_exists"].sum()))
print("Files missing:", int((~quality["file_exists"]).sum()))

display(quality.head(10))

In [ ]:
def review_flags(row):
    flags = []

    if not row["file_exists"]:
        flags.append("MISSING_FILE")

    if row["read_error"]:
        flags.append("READ_ERROR")

    if row["file_size_bytes"] == 0:
        flags.append("EMPTY_FILE")

    if row["cue_count"] == 0:
        flags.append("ZERO_CUES")

    if row["malformed_timestamp_count"] > 0:
        flags.append("MALFORMED_TIMESTAMPS")

    if row["invalid_duration_count"] > 0:
        flags.append("INVALID_DURATIONS")

    if row["empty_text_cue_count"] > 0:
        flags.append("EMPTY_TEXT_CUES")

    if row["replacement_character_count"] > 0:
        flags.append("REPLACEMENT_CHARACTERS")

    if row["mojibake_indicator_count"] > 0:
        flags.append("POSSIBLE_ENCODING_ISSUE")

    if row["cue_count"] > 0 and row["word_count"] < 500:
        flags.append("LOW_WORD_COUNT_REVIEW")

    if row["word_count"] > 50000:
        flags.append("HIGH_WORD_COUNT_REVIEW")

    if row["cue_count"] > 0 and row["word_count"] / row["cue_count"] < 1:
        flags.append("LOW_WORDS_PER_CUE_REVIEW")

    return ";".join(flags) if flags else "NO_AUTOMATED_FLAGS"


quality["review_flags"] = quality.apply(review_flags, axis=1)

hash_counts = quality.loc[
    quality["sha256"].ne(""), "sha256"
].value_counts()

quality["duplicate_content_hash"] = (
    quality["sha256"].map(hash_counts).fillna(0).astype(int) > 1
)

flagged = quality[
    quality["review_flags"] != "NO_AUTOMATED_FLAGS"
].copy()

print("Records with review flags:", len(flagged))
print("Records sharing an identical file hash:",
      int(quality["duplicate_content_hash"].sum()))

display(
    flagged[[
        "movie_record_id",
        "movie_title",
        "cue_count",
        "word_count",
        "malformed_timestamp_count",
        "invalid_duration_count",
        "replacement_character_count",
        "review_flags"
    ]].reset_index(drop=True)
)

In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MANIFEST_PATH = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")

priority_ids = [
    "2010_3",  # Golmaal 3
    "2021_3",  # Master
    "2006_2",  # Lage Raho Munnabhai
    "2013_2",  # Chennai Express
]

TIMESTAMP_RE = re.compile(
    r"^\s*(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})"
    r"\s*-->\s*"
    r"(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)

def decode_for_inspection(data):
    for encoding in ("utf-8-sig", "utf-16", "cp1252", "latin-1"):
        try:
            return data.decode(encoding), encoding
        except UnicodeDecodeError:
            continue
    return data.decode("utf-8", errors="replace"), "utf-8-replacement"

def ms_from_match(match, offset):
    h, m, s, fraction = match.group(
        offset, offset + 1, offset + 2, offset + 3
    )
    return (
        (int(h) * 60 + int(m)) * 60 + int(s)
    ) * 1000 + int(fraction.ljust(3, "0")[:3])

for movie_id in priority_ids:
    rows = manifest.loc[manifest["movie_record_id"] == movie_id]

    print("\n" + "=" * 75)

    if rows.empty:
        print("Record not found:", movie_id)
        continue

    row = rows.iloc[0]
    print("Movie ID:", movie_id)
    print("Title:", row.get("movie_title", ""))
    path_text = row.get("validated_subtitle_file", "").strip()
    print("Path:", path_text)

    path = Path(path_text) if path_text else None

    if not path or not path.is_file():
        print("ERROR: Validated subtitle file not found.")
        continue

    data = path.read_bytes()
    text, encoding = decode_for_inspection(data)
    lines = text.splitlines()

    print("File size:", len(data), "bytes")
    print("Detected encoding:", encoding)
    print("Total lines:", len(lines))
    print("\nFirst 15 lines:")
    for line in lines[:15]:
        print(repr(line))

    timestamp_lines = [
        (i + 1, line.strip())
        for i, line in enumerate(lines)
        if "-->" in line
    ]

    malformed = [
        (line_no, line)
        for line_no, line in timestamp_lines
        if not TIMESTAMP_RE.match(line)
    ]

    invalid_duration = []
    for line_no, line in timestamp_lines:
        match = TIMESTAMP_RE.match(line)
        if match:
            start = ms_from_match(match, 1)
            end = ms_from_match(match, 5)
            if end <= start:
                invalid_duration.append((line_no, line))

    print("\nTimestamp lines:", len(timestamp_lines))
    print("Malformed timestamp lines:", len(malformed))
    print("Invalid-duration lines:", len(invalid_duration))

    if malformed:
        print("\nFirst 10 malformed timestamp examples:")
        for item in malformed[:10]:
            print(item)

    if invalid_duration:
        print("\nInvalid-duration examples:")
        for item in invalid_duration[:10]:
            print(item)

    print("\nFirst 10 timestamp examples:")
    for item in timestamp_lines[:10]:
        print(item)

In [ ]:
from pathlib import Path
import re

path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/validated/"
    "2010_3_GOLMAAL_3_ENGLISH.srt"
)

data = path.read_bytes()

print("File size:", len(data))
print("First 16 bytes:", data[:16])

for encoding in ["utf-16", "utf-16-le", "utf-16-be", "utf-8-sig", "cp1252"]:
    try:
        text = data.decode(encoding)
        timestamp_count = len(re.findall(
            r"(?m)^\s*\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*-->\s*"
            r"\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*$",
            text
        ))
        print(f"\nEncoding: {encoding}")
        print("Timestamp count:", timestamp_count)
        print("First 8 lines:")
        print("\n".join(repr(line) for line in text.splitlines()[:8]))
    except UnicodeError as e:
        print(f"\nEncoding: {encoding} — decode failed: {e}")

In [ ]:
from pathlib import Path
import re

path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/validated/"
    "2010_3_GOLMAAL_3_ENGLISH.srt"
)

data = path.read_bytes()

print("File size:", len(data))
print("First 32 bytes:", data[:32])
print("BOM:", {
    "UTF-8": data.startswith(b"\xef\xbb\xbf"),
    "UTF-16 LE": data.startswith(b"\xff\xfe"),
    "UTF-16 BE": data.startswith(b"\xfe\xff"),
})
print("NUL bytes:", data.count(b"\x00"))

timestamp_pattern = re.compile(
    r"(?m)^\s*\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*-->\s*"
    r"\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*$"
)

for encoding in ["utf-16", "utf-16-le", "utf-16-be",
                 "utf-8-sig", "cp1252"]:
    try:
        text = data.decode(encoding)
        count = len(timestamp_pattern.findall(text))

        print(f"\n{'=' * 55}")
        print("Encoding:", encoding)
        print("Timestamp count:", count)
        print("First 8 lines:")
        for line in text.splitlines()[:8]:
            print(repr(line[:180]))

    except (UnicodeError, LookupError) as e:
        print(f"\nEncoding: {encoding} — failed: {e}")

In [ ]:
from pathlib import Path
import hashlib
import re

path = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/subtitle_corpus_v2/validated"
    / "2010_3_GOLMAAL_3_ENGLISH.srt"
)

data = path.read_bytes()
text = data.decode("cp1252")

timestamp_pattern = re.compile(
    r"(?m)^\s*\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*-->\s*"
    r"\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*$"
)

lines = text.splitlines()
timestamp_lines = [
    i for i, line in enumerate(lines)
    if timestamp_pattern.match(line)
]

print("FILE:", path.name)
print("SIZE:", len(data), "bytes")
print("SHA256:", hashlib.sha256(data).hexdigest())
print("TIMESTAMP COUNT:", len(timestamp_lines))

def show_sample(label, position):
    if not timestamp_lines:
        print("No standard timestamps found.")
        return

    idx = timestamp_lines[
        min(int(position * (len(timestamp_lines) - 1)),
            len(timestamp_lines) - 1)
    ]

    print(f"\n--- {label} (line {idx + 1}) ---")
    for line in lines[idx:idx + 5]:
        if line.strip():
            print(line[:200])

show_sample("BEGINNING", 0.00)
show_sample("MIDDLE", 0.50)
show_sample("END", 1.00)

In [ ]:
from pathlib import Path
import hashlib
import re

source = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/subtitle_corpus_v2/validated"
    / "2010_3_GOLMAAL_3_ENGLISH.srt"
)

output = source.with_name(
    "2010_3_GOLMAAL_3_ENGLISH_DECODED_UTF8.srt"
)

# Read the original using the encoding confirmed by our inspection.
original_bytes = source.read_bytes()
text = original_bytes.decode("cp1252")

# Save as a separate UTF-8 copy.
if output.exists():
    raise FileExistsError(
        f"Output already exists; not overwriting it: {output}"
    )

output.write_text(text, encoding="utf-8", newline="")

# Verify the new copy.
verified_text = output.read_text(encoding="utf-8")
timestamp_pattern = re.compile(
    r"(?m)^\s*\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*-->\s*"
    r"\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*$"
)

print("Original preserved:", source.exists())
print("Original SHA256 unchanged:",
      hashlib.sha256(source.read_bytes()).hexdigest()
      == hashlib.sha256(original_bytes).hexdigest())
print("Decoded copy created:", output.exists())
print("Decoded copy encoding: UTF-8")
print("Timestamp count:", len(timestamp_pattern.findall(verified_text)))
print("Output size:", output.stat().st_size, "bytes")
print("First dialogue lines:")

for line in verified_text.splitlines():
    if line.strip() and not timestamp_pattern.match(line) and not line.strip().isdigit():
        print(line[:150])
        break

In [ ]:
from pathlib import Path

folder = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/subtitle_corpus_v2/incoming_manual_missing_movies"
)

matches = [
    p for p in folder.rglob("*")
    if p.is_file()
    and "CHENNAI_EXPRESS" in p.name.upper()
    and p.suffix.lower() == ".srt"
]

print("Chennai Express files found:", len(matches))

for p in matches:
    print("\nFilename:", p.name)
    print("Size:", p.stat().st_size, "bytes")
    print("Path:", p)

In [ ]:
from pathlib import Path
import hashlib
import re

path = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/subtitle_corpus_v2/incoming_manual_missing_movies"
    / "2013_2_CHENNAI_EXPRESS_ENGLISH.srt.srt"
)

data = path.read_bytes()

# Try common subtitle encodings without modifying the source.
text = None
encoding_used = None

for encoding in ["utf-8-sig", "utf-8", "cp1252", "utf-16"]:
    try:
        text = data.decode(encoding)
        encoding_used = encoding
        break
    except UnicodeError:
        continue

if text is None:
    raise UnicodeError("Could not decode the subtitle using tested encodings.")

timestamp_pattern = re.compile(
    r"(?m)^\s*\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*-->\s*"
    r"\d{1,2}:\d{2}:\d{2}[,.]\d{1,3}\s*$"
)

lines = text.splitlines()
timestamp_indices = [
    i for i, line in enumerate(lines)
    if timestamp_pattern.match(line)
]

print("Filename:", path.name)
print("Size:", len(data), "bytes")
print("SHA256:", hashlib.sha256(data).hexdigest())
print("Detected decoding:", encoding_used)
print("Timestamp count:", len(timestamp_indices))

for fraction, label in [
    (0.00, "BEGINNING"),
    (0.25, "25%"),
    (0.50, "MIDDLE"),
    (0.75, "75%"),
    (1.00, "END"),
]:
    if not timestamp_indices:
        break

    idx = timestamp_indices[
        round(fraction * (len(timestamp_indices) - 1))
    ]

    print(f"\n--- {label} ---")
    shown = 0

    for line in lines[idx + 1:idx + 12]:
        line = line.strip()

        if not line or line.isdigit() or timestamp_pattern.match(line):
            continue

        print(line[:180])
        shown += 1

        if shown == 4:
            break

In [ ]:
from pathlib import Path
import re
import shutil
import hashlib
import pandas as pd
from datetime import datetime

BASE = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/subtitle_corpus_v2"
)

source = (
    BASE / "incoming_manual_missing_movies"
    / "2013_2_CHENNAI_EXPRESS_ENGLISH.srt.srt"
)

destination = (
    BASE / "validated"
    / "2013_2_CHENNAI_EXPRESS_ENGLISH.srt"
)

if not source.exists():
    raise FileNotFoundError(source)

data = source.read_bytes()
text = data.decode("utf-8-sig")

timestamp_re = re.compile(
    r"^\s*(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*"
    r"(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*$"
)

def to_seconds(h, m, s, ms):
    return (
        int(h) * 3600
        + int(m) * 60
        + int(s)
        + int(ms.ljust(3, "0")[:3]) / 1000
    )

lines = text.splitlines()
timestamp_lines = []
malformed_timestamp_lines = []
invalid_durations = []
empty_text_cues = 0
cue_text = []

for i, line in enumerate(lines):
    if "-->" not in line:
        continue

    match = timestamp_re.match(line)

    if not match:
        malformed_timestamp_lines.append((i + 1, line))
        continue

    timestamp_lines.append(i + 1)
    vals = match.groups()
    start = to_seconds(*vals[:4])
    end = to_seconds(*vals[4:])

    if end <= start:
        invalid_durations.append((i + 1, line))

    # Find text lines until the next timestamp.
    j = i + 1
    text_lines = []

    while j < len(lines) and "-->" not in lines[j]:
        item = lines[j].strip()

        if item and not item.isdigit():
            text_lines.append(item)

        j += 1

    if not text_lines:
        empty_text_cues += 1

    cue_text.extend(text_lines)

word_count = len(re.findall(r"\b[\w']+\b", " ".join(cue_text)))

print("Timestamp lines:", len(timestamp_lines))
print("Malformed timestamp lines:", len(malformed_timestamp_lines))
print("Invalid-duration cues:", len(invalid_durations))
print("Empty-text cues:", empty_text_cues)
print("Approximate word count:", word_count)

# Strict gate: do not replace if timestamps are malformed,
# durations are invalid, or there is no usable subtitle text.
if (
    len(timestamp_lines) < 100
    or malformed_timestamp_lines
    or invalid_durations
    or word_count < 1000
):
    raise ValueError(
        "Structural validation did not pass. No files were replaced."
    )

# Preserve the old file before replacement.
backup = None

if destination.exists():
    backup = destination.with_name(
        destination.stem
        + "_POLISH_BACKUP_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
        + destination.suffix
    )
    shutil.copy2(destination, backup)

shutil.copy2(source, destination)

# Verify destination matches the new source byte-for-byte.
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
destination_hash = hashlib.sha256(destination.read_bytes()).hexdigest()

if source_hash != destination_hash:
    raise RuntimeError(
        "Hash mismatch after copy. Keep backup and investigate."
    )

print("\nREPLACEMENT SUCCESSFUL")
print("New validated file:", destination)
print("SHA256:", destination_hash)
print("Backup of previous file:", backup)
print("Original incoming file preserved:", source.exists())

In [ ]:
from pathlib import Path

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

search_roots = [
    PROJECT / "01_data/processed/subtitle_corpus_v2",
    PROJECT / "01_data/processed",
]

for root in search_roots:
    print(f"\nSearching: {root}")

    if not root.exists():
        print("Folder not found")
        continue

    matches = list(
        root.rglob("ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv")
    )

    if matches:
        for path in matches:
            print("MANIFEST FOUND:", path)
    else:
        print("No manifest found here")

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil
import hashlib

# --------------------------------------------------
# 1. Define confirmed paths
# --------------------------------------------------
PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

CORRECT_SUBTITLE = (
    WORK / "validated/2013_2_CHENNAI_EXPRESS_ENGLISH.srt"
)

# --------------------------------------------------
# 2. Verify files before changing anything
# --------------------------------------------------
assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"
assert CORRECT_SUBTITLE.exists(), (
    f"Corrected subtitle not found: {CORRECT_SUBTITLE}"
)

subtitle_bytes = CORRECT_SUBTITLE.read_bytes()
subtitle_hash = hashlib.sha256(subtitle_bytes).hexdigest()

print("Corrected subtitle:", CORRECT_SUBTITLE)
print("File size:", len(subtitle_bytes), "bytes")
print("SHA-256:", subtitle_hash)

# --------------------------------------------------
# 3. Read manifest and check its structure
# --------------------------------------------------
df = pd.read_csv(MANIFEST, dtype=str).fillna("")

print("\nManifest shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

assert "movie_record_id" in df.columns, (
    "Expected movie_record_id column is missing."
)

matches = df.index[df["movie_record_id"].str.strip() == "2013_2"].tolist()

assert len(matches) == 1, (
    f"Expected exactly one record for 2013_2; found {len(matches)}. "
    "Manifest has not been changed."
)

row_idx = matches[0]

# --------------------------------------------------
# 4. Back up the current manifest
# --------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup = WORK / (
    f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_"
    f"backup_before_chennai_express_update_{timestamp}.csv"
)

shutil.copy2(MANIFEST, backup)
print("\nManifest backup created:", backup)

# --------------------------------------------------
# 5. Update only the Chennai Express record
# --------------------------------------------------
if "validated_subtitle_file" in df.columns:
    df.at[row_idx, "validated_subtitle_file"] = str(CORRECT_SUBTITLE)
else:
    raise KeyError(
        "validated_subtitle_file column is missing. "
        "The original manifest remains unchanged."
    )

# Update the hash only if this column exists.
if "sha256" in df.columns:
    df.at[row_idx, "sha256"] = subtitle_hash

# Set status only if the existing schema supports it.
if "subtitle_status" in df.columns:
    df.at[row_idx, "subtitle_status"] = (
        "Structural_Pass_Content_Identity_Review_Pending"
    )

# Keep identity/language/title/year verification pending until
# those checks have been formally recorded in the manifest.
for col in [
    "content_identity_verified",
    "language_verified",
    "title_verified",
    "year_verified",
]:
    if col in df.columns:
        df.at[row_idx, col] = "False"

# --------------------------------------------------
# 6. Save safely, then read back and verify
# --------------------------------------------------
df.to_csv(MANIFEST, index=False)

check = pd.read_csv(MANIFEST, dtype=str).fillna("")
check_row = check.loc[
    check["movie_record_id"].str.strip() == "2013_2"
]

assert len(check_row) == 1
saved_path = check_row.iloc[0]["validated_subtitle_file"]
assert saved_path == str(CORRECT_SUBTITLE)

assert hashlib.sha256(Path(saved_path).read_bytes()).hexdigest() == subtitle_hash

print("\nManifest update successful.")
print("Movie record: 2013_2 — Chennai Express")
print("Saved subtitle path:", saved_path)
print("Subtitle hash verified:", subtitle_hash)
print("Other movie records were not intentionally modified.")
print("\nUpdated record:")
print(check_row.T)

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import hashlib
import re
import shutil

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
SUBTITLE = WORK / "validated/2013_2_CHENNAI_EXPRESS_ENGLISH.srt"

assert MANIFEST.exists() and SUBTITLE.exists()

df = pd.read_csv(MANIFEST, dtype=str).fillna("")
matches = df.index[df["movie_record_id"].str.strip() == "2013_2"].tolist()
assert len(matches) == 1, f"Expected one Chennai Express row; found {len(matches)}"

i = matches[0]
content = SUBTITLE.read_bytes()
text = content.decode("utf-8-sig", errors="strict")
sha256 = hashlib.sha256(content).hexdigest()

# Structural counts from the replacement itself
timestamp_pattern = re.compile(
    r"(?m)^\s*\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
    r"\d{2}:\d{2}:\d{2}[,.]\d{3}"
)
cue_count = len(timestamp_pattern.findall(text))
word_count = len(re.findall(r"\b[\w’'-]+\b", text, flags=re.UNICODE))
character_count = len(text)

# Backup before metadata reconciliation
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = WORK / (
    f"ICMB_130_SUBTITLE_ACQUISITION_MANIFEST_"
    f"backup_before_chennai_metadata_reconciliation_{stamp}.csv"
)
shutil.copy2(MANIFEST, backup)

# Update only the Chennai Express row
df.at[i, "validated_subtitle_file"] = str(SUBTITLE)

# The old candidate ID belonged to the replaced subtitle.
for col in ["source_id", "source_url"]:
    if col in df.columns:
        df.at[i, col] = ""

if "subtitle_source" in df.columns:
    df.at[i, "subtitle_source"] = "Manual replacement; source not recorded"

if "validation_status" in df.columns:
    df.at[i, "validation_status"] = "Structural_Pass_Content_Identity_Review_Pending"

if "quality_status" in df.columns:
    df.at[i, "quality_status"] = "Structural_Pass_Review_Pending"

if "encoding_status" in df.columns:
    df.at[i, "encoding_status"] = "UTF-8 with BOM"

if "dialogue_blocks" in df.columns:
    df.at[i, "dialogue_blocks"] = str(cue_count)

if "word_count" in df.columns:
    df.at[i, "word_count"] = str(word_count)

if "character_count" in df.columns:
    df.at[i, "character_count"] = str(character_count)

# Require verification against the replacement, not the old candidate.
for col in [
    "title_verified",
    "year_verified",
    "movie_identity_verified",
    "language_verified",
]:
    if col in df.columns:
        df.at[i, col] = "False"

if "notes" in df.columns:
    df.at[i, "notes"] = (
        "Replaced previous incorrect Polish subtitle candidate. "
        "Replacement structurally checked; sampled dialogue is consistent "
        "with Chennai Express. Source identifier not retained because it "
        "belongs to the previous candidate. Formal identity/language review pending. "
        f"Replacement SHA-256: {sha256}"
    )

df.to_csv(MANIFEST, index=False)

# Read-back verification
check = pd.read_csv(MANIFEST, dtype=str).fillna("")
row = check.loc[check["movie_record_id"].str.strip() == "2013_2"]
assert len(row) == 1
assert row.iloc[0]["validated_subtitle_file"] == str(SUBTITLE)
assert hashlib.sha256(SUBTITLE.read_bytes()).hexdigest() == sha256
assert row.iloc[0].get("movie_identity_verified", "False").lower() != "true"

print("Metadata reconciliation successful.")
print("Backup:", backup)
print("Replacement SHA-256:", sha256)
print("Timestamp cues:", cue_count)
print("Word count:", word_count)
print("Character count:", character_count)
print("\nUpdated record:")
print(row.T)

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"

df = pd.read_csv(MANIFEST, dtype=str).fillna("")

print("=" * 65)
print("ICMB-130 — FINAL MANIFEST AND PATH AUDIT")
print("=" * 65)

# 1. Manifest integrity
print("\n1. MANIFEST INTEGRITY")
print("Rows:", len(df))
print("Unique movie IDs:", df["movie_record_id"].nunique())
print("Duplicate movie IDs:",
      int(df["movie_record_id"].duplicated().sum()))

# 2. Validated subtitle paths
def path_status(value):
    if not value.strip():
        return "EMPTY_PATH"
    p = Path(value)
    if p.exists():
        return "EXISTS"
    return "MISSING"

df["path_audit"] = df["validated_subtitle_file"].apply(path_status)

print("\n2. VALIDATED SUBTITLE PATHS")
print(df["path_audit"].value_counts(dropna=False).to_string())

missing = df[df["path_audit"] != "EXISTS"]
if not missing.empty:
    print("\nRecords with missing/empty validated paths:")
    print(missing[["movie_record_id", "movie_title",
                   "validated_subtitle_file", "path_audit"]]
          .to_string(index=False))

# 3. Check repeated validated paths
nonempty = df[df["validated_subtitle_file"].str.strip() != ""]
duplicates = nonempty[
    nonempty["validated_subtitle_file"].duplicated(keep=False)
]

print("\n3. DUPLICATED VALIDATED PATHS")
if duplicates.empty:
    print("None")
else:
    print(duplicates[["movie_record_id", "movie_title",
                      "validated_subtitle_file"]]
          .sort_values("validated_subtitle_file")
          .to_string(index=False))

# 4. Chennai Express replacement check
print("\n4. CHENNAI EXPRESS REPLACEMENT")
ce = df[df["movie_record_id"].str.strip() == "2013_2"]

assert len(ce) == 1, (
    f"Expected one Chennai Express record, found {len(ce)}"
)

ce_path = Path(ce.iloc[0]["validated_subtitle_file"])
expected_path = WORK / "validated/2013_2_CHENNAI_EXPRESS_ENGLISH.srt"

print("Registered path:", ce_path)
print("Expected path:", expected_path)
print("Path matches:", ce_path == expected_path)
print("File exists:", ce_path.exists())

if ce_path.exists():
    print("SHA-256:", hashlib.sha256(ce_path.read_bytes()).hexdigest())

# 5. Verification status summary
print("\n5. VERIFICATION STATUS")
for col in [
    "title_verified",
    "year_verified",
    "movie_identity_verified",
    "language_verified",
    "validation_status",
    "quality_status",
]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].replace("", "(blank)").value_counts().to_string())

# 6. Overall result
print("\n6. AUDIT SUMMARY")
checks = {
    "Exactly 130 manifest rows": len(df) == 130,
    "130 unique movie IDs": df["movie_record_id"].nunique() == 130,
    "No duplicate movie IDs": not df["movie_record_id"].duplicated().any(),
    "All validated paths exist": (df["path_audit"] == "EXISTS").all(),
    "Chennai Express path matches": (
        len(ce) == 1
        and ce.iloc[0]["validated_subtitle_file"] == str(expected_path)
        and expected_path.exists()
    ),
}

for label, passed in checks.items():
    print(f"{'PASS' if passed else 'REVIEW'} — {label}")

print("\nAudit complete. No files or manifest rows were changed.")

In [ ]:
from pathlib import Path
import pandas as pd
import re

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

REPORT = WORK / "ICMB_130_FINAL_CORPUS_READINESS_AUDIT.csv"

assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"

df = pd.read_csv(MANIFEST, dtype=str).fillna("")

# --------------------------------------------------
# 2. Normalize verification flags for reporting only
# --------------------------------------------------
def normalize_flag(value):
    value = str(value).strip().lower()

    if value in {"true", "yes", "verified"}:
        return "Verified"

    if value in {"false", "no"}:
        return "Not verified"

    if value == "":
        return "Blank"

    if "english" in value and "content" in value:
        return "Review required"

    return "Review required"

flag_columns = [
    "title_verified",
    "year_verified",
    "movie_identity_verified",
    "language_verified",
]

for col in flag_columns:
    if col in df.columns:
        df[f"{col}_audit"] = df[col].apply(normalize_flag)

# --------------------------------------------------
# 3. Check each validated subtitle file
# --------------------------------------------------
def inspect_subtitle(path_string):
    result = {
        "file_exists": False,
        "readable": False,
        "timestamp_count": 0,
        "malformed_timestamp_count": 0,
        "empty_cue_count": 0,
        "word_count_audit": 0,
        "structure_status": "FILE_MISSING",
    }

    if not path_string.strip():
        return result

    path = Path(path_string)

    if not path.is_file():
        return result

    result["file_exists"] = True

    try:
        raw = path.read_bytes()

        # Try common subtitle encodings without changing the file.
        text = None
        for encoding in ["utf-8-sig", "utf-8", "cp1252", "latin-1"]:
            try:
                text = raw.decode(encoding)
                break
            except UnicodeDecodeError:
                continue

        if text is None:
            result["structure_status"] = "DECODE_FAILED"
            return result

        result["readable"] = True

        lines = text.splitlines()
        timestamp_pattern = re.compile(
            r"^\s*\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
            r"\d{2}:\d{2}:\d{2}[,.]\d{3}"
        )

        timestamp_indices = [
            i for i, line in enumerate(lines)
            if timestamp_pattern.match(line)
        ]

        result["timestamp_count"] = len(timestamp_indices)

        # Count malformed lines that appear to contain a timestamp arrow.
        result["malformed_timestamp_count"] = sum(
            "-->" in line and not timestamp_pattern.match(line)
            for line in lines
        )

        # Count cue blocks whose text is empty.
        empty_cues = 0
        for pos, start in enumerate(timestamp_indices):
            end = (
                timestamp_indices[pos + 1]
                if pos + 1 < len(timestamp_indices)
                else len(lines)
            )

            cue_text = [
                line.strip()
                for line in lines[start + 1:end]
                if line.strip()
                and not re.fullmatch(r"\d+", line.strip())
            ]

            if not cue_text:
                empty_cues += 1

        result["empty_cue_count"] = empty_cues
        result["word_count_audit"] = len(
            re.findall(r"\b[\w’'-]+\b", text, flags=re.UNICODE)
        )

        if len(timestamp_indices) == 0:
            result["structure_status"] = "NO_TIMESTAMP_CUES"
        elif result["malformed_timestamp_count"] > 0 or empty_cues > 0:
            result["structure_status"] = "STRUCTURE_REVIEW"
        else:
            result["structure_status"] = "STRUCTURE_BASIC_PASS"

    except Exception as exc:
        result["structure_status"] = f"READ_ERROR: {type(exc).__name__}"

    return result

records = []

for _, row in df.iterrows():
    record = {
        "movie_record_id": row.get("movie_record_id", ""),
        "year": row.get("year", ""),
        "rank": row.get("rank", ""),
        "movie_title": row.get("movie_title", ""),
        "language": row.get("language", ""),
        "subtitle_language": row.get("subtitle_language", ""),
        "validation_status": row.get("validation_status", ""),
        "quality_status": row.get("quality_status", ""),
        "validated_subtitle_file": row.get("validated_subtitle_file", ""),
    }

    record.update(
        inspect_subtitle(row.get("validated_subtitle_file", ""))
    )

    for col in flag_columns:
        if col in df.columns:
            record[f"{col}_audit"] = normalize_flag(row.get(col, ""))

    records.append(record)

audit = pd.DataFrame(records)

# --------------------------------------------------
# 4. Coverage and review indicators
# --------------------------------------------------
def status_needs_review(value):
    value = str(value).strip().lower()
    return any(term in value for term in [
        "reject",
        "pending",
        "review",
        "incomplete",
        "gap",
        "limitation",
        "flag",
    ])

audit["validation_needs_review"] = audit["validation_status"].apply(
    status_needs_review
)
audit["quality_needs_review"] = audit["quality_status"].apply(
    status_needs_review
)

audit["verification_incomplete"] = audit.apply(
    lambda row: any(
        row.get(f"{col}_audit", "Blank") != "Verified"
        for col in flag_columns
    ),
    axis=1,
)

audit["basic_structure_pass"] = (
    audit["structure_status"] == "STRUCTURE_BASIC_PASS"
)

# --------------------------------------------------
# 5. Save report
# --------------------------------------------------
audit.to_csv(REPORT, index=False)

print("=" * 65)
print("ICMB-130 FINAL CORPUS READINESS AUDIT")
print("=" * 65)

print("\nRecords:", len(audit))
print("Unique movie IDs:", audit["movie_record_id"].nunique())

print("\nFile and decoding status:")
print(audit["structure_status"].value_counts().to_string())

print("\nVerification status:")
for col in flag_columns:
    audit_col = f"{col}_audit"
    print(f"\n{col}:")
    print(audit[audit_col].value_counts().to_string())

print("\nValidation status:")
print(audit["validation_status"].value_counts().to_string())

print("\nQuality status:")
print(audit["quality_status"].value_counts().to_string())

print("\nSummary:")
print("Files present:", int(audit["file_exists"].sum()))
print("Files readable:", int(audit["readable"].sum()))
print("Basic structure pass:", int(audit["basic_structure_pass"].sum()))
print("Records needing validation review:",
      int(audit["validation_needs_review"].sum()))
print("Records needing quality review:",
      int(audit["quality_needs_review"].sum()))
print("Records with incomplete verification:",
      int(audit["verification_incomplete"].sum()))

print("\nReport saved to:")
print(REPORT)

print("\nThis audit did not modify the manifest or subtitle files.")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

audit_path = WORK / "ICMB_130_FINAL_CORPUS_READINESS_AUDIT.csv"
assert audit_path.exists(), f"Audit not found: {audit_path}"

audit = pd.read_csv(audit_path, dtype=str).fillna("")

exceptions = audit[
    audit["structure_status"].isin(
        ["STRUCTURE_REVIEW", "NO_TIMESTAMP_CUES", "DECODE_FAILED"]
    )
].copy()

columns = [
    "movie_record_id",
    "year",
    "movie_title",
    "subtitle_language",
    "structure_status",
    "timestamp_count",
    "malformed_timestamp_count",
    "empty_cue_count",
    "validation_status",
    "quality_status",
    "validated_subtitle_file",
]

columns = [c for c in columns if c in exceptions.columns]

print("Structural exceptions:", len(exceptions))
print(exceptions[columns].to_string(index=False))

In [ ]:
from pathlib import Path
import re

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

files = {
    "2021_3 Master": (
        WORK / "validated/2021_3_MASTER_ENGLISH.srt",
        "utf-8-sig",
    ),
    "2009_2 Magadheera": (
        WORK / "validated/2009_2_MAGADHEERA_ENGLISH.srt",
        "utf-16",
    ),
}

# Accept either HH:MM:SS,mmm or MM:SS,mmm timestamp formats.
timestamp_pattern = re.compile(
    r"^\s*(?:\d{2}:)?\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
    r"(?:\d{2}:)?\d{2}:\d{2}[,.]\d{3}\s*$"
)

for movie, (path, encoding) in files.items():
    print("\n" + "=" * 70)
    print(movie)

    text = path.read_bytes().decode(encoding, errors="strict")
    lines = text.splitlines()

    timestamp_lines = [
        line for line in lines
        if "-->" in line and timestamp_pattern.match(line)
    ]

    malformed = [
        line for line in lines
        if "-->" in line and not timestamp_pattern.match(line)
    ]

    print("Decoded as:", encoding)
    print("File exists:", path.exists())
    print("Total lines:", len(lines))
    print("Recognized timestamp lines:", len(timestamp_lines))
    print("Malformed timestamp-like lines:", len(malformed))
    print("First timestamp:", timestamp_lines[0] if timestamp_lines else "None")
    print("Last timestamp:", timestamp_lines[-1] if timestamp_lines else "None")

    print("\nFirst 8 decoded lines:")
    for line in lines[:8]:
        print(repr(line))

print("\nRead-only recheck complete. No files or manifest rows changed.")

In [ ]:
from pathlib import Path
from collections import Counter
import re

path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/validated/"
    "2021_3_MASTER_ENGLISH.srt"
)

text = path.read_bytes().decode("utf-8-sig")
lines = text.splitlines()

timestamp_like = [line.strip() for line in lines if "-->" in line]

print("Total lines containing '-->':", len(timestamp_like))

print("\nFirst 30 timestamp-like lines:")
for line in timestamp_like[:30]:
    print(repr(line))

print("\nExamples of lines not matching the common SRT timestamp format:")
common = re.compile(
    r"^\s*(?:\d{2}:)?\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
    r"(?:\d{2}:)?\d{2}:\d{2}[,.]\d{3}\s*$"
)

unmatched = [line for line in timestamp_like if not common.match(line)]

print("Unmatched count:", len(unmatched))
for line in unmatched[:30]:
    print(repr(line))

print("\nNo files or manifest records were changed.")

In [ ]:
from pathlib import Path
import re

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

FILES = {
    "Master": (
        WORK / "validated/2021_3_MASTER_ENGLISH.srt",
        "utf-8-sig",
    ),
    "Magadheera": (
        WORK / "validated/2009_2_MAGADHEERA_ENGLISH.srt",
        "utf-16",
    ),
}

# Accept  MM:SS,mmm  OR  H:MM:SS,mmm  OR  HH:MM:SS,mmm
TS = re.compile(
    r"^\s*(\d{1,2}:)?\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
    r"(\d{1,2}:)?\d{2}:\d{2}[,.]\d{3}\s*$"
)

def to_ms(value):
    value = value.strip().replace(",", ".")
    parts = value.split(":")
    if len(parts) == 2:
        minutes, seconds = parts
        hours = 0
    elif len(parts) == 3:
        hours, minutes, seconds = parts
    else:
        raise ValueError(f"Unexpected timestamp: {value}")

    return (
        int(hours) * 3_600_000
        + int(minutes) * 60_000
        + round(float(seconds) * 1000)
    )

for name, (path, encoding) in FILES.items():
    text = path.read_bytes().decode(encoding)
    lines = text.splitlines()

    valid = []
    malformed = []

    for line in lines:
        if "-->" not in line:
            continue

        if TS.match(line):
            left, right = line.split("-->", 1)
            try:
                start = to_ms(left)
                end = to_ms(right)
                valid.append((start, end))
            except Exception:
                malformed.append(line)
        else:
            malformed.append(line)

    nonpositive = sum(end <= start for start, end in valid)
    out_of_order = sum(
        valid[i][0] < valid[i - 1][0]
        for i in range(1, len(valid))
    )

    print("\n" + "=" * 60)
    print(name)
    print("Timestamp lines recognized:", len(valid))
    print("Unrecognized/malformed timestamp lines:", len(malformed))
    print("Non-positive durations:", nonpositive)
    print("Out-of-order start times:", out_of_order)
    print("First timestamp:", valid[0] if valid else None)
    print("Last timestamp:", valid[-1] if valid else None)

print("\nRead-only check complete; no files or manifest records changed.")

In [ ]:
from pathlib import Path
import re

path = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/validated/"
    "2009_2_MAGADHEERA_ENGLISH.srt"
)

text = path.read_bytes().decode("utf-16")
lines = text.splitlines()

pattern = re.compile(
    r"^\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*$"
)

def to_ms(value):
    h, m, s = value.replace(",", ".").split(":")
    return int(h) * 3600000 + int(m) * 60000 + round(float(s) * 1000)

cues = []

for i, line in enumerate(lines):
    match = pattern.match(line)
    if match:
        cues.append({
            "line_index": i,
            "start": to_ms(match.group(1)),
            "end": to_ms(match.group(2)),
            "timestamp": line.strip(),
        })

anomalies = []

for j in range(1, len(cues)):
    if cues[j]["start"] < cues[j - 1]["start"]:
        anomalies.append((j - 1, j))

print("Ordering anomalies:", len(anomalies))

for previous_idx, current_idx in anomalies:
    for cue_idx in [previous_idx, current_idx]:
        cue = cues[cue_idx]
        print("\nCue index:", cue_idx)
        print("Timestamp:", cue["timestamp"])

        start_line = max(0, cue["line_index"] - 1)
        end_line = min(len(lines), cue["line_index"] + 4)

        for line in lines[start_line:end_line]:
            print(repr(line))

print("\nRead-only inspection; no files changed.")

In [ ]:
from pathlib import Path
import re
import hashlib

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

SOURCE = WORK / "validated/2009_2_MAGADHEERA_ENGLISH.srt"
CLEAN_DIR = WORK / "modelling_ready"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = CLEAN_DIR / "2009_2_MAGADHEERA_ENGLISH_CLEANED.srt"

text = SOURCE.read_bytes().decode("utf-16")
blocks = re.split(r"\r?\n\s*\r?\n", text.strip())

timestamp_pattern = re.compile(
    r"^\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*"
    r"(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*$"
)

def to_ms(value):
    h, m, s = value.replace(",", ".").split(":")
    return int(h) * 3_600_000 + int(m) * 60_000 + round(float(s) * 1000)

kept_blocks = []
removed_blocks = []

for block in blocks:
    lines = block.splitlines()
    timestamp_idx = next(
        (i for i, line in enumerate(lines) if "-->" in line),
        None
    )

    if timestamp_idx is None:
        kept_blocks.append(block)
        continue

    timestamp = lines[timestamp_idx].strip()
    match = timestamp_pattern.match(timestamp)

    cue_text = " ".join(
        line.strip() for line in lines[timestamp_idx + 1:]
    ).lower()

    is_promo = (
        "hdmovie8.com" in cue_text
        or "watch movies and series free" in cue_text
    )

    if is_promo:
        removed_blocks.append(block)
    else:
        kept_blocks.append(block)

assert len(removed_blocks) == 1, (
    f"Expected to remove exactly one promotional cue; "
    f"found {len(removed_blocks)}. No output written."
)

clean_text = "\n\n".join(kept_blocks).strip() + "\n"

# Validate chronology of the derived copy
clean_timestamps = []

for line in clean_text.splitlines():
    match = timestamp_pattern.match(line)
    if match:
        clean_timestamps.append(
            (to_ms(match.group(1)), to_ms(match.group(2)), line.strip())
        )

nonpositive = sum(end <= start for start, end, _ in clean_timestamps)
out_of_order = [
    (clean_timestamps[i - 1], clean_timestamps[i])
    for i in range(1, len(clean_timestamps))
    if clean_timestamps[i][0] < clean_timestamps[i - 1][0]
]

assert nonpositive == 0, "Non-positive durations remain; output not written."
assert len(out_of_order) == 0, (
    f"Other ordering anomalies remain: {len(out_of_order)}. "
    "Output not written."
)

# Write a UTF-8 derived copy; leave the original unchanged.
OUTPUT.write_text(clean_text, encoding="utf-8")

print("Original preserved:", SOURCE.exists())
print("Original SHA-256:", hashlib.sha256(SOURCE.read_bytes()).hexdigest())
print("Cleaned copy:", OUTPUT)
print("Removed promotional cues:", len(removed_blocks))
print("Timestamp cues remaining:", len(clean_timestamps))
print("Non-positive durations:", nonpositive)
print("Out-of-order timestamps:", len(out_of_order))
print("Cleaned copy SHA-256:", hashlib.sha256(OUTPUT.read_bytes()).hexdigest())

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

AUDIT = WORK / "ICMB_130_FINAL_CORPUS_READINESS_AUDIT.csv"
MANIFEST = WORK / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"

audit = pd.read_csv(AUDIT, dtype=str).fillna("")
manifest = pd.read_csv(MANIFEST, dtype=str).fillna("")

def true_count(series):
    return series.astype(str).str.strip().str.lower().isin(
        ["true", "1", "yes"]
    ).sum()

print("=" * 65)
print("ICMB-130 — FINAL STRUCTURAL AND QUALITY SUMMARY")
print("=" * 65)

print("\n1. CORPUS INVENTORY")
print("Manifest records:", len(manifest))
print("Unique movie IDs:", manifest["movie_record_id"].nunique())
print("Audit records:", len(audit))
print("Files present:", int(true_count(audit["file_exists"])))
print("Files readable:", int(true_count(audit["readable"])))

print("\n2. STRUCTURE STATUS FROM ORIGINAL AUDIT")
print(audit["structure_status"].value_counts().to_string())

print("\n3. INVESTIGATED FORMAT EXCEPTIONS")
print("Master (2021): 2,797 timestamps recognized with mixed-format parser.")
print("Magadheera (2009): 1,199 timestamps recognized after UTF-16 decoding.")
print("Magadheera cleaned modelling copy: 1,198 timestamps; chronology passes.")

if "empty_cue_count" in audit.columns:
    empty = pd.to_numeric(
        audit["empty_cue_count"], errors="coerce"
    ).fillna(0)

    flagged = audit.loc[empty > 0]

    print("\n4. EMPTY-CUE WARNINGS")
    print("Records with empty-cue warnings:", len(flagged))

    if not flagged.empty:
        cols = [
            "movie_record_id", "movie_title", "empty_cue_count",
            "validation_status", "quality_status"
        ]
        cols = [c for c in cols if c in flagged.columns]
        print(flagged[cols].to_string(index=False))

print("\n5. VALIDATION AND QUALITY LABELS")
for col in ["validation_status", "quality_status"]:
    if col in manifest.columns:
        values = manifest[col].fillna("").astype(str)
        mask = values.str.contains(
            r"reject|incomplete|pending|review|coverage.?gap|limitation",
            case=False,
            regex=True,
        )
        print(f"\n{col}: {int(mask.sum())} records with review-related labels")
        print(values[mask].value_counts().to_string())

print("\n6. VERIFICATION FLAGS")
for col in [
    "title_verified", "year_verified",
    "movie_identity_verified", "language_verified"
]:
    if col in manifest.columns:
        print(f"\n{col}:")
        print(
            manifest[col].replace("", "(blank)")
            .value_counts()
            .to_string()
        )

print("\n" + "=" * 65)
print("SUMMARY COMPLETE — NO FILES OR MANIFEST ROWS MODIFIED")
print("=" * 65)

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import hashlib

PROJECT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
WORK = PROJECT / "01_data/processed/subtitle_corpus_v2"

assert WORK.exists(), f"Project folder not found: {WORK}"

# Files and directories that must be preserved.
PROTECTED_FILES = {
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv",
    "ICMB_130_FINAL_CORPUS_READINESS_AUDIT.csv",
}

PROTECTED_DIRS = {
    "raw",
    "validated",
    "incoming_manual_missing_movies",
    "incoming_manual_2017_2025",
    "modelling_ready",
}

files = [
    p for p in WORK.rglob("*")
    if p.is_file()
]

records = []
hash_groups = defaultdict(list)

for p in files:
    try:
        size = p.stat().st_size
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
    except Exception as exc:
        records.append({
            "path": str(p),
            "size_bytes": None,
            "sha256": "",
            "category": "REVIEW_READ_ERROR",
            "protected": True,
            "notes": str(exc),
        })
        continue

    relative_parts = p.relative_to(WORK).parts
    in_protected_dir = any(
        part in PROTECTED_DIRS for part in relative_parts[:-1]
    )

    name = p.name.lower()

    if in_protected_dir or p.name in PROTECTED_FILES:
        category = "PRESERVE"
        protected = True
        notes = "Core subtitle, derived modelling, or essential project file"
    elif "backup" in name:
        category = "BACKUP_REVIEW"
        protected = True
        notes = "Keep until backup redundancy is confirmed"
    elif any(term in name for term in [
        "audit", "report", "screening", "proposal",
        "resolution", "verification", "language_quality",
        "repair_log", "copy_report",
    ]):
        category = "REPORT_REVIEW"
        protected = False
        notes = "Potentially redundant report; inspect before cleanup"
    elif p.suffix.lower() in {".tmp", ".temp", ".log"}:
        category = "TEMP_REVIEW"
        protected = False
        notes = "Potential temporary file"
    else:
        category = "OTHER_REVIEW"
        protected = True
        notes = "Keep unless confirmed unnecessary"

    records.append({
        "path": str(p),
        "size_bytes": size,
        "sha256": digest,
        "category": category,
        "protected": protected,
        "notes": notes,
    })

    hash_groups[digest].append(str(p))

inventory = pd.DataFrame(records)

# Identify byte-identical duplicates without deleting any copies.
duplicate_paths = {
    path
    for paths in hash_groups.values() if len(paths) > 1
    for path in paths
}

inventory["byte_identical_duplicate"] = (
    inventory["path"].isin(duplicate_paths)
)

report_path = WORK / "ICMB_130_FILE_CLEANUP_INVENTORY.csv"
inventory.to_csv(report_path, index=False)

print("=" * 65)
print("PROJECT FILE CLEANUP INVENTORY")
print("=" * 65)

print("Files scanned:", len(inventory))
print("\nFiles by category:")
print(inventory["category"].value_counts().to_string())

print("\nByte-identical duplicate files:",
      int(inventory["byte_identical_duplicate"].sum()))

print("\nPotential report/temporary files:")
candidates = inventory[
    inventory["category"].isin([
        "REPORT_REVIEW", "TEMP_REVIEW", "BACKUP_REVIEW"
    ])
]
if candidates.empty:
    print("None identified.")
else:
    print(candidates[
        ["path", "size_bytes", "category", "notes"]
    ].to_string(index=False))

print("\nInventory saved to:", report_path)
print("\nNo files were deleted or moved.")

In [ ]:
from pathlib import Path
import pandas as pd

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

INVENTORY = WORK / "ICMB_130_FILE_CLEANUP_INVENTORY.csv"
assert INVENTORY.exists(), f"Inventory not found: {INVENTORY}"

inv = pd.read_csv(INVENTORY, dtype=str).fillna("")

# Keep these reports because they are current references.
KEEP_REPORTS = {
    "ICMB_130_FINAL_CORPUS_READINESS_AUDIT.csv",
    "ICMB_130_FILE_CLEANUP_INVENTORY.csv",
    "ICMB_130_SUBTITLE_CORPUS_COVERAGE_REPORT.csv",
    "ICMB_130_PATH_AUDIT_FINAL.csv",
}

# Identify old/intermediate reports only.
reports = inv[
    (inv["category"] == "REPORT_REVIEW")
    & (~inv["path"].apply(lambda p: Path(p).name in KEEP_REPORTS))
].copy()

# Identify byte-identical duplicates outside protected categories.
duplicates = inv[
    (inv["byte_identical_duplicate"].str.lower() == "true")
    & (inv["category"] != "PRESERVE")
].copy()

print("=" * 65)
print("CLEANUP PROPOSAL — NO FILES MOVED OR DELETED")
print("=" * 65)

print("\nIntermediate reports for review:", len(reports))
if not reports.empty:
    print(reports[["path", "size_bytes"]].to_string(index=False))

print("\nDuplicate files outside PRESERVE category:", len(duplicates))
if not duplicates.empty:
    print(
        duplicates[
            ["path", "size_bytes", "category", "sha256"]
        ].to_string(index=False)
    )

proposal_path = WORK / "ICMB_130_CLEANUP_PROPOSAL.csv"
pd.concat(
    [
        reports.assign(proposal_reason="INTERMEDIATE_REPORT_REVIEW"),
        duplicates.assign(proposal_reason="DUPLICATE_REVIEW"),
    ],
    ignore_index=True,
).drop_duplicates(subset=["path"]).to_csv(proposal_path, index=False)

print("\nProposal saved to:", proposal_path)

In [ ]:

from pathlib import Path
import pandas as pd
import hashlib

# ---------------------------------------------------------
# ICMB-130 CLEANUP — CONSERVATIVE REVIEW PLAN
# No files will be moved or deleted in this cell.
# ---------------------------------------------------------

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

PROPOSAL_PATH = WORK / "ICMB_130_CLEANUP_PROPOSAL.csv"
PLAN_PATH = WORK / "ICMB_130_REVIEWED_QUARANTINE_PLAN.csv"

proposal = pd.read_csv(PROPOSAL_PATH)

# Exact filenames selected for review only.
# We retain one copy of the corrected audit and review the duplicate.
REVIEW_NAMES = {
    "manual_batch_corrected_audit_20261028_193158.csv":
        "Byte-identical to the later corrected audit dated 193244; "
        "retain the later copy.",

    "ICMB130_BATCH01_HINDI_SEARCH_20261015_105348.csv":
        "Reported as 1 byte; likely empty. Verify before quarantine.",

    "batch03_tmdb_id_search_20261015_172631.csv":
        "Reported as 1 byte; likely empty. Verify before quarantine.",

    "batch03_imdb_id_search_20261015_172932.csv":
        "Reported as 1 byte; likely empty. Verify before quarantine.",

    "batch05_hindi_search_errors_20261024_195426.csv":
        "Reported as 1 byte; likely empty. Verify before quarantine.",

    "batch05_hindi_search_errors_20261024_195543.csv":
        "Reported as 1 byte; likely empty. Verify before quarantine.",

    "batch05_tmdb_id_search_errors_20261024_195804.csv":
        "Reported as 1 byte; likely empty. Verify before quarantine.",

    "batch05_remaining9_search_errors_20261024_200354.csv":
        "Reported as 1 byte; likely empty. Verify before quarantine.",

    "batch08_2016_search_errors_20261027_140052.csv":
        "Reported as 1 byte; likely empty. Verify before quarantine.",
}

rows = []

for _, item in proposal.iterrows():
    path = Path(str(item["path"]))
    if path.name not in REVIEW_NAMES:
        continue

    # Never include anything outside the subtitle-corpus workspace.
    try:
        path.resolve().relative_to(WORK.resolve())
    except ValueError:
        print("SAFETY SKIP — outside workspace:", path)
        continue

    if not path.is_file():
        print("SAFETY SKIP — file not found:", path)
        continue

    content = path.read_bytes()

    rows.append({
        "path": str(path),
        "filename": path.name,
        "actual_size_bytes": len(content),
        "actual_sha256": hashlib.sha256(content).hexdigest(),
        "reason_for_review": REVIEW_NAMES[path.name],
        "action": "REVIEW_ONLY_NOT_MOVED",
        "approved_for_quarantine": False
    })

plan = pd.DataFrame(rows)
plan.to_csv(PLAN_PATH, index=False)

print("=" * 65)
print("REVIEWED QUARANTINE PLAN — NO FILES MOVED OR DELETED")
print("=" * 65)
print("Candidates:", len(plan))
print("Plan saved:", PLAN_PATH)

if not plan.empty:
    print("\nCandidate details:")
    print(plan[[
        "filename", "actual_size_bytes",
        "reason_for_review", "action"
    ]].to_string(index=False))
else:
    print("No matching candidates found. Check the proposal filenames.")


In [ ]:

from pathlib import Path
import pandas as pd
import hashlib

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

PLAN_PATH = WORK / "ICMB_130_REVIEWED_QUARANTINE_PLAN.csv"
plan = pd.read_csv(PLAN_PATH)

print("=" * 70)
print("CANDIDATE VERIFICATION — READ ONLY")
print("=" * 70)

# Compare the two corrected audit reports.
audit_old = WORK / "manual_batch_corrected_audit_20261028_193158.csv"
audit_new = WORK / "manual_batch_corrected_audit_20261028_193244.csv"

if audit_old.is_file() and audit_new.is_file():
    old_bytes = audit_old.read_bytes()
    new_bytes = audit_new.read_bytes()

    print("\n1. Corrected audit comparison")
    print("Earlier file exists:", audit_old.exists())
    print("Later file exists:", audit_new.exists())
    print("Earlier SHA-256:", hashlib.sha256(old_bytes).hexdigest())
    print("Later SHA-256:  ", hashlib.sha256(new_bytes).hexdigest())
    print("Byte-identical:", old_bytes == new_bytes)
else:
    print("\n1. Corrected audit comparison: one or both files are missing.")

# Inspect the exact bytes of each one-byte candidate.
print("\n2. One-byte CSV inspection")
one_byte_rows = plan[plan["actual_size_bytes"] == 1]

for _, row in one_byte_rows.iterrows():
    path = Path(row["path"])

    try:
        path.resolve().relative_to(WORK.resolve())
    except ValueError:
        print("SAFETY SKIP — outside workspace:", path)
        continue

    if not path.is_file():
        print("MISSING:", path.name)
        continue

    data = path.read_bytes()
    print(
        f"{path.name} | size={len(data)} | "
        f"bytes={data!r} | sha256={hashlib.sha256(data).hexdigest()}"
    )

print("\n3. Safety summary")
print("Files moved:", 0)
print("Files deleted:", 0)
print("Manifest backups modified:", 0)
print("Subtitle files modified:", 0)
print("\nPaste this output before we decide what to quarantine.")


In [ ]:

from pathlib import Path
import pandas as pd
import hashlib
import shutil
from datetime import datetime

# ---------------------------------------------------------
# ICMB-130: QUARANTINE CONFIRMED REDUNDANT/EMPTY FILES
# Subtitle files, manifests, backups and master data excluded.
# ---------------------------------------------------------

WORK = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2"
)

CONFIRM_QUARANTINE = True  # Change to True only after reviewing the plan.

QUARANTINE = WORK / "quarantine_cleanup_20261029"
LOG_PATH = WORK / "ICMB_130_QUARANTINE_MOVE_LOG_20261029.csv"

# Exact allowlist: no other file can be moved by this cell.
EXPECTED = {
    "manual_batch_corrected_audit_20261028_193158.csv",
    "ICMB130_BATCH01_HINDI_SEARCH_20261015_105348.csv",
    "batch03_tmdb_id_search_20261015_172631.csv",
    "batch03_imdb_id_search_20261015_172932.csv",
    "batch05_hindi_search_errors_20261024_195426.csv",
    "batch05_hindi_search_errors_20261024_195543.csv",
    "batch05_tmdb_id_search_errors_20261024_195804.csv",
    "batch05_remaining9_search_errors_20261024_200354.csv",
    "batch08_2016_search_errors_20261027_140052.csv",
}

# Confirm the later identical report exists before proceeding.
KEEP_AUDIT = WORK / "manual_batch_corrected_audit_20261028_193244.csv"
REMOVE_AUDIT = WORK / "manual_batch_corrected_audit_20261028_193158.csv"

assert WORK.is_dir(), f"Workspace not found: {WORK}"
assert KEEP_AUDIT.is_file(), "Later corrected audit is missing."
assert REMOVE_AUDIT.is_file(), "Earlier corrected audit is missing."
assert KEEP_AUDIT.read_bytes() == REMOVE_AUDIT.read_bytes(), (
    "Audit files are no longer identical. Stop and review."
)

# Find candidates only from the reviewed plan.
plan_path = WORK / "ICMB_130_REVIEWED_QUARANTINE_PLAN.csv"
plan = pd.read_csv(plan_path)
candidates = plan[plan["filename"].isin(EXPECTED)].copy()

assert set(candidates["filename"]) == EXPECTED, (
    "The candidate list differs from the exact allowlist. Stop."
)
assert candidates["filename"].is_unique, "Duplicate candidate rows detected."

# Verify every candidate exists, remains inside WORK, and is unchanged.
records = []
for _, row in candidates.iterrows():
    src = Path(row["path"]).resolve()
    try:
        src.relative_to(WORK.resolve())
    except ValueError:
        raise RuntimeError(f"Safety stop: file outside workspace: {src}")

    assert src.is_file(), f"Candidate missing: {src}"
    assert src.name in EXPECTED, f"Unexpected filename: {src.name}"

    content = src.read_bytes()
    digest = hashlib.sha256(content).hexdigest()

    assert len(content) == int(row["actual_size_bytes"]), (
        f"Size changed since review: {src.name}"
    )
    assert digest == row["actual_sha256"], (
        f"Content changed since review: {src.name}"
    )

    # Verify the eight tiny CSVs contain only a newline.
    if src.name != REMOVE_AUDIT.name:
        assert content == b"\n", (
            f"Unexpected content in tiny CSV: {src.name}"
        )

    records.append({
        "filename": src.name,
        "source_path": str(src),
        "destination_path": str(QUARANTINE / src.name),
        "size_bytes": len(content),
        "sha256": digest,
        "status": "PENDING" if CONFIRM_QUARANTINE else "DRY_RUN_ONLY",
    })

move_plan = pd.DataFrame(records)

print("=" * 70)
print("QUARANTINE MOVE PLAN")
print("=" * 70)
print("Files:", len(move_plan))
print("Destination:", QUARANTINE)
print(move_plan[["filename", "size_bytes", "status"]].to_string(index=False))

if not CONFIRM_QUARANTINE:
    print("\nDRY RUN ONLY. No files moved.")
    print("Review the list, then set CONFIRM_QUARANTINE = True to proceed.")
else:
    # Refuse to overwrite anything or reuse a previous quarantine folder.
    assert not QUARANTINE.exists(), (
        f"Quarantine folder already exists; refusing to overwrite: {QUARANTINE}"
    )
    assert not LOG_PATH.exists(), (
        f"Move log already exists; refusing to overwrite: {LOG_PATH}"
    )

    # Preflight all destinations before moving any file.
    for record in records:
        assert not Path(record["destination_path"]).exists(), (
            f"Destination already exists: {record['destination_path']}"
        )

    QUARANTINE.mkdir(parents=False, exist_ok=False)

    moved = []
    try:
        for record in records:
            src = Path(record["source_path"])
            dst = Path(record["destination_path"])

            shutil.move(str(src), str(dst))

            # Verify moved file contents match the reviewed hash.
            assert dst.is_file(), f"Move verification failed: {dst}"
            moved_hash = hashlib.sha256(dst.read_bytes()).hexdigest()
            assert moved_hash == record["sha256"], (
                f"Hash mismatch after move: {dst.name}"
            )
            assert not src.exists(), f"Original path still exists: {src}"

            record["status"] = "MOVED_AND_HASH_VERIFIED"
            moved.append(record)

            # Persist progress after each verified move.
            pd.DataFrame(records).to_csv(LOG_PATH, index=False)

        print("\nQUARANTINE COMPLETE")
        print("Moved and hash-verified:", len(moved))
        print("Move log:", LOG_PATH)
        print("Original subtitles and manifest backups were not targeted.")

    except Exception as exc:
        print("\nWARNING: A move failed:", repr(exc))
        print("Attempting to restore files already moved...")

        for record in reversed(moved):
            src = Path(record["source_path"])
            dst = Path(record["destination_path"])
            if dst.exists() and not src.exists():
                shutil.move(str(dst), str(src))
                record["status"] = "RESTORED_AFTER_ERROR"

        pd.DataFrame(records).to_csv(LOG_PATH, index=False)
        print("Check the log and both folders before continuing.")
        raise
